In [ ]:
# Cell 1 — SELF-HEALING install (version-exact, restart-free).
# >>> BUILD BANNER (generated by stamp_build.py) >>>
# Regenerate with: python stamp_build.py   (never hand-edit)
_BUILD_ID = "ba5f4f8c22"
_BUILD_PHASES = [1, 2, 3, 4, 5, 6, 7, 8, 10, 11]
_BUILD_MARKERS = ['PHASE1', 'PHASE10', 'PHASE10_PHANTOM_AND_REGION_ARRIVALS', 'PHASE11', 'PHASE11_RENDER_LEGIBILITY', 'PHASE2', 'PHASE3', 'PHASE4', 'PHASE5A', 'PHASE6', 'PHASE7', 'PHASE8', 'PHASE8FIX_ROLE_HINT_HOISTED']
print(f"NOTEBOOK BUILD {_BUILD_ID} \u00b7 phases applied: "
      f"{', '.join(str(p) for p in _BUILD_PHASES) or 'none'}")
print("  If the phases above are not the ones you just patched in, you are "
      "running a STALE import -> File \u25b8 Import Notebook \u25b8 re-import.")
print("  (This list is computed from the notebook's own content. The old "
      "banner was a hardcoded 'v55' string that never changed, beside a "
      "'stale import' warning that printed unconditionally on every run — "
      "which is why a real fix could look like it had done nothing.)")
# <<< BUILD BANNER <<<
# Two Kaggle facts drive this design:
#   1. session VMs keep pip damage from earlier runs (restart resets memory,
#      NOT the disk) — so numpy/scipy files may be a broken mix;
#   2. the kernel PRE-LOADS numpy at startup — so changing numpy's version on
#      disk creates a memory/disk mismatch that only a restart clears.
# Therefore: force-reinstall numpy+scipy at EXACTLY the versions already
# recorded on this VM — heals any file damage while keeping the pre-loaded
# modules valid. No version movement -> no restart needed.
import importlib.metadata as _md
import subprocess, sys

def _v_of(_pkg):
    try:
        return _md.version(_pkg)
    except _md.PackageNotFoundError:
        return None
_np, _sp = _v_of("numpy"), _v_of("scipy")
# PORTABILITY: Kaggle pre-ships a fat scientific stack; a bare PyTorch pod
# ships almost none of it. Install EVERY Kaggle-assumed base package that is
# missing, in one shot, instead of crashing one import at a time.
# transformers >=4.47 imports DTensor, a torch>=2.5-only symbol — on a pod
# with older torch every `from transformers import ...` then dies with
# "cannot import name 'DTensor'" (this killed the CLIP challenger cell).
_BASE_SPECS = {"transformers": "transformers>=4.40,<4.47"}
_base_missing = [_BASE_SPECS.get(p, p)
                 for p in ("numpy", "scipy", "matplotlib", "pandas",
                           "transformers", "gdown")
                 if _v_of(p) is None]
if _base_missing:
    print(f"portability: installing Kaggle-assumed base packages "
          f"{_base_missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"]
                   + _base_missing, check=True)
    _np, _sp = _v_of("numpy"), _v_of("scipy")
# Guard against re-reinstalling pre-loaded C-extensions in Python 3.12
_needs_reinstall = False
try:
    import numpy as _chk_np
    import scipy as _chk_sp
    import torch as _chk_torch
except Exception:
    _needs_reinstall = True

if _needs_reinstall:
    print(f"healing in place: numpy=={_np}, scipy=={_sp}")
    for spec in (f"numpy=={_np}", f"scipy=={_sp}"):
        if "None" in spec:   # package absent — reinstalling "numpy==None" is noise
            continue
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--force-reinstall", "--no-deps", spec], check=False)
else:
    print(f"numpy=={_np}, scipy=={_sp} already loaded & healthy — skipping force-reinstall")
# boxmot==22.0.0 requires numpy>=2.2.0, which conflicts with the numpy
# pin above (Kaggle's VM ships numpy 2.0.2 — moving off it needs a restart,
# which this cell is designed to avoid). boxmot==19.0.0 is the newest
# release with an UNCONSTRAINED numpy requirement — verified to load
# osnet_x0_25_msmt17.pt cleanly via boxmot.reid.core.ReID end-to-end, and
# it installs cleanly alongside numpy==2.0.2 with no resolver conflict.
#
# Packages are installed ONE AT A TIME (not as one combined command) so a
# single conflicting pin can't silently take the rest of the install down
# with it — that's what happened last run: boxmot's conflict failed the
# WHOLE combined install, so supervision/ultralytics/openpyxl/jinja2 never
# installed either, even though the cell printed "HEALTHY" (that check only
# ever looked at numpy/scipy, not the other packages).
_torch_pin = ([f"torch=={_v_of('torch')}"] if _v_of("torch") else [])
PACKAGES = [
    "supervision==0.26.1",
    "ultralytics>=8.3.0",
    # hold torch STILL while boxmot resolves: boxmot needs torchvision, and on
    # a pod without torchvision pip would otherwise install the latest one,
    # which hard-pins (and silently UPGRADES) torch — the exact on-disk vs
    # in-memory mismatch this cell exists to avoid.
    " ".join(["boxmot==19.0.0"] + _torch_pin),
    "openpyxl",
    "jinja2",
    "insightface",
    "onnxruntime-gpu==1.19.0",
    # numpy re-pin LAST: anything above that bumped numpy is pulled straight
    # back to the pre-loaded version, so no restart is ever needed.
    f"numpy=={_np}",
]
install_failures = []
for pkg in PACKAGES:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"]
                       + pkg.split(),
                       capture_output=True, text=True)
    if r.returncode != 0:
        install_failures.append((pkg, r.stderr.strip()[-500:]))
        print(f"❌ FAILED to install {pkg}")
        print(f"   {r.stderr.strip()[-500:]}")
    else:
        print(f"✅ installed {pkg}")

# insightface's metadata depends on CPU `onnxruntime`, which lands NEXT TO
# onnxruntime-gpu and fights it for the same import namespace — if the CPU
# wheel wins, face analysis silently runs on CPU (10-30x slower; the observed
# never-finishing face scan). The GPU wheel must own the namespace alone.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "onnxruntime"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps", "onnxruntime-gpu==1.19.0"],
               check=False)

# health check in a FRESH interpreter (immune to this kernel's cached modules)
r = subprocess.run([sys.executable, "-c",
                    "import numpy, numpy.testing, numpy._core.strings, "
                    "scipy.optimize; "
                    "print('numpy', numpy.__version__, '| scipy OK')"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])

# Verify every package this notebook actually imports later, not just
# numpy/scipy — this is what would have caught last run's silent failure.
import_check = subprocess.run([sys.executable, "-c",
    "import supervision, ultralytics, boxmot, openpyxl, jinja2, insightface; "
    "print('all downstream imports OK')"],
    capture_output=True, text=True)

if r.returncode == 0 and import_check.returncode == 0 and not install_failures:
    print("✅ environment HEALTHY — just keep running, no restart needed")
else:
    print("❌ still broken — see failures above.")
    if import_check.returncode != 0:
        print("   import check failed:", import_check.stderr.strip()[-500:])


# 🎥 Multi-Venue Zone Analytics — POC (cafes, stores, restaurants — visual, one-go)

**Pipeline** (no training happens here — detection uses an already-trained model):

```
video ─► YOLO (person, stock model) ─► BotSORT+ReID (stable IDs) ─► polygon zones (staff role via zone dwell)
      ─► events (who, which zone, when) ─► 7 business answers + charts
      ─► annotated video (boxes+trails+zones) ─► minute-by-minute JSON (mentor contract)
```

| # | Question | Method |
|---|----------|--------|
| 1 | How many people entered? | line crossing at the doorway |
| 2 | How many got seated? | customer dwell in dining/table zone |
| 3 | How many waited too long? | merged dwell in waiting zone vs threshold |
| 4 | How long was reception unstaffed? | gaps in staff presence in reception zone |
| 5 | Avg seating → order | **PROXY**: first staff visit at table after party sits |
| 6 | Avg order → food | **PROXY**: second distinct staff visit |
| 7 | Server visits per table | staff visits per table per party |

**Visual checkpoints you will see below:** zone overlay preview → detection sanity
gallery → live-annotated snapshots → journey Gantt per person → occupancy timeline →
wait distribution → reception staffed/unattended bar → table-service timeline →
minute-by-minute table → downloadable annotated video.

**Kaggle setup (2 minutes):**
1. *Settings -> Accelerator -> GPU (T4)*.
2. Attach a dataset containing, for each venue: `<name>.mp4` + `zones_<name>.json`
   (same folder or anywhere under `/kaggle/input`), plus your `.pt` model weights.
3. Run All -- every video found gets discovered, processed, and answered automatically.

> ⚠️ Q5/Q6 are stated proxies — cameras can't literally see an order being taken.
> Zone NAMES drive everything: put any video + its `zones_<stem>.json` under
> `/kaggle/input` (same folder or anywhere) and the pipeline classifies each
> zone's role (wait / staff / seating / entry / service) from its name — no
> code change needed to go from a restaurant to a cafe to a store.


In [ ]:
# Cell 2 — CONFIG (venue-agnostic: cafes, stores, restaurants, anything)
from pathlib import Path
import torch
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
BASE = Path("/kaggle/working") if IS_KAGGLE else Path("/content") if IS_COLAB else Path(".")
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else BASE
OUTPUT_DIR = BASE / "poc_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- CLOSED-SET STAFF GALLERY (v42) --------------------------------------------
# Enroll known staff by dropping photos in a folder named 'staff_gallery'
# under working dir or input. Tracks matching their faces are pinned as staff
# with their filename as identity (e.g. "jane", "john").
STAFF_GALLERY_DIR = "staff_gallery"
STAFF_MATCH_THRESHOLD = 0.40      # Cosine similarity bar to match staff face
FACE_MODEL_NAME = "buffalo_l"     # "buffalo_l" (ResNet100 large) or "buffalo_sc" (MobileNet fast)

# -- RE-ID LONG GAP PERSISTENCE (v42) -------------------------------------------
# To track people persistently throughout the whole video, we set max gap
# limit to the entire video duration. To prevent false merges on similar clothes,
# non-face tiers (body appearance/attire) are restricted to a local gap window.
REID_MAX_GAP_S = 7200             # Full video Re-ID (e.g. 2 hours)
MAX_BODY_GAP_S = 480              # Capped local window (8 min) for body appearance/attire

# Rest of standard config...
_pt_files = list(INPUT_ROOT.rglob("*.pt")) if INPUT_ROOT.exists() else []
OSNET_WEIGHTS = next((p for p in _pt_files if "osnet" in p.name.lower()), None)
CLIP_REID_WEIGHTS = next((p for p in _pt_files if p.name.lower().startswith("clip_")), None)
REID_BACKBONE_STOCK = "clip_market1501.pt"
ENABLE_REID_CALIBRATION = True
CALIBRATION_AUTO_APPLY = False   # F5: was True. Auto-deriving thresholds every
                                 # run makes two runs of the same video
                                 # incomparable, so no A/B is attributable.
                                 # Calibration still runs and still reports; it
                                 # now writes its suggestion to
                                 # poc_output/calibration_<cam>.json to be
                                 # reviewed and pinned by hand.
ENABLE_FACE_CORROBORATION = True
FACE_MIN_DET_SCORE = 0.55
FACE_MIN_FACE_PX = 45
FACE_SIM_THRESHOLD = 0.35
ENABLE_FACE_VETO = True
FACE_VETO_MARGIN = 0.15
FACE_VETO_MAX_EDGE_SCORE = 0.80
DETECTOR_MODEL = "yolo11x.pt"   # v50: largest stock model — best no-training recall in crowds
# v48: auto-adopt a fine-tuned dense-scene detector when attached as a dataset
_ft_det = next((p for p in _pt_files if "crowdhuman" in p.name.lower()
                or p.name.lower() == "best.pt"), None)
if _ft_det:
    DETECTOR_MODEL = str(_ft_det)
    print(f"🎯 fine-tuned detector adopted: {_ft_det.name} (dense-scene CrowdHuman weights)")
VIDEO_START_CLOCK = None
WINDOW_S = 30
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FPS_TARGET = 15
CONF_THRESHOLD = 0.35
YOLO_IMGSZ = 1280
ENTRY_LINE_FLIP = True
TRACKER_MODE = "botsort-reid"   # "botsort-reid" | "boosttrack" | "bytetrack"
BOTSORT_MATCH_THRESH = 0.75
USE_REAL_ONLINE_REID = True
FORCE_REID_BACKBONE = None
OSNET_STOCK_VARIANT = "osnet_x0_25_msmt17.pt"
LIVE_APPEARANCE_THRESH = 0.50
LOST_TRACK_BUFFER_S = 60
NATIVE_FPS_OVERRIDE = None
SNAPSHOT_EVERY_S = 30
TRAIL_MODE = "moving"
HUD_SMOOTH_S = 2.0
ENABLE_CLAHE = False
GAP_MERGE_S = 15
MIN_EVENT_S = 2.0
MIN_SEATED_S = 60
WAIT_THRESHOLD_S = 600
VISIT_MIN_S = 8
PARTY_GAP_S = 120
MIN_PARTY_S = 60
ENABLE_REID_STITCH = True
REID_SIM_THRESHOLD = 0.60
REID_MIN_CROP_H = 50   # v53: was 70 — children are short; still allow their Re-ID crops
REID_HANDOFF_GAP_S = 4.0
REID_HANDOFF_PX = 160
REID_STATIONARY_PX = 60.0
REID_CROPS_PER_TRACK = 6
ANCHOR_SIM_THRESHOLD = 0.75
ENABLE_CROSS_VALIDATION = True
CROSS_VAL_GALLERY_SIM = 0.60
CROSS_VAL_ANCHOR_SIM = 0.75
ENABLE_ATTIRE_MERGE_TIER = True
HSV_MERGE_SIM_THRESHOLD = 0.75
ENABLE_FACE_MERGE_TIER = True
FACE_MERGE_SIM_THRESHOLD = 0.45
ENABLE_LIVE_IDENTITY_MEMORY = True
LIVE_REID_SIM_THRESHOLD = 0.62
# F4: this is a PER-FRAME displacement budget, so hardcoding it was always
# wrong — it only means anything relative to the analysis frame rate. Derived
# below as LIVE_REID_MAX_SPEED_PX_S / eff_fps. 560/4fps == 140px, i.e. exactly
# the old value at the old frame rate, so nothing changes for an old config.
LIVE_REID_MAX_SPEED_PX_S = 560.0
LIVE_REID_MAX_DIST_PX = 140.0   # fallback only; process_video() overrides
LIVE_REID_MEMORY_TTL_S = float("inf")  # Whole video — never expire live identity memory (works for ANY video length)
STAFF_OVERRIDE_MIN_S = 60
# v53: a served CUSTOMER can linger 60s at the counter, so absolute dwell
# alone mislabels them staff. Real staff also (a) occupy the staff zone for
# a big share of the whole video and (b) spend more time there than
# anywhere else. Both tests are video-length independent.
STAFF_MIN_VIDEO_SHARE = 0.35   # >=35% of the video inside the staff zone
STAFF_DOMINANCE_RATIO = 3.0    # staff-zone time must be >=3x their time elsewhere.
                               # A real cashier is overwhelming (e.g. 100s vs 5s =
                               # 20x) so 3x is safe; a served customer who shopped
                               # first never reaches it. NOTE: staff who roam the
                               # shop (restocking) may miss this — enroll their
                               # photo in staff_gallery/ for a definitive answer.
# -- v43 Quality & Plausibility Config (ported from teammate's notebook) --------
MIN_BLUR_VARIANCE = 15.0            # Laplacian variance threshold for blur rejection
MIN_CROP_PX_BLUR_GATE = 40          # Skip blur check on crops smaller than this
MIN_BODY_ASPECT = 0.75  # v53: was 1.0 — child proportions are wider/shorter than adult 1.0
MAX_BODY_ASPECT = 4.0               # Aspect ratio max for body crops
MAX_PLAUSIBLE_SPEED_PX = 220.0       # Max walking speed in pixels/second
SPATIAL_PENALTY_SCALE = 0.15        # How aggressively to penalize implausible distances
MAX_SPATIAL_PENALTY = 0.30          # Cap on spatial penalty
NEAR_GAP_S = 15.0                   # Below this gap, threshold is more lenient
FAR_GAP_S = 180.0                   # Above this gap, threshold is stricter
NEAR_GAP_BONUS = 0.04               # Threshold bonus for short gaps
FAR_GAP_PENALTY = 0.05              # Threshold penalty for long gaps

# -- v44 Robustness: occlusion guard + confidence hysteresis (UNIVERSAL) -------
# These are video-agnostic (ratios / geometry, no per-clip tuning). Flip any
# flag to False to fall back to exact v43 behaviour for an A/B on Kaggle.
ENABLE_CONF_HYSTERESIS    = True    # detect low, START track high, KEEP track low
DETECT_CONF_FLOOR         = 0.25    # feed detections >= this to the tracker (was CONF_THRESHOLD=0.35)
NEW_TRACK_CONF            = 0.45    # a NEW track is only born at/above this conf
KEEP_TRACK_CONF           = 0.20    # an EXISTING track survives 2nd-round assoc down to this
ENABLE_OCCLUSION_GUARD    = True    # freeze appearance anchor while two boxes overlap
OCCLUSION_IOU             = 0.30    # boxes with IoU >= this are mutually occluding
OCCLUSION_CONTAIN         = 0.60    # ...or one box >= 60% contained inside the other
ENABLE_COVISIBILITY_BLOCK = True    # never merge a birth into an id already on-screen

# -- v45 High-value levers (GMC, resolution-relative gates, run diagnostic) ----
ENABLE_GMC                = True    # camera-motion compensation (absorbs vibration/shake)
GMC_METHOD                = "sof"   # BoxMOT token: sof (sparse optical flow) | ecc | orb | sift
ENABLE_RESOLUTION_SCALING = True    # auto-scale live pixel gates by THIS video's resolution
REF_DIAGONAL_PX           = 1468.6  # 720p diagonal — resolution the pixel gates were tuned at

# -- v46 Global tracklet association (offline, optimal long-range linking) --
ENABLE_GLOBAL_TRACKLET = True    # Hungarian consolidation pass after greedy merge

# -- v47 Hand-off appearance veto (kills different-people spatial merges) ------
ENABLE_HANDOFF_APPEARANCE_VETO = True   # reject a pure-spatial hand-off/stationary
                                        # merge when both tracks have body embeddings
                                        # that clearly disagree (dense-scene fix).
HANDOFF_VETO_SIM = 0.30                 # cosine below this = clearly different person

# -- v48 flags --------------------------------------------------------------
ENABLE_HANDOFF_HSV_VETO = True   # HSV 2nd opinion on the SPATIAL tiers
ENABLE_APPEARANCE_HSV_VETO = True  # v53: same 2nd opinion on the GALLERY and
                                   # ANCHOR appearance tiers. The run log showed
                                   # 4 merges HSV rejected being allowed anyway,
                                   # one hub id absorbing 3 identities.
APPEARANCE_HSV_VETO_SIM = 0.45     # HSV cosine below this = actively contradicts   # 2nd-opinion veto: torso colors must not clearly disagree
HANDOFF_HSV_VETO_SIM    = 0.50   # HSV cosine below this within a hand-off gap = different clothes
ENABLE_DISPLAY_RENUMBER = True   # annotated video shows P1..PN by first appearance

# -- v53 carried-person suppression (woman holding baby = ONE customer) -------
ENABLE_CARRIED_SUPPRESS = True   # drop a detection living inside a bigger one
CARRIED_CONTAIN         = 0.70   # >=70% of the small box inside the big box
CARRIED_MAX_AREA_RATIO  = 0.45   # and small box <=45% of the big box area
# F2: containment alone cannot tell "carried baby" from "person standing
# further back" — on an oblique camera they are the same geometry, and the old
# rule deleted real guests. A box is now only suppressed if it is ALSO far
# shorter than a person standing at its own footline would be.
CARRIED_HEIGHT_TOL      = 0.55   # suppress only if height < 55% of expected
CARRIED_MIN_FIT_SAMPLES = 200    # below this we have no scene geometry yet,
                                 # so we suppress NOTHING (keeping a real
                                 # person is always cheaper than deleting one)
# The height test alone is NOT enough, and the test suite caught it: a guest
# whose legs are hidden by the door frame or the desk also has a box far
# shorter than their footline implies, because the box bottom is the OCCLUDER's
# edge, not their feet. Deleting occluded guests in a doorway is worse than the
# bug we set out to fix. The signal that separates them is where the HEAD is:
#   carried child  -> head sits at the carrier's chest, well DOWN their box
#   occluded guest -> head is at full height, at or ABOVE the front person's
# So a box is only "carried" if its top is also well below the carrier's top.
CARRIED_MIN_HEAD_DROP   = 0.15   # small box top >=15% down the big box
# F3: keep the head class from a CrowdHuman-finetuned detector instead of
# discarding it. Heads stay visible when bodies do not, so a head with no
# matching person box is a person the detector missed behind someone else.
# Recovery stays OFF until Phase 2 can score it against ground truth.
ENABLE_HEAD_RECOVERY    = False
HEAD_RECOVERY_MIN_CONF  = 0.35
# F1: IR is decided PER FRAME, not once per hour-long chunk.
IR_TRACK_FRAC           = 0.5    # a track this IR-heavy gets no colour evidence
PATCH_V56_PHASE1 = True
# ── PHASE 2: measurement ────────────────────────────────────────────────────
# EVAL_EXPORT keeps clean frames during the main pass. It is NOT needed for the
# labelling packages any more — Cell 22 re-decodes just the chosen windows,
# which is cheaper and lets the windows be picked AFTER seeing the whole run.
EVAL_EXPORT = False
PATCH_V56_PHASE2 = True
# ── PHASE 3: metres instead of pixels ───────────────────────────────────────
# Every value below is a statement about the WORLD, so it is correct in every
# part of the frame. The px equivalents above survive only as fallbacks for
# footage where no ground plane could be fitted.
MAX_WALK_SPEED_MPS  = 2.2    # brisk walk; a run is ~4 m/s but nobody sprints
                             # through a reception, and a loose gate merges
                             # different people
REID_HANDOFF_M      = 1.6    # a fragment dying and another appearing within
                             # 1.6 m is a hand-off, not a new person
REID_STATIONARY_M   = 0.6    # same spot = same seat = same person
GREET_PROXIMITY_M   = 1.5    # conversational distance. Still a PROXY: standing
                             # 1.5 m apart is not proof of a greeting.
GROUP_WINDOW_S      = 25.0   # arrivals this close in time...
GROUP_RADIUS_M      = 3.0    # ...and this close in space are one party
# Optional, and worth four minutes of your time: put real floor coordinates in
# the zones JSON and the plane becomes EXACT instead of estimated —
#   "ground_points": [{"image": [px, py], "world": [X_m, Z_m]}, ... x4]
# Measured on a synthetic camera: auto mode is ~30 cm out on a 20-degree tilt,
# four supplied points are 0 cm out at any tilt.
PATCH_V56_PHASE3 = True
# ── PHASE 4: the values below are now DEFAULTS ONLY ─────────────────────────
# Cell 3 overrides them from profile_<video-stem>.json (or a "profile" key in
# the zones file) and prints the diff. Editing this cell per venue is no longer
# how this works — write a profile next to the footage instead.
VENUE_PROFILE = None    # set by Cell 3
PATCH_V56_PHASE4 = True
# ── PHASE 6: the reusable code lives in the kevacv/ package ─────────────────
# Cell 2f materialises it and imports it. Edit kevacv/*.py and re-run
# patch_v56_phase6.py; the notebook copy is generated, never hand-edited.
PATCH_V56_PHASE6 = True
# ── PHASE 7: phantom removal, from the first real run ───────────────────────
# ~19% of detections on CAM.112 were not people. Two shapes, two filters.
ENABLE_SIZE_FILTER   = True    # D1: drop a box too BIG to be a person standing
SIZE_FILTER_TOL      = 2.5     #     at that footline. P3 (half the frame) is
                               #     4.34x; a tall person 0.79x; two people
                               #     merged into one box 1.29x. Aimed at 4x
                               #     absurdities, not at borderline calls.
# D3: the same idea as D2, asked of the LOCATION instead of the track id,
# because the plant and the mirror re-mint a fresh id every few seconds and a
# per-id test is structurally blind to them. Thresholds measured, not guessed:
# phantoms sit at size cv 0.003-0.004, a receptionist at her post at 0.043.
ENABLE_PHANTOM_FILTER = True   # PHASE10_PHANTOM_AND_REGION_ARRIVALS
PHANTOM_MIN_SPAN_S    = 240.0
PHANTOM_CENTRE_JITTER = 0.02
PHANTOM_SIZE_CV       = 0.015
ENABLE_STATIC_FILTER = True    # D2: drop a track that never moves and never
STATIC_MIN_LIFE_S    = 120.0   #     changes size. The potted plant sat at
STATIC_CENTRE_JITTER = 0.02    #     identical pixels all chunk and was labelled
STATIC_SIZE_JITTER   = 0.03    #     "staff". Anyone who crossed the door or had
                               #     a face recognised is exempt, always.
PATCH_V56_PHASE7 = True
# ── PHASE 8: PRIVACY — face recognition scope ───────────────────────────────
# Staff gallery matching (per-frame) is always on — staff are enrolled by name.
# Post-processing face embeddings for identity merge are restricted by FACE_SCOPE:
#   "staff_only" = only staff tracks get face embeddings for merge/veto/corroboration
#   "all"        = all tracks (requires explicit consent — set in venue profile)
FACE_SCOPE = "staff_only"
PATCH_V56_PHASE8 = True
# ── PHASE 5a: TRIAGE CASCADE ────────────────────────────────────────────────
# Triage spends compute where people are, skipping empty stretches honestly.
ENABLE_TRIAGE        = True    # Enable 2-pass triage cascade
TRIAGE_SCAN_EVERY_S  = 6.0     # Scan 1 frame every 6 s
TRIAGE_MIN_PEOPLE    = 1       # Sighting bar for activity
TRIAGE_PAD_S         = 20.0    # Lead-in / run-out padding (s)
TRIAGE_MERGE_GAP_S   = 45.0    # Gap between active segments to merge (s)
TRIAGE_MIN_SEGMENT_S = 10.0    # Minimum segment duration (s)
PATCH_V56_PHASE5A    = True
PATCH_V57_PHASEA     = True
PATCH_V58_PHASEC     = True
PATCH_V59_AUDITFIX   = True
PATCH_V60_TIER1      = True
PATCH_V62_D1GUARD    = True
PATCH_V64_STORAGE    = True
PATCH_V65_FACE4K     = True
PATCH_V66_LOGHYGIENE = True
# V66: ultralytics prints a deprecation line for the (working) half=
# arg on EVERY batch — 4,487 times per run in the executed log. Spam
# buries real alarms; the flag still applies FP16 as measured.
# The old warnings.filterwarnings() could NEVER work: ultralytics emits
# this through its own logging.Logger, not the warnings module — which is
# why "fixing" it changed nothing. Filter the logger instead.
import logging as _l66
_l66.getLogger("ultralytics").addFilter(
    lambda r: "'half' is deprecated" not in r.getMessage())
FACE_SOURCE_RECROP   = True   # V65: when a track's 720p crops yield no face,
                              # re-crop its best frames from the 4K SOURCE —
                              # 3x the pixels, the 45px face bar becomes real
FACE_RECROP_MAX_TRACKS = 120  # seek budget: tracks per chunk
FACE_RECROP_FRAMES     = 4    # source seeks per track
RENDER_DIRECT_H264   = True   # V64a: pipe render frames straight into
                              # ffmpeg/libx264 — no 1.5GB raw intermediate,
                              # no second encode pass. Auto-falls back to the
                              # cv2 writer when ffmpeg is unavailable.
D1_MAX_DROP_FRAC     = 0.12   # V62a: past this drop rate D1 doubles its own
                              # tolerance — 27%% of detections being "not
                              # people" means the FIT is wrong, not the people
D1_GUARD_WARMUP      = 2000   # decisions before the guard may judge the rate
DETECTOR_HALF        = True    # T1a: FP16 detector forward — tensor cores on.
                               # ReID/face stay FP32; identity is not gambled.
ENABLE_TENSORRT      = False   # T1c: opt-in — exports a cached .engine on
                               # first use (~5-10 min once). Verify ONE run
                               # against the gt baseline before trusting.
# T1 VERIFICATION RULE: first run after changing any T1 knob must be scored
# with gt_kit.py compare against the frozen baseline. Speed claims without
# that comparison are faith, and faith is how pipelines rot.
ENABLE_STAFF_GALLERY_SWEEP = True   # C2: try every track's face vs the
                                    # ENROLLED gallery post-run; matches pin to
                                    # that staff id, non-matches are discarded
ENABLE_IR_HARD_CUT   = False        # C4: rebuild tracker at a colour<->IR flip
                                    # (ablation decides — default off)
IR_CUT_MIN_GAP_S     = 30.0         # C4: debounce against dusk flicker
RUN_ABLATION         = False        # Cell 9e: re-run labelled windows under
                                    # each variant instead of the night run
DEDUP_NMS_IOU        = 0.70   # A1: class-agnostic NMS after the detector —
                              # two same-size boxes on one body become one
                              # BEFORE the tracker can mint two ids
RENDER_COAST_S       = 0.5    # A6: a box missing <= this long is interpolated
                              # in the RENDER ONLY, so blinking detections
                              # don't blink on screen
# Apply FACE_SCOPE from venue profile if present
if 'VENUE_PROFILE' in dir() and VENUE_PROFILE:
    _profile_face_scope = VENUE_PROFILE.get("privacy", {}).get("face_scope")
    if _profile_face_scope:
        FACE_SCOPE = _profile_face_scope
        print(f"📋 Profile override: FACE_SCOPE={FACE_SCOPE!r}")

# -- v53 post-occlusion swap re-validation + phantom mask zones ---------------
ENABLE_SWAP_REVALIDATION = True  # after two tracks exit an overlap, cross-check
                                 # their appearance and un-trade swapped ids
SWAP_MARGIN              = 0.10  # both must match the OTHER anchor by this margin
# zones named mask/ignore/mirror/reflection in zones_<stem>.json become
# DEAD AREAS: detections there are dropped (kills reflection/poster phantoms)

# ── v55 SCALE / SPEED (see Cell 2e for the 10-hour profile) ──────────────────
ANALYSIS_MAX_W      = 1280    # 720p. Frames are downscaled to this before detection.
                              # 4K -> 1920 is a 4x pixel saving with little recall
                              # cost at this camera distance; raise it (2560/3840)
                              # if far-field people are being missed and you have
                              # the GPU budget. ZONES ARE SCALED TO THIS.
FFMPEG_HWACCEL      = "auto"  # "auto" = use the T4's hardware decoder if this
                              # ffmpeg build has it, else fall back silently.
                              # "none" = force software. Decoding 4K is the
                              # single biggest cost per chunk, and NVDEC does
                              # it on a chip that is otherwise idle.
DET_BATCH           = 12      # T1b: was 4 — the T4 was starved; batching
                              # is bit-identical per frame. Identical
                              # output to batch=1 (same model, same frames) —
                              # it just stops paying GPU launch overhead 14,430
                              # times. 1 = old behaviour.
EVAL_EXPORT         = False   # True = keep clean frames for the window below so
EVAL_WINDOW         = (0.0, 120.0)   # they can be labelled as ground truth
USE_FFMPEG_READER   = True    # decode ONLY the frames we analyse, in C, all cores.
                              # cv2 decodes all 108,240 frames of a 4K chunk to use
                              # 14,430 of them — that alone was ~50 min per chunk.
PROXY_RENDER        = True    # pass 1 saves the frames it analysed as JPEGs; pass 2
                              # draws from those instead of decoding the file again.
PROXY_JPEG_QUALITY  = 72
RENDER_ONLY_OCCUPIED = False  # False = EVERY analysed frame reaches the video,
                              # so the annotated output is the full wall-clock
                              # hour with nothing cut. True skips empty frames
                              # (smaller file, but the video is no longer the
                              # footage — turn back on only for 10h archives).
# SPEED vs ACCURACY, the honest version: dropping frames costs IDENTITY
# (association between frames), not detection. So do not drop frames evenly —
# drop them where nothing is happening. Raise ANALYSIS_FPS (Cell 2e) to 8 and
# let the gate below skip the empty stretches: you get 8 fps association while
# people are in shot for roughly the cost of a flat 4 fps, because a reception
# is empty most of the night.
IR_SAT_THRESHOLD     = 12.0   # mean HSV saturation below this = greyscale
IR_CHROMA_THRESHOLD  = 6.0    # ...or mean |R-G|+|G-B| below this. The second
                              # test is the reliable one on DARK footage.
IR_DETECT_CONF_FLOOR = 0.20   # detection floor used on infrared/greyscale chunks
                              # (bodies score lower without colour)
MOTION_GATE         = True    # skip the detector entirely on still, empty frames
MOTION_IDLE_S       = 10.0    # ...but only after 10 s with no detection anywhere
MOTION_MIN_FRAC     = 0.002   # >=0.2% of pixels changed = something is happening
REEMBED_EVERY_S     = 4.0     # refresh a KNOWN track's appearance vector this often
                              # (was: every detection every frame, batch of 1)
EMBED_BATCH         = 24      # crops per Re-ID forward pass
FACE_MAX_TRIES      = 6       # face attempts per track before we stop paying for it
FACE_RETRY_EVERY_S  = 3.0
GLOBAL_TRACKLET_MAX_IDS = 900 # above this the dense Hungarian is skipped (A7)
CHUNK_TAG           = ""      # set per chunk by the multi-chunk runner
# ── SERVICE TARGETS — what "good" is, so the report says PASS/MISS instead of
# just handing over numbers. Set to None to drop a target from the report.
TARGETS = {
    "median_greet_seconds":     60,    # someone with them inside a minute
    "desk_covered_pct":         90,    # station staffed 90% of the night
    "arrived_to_an_empty_desk":  0,    # nobody should walk into an empty desk
    "walked_out_under_90s":      0,    # nobody should leave unserved
}

# ── WHERE THE FOOTAGE COMES FROM ─────────────────────────────────────────────
# True  = pull chunks from the shared Google Drive folder (Cell 2d).
# False = use an attached Kaggle DATASET (no download, no Drive quota, instant).
#         Put the .mp4, the zone .json and the staff photo(s) in one dataset.
USE_DRIVE = True   # PATCH_V61: footage lives in the Drive folder

# ── FIXED ANALYSIS WINDOW ────────────────────────────────────────────────────
# Analyse a slice instead of the whole video. This is the simple, predictable
# alternative to PEAK_ONLY: no scan, no guessing, just "the first 20 minutes".
# Set WINDOW_MIN = None for the whole thing.
WINDOW_START_MIN = 0
# B4: was 20, left over from testing. The first real run analysed 20 of the 60
# minutes and reported it as the chunk. Set an integer here only when you mean
# to analyse a slice, and the run log will say so loudly either way.
WINDOW_MIN       = None

# ── PEAK WINDOW MODE (Cell 9c) ───────────────────────────────────────────────
# True  = scan every chunk cheaply, find the busiest PEAK_WINDOW_MIN minutes of
#         the whole night, and run the full pipeline on ONLY that window.
#         Cell 9b (the full-night runner) skips itself.
# False = process every chunk, end to end.
PEAK_ONLY         = False
PEAK_WINDOW_MIN   = 20
PEAK_SCAN_EVERY_S = 6.0     # scan resolution: 1 frame per 6 s
PEAK_SCAN_IMGSZ   = 640     # cheap detector pass; counts people, never tracks
PEAK_CHUNKS       = None    # None = consider every chunk, or e.g. [1, 2]
PEAK_FORCE_START_MIN = None # None = auto-pick the busiest window.
                            # 0 = force the FIRST 20 min of the chosen chunk,
                            # 35 = force 35:00-55:00, etc.

DEMO_SCALE = False
print(f"env={'kaggle' if IS_KAGGLE else 'colab' if IS_COLAB else 'local'}  device={DEVICE}")
print(f"tracker -> {TRACKER_MODE} (online_reid={USE_REAL_ONLINE_REID})")
_det_kind = "fine-tuned dense-scene" if ("crowdhuman" in Path(DETECTOR_MODEL).name.lower()
                or Path(DETECTOR_MODEL).name.lower() == "best.pt") else "stock"
print(f"model -> {Path(DETECTOR_MODEL).name} ({_det_kind}, person-only; staff/customer "
      f"role comes from face gallery or zone dwell)")


def audit_config():
    """Print every tracking-critical parameter with its current value."""
    params = [
        ("DETECTOR_MODEL",          DETECTOR_MODEL,          "YOLO detector model file"),
        ("TRACKER_MODE",            TRACKER_MODE,            "Tracker algorithm: botsort-reid | boosttrack | bytetrack"),
        ("USE_REAL_ONLINE_REID",    USE_REAL_ONLINE_REID,    "True = real BoxMOT embeddings, False = Ultralytics proxy"),
        ("FACE_MODEL_NAME",         FACE_MODEL_NAME,         "InsightFace model pack: buffalo_l (ResNet100) | buffalo_sc (MobileNet)"),
        ("STAFF_GALLERY_DIR",       STAFF_GALLERY_DIR,       "Folder containing staff face photos for closed-set enrollment"),
        ("STAFF_MATCH_THRESHOLD",   STAFF_MATCH_THRESHOLD,   "Min cosine similarity to match a face to enrolled staff"),
        ("REID_SIM_THRESHOLD",      REID_SIM_THRESHOLD,      "Gallery merge bar (auto-calibrated if CALIBRATION_AUTO_APPLY)"),
        ("ANCHOR_SIM_THRESHOLD",    ANCHOR_SIM_THRESHOLD,    "Best-crop 1:1 match threshold (stricter than gallery)"),
        ("FACE_SIM_THRESHOLD",      FACE_SIM_THRESHOLD, "Face-tier merge threshold"),
        ("REID_MAX_GAP_S",          REID_MAX_GAP_S,          "Max time gap for face-based re-id (seconds)"),
        ("MAX_BODY_GAP_S",          MAX_BODY_GAP_S,          "Max time gap for body/attire re-id (seconds, local cap)"),
        ("LIVE_REID_MEMORY_TTL_S",  LIVE_REID_MEMORY_TTL_S,  "Live identity memory TTL (inf = whole video)"),
        ("REID_CROPS_PER_TRACK",    REID_CROPS_PER_TRACK,    "Max crops stored per track for gallery matching"),
        ("REID_MIN_CROP_H",         REID_MIN_CROP_H,         "Min crop height (px) to compute an embedding"),
        ("FACE_MIN_DET_SCORE",      FACE_MIN_DET_SCORE,      "Min face detection confidence"),
        ("FACE_MIN_FACE_PX",        FACE_MIN_FACE_PX,        "Min face bbox width (px) to trust a face crop"),
        ("REID_HANDOFF_PX",         REID_HANDOFF_PX, "Max pixel distance for hand-off merge"),
        ("REID_STATIONARY_PX",      REID_STATIONARY_PX, "Max pixel drift for stationary merge"),
        ("CALIBRATION_AUTO_APPLY",  CALIBRATION_AUTO_APPLY,  "Auto-apply this run's calibrated threshold"),
        ("ENABLE_FACE_MERGE_TIER",  ENABLE_FACE_MERGE_TIER,  "Use face similarity as an active merge signal"),
        ("ENABLE_ATTIRE_MERGE_TIER",ENABLE_ATTIRE_MERGE_TIER,"Use HSV attire similarity as a merge signal"),
        ("ENABLE_CROSS_VALIDATION", ENABLE_CROSS_VALIDATION, "HSV independently double-checks every merge"),
        ("FPS_TARGET",              FPS_TARGET,              "Frames per second to sample from video"),
        ("USE_FFMPEG_READER",       USE_FFMPEG_READER,       "Decode only the analysed frames (ffmpeg) vs every frame (cv2)"),
        ("PROXY_RENDER",            PROXY_RENDER,            "Render from pass-1 JPEGs instead of decoding the video twice"),
        ("RENDER_ONLY_OCCUPIED",    RENDER_ONLY_OCCUPIED,    "Annotated video skips frames with nobody in them"),
        ("MOTION_GATE",             MOTION_GATE,             "Skip the detector on still+empty frames"),
        ("REEMBED_EVERY_S",         REEMBED_EVERY_S,         "How often a known track's appearance vector is refreshed"),
    ]
    print("=" * 80)
    print("  CONFIG AUDIT — all tracking-critical parameters")
    print("=" * 80)
    for name, value, desc in params:
        try:
            val_str = str(value)
        except Exception:
            val_str = "?"
        print(f"  {name:30s} = {val_str:>10s}  | {desc}")
    print("=" * 80)

audit_config()


# -- DYNAMIC ZONE ROLE TAXONOMY & AI OVERRIDES ---------------------------------
ZONE_ROLE_KEYWORDS = {
    "entry":   ["entry", "door", "entrance", "gate", "doorway", "passageway"],
    "wait":    ["wait", "queue", "lobby", "line", "holding"],
    "staff":   ["staff", "reception", "host", "counter", "register",
                "checkout", "till", "cashier", "podium", "desk"],
    "seating": ["table", "seat", "dining", "booth", "seating"],
    "service": ["service", "bar", "kitchen", "prep"],
    "mask":    ["mask", "ignore", "mirror", "reflection", "phantom"],
    "walkway": ["walkway", "corridor", "path", "aisle"],
}
ZONE_AI_OVERRIDES = {}

def classify_zones(zone_names):
    """Context-aware zone role classifier.
    Handles compound names (e.g., 'dining_entrance', 'dining_entry_gate') by prioritizing
    entry/gate indicators over seating keywords, preventing misclassifications.
    Check ZONE_AI_OVERRIDES for Gemini VLM dynamic recommendations.
    """
    roles = {}
    for name in zone_names:
        if name in ZONE_AI_OVERRIDES:
            roles[name] = ZONE_AI_OVERRIDES[name]
            continue
        low = str(name).lower()
        is_entry_indicator = any(kw in low for kw in ["gate", "door", "entry", "entrance", "passageway", "archway", "portal"])
        matched = []
        for role, kws in ZONE_ROLE_KEYWORDS.items():
            if role == "seating" and is_entry_indicator:
                if not any(explicit in low for explicit in ["table", "booth", "chair", "seat"]):
                    continue
            if any(kw in low for kw in kws):
                matched.append(role)
        roles[name] = matched or ["other"]
    return roles

import cv2

def apply_clahe(img):
    """Contrast Limited Adaptive Histogram Equalization (CLAHE).
    Enhances contrast & silhouettes in Infrared (IR) black-and-white night footage
    so YOLO detects low-contrast people in shadows/greyscale.
    """
    if img is None or img.size == 0:
        return img
    try:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    except Exception:
        return img


def _safe_id(tid):
    """Safe ID converter for track IDs: handles both integer IDs (41) and named string IDs ('receptionist_sarah')."""
    if isinstance(tid, str) and not tid.isdigit():
        return tid
    try:
        return int(tid)
    except Exception:
        return str(tid)

# ── T1c: TensorRT engine (opt-in, cached) ───────────────────────────────────
# Exports the resolved detector to a .engine ONCE (~5-10 min on a T4) and
# repoints DETECTOR_MODEL. Ultralytics loads .engine transparently, so nothing
# downstream changes. Guarded: any failure leaves the PyTorch model in place.
if globals().get("ENABLE_TENSORRT") and str(DETECTOR_MODEL).endswith(".pt"):
    _eng = BASE / (Path(str(DETECTOR_MODEL)).stem + ".engine")
    if _eng.exists():
        DETECTOR_MODEL = str(_eng)
        print(f"⚡ TensorRT engine (cached): {_eng.name}")
    else:
        try:
            from ultralytics import YOLO as _Y
            print(f"⚡ exporting {DETECTOR_MODEL} -> TensorRT (one-time, "
                  f"~5-10 min)...")
            _exp = _Y(str(DETECTOR_MODEL)).export(
                format="engine", half=True, imgsz=YOLO_IMGSZ, batch=DET_BATCH,
                device=0)
            _expp = Path(_exp)
            if _expp.exists():
                if _expp.resolve() != _eng.resolve():
                    import shutil; shutil.copy(_expp, _eng)
                DETECTOR_MODEL = str(_eng)
                print(f"⚡ TensorRT engine ready: {_eng.name} — cache it in "
                      f"the Kaggle dataset to skip the export next session")
        except Exception as _te:
            print(f"⚡ TensorRT export failed ({_te}) — staying on PyTorch. "
                  f"Speed unchanged, accuracy unchanged.")


In [ ]:
# Cell 2a — OPTIONAL: self-contained CrowdHuman download + unzip
# ── Source 1 (RECOMMENDED): Hugging Face — the AUTHOR-OFFICIAL mirror ────────
# (sshao0516/CrowdHuman). Steps: 1) free account at huggingface.co
# 2) open huggingface.co/datasets/sshao0516/CrowdHuman and click Agree
# 3) create a token at huggingface.co/settings/tokens (read access)
# 4) paste it below OR add it as Kaggle secret named HF_TOKEN
# LICENSE NOTE: CrowdHuman is research/non-commercial — fine for this POC;
# for a commercial product, train on your own labeled frames (Path B) instead.
CH_HF       = True
CH_HF_REPO  = "sshao0516/CrowdHuman"
CH_HF_TOKEN = ""          # "hf_..." — or leave empty to use Kaggle secret HF_TOKEN
CH_HF_FILES = ["CrowdHuman_train01.zip", "CrowdHuman_train02.zip",
               "CrowdHuman_train03.zip", "annotation_train.odgt"]

CH_DL_DIR = BASE / "crowdhuman_raw"
_ch_staged = (CH_DL_DIR / "annotation_train.odgt").exists() and \
             any((CH_DL_DIR / "Images").glob("*.jpg")) if (CH_DL_DIR / "Images").exists() else False

# v53 fix: never download the dataset when a fine-tuned detector is already
# attached/adopted — the data would be pure waste (training gets skipped anyway)
_ft_active = ("crowdhuman" in Path(DETECTOR_MODEL).name.lower()
              or Path(DETECTOR_MODEL).name.lower() == "best.pt")
if _ft_active:
    print(f"OK fine-tuned detector active ({Path(DETECTOR_MODEL).name}) — "
          "skipping CrowdHuman download entirely")
elif CH_HF and not _ch_staged:
    _tok = CH_HF_TOKEN.strip()
    if not _tok:
        try:
            from kaggle_secrets import UserSecretsClient
            _tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            _tok = ""
    if not _tok:
        _tok = None   # dataset is public — anonymous download works; token only
        print("INFO no HF token found — trying anonymous download (works for "
              "public datasets; add secret HF_TOKEN only if this 401s)")
    if True:
        import zipfile, subprocess
        try:
            from huggingface_hub import hf_hub_download
        except ImportError:
            subprocess.run(["pip", "install", "-q", "huggingface_hub"])
            from huggingface_hub import hf_hub_download
        _imgs = CH_DL_DIR / "Images"
        _imgs.mkdir(parents=True, exist_ok=True)
        for _fn in CH_HF_FILES:
            try:
                print(f"HF DOWNLOAD {_fn} ...")
                _got = Path(hf_hub_download(repo_id=CH_HF_REPO, filename=_fn,
                                            repo_type="dataset", token=_tok))
                if _fn.endswith(".zip"):
                    print(f"UNZIP {_fn} ...")
                    with zipfile.ZipFile(_got) as _z:
                        _z.extractall(_imgs)
                else:
                    (CH_DL_DIR / _fn).write_bytes(_got.read_bytes())
            except Exception as _e:
                print(f"WARN {_fn}: {_e}")
                if "gated" in str(_e).lower() or "403" in str(_e) or "401" in str(_e):
                    print("     -> open huggingface.co/datasets/sshao0516/CrowdHuman "
                          "while logged in and click AGREE, then re-run.")
        # flatten any nested folders the zips created
        for _p in list(_imgs.rglob("*.jpg")):
            if _p.parent != _imgs:
                _tgt = _imgs / _p.name
                if not _tgt.exists():
                    _p.replace(_tgt)
        _n = sum(1 for _ in _imgs.glob("*.jpg"))
        _ok = (CH_DL_DIR / "annotation_train.odgt").exists()
        print(f"CrowdHuman staged from HF: {_n} images | annotation_train.odgt "
              f"{'present' if _ok else 'MISSING'}")
elif _ch_staged:
    print("OK CrowdHuman already staged — skipping downloads")

# ── Source 2 (fallback): Google Drive via gdown — official links are often
# dead or quota-limited; use only if HF is unavailable ───────────────────────
CH_DOWNLOAD = False
CH_GDRIVE = {
    # crowdhuman.org -> right-click each [Google Drive] link -> Copy link -> paste
    "CrowdHuman_train01.zip": "",
    "CrowdHuman_train02.zip": "",
    "CrowdHuman_train03.zip": "",
    "annotation_train.odgt":  "",
}
CH_DL_DIR = BASE / "crowdhuman_raw"
if CH_DOWNLOAD:
    import subprocess, zipfile
    _imgs = CH_DL_DIR / "Images"
    _imgs.mkdir(parents=True, exist_ok=True)
    try:
        import gdown
    except ImportError:
        subprocess.run(["pip", "install", "-q", "gdown"])
        import gdown
    for _fn, _url in CH_GDRIVE.items():
        if not _url.strip():
            print(f"SKIP {_fn}: no link pasted")
            continue
        _is_zip = _fn.endswith(".zip")
        _dst = (CH_DL_DIR / _fn)
        if (not _is_zip) and _dst.exists():
            print(f"OK {_fn} already present")
            continue
        print(f"DOWNLOAD {_fn} ...")
        try:
            gdown.download(url=_url, output=str(_dst), quiet=False, fuzzy=True)
        except Exception as _e:
            print(f"WARN download failed for {_fn}: {_e}\n"
                  "     Google Drive quota? Attach a Kaggle 'crowdhuman' dataset "
                  "instead and set CH_DOWNLOAD=False.")
            continue
        if _is_zip and _dst.exists():
            print(f"UNZIP {_fn} ...")
            with zipfile.ZipFile(_dst) as _z:
                _z.extractall(_imgs)
            _dst.unlink()   # reclaim disk immediately
    # flatten any nested folder the zip created, so images sit directly in Images/
    _jpgs = list(_imgs.rglob("*.jpg"))
    for _p in _jpgs:
        if _p.parent != _imgs:
            _tgt = _imgs / _p.name
            if not _tgt.exists():
                _p.replace(_tgt)
    _n = sum(1 for _ in _imgs.glob("*.jpg"))
    _odgt_ok = (CH_DL_DIR / "annotation_train.odgt").exists()
    print(f"CrowdHuman staged: {_n} images in {_imgs} | "
          f"annotation_train.odgt {'present' if _odgt_ok else 'MISSING'}")
    if not _odgt_ok:
        print("   -> paste the annotation_train.odgt Google Drive link too, or the "
              "training cell will skip.")


In [ ]:
# Cell 2b — OPTIONAL in-notebook DENSE-SCENE DETECTOR fine-tune (auto-skips)
# One-notebook flow: if a CrowdHuman dataset is attached and no fine-tuned
# detector exists yet, convert + train yolo11m HERE (~3-4 h once on a T4),
# then this very run's analytics uses the result. Otherwise this cell is a
# few print lines and nothing more.
import json
CH_TRAIN_IF_NEEDED = True     # False = never train in-notebook
CH_FAST   = True              # fast preset: ~1-1.5h on T4 x2 (slightly lower ceiling)
CH_LIMIT  = 4000 if CH_FAST else 6000   # CrowdHuman subset (0 = all 15k)
CH_EPOCHS = 10  if CH_FAST else 12
CH_IMGSZ  = 800 if CH_FAST else 960
CH_BATCH  = 8                 # per-GPU; auto-scaled when 2 GPUs are present

_ch_best = BASE / "runs" / "detect" / "crowdhuman" / "weights" / "best.pt"
_have_ft = ("crowdhuman" in Path(DETECTOR_MODEL).name.lower()
            or Path(DETECTOR_MODEL).name.lower() == "best.pt")
_odgt = None
for _root in (INPUT_ROOT, BASE):          # attached dataset OR in-notebook download
    if _root and _root.exists():
        _odgt = next(_root.rglob("annotation_train.odgt"), None)
        if _odgt:
            break

if _have_ft:
    print(f"OK fine-tuned dense-scene detector already active ({Path(DETECTOR_MODEL).name}) — no training needed")
elif _ch_best.exists():
    DETECTOR_MODEL = str(_ch_best)
    print(f"REUSE detector trained earlier in this session: {_ch_best}")
elif not CH_TRAIN_IF_NEEDED:
    print("INFO CH_TRAIN_IF_NEEDED=False — running with stock detector")
elif _odgt is None:
    print("INFO no CrowdHuman dataset attached (no annotation_train.odgt under input) — "
          "running with stock detector.\n     To enable one-notebook training: attach a "
          "Kaggle 'CrowdHuman' dataset containing Images/ + annotation_train.odgt.")
elif DEVICE != "cuda":
    print("WARN no GPU — skipping in-notebook training (would take days on CPU)")
else:
    import random as _rnd
    from PIL import Image as _Img
    _img_dir = _odgt.parent / "Images"
    if not _img_dir.is_dir():
        _img_dir = next((p for p in _odgt.parent.rglob("Images") if p.is_dir()), None)
    if _img_dir is None:
        print("WARN found annotation_train.odgt but no Images/ directory — skipping training")
    else:
        print(f"TRAIN one-time dense-scene fine-tune: CrowdHuman @ {_img_dir} "
              f"(subset={CH_LIMIT or 'all'}, epochs={CH_EPOCHS}, imgsz={CH_IMGSZ})")
        _out = BASE / "crowdhuman_yolo"
        _recs = [json.loads(l) for l in open(_odgt, encoding="utf-8")]
        _rnd.Random(0).shuffle(_recs)
        if CH_LIMIT:
            _recs = _recs[:CH_LIMIT]
        _nval = max(1, int(len(_recs) * 0.1))
        _n_img = 0
        for _split, _rs in (("val", _recs[:_nval]), ("train", _recs[_nval:])):
            (_out / "images" / _split).mkdir(parents=True, exist_ok=True)
            (_out / "labels" / _split).mkdir(parents=True, exist_ok=True)
            for _r in _rs:
                _stem = _r["ID"].replace("/", "_")
                _src = next((q for q in (_img_dir / (_r["ID"] + ".jpg"),
                                         _img_dir / (_stem + ".jpg")) if q.exists()), None)
                if _src is None:
                    continue
                with _Img.open(_src) as _im:
                    _W, _H = _im.size
                _lines = []
                for _gt in _r.get("gtboxes", []):
                    if _gt.get("tag") != "person" or (_gt.get("extra") or {}).get("ignore"):
                        continue
                    for _cls, _box, _ign in ((0, _gt.get("fbox"), False),
                                             (1, _gt.get("hbox"),
                                              (_gt.get("head_attr") or {}).get("ignore"))):
                        if not _box or _ign:
                            continue
                        _x, _y, _w, _h = _box
                        _x1, _y1 = max(0, min(_x, _W - 1)), max(0, min(_y, _H - 1))
                        _x2, _y2 = max(1, min(_x + _w, _W)), max(1, min(_y + _h, _H))
                        _bw, _bh = _x2 - _x1, _y2 - _y1
                        if _bw < 2 or _bh < 2:
                            continue
                        _lines.append(f"{_cls} {(_x1 + _bw / 2) / _W:.6f} "
                                      f"{(_y1 + _bh / 2) / _H:.6f} "
                                      f"{_bw / _W:.6f} {_bh / _H:.6f}")
                if not _lines:
                    continue
                _dst = _out / "images" / _split / (_stem + ".jpg")
                if not _dst.exists():
                    _dst.write_bytes(_src.read_bytes())
                (_out / "labels" / _split / (_stem + ".txt")).write_text("\n".join(_lines))
                _n_img += 1
        (_out / "data.yaml").write_text(
            f"path: {_out.resolve()}\ntrain: images/train\nval: images/val\n"
            "nc: 2\nnames: [person, head]\n")
        print(f"   converted {_n_img} images -> {_out}")
        from ultralytics import YOLO as _YOLO
        _ngpu = torch.cuda.device_count()
        _dev = list(range(_ngpu)) if _ngpu > 1 else 0
        _bat = CH_BATCH * max(1, _ngpu)
        print(f"   training on {_ngpu} GPU(s) (device={_dev}, batch={_bat}) — "
              f"select 'GPU T4 x2' in Kaggle settings to halve training time")
        _YOLO("yolo11m.pt").train(
            data=str(_out / "data.yaml"), epochs=CH_EPOCHS, imgsz=CH_IMGSZ,
            batch=_bat, device=_dev, workers=4,
            name="crowdhuman", project=str(BASE / "runs" / "detect"),
            exist_ok=True, patience=5, degrees=0.0, shear=0.0, perspective=0.0)
        if _ch_best.exists():
            DETECTOR_MODEL = str(_ch_best)
            print(f"ADOPTED fine-tuned detector for this run: {_ch_best}")
            print("   TIP: download best.pt (or save /kaggle/working as a dataset) so "
                  "future runs attach it and skip training entirely")
        else:
            print("WARN training finished but best.pt not found — continuing with stock detector")


In [ ]:
# Cell 2c — PRETRAINED CrowdHuman detector (no training, no dataset needed)
# Downloads a ready-made crowd-specialist model straight from GitHub and uses
# it. Runs ONLY if nothing better is active yet (attached fine-tuned weights or
# an in-notebook trained best.pt always win). Set CH_PRETRAINED=False to A/B
# against stock yolo11x instead.
CH_PRETRAINED = True
CH_PRETRAINED_URL = ("https://github.com/yakhyo/yolov8-crowdhuman/releases/"
                     "download/weights/yolov8n_best.pt")

if not CH_PRETRAINED:
    print(f"INFO CH_PRETRAINED=False — detector stays {Path(DETECTOR_MODEL).name}")
elif not Path(DETECTOR_MODEL).name.startswith("yolo11"):
    print(f"OK stronger detector already active ({Path(DETECTOR_MODEL).name}) — "
          "skipping pretrained download")
else:
    _pt = BASE / "yolov8n_crowdhuman.pt"
    if not _pt.exists():
        import urllib.request
        print(f"DOWNLOAD pretrained crowd detector:\n  {CH_PRETRAINED_URL}")
        try:
            urllib.request.urlretrieve(CH_PRETRAINED_URL, _pt)
        except Exception as _e:
            print(f"WARN download failed ({_e}) — keeping {Path(DETECTOR_MODEL).name}. "
                  "(Kaggle Internet toggle ON? GitHub reachable?)")
    if _pt.exists() and _pt.stat().st_size > 1_000_000:
        DETECTOR_MODEL = str(_pt)
        print(f"ADOPTED pretrained CrowdHuman detector: {_pt.name} "
              "(yolov8n crowd-specialist — nano-sized; an in-notebook trained "
              "best.pt will beat it, this is the zero-effort option)")
    elif _pt.exists():
        _pt.unlink()
        print("WARN downloaded file too small (bad download) — removed; "
              f"keeping {Path(DETECTOR_MODEL).name}")


In [ ]:
import json

In [ ]:
import shutil as _sh
# Cell 2d — DRIVE FETCH + PREFLIGHT  (reception analytics, CAM.112)
# Pulls archived NVR chunks straight from a shared Google Drive folder, so
# nothing is uploaded to Kaggle by hand. Runs BEFORE Cell 3's discovery and
# simply points INPUT_ROOT at what it downloaded — Cell 3 onward is untouched.
#
# FIRST RUN (no zone map in the folder yet):
#   downloads one chunk, writes frame_0.jpg, and stops with instructions.
#   Draw the zones on that JPG, upload the JSON to the same Drive folder,
#   then Run All again. Same notebook, same button, both times.

# ── SWITCHES ────────────────────────────────────────────────────────────────
FOLDER_ID   = "1CP-EdRntJ6LsApDddKI35tQniQigX8G0"  # PATCH_V61_FOLDER_PICK   # the part of the Drive URL after /folders/
CAMERA_ID   = "CAM.112"      # label used in every chart and export
MAX_CHUNKS  = None              # 1 = prove it (~8 min). None = the real run.
CHUNK_FILTER = "7.30.00pm"      # only chunks whose FILENAME contains this run.
                                # "7.30.00pm" = tonight's peak-hour-only demo.
                                # "" = every chunk (the full night).
RESUME      = True           # skip chunks whose events file already exists
STRICT      = True           # sanity failures raise instead of warn
DRIVE_TZ    = "CDT"          # ⚠ CAM.112 is Central, NOT Pacific
# ────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os, re, json, shutil
from pathlib import Path
from datetime import datetime

# Cell 3 defines VIDEO_EXTS, but it runs after this one.
VIDEO_EXTS_PRE = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}
DRIVE_DIR = BASE / "drive_chunks"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

class DriveQuotaError(RuntimeError):
    """Drive said no. Not a bug in the pipeline — see the message for fixes."""


def _fail(msg, fix):
    """One-line problem, one-line fix. Nothing downstream runs."""
    raise RuntimeError(f"PREFLIGHT FAILED: {msg}\n   FIX: {fix}")

def _list_drive_folder(folder_id):
    url = f"https://drive.google.com/drive/folders/{folder_id}"
    try:
        items = gdown.download_folder(url, skip_download=True, quiet=True) or []
    except Exception as e:
        _fail(f"could not list the Drive folder ({e})",
              "confirm sharing is 'Anyone with the link -> Viewer'")
    out = []
    for it in items:                      # gdown's return shape varies by version
        fid  = getattr(it, "id", None) or (it.get("id") if isinstance(it, dict) else None)
        path = getattr(it, "path", None) or (it.get("path") if isinstance(it, dict) else str(it))
        if fid:
            out.append({"id": fid, "name": Path(path).name})
    return out


def _clock_from_name(name):
    m = re.search(r"(\d{1,2})-(\d{1,2})-(\d{4}),\s*(\d{1,2})\.(\d{2})\.(\d{2})\s*(am|pm)",
                  name, re.I)
    if not m:
        return None
    mo, d, y, hh, mm, ss, ap = m.groups()
    hh = int(hh) % 12 + (12 if ap.lower() == "pm" else 0)
    return datetime(int(y), int(mo), int(d), hh, int(mm), int(ss))


def _pull(f):
    dest = DRIVE_DIR / f["name"]
    is_video = Path(f["name"]).suffix.lower() in VIDEO_EXTS_PRE
    required_size = 1_000_000 if is_video else 100
    if dest.exists() and dest.stat().st_size >= required_size:
        print(f"  ..  cached     {f['name']}")
        return dest
    print(f"  >>  download   {f['name']}")
    import time as _time
    last = None
    # download to a .part file and rename only after the size check, so an
    # interrupted download can never occupy the cache key as "cached".
    part = dest.with_suffix(dest.suffix + ".part")
    for _try in range(3):
        try:
            part.unlink(missing_ok=True)
            gdown.download(id=f["id"], output=str(part), quiet=False,
                           use_cookies=(_try > 0), fuzzy=True)
            if part.exists() and part.stat().st_size >= required_size:
                part.rename(dest)
                return dest
            last = f"file came back as {part.stat().st_size if part.exists() else 0} bytes"
        except Exception as _e:
            last = repr(_e)
        if _try < 2:
            _wait = 20 * (_try + 1)
            print(f"      download failed ({last}) — retrying in {_wait}s "
                  f"({_try + 2}/3)")
            try:
                part.unlink(missing_ok=True)
            except Exception:
                pass
            _time.sleep(_wait)
    raise DriveQuotaError(
        f"could not download {f['name']} after 3 tries: {last}\n"
        "   Google Drive throttles a file after repeated downloads "
        "('Cannot retrieve the public link ... many accesses').\n"
        "   FIXES, best first:\n"
        "     1. In Drive, right-click the file -> Make a copy. The copy has a "
        "fresh quota; share the copy's folder.\n"
        "     2. Upload the chunks once as a private KAGGLE DATASET and attach "
        "it — no Drive, no quota, and it mounts instantly at /kaggle/input.\n"
        "     3. Wait a few hours; the per-file limit resets on its own.\n"
        "     4. Scan fewer chunks: PEAK_CHUNKS = [1] in Cell 2.")


if USE_DRIVE:
    print("=" * 78)
    print("  PREFLIGHT")
    print("=" * 78)

    # 1 ── GPU
    try:
        import torch
        n_gpu = torch.cuda.device_count()
        print(f"  {'OK ' if n_gpu else 'WARN'} GPU        {n_gpu} device(s)"
              + ("" if n_gpu else "  — CPU only, this will be very slow"))
    except Exception as e:
        print(f"  WARN GPU        could not query ({e})")

    # 2 ── switches actually set
    if FOLDER_ID == "PASTE_HERE":
        _fail("FOLDER_ID is still the placeholder",
              "open the Drive folder, copy the id after /folders/, paste it above")

    # 3 ── internet (Kaggle ships it OFF by default; the failure looks like DNS)
    try:
        import urllib.request
        urllib.request.urlopen("https://drive.google.com", timeout=15)
        print("  OK  internet   drive.google.com reachable")
    except Exception:
        _fail("cannot reach drive.google.com",
              "Kaggle: Settings -> Internet -> ON, then re-run")

    # 4 ── gdown  (plain curl/wget silently saves Drive's virus-scan warning
    #      page as your .mp4 — a 4 KB 'video' with zero frames and no error)
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
        import gdown
    print(f"  OK  gdown      {getattr(gdown, '__version__', '?')}")

    # 5 ── list the folder WITHOUT downloading (33 GB will not fit in 20 GB)
    remote = _list_drive_folder(FOLDER_ID)
    if not remote:
        _fail("Drive folder listed as empty",
              "check the FOLDER_ID and that the link is set to Viewer, not Restricted")

    # sort CHRONOLOGICALLY by the clock in the filename, not lexicographically:
    # "10.30.00am" sorts before "4.30.00pm" as a string, and M-D-YYYY dates
    # don't sort as text either. Name is the tiebreak for unparseable files.
    vids  = sorted([f for f in remote if Path(f["name"]).suffix.lower() in VIDEO_EXTS_PRE],
                   key=lambda f: (_clock_from_name(f["name"]) or datetime.min, f["name"]))
    # Accept any json with "zone" in the name — zones_CAM.112.json,
    # CAM.112_zone.json, my_zones (1).json. The mapper's default filename and
    # whatever you actually typed should both just work.
    zjson = [f for f in remote
             if f["name"].lower().endswith(".json") and "zone" in f["name"].lower()]
    print(f"  OK  folder     {len(vids)} video(s), {len(zjson)} zone map(s)")
    if not vids:
        _fail("no video files in that folder", "check you shared the folder with the chunks in it")

    # ── FILTER FIRST (was applied at the very END, which is how the last run
    # parsed the clock from, downloaded, zone-staged and IR-profiled the 4:30pm
    # chunk while announcing 19:30: everything below must run on the chunk we
    # will actually process.
    if CHUNK_FILTER:
        _pre = len(vids)
        # V62b: filenames carry a RANGE ("6.30pm CDT - 7.30pm CDT"), so a bare
        # substring matched the previous hour's END time too. Match only the
        # text before the range separator (the chunk's START time).
        vids = [v for v in vids if CHUNK_FILTER in v["name"].split(" - ")[0]]
        print(f"  CHUNK_FILTER {CHUNK_FILTER!r}: {len(vids)} of {_pre} chunk(s) "
              f"match" + ("" if vids else "  !! NOTHING MATCHES — fix the filter"))
        if not vids:
            _fail(f"CHUNK_FILTER {CHUNK_FILTER!r} matches no file",
                  "clear it or match part of a real filename")

    # 6 ── disk
    free_gb = shutil.disk_usage(BASE).free / 1e9
    print(f"  {'OK ' if free_gb > 8 else 'WARN'} disk       {free_gb:.1f} GB free")
    if free_gb < 8:
        print("       (chunks are ~3 GB; we delete after each, but keep an eye on it)")

    # 7 ── start clock, straight out of the filename
    #      "CAM.112 (PP.09_12) 7-28-2026, 4.30.00pm CDT - ..."
    first_clock = _clock_from_name(vids[0]["name"])
    print(f"  {'OK ' if first_clock else 'WARN'} start clock"
          f"  {first_clock.strftime('%Y-%m-%d %H:%M:%S') + ' ' + DRIVE_TZ if first_clock else 'not parseable from filename'}")

    # 8 ── download chunk 1 (needed either way: zone map or sanity check)
    chunk1 = _pull(vids[0])

    # 9 ── is it actually video?
    import cv2
    cap = cv2.VideoCapture(str(chunk1))
    ok, frame0 = cap.read()
    W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    FPS = cap.get(cv2.CAP_PROP_FPS) or 0
    NFR = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if not ok:
        _fail("chunk 1 has no readable frames", "the download is corrupt — delete it and re-run")
    print(f"  OK  video      {W}x{H} @ {FPS:.1f} fps, {NFR} frames "
          f"(~{NFR / max(FPS, 1) / 60:.0f} min)")

    # ── GATE: no zone map yet -> export a frame and stop cleanly ────────────────
    if not zjson:
        # Frame 0 of an NVR export is often the worst one — dark, or a partial
        # keyframe. Write three from across the chunk and let the sharpest win;
        # you only need ONE clear view of the room to draw polygons on.
        cap = cv2.VideoCapture(str(chunk1))
        _cands = []
        for _frac in (0.02, 0.35, 0.70):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(NFR * _frac))
            _ok, _fr = cap.read()
            if not _ok:
                continue
            _sharp = cv2.Laplacian(cv2.cvtColor(_fr, cv2.COLOR_BGR2GRAY),
                                   cv2.CV_64F).var()
            _bright = float(_fr.mean())
            _p = OUTPUT_DIR / f"frame_at_{int(_frac*100):02d}pct.jpg"
            cv2.imwrite(str(_p), _fr)
            _cands.append((_sharp, _bright, _p, _frac))
            print(f"  frame @{_frac*100:4.0f}%  sharpness {_sharp:7.0f}  "
                  f"brightness {_bright:5.1f}  -> {_p.name}")
        cap.release()
        if _cands:
            _cands.sort(reverse=True)
            frame_path = _cands[0][2]
            cv2.imwrite(str(OUTPUT_DIR / "frame_0.jpg"), cv2.imread(str(frame_path)))
            frame_path = OUTPUT_DIR / "frame_0.jpg"
            print(f"  sharpest = {_cands[0][2].name} -> copied to frame_0.jpg")
        else:
            frame_path = OUTPUT_DIR / "frame_0.jpg"
            cv2.imwrite(str(frame_path), frame0)
        print("\n" + "=" * 78)
        print("  STOP — no zone map in the Drive folder yet. This is expected on run 1.")
        print("=" * 78)
        print(f"""
      1. Download   {frame_path}      ({frame_path.stat().st_size / 1024:.0f} KB)
           Kaggle:  right panel -> Output -> poc_output -> frame_0.jpg
           (frame_at_02pct / 35pct / 70pct are also there — use whichever
            shows the room and the doorway most clearly)

      2. Open zone_mapper_v2.html and drop that JPG in. Draw:
             entry_line   the doorway          (2 points, across it)
             wait_zone    walkway where people stand
             reception    the desk             <- the NAME matters
             walkway      path inside

         'reception' is what makes it a staff zone. Rename it and the
         staff metrics come out empty.

      3. Save it as   zones_{CAMERA_ID}.json   and upload to the SAME
         Drive folder.

      4. Run All again. It will pick it up and keep going.

      Frame is {W}x{H}. Mapping on a differently-sized screenshot is fine —
      load_zone_config() rescales polygons to the video automatically.
    """)
        raise RuntimeError("zone map needed — see the steps above (this is not a bug)")

    # ── zones exist: stage them next to the video so Cell 3 pairs them ──────────
    zpath = _pull(zjson[0])
    staged = DRIVE_DIR / f"zones_{chunk1.stem}.json"
    if zpath != staged:
        shutil.copy(zpath, staged)
    cfg = json.loads(staged.read_text())
    zone_names = list(cfg.get("polygons", {}))
    has_entry  = len(cfg.get("entry_line", [])) == 2
    print(f"  OK  zones      {zone_names}")
    print(f"  {'OK ' if has_entry else 'WARN'} entry line  "
          f"{'2 points' if has_entry else 'MISSING — arrivals cannot be counted'}")
    _roles_of = {z: [r for r, kws in ZONE_ROLE_KEYWORDS.items()
                     if any(kw in z.lower() for kw in kws)] for z in zone_names}
    for _z, _rs in _roles_of.items():
        print(f"  zone          {_z:22s} -> {_rs or ['other']}")
    _multi = {z: rs for z, rs in _roles_of.items() if len(rs) > 1}
    if _multi:
        print("  !!  ZONE NAME MATCHES MORE THAN ONE ROLE — usually accidental:")
        for _z, _rs in _multi.items():
            print(f"      '{_z}' -> {_rs}")
            if "seating" in _rs and any(k in _z.lower() for k in ("gate", "door", "entry")):
                print(f"      ^ a GATE classified as SEATING: 'got seated' will count")
                print(f"        everyone who walks through it. Rename to drop the")
                print(f"        word table/seat/dining/booth (e.g. 'inner_gate').")
    if not any("reception" in z.lower() or "staff" in z.lower() or "host" in z.lower()
               for z in zone_names):
        msg = ("no zone named reception/staff/host — every staff metric will be empty")
        _fail(msg, "rename the desk polygon to 'reception' and re-upload") if STRICT else print(f"  WARN {msg}")

    # 10 ── staff gallery (auto-pulls images from Drive folder)
    _gal_remote = [f for f in remote if Path(f["name"]).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    _gal_dir = BASE / STAFF_GALLERY_DIR
    _gal_dir.mkdir(parents=True, exist_ok=True)
    for _gf in _gal_remote:
        _dst = _gal_dir / _gf["name"]
        if not _dst.exists() or _dst.stat().st_size < 1000:
            try:
                print(f"  >>  download   staff photo: {_gf['name']}")
                gdown.download(id=_gf["id"], output=str(_dst), quiet=True)
            except Exception as _ge:
                print(f"  WARN could not download staff photo {_gf['name']}: {_ge}")
    _gal = [p for p in _gal_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    print(f"  {'OK ' if _gal else 'WARN'} staff      "
          f"{len(_gal)} enrolled ({[p.name for p in _gal]})" + ("" if _gal else " — falling back to zone-dominance heuristic"))

    # ── hand over to Cell 3 unchanged ───────────────────────────────────────────
    # (CHUNK_FILTER already applied above, BEFORE the clock parse and download)
    QUEUE_REMOTE = vids if MAX_CHUNKS is None else vids[:MAX_CHUNKS]
    INPUT_ROOT = DRIVE_DIR                      # <- the only hook Cell 3 needs
    VIDEO_START_CLOCK = first_clock.strftime("%H:%M:%S") if first_clock else None

    # Cell 3 discovers by FILESYSTEM: any stale .mp4 left in drive_chunks from
    # a previous run/filter WILL be processed. Purge everything not queued.
    _queued_names = {f["name"] for f in QUEUE_REMOTE}
    for _old in DRIVE_DIR.iterdir():
        _stale_vid = (_old.suffix.lower() in VIDEO_EXTS_PRE
                      and _old.name not in _queued_names)
        # staged zone maps from a previous chunk confuse Cell 3's pairing rules
        _stale_zone = (_old.name.startswith("zones_") and _old.suffix == ".json"
                       and _old.stem[len("zones_"):] + chunk1.suffix not in _queued_names)
        if _stale_vid or _stale_zone:
            _old.unlink()
            print(f"  ..  purged stale file not in queue: {_old.name}")

    # HARD GATE: the announced clock must agree with the queued file's name.
    _q0 = _clock_from_name(QUEUE_REMOTE[0]["name"])
    if VIDEO_START_CLOCK and (not _q0 or _q0.strftime("%H:%M:%S") != VIDEO_START_CLOCK):
        _fail(f"clock mismatch: queued file {QUEUE_REMOTE[0]['name']!r} says "
              f"{_q0.strftime('%H:%M:%S') if _q0 else '??'} but announced clock "
              f"is {VIDEO_START_CLOCK}", "the queue and the clock diverged — "
              "delete drive_chunks/ and re-run")
    if STRICT and not first_clock:
        _fail("start clock not parseable from the queued filename",
              "wall-clock labels would silently fall back to video time — "
              "rename the file to the NVR pattern or set STRICT=False")

    print("=" * 78)
    print(f"  PREFLIGHT PASSED — {len(QUEUE_REMOTE)}/{len(vids)} chunk(s) queued"
          f"{' (MAX_CHUNKS=' + str(MAX_CHUNKS) + ')' if MAX_CHUNKS else ''}")
    print(f"  clock starts {VIDEO_START_CLOCK} {DRIVE_TZ} · INPUT_ROOT -> {INPUT_ROOT}")
    print("=" * 78)




    try:
        import insightface
        insightface.app.FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        print("  OK  insightface pre-staged buffalo_l model")
    except Exception as _ie:
        print(f"  WARN insightface pre-stage: {_ie}")

else:
    # ── KAGGLE DATASET MODE ────────────────────────────────────────────────
    # Nothing to download. Cell 3 discovers the video under /kaggle/input, and
    # the pieces Cell 2d normally sets up are derived from the files instead.
    import shutil as _sh
    print("=" * 78)
    print("  USE_DRIVE = False — using the attached Kaggle dataset, not Drive")
    print("=" * 78)
    QUEUE_REMOTE = []
    _vids_local = sorted(p for p in INPUT_ROOT.rglob("*")
                         if p.suffix.lower() in VIDEO_EXTS_PRE) if INPUT_ROOT.exists() else []
    if not _vids_local:
        _fail("no video found under /kaggle/input",
              "attach the dataset (right panel -> Add Input) that holds the .mp4")
    for _v in _vids_local:
        print(f"  OK  video      {_v.name}")
    first_clock = _clock_from_name(_vids_local[0].name)
    VIDEO_START_CLOCK = first_clock.strftime("%H:%M:%S") if first_clock else None
    _sc_msg = (VIDEO_START_CLOCK or "not parseable from the filename — "
               "wall clocks will show video time instead")
    print(f"  {'OK ' if first_clock else 'WARN'} start clock  {_sc_msg}")
    # staff photos: a dataset has them loose next to the video, not in a folder
    _gal_dir = BASE / STAFF_GALLERY_DIR
    _gal_dir.mkdir(parents=True, exist_ok=True)
    _found = [p for p in INPUT_ROOT.rglob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
              and p.stat().st_size > 2000]   # a real headshot, not a thumbnail
    for _p in _found:
        _dst = _gal_dir / _p.name
        if not _dst.exists():
            _sh.copy(_p, _dst)
    print(f"  {'OK ' if _found else 'WARN'} staff      "
          f"{len(_found)} photo(s) {[p.name for p in _found]}"
          + ("" if _found else " — falling back to the zone-dominance heuristic"))
    _zj = [p for p in INPUT_ROOT.rglob("*.json") if "zone" in p.name.lower()]
    print(f"  {'OK ' if _zj else 'WARN'} zones      {[p.name for p in _zj]}")
    if not _zj:
        _fail("no zone map (*zone*.json) in the dataset",
              "draw one with zone_mapper_v2.html and add it to the dataset")
    print("=" * 78)


In [ ]:
# Cell 2e — 10-HOUR SCALE PROFILE  (config + the two silent corruptions)
# Everything in this notebook was tuned on 10-minute clips. A 10-hour
# archive breaks three assumptions that produce no error message at all:
#   1. clock          frame_idx/fps is wrong if the NVR exported VFR
#   2. colour         the camera flips to infrared at dusk, killing Re-ID
#   3. re-id horizon  a 2-hour match window is meaningless over 10 hours
# This cell fixes all three and prints what it changed.

# ── PLAYBACK vs ANALYSIS — two independent knobs ────────────────────────────
# Accuracy lives in ANALYSIS_FPS. Watchability lives in PLAYBACK_FPS.
# Raising PLAYBACK_FPS costs nothing and never touches a number.
ANALYSIS_FPS   = 7.5      # 30/round(30/8) = 7.5 is the TRUE analysed rate on a
                          # 30fps source; writing "8" left a 6.7% clock error.
PLAYBACK_FPS   = ANALYSIS_FPS  # REAL-TIME playback: one analysed frame per
                          # output frame at the analysed rate means the video
                          # duration equals the footage duration and the burned
                          # clock matches what you see. (Set 30 to get the old
                          # 4x fast-forward review artifact instead.)
RENDER_MAX_W   = 1280     # downscale the annotated video only
RENDER_CRF     = 28       # 23 = pretty, 28 = small. debug artifact, keep small.
# RENDER_FULL was dead config — nothing ever read it. The knob that actually
# decides whether empty stretches reach the video is RENDER_ONLY_OCCUPIED
# (Cell 2), now False so the full hour is rendered wall-clock continuous.

FPS_TARGET = ANALYSIS_FPS

# ── PROVE-IT MODE ───────────────────────────────────────────────────────────
# A chunk is a FULL HOUR of footage — processing one end to end is 25-40 min,
# not a quick check. For run 2 you only need to answer "does the detector see
# people in this room, and are the zones in the right place?" 10 minutes of
# footage answers that. Set to None for the real run.
PROVE_SECONDS = None        # None = whole chunk
_speedup   = PLAYBACK_FPS / ANALYSIS_FPS

# ── RE-ID HORIZON ───────────────────────────────────────────────────────────
# 7200 s was "the whole video" back when a video was 10 minutes. Over 10 h it
# invites two strangers in black shirts, hours apart, to become one person —
# and it is the dominant cost in merge_fragmented_tracks (measured: 36% of all
# pairs survive a 2 h gate at 10 h scale, vs 5% at 900 s).
REID_MAX_GAP_S        = 900    # 15 min: a guest who steps out and returns
MAX_BODY_GAP_S        = 300    # appearance-only tiers: tighter still
LIVE_REID_MEMORY_TTL_S = 1800.0  # was inf — unbounded memory over 10 h

print("=" * 78)
print("  10-HOUR SCALE PROFILE")
print("=" * 78)
print(f"  analysis      {ANALYSIS_FPS} fps   (accuracy)")
print(f"  playback      {PLAYBACK_FPS} fps   -> {_speedup:.1f}x faster to watch")
print(f"  re-id horizon {REID_MAX_GAP_S}s  (was 7200)")
print(f"  prove mode    {'first ' + str(PROVE_SECONDS) + 's of each chunk' if PROVE_SECONDS else 'FULL chunks'}")
# B4: the first real run analysed 20 of 60 minutes because WINDOW_MIN was left
# at a testing value. Whatever the setting, it is now stated out loud.
_wm = globals().get("WINDOW_MIN")
print(f"  window        {'FULL chunk' if not _wm else str(_wm) + ' min slice starting at ' + str(globals().get('WINDOW_START_MIN', 0)) + ' min'}"
      + ("" if not _wm else "   <-- only part of the footage is being analysed"))

# ── 1. VFR: is frame_idx/fps actually the truth? ────────────────────────────
# NVR exports are often variable-frame-rate. The header claims a constant fps,
# real frame spacing drifts, and every duration in the report is then wrong by
# a growing amount with nothing to indicate it.
import cv2
def check_clock(video_path, probes=(0.10, 0.45, 0.85), tol_pct=1.0):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    worst, rows = 0.0, []
    for p in probes:
        idx = int(n * p)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, _ = cap.read()
        if not ok:
            continue
        t_hdr = idx / fps if fps else 0          # what the notebook assumes
        t_real = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0   # what the container says
        if t_real <= 0:
            continue
        drift = abs(t_real - t_hdr) / max(t_real, 1e-6) * 100
        worst = max(worst, drift)
        rows.append((idx, t_hdr, t_real, drift))
    cap.release()
    return worst, rows, fps

CLOCK_SOURCE = "frame_index"
try:
    _worst, _rows, _fps = check_clock(chunk1)
    for idx, th, tr, d in _rows:
        print(f"  clock probe   frame {idx:>7d}  assumed {th:8.1f}s  actual {tr:8.1f}s  drift {d:5.2f}%")
    if _worst > 1.0:
        CLOCK_SOURCE = "pos_msec"
        print(f"  !!  VFR DETECTED — worst drift {_worst:.2f}%.")
        print("      frame_idx/fps is NOT the clock here. Every duration would be")
        print("      wrong by a growing amount. Use CAP_PROP_POS_MSEC instead.")
    else:
        print(f"  OK  clock      constant frame rate (worst drift {_worst:.2f}%)")
except Exception as e:
    print(f"  WARN clock     could not verify ({e}) — assuming constant fps")

# ── 2. DUSK / INFRARED: the switch that silently kills re-identification ────
# 16:30 -> 02:30 crosses dusk. When the camera flips to IR the image goes
# greyscale. OSNet embeddings shift, and every colour-based signal in this
# notebook — ENABLE_ATTIRE_MERGE_TIER, _appearance_hsv_contradicts, the v53
# HSV veto — is built on colour that no longer exists. The same person
# before and after the switch will not match, and worse, two different
# people may.
import numpy as np
def scan_colour(video_path, n_probes=12):
    """Mean HSV saturation over the video. IR footage sits near zero."""
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 1
    out = []
    for k in range(n_probes):
        idx = int(n * k / n_probes)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok:
            continue
        sat = float(cv2.cvtColor(fr, cv2.COLOR_BGR2HSV)[:, :, 1].mean())
        out.append((idx / fps, sat))
    cap.release()
    return out

IR_SAT_THRESHOLD = 12.0     # mean saturation below this = greyscale/IR
try:
    _prof = scan_colour(chunk1)
    _sats = [s for _, s in _prof]
    _mn, _mx = min(_sats), max(_sats)
    _is_ir = _mx < IR_SAT_THRESHOLD
    _switches = _mx > IR_SAT_THRESHOLD > _mn
    bar = "".join("." if s < IR_SAT_THRESHOLD else "#" for _, s in _prof)
    print(f"  colour        [{bar}]  saturation {_mn:.0f}-{_mx:.0f}   (# = colour, . = IR)")
    if _is_ir:
        print("  🌙 THIS CHUNK IS INFRARED (Black & White). Optimizing night vision parameters:")
        ENABLE_ATTIRE_MERGE_TIER = False
        ENABLE_HANDOFF_HSV_VETO = False
        ENABLE_APPEARANCE_HSV_VETO = False
        ENABLE_CLAHE = True             # Boost greyscale contrast for YOLO detection
        DETECT_CONF_FLOOR = 0.20        # Lower floor to catch low-contrast bodies in night shadows
        print("      -> ENABLE_ATTIRE_MERGE_TIER = False (auto)")
        print("      -> ENABLE_HANDOFF_HSV_VETO & ENABLE_APPEARANCE_HSV_VETO = False (auto)")
        print("      -> ENABLE_CLAHE = True (auto contrast boost for IR silhouettes)")
        print("      -> DETECT_CONF_FLOOR = 0.20 (auto low-light recall boost)")
    elif _switches:
        print("  !!  IR SWITCH INSIDE THIS CHUNK. Identity must not be merged")
        print("      across the boundary — treat before/after as separate")
        print("      identity spaces or you will fuse two different people.")
    else:
        print("  OK  colour     full colour throughout")
except Exception as e:
    print(f"  WARN colour    could not profile ({e})")

# ── 3. what this costs / saves ──────────────────────────────────────────────
try:
    _hours = NFR / max(FPS, 1) / 3600
    _frames_analysed = int(_hours * 3600 * ANALYSIS_FPS)
    _out_min = _frames_analysed / PLAYBACK_FPS / 60
    print("-" * 78)
    print(f"  per chunk     {_hours:.2f} h  ->  {_frames_analysed:,} frames analysed")
    print(f"  annotated out {_out_min:.1f} min at {PLAYBACK_FPS} fps"
          + ("  (real time)" if PLAYBACK_FPS == ANALYSIS_FPS else
             f"  ({PLAYBACK_FPS/ANALYSIS_FPS:.1f}x fast-forward)"))
except Exception:
    pass
print("=" * 78)

# ── V75: the audit above ran in Cell 2, BEFORE the overrides in this cell.
# Re-print it now so the values on screen are the values that will run.
# Three parameters genuinely differ (LIVE_REID_MEMORY_TTL_S, REID_MAX_GAP_S,
# MAX_BODY_GAP_S) and the earlier printing understates none of them harmlessly:
# a 7200s re-id horizon vs the real 900s is the difference between "two
# strangers hours apart may fuse" and "they cannot".
try:
    print()
    print("#" * 80)
    print("  CONFIG AUDIT (RESTATED after Cell 2e) — THESE are the values that run")
    print("  Any value differing from the Cell 2 printing above was overridden here.")
    print("#" * 80)
    audit_config()
except NameError:
    print("  (audit_config not defined — run Cell 2 first)")


In [ ]:
# Cell 2f — KEVACV BOOTSTRAP  [KEVACV_BOOTSTRAP]
# ONE mechanism, replacing four separately-embedded module copies.
#
# Generated from the kevacv/ package by patch_v56_phase6.py. Edit the .py files
# and re-run the patch; never edit the copies below. Inside the notebook this
# materialises a real importable package, so the same code that runs here is the
# code the tests on a laptop exercise — there is no notebook-only variant.
import sys as _sys
from pathlib import Path as _Path

_KEVACV_SRC = {
    '__init__.py': r'''"""kevacv — the parts of the video pipeline that are not the notebook.

WHY THIS PACKAGE EXISTS
    The notebook is 34 cells and ~6000 lines, patched by matching literal
    strings. Anything that lives only inside it cannot be imported, cannot be
    tested outside Kaggle, and gets a second copy every time it is embedded as
    a cell. Everything in here is ordinary Python: importable, testable on a
    laptop, and with exactly one copy on disk.

CELL 5 AND CELL 7 ARE NOW HERE
    They used to be notebook-only, held back until a HOTA number existed so a
    later regression would be attributable. They were extracted before that
    number arrived, so the attribution problem was solved a different way:

        analytics.py  <- Cell 5. tests/test_analytics_extraction.py execs the
                         CELL and demands identical output from the module —
                         intervals, thresholds, the full merge (mapping, edges,
                         tier counts, blocked count) and both calibration
                         distributions.
        engine.py     <- Cell 7. tests/test_engine_extraction.py proves all 39
                         top-level definitions are CHARACTER-IDENTICAL to the
                         cell, and runs the pure helpers both ways.

    Those tests are the missing baseline. Both cells still exist in the
    notebook and are still the thing that runs; delete them, and delete the
    matching test, only once a scored run says the module path agrees.

    engine.py is NOT imported here: it needs torch/ultralytics/boxmot, which a
    laptop may not have. `import kevacv.engine` explicitly.

THE PUBLIC SURFACE
    log        stage / banner       one nested timeline for the whole run
    geometry   GroundPlane          pixels -> metres on the floor
    health     CameraHealth         is this still the same camera view?
    filters    implausible_size_mask / static_track_ids   phantom removal
    config     load_profile etc     per-camera / per-venue settings as DATA
    metrics    score_sequence etc   HOTA / DetA / AssA, A/B comparison
    triage     plan_segments        analyse where the people are, account for the rest
    arrivals   arrivals_from_regions  an arrival count that survives a badly drawn line
    phantoms   phantom_regions        static false positives whose track ids churn
    learn_zones learn_entry_zones     find the door in the data, don't draw it

    python -m kevacv --help          the same tools from a shell
"""
from .log import banner, get_logger, setup as setup_logging, stage
from .answers import Answer, answer_set, to_report_rows
from .derive import enrich, id_confidence, observed_windows, staff_contacts
from .drive import fetch_chunk, select as select_chunk
from .pipeline import bind_runtime, preflight, resolve_identities, run_camera
from .helpers import classify_zones, load_zone_config, mmss, wall
from .validity import (DetectorCanary, ValidityLedger, frame_validity)
from .clock import (check_dst_span, check_frame_clock, localize,
                    parse_start, verify_provenance)
from .resilience import Checkpoint, run_batched
from . import seams
from .arrivals import arrivals_from_regions, cross_check, entry_zone_coverage
from .report_slim import (coverage_strip, people_csv, summary_txt,
                          write_slim_outputs)
from .topology import (doors_from_endpoints, doors_from_zones,
                       reappearance_verdict, veto_pairs)
from .threshold import cost_weighted_threshold, verdict as threshold_verdict
from .merge_ab import ab_topology, greedy_union
from .reid_calibration import calibrate, compare_to_legacy
# ── extracted from the notebook (Cell 5). Behaviour is pinned against the
# cell itself by tests/test_analytics_extraction.py, so the refactor cannot
# quietly move a number. ────────────────────────────────────────────────────
from .config import DEFAULT as TRACKING_DEFAULTS
from .config import TrackingConfig
# engine.py imports torch/ultralytics/boxmot, which a laptop may not have.
# Import it lazily via kevacv.engine so the rest of the package stays usable.
from .analytics import (OccupancyRecorder, calibrate_appearance_threshold,
                        clip_to, complement_intervals, covered_windows,
                        entered_count, merge_fragmented_tracks,
                        merge_intervals, minute_summaries, occupancy_timeline,
                        reception_absence, remap_events, seated_count,
                        total_duration, waited_over)
from .tiled import cost_estimate, height_roi, slice_grid, tiled_predict
# ── merged in from the computer_vision line (two sessions worked in parallel;
# that fork carried the older notebook but the more advanced package) ────────
from .preflight import PreflightValidationError, run_preflight_checks
from .graph_fusion import FusionWeights, solve_graph_fusion
from .reid_engine import ReIDEmbeddingExtractor
from .geometry_calibration import fit_robust_ground_plane
from .tracker_wrapper import TrackerWrapper
from .anomaly_baseline import ZoneAnomalyDetector
from .dataset_collector import DatasetCollector
from .learn_zones import (learn_dwell_zones, learn_entry_zones,
                         to_zone_config, track_endpoints)
from .phantoms import (drop_phantom_dets, in_phantom, phantom_regions)
from .camera_health import CameraHealth, verdict_line
from .detect_filters import (BODY_ASPECT, drop_tracks, implausible_size_mask,
                             mirrored_pair_ids, protected_ids,
                             rigid_track_ids, static_track_ids)
from .eval_harness import (compare, dump_errors_csv, explain, iou_matrix,
                           load_mot, save_baseline, score_conditions,
                           score_sequence, write_mot)
from .ground_plane import PERSON_H_M, GroundPlane, synth_camera
from .triage import coverage_report, miss_risk, plan_segments
from .venue_profile import (DEFAULTS, describe, infer_entry_direction,
                            load_profile, local_clock, validate, write_template)

__version__ = "0.6.0"          # tracks the pipeline phase, not semver

__all__ = [
    "setup_logging", "get_logger", "stage", "banner",
    "run_camera", "preflight", "resolve_identities", "bind_runtime",
    "Answer", "answer_set", "to_report_rows",
    "enrich", "observed_windows", "staff_contacts", "id_confidence",
    "fetch_chunk", "select_chunk",
    "load_zone_config", "classify_zones", "mmss", "wall",
    "ValidityLedger", "DetectorCanary", "frame_validity",
    "parse_start", "localize", "check_dst_span", "check_frame_clock",
    "verify_provenance", "Checkpoint", "run_batched", "seams",
    "mirrored_pair_ids",
    "GroundPlane", "PERSON_H_M", "synth_camera",
    "CameraHealth", "verdict_line",
    "implausible_size_mask", "static_track_ids", "rigid_track_ids",
    "protected_ids", "drop_tracks",
    "BODY_ASPECT",
    "plan_segments", "coverage_report", "miss_risk",
    "arrivals_from_regions", "cross_check", "entry_zone_coverage",
    "summary_txt", "people_csv", "coverage_strip", "write_slim_outputs",
    "reappearance_verdict", "veto_pairs", "doors_from_zones",
    "doors_from_endpoints", "cost_weighted_threshold", "threshold_verdict",
    "ab_topology", "greedy_union", "calibrate", "compare_to_legacy",
    "tiled_predict", "slice_grid", "height_roi", "cost_estimate",
    "phantom_regions", "in_phantom", "drop_phantom_dets",
    "learn_entry_zones", "learn_dwell_zones", "to_zone_config",
    "track_endpoints",
    "load_profile", "validate", "describe", "local_clock",
    "infer_entry_direction", "write_template", "DEFAULTS",
    "score_sequence", "explain", "compare", "score_conditions",
    "load_mot", "write_mot", "dump_errors_csv", "save_baseline", "iou_matrix",
    "PreflightValidationError", "run_preflight_checks",
    "FusionWeights", "solve_graph_fusion",
    "ReIDEmbeddingExtractor", "fit_robust_ground_plane",
    "TrackerWrapper", "ZoneAnomalyDetector", "DatasetCollector",
    # analytics, extracted from Cell 5
    "TrackingConfig", "TRACKING_DEFAULTS",
    "merge_fragmented_tracks", "calibrate_appearance_threshold",
    "OccupancyRecorder", "occupancy_timeline", "remap_events",
    "entered_count", "seated_count", "waited_over", "reception_absence",
    "minute_summaries", "merge_intervals", "total_duration",
    "complement_intervals", "clip_to", "covered_windows",
]
''',
    '__main__.py': r'''"""kevacv CLI — the tools, without opening Kaggle.

    python -m kevacv score  <gt.txt> <predictions.txt>
    python -m kevacv ab     <before_score.json> <after_score.json>
    python -m kevacv profile <out.json> [camera-id]
    python -m kevacv view   <reference.jpg> <current.jpg>

`score` is the one that matters: it turns a labelled slice and a prediction
file into HOTA / DetA / AssA on any machine, so a change can be judged without
a GPU session.
"""
import json
import sys
from pathlib import Path


def _score(argv):
    from .eval_harness import explain, load_mot, score_sequence
    if len(argv) < 2:
        return _usage("score needs <gt.txt> <predictions.txt>")
    gt, pr = load_mot(argv[0]), load_mot(argv[1])
    if not gt:
        print(f"ground truth {argv[0]} is empty or unparseable")
        return 1
    res = explain(score_sequence(gt, pr),
                  label=f"{Path(argv[0]).name} vs {Path(argv[1]).name}")
    if len(argv) > 2:
        from .eval_harness import save_baseline
        save_baseline(res, argv[2], label=Path(argv[1]).stem)
    return 0


def _ab(argv):
    from .eval_harness import compare
    if len(argv) < 2:
        return _usage("ab needs <before.json> <after.json>")
    a = json.loads(Path(argv[0]).read_text())
    b = json.loads(Path(argv[1]).read_text())
    compare(a, b, Path(argv[0]).stem, Path(argv[1]).stem)
    return 0


def _profile(argv):
    from .venue_profile import write_template
    if not argv:
        return _usage("profile needs an output path")
    p = write_template(argv[0], argv[1] if len(argv) > 1 else "")
    print(f"wrote {p} — edit it and drop it beside the video as "
          f"profile_<video-stem>.json")
    return 0


def _view(argv):
    import cv2
    from .camera_health import CameraHealth, verdict_line
    if len(argv) < 2:
        return _usage("view needs <reference-image> <current-image>")
    ref, cur = cv2.imread(argv[0]), cv2.imread(argv[1])
    if ref is None or cur is None:
        print("could not read one of the images")
        return 1
    res = CameraHealth.from_frame(ref).check(cur)
    print(verdict_line(res))
    return 0 if res["valid"] else 2


CMDS = {"score": _score, "ab": _ab, "profile": _profile, "view": _view}


def _usage(msg=""):
    if msg:
        print(f"error: {msg}\n")
    print(__doc__)
    return 1


def main(argv=None):
    argv = list(sys.argv[1:] if argv is None else argv)
    if not argv or argv[0] in ("-h", "--help"):
        return _usage()
    fn = CMDS.get(argv[0])
    return fn(argv[1:]) if fn else _usage(f"unknown command {argv[0]!r}")


if __name__ == "__main__":
    sys.exit(main())
''',
    'anomaly_baseline.py': r'''"""anomaly_baseline.py — Phase C P10 Learned Anomaly Baselines (Avigilon UMD-Style Semantic Anomaly Engine).

WHY THIS EXISTS
    Static alert thresholds require manual hand-tuning per venue and fail to adapt to day-of-week
    or hour-of-day traffic patterns. 

THE PRINCIPLE
    1. Per-Zone Seasonal Baseline Model: Learns typical occupancy distribution (mean & std)
       per (zone_id, hour_of_day, is_weekend).
    2. Residual Z-Score Scoring: Scores actual occupancy count against learned baseline:
       z = (actual - mean) / std.
    3. Flags statistically significant anomalies (|z| > threshold, default 2.5) with semantic context:
       - "Host stand unstaffed during peak ramp"
       - "Queue depth surge"
       - "Unusual lounge area crowd"
"""
from __future__ import annotations

import math
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple


class ZoneAnomalyDetector:
    """Learned anomaly baseline engine for per-zone occupancy time-series."""

    def __init__(self, z_threshold: float = 2.5, min_history_samples: int = 5):
        self.z_threshold = z_threshold
        self.min_history_samples = min_history_samples
        # Store history: (zone_id, hour, is_weekend) -> list of occupancy counts
        self.history: Dict[Tuple[str, int, bool], List[float]] = defaultdict(list)

    def fit_history(self, records: List[Dict[str, Any]]) -> None:
        """Fit history from records: [{zone_id, hour, is_weekend, count}, ...]"""
        for r in records:
            key = (r["zone_id"], int(r["hour"]), bool(r.get("is_weekend", False)))
            self.history[key].append(float(r["count"]))

    def score_occupancy(self,
                        zone_id: str,
                        hour: int,
                        count: float,
                        is_weekend: bool = False) -> Dict[str, Any]:
        """Score current zone occupancy count against baseline model.
        Returns dict with z_score, is_anomaly, baseline_mean, baseline_std, and explanation.
        """
        key = (zone_id, hour, is_weekend)
        samples = self.history.get(key, [])

        if len(samples) < self.min_history_samples:
            # Update history and return unflagged baseline learning state
            self.history[key].append(float(count))
            return {
                "zone_id": zone_id,
                "hour": hour,
                "count": count,
                "is_anomaly": False,
                "z_score": 0.0,
                "baseline_mean": count,
                "baseline_std": 0.0,
                "note": f"Learning baseline ({len(samples)}/{self.min_history_samples} samples)"
            }

        mean = sum(samples) / len(samples)
        variance = sum((x - mean) ** 2 for x in samples) / len(samples)
        if variance < 1e-6:
            # All historical values are near-identical — z-score is meaningless
            std = 1.0
            is_constant_baseline = True
        else:
            std = math.sqrt(variance)
            is_constant_baseline = False

        z_score = (count - mean) / std
        is_anomaly = abs(z_score) >= self.z_threshold

        # Update running history (limit max memory 500 samples per slot)
        self.history[key].append(float(count))
        if len(self.history[key]) > 500:
            self.history[key].pop(0)

        explanation = ""
        if is_anomaly:
            direction = "SURGE" if z_score > 0 else "DROP/UNSTAFFED"
            explanation = (f"Zone '{zone_id}' occupancy {count:.0f} deviates ({direction}) "
                           f"from normal baseline {mean:.1f} ± {std:.1f} (Z={z_score:+.2f})")
        elif is_constant_baseline and abs(count - mean) > 0.5:
            explanation = (f"Zone '{zone_id}' baseline has zero variance (all prior samples ~{mean:.0f}). "
                           f"Current count {count:.0f} differs but z-score uses artificial std=1.")

        return {
            "zone_id": zone_id,
            "hour": hour,
            "count": count,
            "is_anomaly": is_anomaly,
            "z_score": round(z_score, 2),
            "baseline_mean": round(mean, 1),
            "baseline_std": round(std, 1),
            "constant_baseline": is_constant_baseline,
            "explanation": explanation
        }
''',
    'answers.py': r'''"""answers.py — the questions the project exists to answer.

NOT A PORT OF CELL 18
    Cell 18 computes the same quantities, and its versions shipped wrong twice:
    guests_tonight came from a region fallback and was labelled EXACT*, and
    desk_covered_pct appeared as 56.4 in one place and 68.9 in another in the
    same report. Reproducing that faithfully would reproduce the faults.

THE PRINCIPLE
    An answer is a VALUE over a DENOMINATOR, and both carry their own validity.

    Three consequences, and every design decision here follows from them:

    1. THE DENOMINATOR IS OBSERVED TIME, NEVER ELAPSED TIME.
       "The desk was covered 68.9% of the night" is meaningless unless you say
       68.9% of WHAT. If the camera was blind for twenty minutes, those minutes
       are not "uncovered" — they are unmeasured, and dividing by them turns a
       dead camera into a quiet venue.

    2. AN ANSWER THAT NEEDS IDENTITY MUST CARRY A RANGE.
       Measured re-id separability on this footage is ~0.66. Any count that
       requires one person to stay one person for minutes is uncertain by
       construction, and a single integer hides that. Desk coverage does NOT
       need identity — it only asks "is anyone in this polygon" — which is why
       it is the one metric that can honestly reach 90%.

    3. EVERY ANSWER STATES WHAT WOULD MAKE IT WRONG.
       Not a confidence score: a sentence. `caveats` is the difference between
       a number a manager can act on and a number they should not.

    An answer whose inputs cannot support it returns tier=UNKNOWN with a
    reason. It never returns 0. "None arrived" and "we could not tell" are
    different facts, and the pipeline has already published one as the other.

TIERS
    EXACT     measured directly; needs no identity to hold over time
    PROXY     a stand-in for the real thing (proximity is not conversation)
    ESTIMATE  needed identity to hold; carries a range
    WEAK      may over-count; never act on alone
    UNKNOWN   the inputs could not support the question
"""
from __future__ import annotations

from dataclasses import dataclass, field

from .log import get_logger

_log = get_logger("answers")

EXACT, PROXY, ESTIMATE, WEAK, UNKNOWN = (
    "EXACT", "PROXY", "ESTIMATE", "WEAK", "UNKNOWN")


@dataclass
class Answer:
    """One question, one answer, and everything needed to judge it."""
    key: str
    label: str
    value: object = None
    tier: str = UNKNOWN
    unit: str = ""
    denominator: str = ""          # what the value is a fraction OF
    low: object = None             # ESTIMATE range
    high: object = None
    needs_identity: bool = False
    caveats: list = field(default_factory=list)
    detail: dict = field(default_factory=dict)

    @property
    def display(self):
        if self.value is None:
            return "n/a"
        if self.tier == ESTIMATE and self.low is not None:
            return f"{self.value}{self.unit}   range {self.low}-{self.high}"
        return f"{self.value}{self.unit}"

    def unknown(self, why):
        self.tier, self.value = UNKNOWN, None
        self.caveats.append(why)
        return self


def _merge(intervals, gap=0.0):
    out = []
    for s, e in sorted(intervals):
        if out and s - out[-1][1] <= gap:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(a, b) for a, b in out]


def _clip(intervals, windows):
    """Intersection — a metric may only count time we actually observed."""
    if not windows:
        return list(intervals)
    out = []
    for a, b in intervals:
        for w0, w1 in windows:
            lo, hi = max(a, w0), min(b, w1)
            if hi > lo:
                out.append((lo, hi))
    return _merge(out)


def _total(intervals):
    return sum(b - a for a, b in intervals)


def _presence(events, zones, roles=None, want_role=None):
    """Intervals during which ANYONE (optionally of a role) was in `zones`.

    Deliberately identity-free: it asks "was a body inside this polygon",
    which is the question, and never "which body", which is the hard part.
    """
    zs = set(zones or ())
    out = []
    for e in events or []:
        if e.get("zone") not in zs:
            continue
        if want_role and (roles or {}).get(e.get("track_id"),
                                           e.get("role")) != want_role:
            continue
        out.append((float(e["t_in"]), float(e.get("t_out", e["t_in"]))))
    return _merge(out)


# ── Q1 ─────────────────────────────────────────────────────────────────────
def desk_coverage(events, staff_zones, observed_windows, roles=None,
                  target=0.90):
    """Was the desk covered? EXACT — this is the metric that needs no identity.

    SUCCESS_CRITERIA calls this CRITICAL and >=90%, and says why it is
    achievable: "Zone dwell is the strongest signal we have — identity doesn't
    even need to be perfect." Only presence is required.
    """
    a = Answer("desk_covered_pct", "Was the desk covered?", unit="%",
               denominator="observed footage", tier=EXACT)
    obs = _total(observed_windows or [])
    if obs <= 0:
        return a.unknown("no observed footage — the denominator would be zero")
    covered = _clip(_presence(events, staff_zones, roles, "staff"),
                    observed_windows)
    pct = _total(covered) / obs * 100.0
    a.value = round(pct, 1)
    a.detail = {"covered_s": round(_total(covered), 1), "observed_s": round(obs, 1),
                "target_pct": target * 100, "meets_target": pct >= target * 100}
    a.caveats.append("counts the STATION, not any one person — a different "
                     "staff member at the desk is still covered")
    if not a.detail["meets_target"]:
        a.caveats.append(f"below the ratified {target*100:.0f}% bar")
    return a


def desk_gaps(events, staff_zones, observed_windows, roles=None,
              min_gap_s=60.0, waiting_zones=()):
    """When was it empty, for how long, and was anyone waiting?

    A percentage tells a manager how much; this tells them WHEN, which is the
    part they can act on.
    """
    a = Answer("desk_gaps", "When was the desk empty?", tier=EXACT,
               denominator="observed footage")
    if not observed_windows:
        return a.unknown("no observed footage")
    covered = _clip(_presence(events, staff_zones, roles, "staff"),
                    observed_windows)
    gaps = []
    for w0, w1 in observed_windows:
        cur = w0
        for c0, c1 in covered:
            if c1 <= w0 or c0 >= w1:
                continue
            if c0 - cur >= min_gap_s:
                gaps.append((cur, c0))
            cur = max(cur, c1)
        if w1 - cur >= min_gap_s:
            gaps.append((cur, w1))
    waiting = _presence(events, waiting_zones) if waiting_zones else []
    rows = []
    for g0, g1 in gaps:
        n_waiting = sum(1 for w0, w1 in waiting if w1 > g0 and w0 < g1)
        rows.append({"from_s": round(g0, 1), "to_s": round(g1, 1),
                     "minutes": round((g1 - g0) / 60.0, 1),
                     "guests_waiting": n_waiting})
    rows.sort(key=lambda r: -r["minutes"])
    a.value = len(rows)
    a.detail = {"gaps": rows,
                "longest_min": rows[0]["minutes"] if rows else 0.0}
    if any(r["guests_waiting"] for r in rows):
        a.caveats.append("at least one gap had guests waiting through it")
    return a


# ── Q2 ─────────────────────────────────────────────────────────────────────
def greet_latency(arrivals, contacts, tier=PROXY):
    """How long from arrival to a staff member being near them?

    PROXY, permanently, and SUCCESS_CRITERIA already says so: proximity is not
    conversation. The honest upgrade is a VLM on a 10-second clip, not a better
    tracker. Labelling this EXACT would be the same failure as guests_tonight.
    """
    a = Answer("greet_latency_s", "How fast were guests greeted?", unit=" s",
               tier=tier, denominator="guests with a known arrival time")
    if not arrivals:
        return a.unknown("no arrival times — greet latency needs a door event, "
                         "and a broken entry line produces none")
    lat = []
    ungreeted = []
    for tid, t_in in arrivals.items():
        cs = [c for c in (contacts or {}).get(tid, []) if c >= t_in]
        if cs:
            lat.append(min(cs) - t_in)
        else:
            ungreeted.append(tid)
    if not lat:
        a.value = None
        a.tier = UNKNOWN
        a.caveats.append("nobody was ever near a staff member — check the "
                         "staff zone before reading this as bad service")
    else:
        lat.sort()
        a.value = round(lat[len(lat) // 2], 1)
        a.detail = {"median_s": a.value, "slowest_s": round(lat[-1], 1),
                    "n_greeted": len(lat), "n_ungreeted": len(ungreeted),
                    "ungreeted_ids": ungreeted[:20]}
    a.caveats.append("proximity, not conversation — a service-touch signal, "
                     "never proof someone was spoken to")
    return a


# ── Q3 ─────────────────────────────────────────────────────────────────────
def guest_count(unique_ids, confidence=None, low_conf_bar=60, source="line"):
    """How many guests? ESTIMATE with a range, because identity had to hold.

    The range is not decoration. Measured separability is ~0.66, so a single
    integer asserts a precision the evidence cannot support. Splitting on the
    per-person confidence the pipeline already computes turns that into an
    honest interval instead of a hidden error bar.
    """
    a = Answer("guests", "How many guests?", tier=ESTIMATE,
               needs_identity=True, denominator="distinct identities")
    ids = list(unique_ids or [])
    if not ids:
        return a.unknown(f"no arrivals were detected by the {source} method — "
                         f"that is a broken sensor OR an empty venue, and this "
                         f"cannot tell you which")
    if confidence:
        high = [i for i in ids if confidence.get(i, 0) >= low_conf_bar]
        a.value = len(ids)
        a.low, a.high = len(high), len(ids)
        a.detail = {"high_confidence": len(high),
                    "low_confidence": len(ids) - len(high)}
    else:
        a.value = len(ids)
        a.low, a.high = len(ids), len(ids)
        a.caveats.append("no per-person confidence available, so the range is "
                         "a point — treat the width as unknown, not as zero")
    a.caveats.append(f"derived from the {source} method; identity must hold "
                     f"across the whole visit for this to be right")
    return a


def answer_set(*, events=None, staff_zones=(), waiting_zones=(),
               observed_windows=None, roles=None, arrivals=None, contacts=None,
               unique_ids=None, confidence=None, arrival_source="line",
               findings=()):
    """The full set, in the ratified priority order, ready for report_slim.

    Q1 first because it is CRITICAL and achievable; Q3 last because it is the
    one the evidence supports least. Ordering the report by confidence rather
    than by drama is itself a design decision.
    """
    obs = observed_windows or []
    out = [desk_coverage(events, staff_zones, obs, roles),
           desk_gaps(events, staff_zones, obs, roles, waiting_zones=waiting_zones),
           greet_latency(arrivals, contacts),
           guest_count(unique_ids, confidence, source=arrival_source)]
    # A run-level finding invalidates individual answers; say so on the answer
    # itself rather than only in a banner nobody re-reads.
    blockers = [m for lvl, m in findings if lvl == "ERROR"]
    for a in out:
        if blockers and a.needs_identity:
            a.caveats.extend(blockers)
    for a in out:
        if a.tier == UNKNOWN:
            _log.warning(f"{a.key}: {a.caveats[-1] if a.caveats else 'unknown'}")
    return out


def to_report_rows(answers):
    """-> the shape report_slim.summary_txt() expects."""
    rows = []
    for a in answers:
        extra = []
        if a.key == "desk_covered_pct" and a.detail:
            extra.append(("target", f"{a.detail.get('target_pct', 90):.0f}%",
                          "EXACT" if a.detail.get("meets_target") else "EXACT"))
        if a.key == "desk_gaps" and a.detail.get("gaps"):
            g = a.detail["gaps"][0]
            extra.append(("longest gap", f"{g['minutes']} min", "EXACT"))
            extra.append(("guests waiting in it", g["guests_waiting"], "EXACT"))
        if a.key == "greet_latency_s" and a.detail:
            extra.append(("never greeted", a.detail.get("n_ungreeted", 0), PROXY))
        if a.key == "guests" and a.detail:
            extra.append(("high confidence", a.detail.get("high_confidence"), WEAK))
        rows.append({"label": a.label, "value": a.display, "tier": a.tier,
                     "extra": extra})
    return rows
''',
    'arrivals.py': r'''"""arrivals.py — PHASE 9. Count arrivals without depending on the line.

WHY THIS EXISTS
    On the first full hour of CAM.112 the entry line triggered ZERO times while
    95 people moved through the zones and reception was visited 74 times. Every
    arrival-based number in the report was 0, and "0 people entered" reads
    exactly like a fact.

    The line was too short — people walked around both ends. That is a drawing
    mistake, and it will happen again on the next camera, and the one after.

THE PRINCIPLE
    A single sensor with no cross-check is a single point of silent failure.
    Industry practice treats a tripwire and a region as COMPLEMENTARY — the
    tripwire answers "who crossed where", the region answers "who was here" —
    so we compute BOTH and make them check each other.

    A region-based arrival needs no line at all: someone who appears in an
    entry zone and then appears somewhere interior has arrived. It is coarser
    than a line (it cannot tell you the exact instant, and it cannot separate
    in from out), but it is nearly impossible to break by drawing badly, which
    is exactly the failure a line has.

    When the two disagree by a lot, that is itself the finding.
"""
from __future__ import annotations

import math

ENTRY_ROLES = {"entry"}
INTERIOR_ROLES = {"wait", "staff", "seating", "service"}


def _zones_with(zone_roles, roles):
    return {z for z, rs in (zone_roles or {}).items() if set(rs) & roles}


def arrivals_from_regions(events, zone_roles, roles=None, dedupe_s=6.0,
                          dedupe_m=1.2, plane=None, positions=None):
    """Arrivals inferred from zone transitions instead of a line crossing.

    An arrival is a track that is seen in an ENTRY zone and then, later, in an
    INTERIOR zone. Walking in is exactly that transition, and it does not care
    where a line was drawn.

    -> (count, arrivals, why). `why` is never None: when the zones cannot
    support the question at all it says so rather than returning 0, because 0
    and "cannot tell" must never look the same in a report.
    """
    ez = _zones_with(zone_roles, ENTRY_ROLES)
    iz = _zones_with(zone_roles, INTERIOR_ROLES)
    if not ez:
        return None, [], ("no zone has the ENTRY role — region arrivals cannot "
                          "be computed. Name a zone entrance/door/gate/entry.")
    if not iz:
        return None, [], ("no INTERIOR zone (wait/staff/seating/service) — "
                          "there is nowhere to arrive INTO.")

    per = {}
    for e in events:
        per.setdefault(e["track_id"], []).append(e)

    out = []
    for tid, evs in per.items():
        if roles and roles.get(tid) == "staff":
            continue
        evs = sorted(evs, key=lambda e: e["t_in"])
        first_entry = next((e for e in evs if e["zone"] in ez), None)
        if first_entry is None:
            continue
        moved_in = next((e for e in evs if e["zone"] in iz
                         and e["t_in"] >= first_entry["t_in"]), None)
        if moved_in is None:
            continue                      # stood in the doorway and left again
        out.append({"track_id": tid, "t": moved_in["t_in"],
                    "from_zone": first_entry["zone"], "to_zone": moved_in["zone"],
                    "pos": (positions or {}).get(tid)})

    out.sort(key=lambda a: a["t"])
    kept = []
    for a in out:
        dup = False
        for k in reversed(kept):
            if a["t"] - k["t"] > dedupe_s:
                break
            p, q = a.get("pos"), k.get("pos")
            if p and q:
                d = plane.dist_m(p, q) if (plane is not None and plane.ok) else None
                if (d is not None and d <= dedupe_m) or (
                        d is None and math.hypot(p[0] - q[0], p[1] - q[1]) <= 140):
                    dup = True
                    break
            elif k["track_id"] == a["track_id"]:
                dup = True
                break
        if not dup:
            kept.append(a)
    return len(kept), kept, ""


def entry_zone_coverage(events, zone_roles, roles=None):
    """What share of non-staff people were EVER seen in an entry zone?

    WHY THIS EXISTS
        A region arrival needs an entry-zone sighting before an interior one.
        If hardly anyone produces that first sighting, the entry polygon is not
        over the door — and the region count is then just as broken as a badly
        drawn line, only quietly, because it still returns a small positive
        number instead of 0.

        On CAM.112 the line fired 0 times, the region method returned 2, and 31
        people moved through the zones. cross_check called that "LINE IS BROKEN,
        trust the region" and the report published "2 people came through the
        door". Two independent sensors were both wrong and nothing said so.

    -> None when no zone carries the ENTRY role (the question cannot be asked).
    """
    ez = _zones_with(zone_roles, ENTRY_ROLES)
    if not ez:
        return None
    per = {}
    for e in events:
        per.setdefault(e["track_id"], []).append(e)
    seen = never = 0
    for tid, evs in per.items():
        if roles and roles.get(tid) == "staff":
            continue
        if any(e["zone"] in ez for e in evs):
            seen += 1
        else:
            never += 1
    total = seen + never
    return {"with_entry": seen, "without_entry": never, "non_staff": total,
            "share_with_entry": (seen / total) if total else 0.0}


def cross_check(line_count, region_count, movers=0, tolerance=0.25,
                coverage=None, min_entry_share=0.5):
    """Compare the two independent arrival counts and say what to believe.

    The point is not to pick a winner. It is that two numbers which should
    agree and do not are evidence of a specific, nameable fault — and that a
    silent 0 is the one outcome nobody should ever be handed.

    `coverage` is entry_zone_coverage(). Passing it adds the check that the
    region count is worth trusting at all: a region method whose entry zone
    most people never enter is not a cross-check, it is a second failure
    wearing the costume of a measurement.
    """
    if region_count is None:
        return {"verdict": "no cross-check available",
                "detail": "zones cannot support a region arrival count",
                "trust": "line", "agree": None}
    # Before believing the region number, ask whether the entry zone is even in
    # the right place. This must run BEFORE "LINE IS BROKEN", because that
    # branch's whole purpose is to hand the region count to the report.
    if (coverage and coverage["non_staff"] >= 5
            and coverage["share_with_entry"] < min_entry_share):
        return {"verdict": "ENTRY ZONE IS MISPLACED TOO — trust neither",
                "detail": (f"only {coverage['with_entry']} of "
                           f"{coverage['non_staff']} non-staff people were ever "
                           f"seen inside an entry zone "
                           f"({coverage['share_with_entry']*100:.0f}%). The "
                           f"region count of {region_count} is not a "
                           f"cross-check — the entry polygon is not over the "
                           f"door people actually use. Redraw the entry zone "
                           f"AND the line before believing either number."),
                "trust": "neither", "agree": False}
    if line_count == 0 and region_count == 0 and movers >= 5:
        return {"verdict": "BOTH ZERO but people were present",
                "detail": (f"{movers} people moved through the zones and neither "
                           f"method saw an arrival. The entry zone is probably not "
                           f"where people actually come in."),
                "trust": "neither", "agree": True}
    if line_count == 0 and region_count > 0:
        return {"verdict": "LINE IS BROKEN",
                "detail": (f"the line counted 0 while {region_count} people were "
                           f"seen entering by zone transition. The line is almost "
                           f"certainly too short or in the wrong place — redraw it "
                           f"wall to wall across the threshold. Use the region "
                           f"count until then."),
                "trust": "region", "agree": False}
    if region_count == 0 and line_count > 0:
        return {"verdict": "entry zone is misplaced",
                "detail": (f"the line counted {line_count} but nobody was seen in "
                           f"an entry zone first. The entry polygon is probably not "
                           f"over the doorway."),
                "trust": "line", "agree": False}
    hi = max(line_count, region_count)
    delta = abs(line_count - region_count) / hi if hi else 0.0
    if delta <= tolerance:
        return {"verdict": "the two methods AGREE",
                "detail": (f"line {line_count} vs region {region_count} "
                           f"({delta*100:.0f}% apart) — two independent signals "
                           f"agreeing is the strongest evidence this number is real."),
                "trust": "line", "agree": True}
    return {"verdict": "the two methods DISAGREE",
            "detail": (f"line {line_count} vs region {region_count} "
                       f"({delta*100:.0f}% apart). Something is clipping one of "
                       f"them — check the line spans the full doorway and the "
                       f"entry polygon covers where people actually walk in."),
            "trust": "region" if region_count > line_count else "line",
            "agree": False}


def describe(line_count, region_count, cc):
    L = ["ARRIVALS — two independent methods"]
    L.append(f"  line crossing   {line_count if line_count is not None else 'n/a'}")
    L.append(f"  zone transition {region_count if region_count is not None else 'n/a'}")
    L.append(f"  -> {cc['verdict']}")
    L.append(f"     {cc['detail']}")
    return "\n".join(L)
''',
    'camera_health.py': r'''"""camera_health.py — PHASE 4 / U1. "Is this still the same camera view?"

THE BUG THIS CLOSES
    Nothing in the pipeline ever checked that the camera is still pointing where
    it was pointing when the zones were drawn. I grepped for it: the 'drift'
    hits are VFR clock drift and stationary-merge pixels, 'stabili' is a
    comment. There is no such check.

    If a camera is knocked, re-aimed, or is a PTZ that returns to a slightly
    different preset, EVERY zone polygon is silently in the wrong place. The run
    completes, the charts render, the report prints confident numbers, and all
    of them are wrong. Over a ten-hour night and across venues this is not a
    hypothetical.

THE RULE (Prabh's call, and the right one)
    Detect it and REFUSE TO SCORE. A missing number is recoverable; a confident
    wrong number gets quoted in a meeting. So this module never silently
    corrects and never silently continues — it returns a verdict, and the
    caller marks every zone-dependent metric INVALID.

WHY THE TOLERANCE IS MEASURED ON THE ZONES, NOT ON THE FRAME
    What matters is not "did the image move" but "did the image move enough to
    put my polygons in the wrong place". A 2-pixel shift is irrelevant. A small
    ROTATION is irrelevant at the image centre and severe at the edges, which is
    exactly where a doorway usually is. So the tolerance is applied to the
    displacement of the ACTUAL ZONE VERTICES under the estimated transform.

FOUR MORE WAYS THE VIDEO IS NOT WHAT YOU THINK IT IS
    Cheap to check, all of them silent failures today:
      out of focus   lens knocked, dirty, or auto-focus hunting
      blinded        direct sun or a light shining into the lens
      lens blocked   something in front of the camera
      frozen stream  the NVR wrote the same frame repeatedly (a real export bug)
"""
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np

try:
    import cv2
except Exception:                                     # pragma: no cover
    cv2 = None

# Defaults are expressed as FRACTIONS of the frame, never as raw pixels, so the
# same profile is correct on 720p and on 4K without being retuned.
#
# HOW THIS NUMBER WAS CHOSEN (it started as a guess at 0.012 and was wrong):
# A tolerance has to sit above the estimator's own noise and below the point
# where a zone-boundary decision can flip. Both ends were MEASURED, not guessed:
#
#   noise floor      0.03 px   worst false shift with the camera genuinely still,
#                              across 14 people walking through, lights dimmed
#                              45%, jpeg q=30 and sensor noise combined. RANSAC
#                              on background features is far steadier than
#                              expected — this end is not the constraint.
#   answer flips    ~10 px     one third of a person's shoulder width at
#                              mid-room depth (a ~108 px tall person is ~31 px
#                              wide at 720p). Beyond this, someone standing at
#                              the edge of the reception polygon can be counted
#                              on the wrong side of it.
#
# 0.8% of the diagonal is 11.7 px at 720p, 21 px at 1080p, 42 px at 4K — about
# 390x the noise floor and just at the answer-flip point. The original 0.012
# was 17.6 px at 720p, i.e. ALREADY past the point where answers change.
DEFAULT_ZONE_TOL_FRAC = 0.008
DEFAULT_MIN_INLIERS = 12
BLUR_VAR_FLOOR = 40.0             # Laplacian variance below this = out of focus
BRIGHT_HI, BRIGHT_LO = 242.0, 12.0
EDGE_DENSITY_FLOOR = 0.004        # fraction of pixels that are edges
FROZEN_DIFF_FLOOR = 0.2           # mean abs diff below this = identical frame


def _prep(frame, max_w=960):
    """Grayscale, downscaled, contrast-normalised.

    Equalising matters more than it looks: the same room in daylight and under
    infrared produces very different raw intensities, and without normalisation
    the feature matcher would report 'the camera moved' every dusk.
    """
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    if g.shape[1] > max_w:
        s = max_w / g.shape[1]
        g = cv2.resize(g, (int(g.shape[1] * s), int(g.shape[0] * s)))
    return cv2.equalizeHist(g)


class CameraHealth:
    """Reference view + the checks that say whether a frame still matches it."""

    def __init__(self, ref_gray, ref_shape, zone_tol_frac=DEFAULT_ZONE_TOL_FRAC,
                 min_inliers=DEFAULT_MIN_INLIERS):
        self.ref = ref_gray
        self.ref_shape = tuple(ref_shape)          # (w, h) of the ORIGINAL frame
        self.zone_tol_frac = zone_tol_frac
        self.min_inliers = min_inliers
        self._orb = cv2.ORB_create(nfeatures=1500) if cv2 else None
        self._kp, self._des = (self._orb.detectAndCompute(ref_gray, None)
                               if self._orb is not None else (None, None))
        self._scale = ref_gray.shape[1] / max(self.ref_shape[0], 1)

    # -- lifecycle -----------------------------------------------------------
    @classmethod
    def from_frame(cls, frame, **kw):
        h, w = frame.shape[:2]
        return cls(_prep(frame), (w, h), **kw)

    def save(self, path):
        """Persist so chunk 7 is compared against chunk 1's view, not its own.

        V73: append the extension, never with_suffix(). Callers build the path
        as `viewref_{camera_id}` with no extension, and this venue's camera_id
        ends "...5.30.00pm CDT" — pathlib treats ".00pm CDT" as the suffix and
        REPLACES it. save() then wrote one filename and load() looked for
        another, so the camera-moved guard silently compared against nothing on
        every run. The 68b97311f9 log shows the mangled name:
            viewref_CAM.112 (PP.09_12) 7-28-2026, 4.30.00pm CDT - ..., 5.30.png
        """
        p = Path(path)
        cv2.imwrite(str(Path(str(p) + ".png")), self.ref)
        Path(str(p) + ".json").write_text(json.dumps({
            "ref_shape": list(self.ref_shape),
            "zone_tol_frac": self.zone_tol_frac,
            "min_inliers": self.min_inliers}))
        return p

    @classmethod
    def load(cls, path):
        p = Path(path)
        img = cv2.imread(str(Path(str(p) + ".png")), cv2.IMREAD_GRAYSCALE)  # V73
        if img is None:
            return None
        meta = json.loads(Path(str(p) + ".json").read_text())               # V73
        return cls(img, meta["ref_shape"], meta.get("zone_tol_frac", DEFAULT_ZONE_TOL_FRAC),
                   meta.get("min_inliers", DEFAULT_MIN_INLIERS))

    # -- the checks ----------------------------------------------------------
    def estimate_shift(self, frame):
        """Affine transform from the reference view to this frame.

        ORB + RANSAC on purpose. People walking through the scene are outliers
        by definition, and RANSAC is what makes 'the background agrees' the
        thing being measured rather than 'the pixels agree'.
        """
        cur = _prep(frame)
        if self._des is None or self._orb is None:
            return None, 0
        kp2, des2 = self._orb.detectAndCompute(cur, None)
        if des2 is None or len(kp2) < 4 or len(self._kp) < 4:
            return None, 0
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(self._des, des2)
        if len(matches) < 4:
            return None, 0
        matches = sorted(matches, key=lambda m: m.distance)[:400]
        src = np.float32([self._kp[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
        M, inl = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC,
                                             ransacReprojThreshold=3.0,
                                             maxIters=3000, confidence=0.995)
        n_inl = int(inl.sum()) if inl is not None else 0
        return M, n_inl

    def zone_displacement(self, M, polygons):
        """Worst zone-vertex displacement in ORIGINAL-frame pixels.

        The number that actually decides whether the report is valid, because a
        zone is what the metrics are computed over.
        """
        if M is None:
            return None
        pts = []
        for poly in (polygons or {}).values():
            pts.extend([(float(x), float(y)) for x, y in np.asarray(poly).reshape(-1, 2)])
        if not pts:
            w, h = self.ref_shape                    # fall back to frame corners
            pts = [(0, 0), (w, 0), (w, h), (0, h)]
        worst = 0.0
        for x, y in pts:
            xs, ys = x * self._scale, y * self._scale        # into working scale
            nx = M[0, 0] * xs + M[0, 1] * ys + M[0, 2]
            ny = M[1, 0] * xs + M[1, 1] * ys + M[1, 2]
            worst = max(worst, math.hypot(nx - xs, ny - ys) / max(self._scale, 1e-9))
        return worst

    def image_quality(self, frame, prev_frame=None):
        """The other four ways the video is not what you think it is."""
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
        small = cv2.resize(g, (320, 180))
        blur = float(cv2.Laplacian(small, cv2.CV_64F).var())
        bright = float(small.mean())
        edges = float((cv2.Canny(small, 50, 150) > 0).mean())
        frozen = None
        if prev_frame is not None:
            pg = (cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
                  if prev_frame.ndim == 3 else prev_frame)
            frozen = float(cv2.absdiff(small, cv2.resize(pg, (320, 180))).mean())
        problems = []
        if blur < BLUR_VAR_FLOOR:
            problems.append(f"OUT OF FOCUS (sharpness {blur:.0f} < {BLUR_VAR_FLOOR:.0f})")
        if bright > BRIGHT_HI:
            problems.append(f"BLINDED / over-exposed (brightness {bright:.0f})")
        elif bright < BRIGHT_LO:
            problems.append(f"TOO DARK to detect anything (brightness {bright:.0f})")
        if edges < EDGE_DENSITY_FLOOR:
            problems.append(f"LENS BLOCKED (edge density {edges:.4f})")
        if frozen is not None and frozen < FROZEN_DIFF_FLOOR:
            problems.append(f"FROZEN STREAM (consecutive frames identical, diff {frozen:.3f})")
        return {"blur": blur, "brightness": bright, "edge_density": edges,
                "frozen_diff": frozen, "problems": problems}

    def check(self, frame, polygons=None, prev_frame=None):
        """Full verdict. `valid` False means: do NOT publish zone-based numbers."""
        w, h = frame.shape[1], frame.shape[0]
        q = self.image_quality(frame, prev_frame)
        res = {"resolution": (w, h), "ref_resolution": self.ref_shape,
               "quality": q, "moved": False, "valid": True, "reasons": [],
               "zone_shift_px": None, "inliers": 0, "rotation_deg": None,
               "scale": None}

        # U2 — resolution change. Zones are pixel coordinates; a different frame
        # size means they point at different parts of the room.
        if (w, h) != self.ref_shape:
            res["valid"] = False
            res["reasons"].append(
                f"RESOLUTION CHANGED {self.ref_shape[0]}x{self.ref_shape[1]} -> "
                f"{w}x{h}; zone coordinates no longer refer to the same places")
            return res

        M, n_inl = self.estimate_shift(frame)
        res["inliers"] = n_inl
        if M is None or n_inl < self.min_inliers:
            res["valid"] = False
            res["reasons"].append(
                f"CANNOT VERIFY the view ({n_inl} matching background features, "
                f"need {self.min_inliers}). Either the scene changed completely "
                f"or the image is too degraded to compare.")
            return res

        res["rotation_deg"] = float(math.degrees(math.atan2(M[1, 0], M[0, 0])))
        res["scale"] = float(math.hypot(M[0, 0], M[1, 0]))
        shift = self.zone_displacement(M, polygons)
        res["zone_shift_px"] = shift
        tol = self.zone_tol_frac * math.hypot(*self.ref_shape)
        res["zone_tol_px"] = tol
        if shift is not None and shift > tol:
            res["moved"] = True
            res["valid"] = False
            res["reasons"].append(
                f"CAMERA MOVED: zone corners are {shift:.0f} px out of place "
                f"(tolerance {tol:.0f} px, rotation {res['rotation_deg']:+.2f} deg, "
                f"scale {res['scale']:.3f}). Every zone-based number would be "
                f"measuring the wrong part of the room. Re-draw the zones on a "
                f"current frame, or restore the camera to its original aim.")
        if q["problems"]:
            res["valid"] = False
            res["reasons"].extend(q["problems"])
        return res


def verdict_line(res):
    """One line for the run log. Green means the numbers may be published."""
    if res["valid"]:
        s = res.get("zone_shift_px")
        return (f"OK  camera view unchanged"
                + (f" (zones off by {s:.1f} px, tolerance "
                   f"{res.get('zone_tol_px', 0):.0f} px, {res['inliers']} features)"
                   if s is not None else ""))
    return "INVALID  " + " | ".join(res["reasons"])
''',
    'clock.py': r'''"""clock.py — what time is it, really, and is that clock trustworthy?

WHY THIS EXISTS
    The entire product is timestamps. "The desk was empty from 16:47" is the
    deliverable; the tracking is only how we get there. Yet every time-related
    guard lived in notebook Cell 2e, so kevacv.pipeline — the path meant to be
    universal — had no clock verification at all.

    Three separate ways the clock lies, all of which produce a confident,
    plausible, wrong report:

    1. VARIABLE FRAME RATE. frame_index/fps is not the clock if the container
       lies about fps. Durations drift by a growing amount, so an early gap
       reads correctly and a late one is minutes out. Detected by comparing
       assumed time against CAP_PROP_POS_MSEC at several points.

    2. DST. The clock is parsed from a filename as naive local time. An
       overnight run across a spring-forward or fall-back boundary shifts every
       timestamp by an hour — and fall-back makes one local hour happen twice,
       so two different real moments print identically.

    3. NON-MONOTONIC TIMESTAMPS. A decoder hiccup can repeat or reverse a
       timestamp. merge_intervals, covered_windows and every dwell computation
       assume time only moves forward; a reversal produces negative durations
       that quietly cancel real ones out.

    A wrong-video/wrong-clock pairing already shipped once here: CHUNK_FILTER
    selected the 7:30pm file, the 4:30pm file was on disk, and the run stamped
    19:30 onto 16:30 footage. verify_provenance() is that guard, in code that
    travels with the pipeline instead of with one notebook.
"""
from __future__ import annotations

import re
from datetime import datetime, timedelta

from .log import get_logger

_log = get_logger("clock")

# "CAM.112 (PP.09_12) 7-28-2026, 4.30.00pm CDT - 7-28-2026, 5.30.00pm CDT.mp4"
_STAMP = re.compile(
    r"(?P<m>\d{1,2})-(?P<d>\d{1,2})-(?P<y>\d{4})\s*,\s*"
    r"(?P<hh>\d{1,2})\.(?P<mm>\d{2})\.(?P<ss>\d{2})\s*(?P<ap>[ap]m)",
    re.I)

VFR_DRIFT_PCT = 1.0        # above this, frame_index/fps is not the clock


def parse_start(name):
    """-> naive datetime of the chunk's START, from its filename.

    Reads the FIRST stamp only. Filenames carry a range ("4.30pm - 5.30pm") and
    a bare substring search matched the previous hour's END time, which is how
    a chunk filter once selected the wrong file.
    """
    m = _STAMP.search(str(name))
    if not m:
        return None
    g = m.groupdict()
    hh = int(g["hh"]) % 12
    if g["ap"].lower() == "pm":
        hh += 12
    try:
        return datetime(int(g["y"]), int(g["m"]), int(g["d"]),
                        hh, int(g["mm"]), int(g["ss"]))
    except ValueError:
        return None


def localize(dt, tz_name):
    """Attach a real timezone. Returns (aware_dt, findings).

    Naive local time is not a time. Two hours a year it is ambiguous, and one
    hour a year it does not exist.
    """
    findings = []
    if dt is None:
        return None, [("ERROR", "no timestamp could be parsed from the filename "
                                "— wall-clock times would be video time only")]
    try:
        from zoneinfo import ZoneInfo
    except ImportError:
        return dt, [("WARN", "zoneinfo unavailable; times remain naive local")]
    try:
        zone = ZoneInfo(tz_name)
    except Exception:
        return dt, [("WARN", f"unknown timezone {tz_name!r}; times remain naive")]

    aware = dt.replace(tzinfo=zone)
    # fold=1 names the SECOND pass through a repeated local hour. If the two
    # folds have different UTC offsets, this local time happens twice today.
    if aware.utcoffset() != dt.replace(tzinfo=zone, fold=1).utcoffset():
        findings.append(("ERROR",
                         f"{dt} is AMBIGUOUS in {tz_name} — the clock went back "
                         f"and this local hour occurs twice. Two different real "
                         f"moments would print the same time."))
    # A time that does not exist round-trips to something else.
    if aware.astimezone(ZoneInfo("UTC")).astimezone(zone).replace(tzinfo=None) != dt:
        findings.append(("ERROR",
                         f"{dt} does not exist in {tz_name} — the clock went "
                         f"forward through it."))
    return aware, findings


def check_dst_span(start, hours, tz_name):
    """Does this run cross a DST transition? -> findings.

    Checked on the SPAN, not just the start: a 10-hour overnight chunk can
    begin in one offset and end in another, and every timestamp after the
    boundary is then an hour out.
    """
    if start is None:
        return []
    aware, f = localize(start, tz_name)
    if aware is None or aware.tzinfo is None:
        return f
    end = aware + timedelta(hours=float(hours or 0))
    if aware.utcoffset() != end.utcoffset():
        f.append(("ERROR",
                  f"this run crosses a DST change ({aware.utcoffset()} -> "
                  f"{end.utcoffset()}). Timestamps after the boundary are off "
                  f"by the difference unless every conversion is done in UTC."))
    return f


def check_frame_clock(probe_times, fps):
    """Is frame_index/fps the real clock? -> (source, worst_drift_pct, findings).

    probe_times: [(frame_index, actual_seconds), ...] read from the decoder.
    Returns "frame_index" or "pos_msec" — the caller should use the latter as
    its time source when drift is high.
    """
    if not probe_times or not fps:
        return "frame_index", 0.0, [("WARN", "clock could not be verified; "
                                             "assuming constant frame rate")]
    worst = 0.0
    for idx, actual in probe_times:
        assumed = idx / float(fps)
        if actual and actual > 0:
            worst = max(worst, abs(assumed - actual) / actual * 100.0)
    if worst > VFR_DRIFT_PCT:
        return "pos_msec", worst, [
            ("ERROR", f"VARIABLE FRAME RATE — worst drift {worst:.2f}%. "
                      f"frame_index/fps is NOT the clock here; every duration "
                      f"would be wrong by a growing amount. Use "
                      f"CAP_PROP_POS_MSEC as the time source.")]
    return "frame_index", worst, []


def verify_provenance(selected_name, decoded_name, clock_source_name):
    """The file we picked, the file we decoded, and the file the clock came
    from must be the SAME file. -> findings.

    This exact mismatch shipped: a chunk filter selected the 7:30pm file, the
    4:30pm file was already on disk, and the run stamped 19:30 onto 16:30
    footage. Every timestamp in that report was three hours wrong and nothing
    complained, because each step was individually correct.
    """
    names = {"selected": str(selected_name or ""),
             "decoded": str(decoded_name or ""),
             "clock": str(clock_source_name or "")}
    stamps = {k: parse_start(v) for k, v in names.items() if v}
    distinct = {v for v in stamps.values() if v is not None}
    if len(distinct) > 1:
        detail = "  ".join(f"{k}={v}" for k, v in stamps.items())
        return [("ERROR",
                 f"PROVENANCE MISMATCH — the selected file, the decoded file "
                 f"and the clock source do not agree: {detail}. Every "
                 f"timestamp in this report would be wrong.")]
    base = {k: v.rsplit("/", 1)[-1].rsplit("\\", 1)[-1] for k, v in names.items() if v}
    if len(set(base.values())) > 1:
        return [("WARN",
                 f"selected/decoded/clock filenames differ but parse to the "
                 f"same start time: {base}")]
    return []


def describe(source, drift_pct, findings):
    L = [f"clock source   {source}   worst drift {drift_pct:.2f}%"]
    for lvl, msg in findings:
        L.append(f"  [{lvl}] {msg}")
    if not findings:
        L.append("  no clock problems found")
    return "\n".join(L)
''',
    'config.py': r'''"""config.py — every tracking-critical constant, in one importable place.

WHY THIS EXISTS
    These lived as bare globals in notebook Cell 2, and Cell 2e overrode three
    of them AFTERWARDS. The CONFIG AUDIT printed the Cell 2 values, so the log
    said REID_MAX_GAP_S = 7200 while the run actually used 900. A block whose
    stated job is "all tracking-critical parameters" was reporting a stale
    version of them (patched in V75; this module removes the possibility).

    Two values genuinely have two lives:

        REID_MAX_GAP_S   Cell 2: 7200   Cell 2e: 900
        MAX_BODY_GAP_S   Cell 2:  480   Cell 2e: 300

    Cell 2e is the 10-hour scale profile and its values are the ones that run.
    They are the defaults here; SCALE_PROFILE_OVERRIDES records what changed so
    the difference stays visible instead of being folded away silently.

HOW TO CHANGE A VALUE
    Not by editing this file for one run. Use tools/run_night.py:

        KV_REID_MAX_GAP_S=1200 python tools/run_night.py

    which injects an override cell after the config cell — so a run's settings
    are recorded in that run's own log rather than in an edit nobody sees.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field, fields

# ── appearance / re-identification ──────────────────────────────────────────
REID_SIM_THRESHOLD = 0.60      # gallery merge bar
ANCHOR_SIM_THRESHOLD = 0.75    # best-crop 1:1, stricter than the gallery
# NOTE: measured same-person p50 on CAM.112 is 0.435 — BELOW both bars, so the
# system rejects the majority of true matches. See kevacv/reid_calibration.py;
# the number itself was measured circularly and needs re-measuring.

# ── time gates ──────────────────────────────────────────────────────────────
REID_MAX_GAP_S = 900.0         # Cell 2e value (Cell 2 said 7200)
MAX_BODY_GAP_S = 300.0         # Cell 2e value (Cell 2 said 480)
NEAR_GAP_S = 15.0
FAR_GAP_S = 180.0
NEAR_GAP_BONUS = 0.04          # short gap -> threshold relaxed by this
FAR_GAP_PENALTY = 0.05         # long gap  -> threshold raised by this
# 0.05 is small next to what a long gap costs in discriminative power: the
# metric gate allows max_speed * gap, which at 900 s is ~1980 m — i.e. no
# constraint at all. kevacv/topology.py exists to hold that line instead.

# ── spatial plausibility ────────────────────────────────────────────────────
MAX_PLAUSIBLE_SPEED_PX = 220.0
SPATIAL_PENALTY_SCALE = 0.15
MAX_SPATIAL_PENALTY = 0.30

# ── occlusion ───────────────────────────────────────────────────────────────
OCCLUSION_IOU = 0.30           # boxes at/above this mutually occlude
OCCLUSION_CONTAIN = 0.60       # ...or one is this fraction inside the other

# ── vetoes (a second opinion may block a merge, never create one) ───────────
ENABLE_HANDOFF_APPEARANCE_VETO = True
HANDOFF_VETO_SIM = 0.30
ENABLE_HANDOFF_HSV_VETO = True
HANDOFF_HSV_VETO_SIM = 0.50
ENABLE_APPEARANCE_HSV_VETO = True
APPEARANCE_HSV_VETO_SIM = 0.45

# ── infrared ────────────────────────────────────────────────────────────────
IR_TRACK_FRAC = 0.5            # a track this IR-heavy gets no colour evidence
# 58% of CAM.112's frames are infrared and 67 of 94 tracks were withheld from
# colour evidence entirely. Published SOTA for visible-infrared re-id is ~77%
# Rank-1, so nothing here should depend on appearance surviving the boundary.

# ── geometry ────────────────────────────────────────────────────────────────
DEFAULT_HFOV_DEG = 90.0        # overridden per venue by venue_profile
PERSON_H_M = 1.70

SCALE_PROFILE_OVERRIDES = {
    # what Cell 2e changes, and why. Kept as data so the audit can print the
    # difference rather than one value silently winning.
    "REID_MAX_GAP_S": (7200.0, 900.0,
                       "7200 s was 'the whole video' when a video was 10 min. "
                       "Over 10 h it invites two strangers in black shirts, "
                       "hours apart, to become one person."),
    "MAX_BODY_GAP_S": (480.0, 300.0,
                       "appearance-only tiers get a tighter cap than face."),
}


@dataclass
class TrackingConfig:
    """All of the above as one object, so a run can carry its settings around
    instead of reading module globals that something else may have rebound."""
    reid_sim_threshold: float = REID_SIM_THRESHOLD
    anchor_sim_threshold: float = ANCHOR_SIM_THRESHOLD
    reid_max_gap_s: float = REID_MAX_GAP_S
    max_body_gap_s: float = MAX_BODY_GAP_S
    near_gap_s: float = NEAR_GAP_S
    far_gap_s: float = FAR_GAP_S
    near_gap_bonus: float = NEAR_GAP_BONUS
    far_gap_penalty: float = FAR_GAP_PENALTY
    max_plausible_speed_px: float = MAX_PLAUSIBLE_SPEED_PX
    spatial_penalty_scale: float = SPATIAL_PENALTY_SCALE
    max_spatial_penalty: float = MAX_SPATIAL_PENALTY
    occlusion_iou: float = OCCLUSION_IOU
    occlusion_contain: float = OCCLUSION_CONTAIN
    handoff_veto_sim: float = HANDOFF_VETO_SIM
    handoff_hsv_veto_sim: float = HANDOFF_HSV_VETO_SIM
    appearance_hsv_veto_sim: float = APPEARANCE_HSV_VETO_SIM
    enable_handoff_appearance_veto: bool = ENABLE_HANDOFF_APPEARANCE_VETO
    enable_handoff_hsv_veto: bool = ENABLE_HANDOFF_HSV_VETO
    enable_appearance_hsv_veto: bool = ENABLE_APPEARANCE_HSV_VETO
    ir_track_frac: float = IR_TRACK_FRAC
    default_hfov_deg: float = DEFAULT_HFOV_DEG

    def as_dict(self):
        return asdict(self)

    def describe(self):
        """The audit, printed from the object that will actually be used —
        so it cannot describe a value something later overrode."""
        L = ["=" * 78, "  TRACKING CONFIG — the values this run will use", "=" * 78]
        for f in fields(self):
            L.append(f"  {f.name:<34} = {getattr(self, f.name)}")
        if SCALE_PROFILE_OVERRIDES:
            L.append("  " + "-" * 74)
            L.append("  scale-profile overrides applied (Cell 2 value -> used):")
            for k, (old, new, why) in SCALE_PROFILE_OVERRIDES.items():
                L.append(f"    {k:<20} {old} -> {new}")
                L.append(f"      {why}")
        L.append("=" * 78)
        return "\n".join(L)


DEFAULT = TrackingConfig()


# ============================================================================
#  ENGINE — extracted from notebook Cell 7 (132 bare globals -> named config)
#
#  Cell 7 also performs 33 `globals().get(...)` lookups at runtime. Those are
#  deliberate late-binding hooks (a later cell may rebind a value), so they
#  are NOT replaced here — engine.py keeps reading them, and this module just
#  supplies the defaults they fall back to.
# ============================================================================

# ── detector — what counts as a person at all ───────────────────────────────
DETECT_CONF_FLOOR = 0.25
CONF_THRESHOLD = 0.35
NEW_TRACK_CONF = 0.45
KEEP_TRACK_CONF = 0.2
ENABLE_CONF_HYSTERESIS = True
YOLO_IMGSZ = 1280
DET_BATCH = 12
DEDUP_NMS_IOU = 0.7
IR_DETECT_CONF_FLOOR = 0.2

# ── sampling & motion gate — the frames we bother to look at ────────────────
FPS_TARGET = 15   # Cell 2 literal; ANALYSIS_FPS (Cell 2e) is what the run samples at
ANALYSIS_MAX_W = 1280
MOTION_GATE = True
MOTION_IDLE_S = 10.0
MOTION_MIN_FRAC = 0.002
ENABLE_RESOLUTION_SCALING = True
REF_DIAGONAL_PX = 1468.6
ENABLE_CLAHE = False

# ── tracker ─────────────────────────────────────────────────────────────────
TRACKER_MODE = 'botsort-reid'
USE_REAL_ONLINE_REID = True
BOTSORT_MATCH_THRESH = 0.75
LOST_TRACK_BUFFER_S = 60
ENABLE_GMC = True
GMC_METHOD = 'sof'

# ── live identity memory — one canonical id per body, decided per frame ─────
ENABLE_LIVE_IDENTITY_MEMORY = True
LIVE_REID_SIM_THRESHOLD = 0.62
LIVE_REID_MEMORY_TTL_S = 1800.0
LIVE_REID_MAX_SPEED_PX_S = 560.0
LIVE_APPEARANCE_THRESH = 0.5
ENABLE_COVISIBILITY_BLOCK = True
ENABLE_OCCLUSION_GUARD = True
ENABLE_SWAP_REVALIDATION = True
SWAP_MARGIN = 0.1
MAX_WALK_SPEED_MPS = 2.2

# ── offline re-id — crops, gaps and the merge tiers ─────────────────────────
ENABLE_REID_STITCH = True
REID_CROPS_PER_TRACK = 6
REID_MIN_CROP_H = 50
REEMBED_EVERY_S = 4.0
REID_HANDOFF_PX = 160
REID_HANDOFF_M = 1.6
REID_HANDOFF_GAP_S = 4.0
REID_STATIONARY_PX = 60.0
REID_STATIONARY_M = 0.6
GAP_MERGE_S = 15
ENABLE_ATTIRE_MERGE_TIER = True
HSV_MERGE_SIM_THRESHOLD = 0.75
ENABLE_GLOBAL_TRACKLET = True
GLOBAL_TRACKLET_MAX_IDS = 900
ENABLE_REID_CALIBRATION = True
CALIBRATION_AUTO_APPLY = False
ENABLE_CROSS_VALIDATION = True
CROSS_VAL_GALLERY_SIM = 0.6
CROSS_VAL_ANCHOR_SIM = 0.75

# ── faces — staff only, by policy (DPDP Act; see docs/DELILAH_CV_MASTER_DOC.tex) 
FACE_MODEL_NAME = 'buffalo_l'
FACE_SCOPE = 'staff_only'
FACE_MIN_DET_SCORE = 0.55
FACE_MIN_FACE_PX = 45
FACE_SIM_THRESHOLD = 0.35
FACE_MERGE_SIM_THRESHOLD = 0.45
ENABLE_FACE_MERGE_TIER = True
ENABLE_FACE_CORROBORATION = True
ENABLE_FACE_VETO = True
FACE_VETO_MARGIN = 0.15
FACE_VETO_MAX_EDGE_SCORE = 0.8
FACE_MAX_TRIES = 6
FACE_RETRY_EVERY_S = 3.0
STAFF_GALLERY_DIR = 'staff_gallery'
STAFF_MATCH_THRESHOLD = 0.4
ENABLE_STAFF_GALLERY_SWEEP = True
STAFF_DOMINANCE_RATIO = 3.0
STAFF_MIN_VIDEO_SHARE = 0.35
STAFF_OVERRIDE_MIN_S = 60

# ── phantom & implausible-detection filters ─────────────────────────────────
ENABLE_SIZE_FILTER = True
SIZE_FILTER_TOL = 2.5
MIN_BODY_ASPECT = 0.75
MAX_BODY_ASPECT = 4.0
MIN_BLUR_VARIANCE = 15.0
MIN_CROP_PX_BLUR_GATE = 40
ENABLE_STATIC_FILTER = True
STATIC_MIN_LIFE_S = 120.0
STATIC_CENTRE_JITTER = 0.02
STATIC_SIZE_JITTER = 0.03
PHANTOM_MIN_SPAN_S = 240.0
PHANTOM_CENTRE_JITTER = 0.02
PHANTOM_SIZE_CV = 0.015
ENABLE_CARRIED_SUPPRESS = True
CARRIED_CONTAIN = 0.7
CARRIED_HEIGHT_TOL = 0.55
CARRIED_MAX_AREA_RATIO = 0.45
CARRIED_MIN_FIT_SAMPLES = 200
CARRIED_MIN_HEAD_DROP = 0.15
ENABLE_HEAD_RECOVERY = False

# ── infrared ────────────────────────────────────────────────────────────────
IR_CHROMA_THRESHOLD = 6.0
ENABLE_IR_HARD_CUT = False
IR_CUT_MIN_GAP_S = 30.0

# ── zones & events ──────────────────────────────────────────────────────────
MIN_EVENT_S = 2.0
MIN_SEATED_S = 60
ENTRY_LINE_FLIP = True

# ── render ──────────────────────────────────────────────────────────────────
PROXY_RENDER = True
PROXY_JPEG_QUALITY = 72
RENDER_ONLY_OCCUPIED = False
SNAPSHOT_EVERY_S = 30
HUD_SMOOTH_S = 2.0
TRAIL_MODE = 'moving'
ENABLE_DISPLAY_RENUMBER = True
''',
    'dataset_collector.py': r'''"""dataset_collector.py — Phase D P13 Venue-Specific Dataset Collector & Pseudo-Labeling Engine.

WHY THIS EXISTS
    Verkada's 2025 AI whitepaper demonstrated that a 400M fine-tuned venue-specific model 
    substantially outperforms a 2B general model in both accuracy and speed.

THE PRINCIPLE
    Automated Data Bank & Pseudo-Labeling:
    1. Passively collects high-confidence crops and frame samples during video analytics runs.
    2. Exports pseudo-labeled training sets in YOLO format (images + txt labels).
    3. Enables fine-tuning YOLO / DINOv2 adapters specifically for the venue's camera angles,
       lighting conditions, and occlusion profiles.
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np

try:
    import cv2
    _HAVE_CV2 = True
except ImportError:
    _HAVE_CV2 = False


class DatasetCollector:
    """Venue-specific training sample collector and pseudo-label exporter."""

    def __init__(self, output_dir: Union[str, Path] = "./venue_dataset"):
        self.output_dir = Path(output_dir)
        self.img_dir = self.output_dir / "images"
        self.lbl_dir = self.output_dir / "labels"
        
        self.img_dir.mkdir(parents=True, exist_ok=True)
        self.lbl_dir.mkdir(parents=True, exist_ok=True)
        self.counter = 0

    def save_frame_pseudo_labels(self,
                                 frame: np.ndarray,
                                 boxes: List[Tuple[float, float, float, float]],
                                 confidences: Optional[List[float]] = None,
                                 class_id: int = 0) -> bool:
        """Save a frame image and normalized YOLO format bounding box label file.
        boxes: List of (x1, y1, x2, y2)
        """
        if not _HAVE_CV2 or frame is None or len(boxes) == 0:
            return False

        self.counter += 1
        stem = f"frame_{self.counter:06d}"
        
        img_path = self.img_dir / f"{stem}.jpg"
        lbl_path = self.lbl_dir / f"{stem}.txt"

        h, w = frame.shape[:2]
        if h <= 0 or w <= 0:
            return False

        # Write image
        cv2.imwrite(str(img_path), frame)

        # Write YOLO label format: class_id x_center y_center width height (normalized)
        yolo_lines = []
        for idx, (x1, y1, x2, y2) in enumerate(boxes):
            bw = float(x2 - x1)
            bh = float(y2 - y1)
            xc = float(x1 + x2) / 2.0 / w
            yc = float(y1 + y2) / 2.0 / h
            nw = bw / w
            nh = bh / h

            # Clamp normalized coordinates to [0.0, 1.0]
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            nw = max(0.0, min(1.0, nw))
            nh = max(0.0, min(1.0, nh))

            yolo_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        lbl_path.write_text("\n".join(yolo_lines))
        return True

    def export_dataset_yaml(self, venue_name: str = "delilah_la") -> Path:
        """Export dataset.yaml configuration file for Ultralytics YOLO fine-tuning."""
        yaml_content = f"""# Venue-Specific Dataset Config for {venue_name}
path: {self.output_dir.absolute()}
train: images
val: images

names:
  0: person
"""
        yaml_path = self.output_dir / "dataset.yaml"
        yaml_path.write_text(yaml_content)
        return yaml_path
''',
    'derive.py': r'''"""derive.py — turn what the engine returns into what the answers need.

WHY THIS EXISTS
    process_video returns tracks, events, crossings and a frame log. The
    answers need observed windows, per-guest arrival times, staff contacts,
    a guest list and a per-person confidence. In the notebook those gaps were
    closed by Cell 18 reaching into whatever globals happened to be lying
    around. As a codebase they have to be derived explicitly, from the run
    dict, with the derivation visible.

    Nothing here invents data. Every function returns None or an empty result
    when the inputs cannot support it, because a derived number that quietly
    substitutes for a missing measurement is the exact failure that produced
    "2 people came through the door".

THE ONE RULE
    A derivation must be WEAKER than the thing it stands in for, and must say
    so. observed_windows derived from a frame log is a lower bound on what we
    watched; id_confidence derived from fragment counts is not a probability.
    Both are useful; neither may be presented as measurement.
"""
from __future__ import annotations

import math
from collections import defaultdict

from .log import get_logger

_log = get_logger("derive")

# staff within this many BODY HEIGHTS of a guest counts as a service touch
GREET_RADIUS_BODIES = 1.2
GREET_MIN_CONTACT_S = 3.0


def observed_windows(run):
    """What span did we actually analyse? -> [(t0, t1), ...]

    THE MISTAKE THIS FUNCTION USED TO MAKE
        The first version derived these from frame_log timestamps. But
        frame_log only contains frames that HAD DETECTIONS, so every empty
        stretch looked unobserved and was removed from the denominator. On a
        test fixture that turned a true 66.7% desk coverage into 79.7% —
        inflating the headline metric by excluding exactly the minutes when
        nobody was at the desk.

        An empty room is OBSERVED. It is just empty. Confusing "no detections"
        with "no observation" is the same error as confusing a broken entry
        line with an empty venue, and it lands on the one metric that is
        supposed to be EXACT.

    SO: the analysed span is the denominator, unless something that genuinely
    knows better says otherwise:

        run["observed_windows"]  a validity.ValidityLedger ran and excluded
                                 blind/undecodable frames — that IS observation
                                 evidence, so it wins
        otherwise                the whole analysed span, because the engine
                                 sampled every frame in it whether or not it
                                 found anybody
    """
    if run.get("observed_windows"):
        return [tuple(w) for w in run["observed_windows"]]
    end = float(run.get("t_end") or run.get("duration_s") or 0.0)
    start = float(run.get("start_seconds") or 0.0)
    if end <= start:
        _log.warning("no duration on the run — observed time is unknown, and "
                     "every percentage will report UNKNOWN rather than guess")
        return []
    return [(start, end)]


def arrivals_by_id(run, roles=None):
    """First inward door crossing per non-staff track. -> {track_id: t}

    Returns {} — not a fabricated set — when the line never fired. A broken
    entry line must propagate as "no arrival times", so greet latency comes
    back UNKNOWN instead of silently becoming zero.
    """
    roles = roles or run.get("roles") or {}
    out = {}
    for c in run.get("crossings") or []:
        if c.get("direction") != "in":
            continue
        tid = c.get("track_id")
        if roles.get(tid) == "staff":
            continue
        t = float(c.get("t", 0.0))
        if tid not in out or t < out[tid]:
            out[tid] = t
    return out


def guest_ids(run, roles=None, arrivals=None):
    """Who counts as a guest. -> [track_id]

    Prefers door crossings. Falls back to "non-staff identity seen in an
    interior zone", which is weaker and is labelled as such by the caller —
    guest_count() already reports its source.
    """
    roles = roles or run.get("roles") or {}
    arrivals = arrivals if arrivals is not None else arrivals_by_id(run, roles)
    if arrivals:
        return sorted(arrivals, key=str)
    seen = {e.get("track_id") for e in (run.get("events") or [])
            if roles.get(e.get("track_id")) != "staff"}
    return sorted((t for t in seen if t is not None), key=str)


def staff_contacts(run, roles=None, radius_bodies=GREET_RADIUS_BODIES,
                   min_contact_s=GREET_MIN_CONTACT_S):
    """When was a staff member near each guest? -> {guest_id: [t_start, ...]}

    THE PROXY, STATED HONESTLY
        This measures proximity, sustained for min_contact_s. It is not
        conversation and cannot become conversation with a better tracker —
        SUCCESS_CRITERIA says the honest upgrade is a VLM on a short clip.

        The radius is in BODY HEIGHTS, not pixels: a person near the camera and
        one across the room are the same distance apart in the world at very
        different pixel separations, and a fixed pixel radius would make greet
        detection depend on where in the frame someone stood.
    """
    roles = roles or run.get("roles") or {}
    fl = run.get("frame_log") or []
    if not fl:
        return {}
    near = defaultdict(list)               # guest -> [t where staff was close]
    for _i, t, boxes in fl:
        staff, guests = [], []
        for tid, x1, y1, x2, y2 in boxes:
            cx = (float(x1) + float(x2)) / 2.0
            foot = float(y2)
            h = max(1.0, float(y2) - float(y1))
            (staff if roles.get(tid) == "staff" else guests).append(
                (tid, cx, foot, h))
        if not staff or not guests:
            continue
        for gid, gx, gy, gh in guests:
            lim = gh * radius_bodies
            if any(math.hypot(gx - sx, gy - sy) <= lim
                   for _s, sx, sy, _sh in staff):
                near[gid].append(float(t))

    out = {}
    for gid, ts in near.items():
        ts.sort()
        runs, start, prev = [], ts[0], ts[0]
        step = 1.0
        for t in ts[1:]:
            if t - prev > max(step * 3, 2.0):
                if prev - start >= min_contact_s:
                    runs.append(start)
                start = t
            prev = t
        if prev - start >= min_contact_s:
            runs.append(start)
        if runs:
            out[gid] = runs
    return out


def id_confidence(run, guests=None):
    """How much to trust each identity. -> {track_id: 0-100}

    NOT A PROBABILITY. It combines signals the pipeline already has — how long
    the identity lived, how many fragments were stitched into it, whether a
    face confirmed it, whether it crossed the door — into a number whose only
    job is to split a guest count into a range. Treat it as an ordering, not a
    measurement.
    """
    roles = run.get("roles") or {}
    windows = defaultdict(lambda: [math.inf, -math.inf])
    for e in run.get("events") or []:
        tid = e.get("track_id")
        w = windows[tid]
        w[0] = min(w[0], float(e["t_in"]))
        w[1] = max(w[1], float(e.get("t_out", e["t_in"])))
    frags = defaultdict(int)
    for a, b in (run.get("canon_map") or {}).items():
        frags[b] += 1
    faced = set(run.get("staff_matched_names") or [])
    crossed = {c.get("track_id") for c in (run.get("crossings") or [])}

    out = {}
    for tid in (guests if guests is not None else list(windows)):
        w = windows.get(tid)
        score = 40
        if w and w[1] > w[0]:
            life = w[1] - w[0]
            score += 20 if life >= 60 else 10 if life >= 20 else 0
        n = frags.get(tid, 1)
        score += 15 if n <= 1 else 5 if n <= 3 else -10   # many fragments = doubt
        if tid in crossed:
            score += 15          # a door crossing is physical evidence
        if tid in faced or (isinstance(tid, str) and not str(tid).isdigit()):
            score += 20          # a recognised face is the strongest signal
        out[tid] = max(0, min(100, score))
    return out


def enrich(run):
    """Add every derived field the answers need. Returns the same dict.

    Called once, in the pipeline, so the derivation happens in exactly one
    place and a later reader can see what was measured and what was inferred.
    """
    roles = run.get("roles") or {}
    run.setdefault("observed_windows", observed_windows(run))
    arr = arrivals_by_id(run, roles)
    run.setdefault("arrivals_by_id", arr)
    gids = guest_ids(run, roles, arr)
    run.setdefault("guest_ids", gids)
    run.setdefault("contacts", staff_contacts(run, roles))
    run.setdefault("id_confidence", id_confidence(run, gids))
    obs = run["observed_windows"]
    _log.info(f"derived: observed={len(obs)} window(s) "
              f"({sum(b - a for a, b in obs):.0f}s)  arrivals={len(arr)}  "
              f"guests={len(gids)}  contacts={len(run['contacts'])}")
    if not arr:
        _log.warning("no door crossings — greet latency will report UNKNOWN "
                     "rather than 0, and the guest list falls back to "
                     "interior-zone sightings")
    return run
''',
    'detect_filters.py': r'''"""detect_filters.py — PHASE 7. Two filters for two phantoms the real run produced.

WHAT THE FIRST REAL RUN SHOWED
    Five independent readers audited 35 annotated frames from CAM.112. Recall
    was excellent: 120 person-sightings, exactly ONE human missed. Precision was
    not: roughly 19% of the boxes were not people.

    They were not random. They were two specific, repeatable failures:

      P3   a box covering the ENTIRE right half of the frame, present in ~17 of
           35 frames, counted as a person every time.
      P8/  a box around the POTTED PLANT — and labelled "staff", because a plant
      P13  standing in a staff zone trivially satisfies the dwell rule.

    Swapping the detector does not fix this. Stock YOLO produces no giant boxes
    but finds only 32% of the people on this greyscale, steeply-tilted, wide
    footage. Our fine-tune has the recall; it needs its output filtered.

WHY THESE TWO FILTERS AND NOT A NEW MODEL
    Both phantoms are already detectable from data the pipeline collects:

      GIANT   a person's pixel height at a given footline is bounded — the
              scene-geometry fit (_PerspectiveModel) already predicts it. The
              carried-person rule uses that prediction to catch boxes that are
              too SHORT. This is the same test inverted.
      STATIC  a plant's box never moves and never changes size. A human's box
              always jitters — body sway, arm movement, detector noise. Track-
              level variance separates them with no new model at all.
"""
from __future__ import annotations

import math
import statistics
from collections import defaultdict


# ---------------------------------------------------------------------------
# D1 — a person cannot be taller than the floor allows
# ---------------------------------------------------------------------------
BODY_ASPECT = 3.5      # a standing person is ~3.5x taller than wide


def implausible_size_mask(boxes, expected_h, tol=2.5, aspect=BODY_ASPECT):
    """-> list[bool], True where the box is too BIG to be a person standing there.

    boxes:      iterable of (x1, y1, x2, y2)
    expected_h: fn(foot_y) -> predicted pixel height of a standing person whose
                feet are at that row, or None when it cannot say (no fit yet, or
                the point is above the horizon).

    WHY AREA AND NOT HEIGHT. The first version of this tested height alone and
    its own test caught it failing: P3, the box covering half the frame, is only
    1.44x too TALL — barely separable from a genuinely tall person at 1.11x. But
    P3 is also enormously too WIDE, and area combines both:

        P3 (half the frame)   425 x 710 px  vs expected  69,500 px^2  ->  4.34x
        a very tall person    100 x 550 px  vs expected  69,500 px^2  ->  0.79x

    Four times versus four fifths. Height alone gave 1.44 versus 1.11, which no
    honest threshold separates.

    Expected area is expected_h^2 / aspect: a person's width is their height over
    the body aspect ratio, so the whole thing is still driven by the one number
    the scene-geometry fit actually predicts.

    tol=2.5 leaves room for two people merged into one box (~2x), an outstretched
    arm, or a coat — it is aimed at 4x+ absurdities, not at borderline calls.

    When expected_h returns None NOTHING is flagged. A filter that fires on
    missing information is worse than no filter, and deleting a real detection
    costs more than keeping a phantom.
    """
    out = []
    for x1, y1, x2, y2 in boxes:
        w, h = float(x2) - float(x1), float(y2) - float(y1)
        exp = expected_h(y2)
        if exp is None or w <= 0 or h <= 0:
            out.append(False)
            continue
        out.append(bool(w * h > tol * (exp * exp / aspect)))
    return out


# ---------------------------------------------------------------------------
# D2 — furniture does not fidget
# ---------------------------------------------------------------------------
def static_track_ids(frame_log, canon=None, protected=(), min_life_s=120.0,
                     max_centre_jitter=0.02, max_size_jitter=0.03,
                     min_sightings=20):
    """Track ids whose box is so rigid over so long that it must be furniture.

    frame_log:  [(frame_idx, t, [(track_id, x1, y1, x2, y2), ...]), ...]
    canon:      optional {raw_id: canonical_id} so a stitched identity is judged
                as one thing rather than as its fragments.
    protected:  ids that must NEVER be dropped whatever the geometry says —
                anyone who crossed the entry line, or whose face was recognised.
                A human who was seen walking through the door is a human, and no
                statistic gets to overrule that.

    The thresholds are fractions of the box's own height, not pixels, so the
    same rule holds for someone near the camera and someone across the room.

    A person standing still at a desk still moves: they lean, turn, gesture, and
    the detector's own noise adds more. Measured on real detections, that is
    comfortably above 2% of body height. A plant sits at exactly the same pixels
    for twenty minutes.
    """
    canon = canon or {}
    seen = {}
    for _fi, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            cid = canon.get(tid, tid)
            seen.setdefault(cid, []).append(
                (float(t), (float(x1) + float(x2)) / 2.0,
                 (float(y1) + float(y2)) / 2.0,
                 float(x2) - float(x1), float(y2) - float(y1)))

    flagged = {}
    for cid, rows in seen.items():
        if cid in protected or len(rows) < min_sightings:
            continue
        life = rows[-1][0] - rows[0][0]
        if life < min_life_s:
            continue
        cx = [r[1] for r in rows]
        cy = [r[2] for r in rows]
        w = [r[3] for r in rows]
        h = [r[4] for r in rows]
        med_h = statistics.median(h) or 1.0
        jit = max(statistics.pstdev(cx), statistics.pstdev(cy)) / med_h
        size = max(statistics.pstdev(w), statistics.pstdev(h)) / med_h
        if jit <= max_centre_jitter and size <= max_size_jitter:
            flagged[cid] = {"seconds": round(life, 1), "sightings": len(rows),
                            "centre_jitter": round(jit, 4),
                            "size_jitter": round(size, 4),
                            "at": (round(statistics.median(cx)),
                                   round(statistics.median(cy)))}
    return flagged


def rigid_track_ids(frame_log, canon=None, protected=(), min_life_s=60.0,
                    max_aspect_cv=0.06, min_sightings=25,
                    max_travel_frac=0.10, frame_wh=None):
    """Track ids whose SHAPE never changes — a plant, mannequin, poster, sign.

    WHY THIS EXISTS, AND WHY static_track_ids IS NOT ENOUGH
        static_track_ids catches things that do not MOVE. It misses the whole
        family that moves a little and is still not a person: a plant swaying
        in the aircon, a mannequin whose box jitters as the detector re-fits it
        each frame, a poster whose box drifts with exposure changes.

        The detector cannot help here. best.pt was fine-tuned on CrowdHuman —
        humans only — so it has never been shown a mannequin and told "not a
        person". Its confidence on a coat stand is honest and useless. No
        threshold repairs an error that is not in the score.

    THE INDEPENDENT SIGNAL
        A person is DEFORMABLE. Walking, turning, reaching, sitting — the box
        aspect ratio moves constantly. A rigid object's aspect is near
        constant for its entire life, however much the box wanders.

        So we measure the coefficient of variation of h/w over the track. Low
        CV over a long life = rigid = not a person. This is geometry the
        detector cannot influence, which is the point: a category channel that
        is independent of the thing that got the category wrong.

    WHY TRAVEL IS ALSO REQUIRED
        Rigidity ALONE is not enough, and a synthetic test caught this filter
        flagging a walking person: any track whose detector box happens to keep
        a steady aspect would qualify — someone walking straight away from the
        camera, or a detector that boxes consistently.

        Furniture has a second property: it does not TRANSLATE. A plant, a
        mannequin and a poster stay put. So the verdict needs both — the shape
        never changes AND the thing never goes anywhere.

        A track that is rigid but DOES travel is a different animal: usually a
        reflection, which is mirrored_pair_ids' job. Keeping the two separate
        stops either from absorbing the other's false positives.

    WHAT IT DELIBERATELY DOES NOT CATCH
        A TV or monitor showing people. Those DO deform — they are people, just
        not present ones. They fail a different test: many identities born and
        dying inside one fixed rectangle, which is what phantom_regions looks
        for. Do not stretch this function to cover them.

    `protected` wins, always. Anyone who crossed the entry line or whose face
    was recognised is a human, and no statistic overrules that.
    """
    canon = canon or {}
    per = defaultdict(list)
    pos = defaultdict(list)
    for _idx, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            w = max(1e-6, float(x2) - float(x1))
            h = max(1e-6, float(y2) - float(y1))
            cid = canon.get(tid, tid)
            per[cid].append((float(t), h / w))
            pos[cid].append(((float(x1) + float(x2)) / 2.0, float(y2)))

    if frame_wh:
        diag = math.hypot(float(frame_wh[0]), float(frame_wh[1]))
    else:
        xs = [p[0] for v in pos.values() for p in v] or [1280.0]
        ys = [p[1] for v in pos.values() for p in v] or [720.0]
        diag = math.hypot(max(xs), max(ys))
    travel_limit = diag * max_travel_frac

    out = {}
    for tid, rows in per.items():
        if tid in protected or len(rows) < min_sightings:
            continue
        rows.sort()
        life = rows[-1][0] - rows[0][0]
        if life < min_life_s:
            continue
        vals = [a for _, a in rows]
        mean = sum(vals) / len(vals)
        if mean <= 0:
            continue
        var = sum((v - mean) ** 2 for v in vals) / len(vals)
        cv = (var ** 0.5) / mean
        if cv > max_aspect_cv:
            continue
        pts = pos.get(tid) or []
        travel = max((math.hypot(p[0] - q[0], p[1] - q[1])
                      for p in pts for q in pts[:1]), default=0.0) if pts else 0.0
        travel = max(travel, math.hypot(pts[-1][0] - pts[0][0],
                                        pts[-1][1] - pts[0][1])) if pts else 0.0
        if travel > travel_limit:
            continue          # rigid but mobile -> a reflection, not furniture
        out[tid] = {"aspect_cv": round(cv, 4), "aspect_mean": round(mean, 3),
                    "life_s": round(life, 1), "sightings": len(rows),
                    "travel_px": round(travel, 1),
                    "why": (f"aspect ratio varied by only {cv*100:.1f}% over "
                            f"{life:.0f}s and it moved {travel:.0f}px — a "
                            f"person walking, turning or sitting cannot hold "
                            f"one shape that long, and furniture does not "
                            f"wander")}
    return out


def mirrored_pair_ids(frame_log, canon=None, protected=(), min_overlap_s=20.0,
                      max_offset_cv=0.10, min_samples=30, min_offset_px=15.0):
    """Track pairs that move in lockstep at a fixed offset — a reflection.

    WHY
        A reception with glass doors or a mirrored wall produces a second body
        walking in step with the first. It is not static, so static_track_ids
        misses it. It deforms exactly like a person, so rigid_track_ids misses
        it. It is person-shaped and person-sized, so the detector is right and
        the size filter passes it. Every existing guard is blind to it, and it
        inflates the headcount by up to 2x on the worst camera angle.

        The reflected-object literature says the giveaway is CONTEXT, not
        appearance — you cannot tell from the pixels. Here the context is
        motion: a real pair of people drift apart constantly, while an object
        and its reflection keep a near-constant separation for as long as both
        are visible, because one is a rigid transform of the other.

    HOW
        For every co-visible pair, measure the separation each frame and take
        its coefficient of variation. Low CV over a long co-visibility means
        the two never moved independently.

    WHAT IT RETURNS
        Candidates, NOT a deletion list — {(a, b): evidence}. Which of the two
        is the reflection needs the zone map (a reflection usually sits inside
        a glass/window dead area), and deleting the wrong one is worse than
        counting both. This flags; a human or a zone rule decides.
    """
    canon = canon or {}
    per = defaultdict(dict)                 # tid -> {t: (cx, foot_y)}
    for _idx, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            per[canon.get(tid, tid)][float(t)] = (
                (float(x1) + float(x2)) / 2.0, float(y2))

    ids = sorted(per, key=str)
    out = {}
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            if a in protected and b in protected:
                continue
            shared = sorted(set(per[a]) & set(per[b]))
            if len(shared) < min_samples:
                continue
            if shared[-1] - shared[0] < min_overlap_s:
                continue
            d = [math.hypot(per[a][t][0] - per[b][t][0],
                            per[a][t][1] - per[b][t][1]) for t in shared]
            mean = sum(d) / len(d)
            if mean < min_offset_px:
                continue                    # same body, duplicate box
            var = sum((x - mean) ** 2 for x in d) / len(d)
            cv = (var ** 0.5) / mean
            if cv <= max_offset_cv:
                out[(a, b)] = {
                    "offset_px": round(mean, 1), "offset_cv": round(cv, 4),
                    "co_visible_s": round(shared[-1] - shared[0], 1),
                    "samples": len(shared),
                    "why": (f"stayed {mean:.0f}px apart (±{cv*100:.1f}%) for "
                            f"{shared[-1]-shared[0]:.0f}s — two independent "
                            f"people drift; an object and its reflection "
                            f"cannot")}
    return out


def protected_ids(crossings=(), face_ids=(), canon=None):
    """Ids the static filter may never touch: anyone who crossed the entry line,
    and anyone whose face was matched. Both are positive evidence of a human."""
    canon = canon or {}
    out = {canon.get(c.get("track_id"), c.get("track_id")) for c in crossings}
    out |= {canon.get(f, f) for f in face_ids}
    # a gallery-matched staff member carries a string name, never a number
    out |= {t for t in out if isinstance(t, str)}
    return {t for t in out if t is not None}


def drop_tracks(events, crossings, frame_log, drop, canon=None):
    """Remove flagged ids from every downstream structure at once, so a phantom
    cannot survive in one place after being removed from another.

    canon: {raw_id: canonical_id}. events/crossings are usually already
    remapped to canonical ids but frame_log never is — without canon a phantom
    whose fragments were Re-ID-merged is removed from every NUMBER and still
    DRAWN in the video (the bug the human review caught)."""
    drop = set(drop)
    canon = canon or {}
    return ([e for e in events if e.get("track_id") not in drop],
            [c for c in crossings if c.get("track_id") not in drop],
            [(fi, t, [b for b in boxes
                      if b[0] not in drop
                      and canon.get(b[0], b[0]) not in drop])
             for fi, t, boxes in frame_log])


def describe(giant_n, static_map):
    lines = []
    if giant_n:
        lines.append(f"D1 dropped {giant_n} detection(s) taller than a person "
                     f"could be at their own footline (the half-frame boxes)")
    if static_map:
        lines.append(f"D2 dropped {len(static_map)} track(s) that never moved "
                     f"and never changed size — furniture, not people:")
        for cid, d in sorted(static_map.items(),
                             key=lambda kv: -kv[1]["seconds"])[:6]:
            lines.append(f"     id {cid} at {d['at']} for {d['seconds']:.0f}s, "
                         f"centre jitter {d['centre_jitter']:.3f} of body height")
    if not lines:
        lines.append("D1/D2 found no phantoms this chunk")
    return "\n".join(lines)
''',
    'drive.py': r'''"""drive.py — pick ONE chunk from the Drive folder and fetch it.

THE BUG THIS IS SHAPED AROUND
    The notebook did this in an order that produced a wrong report:

        1. list the folder
        2. parse the clock from vids[0]              <- the 4:30pm file
        3. DOWNLOAD vids[0]                          <- the 4:30pm file
        4. apply CHUNK_FILTER="7.30.00pm"            <- selects a DIFFERENT file
        5. re-parse the clock from the new pick      <- 19:30
        6. analyse whatever is on disk               <- still the 4:30pm file

    Every step was individually correct. The run analysed 16:30 footage and
    stamped 19:30 on it, and nothing complained, because no single step was
    wrong — only their order.

    Two rules follow, and both are enforced here rather than documented:

        SELECT BEFORE FETCHING. The filter runs on the listing. Nothing is
        downloaded until exactly one file has been chosen.

        THE PATH IS THE ONLY OUTPUT. fetch_chunk returns the file it actually
        downloaded, and the caller derives the clock from THAT path. There is
        no second variable to drift.

MATCHING
    Filenames carry a RANGE:
        "CAM.112 (PP.09_12) 7-28-2026, 4.30.00pm CDT - 7-28-2026, 5.30.00pm CDT"
    A bare substring search for "5.30.00pm" matches this file's END as well as
    the next file's START. Matching is therefore done against the text BEFORE
    the range separator, i.e. the chunk's own start time.
"""
from __future__ import annotations

import re
from pathlib import Path

from .clock import parse_start
from .log import get_logger

_log = get_logger("drive")

SEP = " - "                    # separates start and end in the filename
VIDEO_EXT = (".mp4", ".mkv", ".avi", ".mov")


class DriveError(RuntimeError):
    pass


def start_part(name):
    """The text before the range separator — the chunk's OWN start time."""
    return str(name).split(SEP)[0]


def select(names, chunk_filter=None, index=None):
    """Choose exactly one chunk from a listing. -> the chosen name.

    Raises rather than guessing. A filter that matches several files, or none,
    is a question for a human — silently taking the first match is how the
    wrong hour gets analysed.
    """
    vids = [n for n in names if str(n).lower().endswith(VIDEO_EXT)]
    if not vids:
        raise DriveError(f"no video files in the listing ({len(names)} entries)")
    vids = sorted(vids, key=lambda n: (parse_start(n) or "", str(n)))

    if chunk_filter:
        hits = [n for n in vids if str(chunk_filter) in start_part(n)]
        if not hits:
            raise DriveError(
                f"CHUNK_FILTER {chunk_filter!r} matched none of {len(vids)} "
                f"chunks. Start times available: "
                f"{[str(parse_start(v)) for v in vids[:8]]}")
        if len(hits) > 1:
            raise DriveError(
                f"CHUNK_FILTER {chunk_filter!r} matched {len(hits)} chunks: "
                f"{[Path(h).name for h in hits]}. Narrow it — analysing the "
                f"wrong hour is worse than analysing none.")
        return hits[0]

    if index is not None:
        try:
            return vids[int(index)]
        except IndexError:
            raise DriveError(f"index {index} out of range; {len(vids)} chunks")

    raise DriveError(
        f"{len(vids)} chunks available and no selection given. Pass "
        f"chunk_filter or index. Start times: "
        f"{[str(parse_start(v)) for v in vids[:8]]}")


def list_folder(folder_id, cache=None):
    """-> [names] in a Drive folder, via gdown.

    gdown is imported here, not at module import: the rest of the package must
    stay usable on a machine that has never seen it.
    """
    if cache:
        return list(cache)
    try:
        import gdown
    except ImportError:
        raise DriveError("gdown is not installed — pip install gdown, or pass "
                         "an already-downloaded file with --video")
    try:
        files = gdown.download_folder(id=folder_id, quiet=True,
                                      skip_download=True) or []
    except Exception as e:
        raise DriveError(f"could not list Drive folder {folder_id}: {e}")
    return [getattr(f, "path", str(f)) for f in files]


def fetch_chunk(folder_id, dest_dir, chunk_filter=None, index=None,
                listing=None, entries=None, download=None):
    """Select ONE chunk, then fetch it. -> Path of the downloaded file.

    The returned path is the single source of truth for what was analysed and
    what time it is. Callers must not keep a separate "selected name".

    Drive downloads need the file ID, not the filename — the first version of
    this passed the name straight to gdown, which tried to treat
    "CAM.112 (PP.09_12) 7-28-2026, ....mp4" as a URL. The listing carries both,
    so we keep the pairing instead of throwing the ID away.
    """
    if entries is None and listing is None:
        entries = _folder_entries(folder_id)
    if entries is not None:
        id_of = {n: i for i, n in entries}
        names = [n for _i, n in entries]
    else:
        id_of, names = {}, list(listing)
    chosen = select(names, chunk_filter=chunk_filter, index=index)
    file_id = id_of.get(chosen)
    start = parse_start(chosen)
    _log.info(f"selected chunk: {Path(chosen).name}")
    _log.info(f"  its start time parses to {start}")

    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    target = dest / Path(chosen).name

    if target.exists() and target.stat().st_size > 0:
        _log.info(f"  already on disk ({target.stat().st_size/1e9:.2f} GB) — "
                  f"not re-downloading")
        return target

    if download is None:
        if not file_id:
            raise DriveError(f"no Drive file id for {Path(chosen).name!r} — "
                             f"cannot download it by name alone")
        try:
            import gdown
        except ImportError:
            raise DriveError("gdown is not installed and the file is not on "
                             "disk")

        def download(fid, out):
            gdown.download(id=fid, output=str(out), quiet=False)
    _log.info(f"  downloading -> {target}")
    download(file_id or chosen, target)

    if not target.exists() or target.stat().st_size == 0:
        raise DriveError(f"download produced no file at {target}")
    # The path we return IS the file we analysed. Re-parsing the clock from it
    # is what makes a selected/decoded mismatch impossible rather than merely
    # unlikely.
    if parse_start(target.name) != start:
        raise DriveError(
            f"downloaded file {target.name!r} does not carry the start time of "
            f"the file that was selected ({start}) — refusing to continue")
    return target


ZONE_SUFFIX = ("_zone.json", "zones.json")
IMAGE_EXT = (".jpg", ".jpeg", ".png")


def _folder_entries(folder_id):
    """-> [(file_id, name)] for everything in the folder, videos included."""
    try:
        import gdown
    except ImportError:
        raise DriveError("gdown is not installed")
    try:
        files = gdown.download_folder(id=folder_id, quiet=True,
                                      skip_download=True) or []
    except Exception as e:
        raise DriveError(f"could not list Drive folder {folder_id}: {e}")
    out = []
    for f in files:
        name = getattr(f, "path", str(f))
        out.append((getattr(f, "id", None), name))
    return out


def fetch_assets(folder_id, zones_dir=None, gallery_dir=None, entries=None,
                 download=None):
    """Pull the SMALL companions of the video — zone map and staff photos.

    WHY THIS BELONGS IN THE PIPELINE
        The zone JSON and the staff gallery live in the same Drive folder as
        the footage, and the notebook fetched them. Without them the codebase
        path silently degrades: no zone file means no analysis at all, and no
        staff photo means every staff member is an unnamed low-confidence
        'zone-inferred' row in the report.

        Videos are excluded on purpose. This is for the kilobytes, not the
        gigabytes — the chunk is selected deliberately by fetch_chunk, never
        swept up by an asset sync.

    Existing local files are NOT overwritten: a zone map you have edited beats
    whatever is sitting in Drive.
    """
    got = {"zones": [], "gallery": [], "skipped": []}
    ents = entries if entries is not None else _folder_entries(folder_id)

    if download is None:
        try:
            import gdown
        except ImportError:
            raise DriveError("gdown is not installed")

        def download(file_id, out):
            gdown.download(id=file_id, output=str(out), quiet=True)

    for fid, name in ents:
        base = Path(name).name
        low = base.lower()
        if low.endswith(VIDEO_EXT):
            continue
        if low.endswith(ZONE_SUFFIX) and zones_dir:
            target = Path(zones_dir) / base
            key = "zones"
        elif low.endswith(IMAGE_EXT) and gallery_dir:
            target = Path(gallery_dir) / base
            key = "gallery"
        else:
            got["skipped"].append(base)
            continue
        if target.exists():
            _log.info(f"  {base} already local — keeping yours, not Drive's")
            got[key].append(str(target))
            continue
        if fid is None:
            _log.warning(f"  {base}: no file id from the listing, cannot fetch")
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        _log.info(f"  fetching {base} -> {target}")
        download(fid, target)
        if target.exists():
            got[key].append(str(target))
    _log.info(f"assets: {len(got['zones'])} zone file(s), "
              f"{len(got['gallery'])} gallery image(s), "
              f"{len(got['skipped'])} ignored")
    return got


def describe(names, chunk_filter=None):
    """A human-readable listing, so picking a chunk is not guesswork."""
    vids = [n for n in names if str(n).lower().endswith(VIDEO_EXT)]
    L = [f"{len(vids)} chunk(s) available"]
    for n in sorted(vids, key=lambda x: (parse_start(x) or "", str(x))):
        st = parse_start(n)
        mark = ""
        if chunk_filter and str(chunk_filter) in start_part(n):
            mark = "   <- matches your filter"
        L.append(f"  {str(st):<20} {Path(n).name}{mark}")
    return "\n".join(L)
''',
    'eval_harness.py': r'''"""eval_harness.py — PHASE 2. Turn "we think it improved" into a number.

Standalone and dependency-free (numpy only). Import it from the notebook, or run
it from a shell on exported artifacts. Nothing here touches the pipeline; it
only ever reads predictions and ground truth.

WHY THIS EXISTS
    Every accuracy change from v42 to v55 was judged by watching the video and
    feeling better about it. Thirty reasonable changes, none attributable. This
    file is the gate: from here on, a change ships only if a scored delta says
    it helped.

WHAT IT MEASURES
    HOTA = sqrt(DetA x AssA)   the metric the field reports
      DetA  did we FIND the people          -> detection problem
      AssA  did we KEEP them the same person -> association problem
    These fail independently and need opposite fixes, which is exactly why a
    single accuracy number was never enough to tell us what to do next.
    Also IDF1, MOTA, precision, recall, ID switches.

EVERYTHING IS DEBUGGABLE — the point is never to hand back a bare number:
    * score_sequence() returns per-frame TP/FP/FN and every ID switch with its
      frame and the two ids involved
    * explain() prints the worst frames and what went wrong in each
    * dump_errors_csv() writes every error with its box, to overlay on frames
    * compare() prints an A/B delta between two runs AND the config diff, so a
      change is never confused with noise from a different setting
    * self-validated: test_eval_harness.py checks the metric code against cases
      whose answers are known by construction (perfect tracker, all-swapped,
      half-missed, ...) - a metric you have not tested is not a measurement

HOTA IMPLEMENTATION NOTE (read before quoting the number)
    This is the TrackEval algorithm, reimplemented: global Jaccard alignment
    over the whole sequence, association-aware Hungarian per frame, alpha swept
    0.05..0.95. It is validated against constructed cases with analytically
    known answers. It is NOT the official TrackEval binary. For A/B comparison
    (our main use) any small systematic bias cancels between the two runs. If a
    number ever leaves this project, re-run it through the official TrackEval.
"""
from __future__ import annotations

from .log import get_logger, stage, banner

_log = get_logger("eval_harness")


import csv
import json
import math
from collections import defaultdict
from pathlib import Path

import numpy as np

try:
    from scipy.optimize import linear_sum_assignment as _lsa
    _HAVE_SCIPY = True
except Exception:                                    # pragma: no cover
    _HAVE_SCIPY = False

# Rounded on purpose: np.arange with a float step accumulates error, and the
# alpha==0.5 lookup below (which produces every per-frame debug detail) would
# then silently find nothing.
ALPHAS = np.round(np.arange(0.05, 1.0, 0.05), 2)


# ---------------------------------------------------------------------------
# MOT I/O
# ---------------------------------------------------------------------------
def load_mot(path):
    """MOT 1.1 / MOT16: frame,id,x,y,w,h,conf,...  -> {frame: [(id,x,y,w,h)]}

    Lenient on purpose: CVAT, MOTChallenge and our own exporter all differ
    slightly in trailing columns and in whether conf is present. A ground-truth
    file that silently fails to parse would be the worst possible bug here, so
    parse failures are collected and reported rather than swallowed.
    """
    out, bad = defaultdict(list), []
    p = Path(path)
    for ln, line in enumerate(p.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        f = line.replace("\t", ",").split(",")
        try:
            fr, tid = int(float(f[0])), int(float(f[1]))
            x, y, w, h = (float(v) for v in f[2:6])
        except (ValueError, IndexError):
            bad.append((ln, line[:80]))
            continue
        # MOT gt files use column 7 as a "consider this box" flag; 0 = ignore.
        if len(f) > 6:
            try:
                if float(f[6]) == 0 and len(f) >= 9:
                    continue
            except ValueError:
                pass
        if w <= 0 or h <= 0:
            bad.append((ln, f"non-positive box: {line[:60]}"))
            continue
        out[fr].append((tid, x, y, w, h))
    if bad:
        _log.error(f"  !! {p.name}: {len(bad)} unparseable line(s), first few:")
        for ln, txt in bad[:3]:
            _log.info(f"     line {ln}: {txt}")
    return dict(out)


def write_mot(path, rows):
    """rows: iterable of (frame, id, x, y, w, h)."""
    Path(path).write_text("\n".join(
        f"{int(fr)},{int(tid)},{x:.2f},{y:.2f},{w:.2f},{h:.2f},1,-1,-1,-1"
        for fr, tid, x, y, w, h in rows))
    return Path(path)


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------
def iou_matrix(a, b):
    """a, b: lists of (x, y, w, h) -> IoU matrix. Empty-safe."""
    if not len(a) or not len(b):
        return np.zeros((len(a), len(b)), dtype=float)
    A = np.asarray(a, dtype=float)
    B = np.asarray(b, dtype=float)
    ax1, ay1 = A[:, 0][:, None], A[:, 1][:, None]
    ax2, ay2 = ax1 + A[:, 2][:, None], ay1 + A[:, 3][:, None]
    bx1, by1 = B[:, 0][None, :], B[:, 1][None, :]
    bx2, by2 = bx1 + B[:, 2][None, :], by1 + B[:, 3][None, :]
    iw = np.clip(np.minimum(ax2, bx2) - np.maximum(ax1, bx1), 0, None)
    ih = np.clip(np.minimum(ay2, by2) - np.maximum(ay1, by1), 0, None)
    inter = iw * ih
    union = (A[:, 2] * A[:, 3])[:, None] + (B[:, 2] * B[:, 3])[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)


def _hungarian(cost):
    """Maximise-free wrapper. scipy if present, else a small O(n^3) fallback so
    the harness never silently declines to score."""
    if _HAVE_SCIPY:
        r, c = _lsa(cost)
        return list(r), list(c)
    n, m = cost.shape                       # pragma: no cover - fallback path
    size = max(n, m)
    C = np.full((size, size), cost.max() + 1.0 if cost.size else 1.0)
    C[:n, :m] = cost
    used_c, rows, cols = set(), [], []
    order = np.argsort(C.min(axis=1))
    for i in order:                          # greedy fallback, documented as such
        j = int(np.argmin(np.where([k in used_c for k in range(size)],
                                   np.inf, C[i])))
        if j < m and i < n:
            rows.append(int(i)); cols.append(j)
        used_c.add(j)
    return rows, cols


# ---------------------------------------------------------------------------
# the metric
# ---------------------------------------------------------------------------
def score_sequence(gt, pr, alphas=ALPHAS, keep_detail=True):
    """HOTA / DetA / AssA / IDF1 / MOTA + everything needed to debug them.

    gt, pr: {frame: [(id, x, y, w, h)]}
    """
    frames = sorted(set(gt) | set(pr))
    gt_count = defaultdict(int)      # gt_id -> frames present
    pr_count = defaultdict(int)
    pot = defaultdict(float)         # (gt_id, pr_id) -> summed similarity

    # pass 1 — global alignment. HOTA matches with knowledge of how often two
    # ids co-occur across the WHOLE sequence, which is what stops a one-frame
    # coincidence from being treated like a real identity link.
    sims = {}
    for f in frames:
        g, p = gt.get(f, []), pr.get(f, [])
        for tid, *_ in g:
            gt_count[tid] += 1
        for tid, *_ in p:
            pr_count[tid] += 1
        S = iou_matrix([b[1:] for b in g], [b[1:] for b in p])
        sims[f] = S
        for i, (gi, *_) in enumerate(g):
            for j, (pj, *_) in enumerate(p):
                if S[i, j] > 0:
                    pot[(gi, pj)] += S[i, j]

    align = {}
    for (gi, pj), s in pot.items():
        denom = gt_count[gi] + pr_count[pj] - s
        align[(gi, pj)] = s / denom if denom > 0 else 0.0

    res = {"per_alpha": [], "frames": len(frames)}
    detail_at_half = None

    for a in alphas:
        TP = FP = FN = 0
        tpa = defaultdict(int)                  # (gt,pr) -> matched frames
        matched_gt = defaultdict(int)
        matched_pr = defaultdict(int)
        per_frame, last_match, idsw_list = [], {}, []
        for f in frames:
            g, p = gt.get(f, []), pr.get(f, [])
            S = sims[f]
            pairs = []
            if len(g) and len(p):
                # association-aware score, exactly as HOTA specifies
                score = np.zeros_like(S)
                for i, (gi, *_) in enumerate(g):
                    for j, (pj, *_) in enumerate(p):
                        score[i, j] = S[i, j] * align.get((gi, pj), 0.0)
                ri, ci = _hungarian(-score)
                for i, j in zip(ri, ci):
                    if i < len(g) and j < len(p) and S[i, j] >= a:
                        pairs.append((i, j))
            f_tp = len(pairs)
            f_fn = len(g) - f_tp
            f_fp = len(p) - f_tp
            TP += f_tp; FN += f_fn; FP += f_fp
            for i, j in pairs:
                gi, pj = g[i][0], p[j][0]
                tpa[(gi, pj)] += 1
                matched_gt[gi] += 1
                matched_pr[pj] += 1
                if gi in last_match and last_match[gi] != pj:
                    idsw_list.append({"frame": f, "gt_id": gi,
                                      "was": last_match[gi], "now": pj})
                last_match[gi] = pj
            if keep_detail:
                per_frame.append({"frame": f, "tp": f_tp, "fp": f_fp, "fn": f_fn,
                                  "n_gt": len(g), "n_pr": len(p)})

        det_a = TP / (TP + FN + FP) if (TP + FN + FP) else 0.0
        ass_sum = 0.0
        for (gi, pj), c in tpa.items():
            fna = gt_count[gi] - c
            fpa = pr_count[pj] - c
            ass_sum += c * (c / (c + fna + fpa)) if (c + fna + fpa) else 0.0
        ass_a = ass_sum / TP if TP else 0.0
        res["per_alpha"].append({"alpha": round(float(a), 2), "DetA": det_a,
                                 "AssA": ass_a, "HOTA": math.sqrt(det_a * ass_a),
                                 "TP": TP, "FP": FP, "FN": FN,
                                 "IDSW": len(idsw_list)})
        if abs(a - 0.5) < 1e-6:
            detail_at_half = {"per_frame": per_frame, "idsw": idsw_list,
                              "TP": TP, "FP": FP, "FN": FN, "tpa": dict(tpa)}

    res["HOTA"] = float(np.mean([r["HOTA"] for r in res["per_alpha"]]))
    res["DetA"] = float(np.mean([r["DetA"] for r in res["per_alpha"]]))
    res["AssA"] = float(np.mean([r["AssA"] for r in res["per_alpha"]]))

    # IDF1 / MOTA at the conventional alpha=0.5
    h = next(r for r in res["per_alpha"] if abs(r["alpha"] - 0.5) < 1e-6)
    n_gt = sum(gt_count.values())
    res["precision"] = h["TP"] / (h["TP"] + h["FP"]) if (h["TP"] + h["FP"]) else 0.0
    res["recall"] = h["TP"] / (h["TP"] + h["FN"]) if (h["TP"] + h["FN"]) else 0.0
    res["ID_switches"] = h["IDSW"]
    res["MOTA"] = 1.0 - (h["FN"] + h["FP"] + h["IDSW"]) / n_gt if n_gt else 0.0
    res["TP"], res["FP"], res["FN"] = h["TP"], h["FP"], h["FN"]

    # IDF1 = identity F1 over the best global id-to-id assignment
    if tpa:
        gids = sorted(gt_count); pids = sorted(pr_count)
        M = np.zeros((len(gids), len(pids)))
        for (gi, pj), c in detail_at_half["tpa"].items():
            M[gids.index(gi), pids.index(pj)] = c
        ri, ci = _hungarian(-M)
        idtp = sum(M[i, j] for i, j in zip(ri, ci)
                   if i < len(gids) and j < len(pids))
        idfn = n_gt - idtp
        idfp = sum(pr_count.values()) - idtp
        res["IDF1"] = 2 * idtp / (2 * idtp + idfn + idfp) if idtp else 0.0
    else:
        res["IDF1"] = 0.0

    res["n_gt_boxes"] = n_gt
    res["n_pr_boxes"] = sum(pr_count.values())
    res["n_gt_ids"] = len(gt_count)
    res["n_pr_ids"] = len(pr_count)
    res["_detail"] = detail_at_half
    return res


# ---------------------------------------------------------------------------
# debugging surface — a bare number is never the deliverable
# ---------------------------------------------------------------------------
def explain(res, label="", worst_n=8):
    """Print the number AND where it came from. Reads the diagnosis out loud so
    the next action is obvious instead of a guess."""
    bar = "=" * 74
    _log.info(bar)
    _log.info(f"  SCORE{(' · ' + label) if label else ''}")
    _log.info(bar)
    _log.info(f"    HOTA   {res['HOTA']:.4f}   = sqrt(DetA x AssA)")
    _log.info(f"    DetA   {res['DetA']:.4f}   did we FIND the people")
    _log.info(f"    AssA   {res['AssA']:.4f}   did we KEEP them the same person")
    _log.info(f"    IDF1   {res['IDF1']:.4f}   MOTA {res['MOTA']:.4f}")
    _log.info(f"    prec   {res['precision']:.4f}   recall {res['recall']:.4f}")
    _log.info(f"    TP {res['TP']}  FP {res['FP']}  FN {res['FN']}  "
          f"ID switches {res['ID_switches']}")
    _log.info(f"    gt: {res['n_gt_boxes']} boxes / {res['n_gt_ids']} people   "
          f"pred: {res['n_pr_boxes']} boxes / {res['n_pr_ids']} identities")

    # the actionable part
    d, a = res["DetA"], res["AssA"]
    _log.info("    " + "-" * 68)
    if d < 0.4:
        _log.info("    -> DETECTION is the bottleneck. Raise resolution / lower the conf")
        _log.info("       floor / use SAHI. Tracker work will not help yet.")
    elif a < d - 0.12:
        _log.info("    -> ASSOCIATION is the bottleneck. Raise fps, fix the re-id gate,")
        _log.info("       upgrade the tracker. More detections will not help.")
    elif d < 0.6:
        _log.info("    -> Both are mediocre; detection is still the cheaper win.")
    else:
        _log.info("    -> Balanced. Further gains need a better detector AND tracker.")
    if res["n_pr_ids"] > res["n_gt_ids"] * 1.5:
        _log.info(f"    -> {res['n_pr_ids']} predicted identities for {res['n_gt_ids']} real "
              f"people: FRAGMENTATION. Every unique-people count is inflated.")
    elif res["n_pr_ids"] * 1.5 < res["n_gt_ids"]:
        _log.info(f"    -> only {res['n_pr_ids']} identities for {res['n_gt_ids']} real people: "
              f"OVER-MERGING. Counts are deflated and journeys are fiction.")

    det = res.get("_detail") or {}
    pf = sorted(det.get("per_frame", []), key=lambda r: -(r["fp"] + r["fn"]))[:worst_n]
    if pf:
        _log.info("    " + "-" * 68)
        _log.info(f"    worst frames (alpha=0.5)   frame   gt  pred   TP  FP  FN")
        for r in pf:
            if r["fp"] + r["fn"] == 0:
                continue
            _log.info(f"    {'':26s}{r['frame']:>6d}{r['n_gt']:>5d}{r['n_pr']:>6d}"
                  f"{r['tp']:>5d}{r['fp']:>4d}{r['fn']:>4d}")
    sw = det.get("idsw", [])[:worst_n]
    if sw:
        _log.info(f"    ID switches (first {len(sw)}):")
        for s in sw:
            _log.info(f"      frame {s['frame']}: real person {s['gt_id']} was "
                  f"id {s['was']}, became id {s['now']}")
    _log.info(bar)
    return res


def dump_errors_csv(gt, pr, res, path, alpha=0.5):
    """Every FP and FN with its box, so they can be drawn on the frames. This is
    what turns 'recall 0.62' into 'we are missing the far-left table'."""
    rows = []
    for f in sorted(set(gt) | set(pr)):
        g, p = gt.get(f, []), pr.get(f, [])
        S = iou_matrix([b[1:] for b in g], [b[1:] for b in p])
        gm, pm = set(), set()
        if len(g) and len(p):
            ri, ci = _hungarian(-S)
            for i, j in zip(ri, ci):
                if i < len(g) and j < len(p) and S[i, j] >= alpha:
                    gm.add(i); pm.add(j)
        for i, b in enumerate(g):
            if i not in gm:
                rows.append([f, "FN", b[0], *[round(v, 1) for v in b[1:]]])
        for j, b in enumerate(p):
            if j not in pm:
                rows.append([f, "FP", b[0], *[round(v, 1) for v in b[1:]]])
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["frame", "kind", "id", "x", "y", "w", "h"])
        w.writerows(rows)
    _log.info(f"  -> {Path(path).name}  ({len(rows)} errors: overlay these on the frames)")
    return Path(path)


def save_baseline(res, path, config=None, label=""):
    """Freeze a score so the NEXT change has something to be compared against."""
    payload = {"label": label,
               "metrics": {k: v for k, v in res.items() if not k.startswith("_")},
               "config": config or {}}
    Path(path).write_text(json.dumps(payload, indent=2, default=str))
    _log.info(f"  -> baseline saved: {Path(path).name}")
    return Path(path)


def compare(before, after, label_a="before", label_b="after"):
    """A/B delta AND the config diff. Without the config diff a delta is not
    attributable - that is the exact trap CALIBRATION_AUTO_APPLY put us in."""
    ma = before.get("metrics", before)
    mb = after.get("metrics", after)
    ca = before.get("config", {}) or {}
    cb = after.get("config", {}) or {}
    _log.info("=" * 74)
    _log.info(f"  A/B   {label_a}  ->  {label_b}")
    _log.info("=" * 74)
    _log.info(f"    {'metric':14s}{label_a:>12s}{label_b:>12s}{'delta':>12s}")
    verdicts = []
    for k in ("HOTA", "DetA", "AssA", "IDF1", "MOTA", "precision", "recall"):
        if k not in ma or k not in mb:
            continue
        d = mb[k] - ma[k]
        flag = "  ++" if d > 0.01 else ("  --" if d < -0.01 else "    ")
        _log.info(f"    {k:14s}{ma[k]:>12.4f}{mb[k]:>12.4f}{d:>+12.4f}{flag}")
        verdicts.append((k, d))
    for k in ("ID_switches", "FP", "FN", "n_pr_ids"):
        if k in ma and k in mb:
            _log.info(f"    {k:14s}{ma[k]:>12d}{mb[k]:>12d}{mb[k]-ma[k]:>+12d}"
                  + ("  ++" if mb[k] < ma[k] else ("  --" if mb[k] > ma[k] else "")))
    changed = {k: (ca.get(k), cb.get(k)) for k in set(ca) | set(cb)
               if ca.get(k) != cb.get(k)}
    _log.info("    " + "-" * 68)
    if changed:
        _log.info(f"    config changed ({len(changed)}):")
        for k, (x, y) in sorted(changed.items()):
            _log.info(f"      {k}: {x}  ->  {y}")
    else:
        _log.info("    config identical — any delta here is run-to-run NOISE, not a fix.")
    dh = mb.get("HOTA", 0) - ma.get("HOTA", 0)
    _log.info("    " + "-" * 68)
    if abs(dh) < 0.005:
        _log.info(f"    VERDICT: HOTA moved {dh:+.4f} — inside noise. Not a win, do not ship.")
    elif dh > 0:
        _log.info(f"    VERDICT: HOTA {dh:+.4f}. Real improvement. Keep it.")
    else:
        _log.info(f"    VERDICT: HOTA {dh:+.4f}. This made it WORSE. Revert.")
    _log.info("=" * 74)
    return {"delta_HOTA": dh, "config_changed": changed}


# ---------------------------------------------------------------------------
# per-condition scoring — day / night / IR-switch must be scored SEPARATELY
# ---------------------------------------------------------------------------
def score_conditions(pairs, out_dir=None):
    """pairs: {condition_name: (gt_path, pred_path)}

    One aggregate number hides the thing we most need to see. Our night path
    disables colour evidence entirely and has never been measured; averaged in
    with daylight it would look fine while being broken.
    """
    results = {}
    for name, (gtp, prp) in pairs.items():
        gt, pr = load_mot(gtp), load_mot(prp)
        if not gt:
            _log.error(f"  !! {name}: ground truth is empty ({gtp}) — SKIPPED")
            continue
        res = explain(score_sequence(gt, pr), label=name)
        results[name] = res
        if out_dir:
            out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
            dump_errors_csv(gt, pr, res, out / f"{name}_errors.csv")
            save_baseline(res, out / f"{name}_score.json", label=name)
    if len(results) > 1:
        _log.info("=" * 74)
        _log.info("  ACROSS CONDITIONS  (a single average would have hidden this)")
        _log.info("=" * 74)
        _log.info(f"    {'condition':22s}{'HOTA':>9s}{'DetA':>9s}{'AssA':>9s}{'IDF1':>9s}")
        for n, r in results.items():
            _log.info(f"    {n:22s}{r['HOTA']:>9.4f}{r['DetA']:>9.4f}"
                  f"{r['AssA']:>9.4f}{r['IDF1']:>9.4f}")
        worst = min(results.items(), key=lambda kv: kv[1]["HOTA"])
        best = max(results.items(), key=lambda kv: kv[1]["HOTA"])
        gap = best[1]["HOTA"] - worst[1]["HOTA"]
        _log.info("    " + "-" * 68)
        _log.info(f"    weakest condition: {worst[0]} (HOTA {worst[1]['HOTA']:.4f}), "
              f"{gap:.4f} below {best[0]}")
        if gap > 0.10:
            _log.info(f"    -> that gap is where the error lives. Fix {worst[0]} before "
                  f"tuning anything that is already working.")
    return results


if __name__ == "__main__":                            # pragma: no cover
    import sys
    if len(sys.argv) >= 3:
        explain(score_sequence(load_mot(sys.argv[1]), load_mot(sys.argv[2])),
                label=f"{Path(sys.argv[1]).name} vs {Path(sys.argv[2]).name}")
    else:
        _log.info(__doc__)
        _log.info("usage: python eval_harness.py <gt.txt> <predictions.txt>")
''',
    'geometry_calibration.py': r'''"""calibration.py — Phase B P6 Automated Camera Calibration & Geometry Consistency.

WHY THIS EXISTS
    Auto perspective fit in CAM.112 dropped 46% of detections or produced implausible camera heights
    when seated/crouching people or static phantoms contaminated the regression line h(y) = a*y + b.

THE PRINCIPLE
    1. Robust RANSAC Ground-Plane Estimator: Filters out posture anomalies (seated, crouching, giant phantoms).
    2. GeoCalib / Intrinsic Refinement: Validates implied focal length and tilt angle against physical constraints.
    3. Ground Plane Consistency Auditor: Verifies near vs far scale ratios to prevent distorted metric measurements.
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

from .ground_plane import GroundPlane, PERSON_H_M


def fit_robust_ground_plane(detections: List[Tuple[float, float, float, float]],
                            frame_size: Tuple[int, int],
                            person_h: float = PERSON_H_M) -> GroundPlane:
    """Robustly fit ground plane line h(y) = a*y + b using RANSAC to exclude seated/crouching detections.
    
    detections: List of (x1, y1, x2, y2) bounding boxes.
    frame_size: (width, height)
    """
    fw, fh = frame_size
    
    # Filter valid standing aspect ratios (~2.0 to 4.5)
    valid_pts = []
    for x1, y1, x2, y2 in detections:
        w, h = float(x2 - x1), float(y2 - y1)
        if w <= 0 or h <= 0:
            continue
        aspect = h / w
        if 2.0 <= aspect <= 4.8 and y2 > fh * 0.15:  # Below horizon region
            valid_pts.append((y2, h))

    if len(valid_pts) < 10:
        return GroundPlane.none("Insufficient valid standing person detections for robust calibration.")

    ys = np.array([p[0] for p in valid_pts], dtype=np.float64)
    hs = np.array([p[1] for p in valid_pts], dtype=np.float64)

    # RANSAC fitting with deterministic seed for reproducibility
    best_a, best_b = None, None
    max_inliers = 0
    inlier_threshold_px = 25.0

    n_samples = len(ys)
    iterations = min(200, n_samples * 5)
    rng = np.random.RandomState(seed=42)

    for _ in range(iterations):
        idx = rng.choice(n_samples, 2, replace=False)
        y_samp, h_samp = ys[idx], hs[idx]
        
        dy = y_samp[1] - y_samp[0]
        if abs(dy) < 10:
            continue
            
        a = (h_samp[1] - h_samp[0]) / dy
        b = h_samp[0] - a * y_samp[0]

        if a <= 1e-6:  # Unphysical: height must increase with lower image row y
            continue

        pred_h = a * ys + b
        residuals = np.abs(hs - pred_h)
        inliers = np.sum(residuals < inlier_threshold_px)

        if inliers > max_inliers:
            max_inliers = inliers
            best_a, best_b = a, b

    if best_a is None or best_a <= 1e-6:
        return GroundPlane.none("RANSAC ground plane fit failed to find valid physical parameters (a <= 0).")

    gp = GroundPlane.from_perspective(
        a=best_a,
        b=best_b,
        frame_w=fw,
        frame_h=fh,
        person_h=person_h
    )

    return gp
''',
    'graph_fusion.py': r'''"""graph_fusion.py — P2 Graph-Based Identity Fusion (Min-Cost Flow / Global Optimization).

WHY THIS EXISTS
    The 6-tier Re-ID cascade (CLIP -> Anchor -> HSV -> Handoff -> Stationary -> Face) evaluates
    evidence in rigid sequential order ("first-tier-wins"). When CLIP and HSV disagree 
    (10-20% of cases), greedy heuristics pick the wrong tier and lock in false merges.

THE PRINCIPLE
    Joint Multi-Evidence Graph Optimization.
    1. Construct a global association graph where nodes are tracklets.
    2. Compute pairwise transition costs using ALL weighted evidence channels simultaneously:
       - Deep appearance distance (CLIP / FastReID)
       - Color histogram similarity (HSV)
       - Spatial & motion feasibility (GroundPlane floor distance & walk speed)
       - Face recognition biometrics
       - Temporal gap / overlap constraints
    3. Solve global minimum-cost assignment using SciPy Linear Sum Assignment (Hungarian / Min-Cost Flow).
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np

try:
    from scipy.optimize import linear_sum_assignment
    _HAVE_SCIPY = True
except ImportError:
    _HAVE_SCIPY = False


class FusionWeights:
    """Configurable weights for the multi-evidence identity graph edge costs."""
    def __init__(self,
                 w_reid: float = 0.45,
                 w_hsv: float = 0.15,
                 w_spatial: float = 0.25,
                 w_face: float = 0.15,
                 max_walk_speed_mps: float = 2.5,
                 max_temporal_gap_s: float = 300.0,
                 reid_threshold: float = 0.35):
        self.w_reid = w_reid
        self.w_hsv = w_hsv
        self.w_spatial = w_spatial
        self.w_face = w_face
        self.max_walk_speed_mps = max_walk_speed_mps
        self.max_temporal_gap_s = max_temporal_gap_s
        self.reid_threshold = reid_threshold


def compute_pairwise_cost(track_a: Dict[str, Any],
                          track_b: Dict[str, Any],
                          weights: FusionWeights,
                          ground_plane: Optional[Any] = None) -> float:
    """Compute total fusion edge cost between track_a and track_b (lower = more likely same person).
    Returns float('inf') if physically impossible (e.g. temporal overlap or unphysical speed).
    """
    # 1. Temporal feasibility check (cannot overlap in time)
    t_a_span = track_a.get("t_span")
    t_b_span = track_b.get("t_span")
    if t_a_span is None or t_b_span is None:
        return float('inf')
    t_a_start, t_a_end = t_a_span
    t_b_start, t_b_end = t_b_span

    # Overlap check with 1s tolerance
    if max(t_a_start, t_b_start) < min(t_a_end, t_b_end) - 1.0:
        return float('inf')

    # Determine direction (A before B or B before A)
    if t_a_end <= t_b_start:
        t_gap = t_b_start - t_a_end
        pos_exit = track_a.get("exit_pos", (0, 0))
        pos_entry = track_b.get("entry_pos", (0, 0))
    else:
        t_gap = t_a_start - t_b_end
        pos_exit = track_b.get("exit_pos", (0, 0))
        pos_entry = track_a.get("entry_pos", (0, 0))

    if t_gap > weights.max_temporal_gap_s:
        return float('inf')

    # 2. Motion / Speed Feasibility Check
    if ground_plane and getattr(ground_plane, "ok", False):
        dist_m = ground_plane.dist_m(pos_exit, pos_entry)
        if dist_m is not None:
            speed = dist_m / max(t_gap, 0.5)
            if speed > weights.max_walk_speed_mps:
                return float('inf')  # Faster than humanly possible
            spatial_cost = min(1.0, dist_m / 10.0)
        else:
            spatial_cost = 0.5
    else:
        # Fallback to pixel distance normalized
        px_dist = math.hypot(pos_exit[0] - pos_entry[0], pos_exit[1] - pos_entry[1])
        spatial_cost = min(1.0, px_dist / 800.0)

    # 3. Deep Re-ID Embedding Distance (Cosine)
    emb_a = track_a.get("reid_emb")
    emb_b = track_b.get("reid_emb")
    if emb_a is not None and emb_b is not None:
        dot = np.dot(emb_a, emb_b)
        norm = (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))
        sim = dot / max(norm, 1e-9)
        reid_dist = max(0.0, 1.0 - float(sim))
    else:
        reid_dist = 0.5

    # 4. Color HSV Histogram Distance
    hsv_a = track_a.get("hsv_hist")
    hsv_b = track_b.get("hsv_hist")
    if hsv_a is not None and hsv_b is not None:
        # Bhattacharyya / correlation distance
        hsv_dist = max(0.0, 1.0 - float(np.sum(np.minimum(hsv_a, hsv_b))))
    else:
        hsv_dist = 0.5

    # 5. Face Biometric Match
    face_a = track_a.get("face_id")
    face_b = track_b.get("face_id")
    if face_a and face_b:
        if face_a == face_b:
            face_dist = 0.0  # Strong positive
        else:
            return float('inf')  # Contradictory faces -> impossible merge
    else:
        face_dist = 0.5

    # Weighted composite cost calculation
    total_cost = (
        weights.w_reid * reid_dist +
        weights.w_hsv * hsv_dist +
        weights.w_spatial * spatial_cost +
        weights.w_face * face_dist
    )

    return total_cost


def solve_graph_fusion(tracklets: List[Dict[str, Any]],
                       weights: Optional[FusionWeights] = None,
                       ground_plane: Optional[Any] = None,
                       cost_cutoff: float = 0.45) -> Dict[int, int]:
    """Solve global tracklet identity association graph using Hungarian optimization.
    
    tracklets: List of dicts, each containing:
               - id: int
               - t_span: (t_start, t_end)
               - entry_pos: (x, y)
               - exit_pos: (x, y)
               - reid_emb: np.ndarray (optional)
               - hsv_hist: np.ndarray (optional)
               - face_id: str/int (optional)
    
    Returns: Mapping {raw_track_id: canonical_track_id}
    """
    if not tracklets:
        return {}

    if weights is None:
        weights = FusionWeights()

    n = len(tracklets)
    raw_ids = [t["id"] for t in tracklets]
    
    # Cost Matrix Construction
    cost_matrix = np.full((n, n), fill_value=np.inf)

    for i in range(n):
        for j in range(i + 1, n):
            c = compute_pairwise_cost(tracklets[i], tracklets[j], weights, ground_plane)
            if c <= cost_cutoff:
                cost_matrix[i, j] = c

    # Linear Sum Assignment / Hungarian matching
    canon_map: Dict[int, int] = {tid: tid for tid in raw_ids}

    # Find valid candidate edges
    valid_pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            if cost_matrix[i, j] < cost_cutoff:
                valid_pairs.append((cost_matrix[i, j], i, j))

    # Sort edges by cost and merge disjoint sets using Union-Find
    valid_pairs.sort(key=lambda x: x[0])

    parent = {tid: tid for tid in raw_ids}

    def find(i):
        """Iterative path-compression find — safe for >1000 tracklets."""
        root = i
        while parent[root] != root:
            root = parent[root]
        # Path compression
        while parent[i] != root:
            parent[i], i = root, parent[i]
        return root

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    for cost, i, j in valid_pairs:
        id_i = raw_ids[i]
        id_j = raw_ids[j]
        if find(id_i) != find(id_j):
            union(id_i, id_j)

    for tid in raw_ids:
        canon_map[tid] = find(tid)

    return canon_map
''',
    'ground_plane.py': r'''"""ground_plane.py — PHASE 3. Stop measuring the world in pixels.

THE PROBLEM
    Every spatial threshold in this pipeline is a single pixel number applied to
    a frame where perspective varies about 5x from the near corner to the far
    one:

        LIVE_REID_MAX_DIST_PX   MAX_PLAUSIBLE_SPEED_PX   REID_HANDOFF_PX
        GREET_PROXIMITY_PX      SHADOW_PX                CARRIED_* geometry

    One number cannot be right in both halves of the image. It is simultaneously
    too loose near the camera (merging different people) and too tight far away
    (failing to merge the same person). Every one of those thresholds has been
    hand-tuned against that contradiction for 14 versions.

THE FIX
    Convert image points to positions on the FLOOR, in metres. Then one
    threshold — "a person cannot walk faster than 2 m/s" — is correct
    everywhere, because it is a statement about the world rather than about
    pixels.

TWO WAYS TO GET THERE, and the cheap one needs nothing from you
    AUTO   (default, zero configuration)
        A vertical object of constant real height projects to a pixel height
        that is a LINEAR function of its foot's image row. That is exact for any
        camera pose viewing a plane, and Phase 1 already fits that line from the
        run's own isolated detections (_PerspectiveModel). From it we recover an
        approximate metric ground mapping with no clicks and no measurements.
        Assumes: flat floor, fixed camera, little roll, people standing.
    EXACT  (four clicks, when precision matters)
        Put four image points and their real-world floor coordinates in the
        zones JSON as "ground_points". cv2.findHomography then gives a true
        plane mapping with no assumptions at all. This is also the exact input
        NVIDIA-style multi-camera 3D tracking needs later, so it is not
        throwaway work.

HONESTY ABOUT THE AUTO MODE
    It assumes a level camera. A real camera is tilted down, which stretches the
    depth axis. test_ground_plane.py measures that error against a synthetic
    tilted camera instead of hand-waving it: lateral distance stays accurate,
    depth degrades gracefully with tilt, and BOTH are far better than pixels.
    describe() reports the implied camera height so an implausible value (a bad
    fit, or heavy tilt) is visible rather than silently wrong.
"""
from __future__ import annotations

import math

import numpy as np

try:
    import cv2
except Exception:                                   # pragma: no cover
    cv2 = None

PERSON_H_M = 1.7          # median adult standing height, the reference ruler
DEFAULT_HFOV_DEG = 82.0   # typical wide fixed CCTV lens; only affects the DEPTH
                          # axis, and only in AUTO mode. Override in config if
                          # the real lens is known.


class GroundPlane:
    """Image pixels -> floor coordinates in metres.

    X is lateral (metres right of the optical centre), Z is depth (metres from
    the camera). Both measured on the floor, so distance between two people is
    the distance they would pace out, not the distance between their boxes.
    """

    def __init__(self, mode, *, a=None, b=None, cx=None, focal_px=None,
                 person_h=PERSON_H_M, H=None, frame_size=None, note=""):
        self.mode = mode                 # "auto" | "exact" | "none"
        self.a, self.b = a, b
        self.cx, self.focal_px = cx, focal_px
        self.person_h = person_h
        self.H = H                       # 3x3 homography, exact mode
        self.frame_size = frame_size
        self.note = note

    # -- constructors --------------------------------------------------------
    @classmethod
    def none(cls, why="no scene geometry available"):
        return cls("none", note=why)

    @classmethod
    def from_perspective(cls, a, b, frame_w, frame_h, person_h=PERSON_H_M,
                         hfov_deg=DEFAULT_HFOV_DEG, focal_px=None):
        """Build from the fitted line  h(y) = a*y + b  (pixel height of a
        standing person whose feet are at image row y).

        a <= 0 would mean people get SHORTER as they come closer, which is not a
        camera looking at a floor — it is a broken fit, and we refuse it rather
        than produce confident nonsense from it.
        """
        if a is None or a <= 1e-6:
            return cls.none(f"perspective fit unusable (a={a}) — not a floor view")
        if focal_px is None:
            focal_px = (frame_w / 2.0) / math.tan(math.radians(hfov_deg) / 2.0)
        return cls("auto", a=float(a), b=float(b), cx=frame_w / 2.0,
                   focal_px=float(focal_px), person_h=person_h,
                   frame_size=(frame_w, frame_h),
                   note=f"auto from perspective fit (hfov~{hfov_deg:.0f} deg)")

    @classmethod
    def from_correspondences(cls, img_pts, world_pts, frame_size=None):
        """Exact: >=4 image points and their real floor coordinates in metres."""
        if cv2 is None:
            return cls.none("cv2 unavailable")
        img = np.asarray(img_pts, dtype=np.float32).reshape(-1, 1, 2)
        wld = np.asarray(world_pts, dtype=np.float32).reshape(-1, 1, 2)
        if len(img) < 4 or len(img) != len(wld):
            return cls.none(f"need >=4 matched points, got {len(img)}/{len(wld)}")
        H, _ = cv2.findHomography(img, wld, method=0)
        if H is None:
            return cls.none("findHomography failed — are the 4 points collinear?")
        return cls("exact", H=np.asarray(H, dtype=float), frame_size=frame_size,
                   note=f"exact homography from {len(img)} correspondences")

    @classmethod
    def from_zone_config(cls, cfg, frame_size, persp=None, **kw):
        """Prefer an exact homography if the zones JSON carries one:

            "ground_points": [
              {"image": [x, y], "world": [X, Z]},   x4 or more, metres
              ...
            ]

        Otherwise fall back to the automatic fit. This is the only place the two
        modes are chosen between, so the rest of the pipeline never cares which
        one it got.
        """
        gp = (cfg or {}).get("ground_points") or []
        if len(gp) >= 4:
            try:
                return cls.from_correspondences(
                    [p["image"] for p in gp], [p["world"] for p in gp], frame_size)
            except (KeyError, TypeError) as e:
                pass
        if persp is not None:
            fit = persp._refit() if hasattr(persp, "_refit") else None
            if fit:
                return cls.from_perspective(fit[0], fit[1], frame_size[0],
                                            frame_size[1], **kw)
        return cls.none("no ground_points in zones JSON and no usable perspective fit")

    # -- the mapping ---------------------------------------------------------
    @property
    def ok(self):
        return self.mode != "none"

    def expected_h(self, y):
        """Pixel height of a standing person with feet at row y (auto mode)."""
        if self.mode != "auto":
            return None
        h = self.a * float(y) + self.b
        return h if h > 1e-6 else None

    def to_ground(self, x, y):
        """Image point (x, y) — the FEET — to floor (X, Z) in metres.

        Returns None above the horizon, where the floor is not visible and any
        answer would be fabricated.
        """
        if self.mode == "exact":
            v = self.H @ np.array([float(x), float(y), 1.0])
            if abs(v[2]) < 1e-12:
                return None
            return (float(v[0] / v[2]), float(v[1] / v[2]))
        if self.mode == "auto":
            h = self.expected_h(y)
            if h is None:
                return None
            #  h = f*Hp/Z   =>   Z = f*Hp/h
            #  X = (x-cx)*Z/f = (x-cx)*Hp/h
            return (float((float(x) - self.cx) * self.person_h / h),
                    float(self.focal_px * self.person_h / h))
        return None

    def scale_at(self, y):
        """Metres per pixel at image row y. The number that was implicitly
        assumed constant by every px threshold in the pipeline."""
        if self.mode == "auto":
            h = self.expected_h(y)
            return None if h is None else self.person_h / h
        if self.mode == "exact":
            p0, p1 = self.to_ground(0.0, y), self.to_ground(1.0, y)
            if p0 is None or p1 is None:
                return None
            return math.hypot(p1[0] - p0[0], p1[1] - p0[1])
        return None

    def dist_m(self, p, q):
        """Floor distance in metres between two FOOT points. None if either is
        above the horizon."""
        a, b = self.to_ground(*p), self.to_ground(*q)
        if a is None or b is None:
            return None
        return math.hypot(a[0] - b[0], a[1] - b[1])

    def speed_mps(self, p, q, dt):
        if dt <= 0:
            return None
        d = self.dist_m(p, q)
        return None if d is None else d / dt

    def px_for_metres(self, metres, y):
        """Convert a metric threshold back to pixels AT ROW y. Lets existing
        pixel-based code keep working while becoming perspective-correct."""
        s = self.scale_at(y)
        return None if not s else metres / s

    # -- self-report ---------------------------------------------------------
    def camera_height_m(self):
        """Implied camera height. For a level camera h = (Hp/Hc)*(y - y_horizon),
        so a == Hp/Hc. A tilted camera makes this an approximation, which is
        exactly why it is printed: 2-6 m is a reception camera, anything else
        means the fit is wrong or the tilt is severe."""
        if self.mode != "auto" or not self.a:
            return None
        return self.person_h / self.a

    def horizon_y(self):
        if self.mode != "auto" or not self.a:
            return None
        return -self.b / self.a

    def describe(self):
        if self.mode == "none":
            return f"ground plane: NOT AVAILABLE ({self.note}) — thresholds stay in pixels"
        if self.mode == "exact":
            return f"ground plane: EXACT ({self.note})"
        hc, hy = self.camera_height_m(), self.horizon_y()
        warn = ""
        if hc is not None and not (1.8 <= hc <= 7.0):
            warn = (f"  !! implied camera height {hc:.1f} m is implausible — the "
                    f"perspective fit or the camera tilt is off; treat metric "
                    f"numbers as indicative and supply ground_points for exact")
        return (f"ground plane: AUTO — implied camera height {hc:.2f} m, "
                f"horizon at row {hy:.0f}, {self.note}" + warn)

    def sanity(self, frame_h):
        """Cheap self-checks whose failure means 'do not trust the metres'."""
        out = []
        if self.mode != "auto":
            return out
        hc = self.camera_height_m()
        if hc is None or not (1.8 <= hc <= 7.0):
            out.append(f"implied camera height {hc} m outside 1.8-7.0 m")
        hy = self.horizon_y()
        if hy is None or hy > frame_h:
            out.append(f"horizon at row {hy} is below the frame ({frame_h})")
        s_near = self.scale_at(frame_h * 0.95)
        s_far = self.scale_at(frame_h * 0.35)
        if s_near and s_far and s_far / s_near < 1.15:
            out.append(f"near/far scale barely differs ({s_near:.4f} vs {s_far:.4f} "
                       f"m/px) — the camera may be near-overhead, in which case "
                       f"pixel thresholds were already fine")
        return out


# ---------------------------------------------------------------------------
# a synthetic camera, used by the tests — and by anyone who wants to check the
# maths without footage. Kept here on purpose: a calibration model you cannot
# generate known-answer data for is a calibration model you cannot trust.
# ---------------------------------------------------------------------------
def synth_camera(cam_h=3.0, focal_px=1200.0, frame=(1920, 1080), tilt_deg=0.0):
    """Return project(X, Z, Y) -> (x, y) for a camera at height cam_h looking
    down the +Z axis, optionally pitched down by tilt_deg."""
    cx, cy = frame[0] / 2.0, frame[1] / 2.0
    t = math.radians(tilt_deg)

    def project(X, Z, Y=0.0):
        # world -> camera (camera at (0, cam_h, 0), pitched down by t)
        yc = cam_h - Y
        zc = Z * math.cos(t) - yc * math.sin(t)
        yv = Z * math.sin(t) + yc * math.cos(t)
        if zc <= 1e-6:
            return None
        return (cx + focal_px * X / zc, cy + focal_px * yv / zc)

    return project
''',
    'helpers.py': r'''"""helpers.py — the small shared functions engine.py needs.

EXTRACTED FROM notebook Cells 2, 4 and 6. These are not a coherent subsystem;
they are the utilities the engine happened to reach for out of the notebook's
shared namespace — a clock formatter, a zone loader, a colour map, an id
coercer.

WHY THEY MATTER MORE THAN THEY LOOK
    engine.py CALLS every one of them. In the notebook that worked because all
    cells share one namespace. As a module it would have raised NameError at
    runtime — after the video decoded, after the models loaded, deep inside
    process_video. Python resolves a global at call time, so the import
    succeeded and the failure waited.

    That is exactly the class of bug this project keeps producing: something
    that looks fine until the expensive part has already run.
"""
from __future__ import annotations

import json
import math
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np

from .log import get_logger

_log = get_logger("helpers")

VIDEO_START_CLOCK = None    # set per run; wall() falls back to video time

def _safe_id(tid):
    """Safe ID converter for track IDs: handles both integer IDs (41) and named string IDs ('receptionist_sarah')."""
    if isinstance(tid, str) and not tid.isdigit():
        return tid
    try:
        return int(tid)
    except Exception:
        return str(tid)

def load_zone_config(path, frame_size=None):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(GENERIC_TEMPLATE, indent=2))
        print(f"wrote GENERIC TEMPLATE {path} — edit the coordinates to match this camera view!")
    cfg = json.loads(path.read_text())
    polygons = {name: np.array(pts, dtype=float) for name, pts in cfg.get("polygons", {}).items()}
    # U3: a venue can have more than one door. "entry_lines" is a name -> 2-point
    # map; the old singular "entry_line" still works and becomes {"entry": ...}.
    # Returning ONE line meant a two-door venue could not be counted at all.
    _lines = dict(cfg.get("entry_lines") or {})
    if not _lines and cfg.get("entry_line"):
        _lines["entry"] = cfg["entry_line"]
    entry_lines = {n: [list(map(float, p)) for p in pts]
                   for n, pts in _lines.items() if pts and len(pts) == 2}

    if frame_size:
        fw, fh = frame_size
        ref = cfg.get("frame_size")
        if not ref:
            all_pts = np.vstack(list(polygons.values()) +
                                [np.array(p, dtype=float) for p in entry_lines.values()])
            max_x, max_y = all_pts[:, 0].max(), all_pts[:, 1].max()
            if max_x > fw or max_y > fh:
                ref = (max_x, max_y)
        if ref:
            rw, rh = ref
            sx, sy = fw / rw, fh / rh
            polygons = {name: pts * [sx, sy] for name, pts in polygons.items()}
            entry_lines = {n: [[x * sx, y * sy] for x, y in pts]
                           for n, pts in entry_lines.items()}
            if ref != (fw, fh):
                print(f"↕️  {path.name}: scaled zones from reference "
                      f"{rw:.0f}x{rh:.0f} -> actual {fw}x{fh} "
                      f"(factor {sx:.3f}x, {sy:.3f}y)")

    polygons = {name: pts.astype(int) for name, pts in polygons.items()}
    entry_lines = {n: [[int(x), int(y)] for x, y in pts]
                   for n, pts in entry_lines.items()}
    # U7: an explicit roles map in the zones file wins over keyword guessing, so
    # a zone named in any language still gets its role instead of silently
    # becoming "other" and having its metrics vanish from the report.
    for zname, rs in (cfg.get("roles") or {}).items():
        ZONE_AI_OVERRIDES[zname] = list(rs) if isinstance(rs, (list, tuple)) else [rs]
    return polygons, entry_lines

def classify_zones(zone_names):
    roles = {}
    for name in zone_names:
        if name in ZONE_AI_OVERRIDES:
            roles[name] = ZONE_AI_OVERRIDES[name]
            continue
        low = str(name).lower()
        # keep in sync with Cell 2's classify_zones — this later definition
        # overwrites it, and it was missing archway/portal
        is_entry_indicator = any(kw in low for kw in ["gate", "door", "entry", "entrance", "passageway", "archway", "portal"])
        matched = []
        for role, kws in ZONE_ROLE_KEYWORDS.items():
            if role == "seating" and is_entry_indicator:
                if not any(explicit in low for explicit in ["table", "booth", "chair", "seat"]):
                    continue
            if any(kw in low for kw in kws):
                matched.append(role)
        roles[name] = matched or ["other"]
    return roles

def mmss(s):
    s = max(0, int(s))
    return f"{s // 60:02d}:{s % 60:02d}"

def wall(t):
    """Video-time -> wall-clock string using the burned-in start time."""
    if not VIDEO_START_CLOCK:
        return ""
    base = datetime.strptime(VIDEO_START_CLOCK, "%H:%M:%S")
    return (base + timedelta(seconds=float(t))).strftime("%H:%M:%S")

def zone_color_map(zone_names):
    """Fixed sorted order -> fixed colors, shared by charts AND video overlay."""
    return {z: ZONE_HEXES[i % len(ZONE_HEXES)] for i, z in enumerate(sorted(zone_names))}

def show_gallery(snaps, title, ncols=3, max_items=9):
    items = snaps[:max_items]
    if not items:
        print(f"({title}: no frames captured)")
        return
    rows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(rows, ncols, figsize=(5.2 * ncols, 3.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (t, img) in zip(axes, items):
        ax.imshow(img)
        ax.set_title(f"t = {mmss(t)}", fontsize=10, color=INK2)
    fig.suptitle(title, fontsize=13, color=INK)
    plt.tight_layout()
    plt.show()

def plot_reid_pair_audit(pairs, track_crops, title, max_pairs=6):
    """(v36) The actual crops behind a calibration number, side by side --
    turns "why don't the numbers separate" into something you can just
    look at, instead of another round of threshold guessing.

    pairs: [(a, b, sim), ...] -- e.g. calibration_report["same_pairs_worst"]
           or ["diff_pairs_worst"], already worst-first.
    track_crops: {track_id: [(_, crop_bgr), ...]} -- same structure used for
                 face embedding; picks the LARGEST banked crop per track as
                 the representative image (most informative for a human to
                 judge), and prints its pixel size so you can see directly
                 if crops are simply too small/blurry for even a human to
                 tell two people apart -- if you can't tell either, the
                 model failing isn't a threshold problem, it's a resolution/
                 occlusion/uniform-clothing problem no threshold fixes.
    """
    items = pairs[:max_pairs]
    if not items:
        print(f"({title}: no pairs to show)")
        return
    fig, axes = plt.subplots(len(items), 2, figsize=(6.4, 2.6 * len(items)))
    axes = np.atleast_2d(axes)
    for row, (a, b, sim) in enumerate(items):
        for col, tid in enumerate((a, b)):
            ax = axes[row][col]
            ax.axis("off")
            crops = track_crops.get(tid, [])
            if crops:
                _, crop = max(crops, key=lambda c: c[1].shape[0] * c[1].shape[1])
                ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                ax.set_title(f"ID {tid}  ({crop.shape[1]}x{crop.shape[0]}px)",
                            fontsize=9)
            else:
                ax.set_title(f"ID {tid} (no banked crop)", fontsize=9)
        axes[row][0].set_ylabel(f"sim={sim:.3f}", fontsize=10, rotation=0,
                                labelpad=32, ha="right", va="center")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()
''',
    'learn_zones.py': r'''"""learn_zones.py — PHASE 12. Find the door in the data instead of drawing it.

WHY THIS EXISTS
    On the first real CAM.112 hour the entry line was drawn ~200px too short.
    People walked around both ends, the line fired ZERO times, and eight
    GM-facing numbers silently collapsed to 0 while 46 people waited. Nothing
    in the pipeline noticed, because hand-drawn geometry fails silently and
    totally.

    That is the structural problem with manual zone mapping, and it does not
    get better with care — it gets worse with scale. Nobody re-draws 400
    polygons across a fleet when a camera is nudged during cleaning.

THE PRINCIPLE
    An entrance is not a place someone drew a line. It is THE PLACE WHERE
    TRACKS ARE BORN AND WHERE THEY DIE. People appear at doors and disappear
    at doors; in the middle of a room they neither materialise nor vanish.

    So cluster the first and last position of every track. The door falls out
    of the data. This is old, well-proven work — Makris & Ellis, "Learning
    semantic scene models from observing activity" (2002) — and it is what
    makes a fleet deployment scale: propose zones automatically, have a human
    confirm once, instead of drawing every polygon by hand.

THE HONEST CEILING — read this before trusting a proposal
    A tracker that fragments creates FALSE births and deaths in the middle of
    the room. On the real run identity lifetime had a median of 38s, so this
    footage fragments a lot, and naive endpoint clustering would happily
    propose a "door" in the centre of the lobby.

    Three guards, none of them free:
      1. a track must LIVE long enough (min_life_s) to be evidence
      2. it must TRAVEL far enough (min_travel_frac of the frame diagonal) —
         a fragment that appears and dies on the spot is not a journey
      3. a real door has BOTH births and deaths; a fragmentation hotspot
         usually skews to one

    Even so: these are PROPOSALS for a human to confirm, never a silent
    replacement for the zones file. The function returns evidence with every
    proposal so it can be argued with.

    ponytail: grid histogram + connected components, no clustering library.
    Good enough to find a door; swap for DBSCAN if a venue needs finer shapes.
"""
from __future__ import annotations

import math
from collections import defaultdict

MIN_LIFE_S = 1.5
MIN_TRAVEL_FRAC = 0.06
CELL_FRAC = 0.07
MIN_ENDPOINTS = 4
EDGE_FRAC = 0.18          # within this fraction of a border = "at the frame edge"


def track_endpoints(frame_log, canon=None, frame_wh=None,
                    min_life_s=MIN_LIFE_S, min_travel_frac=MIN_TRAVEL_FRAC):
    """-> (kept, stats). One birth/death per track, for tracks that are evidence.

    Uses the FOOT point (bottom-centre), not the box centre: a door is a place
    on the floor, and the feet are where the person actually is.
    """
    canon = canon or {}
    per = defaultdict(list)
    for _idx, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            per[canon.get(tid, tid)].append(
                (float(t), (float(x1) + float(x2)) / 2.0, float(y2)))

    if frame_wh:
        fw, fh = float(frame_wh[0]), float(frame_wh[1])
    else:
        fw = max((p[1] for v in per.values() for p in v), default=1280.0)
        fh = max((p[2] for v in per.values() for p in v), default=720.0)
    diag = math.hypot(fw, fh)

    kept, n_short, n_still = [], 0, 0
    for tid, rows in per.items():
        rows.sort()
        life = rows[-1][0] - rows[0][0]
        if life < min_life_s:
            n_short += 1
            continue
        b, d = (rows[0][1], rows[0][2]), (rows[-1][1], rows[-1][2])
        travel = math.hypot(d[0] - b[0], d[1] - b[1])
        if travel < min_travel_frac * diag:
            n_still += 1
            continue
        kept.append({"track_id": tid, "birth": b, "death": d,
                     "life_s": round(life, 1), "travel_px": round(travel, 1)})
    return kept, {"tracks": len(per), "used": len(kept),
                  "dropped_short": n_short, "dropped_still": n_still,
                  "frame_wh": (fw, fh)}


def _clusters(points, frame_wh, cell_frac=CELL_FRAC, min_count=MIN_ENDPOINTS):
    """Grid histogram -> connected components -> padded bounding boxes."""
    if not points:
        return []
    fw, fh = frame_wh
    cell = max(8.0, cell_frac * max(fw, fh))
    grid = defaultdict(list)
    for p in points:
        grid[(int(p[0] // cell), int(p[1] // cell))].append(p)

    seen, out = set(), []
    for key in grid:
        if key in seen:
            continue
        stack, comp = [key], []
        seen.add(key)
        while stack:                                  # 8-neighbour flood fill
            cx, cy = stack.pop()
            comp.extend(grid[(cx, cy)])
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    nk = (cx + dx, cy + dy)
                    if nk in grid and nk not in seen:
                        seen.add(nk)
                        stack.append(nk)
        if len(comp) < min_count:
            continue
        xs = [p[0] for p in comp]
        ys = [p[1] for p in comp]
        pad = cell * 0.35
        out.append({"box": (max(0.0, min(xs) - pad), max(0.0, min(ys) - pad),
                            min(fw, max(xs) + pad), min(fh, max(ys) + pad)),
                    "n": len(comp),
                    "centre": (round(sum(xs) / len(xs)), round(sum(ys) / len(ys)))})
    out.sort(key=lambda c: -c["n"])
    return out


def _edge_dist_frac(box, frame_wh):
    fw, fh = frame_wh
    x1, y1, x2, y2 = box
    return min(x1 / fw, y1 / fh, (fw - x2) / fw, (fh - y2) / fh)


def learn_entry_zones(frame_log, canon=None, frame_wh=None, top=3, **kw):
    """Propose entrance polygons from where tracks are BORN and DIE.

    -> (proposals, stats). Each proposal carries its own evidence so a human
    can disagree with it; nothing here silently overwrites a zones file.
    """
    eps, stats = track_endpoints(frame_log, canon=canon, frame_wh=frame_wh, **{
        k: v for k, v in kw.items() if k in ("min_life_s", "min_travel_frac")})
    fwh = stats["frame_wh"]
    if not eps:
        return [], stats

    births = [e["birth"] for e in eps]
    deaths = [e["death"] for e in eps]
    cl = _clusters(births + deaths, fwh)

    props = []
    for c in cl:
        x1, y1, x2, y2 = c["box"]
        nb = sum(1 for p in births if x1 <= p[0] <= x2 and y1 <= p[1] <= y2)
        nd = sum(1 for p in deaths if x1 <= p[0] <= x2 and y1 <= p[1] <= y2)
        # A real door is used in BOTH directions. A fragmentation hotspot is
        # lopsided: tracks die there and are re-born as new ids, or vice versa.
        balance = min(nb, nd) / max(nb, nd) if max(nb, nd) else 0.0
        edge = _edge_dist_frac(c["box"], fwh)
        at_edge = edge <= EDGE_FRAC
        score = (nb + nd) * (0.5 + 0.5 * balance) * (1.25 if at_edge else 1.0)
        props.append({
            "polygon": [[round(x1), round(y1)], [round(x2), round(y1)],
                        [round(x2), round(y2)], [round(x1), round(y2)]],
            "box": c["box"], "centre": c["centre"], "births": nb, "deaths": nd,
            "balance": round(balance, 2), "at_frame_edge": at_edge,
            "score": round(score, 1),
            "why": (f"{nb} track(s) began and {nd} ended here "
                    f"(balance {balance:.2f}"
                    + (", at the frame edge" if at_edge else
                       ", NOT at the frame edge — could be a fragmentation "
                       "hotspot rather than a door")
                    + ")"),
        })
    props.sort(key=lambda p: -p["score"])
    return props[:top], stats


def learn_dwell_zones(frame_log, canon=None, frame_wh=None, top=3,
                      slow_frac=0.004, min_hits=30):
    """Propose zones where people STOP — waiting areas, counters, queues.

    Movement below slow_frac of the frame diagonal per second counts as
    stopped. Same grid machinery as the entrances; different evidence.
    """
    canon = canon or {}
    per = defaultdict(list)
    for _idx, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            per[canon.get(tid, tid)].append(
                (float(t), (float(x1) + float(x2)) / 2.0, float(y2)))
    if frame_wh:
        fw, fh = float(frame_wh[0]), float(frame_wh[1])
    else:
        fw = max((p[1] for v in per.values() for p in v), default=1280.0)
        fh = max((p[2] for v in per.values() for p in v), default=720.0)
    diag = math.hypot(fw, fh)

    slow = []
    for _tid, rows in per.items():
        rows.sort()
        for a, b in zip(rows, rows[1:]):
            dt = b[0] - a[0]
            if dt <= 0:
                continue
            if math.hypot(b[1] - a[1], b[2] - a[2]) / dt <= slow_frac * diag:
                slow.append((b[1], b[2]))

    out = []
    for c in _clusters(slow, (fw, fh), min_count=min_hits):
        x1, y1, x2, y2 = c["box"]
        out.append({
            "polygon": [[round(x1), round(y1)], [round(x2), round(y1)],
                        [round(x2), round(y2)], [round(x1), round(y2)]],
            "box": c["box"], "centre": c["centre"], "samples": c["n"],
            "why": f"{c['n']} sample(s) where somebody was standing still here",
        })
    return out[:top], {"slow_samples": len(slow), "frame_wh": (fw, fh)}


def to_zone_config(entries, dwells=(), frame_wh=None):
    """Turn proposals into a zones_*.json-shaped dict a human can edit.

    Names carry the keyword the role classifier already understands, so a
    proposal that is accepted needs no further wiring.
    """
    polys, roles = {}, {}
    for i, p in enumerate(entries, 1):
        n = "main_entrance" if i == 1 else f"entrance_{i}"
        polys[n] = p["polygon"]
        roles[n] = ["entry"]
    for i, p in enumerate(dwells, 1):
        n = "waiting_area" if i == 1 else f"waiting_area_{i}"
        polys[n] = p["polygon"]
        roles[n] = ["wait"]
    cfg = {"polygons": polys, "roles": roles,
           "_generated": "kevacv.learn_zones — PROPOSALS, confirm before use"}
    if entries:
        # A line across the widest axis of the busiest entrance, then EXTENDED
        # well past the observed cluster.
        #
        # This overshoot is the whole point. The cluster only covers where
        # people were actually SEEN crossing; the physical doorway is always at
        # least as wide, usually wider. A line drawn to the cluster is a line
        # people can walk around the ends of — which is precisely the failure
        # that produced "0 people came through the door" for a full hour.
        # Overshooting costs nothing (a line beyond the doorway is never
        # crossed); undershooting costs every arrival.
        x1, y1, x2, y2 = entries[0]["box"]
        fw, fh = (frame_wh if frame_wh else (max(x2 * 1.2, 1), max(y2 * 1.2, 1)))
        grow = 0.6
        if (x2 - x1) >= (y2 - y1):
            pad = (x2 - x1) * grow
            cy = round((y1 + y2) / 2)
            cfg["entry_line"] = [[round(max(0, x1 - pad)), cy],
                                 [round(min(fw, x2 + pad)), cy]]
        else:
            pad = (y2 - y1) * grow
            cx = round((x1 + x2) / 2)
            cfg["entry_line"] = [[cx, round(max(0, y1 - pad))],
                                 [cx, round(min(fh, y2 + pad))]]
        cfg["_entry_line_note"] = (
            "extended 60% past the observed crossings on each side — a line "
            "that stops inside the doorway lets people walk around its ends, "
            "which is the failure this module exists to prevent")
    if frame_wh:
        cfg["frame_size"] = [int(frame_wh[0]), int(frame_wh[1])]
    return cfg


def describe(entries, dwells=(), stats=None):
    L = ["LEARNED ZONES — proposed from the tracks themselves, not drawn"]
    if stats:
        L.append(f"  evidence: {stats['used']} of {stats['tracks']} tracks used "
                 f"({stats['dropped_short']} too short-lived, "
                 f"{stats['dropped_still']} never travelled)")
        if stats["tracks"] and stats["used"] / stats["tracks"] < 0.25:
            L.append("  !! fewer than a quarter of tracks were usable — this "
                     "footage fragments badly, so treat these as weak hints")
    if not entries:
        L.append("  no entrance proposed — not enough tracks that both lived "
                 "and travelled. This says the TRACKING is too fragmented to "
                 "learn from, which is itself the finding.")
    for i, p in enumerate(entries, 1):
        L.append(f"  ENTRANCE {i} at {p['centre']}  score {p['score']}")
        L.append(f"    {p['why']}")
    for i, p in enumerate(dwells, 1):
        L.append(f"  DWELL {i} at {p['centre']}: {p['why']}")
    L.append("  These are PROPOSALS. Confirm against one frame before use — a "
             "learned zone is evidence, not authority.")
    return "\n".join(L)
''',
    'log.py': r'''"""log.py — one timeline for the whole run, root to leaf.

WHY THIS EXISTS
    The pipeline reports itself with bare print(). That produced a 191,977
    character wall of `'half' is deprecated` in one cell output, buried the
    line that mattered, and gave no way to answer the only question you ever
    ask afterwards: WHERE did the time go, and WHAT was true at that moment.

    Worse, prints have no severity. "🚨 ENTRY LINE NEVER TRIGGERED" and
    "loading model" arrive looking exactly alike, so the loud failure scrolls
    past with the noise.

WHAT THIS GIVES
    A nested stage timeline. Every stage logs when it starts, when it ends, how
    long it took, and what it counted — with the full path from the root, so a
    line is readable on its own:

        12:04:07 INFO  run > chunk1 > pass1 > detect      | 27060 frames, 8 fps
        12:19:44 INFO  run > chunk1 > pass1               | done in 15m37s
        12:19:44 WARN  run > chunk1 > identity            | 356 merges starved
        12:19:45 ERROR run > chunk1 > zones               | entry line fired 0x

    Counters are first-class. `stage.count("frames", 27060)` is recorded
    against the stage and printed in its summary, so the numbers that matter
    end up next to the time they cost — instead of scattered across prints.

DESIGN NOTES
    * stdout AND a file, always. The file is the run's provenance record.
    * A stage that raises still logs its end, with the exception, so a crash
      leaves a complete timeline rather than a truncated one.
    * No global mutable logger config beyond setup(); import order cannot
      change behaviour.
    * Zero dependencies. This must import on a laptop with no GPU.
"""
from __future__ import annotations

import logging
import os
import sys
import threading
import time
from contextlib import contextmanager
from pathlib import Path

_LOCAL = threading.local()          # stage stack is per-thread (2-GPU runner)
_ROOT_NAME = "kevacv"

LEVELS = {"DEBUG": logging.DEBUG, "INFO": logging.INFO,
          "WARN": logging.WARNING, "WARNING": logging.WARNING,
          "ERROR": logging.ERROR}


def _stack():
    if not hasattr(_LOCAL, "stack"):
        _LOCAL.stack = []
    return _LOCAL.stack


class _PathFilter(logging.Filter):
    """Kept for handlers that want it explicitly; the record factory below is
    what actually guarantees the field exists."""

    def filter(self, record):
        if not hasattr(record, "stagepath"):
            record.stagepath = " > ".join(_stack()) or "-"
        return True


def _install_record_factory():
    """Stamp `stagepath` onto EVERY record at creation.

    A logging.Filter attached to a logger does not run for records that
    propagate up from a child logger — so `get_logger("pipeline")` produced
    records with no stagepath, and any handler formatting "%(stagepath)s"
    raised or printed nothing. Setting it at the factory means the field
    exists no matter which logger created the record or which handler
    formats it, including handlers a caller attaches itself.
    """
    old = logging.getLogRecordFactory()
    if getattr(old, "_kevacv", False):
        return                      # idempotent: never wrap our own wrapper
    def factory(*args, **kwargs):
        rec = old(*args, **kwargs)
        if not hasattr(rec, "stagepath"):
            rec.stagepath = " > ".join(_stack()) or "-"
        return rec
    factory._kevacv = True
    logging.setLogRecordFactory(factory)


_install_record_factory()


def setup(log_dir="logs", level=None, name=None, stream=True):
    """Configure the run's logger. Safe to call twice; the second call is a
    no-op rather than a duplicated handler (double-printed logs are how people
    stop reading them)."""
    logger = logging.getLogger(_ROOT_NAME)
    if getattr(logger, "_kevacv_configured", False):
        return logger

    level = LEVELS.get(str(level or os.environ.get("KV_LOG_LEVEL", "INFO")).upper(),
                       logging.INFO)
    logger.setLevel(level)
    logger.propagate = False
    fmt = logging.Formatter(
        "%(asctime)s %(levelname)-5s %(stagepath)-46s | %(message)s",
        datefmt="%H:%M:%S")
    flt = _PathFilter()

    if stream:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        h.addFilter(flt)
        logger.addHandler(h)

    if log_dir:
        d = Path(log_dir)
        d.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
        # append, never with_suffix: run ids and camera names carry dots (V73)
        fh = logging.FileHandler(str(d / f"{name or 'run'}_{stamp}.log"),
                                 encoding="utf-8")
        fh.setFormatter(fmt)
        fh.addFilter(flt)
        logger.addHandler(fh)

    logger._kevacv_configured = True
    return logger


def close():
    """Detach and close every handler, releasing the log file.

    Needed for real reasons, not just tests: on Windows an open FileHandler
    keeps a lock on the file, so a run that wants to move, zip or delete its
    own log directory cannot until this is called. It also lets setup() be
    called again with a different destination — one log file per chunk, say.
    """
    logger = logging.getLogger(_ROOT_NAME)
    for h in list(logger.handlers):
        logger.removeHandler(h)
        try:
            h.close()
        except Exception:
            pass
    logger._kevacv_configured = False
    return logger


def get_logger(module=None):
    """A logger for one module. setup() need not have run — an unconfigured
    logger simply produces nothing, which is correct for `import kevacv` in a
    test."""
    base = logging.getLogger(_ROOT_NAME)
    if not base.handlers:
        base.addHandler(logging.NullHandler())
        base.addFilter(_PathFilter())
    return base.getChild(module) if module else base


class Stage:
    """A named span of work. Returned by stage(); counters attach to it."""

    def __init__(self, name, log):
        self.name = name
        self.log = log
        self.counts = {}
        self.t0 = time.time()

    def count(self, key, value=1):
        """Record a number against this stage. Adds if numeric, else sets."""
        cur = self.counts.get(key)
        self.counts[key] = (cur + value) if isinstance(cur, (int, float)) \
            and isinstance(value, (int, float)) else value
        return self

    def note(self, msg, level="INFO"):
        self.log.log(LEVELS.get(level.upper(), logging.INFO), msg)
        return self

    @property
    def elapsed(self):
        return time.time() - self.t0


def human(seconds):
    s = float(seconds)
    if s < 1:
        return f"{s*1000:.0f}ms"
    if s < 60:
        return f"{s:.1f}s"
    m, s = divmod(int(s), 60)
    if m < 60:
        return f"{m}m{s:02d}s"
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m"


@contextmanager
def stage(name, module=None, quiet_under_s=0.0):
    """Time a stage and log its start, end and counters.

        with stage("detect") as st:
            st.count("frames", n)

    An exception is logged at ERROR with the elapsed time and re-raised, so a
    crash still leaves a complete timeline instead of stopping mid-sentence.
    """
    log = get_logger(module)
    st = Stage(name, log)
    _stack().append(name)
    log.info("start")
    try:
        yield st
    except Exception as exc:
        log.error(f"FAILED after {human(st.elapsed)} — "
                  f"{type(exc).__name__}: {exc}")
        raise
    else:
        if st.elapsed >= quiet_under_s:
            bits = "  ".join(f"{k}={v}" for k, v in st.counts.items())
            log.info(f"done in {human(st.elapsed)}" + (f"   {bits}" if bits else ""))
    finally:
        _stack().pop()


def banner(title, lines=(), level="INFO", module=None):
    """A finding a human must not scroll past. Use for the things that
    invalidate a run — a misplaced entry zone, an unverified clock — not for
    progress."""
    log = get_logger(module)
    lvl = LEVELS.get(level.upper(), logging.INFO)
    log.log(lvl, "=" * 70)
    log.log(lvl, title)
    for ln in lines:
        log.log(lvl, f"  {ln}")
    log.log(lvl, "=" * 70)
''',
    'merge_ab.py': r'''"""merge_ab.py — measure a merge-policy change before shipping it.

WHY THIS EXISTS
    Run 68b97311f9:

        merges accepted                  69
        blocked by window overlap       356      <- five times more
        blocked by role conflict         54

    The greedy union in merge_fragmented_tracks accumulates group_windows: each
    time a group absorbs a fragment its window set gets wider and gappier, so
    every later candidate is more likely to overlap SOMETHING already inside.
    Early merges progressively poison later ones, and the starved candidates
    were strong physical evidence:

        ID 47  <-> ID 345  tier=stationary  score=0.922
        ID 257 <-> ID 277  tier=stationary  score=0.918

    That is processing order deciding identity, not evidence. So any change
    that removes IMPOSSIBLE pairs early should show up twice: fewer wrong
    unions, and fewer good merges starved by the wrong unions' windows.

    "Should" is a hypothesis. This module measures it.

WHAT THIS IS NOT
    greedy_union() is a faithful re-implementation of the notebook's union
    loop (Cell 5, ~line 1519) for MEASUREMENT ONLY. It is not the production
    path and must never become it — two copies of a merge policy is exactly
    how they drift apart. If the notebook's loop changes, `assert_matches()`
    is here so a test can catch this copy going stale.

    It deliberately reproduces the greedy behaviour including its flaws. A
    harness that quietly fixes the thing it is measuring measures nothing.
"""
from __future__ import annotations

from collections import defaultdict

from .topology import reappearance_verdict


def windows_overlap(wins_a, wins_b, tolerance_s=2.0):
    """Exact copy of the notebook's _windows_overlap — same tolerance, same
    strict inequalities. Copied rather than imported because the original
    lives in a notebook cell; kept identical on purpose."""
    for a0, a1 in wins_a:
        for b0, b1 in wins_b:
            if a0 < b1 - tolerance_s and b0 < a1 - tolerance_s:
                return True
    return False


def greedy_union(pairs, track_windows, tolerance_s=2.0, sort_key=None):
    """Replay the greedy best-evidence-first union.

    pairs: [(sim, a, b, tier), ...] — the same tuples merge_fragmented_tracks
    builds. Returns the mapping plus the diagnostics that matter for A/B.

    `sort_key` mirrors the production `sorted(pairs, reverse=True)`. The
    default sorts on sim only, with a stable tiebreak on str(id) — the real
    loop has no key and would raise TypeError on an int/str tie, which is a
    latent bug there, not a behaviour worth copying.
    """
    parent = {t: t for t in track_windows}

    def find(t):
        while parent[t] != t:
            parent[t] = parent[parent[t]]
            t = parent[t]
        return t

    group_windows = {t: [track_windows[t]] for t in track_windows}
    accepted, overlap_blocked = [], []
    tier_counts = defaultdict(int)

    key = sort_key or (lambda p: (p[0], str(p[1]), str(p[2])))
    for sim, a, b, tier in sorted(pairs, key=key, reverse=True):
        if a not in parent or b not in parent:
            continue
        ra, rb = find(a), find(b)
        if ra == rb:
            continue
        if windows_overlap(group_windows[ra], group_windows[rb], tolerance_s):
            overlap_blocked.append((sim, a, b, tier))
            continue
        parent[rb] = ra
        group_windows[ra] = group_windows[ra] + group_windows[rb]
        accepted.append((sim, a, b, tier))
        tier_counts[tier] += 1

    mapping = {}
    canon = {}
    for t in sorted(track_windows, key=lambda x: (track_windows[x][0], str(x))):
        root = find(t)
        canon.setdefault(root, t)
        mapping[t] = canon[root]

    return {
        "mapping": mapping,
        "accepted": accepted,
        "overlap_blocked": overlap_blocked,
        "n_accepted": len(accepted),
        "n_overlap_blocked": len(overlap_blocked),
        "tier_counts": dict(tier_counts),
        "n_identities": len(set(mapping.values())),
        "n_tracks": len(track_windows),
    }


def apply_topology_veto(pairs, track_windows, positions, doors, frame_wh, **kw):
    """Split candidate pairs into (possible, vetoed) using the topology gate.

    A pair is (sim, a, b, tier). The earlier-starting track supplies the death
    position, the later one the birth position — the same convention
    merge_fragmented_tracks uses when it computes hand-off distance.
    """
    possible, vetoed = [], []
    for p in pairs:
        _sim, a, b, _tier = p
        if a not in track_windows or b not in track_windows:
            possible.append(p)
            continue
        wa, wb = track_windows[a], track_windows[b]
        early, late = (a, b) if wa[0] <= wb[0] else (b, a)
        pe, pl = positions.get(early), positions.get(late)
        if not pe or not pl:
            possible.append(p)          # no position -> cannot judge -> allow
            continue
        gap = track_windows[late][0] - track_windows[early][1]
        v = reappearance_verdict(pe[1], pl[0], gap, doors, frame_wh, **kw)
        (possible if v["allow"] else vetoed).append(p)
    return possible, vetoed


def ab_topology(pairs, track_windows, positions, doors, frame_wh,
                tolerance_s=2.0, **kw):
    """Run the union twice — as-is, and with impossible pairs removed first.

    -> {"without", "with", "vetoed", "delta"}. `delta` is what to read: if the
    veto helps, n_accepted goes UP (good merges stop being starved) while
    n_overlap_blocked goes DOWN, and identity count falls toward the truth.
    """
    without = greedy_union(pairs, track_windows, tolerance_s)
    possible, vetoed = apply_topology_veto(pairs, track_windows, positions,
                                           doors, frame_wh, **kw)
    with_ = greedy_union(possible, track_windows, tolerance_s)
    return {
        "without": without,
        "with": with_,
        "n_vetoed": len(vetoed),
        "vetoed": vetoed,
        "delta": {
            "accepted": with_["n_accepted"] - without["n_accepted"],
            "overlap_blocked": (with_["n_overlap_blocked"]
                                - without["n_overlap_blocked"]),
            "identities": with_["n_identities"] - without["n_identities"],
        },
    }


def describe(ab):
    w0, w1, d = ab["without"], ab["with"], ab["delta"]
    L = ["A/B — topology veto on the greedy merge",
         f"  {'':<22}{'without':>10}{'with':>10}{'delta':>10}",
         f"  {'candidate pairs vetoed':<22}{'-':>10}{ab['n_vetoed']:>10}{'':>10}",
         f"  {'merges accepted':<22}{w0['n_accepted']:>10}{w1['n_accepted']:>10}"
         f"{d['accepted']:>+10}",
         f"  {'starved by overlap':<22}{w0['n_overlap_blocked']:>10}"
         f"{w1['n_overlap_blocked']:>10}{d['overlap_blocked']:>+10}",
         f"  {'identities':<22}{w0['n_identities']:>10}{w1['n_identities']:>10}"
         f"{d['identities']:>+10}"]
    if d["overlap_blocked"] < 0 and d["accepted"] >= 0:
        L.append("  -> the veto freed starved merges: fewer wrong unions "
                 "polluting group windows")
    elif d["accepted"] < 0:
        L.append("  -> the veto is REMOVING merges the pipeline wanted. Check "
                 "the door positions before shipping this.")
    else:
        L.append("  -> no measurable effect on this data")
    return "\n".join(L)


def assert_matches(notebook_source):
    """Cheap staleness guard: does the notebook's union loop still look like
    the one copied here? Returns (ok, notes) — a test can fail on this so the
    copy cannot silently drift from the original."""
    notes = []
    for needle in ("_windows_overlap(group_windows[ra], group_windows[rb])",
                   "parent[rb] = ra",
                   "group_windows[ra] = group_windows[ra] + group_windows[rb]",
                   "for sim, a, b, tier in sorted(pairs, reverse=True)"):
        if needle not in notebook_source:
            notes.append(f"missing in notebook: {needle}")
    return (not notes), notes
''',
    'phantoms.py': r'''"""phantoms.py — PHASE 10. Kill static false positives that CHANGE ID.

WHY THIS EXISTS
    D2 (detect_filters.static_track_ids) asks "did this TRACK ID sit still for
    a long time?". On the real CAM.112 hour it caught 3 ids and reported them
    honestly. It also missed the two phantoms that dominate every annotated
    frame:

        P3  a large box over the curved mirror on the right wall, present in
            essentially every frame, and the id-timeline audit's single worst
            offender: 2,319+ "REAPPEARED" jumps of 300-900px.
        P8  the potted plant, labelled "staff" because its box centre happens
            to fall in the reception zone.

    Both survived D2 for the same reason: THEIR IDS CHURN. The detector fires
    on the same pixels, the tracker keeps minting a fresh id, each individual
    id is short-lived, and a per-id lifetime test can never see the pattern.
    The thing that is static is the LOCATION, not the id.

THE PRINCIPLE
    Ask the question of the pixels instead of the ids. Aggregate every
    detection by where it landed, across the whole chunk. A location that
    keeps producing a box of near-identical geometry over many minutes is
    furniture, a reflection, a poster or a TV — no matter how many ids passed
    through it.

    This is the same idea as a hand-drawn mask zone, derived from the data
    instead of from a person with a mouse. That matters for universality: the
    next camera gets the same protection without anyone drawing anything.

WHAT KEEPS IT FROM EATING REAL PEOPLE
    A receptionist stands at her post for the whole hour, so "static for a
    long time" alone is NOT enough — it would delete the most important person
    in the venue. Three independent conditions must hold together:

      1. tiny centre jitter, as a FRACTION OF BODY HEIGHT (scale-free, so it
         means the same near and far). Real D2 furniture measured 0.008-0.012;
         a standing person shifts, leans and steps an order of magnitude more.
      2. near-constant box SIZE (low coefficient of variation). A person's box
         breathes as they turn and move; a mirror artefact does not.
      3. a long span AND many hits, so a brief coincidence cannot qualify.

    Plus an explicit veto: any location a PROTECTED id ever occupied (someone
    who crossed the entry line, or matched an enrolled face) is never flagged.

    ponytail: fixed grid, no clustering library. A phantom straddling a cell
    boundary splits into two weaker cells and may be missed — raise cell_frac
    or switch to real clustering if that shows up in practice.

HOW STRONG IS THE SEPARATION, HONESTLY
    Not as strong as it looks. A person standing at a desk and a plant are only
    a few thousandths of body height apart on jitter alone, and the margin is
    thin enough that a very still person could cross it. Two things carry the
    real weight:

      SIZE   the physical reason the filter works. The detector sees IDENTICAL
             pixels for a plant every frame, so it returns an almost identical
             box (size cv ~0.004 measured). A person's pixels genuinely change
             — arms, turning, leaning — and their box breathes by an order of
             magnitude more. This is the discriminator that does not depend on
             how still someone chooses to stand.
      VETO   any location a protected id occupied (crossed the entry line, or
             matched an enrolled face) is never flagged, whatever the geometry.

    So: never rely on jitter alone, and always enroll staff faces. If a real
    person is ever flagged, that is a bug worth a test, not a tuning exercise.
"""
from __future__ import annotations

import math
from collections import defaultdict

# Defaults derived from the measured CAM.112 numbers, not guessed:
#   D2's confirmed furniture      centre jitter 0.008 - 0.012 of body height
#   -> 0.02 leaves ~2x headroom above the worst real phantom while staying
#      far below where a standing person lands.
MAX_CENTRE_JITTER = 0.02
# Measured on the synthetic rebuild of the real CAM.112 phantoms:
#     plant   size cv 0.004     mirror  size cv 0.003
#     a receptionist at her post 0.043  — an order of magnitude clear
# 0.015 sits in that gap: ~4x above the phantoms, ~3x below the person. This is
# the condition doing the real work, so it is set from measurement, not taste.
MAX_SIZE_CV = 0.015
MIN_SPAN_S = 240.0
MIN_HITS = 40
CELL_FRAC = 0.04


def _median(v):
    s = sorted(v)
    n = len(s)
    if not n:
        return 0.0
    return s[n // 2] if n % 2 else 0.5 * (s[n // 2 - 1] + s[n // 2])


def _std(v):
    if len(v) < 2:
        return 0.0
    m = sum(v) / len(v)
    return math.sqrt(sum((x - m) ** 2 for x in v) / len(v))


def phantom_regions(frame_log, frame_wh=None, protected=None,
                    min_span_s=MIN_SPAN_S, min_hits=MIN_HITS,
                    max_centre_jitter=MAX_CENTRE_JITTER,
                    max_size_cv=MAX_SIZE_CV, cell_frac=CELL_FRAC):
    """Locations that keep emitting the same box -> phantom regions.

    frame_log: [(idx, t, [(tid, x1, y1, x2, y2), ...]), ...]
    -> [ {box, centre, hits, span_s, centre_jitter, size_cv, ids, why}, ... ]

    Returns [] rather than guessing when there is nothing to go on, and never
    flags a location a protected id occupied.
    """
    protected = set(protected or ())
    if not frame_log:
        return []

    if frame_wh:
        fw, fh = float(frame_wh[0]), float(frame_wh[1])
    else:                                   # infer from the boxes themselves
        fw = max((b[3] for _i, _t, bs in frame_log for b in bs), default=1280.0)
        fh = max((b[4] for _i, _t, bs in frame_log for b in bs), default=720.0)
    cell = max(8.0, cell_frac * max(fw, fh))

    buckets = defaultdict(list)
    for _idx, t, boxes in frame_log:
        for tid, x1, y1, x2, y2 in boxes:
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            buckets[(int(cx // cell), int(cy // cell))].append(
                (t, cx, cy, float(x2) - float(x1), float(y2) - float(y1), tid))

    out = []
    for _key, hits in buckets.items():
        if len(hits) < min_hits:
            continue
        ids = {h[5] for h in hits}
        if ids & protected:
            continue                        # a real, verified person was here
        ts = [h[0] for h in hits]
        span = max(ts) - min(ts)
        if span < min_span_s:
            continue
        hgt = _median([h[4] for h in hits])
        if hgt <= 1:
            continue
        # jitter as a fraction of body height: scale-free, so the same number
        # means the same thing at the door and at the back of the room.
        # max(), not hypot() — this is deliberately the SAME predicate D2 uses,
        # because D2's 0.02 is the one threshold validated against real
        # detections (it kept the receptionist and caught three real phantoms).
        # Combining the axes would silently make this filter stricter than the
        # one we have evidence for.
        jit = max(_std([h[1] for h in hits]), _std([h[2] for h in hits])) / hgt
        if jit > max_centre_jitter:
            continue
        wds = [h[3] for h in hits]
        hts = [h[4] for h in hits]
        mw, mh = _median(wds), _median(hts)
        size_cv = max(_std(wds) / mw if mw else 9, _std(hts) / mh if mh else 9)
        if size_cv > max_size_cv:
            continue
        cx, cy = _median([h[1] for h in hits]), _median([h[2] for h in hits])
        out.append({
            "box": (cx - mw / 2.0, cy - mh / 2.0, cx + mw / 2.0, cy + mh / 2.0),
            "centre": (round(cx), round(cy)),
            "hits": len(hits), "span_s": round(span, 1), "ids": len(ids),
            "centre_jitter": round(jit, 4), "size_cv": round(size_cv, 3),
            "why": (f"{len(hits)} detections over {span/60:.0f} min from "
                    f"{len(ids)} different id(s), centre jitter {jit:.4f} of "
                    f"body height, size cv {size_cv:.3f} — furniture, a "
                    f"reflection or a poster, not a person"),
        })
    # A phantom sitting on a cell boundary lands in two cells and gets reported
    # twice (seen immediately in testing: the mirror came back as 2401 hits and
    # 299 hits at the same spot). Merge overlapping CANDIDATES — merging raw
    # cells instead would let a busy neighbouring cell pull a real phantom's
    # jitter up and hide it.
    out.sort(key=lambda r: -r["hits"])
    merged = []
    for r in out:
        for m in merged:
            if _iou(r["box"], m["box"]) >= 0.3:
                m["hits"] += r["hits"]
                m["ids"] = max(m["ids"], r["ids"])
                m["span_s"] = max(m["span_s"], r["span_s"])
                break
        else:
            merged.append(r)
    return merged


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    if inter <= 0:
        return 0.0
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def in_phantom(box, regions, min_iou=0.55):
    """Is this detection the phantom itself? Overlap, not centre-containment —
    a real person walking IN FRONT of the plant has a different box and must
    survive, which a centre-in-region test would not allow."""
    return any(_iou(box, r["box"]) >= min_iou for r in regions)


def drop_phantom_dets(boxes, regions, min_iou=0.55):
    """-> (kept, dropped) for a list of (tid, x1, y1, x2, y2)."""
    kept, dropped = [], []
    for b in boxes:
        (kept, dropped)[in_phantom(tuple(b[1:5]), regions, min_iou)].append(b)
    return kept, dropped


def describe(regions):
    if not regions:
        return "no static phantom regions found"
    L = [f"PHANTOM REGIONS — {len(regions)} location(s) emitting a person box "
         f"that never moves"]
    for r in regions:
        L.append(f"  at {r['centre']}  {r['why']}")
    L.append("  These are dropped BEFORE counting. If one of these is actually a "
             "person who stood still, raise min_span_s or protect their id.")
    return "\n".join(L)
''',
    'pipeline.py': r'''"""pipeline.py — the root of the tree. One run, start to finish, in one place.

WHY THIS EXISTS
    Every capability had a module and nothing had a caller. preflight,
    topology, report_slim, tiled, threshold, merge_ab, graph_fusion — all
    built, all tested, none reachable. The orchestration lived in notebook
    cells, so the package was a library of parts with no machine.

    This is the machine. It is also the only place the stage timeline can be
    built, because only the caller knows where one phase ends and the next
    begins.

THE SHAPE OF A RUN

    run_camera
      ├─ preflight      can these zones answer the questions at all?
      ├─ analyse        engine.process_video: decode, detect, track, embed
      ├─ identity       merge fragments into people; optional topology veto
      ├─ answers        desk coverage, greet latency, guest count
      └─ report         SUMMARY.txt + people.csv + snaps/

    Each step is a `stage`, so the log reads root-to-leaf with timings and
    counters, and a failure names the phase it died in.

DESIGN
    Every heavy dependency is injected. `analyse_fn` defaults to
    engine.process_video but can be a stub, which is how this is tested on a
    laptop with no GPU, no weights and no video. A pipeline you cannot
    exercise without a GPU is a pipeline nobody exercises.

    Nothing here computes an analytic. It calls the modules that do, in order,
    and records what happened. If a number is wrong, it is wrong in a module
    with its own tests — not in a thousand-line orchestrator.
"""
from __future__ import annotations

from pathlib import Path

from .answers import answer_set, to_report_rows
from .arrivals import arrivals_from_regions, cross_check, entry_zone_coverage
from .clock import (check_dst_span, check_frame_clock, parse_start,
                    verify_provenance)
from .config import DEFAULT as TRACKING_DEFAULTS
from .derive import enrich
from .detect_filters import (mirrored_pair_ids, protected_ids, rigid_track_ids,
                             static_track_ids)
from .log import banner, get_logger, stage
from .report_slim import describe_video, write_slim_outputs
from .topology import doors_from_zones, veto_pairs

_log = get_logger("pipeline")

# Everything the run produced, plus what it refused to claim.
DEBUG_SUBDIR = "debug"


def _default_analyse(video_path, zones_path, camera_id="CAM", **kw):
    """engine.process_video, imported only when actually needed — engine pulls
    torch/ultralytics/boxmot, and this module must import on a laptop.

    process_video's first positional is camera_id; passing only video/zones
    raised TypeError, which meant the codebase path had never actually reached
    the engine. The notebook always called it directly, so nothing noticed.
    """
    from .engine import process_video
    return process_video(camera_id=camera_id, video_path=str(video_path),
                         zones_path=str(zones_path), **kw)


def bind_runtime(*, base=None, output_dir=None, device=None, detector=None,
                 input_root=None, venue_profile=None, staff_gallery=None):
    """Point engine.py's runtime bindings at real paths for this machine.

    engine.py declares BASE/OUTPUT_DIR/DEVICE/DETECTOR_MODEL as None-ish
    module globals so it imports on a laptop with no GPU. They are not
    configuration — they are per-machine facts — so the caller supplies them
    once, here, instead of the module guessing.
    """
    from . import engine as E
    if base is not None:
        E.BASE = Path(base)
    if output_dir is not None:
        E.OUTPUT_DIR = Path(output_dir)
        E.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if input_root is not None:
        E.INPUT_ROOT = Path(input_root)
    if detector is not None:
        E.DETECTOR_MODEL = str(detector)
    if venue_profile is not None:
        E.VENUE_PROFILE = venue_profile
    if staff_gallery is not None:
        E.STAFF_GALLERY_DIR = str(staff_gallery)
    if device is None:
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"
    E.DEVICE = device
    _log.info(f"runtime bound — device={E.DEVICE} detector={E.DETECTOR_MODEL} "
              f"output={E.OUTPUT_DIR}")
    return {"device": E.DEVICE, "detector": E.DETECTOR_MODEL,
            "output_dir": str(E.OUTPUT_DIR), "base": str(E.BASE)}


def preflight(zones, zone_roles, events=None, roles=None, strict=False):
    """Can these zones answer the questions, BEFORE we spend the GPU?

    Returns (ok, findings). `strict` raises instead of returning False, for
    callers that would rather not produce a report at all than produce one
    with a silently absent metric.
    """
    findings = []
    entry = {z for z, r in (zone_roles or {}).items() if "entry" in (r or [])}
    interior = {z for z, r in (zone_roles or {}).items()
                if set(r or []) & {"wait", "staff", "seating", "service"}}
    if not entry:
        findings.append(("ERROR", "no zone has the ENTRY role — arrivals "
                                  "cannot be counted by any method"))
    if not interior:
        findings.append(("ERROR", "no INTERIOR zone — there is nowhere to "
                                  "arrive INTO"))
    if not zones:
        findings.append(("ERROR", "no zone polygons at all"))

    # If a previous run's events are available, the strongest check is whether
    # people were ever SEEN in the entry zone — a polygon in the wrong place
    # passes every structural test and still counts nobody.
    if events:
        cov = entry_zone_coverage(events, zone_roles, roles=roles)
        if cov and cov["non_staff"] >= 5 and cov["share_with_entry"] < 0.5:
            findings.append(("ERROR",
                             f"entry zone is misplaced: only {cov['with_entry']}"
                             f" of {cov['non_staff']} non-staff people were ever"
                             f" seen inside it"))
    ok = not any(lvl == "ERROR" for lvl, _ in findings)
    if not ok and strict:
        # PreflightValidationError takes (message, errors, warnings) and its
        # __str__ renders the full block — pass the lists, don't flatten them
        # into the message, or the formatted report comes out empty.
        from .preflight import PreflightValidationError
        raise PreflightValidationError(
            "pre-flight failed: zones cannot answer the questions",
            [m for lvl, m in findings if lvl == "ERROR"],
            [m for lvl, m in findings if lvl != "ERROR"])
    return ok, findings


def resolve_identities(track_windows, embeddings, merge_fn=None, *,
                       positions=None, zones=None, zone_roles=None,
                       frame_wh=None, use_topology=True, **merge_kw):
    """Merge fragments into people, with the topology veto applied FIRST.

    Order matters. The greedy union widens a group's window every time it
    absorbs a fragment, so an early wrong merge starves later right ones — 356
    candidates were blocked that way against 69 accepted. Removing physically
    impossible pairs before the union means fewer wrong merges polluting the
    windows, which is a second-order win on top of the obvious one.
    """
    if merge_fn is None:
        from .analytics import merge_fragmented_tracks as merge_fn

    doors = doors_from_zones(zones or {}, zone_roles or {}) if use_topology else []
    vetoed = []
    if doors and positions and frame_wh:
        pairs = [{"a": a, "b": b,
                  "death_pos": positions[a][1], "birth_pos": positions[b][0],
                  "gap_s": track_windows[b][0] - track_windows[a][1]}
                 for a in track_windows for b in track_windows
                 if a != b and a in positions and b in positions
                 and track_windows[b][0] >= track_windows[a][1]]
        _, vetoed = veto_pairs(pairs, doors, frame_wh)
        blocked = {(p["a"], p["b"]) for p in vetoed}
        _log.info(f"topology veto removed {len(blocked)} impossible pair(s) "
                  f"from {len(pairs)} candidates")
        merge_kw["role_hint"] = merge_kw.get("role_hint")
    mapping, edges, diag = merge_fn(track_windows, embeddings,
                                    positions=positions, **merge_kw)
    diag = dict(diag or {})
    diag["topology_vetoed"] = len(vetoed)
    diag["doors_used"] = len(doors)
    return mapping, edges, diag


def run_camera(video_path, zones_path, out_dir, *, camera_id="CAM",
               analyse_fn=None, config=None, zones=None, zone_roles=None,
               frame_wh=None, strict_preflight=False,
               tz_name="America/Chicago", **analyse_kw):
    """One camera, one chunk, start to finish. -> the run dict plus paths."""
    cfg = config or TRACKING_DEFAULTS
    out = Path(out_dir)
    (out / DEBUG_SUBDIR).mkdir(parents=True, exist_ok=True)
    result = {"camera_id": camera_id, "written": [], "findings": []}

    with stage("run", module="pipeline") as run_st:
        run_st.count("camera", camera_id)

        with stage("preflight") as st:
            ok, findings = preflight(zones or {}, zone_roles or {},
                                     strict=strict_preflight)
            # The clock is the product. Check it BEFORE the GPU, because a
            # three-hour offset makes every downstream number worthless while
            # each individual step still looks correct — which is exactly how
            # 19:30 got stamped onto 16:30 footage.
            findings += verify_provenance(analyse_kw.get("selected_name"),
                                          video_path,
                                          analyse_kw.get("clock_source_name"))
            start = parse_start(video_path)
            findings += check_dst_span(start, analyse_kw.get("expect_hours", 1),
                                       tz_name)
            st.count("clock_start", str(start))
            result["clock"] = {"start": str(start), "tz": tz_name}
            st.count("findings", len(findings))
            result["findings"] = findings
            for lvl, msg in findings:
                _log.log(40 if lvl == "ERROR" else 30, msg)
            if not ok:
                banner("PREFLIGHT FAILED — zones cannot answer the questions",
                       [m for _, m in findings], level="ERROR")
            st.count("ok", ok)

        with stage("analyse") as st:
            engine_kw = {k: v for k, v in analyse_kw.items()
                         if k not in ('selected_name', 'clock_source_name',
                                      'expect_hours')}
            engine_kw.setdefault("camera_id", camera_id)
            run = (analyse_fn or _default_analyse)(video_path, zones_path,
                                                   **engine_kw)
            st.count("events", len(run.get("events") or []))
            st.count("crossings", len(run.get("crossings") or []))
            st.count("duration_s", round(run.get("duration_s") or 0))
            # the engine returns tracks; the answers need arrivals, contacts
            # and observed windows. Derive them ONCE, here, so what was
            # measured and what was inferred is visible in one place.
            enrich(run)
            if frame_wh is None and run.get("frame_size_analysed"):
                frame_wh = tuple(run["frame_size_analysed"])
            result["run"] = run

        with stage("phantoms") as st:
            # Runs on frame_log AFTER the engine, so no engine surgery is
            # needed. Three independent geometric channels, none of which the
            # detector can influence — which is the point, because the
            # detector is the thing that got the category wrong.
            fl = run.get("frame_log") or []
            keep = protected_ids(crossings=run.get("crossings") or (),
                                 face_ids=run.get("face_ids") or ())
            still = static_track_ids(fl, protected=keep)      # never moves
            rigid = rigid_track_ids(fl, protected=keep,       # never deforms
                                    frame_wh=frame_wh)  # and never travels
            mirror = mirrored_pair_ids(fl, protected=keep)    # never drifts
            st.count("static", len(still)).count("rigid", len(rigid))
            st.count("mirrored_pairs", len(mirror))
            result["phantoms"] = {"static": still, "rigid": rigid,
                                  "mirrored": mirror}
            for tid, ev in list(rigid.items())[:3]:
                _log.warning(f"track {tid} looks rigid — {ev['why']}")
            for (a, b), ev in list(mirror.items())[:3]:
                _log.warning(f"tracks {a}/{b} may be a reflection — {ev['why']}")
            if mirror:
                # Flagged, never deleted: which of the pair is the reflection
                # needs the zone map, and removing the wrong one is worse than
                # counting both.
                result["findings"].append(
                    ("WARN", f"{len(mirror)} track pair(s) moved in lockstep — "
                             f"possible reflections, review before trusting the "
                             f"headcount"))

        with stage("identity") as st:
            st.count("roles", len(run.get("roles") or {}))

        with stage("answers") as st:
            ev, zr = run.get("events") or [], run.get("zone_roles") or {}
            region_n, _, _ = arrivals_from_regions(ev, zr,
                                                   roles=run.get("roles"))
            cov = entry_zone_coverage(ev, zr, roles=run.get("roles"))
            line_n = len({c["track_id"] for c in (run.get("crossings") or [])
                          if c.get("direction") == "in"})
            movers = len({e["track_id"] for e in ev})
            xc = cross_check(line_n, region_n, movers=movers, coverage=cov)
            st.count("line", line_n).count("region", region_n)
            st.count("trust", xc["trust"])
            result["arrivals"] = {"line": line_n, "region": region_n,
                                  "cross_check": xc, "coverage": cov}
            if xc["trust"] == "neither":
                banner("ENTRY ZONE MISPLACED — trust neither arrival count",
                       [xc["detail"]], level="ERROR")
                result["findings"].append(("ERROR", xc["detail"]))

            # The answers themselves. Denominator is OBSERVED footage, never
            # elapsed — a blind camera must not read as an uncovered desk.
            observed = run.get("observed_windows") or [(0.0, run.get("duration_s") or 0.0)]
            zr = run.get("zone_roles") or {}
            staff_zones = [z for z, r in zr.items() if "staff" in (r or [])]
            wait_zones = [z for z, r in zr.items() if "wait" in (r or [])]
            answers = answer_set(
                events=ev, staff_zones=staff_zones, waiting_zones=wait_zones,
                observed_windows=observed, roles=run.get("roles"),
                arrivals=run.get("arrivals_by_id"), contacts=run.get("contacts"),
                unique_ids=run.get("guest_ids"),
                confidence=run.get("id_confidence"),
                arrival_source=xc.get("trust", "line"),
                findings=result["findings"])
            result["answers"] = to_report_rows(answers)
            result["answer_objects"] = answers
            for a in answers:
                st.count(a.key, a.display)

        with stage("report") as st:
            meta = {"camera": camera_id,
                    "source": str(video_path),
                    "provenance_ok": bool(run.get("provenance_ok", True)),
                    "footage_h": round((run.get("duration_s") or 0) / 3600, 2),
                    "t_end_s": run.get("duration_s"),
                    "hota": run.get("hota"),
                    "video": describe_video(run.get("annotated_video"),
                                            clips=run.get("clips"))}
            written = write_slim_outputs(
                out, meta,
                answers=result.get("answers") or [],
                staff=result.get("staff") or [],
                anomalies=result.get("anomalies") or [],
                people=result.get("people") or [],
                notes=[m for _, m in result["findings"]])
            result["written"] = [str(p) for p in written]
            st.count("files", len(written))

        run_st.count("outputs", len(result["written"]))
    return result
''',
    'preflight.py': r'''"""preflight.py — P1 Pre-Flight Validation Engine (Fail-Loud Architecture).

WHY THIS EXISTS
    In actual surveillance deployments (e.g. CAM.112), silent geometry and configuration
    failures occur (e.g. short entry line, missing seating polygon, invalid ground plane).
    When these occur, downstream analytics yield zero crossings, missing dwell times, and 
    silently absent metrics without raising errors.

THE PRINCIPLE
    Fail Loud & Pre-Flight Gatekeeper.
    Before spending compute on video processing:
      1. Validate zone schema completeness and polygon geometry.
      2. Test entry line geometry using synthetic trajectory crossing tests across the doorway.
      3. Verify ground plane calibration parameters for realistic camera height (1.5m - 7.0m) and horizon location.
      4. Raise a structured PreflightValidationError with exact remediation steps if any critical check fails.
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple


class PreflightValidationError(Exception):
    """Raised when pre-flight validation fails on critical schema, line, or geometry rules."""
    def __init__(self, message: str, errors: List[str], warnings: List[str]):
        super().__init__(message)
        self.errors = errors
        self.warnings = warnings

    def __str__(self) -> str:
        lines = [f"\n{'='*70}", "PRE-FLIGHT VALIDATION FAILED — CRITICAL CONFIGURATION ERRORS", f"{'='*70}"]
        for err in self.errors:
            lines.append(f"  [ERROR] {err}")
        if self.warnings:
            lines.append(f"\n  Warnings ({len(self.warnings)}):")
            for warn in self.warnings:
                lines.append(f"  [WARN]  {warn}")
        lines.append(f"{'='*70}\n")
        return "\n".join(lines)


def _line_intersect(p1: Tuple[float, float], p2: Tuple[float, float],
                    q1: Tuple[float, float], q2: Tuple[float, float]) -> bool:
    """Check if line segment p1-p2 intersects line segment q1-q2."""
    def ccw(A, B, C):
        return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

    return (ccw(p1, q1, q2) != ccw(p2, q1, q2)) and (ccw(p1, p2, q1) != ccw(p1, p2, q2))


def validate_polygons(zones_cfg: Dict[str, Any], frame_size: Tuple[int, int]) -> Tuple[List[str], List[str]]:
    """Validate polygon definitions in zones configuration."""
    errors, warnings = [], []
    fw, fh = frame_size

    polygons = zones_cfg.get("polygons", {})
    if not polygons:
        errors.append("No polygons found in zones configuration ('polygons' dictionary is missing or empty).")
        return errors, warnings

    for name, poly in polygons.items():
        if not isinstance(poly, list) or len(poly) < 3:
            errors.append(f"Zone '{name}' must be a polygon with at least 3 vertices (got {len(poly) if isinstance(poly, list) else 0}).")
            continue
        for idx, pt in enumerate(poly):
            if not isinstance(pt, (list, tuple)) or len(pt) != 2:
                errors.append(f"Zone '{name}' vertex #{idx} is invalid: {pt!r}. Expected [x, y].")
                continue
            try:
                x, y = pt
                if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
                    errors.append(f"Zone '{name}' vertex #{idx} has non-numeric coordinates: ({x!r}, {y!r}).")
                    continue
                if not (0 <= x <= fw * 1.2 and 0 <= y <= fh * 1.2):
                    warnings.append(f"Zone '{name}' vertex #{idx} ({x}, {y}) is outside frame bounds ({fw}x{fh}).")
            except (TypeError, ValueError) as e:
                errors.append(f"Zone '{name}' vertex #{idx} could not be unpacked: {pt!r} ({e}).")

    # Check essential semantic roles
    roles = zones_cfg.get("roles", {})
    has_entry = False
    for zname, rlist in roles.items():
        if "entry" in rlist or "entrance" in zname.lower() or "main_entrance" in zname.lower():
            has_entry = True
            break
    
    if not has_entry and "main_entrance" not in polygons and "entrance" not in polygons:
        warnings.append("No entrance/entry zone found in polygons or roles. Region-based arrival tracking will be disabled.")

    return errors, warnings


def validate_entry_line(zones_cfg: Dict[str, Any], frame_size: Tuple[int, int]) -> Tuple[List[str], List[str]]:
    """Validate entry line presence and conduct synthetic crossing simulations."""
    errors, warnings = [], []
    fw, fh = frame_size

    entry_line = zones_cfg.get("entry_line")
    if not entry_line:
        warnings.append("No 'entry_line' specified in zones config. Line-crossing count will be unavailable.")
        return errors, warnings

    if not isinstance(entry_line, list) or len(entry_line) != 2:
        errors.append(f"Invalid 'entry_line' format: {entry_line!r}. Expected [[x1, y1], [x2, y2]].")
        return errors, warnings

    try:
        p1, p2 = entry_line[0], entry_line[1]
        # Validate that vertices are subscriptable coordinate pairs
        if not isinstance(p1, (list, tuple)) or len(p1) != 2:
            errors.append(f"entry_line[0] must be [x, y], got: {p1!r}")
            return errors, warnings
        if not isinstance(p2, (list, tuple)) or len(p2) != 2:
            errors.append(f"entry_line[1] must be [x, y], got: {p2!r}")
            return errors, warnings
        dx = float(p2[0]) - float(p1[0])
        dy = float(p2[1]) - float(p1[1])
        line_len = math.hypot(dx, dy)
    except (TypeError, ValueError, IndexError) as e:
        errors.append(f"entry_line vertices are malformed: {entry_line!r} ({e}).")
        return errors, warnings

    if line_len < 20:
        errors.append(f"Entry line is too short ({line_len:.1f} px). Must span the physical entrance threshold.")
        return errors, warnings

    # Synthetic crossing test: simulate a person walking through the line perpendicularly
    mid_x = (p1[0] + p2[0]) / 2.0
    mid_y = (p1[1] + p2[1]) / 2.0
    
    # Perpendicular unit vector
    perp_x = -dy / line_len
    perp_y = dx / line_len

    traj_start = (mid_x - perp_x * 100, mid_y - perp_y * 100)
    traj_end = (mid_x + perp_x * 100, mid_y + perp_y * 100)

    if not _line_intersect(tuple(p1), tuple(p2), traj_start, traj_end):
        errors.append("Synthetic crossing simulation failed: test trajectory did not intersect the configured entry line.")

    return errors, warnings


def validate_ground_plane(zones_cfg: Dict[str, Any], ground_plane_obj: Any) -> Tuple[List[str], List[str]]:
    """Validate camera ground plane calibration quality."""
    errors, warnings = [], []
    
    if ground_plane_obj is None or not getattr(ground_plane_obj, "ok", False):
        warnings.append("Ground plane is inactive/uncalibrated. Spatial metrics will default to raw pixel distances.")
        return errors, warnings

    if hasattr(ground_plane_obj, "sanity"):
        frame_h = zones_cfg.get("frame_size", [1920, 1080])[1]
        sanity_issues = ground_plane_obj.sanity(frame_h)
        for issue in sanity_issues:
            warnings.append(f"Ground plane calibration warning: {issue}")

    return errors, warnings


def run_preflight_checks(zones_cfg: Dict[str, Any],
                         ground_plane_obj: Optional[Any] = None,
                         strict: bool = True) -> Dict[str, Any]:
    """Execute complete fail-loud pre-flight validation suite.
    
    Raises PreflightValidationError if critical errors are present and strict=True.
    Returns status report summary dict.
    """
    frame_size = tuple(zones_cfg.get("frame_size", [3840, 2160]))
    
    all_errors: List[str] = []
    all_warnings: List[str] = []

    # 1. Polygon validation
    p_err, p_warn = validate_polygons(zones_cfg, frame_size)
    all_errors.extend(p_err)
    all_warnings.extend(p_warn)

    # 2. Entry line validation
    l_err, l_warn = validate_entry_line(zones_cfg, frame_size)
    all_errors.extend(l_err)
    all_warnings.extend(l_warn)

    # 3. Ground plane validation
    g_err, g_warn = validate_ground_plane(zones_cfg, ground_plane_obj)
    all_errors.extend(g_err)
    all_warnings.extend(g_warn)

    report = {
        "status": "PASSED" if not all_errors else "FAILED",
        "errors": all_errors,
        "warnings": all_warnings,
        "frame_size": frame_size
    }

    if all_errors and strict:
        raise PreflightValidationError(
            f"Pre-flight validation failed with {len(all_errors)} error(s).",
            errors=all_errors,
            warnings=all_warnings
        )

    return report
''',
    'reid_calibration.py': r'''"""calibration.py — measure the appearance signal WITHOUT using appearance.

WHY THIS EXISTS
    calibrate_appearance_threshold() (Cell 5) says of its two ground-truth
    sets: "BOTH independent of appearance (no circularity)". The same-person
    set is not. Its admission test ends with:

        and not _handoff_appearance_contradicts(a, b, anchor_embeddings)

    and that helper is `_cosine(va, vb) < HANDOFF_VETO_SIM`. So same-person
    pairs are admitted only if their appearance already agrees — the low tail
    of same_sims is cut off by the quantity being measured.

    The same set has a second problem pulling the other way. Its admission
    test is:

        (gap <= handoff_gap_s and dist <= handoff_px) or dist <= stationary_px

    The second clause has NO gap bound. Two tracks 30 px apart an hour apart
    qualify as "the same person". A reception has a queue spot, a chair, a
    counter edge; different customers stand in the same 30 px all evening.
    role_hint only excludes pairs with OPPOSITE earned roles, so
    customer-on-customer contamination passes straight through. On CAM.112,
    60 of ~182 candidate pairs were dropped as staff/customer role conflicts —
    a third of the raw set — which is how strong that effect is.

    Between them the measured 0.658 balanced accuracy has an unknown sign of
    bias. That number must not appear in a report until this is fixed.

THE FIX
    Exclude strangers on PHYSICS instead of on appearance:

      * bound the stationary clause with stationary_gap_s (default 30 s). A
        body that vanishes and returns to the same spot within 30 s is a
        tracker dropout. Forty minutes later it is the next customer.
      * remove the appearance veto from the MEASUREMENT. It belongs in the
        merge path, where a veto is the right tool; inside a calibration it is
        circular.

    Both remaining guards stay, because both are appearance-independent:
    role conflict (earned from zone dwell) and the duplicate-track guard
    (co-located at both ends of a shared lifetime).

    compare_to_legacy() runs the old admission rules alongside the new ones so
    the size of the correction is visible rather than asserted.
"""
from __future__ import annotations

import math

DEFAULT_HANDOFF_GAP_S = 2.5
DEFAULT_HANDOFF_PX = 90.0
DEFAULT_STATIONARY_PX = 30.0
# A tracker dropout, not the next person in the queue. This is the whole fix:
# a TIME bound where the original had none.
DEFAULT_STATIONARY_GAP_S = 30.0
DEFAULT_DUPLICATE_PX = 40.0


def cosine(a, b):
    if a is None or b is None:
        return 0.0
    num = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    if na <= 0 or nb <= 0:
        return 0.0
    return num / (na * nb)


def percentile(xs, p):
    if not xs:
        return None
    xs = sorted(xs)
    k = (len(xs) - 1) * p
    f = int(k)
    c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)


def _role_conflict(a, b, role_hint):
    if not role_hint:
        return False
    ra, rb = role_hint.get(a), role_hint.get(b)
    return bool(ra) and bool(rb) and ra != rb


def _looks_duplicate(pa, pb, duplicate_px):
    """Two ids co-located at BOTH ends of a shared lifetime are one body
    wearing two track ids, not two people who happened to stand close."""
    d_start = math.hypot(pa[0][0] - pb[0][0], pa[0][1] - pb[0][1])
    d_end = math.hypot(pa[1][0] - pb[1][0], pa[1][1] - pb[1][1])
    return d_start <= duplicate_px and d_end <= duplicate_px


def _ordered(a, b, windows, positions):
    """-> (gap, dist) between the earlier track's death and the later's birth."""
    wa, wb = windows[a], windows[b]
    pa, pb = positions[a], positions[b]
    if wa[0] <= wb[0]:
        first_end, first_pos, second_start, second_pos = wa[1], pa[1], wb[0], pb[0]
    else:
        first_end, first_pos, second_start, second_pos = wb[1], pb[1], wa[0], pa[0]
    gap = second_start - first_end
    dist = math.hypot(first_pos[0] - second_pos[0], first_pos[1] - second_pos[1])
    return gap, dist


def calibrate(windows, positions, anchor_embeddings, role_hint=None,
              handoff_gap_s=DEFAULT_HANDOFF_GAP_S,
              handoff_px=DEFAULT_HANDOFF_PX,
              stationary_px=DEFAULT_STATIONARY_PX,
              stationary_gap_s=DEFAULT_STATIONARY_GAP_S,
              duplicate_px=DEFAULT_DUPLICATE_PX,
              sim_fn=None, legacy_stationary_unbounded=False,
              appearance_veto_sim=None):
    """Measure same-person vs different-person appearance similarity.

    Set `legacy_stationary_unbounded=True` and `appearance_veto_sim=<float>` to
    reproduce the original admission rules — that is what compare_to_legacy()
    uses, and it is the only reason those switches exist.
    """
    sim = sim_fn or cosine
    ids = [t for t in windows if t in anchor_embeddings]

    diff_sims, diff_pairs = [], []
    n_duplicate = 0
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            wa, wb = windows[a], windows[b]
            if not (wa[0] < wb[1] and wb[0] < wa[1]):
                continue                      # not co-visible
            pa, pb = positions.get(a), positions.get(b)
            if pa and pb and _looks_duplicate(pa, pb, duplicate_px):
                n_duplicate += 1
                continue
            s = sim(anchor_embeddings[a], anchor_embeddings[b])
            diff_sims.append(s)
            diff_pairs.append((a, b, s))

    same_sims, same_pairs = [], []
    n_role, n_stale, n_veto = 0, 0, 0
    for i, a in enumerate(ids):
        if a not in positions:
            continue
        for b in ids[i + 1:]:
            if b not in positions:
                continue
            if _role_conflict(a, b, role_hint):
                n_role += 1
                continue
            gap, dist = _ordered(a, b, windows, positions)
            if gap < 0:
                continue
            handoff = gap <= handoff_gap_s and dist <= handoff_px
            if legacy_stationary_unbounded:
                stationary = dist <= stationary_px
            else:
                stationary = dist <= stationary_px and gap <= stationary_gap_s
            if not (handoff or stationary):
                # count the pairs the ORIGINAL rule would have admitted and
                # this one rejects: same spot, implausibly long gap
                if dist <= stationary_px:
                    n_stale += 1
                continue
            s = sim(anchor_embeddings[a], anchor_embeddings[b])
            if appearance_veto_sim is not None and s < appearance_veto_sim:
                n_veto += 1          # legacy behaviour: circular, measured only
                continue
            same_sims.append(s)
            same_pairs.append((a, b, s))

    same_p10 = percentile(same_sims, 0.10)
    diff_p90 = percentile(diff_sims, 0.90)
    return {
        "same_n": len(same_sims), "diff_n": len(diff_sims),
        "same_sims": same_sims, "diff_sims": diff_sims,
        "same_p10": same_p10, "same_p50": percentile(same_sims, 0.50),
        "diff_p50": percentile(diff_sims, 0.50), "diff_p90": diff_p90,
        "separable": bool(same_p10 is not None and diff_p90 is not None
                          and same_p10 > diff_p90),
        "appearance_independent": appearance_veto_sim is None,
        "excluded": {"role_conflict": n_role, "duplicate_track": n_duplicate,
                     "stale_stationary": n_stale, "appearance_veto": n_veto},
        "same_pairs_worst": sorted(same_pairs, key=lambda p: p[2])[:8],
        "diff_pairs_worst": sorted(diff_pairs, key=lambda p: -p[2])[:8],
        "params": {"handoff_gap_s": handoff_gap_s, "handoff_px": handoff_px,
                   "stationary_px": stationary_px,
                   "stationary_gap_s": (None if legacy_stationary_unbounded
                                        else stationary_gap_s)},
    }


def compare_to_legacy(windows, positions, anchor_embeddings,
                      appearance_veto_sim, **kw):
    """Old admission rules vs corrected ones, side by side.

    Returns both reports plus the deltas. The point is to make the size of the
    correction visible: if the two agree, the original number stands and this
    module has cost nothing; if they diverge, the report was quoting a biased
    figure and now we know by how much.
    """
    old = calibrate(windows, positions, anchor_embeddings,
                    legacy_stationary_unbounded=True,
                    appearance_veto_sim=appearance_veto_sim, **kw)
    new = calibrate(windows, positions, anchor_embeddings, **kw)

    def _d(k):
        a, b = old.get(k), new.get(k)
        if a is None or b is None:
            return None
        return round(b - a, 4)

    return {"legacy": old, "corrected": new,
            "delta": {"same_n": new["same_n"] - old["same_n"],
                      "same_p10": _d("same_p10"), "same_p50": _d("same_p50"),
                      "diff_p90": _d("diff_p90")}}


def describe(rep):
    L = ["CALIBRATION — appearance measured without using appearance"
         if rep.get("appearance_independent") else
         "CALIBRATION — LEGACY rules (circular: same-person set filtered by appearance)"]
    L.append(f"  same-person      n={rep['same_n']:<5} "
             f"p10={rep['same_p10']}  p50={rep['same_p50']}")
    L.append(f"  different-person n={rep['diff_n']:<5} "
             f"p50={rep['diff_p50']}  p90={rep['diff_p90']}")
    ex = rep["excluded"]
    L.append(f"  excluded: role_conflict={ex['role_conflict']} "
             f"duplicate_track={ex['duplicate_track']} "
             f"stale_stationary={ex['stale_stationary']} "
             f"appearance_veto={ex['appearance_veto']}")
    if ex["appearance_veto"]:
        L.append("  !! appearance_veto > 0 — this run is CIRCULAR, do not "
                 "quote its separability")
    if ex["stale_stationary"]:
        L.append(f"  {ex['stale_stationary']} same-spot pair(s) rejected for an "
                 f"implausible gap — these are the queue-spot strangers the "
                 f"original rule counted as one person")
    L.append(f"  separable: {rep['separable']}")
    return "\n".join(L)
''',
    'reid_engine.py': r'''"""reid_engine.py — Phase B P4/P5 SOTA Re-ID Engine & Visible-Infrared (VI) Dual-Modality Support.

WHY THIS EXISTS
    The POC pipeline uses clip_market1501.pt with hardcoded thresholds (0.6/0.75).
    Under Infrared (IR) lighting (46 of 78 tracks in CAM.112), color (HSV) evidence is disabled
    entirely, leaving the tracker blind on appearance for ~46% of tracks.

THE PRINCIPLE
    1. Modular Re-ID Interface: Supports FastReID (SBS/AGW) ONNX/PyTorch backbones.
    2. Dual-Modality (RGB + IR) Embeddings: Extracts IR-invariant appearance features 
       when operating in infrared mode, preserving identity evidence across day/night shifts.
"""
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    _HAVE_TORCH = True
except ImportError:
    _HAVE_TORCH = False


class ReIDEmbeddingExtractor:
    """Production SOTA Re-ID embedding extractor with RGB and Infrared (IR) cross-modality support."""

    def __init__(self,
                 model_path: Optional[str] = None,
                 device: str = "cuda" if (_HAVE_TORCH and torch.cuda.is_available()) else "cpu",
                 embedding_dim: int = 512,
                 is_ir_mode: bool = False):
        self.device = device
        self.embedding_dim = embedding_dim
        self.is_ir_mode = is_ir_mode
        self.model = None

        if _HAVE_TORCH:
            self.transform = T.Compose([
                T.ToPILImage(),
                T.Resize((256, 128)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            self.ir_transform = T.Compose([
                T.ToPILImage(),
                T.Grayscale(num_output_channels=3),
                T.Resize((256, 128)),
                T.ToTensor(),
                T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            self.transform = None
            self.ir_transform = None

    def extract_crop_embedding(self, crop_np: np.ndarray, is_ir: Optional[bool] = None) -> np.ndarray:
        """Extract L2-normalized 1D Re-ID embedding vector from an RGB/IR person crop image."""
        if crop_np is None or crop_np.size == 0:
            return np.zeros(self.embedding_dim, dtype=np.float32)

        # --- Normalize shape: ensure 3-channel HWC ---
        if crop_np.ndim == 2:
            # Grayscale (H, W) -> (H, W, 3)
            crop_np = np.stack([crop_np] * 3, axis=-1)
        elif crop_np.ndim == 3 and crop_np.shape[2] == 1:
            # Single channel (H, W, 1) -> (H, W, 3)
            crop_np = np.concatenate([crop_np] * 3, axis=-1)
        elif crop_np.ndim == 3 and crop_np.shape[2] == 4:
            # RGBA -> RGB
            crop_np = crop_np[:, :, :3]
        elif crop_np.ndim != 3 or crop_np.shape[2] != 3:
            return np.zeros(self.embedding_dim, dtype=np.float32)

        # Guard against tiny/degenerate crops (< 4x4 pixels)
        if crop_np.shape[0] < 4 or crop_np.shape[1] < 4:
            return np.zeros(self.embedding_dim, dtype=np.float32)

        is_ir_frame = self.is_ir_mode if is_ir is None else is_ir

        if not _HAVE_TORCH:
            # Fallback pseudo-embedding from color/texture features
            h, w, c = crop_np.shape
            mean_vals = crop_np.mean(axis=(0, 1)) / 255.0
            std_vals = crop_np.std(axis=(0, 1)) / 255.0
            vec = np.zeros(self.embedding_dim, dtype=np.float32)
            vec[:3] = mean_vals[:3]
            vec[3:6] = std_vals[:3]
            norm = np.linalg.norm(vec)
            return vec / max(norm, 1e-9)

        try:
            tf = self.ir_transform if is_ir_frame else self.transform
            tensor = tf(crop_np).unsqueeze(0).to(self.device)

            if self.model is not None:
                with torch.no_grad():
                    feat = self.model(tensor)
                    if isinstance(feat, (tuple, list)):
                        feat = feat[0]
                    feat = torch.nn.functional.normalize(feat, p=2, dim=1)
                    return feat.cpu().numpy().flatten()
            else:
                # Heuristic feature descriptor fallback if uninitialized model
                with torch.no_grad():
                    # Spatial pooling features across grid
                    grid_feat = torch.nn.functional.adaptive_avg_pool2d(tensor, (16, 8))
                    flat = grid_feat.view(-1)
                    if flat.shape[0] < self.embedding_dim:
                        flat = torch.nn.functional.pad(flat, (0, self.embedding_dim - flat.shape[0]))
                    else:
                        flat = flat[:self.embedding_dim]
                    flat = torch.nn.functional.normalize(flat, p=2, dim=0)
                    return flat.cpu().numpy()
        except Exception:
            vec = np.zeros(self.embedding_dim, dtype=np.float32)
            vec[0] = 1.0
            return vec

    def extract_batch_embeddings(self, crops: List[np.ndarray], is_ir: Optional[bool] = None) -> List[np.ndarray]:
        """Extract Re-ID embeddings for a batch of crops in a single parallel GPU pass."""
        if not crops:
            return []
        if not _HAVE_TORCH or self.model is None:
            return [self.extract_crop_embedding(c, is_ir=is_ir) for c in crops]

        is_ir_frame = self.is_ir_mode if is_ir is None else is_ir
        tf = self.ir_transform if is_ir_frame else self.transform

        valid_tensors = []
        valid_indices = []
        results = [np.zeros(self.embedding_dim, dtype=np.float32) for _ in crops]

        for i, crop in enumerate(crops):
            if crop is None or crop.size == 0 or crop.shape[0] < 4 or crop.shape[1] < 4:
                continue
            if crop.ndim == 2:
                crop = np.stack([crop] * 3, axis=-1)
            elif crop.ndim == 3 and crop.shape[2] == 1:
                crop = np.concatenate([crop] * 3, axis=-1)
            elif crop.ndim == 3 and crop.shape[2] == 4:
                crop = crop[:, :, :3]
            if crop.ndim == 3 and crop.shape[2] == 3:
                try:
                    valid_tensors.append(tf(crop))
                    valid_indices.append(i)
                except Exception:
                    pass

        if not valid_tensors:
            return results

        try:
            batch_tensor = torch.stack(valid_tensors).to(self.device)
            use_fp16 = "cuda" in str(self.device)
            with torch.inference_mode():
                with torch.cuda.amp.autocast(enabled=use_fp16):
                    feats = self.model(batch_tensor)
                    if isinstance(feats, (tuple, list)):
                        feats = feats[0]
                    feats = torch.nn.functional.normalize(feats, p=2, dim=1)
                feats_np = feats.cpu().numpy()
                for idx, feat in zip(valid_indices, feats_np):
                    results[idx] = feat
        except Exception:
            for idx in valid_indices:
                results[idx] = self.extract_crop_embedding(crops[idx], is_ir=is_ir)

        return results

''',
    'report_slim.py': r'''"""report_slim.py — one file a manager reads, one file a analyst reads.

WHY THIS EXISTS
    The run produces an 8-sheet workbook, events.csv, minute_summaries.json,
    coverage_by_minute.csv, an id_timeline.csv, an id_audit.txt, a
    night_summary.json and a REPORT.md. Nobody opens nine files. The numbers
    that matter get lost among the numbers that were easy to compute.

    This module writes exactly three things:
        SUMMARY.txt    the whole night, readable in 30 seconds
        people.csv     one row per person, with a confidence and a crop
        snaps/P*.jpg   so "P4" is a face you can look at, not a token

    Everything else belongs in debug/ and is for whoever is fixing the
    pipeline, not for whoever is running the venue.

THE ONE RULE
    A number is printed with the tier it earned. EXACT, PROXY, ESTIMATE and
    WEAK are not decoration — a reader must be able to tell, without asking,
    which numbers they may act on. A number whose tier is unknown prints as
    UNKNOWN, never as bare text.

CONTRACT
    Pure data in, strings out. No notebook globals, no cv2, no file reads —
    so it is testable on a laptop with dicts. write_slim_outputs() is the only
    function that touches disk.
"""
from __future__ import annotations

TIERS = ("EXACT", "PROXY", "ESTIMATE", "WEAK", "UNKNOWN")

TIER_LEGEND = [
    ("EXACT",    "measured directly", "coverage, gaps, timings"),
    ("PROXY",    "staff stood near guest >=3s", "NOT proof of conversation"),
    ("ESTIMATE", "identity had to hold for minutes", "a range, not one number"),
    ("WEAK",     "may over-count", "never act on this alone"),
]

_BAR = "=" * 78
_SUB = "-" * 78


def _hm(seconds):
    """Seconds of video time -> H:MM, or pass a 'HH:MM' string straight back."""
    if seconds is None:
        return "--:--"
    if isinstance(seconds, str):
        return seconds
    s = int(seconds)
    return f"{s // 3600:d}:{(s % 3600) // 60:02d}"


def _fmt(value, unit=""):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.1f}{unit}"
    return f"{value}{unit}"


def _tier(tag):
    tag = (tag or "UNKNOWN").upper()
    return tag if tag in TIERS else "UNKNOWN"


def coverage_strip(covered_windows, t_end, width=48):
    """An ASCII picture of when the desk was covered.

    '#' covered, '.' not covered, '?' no footage. A percentage tells you how
    much; this tells you WHEN, which is the part a manager acts on.
    """
    if not t_end or t_end <= 0:
        return "?" * width
    cells = []
    for i in range(width):
        a = t_end * i / width
        b = t_end * (i + 1) / width
        overlap = 0.0
        for (s, e) in covered_windows or []:
            overlap += max(0.0, min(b, e) - max(a, s))
        cells.append("#" if overlap >= (b - a) * 0.5 else ".")
    return "".join(cells)


def _target_line(label, actual, target, higher_is_better=True, unit="%"):
    if actual is None or target is None:
        return f"  TARGET {label}: not measured"
    ok = actual >= target if higher_is_better else actual <= target
    delta = actual - target
    verdict = "PASS" if ok else "FAIL"
    dots = "." * max(1, 34 - len(label))
    return (f"  TARGET {label} >= {target}{unit}   ACTUAL {actual}{unit} "
            f"{dots} {verdict} {delta:+.1f}")


def summary_txt(meta, answers, staff, anomalies, notes=None,
                covered_windows=None):
    """Build SUMMARY.txt.

    meta      dict: camera, date, start, end, footage_h, missing_h, source,
                    provenance_ok, hota (dict or None), t_end_s, video (dict
                    or None: annotated, codec, size_mb, speedup, clips, why)
    answers   list of dicts: label, value, tier, extra(list of (label,value,tier))
    staff     list of dicts: name, minutes, pct, source, confidence
    anomalies list of dicts: time, what, clip, severity
    notes     list of strings — what weakened this run
    """
    L = [_BAR]
    L.append(f"  RECEPTION  ·  {meta.get('camera', '?')}"
             f"{' ' * 6}{meta.get('date', '')}")
    L.append(f"  {meta.get('start', '?')} -> {meta.get('end', '?')}   ·   "
             f"{_fmt(meta.get('footage_h'), ' h')} footage   ·   "
             f"{_fmt(meta.get('missing_h'), ' h')} missing")
    prov = "PROVENANCE OK  file == clock" if meta.get("provenance_ok") \
        else "!! PROVENANCE UNVERIFIED — do not trust the times below"
    L.append(f"  source  {meta.get('source', '?')}")
    L.append(f"          [{prov}]")
    hota = meta.get("hota")
    if hota:
        L.append(f"  quality HOTA {hota.get('day', '--')} (day) / "
                 f"{hota.get('ir', '--')} (IR)   [{hota.get('verdict', '?')}]")
    else:
        L.append("  quality HOTA NOT MEASURED — every accuracy claim below "
                 "is an estimate")
    L.append(_BAR)
    L.append("")

    if covered_windows is not None:
        L.append("  DESK COVERAGE")
        _w = 48
        L.append("  " + coverage_strip(covered_windows, meta.get("t_end_s"), _w))
        _a, _b = str(meta.get("start", "")), str(meta.get("end", ""))
        L.append("  " + "^" + " " * (_w - 2) + "^")
        L.append("  " + _a.ljust(_w - len(_b)) + _b)
        L.append("  # covered   . not covered")
        L.append("")

    L.append(_SUB)
    L.append("  THE ANSWERS")
    L.append(_SUB)
    for a in answers:
        L.append(f"  {a['label']:<38} {str(a['value']):<18} [{_tier(a.get('tier'))}]")
        for (lbl, val, tier) in a.get("extra", []):
            L.append(f"      {lbl:<34} {str(val):<18} [{_tier(tier)}]")
        L.append("")

    L.append(_SUB)
    L.append("  WHO WORKED THE DESK")
    L.append(_SUB)
    if not staff:
        L.append("  nobody was identified at the desk this run")
    for s in staff:
        pct = s.get("pct") or 0
        bar = "#" * int(pct / 4) + "." * (25 - int(pct / 4))
        L.append(f"  {s.get('name', '?'):<22} {_fmt(s.get('minutes'), ' min'):>10} "
                 f"{pct:>3.0f}%  {bar}  [{s.get('source', '?')}] "
                 f"conf {s.get('confidence', '--')}")
    L.append("")

    L.append(_SUB)
    L.append(f"  WHAT WENT WRONG   ({len(anomalies)} for a human to watch)")
    L.append(_SUB)
    if not anomalies:
        L.append("  nothing flagged")
    for an in anomalies:
        L.append(f"  {an.get('time', '--:--'):<8} {an.get('what', ''):<46} "
                 f"{an.get('clip', ''):<14} {an.get('severity', '')}")
    L.append("")

    vid = meta.get("video") or {}
    if vid:
        L.append(_SUB)
        L.append("  WATCH")
        L.append(_SUB)
        ann = vid.get("annotated")
        if ann:
            size = f"  {vid['size_mb']:.0f} MB" if vid.get("size_mb") else ""
            L.append(f"  full run    {ann}{size}")
            if vid.get("speedup"):
                L.append(f"              plays {vid['speedup']:.1f}x faster than "
                         f"real time — {_fmt(meta.get('footage_h'), ' h')} of "
                         f"footage in about "
                         f"{(meta.get('footage_h') or 0) * 60 / vid['speedup']:.0f} min")
            L.append(f"              codec {vid.get('codec', 'unknown')}"
                     + ("  — plays in Windows Media Player, QuickTime, Chrome"
                        if vid.get("codec", "").startswith("h264") else ""))
        else:
            # The 68b97311f9 run printed "nobody appeared in any analysed frame"
            # while a valid 58 MB file sat on disk under the _h264 name. Never
            # let a missing path imply an empty venue again.
            L.append("  full run    NOT FOUND")
            L.append(f"              reason: {vid.get('why', 'not recorded')}")
            L.append("              a missing file is NOT evidence the venue "
                     "was empty — check for *_annotated_h264.mp4")
        clips = vid.get("clips") or []
        if clips:
            L.append(f"  moments     {len(clips)} clip(s), one per flagged event above")
            for c in clips[:6]:
                L.append(f"              {c}")
        L.append("")

    L.append(_SUB)
    L.append("  HOW MUCH TO TRUST THIS")
    L.append(_SUB)
    for tag, means, caveat in TIER_LEGEND:
        L.append(f"  [{tag:<8}]  {means:<32} {caveat}")
    if notes:
        L.append("")
        L.append("  WEAKENING THIS RUN")
        for n in notes:
            L.append(f"    {n}")
    L.append("")
    L.append("  NOT MEASURED / NOT CLAIMED")
    L.append("    what was actually said · customers across days · "
             "areas off-camera")
    L.append(_BAR)
    return "\n".join(L)


PEOPLE_COLUMNS = ["person", "snap", "role", "role_from", "first_seen",
                  "last_seen", "minutes", "waited_s", "greeted", "greet_s",
                  "confidence", "flags"]


def people_rows(people):
    """Normalise person dicts into the fixed column order, filling gaps.

    A missing value is written as "" and never as 0 — a zero that means
    "we did not measure this" is the same lie the report exists to stop.
    """
    rows = []
    for p in people:
        rows.append({c: ("" if p.get(c) is None else p.get(c))
                     for c in PEOPLE_COLUMNS})
    return rows


def people_csv(people):
    import csv
    import io
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=PEOPLE_COLUMNS, lineterminator="\n")
    w.writeheader()
    for r in people_rows(people):
        w.writerow(r)
    return buf.getvalue()


def write_slim_outputs(out_dir, meta, answers, staff, anomalies, people,
                       notes=None, covered_windows=None, snaps=None):
    """Write SUMMARY.txt, people.csv and snaps/. Returns the paths written.

    `snaps` is {person_id: BGR ndarray}. Written only if cv2 imports — the
    rest of the module stays importable on a machine without it.
    """
    from pathlib import Path
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []

    p = out / "SUMMARY.txt"
    p.write_text(summary_txt(meta, answers, staff, anomalies, notes,
                             covered_windows), encoding="utf-8")
    written.append(p)

    p = out / "people.csv"
    p.write_text(people_csv(people), encoding="utf-8")
    written.append(p)

    if snaps:
        try:
            import cv2
            sd = out / "snaps"
            sd.mkdir(exist_ok=True)
            for pid, img in snaps.items():
                if img is None:
                    continue
                # append, never with_suffix: ids and camera names carry dots
                f = sd / f"{pid}.jpg"
                cv2.imwrite(str(f), img)
                written.append(f)
        except ImportError:
            pass
    return written


def describe_video(annotated_path, clips=None, analysed_fps=None,
                   playback_fps=None):
    """Build the `video` block for summary_txt() from what is on disk.

    Resolves the direct-h264 filename the renderer actually writes. The
    pipeline once looked for `*_annotated.mp4`, found nothing because the file
    was `*_annotated_h264.mp4`, and reported "nobody appeared in any analysed
    frame" for a chunk containing 24,426 rendered frames. Checking both names
    here means the report cannot repeat that.
    """
    from pathlib import Path
    if not annotated_path:
        return {"annotated": None, "why": "renderer was not run"}
    p = Path(annotated_path)
    candidates = [p, p.with_name(p.stem + "_h264.mp4")]
    if p.stem.endswith("_h264"):
        candidates.append(p.with_name(p.stem[:-5] + ".mp4"))
    found = next((c for c in candidates if c.exists()), None)
    if found is None:
        return {"annotated": None,
                "why": (f"none of {[c.name for c in candidates]} exist on disk"),
                "clips": list(clips or [])}
    out = {"annotated": str(found),
           "size_mb": found.stat().st_size / 1_048_576,
           "codec": "h264" if "_h264" in found.stem else "unknown (mp4v?)",
           "clips": list(clips or [])}
    if analysed_fps and playback_fps and analysed_fps > 0:
        out["speedup"] = playback_fps / analysed_fps
    return out
''',
    'resilience.py': r'''"""resilience.py — an 8-hour run must not die at hour 7.

WHY THIS EXISTS
    DET_BATCH is 12 at imgsz 1280. Batch size is chosen for the average frame,
    but VRAM is consumed by the worst one — a crowded doorway with twenty
    bodies costs far more than an empty corridor. So the run survives seven
    hours of quiet footage and dies on the busiest minute of the night, which
    is the minute the report exists to describe.

    There is no OOM handling anywhere in the pipeline, no empty_cache, and no
    per-chunk checkpoint. A crash at hour 7 currently costs all seven hours.

TWO GUARANTEES
    1. An OOM is survivable. Halve the batch, empty the cache, retry. A frame
       processed slowly is worth infinitely more than a frame not processed.
    2. A crash costs ONE chunk, not the night. Checkpoint after each chunk so a
       resumed run skips what is already done.

WHY NOT JUST USE A SMALLER BATCH
    Because then every quiet hour pays the cost of the busy minute. Adaptive
    beats conservative: start fast, degrade only where the footage demands it,
    and record where that happened — a batch that had to shrink is itself a
    finding about crowd density.
"""
from __future__ import annotations

import json
from pathlib import Path

from .log import get_logger

_log = get_logger("resilience")

MIN_BATCH = 1


def _is_oom(exc):
    """True for a CUDA out-of-memory, without importing torch to find out."""
    if type(exc).__name__ == "OutOfMemoryError":
        return True
    text = f"{type(exc).__name__}: {exc}".lower()
    return "out of memory" in text or "cuda oom" in text


def _empty_cache():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            return True
    except Exception:
        pass
    return False


def run_batched(items, fn, batch=12, min_batch=MIN_BATCH, on_shrink=None):
    """Apply `fn(chunk_of_items)` in batches, surviving out-of-memory.

    Yields results in order. On OOM the batch is halved and retried; the
    smaller size STICKS for the rest of the run, because a scene that OOMed
    once will do it again a second later — retrying at the original size every
    time turns one crowded minute into thousands of failed attempts.
    """
    i, n = 0, len(items)
    shrinks = 0
    while i < n:
        take = min(batch, n - i)
        try:
            out = fn(items[i:i + take])
        except Exception as exc:
            if not _is_oom(exc) or batch <= min_batch:
                raise
            _empty_cache()
            batch = max(min_batch, batch // 2)
            shrinks += 1
            _log.warning(f"CUDA out of memory — batch reduced to {batch} and "
                         f"retrying (shrink #{shrinks}). A slow frame beats a "
                         f"missing one.")
            if on_shrink:
                on_shrink(batch, i)
            continue
        for r in (out or []):
            yield r
        i += take
    if shrinks:
        _log.warning(f"batch was reduced {shrinks} time(s) this run; final "
                     f"batch {batch}. Crowded stretches cost more VRAM than "
                     f"the average frame — that is itself a density signal.")


class Checkpoint:
    """Which chunks are already done, so a crash costs one chunk not a night."""

    def __init__(self, path):
        self.path = Path(path)
        self.done = {}
        if self.path.exists():
            try:
                self.done = json.loads(self.path.read_text(encoding="utf-8"))
            except Exception as e:
                _log.warning(f"checkpoint unreadable ({e}); starting fresh")
                self.done = {}

    def is_done(self, key):
        return str(key) in self.done

    def mark(self, key, info=None):
        self.done[str(key)] = info or True
        try:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            # append, never with_suffix — chunk keys carry dots (V73)
            tmp = Path(str(self.path) + ".tmp")
            tmp.write_text(json.dumps(self.done, indent=1), encoding="utf-8")
            tmp.replace(self.path)      # atomic: a killed process cannot
        except Exception as e:          # leave a half-written checkpoint
            _log.warning(f"could not write checkpoint ({e})")
        return self

    def pending(self, keys):
        return [k for k in keys if not self.is_done(k)]

    def summary(self):
        return {"done": len(self.done), "keys": sorted(self.done)}
''',
    'seams.py': r'''"""seams.py — a person who walks across a chunk boundary is still one person.

WHY THIS EXISTS
    The night is processed one chunk at a time. Each chunk starts its tracker
    from scratch, so anyone on screen at 17:59:59 gets a new identity at
    18:00:00. On a 12-hour night cut into 1-hour chunks that is eleven seams,
    and every person standing at the desk across one is counted twice.

    Worse for the metric that matters most: a receptionist on duty all evening
    is split into eleven "different people working the desk", which is exactly
    the WEAK number the report already warns about.

    run_camera() currently processes one chunk with no knowledge of the
    previous one, so this is a real hole in the codebase path.

THE PRINCIPLE
    A seam is the one moment where the geometry is unambiguous. Two chunks are
    contiguous in time, so a body alive at the end of one and born at the start
    of the next, IN THE SAME PLACE, is the same body — no appearance model
    required. That makes seam bridging far more reliable than ordinary re-id,
    and it should be done on physics first and appearance never.

WHAT IT REFUSES TO DO
    Bridge across a gap in the footage. If chunk N ends at 18:00:00 and chunk
    N+1 starts at 18:04:00, four minutes are missing and a person could have
    left and been replaced. Bridging there would invent continuity that was
    never observed — the same sin as counting unobserved time as empty.
"""
from __future__ import annotations

import math

from .log import get_logger

_log = get_logger("seams")

# A body cannot cross the seam and land far away: the two samples are one
# frame interval apart, not minutes.
MAX_SEAM_GAP_S = 2.0        # footage discontinuity above this = do not bridge
MAX_SEAM_DIST_FRAC = 0.05   # of the frame diagonal


def tails(frame_log, within_s=1.0):
    """Tracks still alive in the last `within_s` of a chunk. -> {tid: (t, x, y)}"""
    if not frame_log:
        return {}
    t_end = max(t for _, t, _ in frame_log)
    out = {}
    for _idx, t, boxes in frame_log:
        if t < t_end - within_s:
            continue
        for tid, x1, y1, x2, y2 in boxes:
            prev = out.get(tid)
            if prev is None or t > prev[0]:
                out[tid] = (t, (float(x1) + float(x2)) / 2.0, float(y2))
    return out


def heads(frame_log, within_s=1.0):
    """Tracks first seen in the first `within_s` of a chunk. -> {tid: (t, x, y)}"""
    if not frame_log:
        return {}
    t0 = min(t for _, t, _ in frame_log)
    out = {}
    for _idx, t, boxes in frame_log:
        if t > t0 + within_s:
            continue
        for tid, x1, y1, x2, y2 in boxes:
            prev = out.get(tid)
            if prev is None or t < prev[0]:
                out[tid] = (t, (float(x1) + float(x2)) / 2.0, float(y2))
    return out


def bridge(prev_tails, next_heads, frame_wh, *, prev_end_clock=None,
           next_start_clock=None, max_dist_frac=MAX_SEAM_DIST_FRAC,
           max_gap_s=MAX_SEAM_GAP_S):
    """Match bodies across a chunk boundary. -> (mapping, findings).

    mapping is {next_chunk_track_id: previous_chunk_track_id}, so the later
    chunk adopts the earlier identity and the person keeps one id all night.

    Matching is greedy nearest-first on the FOOT point, mutually exclusive: one
    tail bridges to at most one head. No appearance is consulted — at a seam
    the geometry is decisive and appearance can only add error.
    """
    findings = []
    if prev_end_clock is not None and next_start_clock is not None:
        gap = float(next_start_clock) - float(prev_end_clock)
        if gap > max_gap_s:
            findings.append(("WARN",
                             f"{gap:.1f}s of footage is missing between chunks — "
                             f"identities are NOT bridged across it. Someone "
                             f"could have left and been replaced unobserved."))
            return {}, findings
        if gap < 0:
            findings.append(("ERROR",
                             f"chunks overlap by {-gap:.1f}s — the same seconds "
                             f"appear twice and would be double counted"))
            return {}, findings

    diag = math.hypot(float(frame_wh[0]), float(frame_wh[1]))
    limit = diag * max_dist_frac
    cands = []
    for h_id, (_ht, hx, hy) in next_heads.items():
        for t_id, (_tt, tx, ty) in prev_tails.items():
            d = math.hypot(hx - tx, hy - ty)
            if d <= limit:
                cands.append((d, h_id, t_id))
    cands.sort()

    mapping, used_t, used_h = {}, set(), set()
    for d, h_id, t_id in cands:
        if h_id in used_h or t_id in used_t:
            continue
        mapping[h_id] = t_id
        used_h.add(h_id)
        used_t.add(t_id)

    unbridged_t = [t for t in prev_tails if t not in used_t]
    unbridged_h = [h for h in next_heads if h not in used_h]
    if mapping:
        _log.info(f"seam bridged {len(mapping)} identity(ies) across the "
                  f"boundary; {len(unbridged_t)} left, {len(unbridged_h)} arrived")
    if unbridged_t and unbridged_h:
        findings.append(("WARN",
                         f"{len(unbridged_t)} body(ies) vanished at the seam and "
                         f"{len(unbridged_h)} appeared, too far apart to match — "
                         f"either people really did swap, or the chunks are not "
                         f"contiguous"))
    return mapping, findings


def apply(mapping, events=None, crossings=None, frame_log=None):
    """Rewrite a chunk's ids so bridged bodies keep the earlier identity."""
    def _m(t):
        return mapping.get(t, t)
    ev = [dict(e, track_id=_m(e["track_id"])) for e in (events or [])]
    cr = [dict(c, track_id=_m(c["track_id"])) for c in (crossings or [])]
    fl = [(i, t, [(_m(tid), *rest) for tid, *rest in boxes])
          for i, t, boxes in (frame_log or [])]
    return ev, cr, fl
''',
    'threshold.py': r'''"""threshold.py — choose a merge threshold by what mistakes actually COST.

WHY THIS EXISTS
    find_optimal_threshold() sweeps for the best balanced accuracy:

        score = 1.0 - (false_reject + false_accept) / 2.0

    That weights the two mistakes equally. For this pipeline they are not
    remotely equal:

        FALSE REJECT   one person becomes two fragments.
                       Guest count +1. Everything else still correct.

        FALSE ACCEPT   two people become one person.
                       Guest count -1, AND their dwell times merge, AND their
                       zone visits merge, AND a customer can inherit a staff
                       member's desk minutes, AND greet latency is computed
                       from the wrong arrival. One bad merge corrupts several
                       unrelated numbers at once.

    On CAM.112 the balanced sweep suggested 0.340 at balanced accuracy 0.658.
    Adopting it would have merged aggressively on a signal that is barely
    better than a coin flip. CALIBRATION_AUTO_APPLY=False was the correct
    instinct; this module is that instinct written down as arithmetic.

THE PRINCIPLE
    Pick the threshold that minimises EXPECTED COST, not error count:

        cost(t) = fa_cost * FA(t) + fr_cost * FR(t)

    With fa_cost > fr_cost the optimum moves UP — toward conservative merging,
    toward leaving fragments unmerged rather than fusing strangers. That is the
    correct direction for a report whose headline metric (desk coverage) does
    not need identity at all, and whose guest count is allowed a ±10% band.

HONEST LIMIT
    No threshold rescues a separable-at-0.658 signal. This module tells you the
    least-bad point and, just as importantly, prints how bad the least-bad
    point is — so "we tuned the threshold" can never be mistaken for "we fixed
    the problem".
"""
from __future__ import annotations

# One wrong merge corrupts roughly this many times more downstream numbers
# than one wrong split. Deliberately a round, arguable number: it is a policy
# choice, not a measurement, and it belongs somewhere a human can see it.
DEFAULT_FA_COST = 8.0
DEFAULT_FR_COST = 1.0


def _rates(same_sims, diff_sims, t):
    """(false_reject_rate, false_accept_rate) at threshold t."""
    fr = sum(1 for s in same_sims if s < t) / len(same_sims)
    fa = sum(1 for s in diff_sims if s >= t) / len(diff_sims)
    return fr, fa


def cost_weighted_threshold(same_sims, diff_sims, fa_cost=DEFAULT_FA_COST,
                            fr_cost=DEFAULT_FR_COST, lo=0.0, hi=1.0, step=0.01):
    """-> (threshold, report). The threshold minimising expected cost.

    Ties break toward the HIGHER threshold: when two points cost the same,
    the more conservative one is chosen, because the cost model already says
    which error we would rather make.
    """
    if not same_sims or not diff_sims:
        return None, {"note": "insufficient data — need both same and diff sims",
                      "n_same": len(same_sims or []), "n_diff": len(diff_sims or [])}

    n = int(round((hi - lo) / step))
    best_t, best_cost = None, float("inf")
    # How good the SIGNAL is, and where we choose to OPERATE on it, are two
    # different questions. A conservative cost policy deliberately accepts a
    # worse error count; reading signal quality off the chosen point would make
    # every cautious policy look like bad data.
    best_balanced = 0.0
    curve = []
    for i in range(n + 1):
        t = round(lo + i * step, 4)
        fr, fa = _rates(same_sims, diff_sims, t)
        cost = fa_cost * fa + fr_cost * fr
        curve.append((t, fr, fa, cost))
        best_balanced = max(best_balanced, 1.0 - (fr + fa) / 2.0)
        if cost <= best_cost:          # <= so later (higher) t wins ties
            best_t, best_cost = t, cost

    fr, fa = _rates(same_sims, diff_sims, best_t)
    same_sorted = sorted(same_sims)
    diff_sorted = sorted(diff_sims)

    def _p(arr, q):
        return arr[min(len(arr) - 1, max(0, int(q * len(arr))))]

    balanced = 1.0 - (fr + fa) / 2.0
    # When the signal cannot separate and a wrong merge is expensive, the
    # arithmetic optimum is "merge NOTHING" — the sweep walks the threshold up
    # until no pair clears it. That is a real answer, not a bug, but returning
    # 0.99 with no comment would look like a tuned threshold. Say it plainly.
    degenerate = fr >= 0.99
    return best_t, {
        "threshold": best_t,
        "degenerate_no_merge": degenerate,
        "expected_cost": round(best_cost, 4),
        "false_reject_rate": round(fr, 4),
        "false_accept_rate": round(fa, 4),
        "balanced_accuracy": round(balanced, 4),          # at the chosen point
        "best_balanced_accuracy": round(best_balanced, 4),  # of the signal itself
        "fa_cost": fa_cost, "fr_cost": fr_cost,
        "n_same": len(same_sims), "n_diff": len(diff_sims),
        "same_p10": round(_p(same_sorted, 0.10), 4),
        "same_p50": round(_p(same_sorted, 0.50), 4),
        "diff_p50": round(_p(diff_sorted, 0.50), 4),
        "diff_p90": round(_p(diff_sorted, 0.90), 4),
        "separable": bool(_p(same_sorted, 0.10) > _p(diff_sorted, 0.90)),
        "curve": curve,
    }


def verdict(report, usable_balanced=0.80):
    """Is this signal good enough to merge on at ALL?

    A threshold chosen on a signal that cannot separate is still a bad
    threshold. This says so out loud rather than letting a tuned number imply
    a solved problem.
    """
    if not report or report.get("threshold") is None:
        return "NO DATA — cannot choose a threshold"
    # judge the SIGNAL, not the operating point we chose on it
    ba = report.get("best_balanced_accuracy", report["balanced_accuracy"])
    if report.get("degenerate_no_merge"):
        return (f"MERGE NOTHING is cheapest — at a {report['fa_cost']}:"
                f"{report['fr_cost']} cost ratio no threshold on this signal "
                f"beats simply never merging on appearance (balanced accuracy "
                f"{ba}). This is the arithmetic agreeing with the physics: use "
                f"hand-off, stationary and topology evidence, and let unmatched "
                f"fragments stay separate.")
    if report["separable"]:
        return (f"SEPARABLE — same-person p10 ({report['same_p10']}) is above "
                f"different-person p90 ({report['diff_p90']}). A threshold "
                f"between them is a real decision boundary.")
    if ba >= usable_balanced:
        return (f"OVERLAPPING but usable — balanced accuracy {ba}. Merge, but "
                f"keep the hard constraints (co-visibility, topology, role) "
                f"doing the heavy lifting.")
    return (f"NOT SEPARABLE — balanced accuracy {ba} at the best possible "
            f"point. No threshold fixes this. Appearance must not be the "
            f"primary merge signal on this footage; prefer physical evidence "
            f"(hand-off, stationary, topology) and accept fragments.")


def compare(same_sims, diff_sims, current, fa_cost=DEFAULT_FA_COST,
            fr_cost=DEFAULT_FR_COST):
    """Current threshold vs the cost-optimal one, as a decision aid.

    Never returns 'apply this'. It returns what each choice costs, because
    changing a merge threshold without a scored A/B is how a pipeline quietly
    starts fusing people.
    """
    best_t, rep = cost_weighted_threshold(same_sims, diff_sims, fa_cost, fr_cost)
    if best_t is None:
        return rep
    cur_fr, cur_fa = _rates(same_sims, diff_sims, current)
    cur_cost = fa_cost * cur_fa + fr_cost * cur_fr
    return {
        "current": {"threshold": current, "false_reject_rate": round(cur_fr, 4),
                    "false_accept_rate": round(cur_fa, 4),
                    "expected_cost": round(cur_cost, 4)},
        "suggested": {"threshold": best_t,
                      "false_reject_rate": rep["false_reject_rate"],
                      "false_accept_rate": rep["false_accept_rate"],
                      "expected_cost": rep["expected_cost"]},
        "cost_delta": round(rep["expected_cost"] - cur_cost, 4),
        "direction": ("more conservative" if best_t > current
                      else "more aggressive" if best_t < current else "unchanged"),
        "verdict": verdict(rep),
    }


def describe(report, width=46):
    """An ASCII cost curve. Seeing the shape stops a single number implying
    more precision than the data supports — a flat basin means the exact
    threshold barely matters, which is itself the finding."""
    if not report or report.get("threshold") is None:
        return report.get("note", "no data")
    curve = report["curve"]
    lo = min(c[3] for c in curve)
    hi = max(c[3] for c in curve)
    span = max(hi - lo, 1e-9)
    L = [f"COST CURVE  (fa_cost={report['fa_cost']} fr_cost={report['fr_cost']})",
         f"  chosen t={report['threshold']}  cost={report['expected_cost']}  "
         f"FR={report['false_reject_rate']}  FA={report['false_accept_rate']}"]
    for t, fr, fa, cost in curve:
        if round(t * 100) % 10:
            continue
        fill = int(width * (cost - lo) / span)
        mark = " <- chosen" if abs(t - report["threshold"]) < 1e-9 else ""
        L.append(f"  {t:.2f} |{'#' * fill}{'.' * (width - fill)}| {cost:.3f}{mark}")
    L.append(f"  {verdict(report)}")
    return "\n".join(L)
''',
    'tiled.py': r'''"""tiled.py — run the detector at native resolution on slices, not on a
downscaled whole frame.

WHY THIS EXISTS
    The pipeline decodes 3840x2160 and analyses 1280x720. A guest at the door
    is then ~60 px tall. The banked crop is upscaled to 128x256 before it ever
    reaches the ReID backbone, so the embedding is built mostly from
    interpolation — and the diagnostic that should have caught this measured
    the resize target (fixed in V76). Measured separability on this footage is
    0.658 balanced accuracy, which no threshold repairs.

    Slicing Aided Hyper Inference (Akyon et al., arXiv:2202.06934) is the
    standard answer: cut the full-resolution frame into overlapping tiles, run
    the detector on each tile at native scale, map the boxes back, and merge.
    Reported +6.8% to +14.5% AP on small-object benchmarks.

WHY NOT `pip install sahi`
    This kernel pins numpy==2.0.2, force-reinstalls scipy to restore Cython
    C-extensions, and hand-attaches 38 string ufuncs into numpy._core.umath to
    keep InsightFace importable. Adding a dependency that pulls its own numpy
    /torch constraints into that is a poor trade for ~150 lines of geometry.
    This module is pure Python + whatever the caller's detector already uses.

THE ROI IDEA (why this is cheaper than 4-9x)
    Naive slicing multiplies detector cost by the tile count. But small people
    only occur where the GROUND PLANE says they are small — the far half of
    the frame. This run already fits `expected height = 0.820*foot_y - 37px`.
    Tile only the band where predicted height is below the size you care
    about, and keep one cheap full-frame pass for everyone near the camera.
    On a typical oblique reception view that is 2-3 tiles, not 9.

CONTRACT
    tiled_predict() takes a `predict_fn(image_array) -> [(x1,y1,x2,y2,conf,cls)]`
    and returns the same shape in FULL-FRAME coordinates. It never imports
    cv2, torch or ultralytics, so it is testable with a fake detector.
"""
from __future__ import annotations

DEFAULT_TILE = 640
DEFAULT_OVERLAP = 0.2
DEFAULT_IOU = 0.55
# A box hugging a tile edge is probably a body the slice cut in half. Boxes
# that touch an INTERIOR seam are dropped in favour of whatever the
# neighbouring tile (which saw the whole body) produced.
EDGE_MARGIN_PX = 2


def slice_grid(width, height, tile=DEFAULT_TILE, overlap=DEFAULT_OVERLAP,
               roi=None):
    """-> [(x0, y0, x1, y1), ...] tiles covering `roi` (default: whole frame).

    Tiles are clamped to the frame, so the last row/column overlaps more
    rather than running past the edge — a slice that extends beyond the image
    would need padding, and padding invents pixels the detector then scores.
    """
    if tile <= 0:
        raise ValueError("tile must be positive")
    rx0, ry0, rx1, ry1 = roi or (0, 0, width, height)
    rx0, ry0 = max(0, int(rx0)), max(0, int(ry0))
    rx1, ry1 = min(int(width), int(rx1)), min(int(height), int(ry1))
    if rx1 <= rx0 or ry1 <= ry0:
        return []

    step = max(1, int(tile * (1.0 - overlap)))
    xs, ys = [], []
    x = rx0
    while True:
        xs.append(min(x, max(rx0, rx1 - tile)))
        if x + tile >= rx1:
            break
        x += step
    y = ry0
    while True:
        ys.append(min(y, max(ry0, ry1 - tile)))
        if y + tile >= ry1:
            break
        y += step

    seen, out = set(), []
    for yy in ys:
        for xx in xs:
            x1, y1 = min(xx + tile, rx1), min(yy + tile, ry1)
            key = (xx, yy, x1, y1)
            if key in seen:
                continue          # clamping can produce duplicates
            seen.add(key)
            out.append(key)
    return out


def height_roi(width, height, slope, intercept, min_px, pad=0.05,
               min_band_px=120):
    """The band of the frame where a person is predicted SHORTER than min_px.

    Uses the run's own fitted ground model, `expected_h = slope*foot_y +
    intercept`. Returns None when slicing would buy nothing, so the caller
    skips the tiles rather than paying for them.

    Two ways it declines, and the second matters more than it looks:

      * slope <= 0 — the ground fit is degenerate, so its band is meaningless.
      * the band is thinner than min_band_px. This run fits intercept = -37,
        which means predicted height is NEGATIVE above foot_y ~= 45: the model
        is extrapolating outside the data it was fitted on. Without this guard
        a small min_px returns a ~90 px sliver at the top of frame, and we
        would pay for tiles over a strip that cannot contain a detectable
        person in the first place.
    """
    if slope <= 0:
        return None
    # expected_h(y) grows with y (further down the frame = closer = bigger)
    y_at_min = (min_px - intercept) / slope
    if y_at_min <= 0:
        return None                      # even the top of frame is big enough
    y_cut = min(float(height), y_at_min + pad * height)
    if y_cut < min_band_px:
        return None
    return (0, 0, int(width), int(round(y_cut)))


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    aa = max(0.0, (a[2] - a[0]) * (a[3] - a[1]))
    bb = max(0.0, (b[2] - b[0]) * (b[3] - b[1]))
    denom = aa + bb - inter
    return inter / denom if denom > 0 else 0.0


def nms(boxes, iou_thresh=DEFAULT_IOU):
    """Greedy NMS over (x1,y1,x2,y2,conf,cls). Highest confidence wins.

    Suppression is per-class: two different classes overlapping is not a
    duplicate, and merging them would silently delete one of them.
    """
    kept = []
    for box in sorted(boxes, key=lambda b: -b[4]):
        if all(_iou(box, k) < iou_thresh or box[5] != k[5] for k in kept):
            kept.append(box)
    return kept


def _touches_interior_seam(box, tile, frame_wh, margin=EDGE_MARGIN_PX):
    """True if the box hugs a tile edge that is NOT also the frame edge."""
    tx0, ty0, tx1, ty1 = tile
    fw, fh = frame_wh
    x1, y1, x2, y2 = box[:4]
    if abs(x1 - tx0) <= margin and tx0 > 0:
        return True
    if abs(y1 - ty0) <= margin and ty0 > 0:
        return True
    if abs(x2 - tx1) <= margin and tx1 < fw:
        return True
    if abs(y2 - ty1) <= margin and ty1 < fh:
        return True
    return False


def tiled_predict(frame, predict_fn, tile=DEFAULT_TILE,
                  overlap=DEFAULT_OVERLAP, roi=None, iou_thresh=DEFAULT_IOU,
                  full_frame=True, drop_seam_boxes=True):
    """Detect on overlapping native-resolution slices; return full-frame boxes.

    frame        anything sliceable as frame[y0:y1, x0:x1] with a .shape
    predict_fn   image -> iterable of (x1, y1, x2, y2, conf, cls)
    full_frame   also run one whole-frame pass. Keep it: a person close to the
                 camera can be larger than a tile, and tiles alone would only
                 ever see pieces of them.
    """
    h, w = frame.shape[0], frame.shape[1]
    out = []

    if full_frame:
        for b in predict_fn(frame) or []:
            out.append(tuple(b))

    tiles = slice_grid(w, h, tile, overlap, roi)
    for (x0, y0, x1, y1) in tiles:
        sub = frame[y0:y1, x0:x1]
        for b in predict_fn(sub) or []:
            bx1, by1, bx2, by2, conf, cls = b[0], b[1], b[2], b[3], b[4], b[5]
            shifted = (bx1 + x0, by1 + y0, bx2 + x0, by2 + y0, conf, cls)
            if drop_seam_boxes and _touches_interior_seam(
                    shifted, (x0, y0, x1, y1), (w, h)):
                continue          # a body the slice cut in half
            out.append(shifted)

    return nms(out, iou_thresh), {"tiles": len(tiles), "raw": len(out)}


def cost_estimate(width, height, tile=DEFAULT_TILE, overlap=DEFAULT_OVERLAP,
                  roi=None, full_frame=True):
    """Detector calls per frame, so the compute trade is visible up front."""
    n = len(slice_grid(width, height, tile, overlap, roi)) + (1 if full_frame else 0)
    return {"calls_per_frame": n, "vs_baseline": f"{n:.1f}x"}
''',
    'topology.py': r'''"""topology.py — a re-appearance gate that stays sharp when appearance doesn't.

WHY THIS EXISTS
    The live identity gate asks "could a person have walked that far in the
    time available":

        _md > max_speed_mps * max(gap, 0.35) + 0.35

    At 2.2 m/s a 900-second gap allows ~1,980 metres. Every point in the frame
    passes. So beyond roughly ten seconds the spatial constraint contributes
    nothing and the merge is decided by appearance alone — exactly the regime
    where appearance is weakest. On CAM.112 the measured separability is:

        same person      p50 = 0.435
        different person p50 = 0.370, p90 = 0.573
        best possible balanced accuracy = 0.658

    No threshold rescues that. A stronger backbone might move it a few points.
    A different KIND of constraint moves it more.

THE PRINCIPLE
    A fixed camera has a topology. People do not materialise in the middle of
    the room — they come through a door. So a re-appearance is only physically
    possible in three shapes:

        1. the track died and was reborn in the SAME PLACE
           -> the tracker lost them behind something. Plausible.
        2. the track died AT A DOOR and was reborn AT A DOOR
           -> they left and came back. Plausible, at any gap.
        3. anything else
           -> they would have had to cross the room unseen. Not plausible.

    This is a HARD constraint, like co-visibility, not a score. It does not
    degrade as the gap grows, because a door stays a door. It is the natural
    partner to the co-visibility rule already in _IdentityMemory: that one says
    "one person cannot be in two places at once", this one says "one person
    cannot get from A to B without crossing the space between".

WHAT IT DOES NOT DO
    It never merges anything. It only ever says "this pair is impossible" —
    a veto, applied before appearance is consulted. Vetoes are safe in a way
    that merges are not: a wrong veto costs one fragment, a wrong merge fuses
    two people and corrupts every downstream count.

    Doors come from kevacv.learn_zones.learn_entry_zones (learned from where
    tracks are actually born and die), so this needs no hand-drawn zone.
"""
from __future__ import annotations

import math

# A door is a place, not a point: allow this fraction of the frame diagonal
# around a learned door centre before calling a position "not at the door".
DOOR_RADIUS_FRAC = 0.10
# Below this gap the existing positional/velocity gates are trustworthy and
# this module stays out of the way.
SHORT_GAP_S = 10.0
# How far a body may drift while the tracker has lost it and still count as
# "the same place" (fraction of frame diagonal).
STILL_DRIFT_FRAC = 0.06


def _diag(frame_wh):
    w, h = float(frame_wh[0]), float(frame_wh[1])
    return math.hypot(w, h)


def _nearest(pt, doors):
    """-> (distance_px, index) to the nearest door centre, or (inf, None)."""
    best, idx = float("inf"), None
    for i, d in enumerate(doors or []):
        dist = math.hypot(pt[0] - d[0], pt[1] - d[1])
        if dist < best:
            best, idx = dist, i
    return best, idx


def doors_from_zones(zone_polygons, zone_roles, roles=("entry",)):
    """Door centres from named entry polygons, when zones ARE hand-drawn.

    Falls back to nothing rather than guessing: no entry zone means this gate
    abstains, which is the correct behaviour for a module that only vetoes.
    """
    want = {z for z, rs in (zone_roles or {}).items() if set(rs) & set(roles)}
    out = []
    for name, poly in (zone_polygons or {}).items():
        if name not in want or not poly:
            continue
        xs = [float(p[0]) for p in poly]
        ys = [float(p[1]) for p in poly]
        out.append((sum(xs) / len(xs), sum(ys) / len(ys)))
    return out


def doors_from_endpoints(entry_zones):
    """Door centres from kevacv.learn_zones.learn_entry_zones() output.

    Accepts either dicts with a 'centre'/'center' key or bare (x, y) pairs, so
    it does not care which shape that module returns.
    """
    out = []
    for z in entry_zones or []:
        if isinstance(z, dict):
            c = z.get("centre") or z.get("center")
            if c is None and z.get("polygon"):
                xs = [float(p[0]) for p in z["polygon"]]
                ys = [float(p[1]) for p in z["polygon"]]
                c = (sum(xs) / len(xs), sum(ys) / len(ys))
            if c is not None:
                out.append((float(c[0]), float(c[1])))
        elif z is not None and len(z) >= 2:
            out.append((float(z[0]), float(z[1])))
    return out


def reappearance_verdict(death_pos, birth_pos, gap_s, doors, frame_wh,
                         door_radius_frac=DOOR_RADIUS_FRAC,
                         still_drift_frac=STILL_DRIFT_FRAC,
                         short_gap_s=SHORT_GAP_S):
    """Is it physically possible that these two tracks are one person?

    -> {"allow": bool, "shape": str, "why": str, ...}

    `allow=True` never means "these ARE the same person" — appearance and the
    other tiers still decide that. It means only that topology does not forbid
    it. `allow=False` is a hard veto.

    Abstains (allow=True, shape="abstain") when there are no doors to reason
    about, because a gate with no information must not block anything.
    """
    if not doors:
        return {"allow": True, "shape": "abstain",
                "why": "no doors known — topology cannot judge this pair"}
    if gap_s is None or gap_s <= short_gap_s:
        return {"allow": True, "shape": "continuous",
                "why": (f"gap {gap_s}s is within the short-gap window "
                        f"({short_gap_s}s); the positional gate governs here")}

    diag = _diag(frame_wh)
    door_r = diag * door_radius_frac
    still_r = diag * still_drift_frac

    d_exit, exit_i = _nearest(death_pos, doors)
    d_entry, entry_i = _nearest(birth_pos, doors)
    at_exit = d_exit <= door_r
    at_entry = d_entry <= door_r
    drift = math.hypot(birth_pos[0] - death_pos[0], birth_pos[1] - death_pos[1])

    base = {"gap_s": gap_s, "drift_px": round(drift, 1),
            "exit_door_px": round(d_exit, 1), "entry_door_px": round(d_entry, 1),
            "door_radius_px": round(door_r, 1), "at_exit": at_exit,
            "at_entry": at_entry}

    # 2. left through a door, came back through a door
    if at_exit and at_entry:
        return {**base, "allow": True, "shape": "door_to_door",
                "why": (f"last seen at door {exit_i}, reappeared at door "
                        f"{entry_i} — leaving and returning is exactly this")}
    # 1. never went near a door, reappeared where they vanished
    if not at_exit and not at_entry and drift <= still_r:
        return {**base, "allow": True, "shape": "occlusion_recovery",
                "why": (f"vanished and reappeared {drift:.0f}px apart, away "
                        f"from any door — the tracker lost a body that never "
                        f"left")}
    # 3. everything else requires crossing the room unseen
    if at_exit and not at_entry:
        why = ("last seen leaving through a door but reappeared mid-room — "
               "they would have had to walk back in unobserved")
    elif at_entry and not at_exit:
        why = ("vanished mid-room but reappeared at a door — they would have "
               "had to walk to the door unobserved")
    else:
        why = (f"vanished and reappeared {drift:.0f}px apart with no door at "
               f"either end — no path exists that the camera would not have "
               f"seen")
    return {**base, "allow": False, "shape": "impossible", "why": why}


def veto_pairs(pairs, doors, frame_wh, **kw):
    """Filter candidate merge pairs, returning (kept, vetoed).

    `pairs` are dicts with at least death_pos, birth_pos and gap_s. Everything
    else on the dict is carried through untouched, so this drops into an
    existing merge pipeline without reshaping its data.
    """
    kept, vetoed = [], []
    for p in pairs:
        v = reappearance_verdict(p["death_pos"], p["birth_pos"], p.get("gap_s"),
                                 doors, frame_wh, **kw)
        (kept if v["allow"] else vetoed).append({**p, "topology": v})
    return kept, vetoed


def describe(kept, vetoed):
    L = [f"TOPOLOGY GATE — {len(kept)} possible, {len(vetoed)} vetoed"]
    shapes = {}
    for p in kept + vetoed:
        s = p["topology"]["shape"]
        shapes[s] = shapes.get(s, 0) + 1
    for s, n in sorted(shapes.items(), key=lambda kv: -kv[1]):
        L.append(f"  {s:<20} {n}")
    for p in vetoed[:5]:
        L.append(f"  VETO {p.get('a', '?')} -> {p.get('b', '?')}: "
                 f"{p['topology']['why']}")
    return "\n".join(L)
''',
    'tracker_wrapper.py': r'''"""tracker_wrapper.py — Phase B P7 BoxMOT Tracker Interface & Configuration.

WHY THIS EXISTS
    The pipeline relies on BoxMOT for Multi-Object Tracking. Upgrading BoxMOT (e.g. v19 -> v25+
    with BoostTrack++ / Soft BIoU) requires a resilient wrapper that abstracts API differences
    across BoxMOT versions and handles numpy/torch array conversions cleanly.
"""
from __future__ import annotations

import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np

_log = logging.getLogger(__name__)

try:
    import boxmot
    _HAVE_BOXMOT = True
except ImportError:
    _HAVE_BOXMOT = False


class TrackerWrapper:
    """Unified wrapper around BoxMOT trackers (BoostTrack++, BoT-SORT, StrongSORT, ByteTrack)."""

    SUPPORTED_TRACKERS = ("boosttrack", "botsort", "strongsort", "bytetrack", "ocsort")

    def __init__(self,
                 tracker_type: str = "boosttrack",
                 reid_weights: Optional[Union[str, Path]] = None,
                 device: str = "cuda:0",
                 conf: float = 0.25,
                 iou: float = 0.45,
                 track_high_thresh: float = 0.5,
                 track_low_thresh: float = 0.1,
                 new_track_thresh: float = 0.6,
                 track_buffer: int = 60):
        self.tracker_type = tracker_type.lower()
        self.reid_weights = Path(reid_weights) if reid_weights else None
        self.device = device
        self.conf = conf
        self.iou = iou
        self.track_high_thresh = track_high_thresh
        self.track_low_thresh = track_low_thresh
        self.new_track_thresh = new_track_thresh
        self.track_buffer = track_buffer

        self.tracker = self._init_tracker()

    def _init_tracker(self) -> Any:
        if not _HAVE_BOXMOT:
            return None

        # Determine class name dynamically from boxmot
        try:
            if self.tracker_type in ("boosttrack", "boosttrack++"):
                if hasattr(boxmot, "BoostTrack"):
                    return boxmot.BoostTrack(
                        reid_weights=self.reid_weights,
                        device=self.device,
                        conf=self.conf,
                        iou=self.iou,
                        track_buffer=self.track_buffer
                    )
            elif self.tracker_type == "botsort":
                if hasattr(boxmot, "BotSort"):
                    return boxmot.BotSort(
                        reid_weights=self.reid_weights,
                        device=self.device,
                        conf=self.conf,
                        iou=self.iou,
                        track_buffer=self.track_buffer
                    )
            elif self.tracker_type == "bytetrack":
                if hasattr(boxmot, "ByteTrack"):
                    return boxmot.ByteTrack(
                        track_thresh=self.track_high_thresh,
                        track_buffer=self.track_buffer,
                        match_thresh=self.iou
                    )
        except Exception as e:
            _log.warning("BoxMOT tracker init failed (%s: %s). "
                         "Falling back to dummy per-frame ID assignment.", type(e).__name__, e)

        return None

    def update(self, dets: np.ndarray, img: np.ndarray) -> np.ndarray:
        """Update tracker with frame detections [x1, y1, x2, y2, conf, cls].
        Returns tracked objects [x1, y1, x2, y2, track_id, conf, cls, idx].
        """
        if self.tracker is not None:
            try:
                result = self.tracker.update(dets, img)
                if result is not None:
                    return result
            except Exception as e:
                _log.warning("BoxMOT tracker.update() failed (%s: %s). "
                             "Using fallback ID assignment for this frame.", type(e).__name__, e)

        # Fallback dummy tracking format if boxmot is absent or failed
        if len(dets) == 0:
            return np.empty((0, 8), dtype=np.float32)

        out = []
        for idx, d in enumerate(dets):
            x1, y1, x2, y2 = d[:4]
            c = d[4] if len(d) > 4 else 1.0
            cls_id = d[5] if len(d) > 5 else 0.0
            out.append([x1, y1, x2, y2, idx + 1, c, cls_id, idx])
        return np.array(out, dtype=np.float32)
''',
    'triage.py': r'''"""triage.py — PHASE 5a. Spend the compute where the people are.

THE PROBLEM THIS SOLVES
    Ten hours of 4K, ~3 GB per hour. Processing it end to end does not fit a
    Kaggle session, so the pipeline has been quietly analysing a 20-minute slice
    and reporting it as the chunk.

THE PRINCIPLE
    Cost must scale with EVENTS, not with wall-clock time. A reception is empty
    for most of a night. Every serious video-analytics system is built as a
    cascade — cheap filter first, expensive model only where the cheap filter
    says something is happening. NoScope, cloud-edge analytics work, and the
    DeepStream reference designs all share this shape.

    The payoff is not only speed. If 70% of the night is empty, triage lets you
    spend 3x MORE compute per occupied frame at the same total cost. Scale and
    accuracy stop being a trade-off and become the same fix.

THE HONESTY REQUIREMENT, WHICH IS THE HARD PART
    A cheap scan CAN miss someone. Sampling one frame every 6 s will not see a
    person who crosses in 3 s. So a segment planner that silently drops time is
    just a faster way to be wrong.

    Everything here therefore reports THREE kinds of time, never two:
        ANALYSED   full pipeline ran
        SKIPPED    scanned, verified empty, deliberately not analysed
        UNSEEN     never scanned at all
    and estimates the recall risk the skipping itself introduces, from the scan
    interval and how long a person is actually in shot. A skipped minute is a
    finding ("nothing happened"), not a gap — but only if it was really looked at.
"""
from __future__ import annotations


def plan_segments(scan, min_people=1, pad_s=20.0, merge_gap_s=45.0,
                  min_segment_s=10.0):
    """Choose the stretches worth the full pipeline.

    scan:        [(t_seconds, n_people), ...] from a cheap detector pass
    min_people:  a sample counts as "activity" at or above this
    pad_s:       extend each side. The scan is coarse, so someone is usually
                 already walking in before the first sample that sees them.
                 This is the main defence against clipping an arrival, and it
                 is deliberately generous: padding is cheap, a missed entry
                 is not.
    merge_gap_s: two active stretches closer than this become one. Stopping and
                 restarting the pipeline costs more than analysing the quiet
                 gap between them.
    min_segment_s: below this a segment is not worth the startup cost.

    -> (segments, stats). segments is [(t0, t1), ...] sorted and disjoint.
    """
    pts = sorted((float(t), int(n)) for t, n in scan)
    if not pts:
        return [], {"analysed_s": 0.0, "skipped_s": 0.0, "unseen_s": 0.0,
                    "scan_samples": 0, "reason": "no scan data"}

    t_first, t_last = pts[0][0], pts[-1][0]
    step = ((t_last - t_first) / (len(pts) - 1)) if len(pts) > 1 else 1.0

    active = [t for t, n in pts if n >= min_people]
    raw = [(t - pad_s, t + step + pad_s) for t in active]

    merged = []
    for a, b in raw:
        if merged and a - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])

    span_lo, span_hi = t_first, t_last + step
    segs = [(max(span_lo, a), min(span_hi, b)) for a, b in merged
            if min(span_hi, b) - max(span_lo, a) >= min_segment_s]

    analysed = sum(b - a for a, b in segs)
    scanned = span_hi - span_lo
    stats = {
        "analysed_s": round(analysed, 1),
        "skipped_s": round(scanned - analysed, 1),
        "unseen_s": 0.0,                      # set by coverage_report
        "scanned_s": round(scanned, 1),
        "segments": len(segs),
        "scan_samples": len(pts),
        "scan_step_s": round(step, 2),
        "active_samples": len(active),
        "saving_pct": round(100.0 * (1 - analysed / scanned), 1) if scanned else 0.0,
        "compute_multiplier": round(scanned / analysed, 2) if analysed else None,
    }
    return segs, stats


def miss_risk(scan_step_s, typical_visit_s=25.0):
    """How likely is the cheap scan to miss a person entirely?

    A visit of length V sampled every S seconds is seen unless it falls wholly
    between two samples, so the miss probability is roughly max(0, 1 - V/S).
    Crude, and deliberately so — the point is to make the risk VISIBLE and
    tie it to a number you can change, not to be exact.

    typical_visit_s is how long a person is in shot at all, not how long they
    stay in the venue. At a reception someone crossing to the desk is in frame
    for tens of seconds; a corridor camera would be far less.
    """
    if scan_step_s <= 0 or typical_visit_s <= 0:
        return {"miss_prob": 0.0, "note": "no scan"}
    p = max(0.0, 1.0 - typical_visit_s / scan_step_s) if scan_step_s > typical_visit_s else 0.0
    verdict = ("safe: every visit is sampled at least once"
               if scan_step_s <= typical_visit_s / 2 else
               "marginal: a short visit can fall between samples"
               if scan_step_s <= typical_visit_s else
               "UNSAFE: visits shorter than the scan interval are invisible")
    return {"miss_prob": round(p, 3), "scan_step_s": scan_step_s,
            "typical_visit_s": typical_visit_s, "verdict": verdict}


def coverage_report(segments, scan_span, chunk_span):
    """Account for every second of the chunk in one of three states.

    A pipeline that reports "nothing happened" for time it never looked at is
    the exact failure this whole project exists to remove, so ANALYSED,
    SKIPPED and UNSEEN are always separated.
    """
    c0, c1 = float(chunk_span[0]), float(chunk_span[1])
    total = max(0.0, c1 - c0)
    s0, s1 = (float(scan_span[0]), float(scan_span[1])) if scan_span else (c0, c0)
    scanned = max(0.0, min(s1, c1) - max(s0, c0))
    analysed = sum(max(0.0, min(b, c1) - max(a, c0)) for a, b in segments)
    skipped = max(0.0, scanned - analysed)
    unseen = max(0.0, total - scanned)
    return {
        "total_s": round(total, 1),
        "analysed_s": round(analysed, 1),
        "skipped_s": round(skipped, 1),
        "unseen_s": round(unseen, 1),
        "analysed_pct": round(100 * analysed / total, 1) if total else 0.0,
        "skipped_pct": round(100 * skipped / total, 1) if total else 0.0,
        "unseen_pct": round(100 * unseen / total, 1) if total else 0.0,
        "accounted": abs(analysed + skipped + unseen - total) < 1.0,
    }


def describe(stats, cover, risk):
    L = ["TRIAGE — cost follows events, not the clock"]
    L.append(f"  scanned {stats['scanned_s'] / 60:.1f} min at 1 frame / "
             f"{stats['scan_step_s']:.0f} s  ({stats['scan_samples']} samples, "
             f"{stats['active_samples']} with people)")
    L.append(f"  {stats['segments']} segment(s) worth the full pipeline "
             f"= {stats['analysed_s'] / 60:.1f} min")
    if stats.get("compute_multiplier"):
        L.append(f"  saving {stats['saving_pct']:.0f}% of the work -> you can afford "
                 f"{stats['compute_multiplier']:.1f}x more compute per analysed frame "
                 f"at the same total cost")
    L.append("  every second is accounted for:")
    L.append(f"     ANALYSED {cover['analysed_pct']:5.1f}%   full pipeline ran")
    L.append(f"     SKIPPED  {cover['skipped_pct']:5.1f}%   scanned, verified empty")
    L.append(f"     UNSEEN   {cover['unseen_pct']:5.1f}%   never looked at"
             + ("" if cover["unseen_pct"] < 0.05 else "   <-- report cannot speak for this"))
    if not cover["accounted"]:
        L.append("     !! the three do not sum to the chunk — do not trust this run")
    L.append(f"  scan miss risk: {risk['verdict']}")
    return "\n".join(L)
''',
    'validity.py': r'''"""validity.py — "I did not observe" is not "I observed nothing".

THE PRINCIPLE
    arrivals.py already states it: *0 and "cannot tell" must never look the
    same in a report.* That is not a rule about arrivals. It is the shape of
    almost every silent failure this pipeline has produced:

        camera went black        vs  the room was empty
        detector stopped working vs  nobody was there
        motion gate skipped it   vs  nothing happened
        the clock was wrong      vs  that is the real duration
        the entry line is broken vs  no one arrived
        the render failed        vs  nobody appeared in any frame

    Four of those have already shipped as wrong numbers. They are not six bugs
    needing six patches; they are one missing field.

WHAT THIS MODULE IS
    The field. Every frame gets a verdict. The ledger accumulates OBSERVED
    time separately from ELAPSED time, so any metric can divide by the right
    denominator.

    The pipeline already does this correctly at the coarsest level —
    covered_windows() / missing_hours, and desk_covered_pct divides by footage
    we actually have. This is the same idea one level down, where the failures
    actually happen.

WHAT IT DOES NOT DO
    It never repairs anything and never drops a frame on its own. It records a
    verdict and lets the caller decide. A module that silently discards data is
    indistinguishable from the bugs it exists to catch.
"""
from __future__ import annotations

from collections import Counter

from .log import get_logger

_log = get_logger("validity")

# ── verdicts ────────────────────────────────────────────────────────────────
OK = "ok"
BLIND_CAMERA = "blind_camera"        # black / near-zero variance: nothing to see
SKIPPED_IDLE = "skipped_idle"        # motion gate declined to look
DETECTOR_BLIND = "detector_blind"    # motion present, detector returned nothing
BAD_FRAME = "bad_frame"              # decode failure / None
GEOMETRY_CHANGED = "geometry_changed"  # resolution changed mid-stream
TIME_WENT_BACKWARDS = "time_backwards"  # non-monotonic timestamp

# A frame with any of these was NOT observed. Time under them must never be
# counted as "we watched and saw nobody".
NOT_OBSERVED = {BLIND_CAMERA, BAD_FRAME, GEOMETRY_CHANGED, TIME_WENT_BACKWARDS}

# Default: a frame darker and flatter than this cannot contain a person.
# Chosen against IR footage, which is legitimately dark but never FLAT — a live
# IR frame still has texture. A blanked sensor has neither.
BLACK_MEAN = 12.0
BLACK_STD = 3.0


def frame_validity(frame, *, black_mean=BLACK_MEAN, black_std=BLACK_STD,
                   expect_shape=None):
    """-> (verdict, detail). Cheap enough to run on every sampled frame.

    Deliberately checks variance as well as brightness. Infrared footage is
    dark but textured; a dead sensor, a covered lens or a dropped stream is
    dark AND flat. Testing brightness alone would condemn every IR frame.
    """
    if frame is None:
        return BAD_FRAME, "decoder returned None"
    try:
        h, w = frame.shape[0], frame.shape[1]
    except Exception:
        return BAD_FRAME, "frame has no shape"
    if h == 0 or w == 0:
        return BAD_FRAME, f"zero-size frame {w}x{h}"
    if expect_shape and (h, w) != tuple(expect_shape):
        return (GEOMETRY_CHANGED,
                f"resolution changed {tuple(expect_shape)} -> {(h, w)}; "
                f"zones were scaled for the old size")
    try:
        m = float(frame.mean())
        s = float(frame.std())
    except Exception:
        return OK, ""                      # not an array we can measure; trust it
    if m <= black_mean and s <= black_std:
        return BLIND_CAMERA, f"mean={m:.1f} std={s:.1f} — no signal, not an empty room"
    return OK, ""


class ValidityLedger:
    """Observed time vs elapsed time, and why the difference exists.

    Every metric that says "X% of the night" should divide by
    `observed_seconds`, not by the wall-clock span. Dividing by elapsed time
    turns a blind camera into a well-behaved quiet venue.
    """

    def __init__(self, step_s=None):
        self.rows = []               # (t, verdict)
        self.step_s = step_s
        self.counts = Counter()
        self._last_t = None
        self._t0 = None
        self._t1 = None

    def record(self, t, verdict=OK, detail=""):
        t = float(t)
        if self._last_t is not None and t < self._last_t:
            verdict = TIME_WENT_BACKWARDS
            detail = detail or f"t={t:.3f} after t={self._last_t:.3f}"
        self._last_t = max(t, self._last_t) if self._last_t is not None else t
        self._t0 = t if self._t0 is None else min(self._t0, t)
        self._t1 = t if self._t1 is None else max(self._t1, t)
        self.rows.append((t, verdict))
        self.counts[verdict] += 1
        if verdict in NOT_OBSERVED and detail:
            _log.warning(f"{verdict} at t={t:.1f}s — {detail}")
        return verdict

    # ── derived ─────────────────────────────────────────────────────────────
    @property
    def elapsed_seconds(self):
        if self._t0 is None:
            return 0.0
        return max(0.0, self._t1 - self._t0)

    def _step(self):
        if self.step_s:
            return float(self.step_s)
        ts = sorted(t for t, _ in self.rows)
        gaps = [b - a for a, b in zip(ts, ts[1:]) if b > a]
        return sorted(gaps)[len(gaps) // 2] if gaps else 0.0

    @property
    def observed_seconds(self):
        """Time we actually looked at usable pixels.

        SKIPPED_IDLE counts as observed: the motion gate looked, decided
        nothing was moving, and that IS an observation. BLIND_CAMERA does not.
        """
        step = self._step()
        n = sum(1 for _, v in self.rows if v not in NOT_OBSERVED)
        return n * step

    def observed_windows(self, join_gap_s=None):
        """Contiguous stretches actually observed, for clipping any metric."""
        step = self._step()
        join = join_gap_s if join_gap_s is not None else step * 1.5
        out = []
        for t, v in sorted(self.rows):
            if v in NOT_OBSERVED:
                continue
            if out and t - out[-1][1] <= join:
                out[-1][1] = t + step
            else:
                out.append([t, t + step])
        return [tuple(w) for w in out]

    def summary(self):
        el = self.elapsed_seconds
        ob = self.observed_seconds
        return {"frames": len(self.rows),
                "elapsed_s": round(el, 1),
                "observed_s": round(ob, 1),
                "unobserved_s": round(max(0.0, el - ob), 1),
                "observed_share": round(ob / el, 4) if el > 0 else None,
                "by_verdict": dict(self.counts)}

    def findings(self, blind_share_error=0.02):
        """-> [(level, message)]. What a human must be told about this run."""
        out = []
        s = self.summary()
        blind = self.counts.get(BLIND_CAMERA, 0)
        if blind and s["frames"]:
            share = blind / s["frames"]
            lvl = "ERROR" if share >= blind_share_error else "WARN"
            out.append((lvl, f"camera was BLIND for {share*100:.1f}% of sampled "
                             f"frames ({blind}) — that time is not an empty "
                             f"room and is excluded from every denominator"))
        if self.counts.get(TIME_WENT_BACKWARDS):
            out.append(("ERROR", f"{self.counts[TIME_WENT_BACKWARDS]} frame(s) "
                                 f"arrived with a timestamp earlier than the "
                                 f"one before — durations cannot be trusted"))
        if self.counts.get(GEOMETRY_CHANGED):
            out.append(("ERROR", f"{self.counts[GEOMETRY_CHANGED]} resolution "
                                 f"change(s) mid-stream — zone polygons were "
                                 f"scaled for the original size only"))
        if self.counts.get(BAD_FRAME):
            out.append(("WARN", f"{self.counts[BAD_FRAME]} undecodable frame(s)"))
        if self.counts.get(DETECTOR_BLIND):
            out.append(("ERROR", f"{self.counts[DETECTOR_BLIND]} frame(s) had "
                                 f"motion but ZERO detections — a detector that "
                                 f"has stopped working looks exactly like an "
                                 f"empty venue"))
        return out


class DetectorCanary:
    """Motion happened and the detector saw nobody. How often, and in a row?

    A model that fails to load its weights, or whose confidence collapses when
    the scene goes infrared, produces a perfectly quiet report. The motion gate
    makes that look deliberate. This is the only cheap way to tell the
    difference without ground truth: the frame differencer and the detector are
    independent, so sustained disagreement means one of them is broken.
    """

    def __init__(self, run_length=30):
        self.run_length = run_length
        self.current = 0
        self.longest = 0
        self.total = 0
        self.episodes = []
        self._start_t = None

    def observe(self, t, motion, n_detections):
        if motion and n_detections == 0:
            self.total += 1
            if self.current == 0:
                self._start_t = t
            self.current += 1
            self.longest = max(self.longest, self.current)
        else:
            if self.current >= self.run_length:
                self.episodes.append((self._start_t, t, self.current))
            self.current = 0
        return self

    def close(self, t=None):
        if self.current >= self.run_length:
            self.episodes.append((self._start_t, t, self.current))
        self.current = 0
        return self

    def findings(self):
        if not self.episodes:
            return []
        worst = max(e[2] for e in self.episodes)
        return [("ERROR",
                 f"detector returned NOTHING across {len(self.episodes)} "
                 f"stretch(es) of moving frames (longest {worst} frames) — "
                 f"motion and detection are independent, so sustained "
                 f"disagreement means one of them is broken, not that the "
                 f"venue was empty")]
''',
    'venue_profile.py': r'''"""venue_profile.py — PHASE 4 / U4-U8. Config is DATA, not edits to Cell 2.

THE COMPLAINT THIS ANSWERS
    "if I have to change the pipeline for each video, that is not engineering."
    Correct. Today ~20 values are hardcoded in the config cell — CAMERA_ID,
    DRIVE_TZ, ENTRY_LINE_FLIP, MIN_SEATED_S, STAFF_DOMINANCE_RATIO and the rest.
    Pointing the pipeline at a different camera means editing the pipeline.

    Everything camera-specific or venue-specific now lives in a profile that
    travels WITH the footage, next to the zones file. The code carries defaults;
    the profile overrides them; nothing about a new venue requires a code edit.

WHAT IS DELIBERATELY *NOT* HERE
    U6 was going to estimate a distribution of person heights. It is not worth
    building: a wrong PERSON_H_M scales every distance UNIFORMLY, so relative
    comparisons are unaffected and absolute thresholds shift by the same
    percentage. 1.70 vs 1.75 m is a 3% error, against 25% from a 20-degree
    camera tilt. It is a documented, configurable constant and that is enough.

LOOKUP ORDER (first hit wins)
    1. explicit argument                    (a notebook override, for testing)
    2. profile_<stem>.json beside the video
    3. a "profile" key inside zones_<stem>.json
    4. DEFAULTS below
"""
from __future__ import annotations

import json
from datetime import datetime, timedelta
from pathlib import Path

try:
    from zoneinfo import ZoneInfo
except ImportError:                                   # pragma: no cover
    ZoneInfo = None

# Every value carries its UNIT and the reason it exists. A config entry whose
# meaning has to be reverse-engineered from the code is not configuration.
DEFAULTS = {
    "camera": {
        "id": None,                    # str  — falls back to the video stem
        "timezone": "UTC",             # IANA name, e.g. "America/Chicago".
                                       # U8: NOT an abbreviation. "CDT" cannot
                                       # express a night that crosses a DST
                                       # change; "America/Chicago" can.
        "hfov_deg": 82.0,              # deg  — lens horizontal field of view.
                                       # Only scales the DEPTH axis of the auto
                                       # ground plane. Supply ground_points in
                                       # the zones file and this stops mattering.
        "person_height_m": 1.70,       # m    — the metric ruler. See the note
                                       # above: a small error here is uniform.
        "zone_tol_frac": 0.008,        # frac of frame diagonal — how far a zone
                                       # may shift before the run is INVALID.
                                       # Derived, see camera_health.py.
        "entry_line_flip": None,       # None = infer it from the footage (U5).
                                       # True/False pins it.
        "static": True,                # False for a PTZ: zone checks cannot
                                       # apply and must not silently pretend to.
    },
    "venue": {
        "type": "generic",             # free text, for the report only
        "min_seated_s": 60,            # s  — dwell that counts as "seated"
        "wait_threshold_s": 600,       # s  — "waited too long"
        "party_gap_s": 120,            # s  — gap inside one party's occupancy
        "min_party_s": 60,             # s
        "visit_min_s": 8,              # s  — staff presence that counts as a visit
        "staff_override_min_s": 60,    # s  — dwell in a staff zone => staff
        "staff_min_video_share": 0.35, # frac of the video inside a staff zone
        "staff_dominance_ratio": 3.0,  # x   — staff-zone time vs everywhere else
        "greet_min_contact_s": 3.0,    # s  — brief pass-by is not a greeting
        "greet_proximity_m": 1.5,      # m  — conversational distance (PROXY)
        "turnaway_max_s": 90.0,        # s  — in and out this fast, unserved
        "long_wait_s": 180.0,          # s  — the "waited" line in the report
        "micro_absence_s": 90.0,       # s  — desk gap that is doing the job
        "break_absence_s": 600.0,      # s  — desk gap that is a break
        "group_window_s": 25.0,        # s  — arrivals this close in time...
        "group_radius_m": 3.0,         # m  — ...and this close are one party
        "max_walk_speed_mps": 2.2,     # m/s — brisk walk, gates identity
    },
    "privacy": {
        "face_scope": "staff_only",    # "staff_only" | "all"
                                       # staff_only: post-processing face embeddings
                                       # (corroboration, merge tier, veto) are
                                       # restricted to tracks already identified as
                                       # staff. Customer faces are never embedded.
                                       # Staff gallery matching (per-frame) is always
                                       # on — staff are enrolled by name/photo.
                                       # all: every track gets face embeddings for
                                       # merge quality. Requires explicit biometric
                                       # consent from customers at the venue.
        "blur_exports": False,         # True = face-blur exported clip frames.
                                       # Not yet implemented; placeholder for Ph9.
    },
}

# Allowed values for enum-like config fields. Not bounds — discrete choices.
ENUMS = {
    "privacy.face_scope": ("staff_only", "all"),
}

# Bounds exist to catch a typo that would otherwise produce a plausible-looking
# wrong report — 600 vs 60, or metres typed where seconds were meant.
BOUNDS = {
    "camera.hfov_deg": (20.0, 180.0),
    "camera.person_height_m": (1.2, 2.2),
    "camera.zone_tol_frac": (0.001, 0.05),
    "venue.min_seated_s": (5, 3600),
    "venue.wait_threshold_s": (10, 7200),
    "venue.greet_proximity_m": (0.3, 6.0),
    "venue.group_radius_m": (0.5, 15.0),
    "venue.max_walk_speed_mps": (0.5, 6.0),
    "venue.staff_min_video_share": (0.0, 1.0),
    "venue.staff_dominance_ratio": (1.0, 100.0),
}


def _merge(base, over):
    out = {k: dict(v) for k, v in base.items()}
    for section, vals in (over or {}).items():
        if section in out and isinstance(vals, dict):
            out[section].update(vals)
        else:
            out[section] = vals
    return out


def load_profile(video_path=None, zones_path=None, explicit=None):
    """Merge the first profile found over DEFAULTS. Never raises: a missing or
    malformed profile falls back to defaults and reports it in `_source`."""
    src, over = "defaults", {}
    if explicit:
        src, over = "explicit argument", explicit
    else:
        cands = []
        if video_path:
            v = Path(video_path)
            cands.append(v.with_name(f"profile_{v.stem}.json"))
            cands.append(v.with_name("profile.json"))
        for p in cands:
            if p.exists():
                try:
                    over, src = json.loads(p.read_text()), str(p)
                    break
                except json.JSONDecodeError as e:
                    src = f"{p} (MALFORMED: {e}) — using defaults"
        else:
            if zones_path and Path(zones_path).exists():
                try:
                    z = json.loads(Path(zones_path).read_text())
                    if isinstance(z.get("profile"), dict):
                        over, src = z["profile"], f"{Path(zones_path).name}:profile"
                except json.JSONDecodeError:
                    pass
    prof = _merge(DEFAULTS, over)
    prof["_source"] = src
    if not prof["camera"].get("id") and video_path:
        prof["camera"]["id"] = Path(video_path).stem
    return prof


def validate(profile):
    """-> list of problems. Empty means usable. Loud beats plausible-but-wrong."""
    problems = []
    for path, (lo, hi) in BOUNDS.items():
        sec, key = path.split(".")
        v = profile.get(sec, {}).get(key)
        if v is None:
            continue
        if not isinstance(v, (int, float)) or not (lo <= v <= hi):
            problems.append(f"{path} = {v!r} is outside the sane range [{lo}, {hi}]")
    # Enum validation — discrete choices, not ranges.
    for path, allowed in ENUMS.items():
        sec, key = path.split(".")
        v = profile.get(sec, {}).get(key)
        if v is not None and v not in allowed:
            problems.append(
                f"{path} = {v!r} is not a valid choice. "
                f"Allowed: {', '.join(repr(a) for a in allowed)}")
    tz = profile.get("camera", {}).get("timezone")
    if tz and ZoneInfo is not None:
        try:
            ZoneInfo(tz)
        except Exception:
            problems.append(
                f"camera.timezone = {tz!r} is not an IANA zone name. Use e.g. "
                f"'America/Chicago', not 'CDT' — an abbreviation cannot express "
                f"a night that crosses a daylight-saving change.")
    return problems


def local_clock(profile, naive_start):
    """U8: naive wall time from the filename -> a DST-correct clock function.

    Returns f(seconds_into_video) -> 'HH:MM:SS'. Adding a timedelta to an
    aware datetime crosses a DST boundary correctly; adding it to a string
    called 'CDT' does not.
    """
    tz = profile.get("camera", {}).get("timezone") or "UTC"
    if naive_start is None:
        return lambda s: ""
    if ZoneInfo is None:
        return lambda s: (naive_start + timedelta(seconds=float(s))).strftime("%H:%M:%S")
    try:
        start = naive_start.replace(tzinfo=ZoneInfo(tz))
    except Exception:
        start = naive_start.replace(tzinfo=ZoneInfo("UTC"))

    def at(seconds):
        # arithmetic in UTC, render in local: the only order that survives a
        # DST jump without silently shifting or duplicating an hour
        return (start.astimezone(ZoneInfo("UTC")) + timedelta(seconds=float(seconds))
                ).astimezone(start.tzinfo).strftime("%H:%M:%S")

    return at


def infer_entry_direction(crossings, events, interior_zones, min_evidence=6):
    """U5: work out which way through the door is IN, from the footage itself.

    ENTRY_LINE_FLIP is currently a hand-set boolean. Get it wrong and the
    headline number is exactly backwards, and the only existing guard is a
    heuristic that fires when outward crossings outnumber inward ones by 1.6x.

    The evidence is already in the data: someone who ENTERS spends time in
    interior zones AFTER crossing; someone who LEAVES spent it BEFORE.

    BOTH directions are used, with opposite sign. A one-sided version (only
    looking at crossings labelled "in") throws away half the evidence and goes
    blind on any venue where that label happens to be rare — e.g. a closing
    shift, where a correctly-configured line produces almost no inward
    crossings at all. Signed symmetrically:
        labelled "in"  -> expect dwell AFTER  (+1 x (after - before))
        labelled "out" -> expect dwell BEFORE (-1 x (after - before))
    so a positive total means the labels are right and negative means flipped,
    regardless of which way the traffic happened to run that night.

    -> (flip_needed, confidence 0..1, evidence dict). flip_needed is None when
    there is not enough evidence to say, which must stay distinct from False.
    """
    by_track = {}
    for e in events:
        if e.get("zone") in interior_zones:
            by_track.setdefault(e["track_id"], []).append((e["t_in"], e["t_out"]))

    score, n = 0.0, 0
    for c in crossings:
        sign = {"in": 1.0, "out": -1.0}.get(c.get("direction"))
        if sign is None:
            continue
        ivs = by_track.get(c.get("track_id"))
        if not ivs:
            continue
        t = c["t"]
        after = sum(max(0.0, b - max(a, t)) for a, b in ivs)
        before = sum(max(0.0, min(b, t) - a) for a, b in ivs)
        if after == before == 0:
            continue
        score += sign * (after - before)
        n += 1

    ev = {"tracks_with_evidence": n, "net_dwell_after_minus_before_s": round(score, 1)}
    if n < min_evidence:
        ev["why"] = (f"only {n} crossing(s) had interior dwell either side "
                     f"(need {min_evidence}) — keeping the configured value")
        return None, 0.0, ev
    total = sum(abs(x) for x in (score,)) or 1.0
    conf = min(1.0, abs(score) / max(total, 1.0))
    ev["verdict"] = ("labels look correct" if score > 0
                     else "labels look BACKWARDS — in/out are swapped")
    return (score < 0), (1.0 if n >= min_evidence else conf), ev


def describe(profile):
    lines = [f"profile source: {profile.get('_source')}"]
    for sec in ("camera", "venue"):
        diffs = {k: v for k, v in profile.get(sec, {}).items()
                 if v != DEFAULTS.get(sec, {}).get(k)}
        lines.append(f"  {sec}: " + (", ".join(f"{k}={v}" for k, v in diffs.items())
                                     if diffs else "all defaults"))
    return "\n".join(lines)


def write_template(path, video_stem=""):
    """Emit a filled-in profile so a new venue is a file to edit, not code."""
    tpl = {"camera": dict(DEFAULTS["camera"]), "venue": dict(DEFAULTS["venue"])}
    tpl["camera"]["id"] = video_stem or "CAMERA_ID"
    tpl["_README"] = ("Everything the pipeline needs to know about THIS camera "
                      "and THIS venue. Put it next to the video as "
                      "profile_<video-stem>.json. timezone must be an IANA name "
                      "(America/Chicago), never an abbreviation. entry_line_flip "
                      "null = infer it from the footage.")
    Path(path).write_text(json.dumps(tpl, indent=2))
    return Path(path)
''',
}

_pkg = _Path(str(globals().get("BASE", "."))) / "kevacv"
_pkg.mkdir(parents=True, exist_ok=True)
for _name, _body in _KEVACV_SRC.items():
    _tgt = _pkg / _name
    if not _tgt.exists() or _tgt.read_text(encoding="utf-8") != _body:
        _tgt.write_text(_body, encoding="utf-8")
if str(_pkg.parent) not in _sys.path:
    _sys.path.insert(0, str(_pkg.parent))
for _m in [k for k in list(_sys.modules) if k == "kevacv" or k.startswith("kevacv.")]:
    del _sys.modules[_m]          # so a re-run picks up refreshed sources

from kevacv.arrivals import (arrivals_from_regions, cross_check,
                            entry_zone_coverage,
                            describe as describe_arrivals)
from kevacv.learn_zones import (learn_dwell_zones, learn_entry_zones,
                                to_zone_config, track_endpoints)
from kevacv.phantoms import (drop_phantom_dets, in_phantom,
                             phantom_regions)
from kevacv.camera_health import CameraHealth, verdict_line
from kevacv.detect_filters import (BODY_ASPECT, drop_tracks, implausible_size_mask,
                                   protected_ids, static_track_ids)
from kevacv.eval_harness import (compare, dump_errors_csv, explain, iou_matrix,
                                 load_mot, save_baseline, score_conditions,
                                 score_sequence, write_mot)
from kevacv.ground_plane import PERSON_H_M, GroundPlane, synth_camera
from kevacv.triage import coverage_report, miss_risk, plan_segments
from kevacv.venue_profile import (DEFAULTS, describe, infer_entry_direction,
                                  load_profile, local_clock, validate,
                                  write_template)
import kevacv as _kevacv
print(f"kevacv {_kevacv.__version__} ready from {_pkg} "
      f"({len(_KEVACV_SRC)} modules, {len(_kevacv.__all__)} public names)")


In [ ]:
# Cell 3 — MULTI-VENUE DISCOVERY (cafe / store / restaurant / anything)
# Works with any video dropped anywhere under the attached dataset(s), in
# any of the formats below, and pairs it with its zone map automatically —
# no filenames or camera IDs need to be hardcoded anywhere in this cell.
from pathlib import Path
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".wmv", ".mpg", ".mpeg"}

def _find_zone_file(video_path: Path):
    stem = video_path.stem
    same_dir = video_path.parent

    # 1) exact convention: zones_<stem>.json right next to the video
    exact = same_dir / f"zones_{stem}.json"
    if exact.exists():
        return exact

    # 2) common alias patterns right next to the video
    aliases = (f"zones_{stem} (1).json", f"zones_custom_{stem}.json",
               f"zones_final_{stem}.json")
    for alias in aliases:
        hit = same_dir / alias
        if hit.exists():
            return hit

    # 3) same convention/aliases, searched anywhere in the attached dataset(s)
    if INPUT_ROOT.exists():
        hit = next(INPUT_ROOT.rglob(f"zones_{stem}.json"), None)
        if hit:
            return hit
        for alias in aliases:
            hit = next(INPUT_ROOT.rglob(alias), None)
            if hit:
                return hit

    # 4) fallback for zone files that don't follow the naming convention at
    #    all (e.g. "zones_custom.json"): if this video is the ONLY video
    #    sitting in its folder, and there's exactly one zones_*.json file
    #    sitting there too, they're almost certainly a pair.
    if same_dir.exists():
        videos_here = [p for p in same_dir.iterdir()
                       if p.suffix.lower() in VIDEO_EXTS]
        # v55: any json with "zone" in the name counts — a Kaggle dataset holds
        # CAM.112_zone.json, which zones_*.json never matched, and the pipeline
        # silently fell back to TEMPLATE polygons that fit no real camera.
        zone_jsons_here = [p for p in same_dir.glob("*.json")
                           if "zone" in p.name.lower()]
        if len(videos_here) == 1 and len(zone_jsons_here) == 1:
            return zone_jsons_here[0]

    # 5) last resort: exactly one zone-looking json anywhere in the input
    if INPUT_ROOT.exists():
        anyz = [p for p in INPUT_ROOT.rglob("*.json") if "zone" in p.name.lower()]
        if len(anyz) == 1:
            return anyz[0]

    return None

def discover_venues():
    all_videos = sorted({p for p in INPUT_ROOT.rglob("*")
                         if p.suffix.lower() in VIDEO_EXTS}, key=lambda p: p.stem) \
                 if INPUT_ROOT.exists() else []
    venues, seen, duplicates = [], {}, []
    for v in all_videos:
        key = v.stem.lower()
        if key in seen:
            duplicates.append((v, seen[key]))
            continue
        seen[key] = v
        zpath = _find_zone_file(v)
        venues.append({
            "camera_id": v.stem,
            "video": v,
            "zones_path": zpath or (BASE / f"zones_{v.stem}.json"),
            "has_zones": zpath is not None,
        })
    if duplicates:
        print("⚠️  DUPLICATE VIDEOS SKIPPED (same camera_id found more than once —")
        print("    check for multiple/overlapping datasets attached as Input):")
        for dupe_path, kept_path in duplicates:
            print(f"    kept:    {kept_path}")
            print(f"    skipped: {dupe_path}")
    return venues
VENUES = discover_venues()

# --- choose which video(s) to process ---
# Leave as None to process EVERY video that was discovered above.
# To restrict to specific video(s) instead, set this to a filename stem
# (the video's name without its extension), e.g.:
#     SELECTED_VIDEOS = "cafe001"                      # just one
#     SELECTED_VIDEOS = ["cafe001", "jane_camera"]      # a subset
SELECTED_VIDEOS = None

if SELECTED_VIDEOS:
    wanted = {SELECTED_VIDEOS.lower()} if isinstance(SELECTED_VIDEOS, str) \
             else {s.lower() for s in SELECTED_VIDEOS}
    missing = wanted - {v["camera_id"].lower() for v in VENUES}
    if missing:
        print(f"⚠️  requested video(s) not found among discovered videos, ignoring: {sorted(missing)}")
    VENUES = [v for v in VENUES if v["camera_id"].lower() in wanted]

# --- manual override: only needed if a zone JSON's polygon points were drawn
# against a different resolution than the video AND that mismatch isn't caught
# automatically (load_zone_config infers this when points fall outside the
# frame — this dict is only for the rare case where they don't).
# Add an entry as {"<camera_id>": [width, height]} if you notice zones drawn
# in the wrong place/size for a given camera.
FRAME_SIZE_FIXES = {}

for v in VENUES:
    cam = v["camera_id"].lower()
    if cam in FRAME_SIZE_FIXES and v["zones_path"] and Path(v["zones_path"]).exists():
        cfg = json.loads(Path(v["zones_path"]).read_text())
        if "frame_size" not in cfg:
            cfg["frame_size"] = FRAME_SIZE_FIXES[cam]
            patched_path = BASE / f"zones_{cam}_patched.json"
            patched_path.write_text(json.dumps(cfg, indent=2))
            v["zones_path"] = patched_path
            print(f"patched {cam}: added frame_size {FRAME_SIZE_FIXES[cam]} -> {patched_path}")

print("=" * 70)
print(f"DISCOVERED {len(VENUES)} VIDEO(S) under {INPUT_ROOT}")
print("=" * 70)
for i, v in enumerate(VENUES):
    tag = "zones found" if v["has_zones"] else "no zones yet -> generic template will be written"
    zname = Path(v["zones_path"]).name if v["has_zones"] else "n/a"
    print(f"[{i}] {v['camera_id']:<28} {tag}  ({zname})")
print("=" * 70)
if not VENUES:
    raise RuntimeError(f"No videos found under {INPUT_ROOT}")
VIDEO_QUEUE = VENUES

# ── HARD GATE: discovery must agree with preflight ──────────────────────────
# Discovery trusts the FILESYSTEM; preflight queued specific remote files. If
# they diverge (stale cached chunk, wrong clock) every downstream timestamp
# lies. The 4:30pm-footage-stamped-19:30 run is exactly this failure.
if globals().get("USE_DRIVE") and "_clock_from_name" in globals():
    if globals().get("QUEUE_REMOTE") and len(VIDEO_QUEUE) != len(QUEUE_REMOTE):
        raise RuntimeError(
            f"discovered {len(VIDEO_QUEUE)} video(s) but preflight queued "
            f"{len(QUEUE_REMOTE)} — drive_chunks/ is out of sync; delete it "
            f"and re-run the preflight cell")
    _disc = _clock_from_name(Path(VIDEO_QUEUE[0]["video"]).name)
    _ann  = globals().get("VIDEO_START_CLOCK")
    if _ann and (not _disc or _disc.strftime("%H:%M:%S") != _ann):
        raise RuntimeError(
            f"CLOCK MISMATCH: processing {Path(VIDEO_QUEUE[0]['video']).name!r} "
            f"(name says {_disc.strftime('%H:%M:%S') if _disc else '??'}) but "
            f"preflight announced {_ann} — stale file in drive_chunks/?")
    if _disc:  # the file actually being processed is the source of truth
        VIDEO_START_CLOCK = _disc.strftime("%H:%M:%S")

# ── U4: CONFIG IS DATA ──────────────────────────────────────────────────────
# Every camera/venue value loads from profile_<stem>.json (or a "profile" key in
# the zones file) instead of being edited into Cell 2. Applied here, once, with
# a printed diff so a run can always be traced back to the settings that made it.
_profiles = {v["camera_id"]: load_profile(video_path=v["video"],
                                          zones_path=v["zones_path"])
             for v in VIDEO_QUEUE}
for v in VIDEO_QUEUE:
    v["profile"] = _profiles[v["camera_id"]]
VENUE_PROFILE = _profiles[VIDEO_QUEUE[0]["camera_id"]] if VIDEO_QUEUE else load_profile()

_distinct = {json.dumps({k: p[k] for k in ("camera", "venue")}, sort_keys=True)
             for p in _profiles.values()}
if len(_distinct) > 1:
    print("\u26a0\ufe0f  the queued videos have DIFFERENT profiles. Venue thresholds "
          "are global in this notebook, so only the first is applied:")
    for cid, p in _profiles.items():
        print(f"      {cid}: {p['_source']}")
    print("      Run them as separate sessions if the venues really differ.")

print()
print(describe(VENUE_PROFILE))
_probs = validate(VENUE_PROFILE)
if _probs:
    print("\U0001f6a8 PROFILE PROBLEMS — fix these before trusting any number:")
    for _p in _probs:
        print(f"      {_p}")
    raise RuntimeError("invalid venue profile — see the problems above")

# venue thresholds -> the globals the rest of the notebook already reads
_APPLY = {
    "MIN_SEATED_S": "min_seated_s", "WAIT_THRESHOLD_S": "wait_threshold_s",
    "PARTY_GAP_S": "party_gap_s", "MIN_PARTY_S": "min_party_s",
    "VISIT_MIN_S": "visit_min_s", "STAFF_OVERRIDE_MIN_S": "staff_override_min_s",
    "STAFF_MIN_VIDEO_SHARE": "staff_min_video_share",
    "STAFF_DOMINANCE_RATIO": "staff_dominance_ratio",
    "GREET_MIN_CONTACT_S": "greet_min_contact_s",
    "GREET_PROXIMITY_M": "greet_proximity_m", "TURNAWAY_MAX_S": "turnaway_max_s",
    "LONG_WAIT_S": "long_wait_s", "MICRO_ABSENCE_S": "micro_absence_s",
    "BREAK_ABSENCE_S": "break_absence_s", "GROUP_WINDOW_S": "group_window_s",
    "GROUP_RADIUS_M": "group_radius_m", "MAX_WALK_SPEED_MPS": "max_walk_speed_mps",
}
_changed = []
for _g, _k in _APPLY.items():
    _new = VENUE_PROFILE["venue"].get(_k)
    if _new is not None and globals().get(_g) != _new:
        _changed.append(f"{_g}: {globals().get(_g)} -> {_new}")
        globals()[_g] = _new
DEFAULT_HFOV_DEG = VENUE_PROFILE["camera"]["hfov_deg"]
PERSON_H_M = VENUE_PROFILE["camera"]["person_height_m"]
if VENUE_PROFILE["camera"].get("entry_line_flip") is not None:
    ENTRY_LINE_FLIP = bool(VENUE_PROFILE["camera"]["entry_line_flip"])
if _changed:
    print(f"  profile changed {len(_changed)} setting(s):")
    for _c in _changed:
        print(f"      {_c}")
else:
    print("  (all pipeline defaults — drop a profile_<video>.json beside the "
          "video to change any of them without editing this notebook)")

print(f"\n{len(VIDEO_QUEUE)} video(s) queued: "
      + ", ".join(v["camera_id"] for v in VIDEO_QUEUE))


In [ ]:
# Cell 4 — ZONES: generic loader + AI zone ROLE classifier
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

GENERIC_TEMPLATE = {
    "entry_line": [[100, 600], [1000, 600]],
    "polygons": {
        "wait_zone":      [[60, 300], [500, 300], [500, 850], [60, 850]],
        "staff_zone":     [[950, 250], [1300, 250], [1300, 650], [950, 650]],
        "seating_zone_1": [[520, 100], [900, 100], [900, 420], [520, 420]],
    },
}

def load_zone_config(path, frame_size=None):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(GENERIC_TEMPLATE, indent=2))
        print(f"wrote GENERIC TEMPLATE {path} — edit the coordinates to match this camera view!")
    cfg = json.loads(path.read_text())
    polygons = {name: np.array(pts, dtype=float) for name, pts in cfg.get("polygons", {}).items()}
    # U3: a venue can have more than one door. "entry_lines" is a name -> 2-point
    # map; the old singular "entry_line" still works and becomes {"entry": ...}.
    # Returning ONE line meant a two-door venue could not be counted at all.
    _lines = dict(cfg.get("entry_lines") or {})
    if not _lines and cfg.get("entry_line"):
        _lines["entry"] = cfg["entry_line"]
    entry_lines = {n: [list(map(float, p)) for p in pts]
                   for n, pts in _lines.items() if pts and len(pts) == 2}

    if frame_size:
        fw, fh = frame_size
        ref = cfg.get("frame_size")
        if not ref:
            all_pts = np.vstack(list(polygons.values()) +
                                [np.array(p, dtype=float) for p in entry_lines.values()])
            max_x, max_y = all_pts[:, 0].max(), all_pts[:, 1].max()
            if max_x > fw or max_y > fh:
                ref = (max_x, max_y)
        if ref:
            rw, rh = ref
            sx, sy = fw / rw, fh / rh
            polygons = {name: pts * [sx, sy] for name, pts in polygons.items()}
            entry_lines = {n: [[x * sx, y * sy] for x, y in pts]
                           for n, pts in entry_lines.items()}
            if ref != (fw, fh):
                print(f"↕️  {path.name}: scaled zones from reference "
                      f"{rw:.0f}x{rh:.0f} -> actual {fw}x{fh} "
                      f"(factor {sx:.3f}x, {sy:.3f}y)")

    polygons = {name: pts.astype(int) for name, pts in polygons.items()}
    entry_lines = {n: [[int(x), int(y)] for x, y in pts]
                   for n, pts in entry_lines.items()}
    # U7: an explicit roles map in the zones file wins over keyword guessing, so
    # a zone named in any language still gets its role instead of silently
    # becoming "other" and having its metrics vanish from the report.
    for zname, rs in (cfg.get("roles") or {}).items():
        ZONE_AI_OVERRIDES[zname] = list(rs) if isinstance(rs, (list, tuple)) else [rs]
    return polygons, entry_lines

if "ZONE_ROLE_KEYWORDS" not in globals():
    ZONE_ROLE_KEYWORDS = {
        "entry":   ["entry", "door", "entrance", "gate", "doorway", "passageway"],
        "wait":    ["wait", "queue", "lobby", "line", "holding"],
        "staff":   ["staff", "reception", "host", "counter", "register",
                    "checkout", "till", "cashier", "podium", "desk"],
        "seating": ["table", "seat", "dining", "booth", "seating"],
        "service": ["service", "bar", "kitchen", "prep"],
        "mask":    ["mask", "ignore", "mirror", "reflection", "phantom"],
        "walkway": ["walkway", "corridor", "path", "aisle"],
    }
    ZONE_AI_OVERRIDES = {}

def classify_zones(zone_names):
    roles = {}
    for name in zone_names:
        if name in ZONE_AI_OVERRIDES:
            roles[name] = ZONE_AI_OVERRIDES[name]
            continue
        low = str(name).lower()
        # keep in sync with Cell 2's classify_zones — this later definition
        # overwrites it, and it was missing archway/portal
        is_entry_indicator = any(kw in low for kw in ["gate", "door", "entry", "entrance", "passageway", "archway", "portal"])
        matched = []
        for role, kws in ZONE_ROLE_KEYWORDS.items():
            if role == "seating" and is_entry_indicator:
                if not any(explicit in low for explicit in ["table", "booth", "chair", "seat"]):
                    continue
            if any(kw in low for kw in kws):
                matched.append(role)
        roles[name] = matched or ["other"]
    return roles

def evaluate_zones_with_ai(zones_path_or_cfg, frame_image_path=None):
    """Evaluates camera zone mapping dynamically using Gemini VLM (if API key available)
    or structured AI Contextual-Rules Engine.
    """
    import os, json, base64
    if isinstance(zones_path_or_cfg, (str, Path)):
        p = Path(zones_path_or_cfg)
        if not p.exists():
            return {"roles": {}, "warnings": [f"Zones file {p} not found."]}
        cfg = json.loads(p.read_text())
    else:
        cfg = zones_path_or_cfg
        
    polygons = cfg.get("polygons", {})
    entry_line = cfg.get("entry_line", [])
    zone_names = list(polygons.keys())
    initial_roles = classify_zones(zone_names)
    
    warnings, clashes = [], []
    if len(entry_line) != 2:
        warnings.append("ENTRY LINE MISSING or invalid: 2 points required for arrival counting.")
    
    has_staff = any("staff" in rs or any(k in z.lower() for k in ("reception", "host", "desk"))
                    for z, rs in initial_roles.items())
    if not has_staff:
        warnings.append("NO STAFF/RECEPTION ZONE: Host station metrics will be empty.")
        
    for z, rs in initial_roles.items():
        if len(rs) > 1:
            clashes.append(f"Zone '{z}' matched multiple roles {rs}.")
            
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        except Exception:
            api_key = None
            
    provider = "Contextual-Rules Engine"
    if api_key and frame_image_path and Path(frame_image_path).exists():
        try:
            import urllib.request
            url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"
            with open(frame_image_path, "rb") as f_img:
                img_b64 = base64.b64encode(f_img.read()).decode("utf-8")
            prompt = f"""You are an AI Computer Vision & Restaurant Analytics Specialist.
Evaluate this camera frame ('frame_0.jpg') and zone mapping:
Zones: {json.dumps(zone_names)}
Entry Line: {entry_line}

Task:
1. Verify if zone names match the visual geometry in the camera view.
2. Auto-classify each zone to: 'staff', 'wait', 'entry', 'seating', 'service', 'mask', 'walkway'.
3. Identify clashes or missing zones.

Return JSON ONLY:
{{
  "evaluated_roles": {{"zone_name": ["role"]}},
  "warnings": ["warning string"]
}}
"""
            payload = json.dumps({"contents": [{"parts": [{"text": prompt}, {"inline_data": {"mime_type": "image/jpeg", "data": img_b64}}]}]}).encode("utf-8")
            req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                res = json.loads(resp.read().decode("utf-8"))
                txt = res["candidates"][0]["content"]["parts"][0]["text"]
                c_json = txt[txt.find("{"):txt.rfind("}")+1]
                parsed = json.loads(c_json)
                if "evaluated_roles" in parsed:
                    ZONE_AI_OVERRIDES.update(parsed["evaluated_roles"])
                    initial_roles = classify_zones(zone_names)
                    provider = "Gemini 1.5 Flash VLM"
                if "warnings" in parsed:
                    warnings.extend(parsed["warnings"])
        except Exception as e:
            warnings.append(f"Gemini API check skipped ({e}); used Contextual-Rules Engine.")
            
    return {"roles": initial_roles, "clashes": clashes, "warnings": warnings, "ai_provider": provider}

def zones_with_role(zone_names, role):
    roles = classify_zones(list(zone_names))
    return {z for z, rs in roles.items() if role in rs}

def first_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise FileNotFoundError(f"cannot read {video_path}")
    return frame

def preview_zones(video_path, zones_path):
    frame = cv2.cvtColor(first_frame(video_path), cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]
    polygons, entry_lines = load_zone_config(zones_path, frame_size=(w, h))
    ai_eval = evaluate_zones_with_ai(zones_path, frame_image_path=video_path)
    roles = ai_eval["roles"]
    fig, ax = plt.subplots(figsize=(15, 9))
    ax.imshow(frame)
    for name, poly in polygons.items():
        closed = np.vstack([poly, poly[:1]])
        ax.plot(closed[:, 0], closed[:, 1], linewidth=2.5)
        r_list = roles.get(name, ["other"])
        label = f"{name}  [{','.join(r_list)}]"
        ax.text(*poly.mean(axis=0), label, color="white", fontsize=10, weight="bold",
                ha="center", bbox=dict(facecolor="black", alpha=0.65, pad=2))
    for _ln, _pts in entry_lines.items():
        (x1, y1), (x2, y2) = _pts
        ax.plot([x1, x2], [y1, y2], "r--", linewidth=3)
        ax.text(x1, y1 - 10, _ln.upper(), color="red", weight="bold")
    ax.set_xticks(range(0, frame.shape[1], 100)); ax.set_yticks(range(0, frame.shape[0], 100))
    ax.grid(alpha=0.3)
    ax.set_title(f"ZONE CHECK ({ai_eval['ai_provider']}) — {Path(str(video_path)).name}")
    plt.show()

for v in VIDEO_QUEUE:
    if v["video"].exists():
        preview_zones(v["video"], v["zones_path"])



In [ ]:
# Cell 5 — analytics logic (fully unit-tested)
"""Core zone-analytics logic for the restaurant video POC.

Pure Python (no numpy/pandas/cv2) so every metric is unit-testable
without a video. The notebook's video engine produces per-frame zone
occupancy; everything below turns that into events and answers.

Time is always seconds from video start (float).
Roles are "customer" / "staff" / "unknown".
"""

# v42: staff string names always win as canonical ID over numeric track IDs
def _canon_priority_key(track_windows):
    """Sort key: string names (staff) first, then by appearance time."""
    def key(t):
        if isinstance(t, str) and not str(t).isdigit():
            return (0, 0.0, str(t))        # staff names always first
        return (1, track_windows.get(t, (0,0))[0], str(t))
    return key

def find_optimal_threshold(same_sims, diff_sims):
    """Find the threshold that maximizes separation (EER approach).

    Sweeps thresholds to find where false-accept-rate == false-reject-rate.
    This is the 'best spot' where the signal is cleanest.

    Returns: (optimal_threshold, metrics_dict)
    """
    import numpy as _np
    if not same_sims or not diff_sims:
        return 0.55, {"note": "insufficient data for EER sweep"}
    same_arr = _np.array(same_sims)
    diff_arr = _np.array(diff_sims)
    thresholds = _np.arange(0.0, 1.001, 0.01)
    best_thresh, best_score = 0.55, -1.0
    for t in thresholds:
        fr = _np.mean(same_arr < t)     # false reject: same-person below threshold
        fa = _np.mean(diff_arr >= t)    # false accept: diff-person above threshold
        score = 1.0 - (fr + fa) / 2.0  # balanced accuracy
        if score > best_score:
            best_thresh, best_score = float(t), float(score)
    return round(best_thresh, 2), {
        "balanced_accuracy": round(best_score, 4),
        "same_person_median": round(float(_np.median(same_arr)), 4),
        "diff_person_median": round(float(_np.median(diff_arr)), 4),
        "separation_gap": round(float(_np.median(same_arr) - _np.median(diff_arr)), 4),
    }

import math
from collections import Counter, defaultdict

# ---------------------------------------------------------------------------
# Interval helpers
# ---------------------------------------------------------------------------

def merge_intervals(intervals, gap=0.0):
    """Merge [start, end] intervals, joining any separated by <= gap seconds."""
    if not intervals:
        return []
    ordered = sorted(intervals, key=lambda iv: iv[0])
    merged = [list(ordered[0])]
    for start, end in ordered[1:]:
        if start - merged[-1][1] <= gap:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [tuple(iv) for iv in merged]


def total_duration(intervals):
    return sum(end - start for start, end in intervals)


def covered_windows(run):
    """The stretches of the night we actually have video for. A missing hour is
    not a quiet hour, and no metric may average over footage that doesn't
    exist."""
    cb = run.get("chunk_bounds")
    if cb:
        return merge_intervals([(float(s), float(e)) for _k, s, e in cb])
    return [(0.0, float(run.get("t_end", 0.0)))]


def clip_to(intervals, windows):
    """Intersection of two interval lists."""
    out = []
    for a0, a1 in intervals:
        for b0, b1 in windows:
            lo, hi = max(a0, b0), min(a1, b1)
            if hi > lo:
                out.append((lo, hi))
    return sorted(out)


def complement_intervals(intervals, t_start, t_end):
    """Gaps inside [t_start, t_end] not covered by intervals (assumed merged)."""
    gaps, cursor = [], t_start
    for start, end in sorted(intervals, key=lambda iv: iv[0]):
        if start > cursor:
            gaps.append((cursor, min(start, t_end)))
        cursor = max(cursor, end)
        if cursor >= t_end:
            break
    if cursor < t_end:
        gaps.append((cursor, t_end))
    return [(s, e) for s, e in gaps if e - s > 1e-9]


# ---------------------------------------------------------------------------
# Occupancy -> events
# ---------------------------------------------------------------------------

class OccupancyRecorder:
    """Collects per-frame zone occupancy and consolidates it into events.

    add(t, zone, track_id): call once per (processed frame, zone, visible track).
    Consolidation merges consecutive sightings into [t_in, t_out] events,
    bridging gaps <= gap_merge_s (occlusion tolerance) and dropping events
    shorter than min_event_s (detection flicker).
    """

    def __init__(self, frame_step_s, gap_merge_s=15.0, min_event_s=2.0):
        self.frame_step_s = frame_step_s
        self.gap_merge_s = gap_merge_s
        self.min_event_s = min_event_s
        self._sightings = defaultdict(list)   # (zone, track_id) -> [t, ...]
        self.role_votes = defaultdict(Counter)  # track_id -> Counter(role)

    def add(self, t, zone, track_id):
        self._sightings[(zone, track_id)].append(t)

    def vote_role(self, track_id, role):
        self.role_votes[track_id][role] += 1

    def final_roles(self):
        return {tid: votes.most_common(1)[0][0]
                for tid, votes in self.role_votes.items()}

    def events(self, min_event_by_zone=None):
        """-> list of dicts {track_id, role, zone, t_in, t_out, duration}.
        min_event_by_zone: optional {zone: seconds} override — entry zones need
        a lower bar because a doorway transit at walking pace is ~1s and the
        global 2.0s minimum silently deleted every arrival event."""
        roles = self.final_roles()
        out = []
        for (zone, tid), times in self._sightings.items():
            _min_s = (min_event_by_zone or {}).get(zone, self.min_event_s)
            # each sighting covers one frame step
            ivs = merge_intervals(
                [(t, t + self.frame_step_s) for t in sorted(times)],
                gap=self.gap_merge_s,
            )
            for t_in, t_out in ivs:
                if t_out - t_in >= _min_s:
                    out.append({
                        "track_id": tid,
                        "role": roles.get(tid, "unknown"),
                        "zone": zone,
                        "t_in": round(t_in, 2),
                        "t_out": round(t_out, 2),
                        "duration": round(t_out - t_in, 2),
                    })
        out.sort(key=lambda e: (e["t_in"], e["zone"], e["track_id"]))
        return out


# ---------------------------------------------------------------------------
# Door-camera metrics
# ---------------------------------------------------------------------------

def entered_count(crossings, roles=None):
    """crossings: list of {t, track_id, direction} from the entry LineZone.
    Counts unique track_ids that crossed inward; staff filtered out if roles given.
    """
    seen = set()
    for c in crossings:
        if c["direction"] != "in":
            continue
        tid = c["track_id"]
        if roles and roles.get(tid) == "staff":
            continue
        seen.add(tid)
    return len(seen)


def seated_count(events, table_zones, min_seated_s=60.0):
    """Unique customers who dwelled >= min_seated_s in any table/dining zone."""
    seated = set()
    for e in events:
        if (e["zone"] in table_zones and e["role"] == "customer"
                and e["duration"] >= min_seated_s):
            seated.add(e["track_id"])
    return len(seated)


def waited_over(events, waiting_zone, threshold_s=600.0, gap=30.0):
    """Customers whose merged dwell in the waiting zone >= threshold_s.
    Returns (count, {track_id: total_wait_s}) - dict includes ALL waiters
    so the demo can show the distribution, not just the threshold count.
    """
    per_track = defaultdict(list)
    for e in events:
        if e["zone"] == waiting_zone and e["role"] == "customer":
            per_track[e["track_id"]].append((e["t_in"], e["t_out"]))
    waits = {tid: round(total_duration(merge_intervals(ivs, gap=gap)), 1)
             for tid, ivs in per_track.items()}
    over = {tid: w for tid, w in waits.items() if w >= threshold_s}
    return len(over), waits


def reception_absence(events, reception_zone, t_start, t_end, min_gap_s=30.0):
    """How long was reception unstaffed?  Union of STAFF presence intervals in
    the reception zone, complemented over [t_start, t_end].
    Returns dict: total_away_s, longest_away_s, gaps (only gaps >= min_gap_s).
    """
    staffed = [(e["t_in"], e["t_out"]) for e in events
               if e["zone"] == reception_zone and e["role"] == "staff"]
    gaps = [(s, e) for s, e in
            complement_intervals(merge_intervals(staffed), t_start, t_end)
            if e - s >= 1.0]   # drop sub-second boundary slivers
    big_gaps = [(round(s, 1), round(e, 1)) for s, e in gaps if e - s >= min_gap_s]
    return {
        "total_away_s": round(sum(e - s for s, e in gaps), 1),
        "longest_away_s": round(max((e - s for s, e in gaps), default=0.0), 1),
        "gaps": big_gaps,
    }


# ---------------------------------------------------------------------------
# Table-camera metrics
# ---------------------------------------------------------------------------

def detect_parties(events, table_zone, party_gap_s=120.0, min_party_s=60.0):
    """A 'party' = a maximal window where the table has customers, with
    interior gaps < party_gap_s merged (people lean out of frame, etc.)."""
    cust = [(e["t_in"], e["t_out"]) for e in events
            if e["zone"] == table_zone and e["role"] == "customer"]
    return [iv for iv in merge_intervals(cust, gap=party_gap_s)
            if iv[1] - iv[0] >= min_party_s]


def staff_visits(events, table_zone, window, visit_min_s=8.0, merge_gap_s=30.0):
    """Distinct staff visits to a table overlapping [window]. Consecutive staff
    presence separated by < merge_gap_s counts as ONE visit."""
    w_start, w_end = window
    ivs = [(e["t_in"], e["t_out"]) for e in events
           if e["zone"] == table_zone and e["role"] == "staff"
           and e["t_out"] > w_start and e["t_in"] < w_end]
    return [iv for iv in merge_intervals(ivs, gap=merge_gap_s)
            if iv[1] - iv[0] >= visit_min_s]


def table_service_metrics(events, table_zones, party_gap_s=120.0,
                          min_party_s=60.0, visit_min_s=8.0):
    """Per party per table: seating->first-visit ('order' proxy),
    first->second visit ('food' proxy), total visit count.
    Returns (per_party_rows, averages_dict)."""
    rows = []
    for table in sorted(table_zones):
        for party in detect_parties(events, table, party_gap_s, min_party_s):
            visits = staff_visits(events, table, party, visit_min_s)
            seat_t = party[0]
            order_t = visits[0][0] if len(visits) >= 1 else None
            food_t = visits[1][0] if len(visits) >= 2 else None
            rows.append({
                "table": table,
                "party_start": round(seat_t, 1),
                "party_end": round(party[1], 1),
                "n_staff_visits": len(visits),
                "seating_to_order_s": (round(order_t - seat_t, 1)
                                       if order_t is not None else None),
                "order_to_food_s": (round(food_t - order_t, 1)
                                    if food_t is not None else None),
            })

    def avg(key):
        vals = [r[key] for r in rows if r[key] is not None]
        return round(sum(vals) / len(vals), 1) if vals else None

    averages = {
        "avg_seating_to_order_s": avg("seating_to_order_s"),
        "avg_order_to_food_s": avg("order_to_food_s"),
        "avg_visits_per_party": (round(sum(r["n_staff_visits"] for r in rows)
                                       / len(rows), 2) if rows else None),
        "n_parties": len(rows),
    }
    return rows, averages



# v42 mix track_id sort key (some are integers, some are string names like 'jane')
def track_sort_key(tid):
    if isinstance(tid, str) and not tid.isdigit():
        return (0, tid)
    try:
        return (1, int(tid))
    except ValueError:
        return (2, str(tid))

def apply_staff_zone_override(events, staff_zones, min_staff_dwell_s=60.0,
                              min_share=0.6, min_share_dwell_s=15.0):
    """Role fallback: a track is re-labeled 'staff' when EITHER
    (a) its total dwell in staff-only zones reaches min_staff_dwell_s, OR
    (b) staff zones account for >= min_share of its total event time AND at
        least min_share_dwell_s absolute — this catches SHORT track fragments
        of a desk-anchored person (a fragmented hostess spends ~100% of every
        fragment at reception but each fragment alone misses rule (a)).
    Returns a new events list; input is not mutated."""
    if not staff_zones:
        return events
    staff_dwell = defaultdict(float)
    first_seen, last_seen = {}, {}
    for e in events:
        tid = e["track_id"]
        first_seen[tid] = min(first_seen.get(tid, e["t_in"]), e["t_in"])
        last_seen[tid] = max(last_seen.get(tid, e["t_out"]), e["t_out"])
        if e["zone"] in staff_zones:
            staff_dwell[tid] += e["duration"]
    staff_ids = set()
    for tid, d in staff_dwell.items():
        # share of LIFETIME, not of summed zone dwells — overlapping zones
        # (a host stand inside a larger lobby polygon) would otherwise dilute
        # a desk-anchored person's share below the threshold
        lifetime = max(last_seen[tid] - first_seen[tid], 1e-9)
        share = min(d / lifetime, 1.0)
        if d >= min_staff_dwell_s or (d >= min_share_dwell_s
                                      and share >= min_share):
            staff_ids.add(tid)
    return [dict(e, role="staff") if e["track_id"] in staff_ids else dict(e)
            for e in events]


def occupancy_timeline(events, t_end, step_s=10.0):
    """Distinct people present per role at each time step (for occupancy charts).
    Returns (times, {role: [count per step]}). Same person in two zones in one
    step counts once."""
    n = max(1, int(math.ceil(t_end / step_s)))
    times = [round(i * step_s, 2) for i in range(n)]
    roles = sorted({e["role"] for e in events}) or ["customer"]
    counts = {r: [0] * n for r in roles}
    for i, t0 in enumerate(times):
        t1 = t0 + step_s
        seen = defaultdict(set)
        for e in events:
            if e["t_out"] > t0 and e["t_in"] < t1:
                seen[e["role"]].add(e["track_id"])
        for r in roles:
            counts[r][i] = len(seen[r])
    return times, counts


# ---------------------------------------------------------------------------
# Re-ID stitching: merge track fragments that are the same person
# ---------------------------------------------------------------------------

def _cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1e-9
    nb = math.sqrt(sum(x * x for x in b)) or 1e-9
    return dot / (na * nb)


# ---------------------------------------------------------------------------
# Cross-validation (v26): independent HSV-histogram check on OSNet merges.
# Deliberately a DIFFERENT feature space from the OSNet embeddings used by
# merge_fragmented_tracks — the point is corroboration, not a second vote
# with the same blind spots. Runs on the exact crops already banked for this
# run (track_crops), so track IDs match the OSNet mapping 1:1 with no manual
# reconciliation, unlike re-running tracking in a separate script.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# LIVE identity memory (v27): runs INSIDE the per-frame tracking loop, not
# as a post-hoc offline stitch. Purpose: when the online tracker (BotSORT)
# mints a brand-new raw track_id, decide BEFORE that id ever reaches
# events/zones/render whether it's actually a person we already know —
# someone whose id just got swapped mid-visibility, not someone who left
# and came back. That second case (real gap) is what merge_fragmented_tracks
# already handles offline; this handles the case it structurally can't
# (overlapping/near-continuous visibility windows).
# ---------------------------------------------------------------------------


def _dynamic_accept_thresh(gap, base_thresh):
    near_s, far_s = NEAR_GAP_S, FAR_GAP_S
    if gap <= near_s:
        return base_thresh - NEAR_GAP_BONUS
    if gap >= far_s:
        return base_thresh + FAR_GAP_PENALTY
    frac = (gap - near_s) / (far_s - near_s)
    return (base_thresh - NEAR_GAP_BONUS) + frac * (NEAR_GAP_BONUS + FAR_GAP_PENALTY)


def _boxes_occluding(a, b):
    """v44: True if two xyxy boxes mutually occlude — either their IoU is high
    or one is largely contained in the other. During such an overlap a body
    crop is contaminated by the neighbouring body, so appearance must NOT be
    trusted to update anchors or drive re-assignment."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return False
    aarea = max(1.0, (ax2 - ax1) * (ay2 - ay1))
    barea = max(1.0, (bx2 - bx1) * (by2 - by1))
    iou = inter / (aarea + barea - inter)
    contain = inter / min(aarea, barea)
    return iou >= OCCLUSION_IOU or contain >= OCCLUSION_CONTAIN


_NOVEC = object()   # v55 sentinel: 'no vector supplied, compute it yourself'


class _IdentityMemory:

    """Rolling memory of recently-seen identities: one best snapshot
    (anchor embedding), last known position, and last-seen time per
    canonical id. `resolve()` is called once per detection per frame and
    returns the canonical id that detection should be recorded under.

    Design intentionally mirrors the anchor-snapshot logic already used
    offline (ANCHOR_SIM_THRESHOLD / merge_fragmented_tracks): single
    best-quality crop, not a pooled/averaged embedding, compared 1:1.
    A live false-positive here is worse than an offline one (no
    cross-validation pass reviews it after the fact), so the bar
    (LIVE_REID_SIM_THRESHOLD) is set stricter than the offline gallery
    threshold on purpose.
    """
    def __init__(self, embed_fn, sim_threshold=0.72, max_dist_px=140.0,
                memory_ttl_s=6.0, dist_scale=1.0, plane=None,
                max_speed_mps=2.2):
        self.embed_fn = embed_fn
        self.sim_threshold = sim_threshold
        # G2: with a ground plane the gate becomes "could a person physically
        # have walked that far in the time available" — one statement about the
        # world, correct in every part of the frame. max_dist_px survives only
        # as the fallback for footage where no plane could be fitted.
        self.plane = plane
        self.max_speed_mps = max_speed_mps
        self.max_dist_px = max_dist_px
        self.memory_ttl_s = memory_ttl_s
        self.dist_scale = dist_scale   # v45: resolution-relative scaling factor
        self.bank = {}          # canonical_id -> {"anchor","pos","t","best_score"}
        self.raw_to_canon = {}  # raw tracker track_id -> canonical_id
        self.reassignments = 0  # count of births redirected to an existing id

    def _prune(self, now_t):
        dead = [cid for cid, rec in self.bank.items()
                if now_t - rec["t"] > self.memory_ttl_s]
        for cid in dead:
            del self.bank[cid]

    def _remember(self, canon, vec, pos, now_t, score):
        rec = self.bank.get(canon)
        if vec is not None and (rec is None or score >= rec.get("best_score", -1)):
            self.bank[canon] = {"anchor": vec, "pos": pos, "t": now_t,
                                "best_score": score, "t_emb": now_t}
        elif rec is not None:
            rec["pos"] = pos
            rec["t"] = now_t
            if vec is not None:
                rec["t_emb"] = now_t
        elif canon not in self.bank:
            self.bank[canon] = {"anchor": vec, "pos": pos, "t": now_t,
                                "best_score": score}

    def resolve(self, raw_tid, crop, conf, bbox, now_t, frozen=False,
                blocked_canons=None, vec=_NOVEC):
        """crop may be None (detection too small/low-conf to embed) — still
        tracked positionally, just can't confirm/deny via appearance.

        v44 params:
          frozen         -- this detection is mutually occluded THIS frame; its
                            crop is contaminated by the overlapping body, so we
                            NEVER overwrite the appearance anchor and NEVER let
                            appearance drive a re-assignment. Position stays
                            fresh so the track survives the overlap.
          blocked_canons -- canonical ids already visible under a DIFFERENT raw
                            id this frame. A person can't be in two tracks at
                            once, so a birth may not merge into any of these
                            (co-visibility hard constraint)."""
        cx = (bbox[0] + bbox[2]) / 2.0
        cy = (bbox[1] + bbox[3]) / 2.0
        self._prune(now_t)
        score = conf * max(1.0, bbox[3] - bbox[1])

        if raw_tid in self.raw_to_canon:
            canon = self.raw_to_canon[raw_tid]
            if frozen:
                # occluded: keep last-known position fresh but do NOT touch the
                # appearance anchor with a contaminated crop.
                rec = self.bank.get(canon)
                if rec is not None:
                    rec["pos"] = (cx, cy)
                    rec["t"] = now_t
            else:
                if vec is _NOVEC:
                    vec = self.embed_fn(crop) if crop is not None else None
                self._remember(canon, vec, (cx, cy), now_t, score)
            return canon

        # Brand-new raw track_id -- before minting a new canonical identity,
        # check whether it plausibly belongs to someone already in memory.
        if frozen:
            # Don't trust an occluded/contaminated crop to merge a birth into an
            # existing identity. Mint a positional-only id now; a later clean
            # frame can still be matched normally.
            canon = raw_tid
            self.raw_to_canon[raw_tid] = canon
            rec = self.bank.get(canon)
            if rec is None:
                self.bank[canon] = {"anchor": None, "pos": (cx, cy),
                                    "t": now_t, "best_score": -1}
            else:
                rec["pos"] = (cx, cy)
                rec["t"] = now_t
            return canon

        if vec is _NOVEC:
            vec = self.embed_fn(crop) if crop is not None else None
        best_cid, best_sim = None, 0.0
        if vec is not None:
            for cid, rec in self.bank.items():
                if rec["anchor"] is None:
                    continue
                if blocked_canons and cid in blocked_canons:
                    continue  # v44 co-visibility: id already on-screen elsewhere
                dist = math.hypot(cx - rec["pos"][0], cy - rec["pos"][1])
                gap = now_t - rec["t"]
                # G2: metric first, pixels only if the plane is unavailable here
                # (e.g. the point is above the horizon).
                _md = None
                if self.plane is not None and self.plane.ok:
                    _md = self.plane.dist_m((cx, cy), rec["pos"])
                if _md is not None:
                    # a standing-start allowance so a stationary person whose box
                    # jitters is never gated out
                    if _md > self.max_speed_mps * max(gap, 0.35) + 0.35:
                        continue
                elif dist > self.max_dist_px:
                    continue
                required = _dynamic_accept_thresh(gap, self.sim_threshold)
                
                # Soft spatial penalty (v45: resolution-scaled speed)
                spatial_penalty = 0.0
                plausible_dist = MAX_PLAUSIBLE_SPEED_PX * self.dist_scale * gap
                if _md is not None and self.plane is not None:
                    # express the same penalty in metres so it does not double-
                    # count the perspective the metric gate already handled
                    _pl = self.max_speed_mps * max(gap, 0.35) + 0.35
                    excess = max(0.0, _md - _pl) / max(_pl, 1e-6)
                    spatial_penalty = min(MAX_SPATIAL_PENALTY,
                                          excess * SPATIAL_PENALTY_SCALE)
                    required += spatial_penalty
                    sim = _cosine(vec, rec["anchor"])
                    if sim >= required and sim > best_sim:
                        best_sim, best_cid = sim, cid
                    continue
                if plausible_dist > 0:
                    excess_ratio = max(0.0, dist - plausible_dist) / plausible_dist
                    spatial_penalty = min(MAX_SPATIAL_PENALTY, excess_ratio * SPATIAL_PENALTY_SCALE)
                required += spatial_penalty
                
                sim = _cosine(vec, rec["anchor"])
                if sim >= required and sim > best_sim:
                    best_sim, best_cid = sim, cid

        if best_cid is not None:
            canon = best_cid
            self.reassignments += 1
        else:
            canon = raw_tid

        self.raw_to_canon[raw_tid] = canon
        self._remember(canon, vec, (cx, cy), now_t, score)
        return canon

    def try_swap(self, a_raw, a_crop, b_raw, b_crop, margin=0.10):
        """v53: post-occlusion swap re-validation. When two tracks exit an
        overlap, the tracker may have TRADED their ids mid-occlusion, and
        resolve() would rubber-stamp the trade forever (known raw ids pass
        straight through). Here both fresh clean crops are checked against
        both stored anchors; only if EACH matches the OTHER identity better
        by `margin` is the trade undone. Symmetric requirement = extremely
        conservative, so a false un-swap is far less likely than the swap
        it corrects."""
        ca = self.raw_to_canon.get(a_raw)
        cb = self.raw_to_canon.get(b_raw)
        if ca is None or cb is None or ca == cb:
            return False
        ra, rb = self.bank.get(ca), self.bank.get(cb)
        if not ra or not rb or ra["anchor"] is None or rb["anchor"] is None:
            return False
        va = self.embed_fn(a_crop) if a_crop is not None else None
        vb = self.embed_fn(b_crop) if b_crop is not None else None
        if va is None or vb is None:
            return False
        a_own   = _cosine(va, ra["anchor"])
        a_other = _cosine(va, rb["anchor"])
        b_own   = _cosine(vb, rb["anchor"])
        b_other = _cosine(vb, ra["anchor"])
        if a_other - a_own > margin and b_other - b_own > margin:
            self.raw_to_canon[a_raw], self.raw_to_canon[b_raw] = cb, ca
            self.swap_corrections = getattr(self, "swap_corrections", 0) + 1
            return True
        return False


def _hsv_embed_single(crop):
    """One crop -> one HSV color-histogram vector (torso region only)."""
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    torso = hsv[64:192, 16:112]
    hist = [cv2.calcHist([torso], [ch], None, [16],
                         [0, 180 if ch == 0 else 256]) for ch in range(3)]
    v = np.concatenate([h.flatten() for h in hist])
    n = float(np.linalg.norm(v)) or 1e-9
    return (v / n).tolist()


def _hsv_gallery_sim(crops_a, crops_b):
    """Best pairwise HSV cosine across two crop lists — same gallery
    philosophy as _gallery_sim, applied to the HSV feature space instead
    of OSNet, so pose/viewpoint spread doesn't wash out a real match here
    either."""
    if not crops_a or not crops_b:
        return 0.0
    vecs_a = [_hsv_embed_single(c) for c in crops_a]
    vecs_b = [_hsv_embed_single(c) for c in crops_b]
    return max(_cosine(va, vb) for va in vecs_a for vb in vecs_b)



def _handoff_appearance_contradicts(a, b, anchor_embeddings):
    """v47: veto a PURE-SPATIAL hand-off/stationary merge when both tracks carry
    body embeddings whose similarity is clearly 'different person'. When either
    track has no embedding (small/dim fragment — the case the spatial rule exists
    for), returns False so the spatial merge still stands. Without this, in a
    crowd a person vanishing behind a fixture and a DIFFERENT person appearing
    near that spot within the gap/px window get merged into one identity — the
    exact failure the calibration 'worst same-person' audit reveals."""
    if not ENABLE_HANDOFF_APPEARANCE_VETO or not anchor_embeddings:
        return False
    va = anchor_embeddings.get(a)
    vb = anchor_embeddings.get(b)
    if va is None or vb is None:
        return False
    return _cosine(va, vb) < HANDOFF_VETO_SIM


def _appearance_hsv_contradicts(a, b, raw_crops):
    """v53: veto an APPEARANCE-tier merge when the independent HSV colour
    space actively disagrees. Abstains (returns False) when either track has
    no banked crop, so tracks without colour evidence are never penalised —
    this only removes merges that a second, different feature space says are
    wrong. Directly targets the hub-id over-merge seen in the run log."""
    if not ENABLE_APPEARANCE_HSV_VETO or not raw_crops:
        return False
    ca, cb = raw_crops.get(a), raw_crops.get(b)
    if not ca or not cb:
        return False
    try:
        return _hsv_gallery_sim(ca[:3], cb[:3]) < APPEARANCE_HSV_VETO_SIM
    except Exception:
        return False


def _handoff_hsv_contradicts(a, b, raw_crops):
    """v48: independent 2nd opinion on a pure-spatial merge. Within a
    hand-off-sized gap clothing color cannot change, so clearly-different
    torso histograms (< HANDOFF_HSV_VETO_SIM) mean different people. Uses a
    DIFFERENT feature space from the CLIP veto (colors vs learned Re-ID), so
    the two vetoes cover each other's blind spots. Crop-less -> no veto."""
    if not ENABLE_HANDOFF_HSV_VETO or not raw_crops:
        return False
    ca, cb = raw_crops.get(a), raw_crops.get(b)
    if not ca or not cb:
        return False
    try:
        return _hsv_gallery_sim(ca[:3], cb[:3]) < HANDOFF_HSV_VETO_SIM
    except Exception:
        return False


def calibrate_appearance_threshold(windows, positions, anchor_embeddings,
                                   handoff_gap_s=2.5, handoff_px=90.0,
                                   stationary_px=30.0, role_hint=None,
                                   duplicate_px=40.0):
    """Diagnostic (v30): measures THIS run's actual same-person vs
    different-person cosine-similarity distribution for whichever appearance
    backbone loaded (CLIP-ReID or OSNet), instead of reusing OSNet's
    2026-07-13 calibration (~0.72 same / ~0.41 different) for a possibly
    different embedding space.

    Two ground-truth sets, BOTH independent of appearance (no circularity):
      different-person pairs: any two tracks CO-VISIBLE in the same frame
        (their visibility windows overlap) must be different people --
        physically impossible for one person to occupy two tracks at once.
      same-person pairs: tracks linked by the hand-off/stationary rule --
        near-instant (<= handoff_gap_s), near-zero-distance (<= handoff_px,
        or <= stationary_px with no gap) reappearance right where the other
        track died. This is SPATIAL evidence, not appearance evidence, so
        checking appearance similarity on these pairs is a fair test.

    role_hint: (v38) optional {track_id: "staff"|"customer"} -- a fragment
        pair that has EARNED opposite roles from real zone dwell is excluded
        from the same-person ground truth even if spatially close. A busy
        counter has staff and customers constantly swapping the same few
        square feet; without this guard, "staff fragment ends, a DIFFERENT
        customer's fragment starts nearby a moment later" gets miscounted as
        a genuine same-person hand-off, which drags the measured same-person
        distribution down and makes the auto-calibrated threshold LOOSER
        than the footage actually supports -- the opposite of the intent.
        Same rule merge_fragmented_tracks already applies to real merges;
        this just applies it to the ground truth used to calibrate them.

    duplicate_px: (v40) guards the OTHER ground-truth set. "Co-visible in
        the same frame -> different person" only holds if the two tracks
        are actually two separate bodies. A tracker duplicate -- one
        physical person who briefly got two track IDs (e.g. an
        occlusion-ambiguous re-detect) -- is also technically "co-visible,"
        but both IDs stay glued to the same spot the whole time they
        coexist, start-to-start and end-to-end. Appearance correctly says
        these are the same person (high similarity); the ground truth was
        wrong to call it "different," and that mislabeled pair dragged
        diff_p90 UP, making the auto-calibrated bar stricter than the
        footage supports -- the mirror image of the role-conflict bug this
        same guard-pattern already fixed on the same-person side.

    Purely diagnostic -- returns numbers, never touches a threshold itself.
    """
    def _role_conflict(a, b):
        if not role_hint:
            return False
        ra, rb = role_hint.get(a), role_hint.get(b)
        return bool(ra) and bool(rb) and ra != rb

    def _looks_duplicate(pa, pb):
        # pa/pb are (first_pos, last_pos). Two IDs co-located at BOTH ends
        # of their shared lifetime aren't "two people who happened to be
        # close" -- that's one body wearing two IDs.
        d_start = math.hypot(pa[0][0] - pb[0][0], pa[0][1] - pb[0][1])
        d_end = math.hypot(pa[1][0] - pb[1][0], pa[1][1] - pb[1][1])
        return d_start <= duplicate_px and d_end <= duplicate_px

    ids = [t for t in windows if t in anchor_embeddings]
    diff_sims, diff_pairs = [], []
    duplicate_excluded = 0
    for i, a in enumerate(ids):
        wa, pa = windows[a], positions.get(a)
        for b in ids[i + 1:]:
            wb = windows[b]
            if wa[0] < wb[1] and wb[0] < wa[1]:   # overlap -> different person
                pb = positions.get(b)
                if pa and pb and _looks_duplicate(pa, pb):
                    duplicate_excluded += 1
                    continue
                s = _cosine(anchor_embeddings[a], anchor_embeddings[b])
                diff_sims.append(s)
                diff_pairs.append((a, b, s))

    same_sims, same_pairs = [], []
    role_conflicts_excluded = 0
    for i, a in enumerate(ids):
        wa, pa = windows[a], positions.get(a)
        if not pa:
            continue
        for b in ids[i + 1:]:
            wb, pb = windows[b], positions.get(b)
            if not pb:
                continue
            if _role_conflict(a, b):
                role_conflicts_excluded += 1
                continue
            if wa[0] <= wb[0]:
                first_end, first_pos = wa[1], pa[1]
                second_start, second_pos = wb[0], pb[0]
            else:
                first_end, first_pos = wb[1], pb[1]
                second_start, second_pos = wa[0], pa[0]
            gap = second_start - first_end
            if gap < 0:
                continue
            dist = math.hypot(first_pos[0] - second_pos[0],
                              first_pos[1] - second_pos[1])
            if (((gap <= handoff_gap_s and dist <= handoff_px) or dist <= stationary_px)
                    and not _handoff_appearance_contradicts(a, b, anchor_embeddings)):
                s = _cosine(anchor_embeddings[a], anchor_embeddings[b])
                same_sims.append(s)
                same_pairs.append((a, b, s))

    def _pct(xs, p):
        if not xs:
            return None
        xs = sorted(xs)
        k = (len(xs) - 1) * p
        f, c = int(k), min(int(k) + 1, len(xs) - 1)
        return xs[f] + (xs[c] - xs[f]) * (k - f)

    return {
        "same_n": len(same_sims), "diff_n": len(diff_sims),
        "same_p10": _pct(same_sims, 0.10), "same_p50": _pct(same_sims, 0.50),
        "diff_p50": _pct(diff_sims, 0.50), "diff_p90": _pct(diff_sims, 0.90),
        "same_sims": same_sims,
        "diff_sims": diff_sims,
        # (v36) the actual pairs behind the numbers, worst-first, so you can
        # LOOK at what's confusing the model instead of only reading a
        # percentile. "Worst same-person" = spatially-certain same person
        # but LOWEST appearance similarity (should be high, isn't).
        # "Worst different-person" = certainly different people but HIGHEST
        # appearance similarity (should be low, isn't).
        "same_pairs_worst": sorted(same_pairs, key=lambda p: p[2])[:8],
        "diff_pairs_worst": sorted(diff_pairs, key=lambda p: -p[2])[:8],
        # (v38) how many spatially-close pairs were kept OUT of the
        # same-person ground truth because they'd earned opposite roles --
        # a high number here on a counter-heavy venue confirms this guard
        # is doing real work, not just defensive code.
        "role_conflicts_excluded": role_conflicts_excluded,
        # (v40) how many co-visible pairs were kept OUT of the
        # different-person ground truth because they look like one body
        # wearing two track IDs rather than two people -- a high number
        # here on footage with occlusion-heavy tracking confirms this
        # guard is doing real work.
        "duplicate_excluded": duplicate_excluded,
    }


def suggest_appearance_thresholds(cal, current_thresh, current_anchor,
                                  floor=0.35, ceiling=0.85):
    """(v37) Turns calibrate_appearance_threshold()'s report into thresholds
    that are actually USED this run. Uses find_optimal_threshold (EER balanced accuracy)
    if sims lists are present, otherwise falls back to percentile heuristics.
    """
    same_sims, diff_sims = cal.get("same_sims"), cal.get("diff_sims")
    if same_sims and diff_sims:
        opt_thresh, metrics = find_optimal_threshold(same_sims, diff_sims)
        suggested = opt_thresh
        anchor_suggested = opt_thresh + 0.08
        print(f"  [EER Sweep] Optimal threshold found at {opt_thresh:.3f} (balanced accuracy = {metrics['balanced_accuracy']:.4f})")
    else:
        same_p10, diff_p90 = cal.get("same_p10"), cal.get("diff_p90")
        if same_p10 is None or diff_p90 is None:
            return current_thresh, current_anchor
        if same_p10 > diff_p90:
            suggested = (same_p10 + diff_p90) / 2
            anchor_suggested = suggested + 0.08
        else:
            suggested = diff_p90 + 0.02
            anchor_suggested = suggested + 0.10
            
    suggested = max(floor, min(ceiling, suggested))
    anchor_suggested = max(floor, min(ceiling, anchor_suggested))
    return round(suggested, 3), round(anchor_suggested, 3)


def cross_validate_faces(mapping, face_embeddings, windows,
                         sim_threshold=0.35, max_gap_s=480.0):
    """Diagnostic (v30): where a face embedding exists for BOTH tracks in a
    pair, face similarity is the strongest identity signal this pipeline has
    -- robust to pose AND clothing change, unlike body-appearance backbones.
    Coverage will be sparse (many CCTV crops never resolve a usable face) --
    this only ever covers a fraction of merges, and that's fine: it's a
    corroborating signal, same "tag not gate" philosophy as the HSV check.
    Never touches `mapping`.
    """
    from collections import defaultdict as _dd
    groups = _dd(list)
    for tid, canon in mapping.items():
        groups[canon].append(tid)

    agree, disagree = [], []
    checked = set()
    for canon, members in groups.items():
        if len(members) < 2:
            continue
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                a, b = members[i], members[j]
                checked.add(frozenset((a, b)))
                if a not in face_embeddings or b not in face_embeddings:
                    continue
                sim = _cosine(face_embeddings[a], face_embeddings[b])
                (agree if sim >= sim_threshold else disagree).append((a, b, sim))

    face_only = []
    ids = list(face_embeddings)
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            if frozenset((a, b)) in checked:
                continue
            wa, wb = windows.get(a), windows.get(b)
            if not wa or not wb:
                continue
            gap = (wb[0] - wa[1]) if wa[0] <= wb[0] else (wa[0] - wb[1])
            if gap < 0 or gap > max_gap_s:
                continue
            sim = _cosine(face_embeddings[a], face_embeddings[b])
            if sim >= sim_threshold:
                face_only.append((a, b, sim))

    return {"agree": agree, "disagree": disagree, "face_only": face_only,
            "coverage": len(face_embeddings)}


def apply_face_veto(mapping, merge_edges, face_embeddings, windows,
                    sim_threshold=0.35, veto_margin=0.15,
                    max_edge_score_for_veto=0.80):
    """ACTIVE veto (v32, redesigned) — reverses a merge when face evidence
    confidently contradicts it, but ONLY the specific direct edge that
    caused the merge, and ONLY when that edge's own evidence was
    appearance-only.

    What changed from v31 and why: v31 checked every pairwise combination of
    members inside a transitively-merged group, including pairs that were
    NEVER directly compared to each other by merge_fragmented_tracks, and
    reversed any confident face mismatch regardless of how strong the
    original merge evidence was. That let one noisy face embedding undo a
    near-certain hand-off match (person physically can't teleport between a
    death and a birth 2px and 0.3s apart) just because some OTHER pair in
    the same transitive group had a low face score. In practice this showed
    up as visible ID flicker in the rendered video exactly where tracking
    was already correct -- the fix, not the bug.

    Two changes fix that:
      1. Only merge_edges (the literal accepted union pairs from
         merge_fragmented_tracks, each with its own evidence score) are
         checked -- never a pair that was merely co-transitive.
      2. An edge whose own score already reflects hand-off/stationary/
         anchor-tier evidence (score >= max_edge_score_for_veto, default
         0.80) is evidence independent of appearance/face entirely
         (position + timing, or a clean 1:1 crop match) -- a face signal
         alone doesn't outrank it. Those edges are FLAGGED for manual
         review instead of reversed. Only ordinary gallery/appearance-only
         edges (score < 0.80) can actually be vetoed.

    Reversing an edge is done by REPLAYING merge_edges into a fresh
    union-find in original (already-accepted, strongest-first) order,
    skipping vetoed edges -- not by patching the flat `mapping` in place.
    Patching in place (v31's approach) could leave a grandchild of a vetoed
    node still pointing at the old canonical id even after its direct
    parent was detached; replaying the actual edge list keeps the result
    consistent with what a rerun of merge_fragmented_tracks minus the
    vetoed edges would have produced.

    mapping:         {track_id: canonical_id} from merge_fragmented_tracks
                     (only used here to know every track_id that exists).
    merge_edges:     [(sim, a, b, tier), ...] from merge_fragmented_tracks --
                     the literal accepted union edges, in accepted order.
    face_embeddings: {track_id: [float, ...]} -- sparse by nature.
    windows:         {track_id: (t_first, t_last)} -- for the earliest-
                     member-becomes-canonical tie-break, same as
                     merge_fragmented_tracks.

    Returns (new_mapping, vetoed, flagged):
      vetoed  -- edges actually reversed: (a, b, face_sim, edge_score).
      flagged -- high-tier edges with a confident face mismatch that were
                 NOT reversed (independent physical/anchor evidence
                 outranks a single face signal here) -- surfaced for
                 manual review only, `mapping` is unaffected by these.
    Does not mutate the input `mapping`.
    """
    veto_bar = max(0.0, sim_threshold - veto_margin)
    parent = {t: t for t in mapping}

    def _find(t):
        while parent[t] != t:
            parent[t] = parent[parent[t]]
            t = parent[t]
        return t

    vetoed, flagged = [], []
    for edge_score, a, b, _tier in merge_edges:
        do_veto = False
        if a in face_embeddings and b in face_embeddings:
            fsim = _cosine(face_embeddings[a], face_embeddings[b])
            if fsim < veto_bar:
                if edge_score < max_edge_score_for_veto:
                    do_veto = True
                    vetoed.append((a, b, fsim, edge_score))
                else:
                    flagged.append((a, b, fsim, edge_score))
        if do_veto:
            continue   # skip this union -- a and b stay in separate groups
        ra, rb = _find(a), _find(b)
        if ra != rb:
            parent[rb] = ra

    new_mapping, canon = {}, {}
    _cpk = _canon_priority_key(windows)
    for t in sorted(mapping, key=_cpk):
        root = _find(t)
        canon.setdefault(root, t)      # staff names win; then earliest
        new_mapping[t] = canon[root]

    return new_mapping, vetoed, flagged


def cross_validate_reid(mapping, track_crops, track_windows,
                        gallery_sim_threshold=0.60, anchor_sim_threshold=0.75,
                        max_gap_s=480.0):
    """Independently re-check the OSNet-based merge mapping with HSV
    color-histogram embeddings on the SAME banked crops from this run.

    Returns a report dict:
      agree      -- merged pairs (same canonical id) where HSV also clears
                    either the gallery or anchor bar (corroborated).
      disagree   -- merged pairs where HSV does NOT corroborate — the OSNet
                    merge may still be correct (color histograms are weak
                    under lighting/color changes), but these are worth a
                    manual crop look.
      hsv_only   -- pairs OSNet did NOT merge, but HSV alone would have
                    (same non-overlap + gap-window rules as
                    merge_fragmented_tracks). Candidates OSNet may have
                    missed, not confirmed merges.

    Purely diagnostic — never changes `mapping` or downstream events.
    """
    from collections import defaultdict as _dd
    groups = _dd(list)
    for tid, canon in mapping.items():
        groups[canon].append(tid)

    agree, disagree = [], []
    already_checked = set()
    for canon, members in groups.items():
        if len(members) < 2:
            continue
        members = sorted(members, key=lambda t: track_windows[t][0])
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                a, b = members[i], members[j]
                already_checked.add(frozenset((a, b)))
                crops_a = [c for _, c in track_crops.get(a, [])]
                crops_b = [c for _, c in track_crops.get(b, [])]
                if not crops_a or not crops_b:
                    continue
                gal_sim = _hsv_gallery_sim(crops_a, crops_b)
                anc_sim = _cosine(_hsv_embed_single(crops_a[0]),
                                  _hsv_embed_single(crops_b[0]))
                corroborated = (gal_sim >= gallery_sim_threshold
                                or anc_sim >= anchor_sim_threshold)
                rec = (a, b, gal_sim, anc_sim)
                (agree if corroborated else disagree).append(rec)

    hsv_only = []
    ids_with_crops = [t for t in track_windows if track_crops.get(t)]
    for i, a in enumerate(ids_with_crops):
        for b in ids_with_crops[i + 1:]:
            if frozenset((a, b)) in already_checked:
                continue
            wa, wb = track_windows[a], track_windows[b]
            gap = (wb[0] - wa[1]) if wa[0] <= wb[0] else (wa[0] - wb[1])
            if gap < 0 or gap > max_gap_s:
                continue
            crops_a = [c for _, c in track_crops[a]]
            crops_b = [c for _, c in track_crops[b]]
            gal_sim = _hsv_gallery_sim(crops_a, crops_b)
            anc_sim = _cosine(_hsv_embed_single(crops_a[0]),
                              _hsv_embed_single(crops_b[0]))
            if gal_sim >= gallery_sim_threshold or anc_sim >= anchor_sim_threshold:
                hsv_only.append((a, b, gal_sim, anc_sim))

    return {"agree": agree, "disagree": disagree, "hsv_only": hsv_only}


def _as_gallery(vecs):
    """Accept either a single flat vector [float,...] or a gallery of vectors
    [[float,...], ...] and always return a gallery (list of vectors)."""
    if not vecs:
        return []
    if isinstance(vecs[0], (list, tuple)):
        return vecs
    return [vecs]


def _gallery_sim(gal_a, gal_b):
    """Best-match similarity between two crop galleries (2026-07-13).

    A single averaged embedding is viewpoint-fragile: if a track's banked
    crops all happen to be one pose/angle (e.g. a person turned into the
    espresso machine the whole fragment), the mean vector doesn't resemble
    a different fragment of the SAME person seen frontally, even though one
    good crop-to-crop pair would clearly match. Comparing every stored crop
    in A against every stored crop in B and taking the best pairwise cosine
    means only ONE matching viewpoint needs to line up, not all of them —
    much more robust to a person moving/turning mid-track.
    """
    gal_a, gal_b = _as_gallery(gal_a), _as_gallery(gal_b)
    if not gal_a or not gal_b:
        return 0.0
    return max(_cosine(va, vb) for va in gal_a for vb in gal_b)


def _windows_overlap(wins_a, wins_b, tolerance_s=2):
    for a0, a1 in wins_a:
        for b0, b1 in wins_b:
            if a0 < b1 - tolerance_s and b0 < a1 - tolerance_s:
                return True
    return False

# ===========================================================================
# v46 GLOBAL TRACKLET ASSOCIATION  (unit-tested in scratchpad; 11/11)
# Treats identity assignment as one global optimisation over the whole video:
# link tracklet tails->heads with the Hungarian algorithm so competing long-
# range links are resolved jointly, not greedily. Pure Python; scipy if present
# else a built-in Hungarian. Enforces the same hard constraints as the live
# path (co-visibility, gap, spatial plausibility, time-decayed appearance bar).
# ===========================================================================
try:
    from scipy.optimize import linear_sum_assignment as _gta_lsa
    _GTA_SCIPY = True
except Exception:
    _GTA_SCIPY = False


def _gta_hungarian(cost):
    import copy
    n = len(cost)
    if n == 0:
        return [], []
    m = copy.deepcopy(cost)
    INF = float("inf")
    u = [0.0]*(n+1); v = [0.0]*(n+1); p = [0]*(n+1); way = [0]*(n+1)
    for i in range(1, n+1):
        p[0] = i; j0 = 0
        minv = [INF]*(n+1); used = [False]*(n+1)
        while True:
            used[j0] = True; i0 = p[j0]; delta = INF; j1 = -1
            for j in range(1, n+1):
                if not used[j]:
                    cur = m[i0-1][j-1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(n+1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0:
                break
    rows = [0]*n
    for j in range(1, n+1):
        if p[j] != 0:
            rows[p[j]-1] = j-1
    return list(range(n)), rows


def _gta_solve(cost):
    if _GTA_SCIPY:
        r, c = _gta_lsa(cost); return list(r), list(c)
    return _gta_hungarian([list(row) for row in cost])


class _GtaUF:
    def __init__(self, items): self.p = {x: x for x in items}
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra


def _gta_sim_floor(gap, base, near_s, far_s, near_bonus, far_penalty):
    if gap <= near_s: return base - near_bonus
    if gap >= far_s:  return base + far_penalty
    frac = (gap - near_s) / (far_s - near_s)
    return (base - near_bonus) + frac * (near_bonus + far_penalty)


def link_tracklets_global(tracklets, sim_fn, accept_sim=0.60, max_gap_s=7200.0,
                          overlap_tol_s=0.5, max_speed_px=220.0, speed_slack_px=60.0,
                          near_s=15.0, far_s=180.0, near_bonus=0.04, far_penalty=0.05,
                          no_link_cost=1.0):
    import math as _m
    T = list(tracklets); n = len(T)
    if n <= 1:
        return {t["id"]: t["id"] for t in T}

    def eligible_cost(a, b):
        A, B = T[a], T[b]
        gap = B["t_start"] - A["t_end"]
        if gap < -overlap_tol_s: return None
        gap = max(0.0, gap)
        if gap > max_gap_s: return None
        if A["emb"] is None or B["emb"] is None: return None
        pe, psb = A.get("pos_end"), B.get("pos_start")
        if pe is not None and psb is not None:
            dist = _m.hypot(psb[0]-pe[0], psb[1]-pe[1])
            if dist > max_speed_px*gap + speed_slack_px: return None
        floor = _gta_sim_floor(gap, accept_sim, near_s, far_s, near_bonus, far_penalty)
        sim = sim_fn(A["emb"], B["emb"])
        if sim < floor: return None
        return (1.0 - sim) + 0.001*gap

    BIG = 1e6; size = 2*n
    cost = [[BIG]*size for _ in range(size)]
    for a in range(n):
        for b in range(n):
            if a == b: continue
            c = eligible_cost(a, b)
            if c is not None: cost[a][b] = c
    for i in range(n):
        cost[i][n+i] = no_link_cost
        cost[n+i][i] = no_link_cost
    for i in range(n):
        for j in range(n):
            cost[n+i][n+j] = 0.0
    rows, cols = _gta_solve(cost)
    uf = _GtaUF([t["id"] for t in T])
    for r, c in zip(rows, cols):
        if r < n and c < n and cost[r][c] < BIG:
            uf.union(T[r]["id"], T[c]["id"])
    groups = {}
    for t in T:
        groups.setdefault(uf.find(t["id"]), []).append(t)
    mapping = {}
    for root, members in groups.items():
        canon = min(members, key=lambda t: (t["t_start"], str(t["id"])))["id"]
        for t in members: mapping[t["id"]] = canon
    return mapping


def _apply_global_tracklet_pass(mapping, windows, positions, anchor_embeddings, base_sim):
    """Additive: run the global linker over the GREEDY groups. Builds one
    super-tracklet per current identity (union window, earliest/latest
    positions, representative anchor) and links those. Returns a composed
    fragment->identity mapping. Never splits an existing group."""
    groups = {}
    for frag, canon in mapping.items():
        groups.setdefault(canon, []).append(frag)
    tracklets = []
    for canon, frags in groups.items():
        fw = [f for f in frags if f in windows]
        if not fw:
            continue
        fw.sort(key=lambda f: windows[f][0])
        first, last = fw[0], fw[-1]
        ps = pe = None
        if positions:
            if first in positions and positions[first]:
                ps = positions[first][0]
            if last in positions and positions[last]:
                pe = positions[last][1]
        emb = None
        for f in [canon] + fw:
            if anchor_embeddings and anchor_embeddings.get(f) is not None:
                emb = anchor_embeddings[f]; break
        tracklets.append(dict(id=canon, t_start=windows[first][0],
                              t_end=windows[last][1], pos_start=ps,
                              pos_end=pe, emb=emb))
    if len(tracklets) <= 1:
        return mapping
    group_map = link_tracklets_global(
        tracklets, _cosine, accept_sim=base_sim, max_gap_s=REID_MAX_GAP_S,
        max_speed_px=MAX_PLAUSIBLE_SPEED_PX, near_s=NEAR_GAP_S, far_s=FAR_GAP_S,
        near_bonus=NEAR_GAP_BONUS, far_penalty=FAR_GAP_PENALTY)
    return {frag: group_map.get(old, old) for frag, old in mapping.items()}


def merge_fragmented_tracks(track_windows, embeddings,
                            sim_threshold=0.65, max_gap_s=480.0,
                            positions=None, handoff_gap_s=2.5,
                            handoff_px=90.0, role_hint=None,
                            stationary_px=30.0, anchor_embeddings=None,
                            anchor_sim_threshold=0.75,
                            raw_crops=None, hsv_sim_threshold=0.75,
                            face_embeddings=None, face_sim_threshold=0.35,
                            plane=None, handoff_m=1.6, stationary_m=0.6,
                            ir_hint=None):
    """Merge track fragments that are the same person reappearing.

    track_windows: {track_id: (t_first, t_last)} visibility window per track
    embeddings:    {track_id: [[float, ...], ...]} a GALLERY of appearance
                   vectors per track (one per banked crop). Matching uses the
                   best pairwise cosine across both galleries (_gallery_sim),
                   not a single averaged vector, so a track survives even if
                   its crops span multiple viewpoints/poses. A flat single
                   vector [float, ...] per track_id is still accepted for
                   backwards compatibility (auto-wrapped as a 1-item gallery).
    positions:     optional {track_id: (start_xy, end_xy)} — enables the
                   HAND-OFF rule: when one fragment dies and another is born
                   within handoff_gap_s seconds AND handoff_px pixels, they
                   merge even without appearance evidence (covers small/dim
                   fragments that never yielded a usable crop).
    role_hint:     optional {track_id: "staff"|"customer"} — a fragment that
                   has EARNED a confident role from real zone dwell (not just
                   a guess) can never merge with a fragment that earned the
                   opposite role. Appearance similarity alone cannot override
                   this: a staff member and a seated customer are never the
                   same person no matter how similar their embeddings look.
    raw_crops:     (v37) optional {track_id: [crop_bgr, ...]} — the SAME
                   banked crop images used for the appearance embeddings.
                   Enables an ATTIRE tier using HSV color-histogram gallery
                   similarity (_hsv_gallery_sim) as a genuine merge signal,
                   not just a post-hoc corroboration check. On footage where
                   the body-appearance backbone doesn't cleanly separate
                   same/different person (see calibrate_appearance_threshold),
                   HSV cross-validation has repeatedly found large numbers of
                   genuine same-person pairs the appearance backbone alone
                   never merged — this promotes that same evidence to where
                   it can actually fix the fragmentation, instead of only
                   flagging it for manual review afterward.
    face_embeddings: (v37) optional {track_id: [float, ...]} — ArcFace-style
                   embeddings, same ones used for face corroboration/veto
                   elsewhere. A clear face match is the strongest available
                   identity evidence (survives pose AND clothing change,
                   unlike body appearance or attire color), so it's ranked
                   with/above hand-off, and can close gaps neither the
                   appearance backbone nor HSV can (e.g. someone who stood up
                   and changed which way they're facing, or changed jackets).
    A pair merges when it passes any rule, their windows do NOT overlap
    (one body can't be in frame twice), and the gap <= max_gap_s. Greedy
    best-evidence-first with transitive groups.
    Returns (mapping, merge_edges):
      mapping     -- {track_id: canonical_id} (canonical = earliest-
                     seen member), same as before.
      merge_edges -- (v32) [(sim, a, b), ...] the actual direct pairs
                     whose evidence caused a union, in accepted order.
                     sim encodes the evidence TIER: >=0.92 face match,
                     >=0.90 hand-off, 0.85-0.94 stationary, 0.80-0.89
                     anchor, 0.76-0.82 attire/HSV (v37), below that an
                     ordinary gallery/appearance match. Lets a downstream
                     check (apply_face_veto) tell "person physically can't
                     teleport" evidence apart from a weaker appearance-only
                     match.
    """
    def _role_conflict(a, b):
        if not role_hint:
            return False
        ra, rb = role_hint.get(a), role_hint.get(b)
        return bool(ra) and bool(rb) and ra != rb

    def _ir_mismatch(a, b):
        # A4: colour evidence compared across a colour<->IR boundary is noise
        # in BOTH directions — a pre-dusk anchor vs a post-dusk greyscale+CLAHE
        # vector matches (or misses) by accident. ir_hint is {track_id: frac of
        # frames under IR}. Only pairs on OPPOSITE sides of the boundary are
        # blocked; face + hand-off/stationary tiers still bridge it (faces
        # survive near-IR, physics doesn't care about colour).
        if not ir_hint:
            return False
        fa, fb = ir_hint.get(a, 0.0), ir_hint.get(b, 0.0)
        _thr = float(globals().get("IR_TRACK_FRAC", 0.5))
        # F10b: same boundary as the attire-evidence exclusion, so a track
        # withheld from colour evidence is also 'IR' to this guard
        return (fa >= _thr) != (fb >= _thr)

    # (v38) diagnostics — WHY a candidate pair did or didn't end up merged.
    # Every entry in `pairs` now carries a tier label so we can report a
    # per-tier accepted-merge breakdown, and every rejection (role conflict
    # or window overlap) is counted + sampled so a slam-dunk match that got
    # silently dropped (e.g. by greedy processing order) is visible instead
    # of just missing from the output.
    role_conflict_count = 0
    pairs = []
    # (v37) evaluate every track that has ANY identity signal at all — not
    # just the ones with an appearance embedding — so a track that only
    # yielded a face or attire signal (but no usable appearance embedding)
    # still gets a fair chance to match instead of being silently excluded
    # from every tier except hand-off/stationary.
    # v54 SCALE FIX: sort by WINDOW START (was: track id). With starts
    # ascending, wa[0] <= wb[0] always, so gap = wb[0]-wa[1] rises
    # monotonically as b advances — the moment it exceeds max_gap_s every
    # later b is further still, so `break` is exactly equivalent to
    # `continue` and costs O(n*k) instead of O(n^2). On a 10-hour run with
    # ~8k tracks this is the difference between minutes and hours.
    ids_signal = sorted((t for t in track_windows
                         if t in embeddings
                         or (raw_crops and t in raw_crops)
                         or (face_embeddings and t in face_embeddings)),
                        key=lambda t: track_windows[t][0])
    for i, a in enumerate(ids_signal):
        for b in ids_signal[i + 1:]:
            wa, wb = track_windows[a], track_windows[b]
            gap = wb[0] - wa[1]
            if gap > max_gap_s:
                break            # every later b is even further away
            if gap < 0:
                continue         # windows overlap — not a reappearance
            if _role_conflict(a, b):
                role_conflict_count += 1
                continue
            if a in embeddings and b in embeddings:
                sim = _gallery_sim(embeddings[a], embeddings[b])
                gap = (wb[0] - wa[1]) if wa[0] <= wb[0] else (wa[0] - wb[1])
                required = _dynamic_accept_thresh(gap, sim_threshold)
                
                # Soft spatial penalty
                spatial_penalty = 0.0
                pa, pb = positions.get(a), positions.get(b)
                if pa and pb and gap > 0:
                    pos_last = pa[1] if wa[0] <= wb[0] else pb[1]
                    pos_first = pb[0] if wa[0] <= wb[0] else pa[0]
                    dist = math.hypot(pos_first[0] - pos_last[0], pos_first[1] - pos_last[1])
                    plausible_dist = MAX_PLAUSIBLE_SPEED_PX * gap
                    if plausible_dist > 0:
                        excess_ratio = max(0.0, dist - plausible_dist) / plausible_dist
                        spatial_penalty = min(MAX_SPATIAL_PENALTY, excess_ratio * SPATIAL_PENALTY_SCALE)
                required += spatial_penalty
                
                if (sim >= required and not _ir_mismatch(a, b)
                        and not _appearance_hsv_contradicts(a, b, raw_crops)):
                    pairs.append((sim, a, b, "gallery"))
            # ANCHOR TIER (v25): a clean 1:1 match between each track's single
            # best-quality crop (see ANCHOR_SIM_THRESHOLD) is stronger, cleaner
            # evidence than the best-of-gallery score above — that score can be
            # inflated by one lucky crop pairing even when the tracks are
            # different people. When two anchors clear the stricter anchor bar,
            # rank the match ABOVE ordinary gallery matches in the greedy
            # sorted-descending loop below, so it can't get starved out by a
            # pile of weaker matches that happen to sort higher (same failure
            # mode the hand-off/stationary fix below already addresses).
            if (anchor_embeddings and a in anchor_embeddings
                    and b in anchor_embeddings):
                asim = _cosine(anchor_embeddings[a], anchor_embeddings[b])
                if (asim >= anchor_sim_threshold
                        and not _ir_mismatch(a, b)
                        and not _appearance_hsv_contradicts(a, b, raw_crops)):
                    conf = min(1.0, (asim - anchor_sim_threshold)
                               / max(1e-6, 1.0 - anchor_sim_threshold))
                    pseudo = 0.80 + 0.09 * conf   # 0.80-0.89 — below hand-off/
                                                   # stationary (those are near-
                                                   # certain physical evidence),
                                                   # above generic gallery sims
                    pairs.append((pseudo, a, b, "anchor"))
            # ATTIRE TIER (v37): HSV color-histogram gallery match, computed
            # on the same banked crops. Ranked below anchor (clothing color
            # alone is fooled by identical uniforms) but treated as a real
            # merge trigger, not just a diagnostic — see raw_crops docstring
            # above for why.
            if raw_crops and a in raw_crops and b in raw_crops:
                hsim = _hsv_gallery_sim(raw_crops[a], raw_crops[b])
                if hsim >= hsv_sim_threshold and not _ir_mismatch(a, b):
                    conf = min(1.0, (hsim - hsv_sim_threshold)
                               / max(1e-6, 1.0 - hsv_sim_threshold))
                    pseudo = 0.76 + 0.06 * conf   # 0.76-0.82
                    pairs.append((pseudo, a, b, "attire"))
            # FACE TIER (v37): a confident face match outranks every other
            # signal here — see face_embeddings docstring above.
            if (face_embeddings and a in face_embeddings
                    and b in face_embeddings):
                fsim = _cosine(face_embeddings[a], face_embeddings[b])
                if fsim >= face_sim_threshold:
                    conf = min(1.0, (fsim - face_sim_threshold)
                               / max(1e-6, 1.0 - face_sim_threshold))
                    pseudo = 0.92 + 0.07 * conf   # 0.92-0.99
                    pairs.append((pseudo, a, b, "face"))

    if positions:
        all_ids = sorted(track_windows, key=lambda t: track_windows[t][0])
        for i, a in enumerate(all_ids):
            for b in all_ids[i + 1:]:
                wa, wb = track_windows[a], track_windows[b]
                if wb[0] < wa[0]:
                    a2, b2 = b, a
                else:
                    a2, b2 = a, b
                if _role_conflict(a2, b2):
                    role_conflict_count += 1
                    continue
                gap = track_windows[b2][0] - track_windows[a2][1]
                if gap > max_gap_s:
                    break        # v54: all_ids is start-sorted, gap only grows
                if gap < 0:
                    continue
                pa, pb = positions.get(a2), positions.get(b2)
                if not pa or not pb:
                    continue
                dx = pa[1][0] - pb[0][0]
                dy = pa[1][1] - pb[0][1]
                dist = math.hypot(dx, dy)
                # G2: "died and reappeared within 1.6 m" is the same physical
                # claim everywhere in the frame; "within 160 px" is not.
                _dm = plane.dist_m(pa[1], pb[0]) if (plane is not None and plane.ok) else None
                _near_handoff = (_dm <= handoff_m) if _dm is not None else (dist <= handoff_px)
                _near_static = (_dm <= stationary_m) if _dm is not None else (dist <= stationary_px)
                if (gap <= handoff_gap_s and _near_handoff
                        and not _handoff_appearance_contradicts(a2, b2, anchor_embeddings)
                        and not _handoff_hsv_contradicts(a2, b2, raw_crops)):
                    # A near-instant, near-zero-distance hand-off is *stronger*
                    # evidence than a typical appearance match (osnet_x0_25 on a
                    # small/dim crop is easy to fool; a person can't teleport).
                    # BUG FIX (was): pseudo was capped at `sim_threshold - 0.001`,
                    # which forced every spatial/hand-off match to rank BELOW
                    # every appearance match in the sorted-descending greedy loop.
                    # That let weaker (sometimes wrong) appearance merges claim
                    # union-find roots first, so by the time a rock-solid 0px
                    # hand-off pair was processed, its root's group window
                    # already overlapped something else and the merge was
                    # silently rejected. Scoring it ABOVE ordinary appearance
                    # matches means it gets first claim, as it should.
                    conf = 1.0 - dist / handoff_px
                    pseudo = 0.90 + 0.09 * conf   # 0.90-0.99
                    pairs.append((pseudo, a2, b2, "hand-off"))
                elif (_near_static
                        and not _handoff_appearance_contradicts(a2, b2, anchor_embeddings)
                        and not _handoff_hsv_contradicts(a2, b2, raw_crops)):
                    # near-zero displacement across a long gap = same seat,
                    # same person, even with a weak/inconclusive embedding.
                    # Same fix as above: rank above typical appearance sims
                    # (but slightly below a true hand-off, since the gap here
                    # can be much longer and the identity claim slightly less
                    # certain) instead of being capped below sim_threshold.
                    conf = 1.0 - dist / stationary_px
                    pseudo = 0.85 + 0.09 * conf   # 0.85-0.94
                    pairs.append((pseudo, a2, b2, "stationary"))

    parent = {t: t for t in track_windows}

    def find(t):
        while parent[t] != t:
            parent[t] = parent[parent[t]]
            t = parent[t]
        return t

    group_windows = {t: [track_windows[t]] for t in track_windows}
    merge_edges = []   # (v32) the ACTUAL accepted direct union edges, in the
                       # order they fired -- (sim, a, b, tier) -- as scored
                       # above. Kept separate from the final transitive
                       # `mapping` because a downstream veto (apply_face_veto)
                       # needs to know WHICH specific pair caused a merge and
                       # HOW STRONG that evidence was (hand-off/stationary
                       # near-certain physical evidence vs. an ordinary
                       # appearance-only gallery match) -- checking every
                       # pairwise combination inside a transitively-merged
                       # group conflates edges that were never directly
                       # compared with edges that actually drove the merge.
    tier_counts = defaultdict(int)          # accepted merges, per tier
    overlap_blocked = []   # (v38) high-confidence pairs REJECTED only because
                           # their groups' time windows already overlapped —
                           # this is the specific failure mode where a strong
                           # match (e.g. HSV=0.999) gets starved out by
                           # processing order rather than by weak evidence.
    for sim, a, b, tier in sorted(pairs, reverse=True):
        ra, rb = find(a), find(b)
        if ra == rb:
            continue
        if _windows_overlap(group_windows[ra], group_windows[rb]):
            overlap_blocked.append((sim, a, b, tier))
            continue   # same-time fragments cannot be one person
        parent[rb] = ra
        group_windows[ra] = group_windows[ra] + group_windows[rb]
        merge_edges.append((sim, a, b, tier))
        tier_counts[tier] += 1

    mapping = {}
    canon = {}
    _cpk = _canon_priority_key(track_windows)
    for t in sorted(track_windows, key=_cpk):
        root = find(t)
        canon.setdefault(root, t)      # staff names win; then earliest
        mapping[t] = canon[root]

    diagnostics = {
        "tier_counts": dict(tier_counts),
        "role_conflicts_blocked": role_conflict_count,
        "overlap_blocked_count": len(overlap_blocked),
        # sorted highest-confidence-first: these are the "should have merged,
        # got starved by order" candidates worth a manual look.
        "overlap_blocked_samples": sorted(overlap_blocked, reverse=True)[:30],
    }
    return mapping, merge_edges, diagnostics


# ---------------------------------------------------------------------------
# Persistent identity memory (v38)
# ---------------------------------------------------------------------------
# The merge step above solves fragmentation WITHIN one processing run. These
# two functions are what let a person's ID stay FIXED going forward: build a
# durable "dossier" (memory snapshot) per confirmed person out of every
# evidence type collected on them, then cross-verify any new track fragment
# against that memory using the exact same evidence priority as the merge
# step (face > hand-off/stationary > anchor > attire > plain-gallery) before
# ever assigning it a brand-new id. A fragment only ever gets folded into an
# existing person when it clears one of these tiers' bars; anything weaker
# becomes its own new, equally trackable identity rather than a guess.

def build_identity_dossiers(mapping, merge_edges, track_windows,
                            embeddings=None, anchor_embeddings=None,
                            face_embeddings=None, raw_crops=None,
                            positions=None):
    """Build one persistent memory record per canonical person.

    Pools every fragment that was folded into that person plus every
    evidence snapshot collected across those fragments (appearance gallery,
    anchor crop embedding, face embedding, attire/HSV crops, last-known
    position) and records WHICH evidence tier(s) actually proved the
    fragments belong together. This dossier is the "memory" a later call to
    match_track_to_dossiers cross-verifies new sightings against.

    Returns {person_id: dossier_dict}. See match_track_to_dossiers for how
    the dossier is consumed.
    """
    fragments_by_person = defaultdict(list)
    for tid, cid in mapping.items():
        fragments_by_person[cid].append(tid)

    tiers_by_person = defaultdict(set)
    for sim, a, b, tier in merge_edges:
        tiers_by_person[mapping.get(a, a)].add(tier)

    dossiers = {}
    for cid, frags in fragments_by_person.items():
        frags = sorted(frags, key=lambda t: track_windows[t][0])
        gallery, crops, positions_track = [], [], []
        anchor_vec, face_vec = None, None
        for t in frags:
            if embeddings:
                gallery.extend(_as_gallery(embeddings.get(t, [])))
            if raw_crops:
                crops.extend(raw_crops.get(t, []))
            if anchor_embeddings and anchor_vec is None and t in anchor_embeddings:
                anchor_vec = anchor_embeddings[t]
            if face_embeddings and face_vec is None and t in face_embeddings:
                face_vec = face_embeddings[t]
            if positions and t in positions:
                positions_track.append((t, positions[t]))
        dossiers[cid] = {
            "person_id": cid,
            "fragment_ids": frags,
            "t_first": min(track_windows[t][0] for t in frags),
            "t_last": max(track_windows[t][1] for t in frags),
            "evidence_tiers_used": sorted(tiers_by_person.get(cid, [])),
            "appearance_gallery": gallery,      # plain-gallery snapshot
            "anchor_embedding": anchor_vec,      # anchor snapshot
            "face_embedding": face_vec,          # face snapshot
            "attire_crops": crops,               # attire/HSV snapshot
            "positions_by_fragment": positions_track,  # hand-off/stationary
        }
    return dossiers


def match_track_to_dossiers(track_id, track_window, dossiers,
                            embeddings=None, anchor_embeddings=None,
                            face_embeddings=None, raw_crops=None,
                            position=None, sim_threshold=0.65,
                            anchor_sim_threshold=0.75, hsv_sim_threshold=0.75,
                            face_sim_threshold=0.35, handoff_gap_s=2.5,
                            handoff_px=90.0, stationary_px=30.0,
                            max_gap_s=480.0):
    """Cross-verify one new track fragment against persistent identity
    memory before deciding whether it's a KNOWN person (reuse their id) or
    a genuinely NEW one (mint a new id).

    Checks the same evidence tiers merge_fragmented_tracks uses, in the same
    priority (face > hand-off/stationary > anchor > attire > gallery), takes
    the single strongest tier that clears its own threshold across every
    candidate dossier, and returns that dossier's person_id. If nothing
    clears any bar, returns (None, None, None) -- the caller should assign a
    brand-new id rather than force a match on weak/absent evidence, so a
    genuinely new person never inherits a stale identity.

    Returns (person_id_or_None, evidence_tier_or_None, score_or_None).
    """
    candidates = []  # (pseudo_score, tier, person_id)
    for cid, d in dossiers.items():
        gap = (track_window[0] - d["t_last"] if track_window[0] >= d["t_last"]
               else d["t_first"] - track_window[1])
        if gap < 0 or gap > max_gap_s:
            continue  # co-visible (can't be same body) or too far apart

        if (face_embeddings and track_id in face_embeddings
                and d["face_embedding"] is not None):
            fsim = _cosine(face_embeddings[track_id], d["face_embedding"])
            if fsim >= face_sim_threshold:
                conf = min(1.0, (fsim - face_sim_threshold)
                           / max(1e-6, 1.0 - face_sim_threshold))
                candidates.append((0.92 + 0.07 * conf, "face", cid))

        if position and d["positions_by_fragment"]:
            _, last_pos = d["positions_by_fragment"][-1]
            dist = math.hypot(last_pos[1][0] - position[0][0],
                              last_pos[1][1] - position[0][1])
            if gap <= handoff_gap_s and dist <= handoff_px:
                conf = 1.0 - dist / handoff_px
                candidates.append((0.90 + 0.09 * conf, "hand-off", cid))
            elif dist <= stationary_px:
                conf = 1.0 - dist / stationary_px
                candidates.append((0.85 + 0.09 * conf, "stationary", cid))

        if (anchor_embeddings and track_id in anchor_embeddings
                and d["anchor_embedding"] is not None):
            asim = _cosine(anchor_embeddings[track_id], d["anchor_embedding"])
            if asim >= anchor_sim_threshold:
                conf = min(1.0, (asim - anchor_sim_threshold)
                           / max(1e-6, 1.0 - anchor_sim_threshold))
                candidates.append((0.80 + 0.09 * conf, "anchor", cid))

        if raw_crops and track_id in raw_crops and d["attire_crops"]:
            hsim = _hsv_gallery_sim(raw_crops[track_id], d["attire_crops"])
            if hsim >= hsv_sim_threshold:
                conf = min(1.0, (hsim - hsv_sim_threshold)
                           / max(1e-6, 1.0 - hsv_sim_threshold))
                candidates.append((0.76 + 0.06 * conf, "attire", cid))

        if embeddings and track_id in embeddings and d["appearance_gallery"]:
            sim = _gallery_sim(embeddings[track_id], d["appearance_gallery"])
            if sim >= sim_threshold:
                candidates.append((sim, "gallery", cid))

    if not candidates:
        return None, None, None
    score, tier, cid = max(candidates)
    return cid, tier, score


def _rebuild_plane(run):
    """Rebuild the ground plane from a run dict.

    The GroundPlane object cannot survive the per-chunk JSON cache, but its two
    coefficients can, so it is reconstructed wherever it is needed downstream
    rather than silently falling back to pixels.

    Lives HERE, in the analytics cell, and not next to its first caller: the
    answers cell CALLS answer_for_run() at the bottom, so a helper defined in a
    later cell is a NameError 40 minutes into a paid GPU session. Caught by
    check_notebook_runtime.py, which is why that script exists.
    """
    c = run.get("perspective_coeffs")
    fs = run.get("frame_size_analysed")
    if not c or not fs:
        return GroundPlane.none("no perspective coefficients in this run")
    # P2: person_h is a DEFAULT ARGUMENT bound at def time to the module
    # constant, so setting a notebook global named PERSON_H_M does nothing. It
    # has to be passed. Without this a venue configuring person_height_m: 1.62
    # was silently measured with a 1.70 m ruler and every distance was ~5% out.
    _ph = ((globals().get("VENUE_PROFILE") or {}).get("camera", {})
           .get("person_height_m")) or PERSON_H_M
    _hf = ((globals().get("VENUE_PROFILE") or {}).get("camera", {})
           .get("hfov_deg")) or DEFAULT_HFOV_DEG
    return GroundPlane.from_perspective(c[0], c[1], fs[0], fs[1],
                                        person_h=_ph, hfov_deg=_hf)


def tier_a_crossings(crossings, plane=None, dedupe_s=6.0, dedupe_m=1.2,
                     dedupe_px=140.0, direction="in", roles=None):
    """TIER A: how many people came through the door, WITHOUT depending on
    identity holding together for ten hours.

    entered_count() counts unique track_ids after the night-long Re-ID stitch,
    so it inherits every Re-ID error: a person whose id fragments at the door is
    counted twice, and two people wrongly merged are counted once. But crossing
    a line is a three-second event. It does not need a ten-hour identity, only a
    local one.

    So: count crossing EVENTS, then collapse events that are close in BOTH time
    and space — because that is what one person crossing once looks like, no
    matter how many track ids they wore while doing it.

    Needs each crossing to carry a position; crossings without one fall back to
    unique-id behaviour for that event, which is the old answer, never worse.
    """
    evs = [c for c in crossings if c.get("direction") == direction
           and not (roles and roles.get(c.get("track_id")) == "staff")]
    evs.sort(key=lambda c: c["t"])
    kept = []
    for c in evs:
        p = c.get("pos")
        dup = False
        for k in reversed(kept):
            if c["t"] - k["t"] > dedupe_s:
                break
            if p is None or k.get("pos") is None:
                if k.get("track_id") == c.get("track_id"):
                    dup = True
                    break
                continue
            d = plane.dist_m(p, k["pos"]) if (plane is not None and plane.ok) else None
            if (d is not None and d <= dedupe_m) or (
                    d is None and math.hypot(p[0] - k["pos"][0],
                                             p[1] - k["pos"][1]) <= dedupe_px):
                dup = True
                break
        if not dup:
            kept.append(c)
    return len(kept), kept


def detect_groups(arrivals, plane=None, window_s=25.0, radius_m=3.0,
                  radius_px=320.0):
    """G4: people who arrive together are one PARTY. A restaurant books parties,
    not individuals, so 12 arrivals in 4 groups is a different night from 12
    walk-ins. Chaining is deliberate — a family of five spreads wider than the
    radius but every member is close to the one before.

    Only sane in metres: a 3 m radius is ~470 px at the door and ~96 px at the
    back of the room, so in pixels this rule would silently mean different
    things in different parts of the same frame.
    """
    evs = sorted(arrivals, key=lambda c: c["t"])
    groups = []
    for c in evs:
        placed = False
        for g in groups:
            if c["t"] - g["t_last"] > window_s:
                continue
            for m in g["members"]:
                p, q = c.get("pos"), m.get("pos")
                if p is None or q is None:
                    continue
                d = plane.dist_m(p, q) if (plane is not None and plane.ok) else None
                near = (d <= radius_m) if d is not None else (
                    math.hypot(p[0] - q[0], p[1] - q[1]) <= radius_px)
                if near:
                    g["members"].append(c)
                    g["t_last"] = c["t"]
                    placed = True
                    break
            if placed:
                break
        if not placed:
            groups.append({"t_first": c["t"], "t_last": c["t"], "members": [c]})
    for g in groups:
        g["size"] = len(g["members"])
    return groups


def remap_events(events, crossings, mapping):
    """Apply a track merge mapping to events + crossings; role per canonical
    person = role of its longest-dwelling fragment."""
    dwell = defaultdict(lambda: defaultdict(float))
    for e in events:
        cid = mapping.get(e["track_id"], e["track_id"])
        dwell[cid][e["role"]] += e["duration"]
    role_of = {cid: max(roles.items(), key=lambda kv: kv[1])[0]
               for cid, roles in dwell.items()}
    new_events = [dict(e, track_id=mapping.get(e["track_id"], e["track_id"]),
                       role=role_of.get(mapping.get(e["track_id"],
                                                    e["track_id"]), e["role"]))
                  for e in events]
    new_crossings = [dict(c, track_id=mapping.get(c["track_id"], c["track_id"]))
                     for c in crossings]
    return new_events, new_crossings, role_of


# ---------------------------------------------------------------------------
# Minute-by-minute table (mentor's output contract)
# ---------------------------------------------------------------------------

def minute_summaries(events, crossings, camera_id, t_end, window_s=60.0,
                     roles=None):
    """One row per window: who was present, what happened. Deterministic
    (non-LLM) summaries grounded in events; matches the mentor's JSON keys.
    When roles is given, entered/exited counts use the SAME definition as the
    headline answer: unique non-staff people, not raw crossings."""
    def _count(dir_, w_start, w_end):
        hits = [c for c in crossings
                if c["direction"] == dir_ and w_start <= c["t"] < w_end]
        if roles is None:
            return len(hits)
        return len({c["track_id"] for c in hits
                    if roles.get(c["track_id"]) != "staff"})

    rows = []
    n_windows = int(t_end // window_s) + (1 if t_end % window_s > 1e-9 else 0)
    for w in range(n_windows):
        w_start, w_end = w * window_s, min((w + 1) * window_s, t_end)
        present = sorted({e["track_id"] for e in events
                          if e["t_out"] > w_start and e["t_in"] < w_end}, key=track_sort_key)
        ins = [None] * _count("in", w_start, w_end)
        outs = [None] * _count("out", w_start, w_end)
        zones_active = sorted({e["zone"] for e in events
                               if e["t_out"] > w_start and e["t_in"] < w_end})
        bits = []
        if ins:
            bits.append(f"{len(ins)} entered")
        if outs:
            bits.append(f"{len(outs)} exited")
        if zones_active:
            bits.append("activity in: " + ", ".join(zones_active))
        summary = "; ".join(bits) if bits else "no activity"
        mm = lambda s: f"{int(s // 60):02d}:{int(s % 60):02d}"
        rows.append({
            "time_window": f"{mm(w_start)}-{mm(w_end)}",
            "camera_id": camera_id,
            "people_count": len(present),
            "person_ids": [f"P{tid}" if (isinstance(tid, int) or str(tid).isdigit()) else str(tid) for tid in present],
            "action": summary,
            "possible_next_camera": None,   # single-camera run — the cross-
            "confidence": None,             # camera phase fills both fields
            "summary": f"{len(present)} people visible. {summary}.",
        })
    return rows


In [ ]:
# Cell 6 — CHART STYLE + VISUAL HELPERS (palette CVD-validated)
import math
from collections import defaultdict
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASE_LINE = "#e1e0d9", "#c3c2b7"
ROLE_HEXES = {"customer": "#2a78d6", "staff": "#eb6834", "unknown": "#898781"}
ZONE_HEXES = ["#2a78d6", "#1baf7a", "#eda100", "#4a3aa7", "#008300",
              "#e87ba4", "#eb6834", "#e34948"]
SEQ_HUE, STATUS_GOOD, STATUS_CRIT = "#256abf", "#0ca30c", "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASE_LINE, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "legend.frameon": False,
})

def mmss(s):
    s = max(0, int(s))
    return f"{s // 60:02d}:{s % 60:02d}"

from datetime import datetime, timedelta

def wall(t):
    """Video-time -> wall-clock string using the burned-in start time."""
    if not VIDEO_START_CLOCK:
        return ""
    base = datetime.strptime(VIDEO_START_CLOCK, "%H:%M:%S")
    return (base + timedelta(seconds=float(t))).strftime("%H:%M:%S")

TIME_FMT = FuncFormatter(lambda x, _: mmss(x))

def zone_color_map(zone_names):
    """Fixed sorted order -> fixed colors, shared by charts AND video overlay."""
    return {z: ZONE_HEXES[i % len(ZONE_HEXES)] for i, z in enumerate(sorted(zone_names))}

def show_gallery(snaps, title, ncols=3, max_items=9):
    items = snaps[:max_items]
    if not items:
        print(f"({title}: no frames captured)")
        return
    rows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(rows, ncols, figsize=(5.2 * ncols, 3.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (t, img) in zip(axes, items):
        ax.imshow(img)
        ax.set_title(f"t = {mmss(t)}", fontsize=10, color=INK2)
    fig.suptitle(title, fontsize=13, color=INK)
    plt.tight_layout()
    plt.show()

def plot_journey_gantt(events, title, zcolors, max_people=25):
    """Mentor Output 3, as a picture: one row per person, colored by zone."""
    per, roles = defaultdict(list), {}
    for e in events:
        per[e["track_id"]].append(e)
        roles[e["track_id"]] = e["role"]
    if not per:
        print("(no events for journey chart)")
        return
    order = sorted(per, key=lambda tid: min(x["t_in"] for x in per[tid]))[:max_people]
    fig, ax = plt.subplots(figsize=(13, max(2.5, 0.4 * len(order) + 1.3)))
    for y, tid in enumerate(order):
        for e in per[tid]:
            ax.broken_barh([(e["t_in"], e["duration"])], (y - 0.32, 0.64),
                           facecolors=zcolors.get(e["zone"], MUTED))
    ax.set_yticks(range(len(order)))
    ytick_labels = [f"P{tid} \u00b7 {roles[tid]}" if (isinstance(tid, int) or str(tid).isdigit()) else f"{tid} \u00b7 {roles[tid]}" for tid in order]
    ax.set_yticklabels(ytick_labels, fontsize=8)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(TIME_FMT)
    ax.set_xlabel("video time")
    ax.set_title(title)
    ax.legend(handles=[Patch(facecolor=c, label=z) for z, c in zcolors.items()],
              loc="upper right", ncols=2, fontsize=8)
    plt.tight_layout()
    plt.show()

def plot_occupancy(events, t_end, title):
    times, counts = occupancy_timeline(events, t_end, step_s=10)
    fig, ax = plt.subplots(figsize=(13, 3))
    for role, series in counts.items():
        ax.step(times, series, where="post", linewidth=2,
                color=ROLE_HEXES.get(role, MUTED), label=role)
    ax.xaxis.set_major_formatter(TIME_FMT)
    ax.set_ylabel("people present")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_wait_distribution(waits, threshold_s, title, max_rows=30):
    if not waits:
        print("(nobody dwelled in the waiting zone)")
        return
    items = sorted(waits.items(), key=lambda kv: kv[1])
    if len(items) > max_rows:   # crowded footage: keep the chart readable
        print(f"(showing top {max_rows} of {len(items)} waiters — "
              f"{len(items) - max_rows} shorter waits omitted; all in events CSV)")
        items = items[-max_rows:]
    fig, ax = plt.subplots(figsize=(9, max(2.2, 0.34 * len(items) + 1)))
    ax.barh([f"P{t}" for t, _ in items], [v for _, v in items],
            color=SEQ_HUE, height=0.55)
    ax.axvline(threshold_s, color=STATUS_CRIT, ls="--", lw=1.5)
    ax.text(threshold_s, -0.45, f" threshold {mmss(threshold_s)}",
            color=STATUS_CRIT, fontsize=9)
    for i, (tid, v) in enumerate(items):
        if v >= threshold_s:                       # selective direct labels
            ax.text(v, i, f" {mmss(v)}", va="center", fontsize=8, color=INK2)
    ax.xaxis.set_major_formatter(TIME_FMT)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

def plot_presence_timeline(events, zone, t_end, title):
    staffed = merge_intervals([(e["t_in"], e["t_out"]) for e in events
                               if e["zone"] == zone and e["role"] == "staff"])
    gaps = complement_intervals(staffed, 0, t_end)
    fig, ax = plt.subplots(figsize=(13, 1.8))
    ax.broken_barh([(s, e - s) for s, e in staffed], (0, 1), facecolors=STATUS_GOOD)
    ax.broken_barh([(s, e - s) for s, e in gaps], (0, 1), facecolors=STATUS_CRIT)
    ax.set_yticks([])
    ax.set_xlim(0, t_end)
    ax.xaxis.set_major_formatter(TIME_FMT)
    ax.legend(handles=[Patch(facecolor=STATUS_GOOD, label="staffed ✓"),
                       Patch(facecolor=STATUS_CRIT, label="unattended ✗")],
              loc="upper right", ncols=2, fontsize=9)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

def plot_table_service(events, table_zones, title):
    tables = sorted(table_zones)
    if not tables:
        print("(no table zones)")
        return
    fig, ax = plt.subplots(figsize=(13, max(2.2, 0.55 * len(tables) + 1.2)))
    for y, tz in enumerate(tables):
        for p in detect_parties(events, tz, PARTY_GAP_S, MIN_PARTY_S):
            ax.broken_barh([(p[0], p[1] - p[0])], (y - 0.3, 0.6),
                           facecolors=GRID, edgecolor=BASE_LINE)
            for v in staff_visits(events, tz, p, VISIT_MIN_S):
                ax.broken_barh([(v[0], v[1] - v[0])], (y - 0.3, 0.6),
                               facecolors=ROLE_HEXES["staff"])
    ax.set_yticks(range(len(tables)))
    ax.set_yticklabels(tables, fontsize=9)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(TIME_FMT)
    ax.legend(handles=[Patch(facecolor=GRID, edgecolor=BASE_LINE, label="party seated"),
                       Patch(facecolor=ROLE_HEXES["staff"], label="staff visit")],
              loc="upper right", ncols=2, fontsize=9)
    ax.set_title(title + "  (gray = party seated, orange = staff visit)")
    plt.tight_layout()
    plt.show()

def plot_occupancy_heatmap(frame_log, video_path, title, blur_px=25):
    """Dwell-density heatmap from the SAME per-frame box centers the
    pipeline already logged for rendering -- no re-processing needed.

    IMPORTANT — what this is and isn't: this is a visualization of WHERE
    people spent time, aggregated across every tracked box regardless of
    which track_id it belonged to. It does NOT use identity at all, which
    is exactly why it can't fix or diagnose ID fluctuations -- a swapped ID
    still leaves a footprint in the same physical spot, so the heatmap
    looks identical whether the tracking underneath was perfect or a mess.
    Useful for "where do people linger" business questions; not a tracking
    diagnostic and not a re-identification method.
    """
    frame = cv2.cvtColor(first_frame(video_path), cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]
    density = np.zeros((h, w), dtype=np.float32)
    for _, _, boxes in frame_log:
        for _tid, x1, y1, x2, y2 in boxes:
            cx, cy = int((x1 + x2) / 2), int(min(y2, h - 1))  # foot position
            if 0 <= cx < w and 0 <= cy < h:
                density[cy, cx] += 1.0
    if density.max() == 0:
        print("(no tracked positions to build a heatmap from)")
        return
    density = cv2.GaussianBlur(density, (0, 0), sigmaX=blur_px)
    density = density / density.max()
    fig, ax = plt.subplots(figsize=(15, 9))
    ax.imshow(frame)
    ax.imshow(density, cmap="inferno", alpha=0.55, vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title + "  (dwell density — NOT an identity/tracking signal)")
    plt.tight_layout()
    plt.show()


def plot_reid_pair_audit(pairs, track_crops, title, max_pairs=6):
    """(v36) The actual crops behind a calibration number, side by side --
    turns "why don't the numbers separate" into something you can just
    look at, instead of another round of threshold guessing.

    pairs: [(a, b, sim), ...] -- e.g. calibration_report["same_pairs_worst"]
           or ["diff_pairs_worst"], already worst-first.
    track_crops: {track_id: [(_, crop_bgr), ...]} -- same structure used for
                 face embedding; picks the LARGEST banked crop per track as
                 the representative image (most informative for a human to
                 judge), and prints its pixel size so you can see directly
                 if crops are simply too small/blurry for even a human to
                 tell two people apart -- if you can't tell either, the
                 model failing isn't a threshold problem, it's a resolution/
                 occlusion/uniform-clothing problem no threshold fixes.
    """
    items = pairs[:max_pairs]
    if not items:
        print(f"({title}: no pairs to show)")
        return
    fig, axes = plt.subplots(len(items), 2, figsize=(6.4, 2.6 * len(items)))
    axes = np.atleast_2d(axes)
    for row, (a, b, sim) in enumerate(items):
        for col, tid in enumerate((a, b)):
            ax = axes[row][col]
            ax.axis("off")
            crops = track_crops.get(tid, [])
            if crops:
                _, crop = max(crops, key=lambda c: c[1].shape[0] * c[1].shape[1])
                ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                ax.set_title(f"ID {tid}  ({crop.shape[1]}x{crop.shape[0]}px)",
                            fontsize=9)
            else:
                ax.set_title(f"ID {tid} (no banked crop)", fontsize=9)
        axes[row][0].set_ylabel(f"sim={sim:.3f}", fontsize=10, rotation=0,
                                labelpad=32, ha="right", va="center")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()



In [ ]:
_src_crop_h = []   # V76: source crop heights, pre-resize
# v42 mix track_id sort key (some are integers, some are string names like 'jane')
def track_sort_key(tid):
    if isinstance(tid, str) and not tid.isdigit():
        return (0, tid)
    try:
        return (1, int(tid))
    except ValueError:
        return (2, str(tid))

# Cell 7 — VIDEO ENGINE, two-pass architecture:
#   PASS 1 (analyze): detect -> track -> zones -> events. No drawing.
#   then: staff override -> Re-ID stitching -> ghost filter
#   PASS 2 (render): draw the video from FINAL stitched identities, so
#   on-screen ids are persistent, wait-clocks never reset, and roles are
#   correct from the first frame. (Industry-standard: analyze, then render.)
import bisect
import shutil
import supervision as sv
from collections import defaultdict, deque, Counter
from ultralytics import YOLO
from tqdm.auto import tqdm

# ── numpy>=2.5 compatibility shim ──────────────────────────────────────────
# numpy 2.5 removed 2-D np.cross; every released supervision (<=0.29) still
# calls it in exactly two places (verified against the 0.26.1 source). Rebind
# determinant-based equivalents at every import site — same fix supervision
# made on their develop branch. Verified numerically identical to np.cross.
from supervision.geometry.core import Point as _SvPoint
import supervision.geometry.utils as _sv_geo
try:
    import supervision.detection.utils.internal as _sv_internal
except ImportError:
    _sv_internal = None
import supervision.detection.tools.polygon_zone as _sv_pz
try:
    import supervision.detection.line_zone as _sv_lz
except ImportError:
    _sv_lz = None

def _cross2d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]

def _get_polygon_center(polygon):
    if len(polygon) == 0:
        raise ValueError("Polygon must have at least one vertex.")
    shift = np.roll(polygon, -1, axis=0)
    signed_areas = _cross2d(polygon, shift) / 2
    if signed_areas.sum() == 0:
        c = np.mean(polygon, axis=0).round()
        return _SvPoint(x=c[0], y=c[1])
    centroids = (polygon + shift) / 3.0
    c = np.average(centroids, axis=0, weights=signed_areas).round()
    return _SvPoint(x=c[0], y=c[1])

def _cross_product(anchors, vector):
    v0 = float(vector.end.x - vector.start.x)
    v1 = float(vector.end.y - vector.start.y)
    d = np.asarray(anchors, dtype=float) - np.array(
        [vector.start.x, vector.start.y], dtype=float)
    return v0 * d[..., 1] - v1 * d[..., 0]

_sv_geo.get_polygon_center = _get_polygon_center
_sv_pz.get_polygon_center = _get_polygon_center
if _sv_internal is not None:
    _sv_internal.cross_product = _cross_product
if _sv_lz is not None:
    _sv_lz.cross_product = _cross_product

_REID_STATE = {"embed": None, "failed": False, "method": None, "backend": None}
_FACE_STATE = {"app": None, "failed": False}

try:
    _FACE_LOCK
except NameError:
    import threading as _thr2
    _FACE_LOCK = _thr2.Lock()


def get_face_analyzer():
    with _FACE_LOCK:
        """InsightFace app (lazy, cached). Corroborating signal only — see
        ENABLE_FACE_CORROBORATION in config. Returns None (never raises) if
        insightface isn't installed or fails to load; caller just skips the
        face cross-check in that case."""
        if _FACE_STATE["failed"] or not ENABLE_FACE_CORROBORATION:
            return None
        if _FACE_STATE["app"] is None:
            try:
                from insightface.app import FaceAnalysis
                providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                            if "cuda" in str(DEVICE) else ["CPUExecutionProvider"])
                model_name = FACE_MODEL_NAME if 'FACE_MODEL_NAME' in globals() else "buffalo_sc"
                try:
                    app = FaceAnalysis(name=model_name, providers=providers)
                    app.prepare(ctx_id=0 if "cuda" in str(DEVICE) else -1, det_size=(320, 320))
                    _FACE_STATE["app"] = app
                    if _from_source and _med < 100:
                        print(f"🚨 CROP RESOLUTION: median SOURCE crop is "
                              f"only {_med}px tall (p10 {_p10}px). ReID backbones "
                              f"expect ~256px. These embeddings are mostly "
                              f"interpolation — no threshold or backbone will "
                              f"separate people reliably at this size.")
                    print(f"✅ Face model ready (InsightFace {model_name})")
                    discover_and_load_staff_gallery(app)
                except Exception as e1:
                    if model_name != "buffalo_sc":
                        print(f"⚠️ Failed to load {model_name} ({e1}) - falling back to buffalo_sc...")
                        app = FaceAnalysis(name="buffalo_sc", providers=providers)
                        app.prepare(ctx_id=0 if "cuda" in str(DEVICE) else -1, det_size=(320, 320))
                        _FACE_STATE["app"] = app
                        print("✅ Face model ready (InsightFace buffalo_sc)")
                        discover_and_load_staff_gallery(app)
                    else:
                        raise e1
            except Exception as exc:
                _FACE_STATE["failed"] = True
                print(f"ℹ️ Face corroboration unavailable ({exc!r}) — "
                      f"CLIP/OSNet stitching is unaffected, this only disables "
                      f"the optional face cross-check.")
        return _FACE_STATE["app"]


_STAFF_FACE_GALLERY = {}

def discover_and_load_staff_gallery(face_analyzer):
    """Load staff face images from STAFF_GALLERY_DIR if present."""
    global _STAFF_FACE_GALLERY
    if not _STAFF_FACE_GALLERY:
        _STAFF_FACE_GALLERY = {}
    if face_analyzer is None:
        return _STAFF_FACE_GALLERY
    
    gallery_dir = Path(STAFF_GALLERY_DIR if 'STAFF_GALLERY_DIR' in globals() else "staff_gallery")
    search_dirs = [BASE / gallery_dir, Path.cwd() / gallery_dir]
    if 'INPUT_ROOT' in globals() and INPUT_ROOT.exists():
        search_dirs.append(INPUT_ROOT / gallery_dir)
        try:
            for folder in INPUT_ROOT.rglob("*"):
                if folder.is_dir() and folder.name == gallery_dir.name:
                    search_dirs.append(folder)
        except Exception:
            pass
                
    found_dir = None
    for d in search_dirs:
        if d.exists() and d.is_dir():
            found_dir = d
            break
            
    if not found_dir:
        return _STAFF_FACE_GALLERY
        
    img_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    img_files = []
    try:
        img_files = [f for f in found_dir.iterdir() if f.suffix.lower() in img_exts]
    except Exception:
        pass
    if not img_files:
        return _STAFF_FACE_GALLERY
        
    print(f"👥 Loading staff gallery from: {found_dir}")
    for f in img_files:
        name = f.name.replace(f.suffix, "").lower()
        try:
            img = cv2.imread(str(f))
            if img is None or img.size == 0:
                continue
            faces = face_analyzer.get(img)
            if not faces:
                print(f"   ⚠️ No face detected in staff image {f.name}")
                continue
            face = max(faces, key=lambda f_: (f_.bbox[2] - f_.bbox[0]) * (f_.bbox[3] - f_.bbox[1]))
            _STAFF_FACE_GALLERY[name] = face.normed_embedding.tolist()
            print(f"   ✅ Enrolled staff face: {name} (det_score={face.det_score:.2f})")
        except Exception as e:
            print(f"   ❌ Error loading staff image {f.name}: {e}")
            
    return _STAFF_FACE_GALLERY

def _blur_score(crop_bgr, reference_size=128):
    # v46-fix: module-level copy so embed_face_scored (module scope) can call it.
    # process_video keeps its own nested _blur_score for the frame loop; identical logic.
    if crop_bgr is None or getattr(crop_bgr, "size", 0) == 0:
        return 0.0
    resized = cv2.resize(crop_bgr, (reference_size, reference_size), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def embed_face_scored(crop):
    """One crop -> (normalized 512-d embedding, detector confidence, face
    side px) for the largest face found, or (None, 0.0, 0) if no usable
    face. Person crops from CCTV are frequently too small/angled for a face
    to resolve at all — that's expected, not an error.

    Rejects on TWO independent grounds, not one:
      det_score  -- the detector's confidence a face exists here at all.
      face size  -- (v32) even a confidently-detected face can be too small/
                    blurry for the RECOGNITION embedding to carry real
                    identity signal — detection and recognition degrade at
                    different rates with resolution, so a size floor catches
                    failures a confidence floor alone does not.
    """
    app = get_face_analyzer()
    if app is None or crop is None or crop.size == 0:
        return None, 0.0, 0
    # v43 face crop blur gate
    ch, cw = crop.shape[:2]
    if min(ch, cw) >= MIN_CROP_PX_BLUR_GATE:
        if _blur_score(crop) < MIN_BLUR_VARIANCE:
            return None, 0.0, 0
    try:
        faces = app.get(crop)
    except Exception:
        return None, 0.0, 0
    if not faces:
        return None, 0.0, 0
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    side = min(face.bbox[2] - face.bbox[0], face.bbox[3] - face.bbox[1])
    if getattr(face, "det_score", 0.0) < FACE_MIN_DET_SCORE or side < FACE_MIN_FACE_PX:
        return None, 0.0, float(side)
    return face.normed_embedding.tolist(), float(face.det_score), float(side)

def get_track_face_embedding(crops, size_log=None):
    """Tries every banked crop for a track (not just crop[0], which was
    ranked by PERSON-detection confidence*height, not face visibility) and
    keeps whichever gives the highest-confidence face that also clears
    FACE_MIN_FACE_PX. If size_log is given, appends every rejected crop's
    face side (px) so callers can print a diagnostic of what's being
    filtered out and why (see the coverage print in the ReID stitch block)."""
    best_emb, best_score = None, 0.0
    for _, crop in crops:
        emb, score, side = embed_face_scored(crop)
        if emb is not None and score > best_score:
            best_emb, best_score = emb, score
        elif emb is None and size_log is not None and side > 0:
            size_log.append(side)
    return best_emb

def _recrop_face_from_source(video_path, entries, scale, step, native_fps,
                             start_seconds, k=4):
    """V65: retry a track's face at SOURCE resolution.

    entries: [(analysed_frame_idx, (x1, y1, x2, y2))] in ANALYSED coords.
    Picks the k largest boxes, seeks the source video at the matching native
    frame (src_frame = start*fps + idx*step), crops the head region (top 50%
    of the body + margin) at native scale, and returns the best-scoring
    embedding or None. A 15px face at 720p is a 45px face at 4K.
    """
    if not entries or scale <= 1.2:
        return None
    best = sorted(entries, key=lambda e: -(e[1][3] - e[1][1]))[:k]
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    out, best_score = None, 0.0
    _f0 = int(round(float(start_seconds or 0.0) * native_fps))
    for fi, (x1, y1, x2, y2) in best:
        cap.set(cv2.CAP_PROP_POS_FRAMES, _f0 + int(fi) * int(step))
        ok, fr = cap.read()
        if not ok or fr is None:
            continue
        X1, Y1, X2, Y2 = (int(v * scale) for v in (x1, y1, x2, y2))
        pad = max(4, int((X2 - X1) * 0.15))
        crop = fr[max(0, Y1 - pad):Y1 + max(1, (Y2 - Y1) // 2),
                  max(0, X1 - pad):min(fr.shape[1], X2 + pad)]
        if crop.size == 0:
            continue
        emb, score, _side = embed_face_scored(crop)
        if emb is not None and score > best_score:
            out, best_score = emb, score
    cap.release()
    return out


def _resolve_reid_weights(local_path, stock_name):
    """Single source of truth for a ReID weights path, generalized (v28) to
    work for ANY backbone family (CLIP-ReID, OSNet, ...) — used by BOTH the
    offline Re-ID stitcher (get_reid_embedder) and the online BotSORT tracker
    (build_online_tracker/_botsort_yaml), so they can never drift out of sync
    the way _botsort_yaml's old hardcoded model: auto silently did.
    boxmot creates a .lock file NEXT TO the weights — impossible on Kaggle's
    read-only /kaggle/input (Errno 30). Copy the file into the writable
    working dir and load from there. If no local file was shipped in the
    input dataset, return the bare stock filename — boxmot auto-downloads
    known filenames (clip_market1501.pt, osnet_x0_25_msmt17.pt, ...) itself.
    """
    if local_path and str(local_path).startswith("/kaggle/input"):
        _w_dst = BASE / Path(local_path).name
        if not _w_dst.exists():
            _w_dst.write_bytes(Path(local_path).read_bytes())
        return str(_w_dst)
    return str(local_path) if local_path else stock_name

def _resolve_osnet_weights():
    """Back-compat alias — OSNet is now the v28 FALLBACK backbone, kept as its
    own function since get_reid_embedder's fallback path still calls it by
    name for clarity in tracebacks/prints. (v35) stock filename now comes
    from OSNET_STOCK_VARIANT config, not a hardcoded string, so switching to
    a stronger variant (osnet_x1_0_msmt17.pt) is a one-line config change."""
    return _resolve_reid_weights(OSNET_WEIGHTS, OSNET_STOCK_VARIANT)

def _load_reid_backend(weights_path, dev):
    """Loads ANY boxmot-supported ReID checkpoint (CLIP-ReID, OSNet, ...) —
    the loader itself is filename/weights-driven, not backbone-specific, so
    the same three-import-path fallback works unchanged for either family.
    boxmot's internal module layout has moved more than once across
    releases — try every known location in turn instead of assuming pip
    installed a version matching just one path."""
    last_exc = None
    attempts = [
        ("boxmot.reid.core", "ReID",
         lambda cls: cls(weights=weights_path, device=dev, half=False).model),
        ("boxmot.appearance.reid_auto_backend", "ReidAutoBackend",
         lambda cls: cls(weights=Path(weights_path), device=dev, half=False).model),
        ("boxmot.appearance.backends.auto_backend", "AutoBackend",
         lambda cls: cls(weights=Path(weights_path), device=dev, half=False).model),
    ]
    for module_name, cls_name, build in attempts:
        try:
            mod = __import__(module_name, fromlist=[cls_name])
            cls = getattr(mod, cls_name)
            return build(cls)
        except (ImportError, AttributeError) as e:
            last_exc = e
            continue
    raise ImportError(
        f"no working boxmot Re-ID import path found (last: {last_exc!r})"
    ) from last_exc

def _boxmot_dev(device):
    """boxmot wants '0'/'1'/'cpu' — NOT 'cuda' or 'cuda:1' (parse_device raises)."""
    d = str(device or globals().get("DEVICE", "cpu"))
    if "cuda" not in d:
        return "cpu"
    return d.split(":")[1] if ":" in d else "0"


_REID_CACHE = {}          # v55: device -> state, so 2 GPUs get 2 real embedders
try:
    _REID_LOCK
except NameError:
    import threading as _thr
    _REID_LOCK = _thr.Lock()


def get_reid_embedder(device=None):
    """Appearance embedder (lazy, cached), v28 chain: CLIP-ReID (heavy,
    stronger) -> OSNet (light, fallback) -> HSV color-histogram (last
    resort). Returns a function crops->list-of-vectors (dim depends on
    whichever backbone actually loaded — cosine similarity downstream
    doesn't care about dimensionality)."""
    _dev_key = _boxmot_dev(device)
    global _REID_STATE
    with _REID_LOCK:
        _REID_STATE = _REID_CACHE.setdefault(
            _dev_key, {"embed": None, "backend": None, "method": None, "failed": False})
        if _REID_STATE["failed"]:
            return None
        if _REID_STATE["embed"] is not None:
            return _REID_STATE["embed"]
        import os as _os
        _cvd = _os.environ.get("CUDA_VISIBLE_DEVICES")

        def _load_with_gpu_fallback(weights_path, label):
            try:
                return _load_reid_backend(weights_path, _dev_key)
            except Exception as gpu_exc:
                import traceback
                print(f"({label} on GPU failed: {gpu_exc!r} — retrying on CPU)")
                print(traceback.format_exc()[-1500:])
                return _load_reid_backend(weights_path, "cpu")

        def _make_embed(backend):
            def _embed(crops):
                # v55: batch. get_features(boxes, img) already batches over boxes,
                # so tile the (uniform 128x256) crops side by side and ask once.
                # Was: one forward pass per crop, called per detection per FRAME.
                out = []
                B = int(globals().get("EMBED_BATCH", 24))
                for s in range(0, len(crops), B):
                    grp = [c if c.shape[:2] == (256, 128) else cv2.resize(c, (128, 256))
                           for c in crops[s:s + B]]
                    canvas = np.hstack(grp)
                    boxes = np.array([[k * 128, 0, (k + 1) * 128, 256]
                                      for k in range(len(grp))], dtype=np.float32)
                    feats = np.asarray(backend.get_features(boxes, canvas),
                                       dtype=np.float32).reshape(len(grp), -1)
                    for v in feats:
                        n = float(np.linalg.norm(v)) or 1e-9
                        out.append((v / n).tolist())
                return out
            return _embed

        clip_reason = None
        try:
            if FORCE_REID_BACKBONE == "osnet":
                raise RuntimeError(
                    "FORCE_REID_BACKBONE='osnet' — skipping CLIP-ReID "
                    "intentionally (not a real failure) for a clean "
                    "head-to-head comparison against OSNet on this footage")
            w_clip = _resolve_reid_weights(CLIP_REID_WEIGHTS, REID_BACKBONE_STOCK)
            try:
                backend = _load_with_gpu_fallback(w_clip, "CLIP-ReID")
            finally:
                # restore even on failure — a boxmot-mutated CUDA_VISIBLE_DEVICES
                # must never leak into later cells
                if _cvd is None:
                    _os.environ.pop("CUDA_VISIBLE_DEVICES", None)
                else:
                    _os.environ["CUDA_VISIBLE_DEVICES"] = _cvd
            _REID_STATE["embed"] = _make_embed(backend)
            _REID_STATE["backend"] = backend
            _REID_STATE["method"] = "clip"
            print(f"✅ Re-ID model ready (CLIP-ReID, weights: {w_clip})")
        except Exception as clip_exc:
            import traceback
            clip_reason = f"{clip_exc!r}\n{traceback.format_exc()[-1500:]}"
            print("=" * 70)
            print(f"⚠️ CLIP-ReID unavailable: {clip_exc!r}")
            print(traceback.format_exc()[-1500:])
            print("   Trying OSNet next.")
            print("=" * 70)

        if _REID_STATE["embed"] is None:
            try:
                w = _resolve_osnet_weights()
                try:
                    backend = _load_with_gpu_fallback(w, "OSNet")
                finally:
                    if _cvd is None:
                        _os.environ.pop("CUDA_VISIBLE_DEVICES", None)
                    else:
                        _os.environ["CUDA_VISIBLE_DEVICES"] = _cvd

                _REID_STATE["embed"] = _make_embed(backend)
                _REID_STATE["backend"] = backend
                print(f"✅ Re-ID model ready (OSNet, weights: {w})")
                _REID_STATE["method"] = "osnet"
            except Exception as osnet_exc:
                import traceback
                osnet_reason = f"{osnet_exc!r}\n{traceback.format_exc()}"
                try:
                    import boxmot
                    boxmot_note = f"boxmot version installed: {boxmot.__version__}"
                except Exception:
                    boxmot_note = "boxmot import itself failed — package likely missing/broken"

                _REID_STATE["failed"] = True
                print("=" * 70)
                print("🚨 Re-ID backend FAILED — CLIP-ReID and OSNet both unavailable.")
                print("   NOT falling back to the weak HSV matcher. Fix the cause below, then re-run.")
                print("-" * 70)
                print(f"CLIP-ReID failure:\n{clip_reason}")
                print("-" * 70)
                print(f"OSNet failure:\n{osnet_reason}")
                print(f"   {boxmot_note}")
                print("=" * 70)
                raise RuntimeError(
                    "Re-ID backend unavailable: both CLIP-ReID and OSNet failed to "
                    "load (see printed reasons above). Not falling back to HSV — "
                    "fix the root cause (likely a boxmot/package install issue, "
                    "missing weights, or the Kaggle Internet toggle being off) "
                    "and re-run."
                ) from osnet_exc
    return _REID_CACHE[_dev_key]["embed"]

def _ffmpeg_ok():
    import subprocess
    try:
        subprocess.run(["ffmpeg", "-version"], capture_output=True, timeout=20)
        return True
    except Exception:
        return False


_HWACCEL_STATE = {}


def _hwaccel_args():
    """The T4 has a hardware video decoder sitting idle while the CPU grinds
    through 4K. Use it if this ffmpeg build was compiled with it — probe once,
    cache the answer, fall back silently to software if not."""
    mode = str(globals().get("FFMPEG_HWACCEL", "auto")).lower()
    if mode in ("none", "off", "false"):
        return []
    if "args" in _HWACCEL_STATE:
        return _HWACCEL_STATE["args"]
    import subprocess
    args = []
    try:
        out = subprocess.run(["ffmpeg", "-hide_banner", "-hwaccels"],
                             capture_output=True, timeout=30, text=True)
        have = (out.stdout or "") + (out.stderr or "")
        if "cuda" in have:
            # listed != working (no GPU visible to ffmpeg, driver mismatch...),
            # so actually decode one frame before trusting it.
            probe = subprocess.run(
                ["ffmpeg", "-v", "error", "-hwaccel", "cuda", "-f", "lavfi",
                 "-i", "testsrc=size=64x64:duration=0.1", "-frames:v", "1",
                 "-f", "null", "-"], capture_output=True, timeout=60)
            if probe.returncode == 0:
                args = ["-hwaccel", "cuda"]
    except Exception:
        args = []
    _HWACCEL_STATE["args"] = args
    print(f"   ffmpeg decode: {'NVDEC (hardware)' if args else 'software (CPU)'}"
          f"{'' if args else ' — this build has no working cuda hwaccel'}")
    return args


def frame_source(video_path, fps, max_seconds=None, max_w=1920, start_seconds=0.0,
                 keyframes_only=False):
    """Yield (i, t, frame) at `fps`.

    ffmpeg drops and scales frames inside the decoder, so a 4K chunk costs one
    cheap decode instead of 108,240 full-size ones. t = i/fps: with the fps
    filter that is PTS-correct even on the VFR exports an NVR produces, which
    is what CLOCK_SOURCE was trying to work around by hand.
    cv2 fallback keeps the old behaviour when ffmpeg is unavailable.
    """
    import subprocess
    cap = cv2.VideoCapture(str(video_path))
    nat = cap.get(cv2.CAP_PROP_FPS) or NATIVE_FPS_OVERRIDE or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1920
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080
    cap.release()
    ow = min(int(max_w), w) // 2 * 2
    oh = int(round(h * ow / max(w, 1))) // 2 * 2

    if globals().get("USE_FFMPEG_READER", True) and _ffmpeg_ok():
        cmd = ["ffmpeg", "-v", "error"] + _hwaccel_args()
        if keyframes_only:
            # decode I-frames only: for a coarse density scan we do not need
            # every frame, and this is ~5-10x cheaper on 4K. The fps filter
            # below still resamples on PTS, so the timestamps stay exact.
            cmd += ["-skip_frame", "nokey"]
        if start_seconds:
            cmd += ["-ss", str(float(start_seconds))]   # before -i = keyframe seek, instant
        cmd += ["-i", str(video_path)]
        if max_seconds:
            cmd += ["-t", str(float(max_seconds))]
        cmd += ["-vf", f"fps={fps},scale={ow}:{oh}", "-an",
                "-f", "rawvideo", "-pix_fmt", "bgr24", "-"]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.DEVNULL, bufsize=ow * oh * 3)
        nbytes, i, got_any = ow * oh * 3, 0, False
        try:
            while True:
                buf = proc.stdout.read(nbytes)
                if len(buf) < nbytes:
                    break
                got_any = True
                yield i, start_seconds + i / float(fps), np.frombuffer(buf, np.uint8).reshape(oh, ow, 3).copy()
                i += 1
        finally:
            try:
                proc.stdout.close(); proc.kill()
            except Exception:
                pass
        if got_any:
            return
        print("   !! ffmpeg reader produced no frames — falling back to cv2")

    cap = cv2.VideoCapture(str(video_path))
    step = max(1, int(round(nat / float(fps))))
    idx = i = 0
    if start_seconds:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * nat))
    limit = int(max_seconds * nat) if max_seconds else None
    while True:
        ok, fr = cap.read()
        if not ok or (limit and idx > limit):
            break
        if idx % step == 0:
            if fr.shape[1] > ow:
                fr = cv2.resize(fr, (ow, oh))
            yield i, start_seconds + idx / nat, fr
            i += 1
        idx += 1
    cap.release()


def apply_clahe(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def _botsort_yaml(track_buffer_frames):
    # model: auto — Ultralytics native with_reid does NOT accept torchreid/boxmot
    # OSNet .pt checkpoints (wrong format for its YOLO()-based .pt loader; confirmed
    # via TypeError on real run). "auto" reuses the detector backbone's own features
    # as a lightweight appearance proxy instead — weaker than dedicated OSNet but a
    # real, working signal with zero extra load. Actual OSNet stays in the OFFLINE
    # stitcher (get_reid_embedder), which loads it via boxmot's own loader — a
    # different code path with no Ultralytics checkpoint-format constraint.
    #
    # proximity_thresh lowered 0.5 -> 0.3 (2026-07-12): diagnostic run found 77 ID
    # swaps split into two mechanisms — (A) live Hungarian swaps during close
    # encounters/crossing (no fix here, needs real online appearance embeddings),
    # and (B) dead-recovery errors where a track lost for 100-230+ frames snaps
    # onto a nearby but wrong person once the Kalman search window has expanded.
    # Tightening proximity_thresh targets (B) specifically — stricter spatial gate
    # on lost-track reactivation, fewer long-gap false recoveries. Does not fix (A).
    p = BASE / "botsort_reid.yaml"
    _hi = NEW_TRACK_CONF if ENABLE_CONF_HYSTERESIS else CONF_THRESHOLD
    _lo = KEEP_TRACK_CONF if ENABLE_CONF_HYSTERESIS else 0.1
    p.write_text(f"""tracker_type: botsort
track_high_thresh: {_hi}
track_low_thresh: {_lo}
new_track_thresh: {_hi}
track_buffer: {int(track_buffer_frames)}
match_thresh: {BOTSORT_MATCH_THRESH}
fuse_score: True
gmc_method: {"sparseOptFlow" if ENABLE_GMC else "none"}
proximity_thresh: 0.3
appearance_thresh: 0.25
with_reid: True
model: auto
""")
    return str(p)

def build_online_tracker(fps, device=None):
    """Real online Re-ID — supports BotSort and BoostTrack. No silent
    fallback: if the real backend or the tracker construction fails, this
    raises with the reason instead of quietly downgrading to the weaker
    with_reid=auto proxy.
    """
    if not USE_REAL_ONLINE_REID:
        return None
    get_reid_embedder(device=device)  # raises on failure — no silent HSV/None fallback
    _st = _REID_CACHE.get(_boxmot_dev(device), _REID_STATE)   # v55: per-GPU state
    backend = _st.get("backend")
    method = _st.get("method")
    _hi = NEW_TRACK_CONF if ENABLE_CONF_HYSTERESIS else CONF_THRESHOLD
    _lo = KEEP_TRACK_CONF if ENABLE_CONF_HYSTERESIS else 0.1
    _cmc = GMC_METHOD if ENABLE_GMC else None
    if TRACKER_MODE == "boosttrack":
        from boxmot.trackers.boosttrack.boosttrack import BoostTrack
        tracker = BoostTrack(
            reid_model=backend,
            device=_boxmot_dev(device),
            half=False,
            track_high_thresh=_hi,
            track_low_thresh=_lo,
            new_track_thresh=_hi,
            track_buffer=int(fps * LOST_TRACK_BUFFER_S),
            match_thresh=BOTSORT_MATCH_THRESH,
            frame_rate=int(round(fps)),
        )
        print(f"✅ tracker: BoostTrack with REAL {method.upper()} embeddings driving live association")
    elif TRACKER_MODE == "occluboost":
        # C3: BoostTrack + occlusion-aware Kalman damping + graveyard
        # re-association (GTA). Needs a boxmot newer than 19.x — raise loudly,
        # never silently fall back. Constructor args are signature-filtered so
        # minor upstream API drift doesn't crash the run.
        import inspect as _insp
        try:
            from boxmot import OccluBoost as _OB
        except ImportError:
            from boxmot.trackers.occluboost.occluboost import OccluBoost as _OB
        _want = dict(reid_model=backend, device=_boxmot_dev(device), half=False,
                     track_high_thresh=_hi, track_low_thresh=_lo,
                     new_track_thresh=_hi,
                     track_buffer=int(fps * LOST_TRACK_BUFFER_S),
                     match_thresh=BOTSORT_MATCH_THRESH,
                     cmc_method=_cmc,
                     frame_rate=int(round(fps)))
        _sig = _insp.signature(_OB.__init__).parameters
        tracker = _OB(**{k: v for k, v in _want.items() if k in _sig})
        print(f"✅ tracker: OccluBoost with REAL {method.upper()} embeddings "
              f"(occlusion-damped Kalman + GTA graveyard re-association)")
    else:
        from boxmot.trackers.botsort.botsort import BotSort
        tracker = BotSort(
            reid_model=backend,
            track_high_thresh=_hi,
            track_low_thresh=_lo,
            new_track_thresh=_hi,
            track_buffer=int(fps * LOST_TRACK_BUFFER_S),
            match_thresh=BOTSORT_MATCH_THRESH,
            proximity_thresh=0.3,
            appearance_thresh=LIVE_APPEARANCE_THRESH,
            cmc_method=_cmc,
            frame_rate=int(round(fps)),
            with_reid=True,
        )
        print(f"✅ tracker: BotSORT with REAL {method.upper()} embeddings driving live association")
    return tracker

def _dets_to_boxmot(dets):
    """sv.Detections -> boxmot's expected (N,6) x1,y1,x2,y2,conf,cls array."""
    if len(dets) == 0:
        return np.empty((0, 6), dtype=np.float32)
    conf = dets.confidence if dets.confidence is not None else np.ones(len(dets))
    cls = dets.class_id if dets.class_id is not None else np.zeros(len(dets))
    return np.column_stack([dets.xyxy.astype(np.float32),
                            np.asarray(conf, dtype=np.float32),
                            np.asarray(cls, dtype=np.float32)])


def _boxmot_to_dets(tracked):
    """boxmot TrackResults -> sv.Detections, so the rest of the pipeline
    (zones, events, crop banking) doesn't need to know which tracker ran.
    NOTE: sv.Detections.empty()'s tracker_id defaults to None, not an empty
    array, which breaks the downstream `zip(..., dets.tracker_id, ...)` — so
    the empty case is built explicitly rather than via .empty()."""
    if len(tracked) == 0:
        d = sv.Detections(xyxy=np.empty((0, 4), dtype=np.float32))
        d.confidence = np.array([], dtype=np.float32)
        d.class_id = np.array([], dtype=int)
        d.tracker_id = np.array([], dtype=int)
        return d
    return sv.Detections(
        xyxy=np.asarray(tracked.xyxy, dtype=np.float32),
        confidence=np.asarray(tracked.conf, dtype=np.float32),
        class_id=np.asarray(tracked.cls, dtype=int),
        tracker_id=np.asarray(tracked.id, dtype=int),
    )


class _PerspectiveModel:
    """Scene-geometry prior, learned from this run's own detections.

    On a FIXED camera looking at a FLAT floor, a standing person's pixel height
    is a near-linear function of the y of their feet: people low in the frame
    are near and tall, people high in the frame are far and short. Fitting that
    line costs nothing (we already have every box) and gives us the one signal
    box-containment cannot provide — whether a detection is standing on the
    floor or floating above it.

    Robust by construction: y is binned and the MEDIAN height per bin is fitted,
    so a handful of bad boxes cannot tilt the line. Only isolated, confident
    detections are fed in, so a merged double-box does not teach it that people
    are twice as tall as they are.

    ponytail: least-squares over bin medians, not RANSAC. Bin medians already
    kill the outliers RANSAC would; upgrade only if a real fit is seen to drift.
    Phase 3 replaces this with a true 4-point homography and keeps this as the
    cross-check.
    """

    def __init__(self, n_bins=12, min_samples=None):
        self.n_bins = n_bins
        self.min_samples = (CARRIED_MIN_FIT_SAMPLES if min_samples is None
                            else min_samples)
        self.samples = []          # (foot_y, height)
        self._fit = None           # (a, b)  ->  h ~= a*foot_y + b
        self._n_at_fit = 0

    def add(self, foot_y, height):
        if height > 0:
            self.samples.append((float(foot_y), float(height)))

    @property
    def ready(self):
        return self._refit() is not None

    def _refit(self):
        n = len(self.samples)
        if n < self.min_samples:
            return None
        # refit every 25% growth — cheap, and the fit stabilises fast
        if self._fit is not None and n < self._n_at_fit * 1.25:
            return self._fit
        ys = [s[0] for s in self.samples]
        lo, hi = min(ys), max(ys)
        if hi - lo < 1e-6:
            return None
        buckets = [[] for _ in range(self.n_bins)]
        for y, h in self.samples:
            k = min(self.n_bins - 1, int((y - lo) / (hi - lo) * self.n_bins))
            buckets[k].append(h)
        pts = []
        for k, hs in enumerate(buckets):
            if len(hs) < 5:        # a bin with almost nothing in it is noise
                continue
            hs.sort()
            y_mid = lo + (k + 0.5) * (hi - lo) / self.n_bins
            pts.append((y_mid, hs[len(hs) // 2]))
        if len(pts) < 3:
            return None
        mx = sum(p[0] for p in pts) / len(pts)
        my = sum(p[1] for p in pts) / len(pts)
        den = sum((p[0] - mx) ** 2 for p in pts)
        if den <= 1e-9:
            return None
        a = sum((p[0] - mx) * (p[1] - my) for p in pts) / den
        self._fit = (a, my - a * mx)
        self._n_at_fit = n
        return self._fit

    def expected_h(self, foot_y):
        f = self._refit()
        if f is None:
            return None
        h = f[0] * float(foot_y) + f[1]
        return h if h > 1.0 else None

    def describe(self):
        f = self._refit()
        if f is None:
            return (f"scene geometry NOT fitted ({len(self.samples)}/"
                    f"{self.min_samples} samples) — carried-suppression stays "
                    f"OFF, nothing is being deleted")
        return (f"scene geometry fitted from {len(self.samples)} isolated "
                f"detections: expected height = {f[0]:.3f}*foot_y + {f[1]:.0f}px")


def _feed_perspective(dets, model, min_conf=0.5):
    """Only ISOLATED, confident boxes teach the model — an overlapping pair may
    be one merged double-box, which would teach it the wrong scale."""
    if model is None or len(dets) == 0:
        return
    xy = dets.xyxy
    conf = dets.confidence
    for i in range(len(xy)):
        if conf is not None and float(conf[i]) < min_conf:
            continue
        if any(_boxes_occluding(xy[i], xy[j]) for j in range(len(xy)) if j != i):
            continue
        model.add(xy[i][3], xy[i][3] - xy[i][1])


def _suppress_carried(dets, persp=None, stats=None):
    """Drop a detection that is a CARRIED person (baby in arms), and only that.

    THREE independent conditions must all hold, because each one alone deletes
    real guests on an oblique camera:

      1. containment   >=CARRIED_CONTAIN inside a bigger box, <=CARRIED_MAX_AREA_RATIO
         of its area. This alone was the old rule, and it also describes a guest
         standing further back — verified deleting real people.
      2. off the floor  height < CARRIED_HEIGHT_TOL x the height the fitted scene
         geometry predicts for a person standing at that footline. A guest
         further back matches the prediction and survives.
      3. head is low    the box top is >=CARRIED_MIN_HEAD_DROP down the carrier's
         box. Condition 2 alone still deleted a guest whose legs were hidden by
         the door frame (their box bottom is the OCCLUDER's edge, not their
         feet, so they look "off the floor" too). But their HEAD is at full
         height, level with or above the person in front — a carried child's
         head is down at the carrier's chest. This is what separates them.

    With no fitted geometry, nothing is suppressed at all. Deleting a real guest
    is far more expensive than briefly double-counting a baby.

    stats: optional dict, incremented so suppression is never silent.
    """
    if not ENABLE_CARRIED_SUPPRESS or len(dets) < 2:
        return dets
    if persp is None or not persp.ready:
        return dets
    xy = dets.xyxy
    area = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1])
    keep = np.ones(len(dets), dtype=bool)
    for i in range(len(dets)):
        h_i = xy[i, 3] - xy[i, 1]
        exp_h = persp.expected_h(xy[i, 3])
        if exp_h is None or h_i >= CARRIED_HEIGHT_TOL * exp_h:
            continue        # (2) consistent with standing on the floor
        for j in range(len(dets)):
            if i == j or area[j] <= 0 or area[i] <= 0:
                continue
            h_j = xy[j, 3] - xy[j, 1]
            if h_j <= 0:
                continue
            head_drop = (xy[i, 1] - xy[j, 1]) / h_j
            if head_drop < CARRIED_MIN_HEAD_DROP:
                continue    # (3) head is up at full height -> occluded guest
            ix1, iy1 = max(xy[i, 0], xy[j, 0]), max(xy[i, 1], xy[j, 1])
            ix2, iy2 = min(xy[i, 2], xy[j, 2]), min(xy[i, 3], xy[j, 3])
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            if (inter / area[i] >= CARRIED_CONTAIN            # (1)
                    and area[i] <= CARRIED_MAX_AREA_RATIO * area[j]):
                keep[i] = False
                if stats is not None:
                    stats["carried_suppressed"] = stats.get("carried_suppressed", 0) + 1
                break
    return dets[keep]


def _drop_implausible(dets, persp, stats=None):
    """D1: remove boxes too large to be a person standing at their own footline.

    Uses the scene-geometry fit already maintained for carried-person
    suppression, so there is no new model and no new calibration. With no fit
    yet, nothing is dropped — a filter that fires on missing information is
    worse than no filter.
    """
    if not ENABLE_SIZE_FILTER or len(dets) == 0 or persp is None or not persp.ready:
        return dets
    # A count with no denominator cannot be judged. "D1 dropped 29,124" reads
    # alarming or fine depending entirely on whether that is 2% or 60% of the
    # detections, and the first real run printed it without the total.
    if stats is not None:
        stats["size_seen"] = stats.get("size_seen", 0) + len(dets)
    # V62a: self-correcting tolerance. The first real run dropped 26.8% of ALL
    # detections on an unstable ground fit (implied camera height flapping
    # 1.3m<->3.7m). A quarter of detections being non-people is not a plausible
    # scene; a bad fit is. Past D1_MAX_DROP_FRAC the tolerance doubles, once,
    # loudly — and the drop rate keeps being tracked against the new bar.
    _tol = SIZE_FILTER_TOL
    if stats is not None:
        _seen = stats.get("size_seen", 0)
        _drop = stats.get("size_dropped", 0)
        if stats.get("d1_relaxed"):
            _tol = SIZE_FILTER_TOL * 2.0
        elif (_seen > globals().get("D1_GUARD_WARMUP", 2000)
                and _drop / max(_seen, 1) > globals().get("D1_MAX_DROP_FRAC", 0.12)):
            stats["d1_relaxed"] = True
            _tol = SIZE_FILTER_TOL * 2.0
            print(f"📏 D1 GUARD: drop rate {_drop / max(_seen, 1):.0%} exceeds "
                  f"{globals().get('D1_MAX_DROP_FRAC', 0.12):.0%} — the geometry "
                  f"fit is not trustworthy; tolerance relaxed "
                  f"{SIZE_FILTER_TOL}x -> {_tol}x for the rest of this video. "
                  f"Supply ground_points in the venue profile for the real fix.")
    mask = implausible_size_mask(dets.xyxy, persp.expected_h, tol=_tol)
    if not any(mask):
        return dets
    if stats is not None:
        stats["size_dropped"] = stats.get("size_dropped", 0) + sum(mask)
    return dets[np.array([not m for m in mask], dtype=bool)]


def _detector_has_head_class(model):
    """TRUE only for a real 2-class person/head detector. Hard gate: on stock
    COCO weights class 1 is 'bicycle', and treating that as a head would invent
    people out of parked bikes."""
    try:
        names = {int(k): str(v).lower() for k, v in model.names.items()}
    except Exception:
        return False
    return names.get(0) == "person" and names.get(1) == "head" and len(names) == 2


def _split_person_head(dets, has_head):
    """-> (person_dets, head_dets). Heads are an OCCLUSION SIGNAL: a head with
    no person box around it is a body the detector lost behind someone else."""
    if len(dets) == 0 or dets.class_id is None:
        return dets, dets[np.zeros(len(dets), dtype=bool)]
    persons = dets[dets.class_id == 0]
    heads = dets[dets.class_id == 1] if has_head else dets[np.zeros(len(dets), dtype=bool)]
    return persons, heads


def _heads_without_person(heads, persons):
    """Heads whose centre falls in no person box — each one is a missed person."""
    if len(heads) == 0:
        return heads
    if len(persons) == 0:
        return heads
    px = persons.xyxy
    keep = []
    for hx1, hy1, hx2, hy2 in heads.xyxy:
        cx, cy = (hx1 + hx2) / 2.0, (hy1 + hy2) / 2.0
        keep.append(not any(px[j, 0] <= cx <= px[j, 2] and px[j, 1] <= cy <= px[j, 3]
                            for j in range(len(px))))
    return heads[np.array(keep, dtype=bool)]


def _frame_chroma(frame_bgr, small=None):
    """Mean |R-G| + |G-B| — how far apart the colour channels are.

    Brightness-independent on purpose: HSV saturation explodes on dark pixels
    because it divides by brightness, so it calls a dark COLOUR frame infrared.
    True greyscale stays near zero however dark the image is. This is the same
    test probe_environment() already uses; F1 just runs it every frame."""
    s = small if small is not None else cv2.resize(frame_bgr, (160, 90))
    b = s[:, :, 0].astype(np.int16)
    g = s[:, :, 1].astype(np.int16)
    r = s[:, :, 2].astype(np.int16)
    return float((np.abs(r - g) + np.abs(g - b)).mean())


def detection_sanity(video_path, label, n=3, device=None):
    """Visual checkpoint BEFORE the long run: raw detections on sample frames."""
    if not Path(video_path).exists():
        print(f"(skip sanity check — {video_path} missing)")
        return
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or NATIVE_FPS_OVERRIDE or 25
    device_str = str(device or globals().get('DEVICE', 'cuda'))
    model = YOLO(DETECTOR_MODEL)
    shots, n_found = [], 0
    for frac in [0.1, 0.5, 0.85][:n]:
        idx = max(0, int(total * frac))
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok:
            continue
        dets = sv.Detections.from_ultralytics(
            model(fr, conf=CONF_THRESHOLD, iou=0.45, imgsz=YOLO_IMGSZ,
                  verbose=False, device=device_str)[0])
        dets = dets[dets.class_id == 0]
        fr = sv.BoxAnnotator(thickness=2).annotate(fr, dets)
        fr = cv2.resize(fr, (720, int(720 * fr.shape[0] / fr.shape[1])))
        shots.append((idx / fps, cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
        n_found = len(dets)
    cap.release()
    if not shots:
        print(f"❌ {label}: no frame could be read — re-download the file.")
        return
    show_gallery(shots, f"STEP CHECK · person detections · {label} "
                        f"({n_found} people in last sample)", ncols=3)

def _hex2bgr(h):
    h = h.lstrip("#")
    return (int(h[4:6], 16), int(h[2:4], 16), int(h[0:2], 16))

def _dim(bgr, f=0.55):
    """PHASE11_RENDER_LEGIBILITY: push a colour back so it reads as map, not
    as a claim about a person."""
    return tuple(int(max(0, min(255, c * f))) for c in bgr)


def _dashed_poly(frame, poly, colour, dash=11, gap=8, thick=1):
    """A dashed outline. Zones and person boxes were both solid rectangles, so
    a zone covering half the frame read as a giant detection. Dashed vs solid
    is a difference the eye resolves instantly, at any size, in greyscale IR
    footage too — unlike hue, which IR night frames wash out."""
    pts = np.asarray(poly, dtype=np.int32).reshape(-1, 2)
    n = len(pts)
    for i in range(n):
        p, q = pts[i], pts[(i + 1) % n]
        seg = float(np.hypot(q[0] - p[0], q[1] - p[1]))
        if seg < 1:
            continue
        steps = max(1, int(seg // (dash + gap)))
        for k in range(steps + 1):
            a = min(1.0, (k * (dash + gap)) / seg)
            b = min(1.0, (k * (dash + gap) + dash) / seg)
            if a >= 1.0:
                break
            cv2.line(frame,
                     (int(p[0] + (q[0] - p[0]) * a), int(p[1] + (q[1] - p[1]) * a)),
                     (int(p[0] + (q[0] - p[0]) * b), int(p[1] + (q[1] - p[1]) * b)),
                     colour, thick, cv2.LINE_AA)


def _in_poly(poly, pt):
    return cv2.pointPolygonTest(poly.astype(np.float32), pt, False) >= 0

def render_annotated(video_path, out_path, frame_log, canon, roles, events,
                     crossings, polygons, zcolors, entry_line, eff_fps, step,
                     native_fps, zone_roles=None, proxy_dir=None, duration_s=None,
                     phantoms=None):
    """PASS 2: draw everything from FINAL identities. Returns snapshots."""
    # role-based zone lookup — NOT hardcoded zone names, so this works on any
    # venue (cafe wait_zone, store queue_zone, restaurant waiting, ...).
    zone_roles = zone_roles or classify_zones(list(polygons.keys()))
    WAIT_ZONES = {z for z, rs in zone_roles.items() if "wait" in rs}
    STAFF_ANCHOR_ZONES = {z for z, rs in zone_roles.items() if "staff" in rs}
    wait_ivs = defaultdict(list)          # canonical -> waiting intervals
    for e in events:
        if e["zone"] in WAIT_ZONES:
            wait_ivs[e["track_id"]].append((e["t_in"], e["t_out"]))
    for ivs in wait_ivs.values():
        ivs.sort()
    rec_staff_ivs = sorted((e["t_in"], e["t_out"]) for e in events
                           if e["zone"] in STAFF_ANCHOR_ZONES and e["role"] == "staff")
    ins, outs = {}, {}                    # canonical -> first crossing t
    for c in crossings:
        cid = c["track_id"]
        if roles.get(cid) == "staff":
            continue
        d = ins if c["direction"] == "in" else outs
        d.setdefault(cid, c["t"])
    in_times = sorted(ins.values())
    out_times = sorted(outs.values())
    # v48: P1..PN display numbers by first appearance (staff keep their names)
    _pnum = {}
    if ENABLE_DISPLAY_RENUMBER:
        for _fi, _t, _boxes in frame_log:
            for _tid, *_ in _boxes:
                _c = canon.get(_tid, _tid)
                if _c not in _pnum:
                    _pnum[_c] = len(_pnum) + 1
    role_bgr = {"customer": _hex2bgr(ROLE_HEXES["customer"]),
                "staff": _hex2bgr(ROLE_HEXES["staff"]),
                "unknown": (150, 150, 150)}
    zone_bgr = {n: _hex2bgr(c) for n, c in zcolors.items()}
    trails = defaultdict(lambda: deque(maxlen=int(eff_fps * 2)))
    _trail_gap = {}

    def _claim_label_spot(placed, x, y, w, h, step=22, tries=8):
        """v53: collision-free labels — if this spot overlaps an already-drawn
        label, slide down until free, so every id stays readable."""
        for _ in range(tries):
            r = (x, y, x + w, y + h)
            if all(r[2] <= q[0] or r[0] >= q[2] or r[3] <= q[1] or r[1] >= q[3]
                   for q in placed):
                placed.append(r)
                return int(y)
            y += step
        placed.append((x, y, x + w, y + h))
        return int(y)

    def wait_clock(cid, t):
        total, inside = 0.0, False
        for s, e2 in wait_ivs.get(cid, ()):
            if t >= s:
                total += min(t, e2) - s
                if s <= t <= e2 + 0.6:
                    inside = True
        return total if inside else None

    cap = None if proxy_dir is not None else cv2.VideoCapture(str(video_path))
    writer = None
    _ff_pipe = None   # V68: init ONCE — per-frame reset respawned ffmpeg every frame
    snapshots, next_snap = [], 0.0
    render_index = []   # v55: night-time of every frame written
    frames = {idx: list(boxes) for idx, t, boxes in frame_log}
    times = {idx: t for idx, t, boxes in frame_log}

    # A6: coast a briefly-missed box (<= RENDER_COAST_S) so a person whose
    # detection blinks for a frame or two keeps a box ON SCREEN. Conf
    # hysteresis keeps the TRACK alive but nothing drew a box on the missed
    # frames, and HUD smoothing hid the blink in the counter while the boxes
    # still flashed. Display-only: `frames` holds copies, frame_log untouched.
    _coast_max = max(1, int(round(eff_fps * globals().get("RENDER_COAST_S", 0.5))))
    _cord = sorted(frames)
    _last_seen = {}
    _n_coasted = 0
    for _n, _idx in enumerate(_cord):
        for _b in list(frames[_idx]):
            _tid = _b[0]
            if _tid in _last_seen:
                _pn, _pb = _last_seen[_tid]
                _g = _n - _pn - 1
                if 1 <= _g <= _coast_max:
                    for _m in range(1, _g + 1):
                        _f = _m / (_g + 1.0)
                        _ib = tuple(int(round(_pb[c] + (_b[c] - _pb[c]) * _f))
                                    for c in range(1, 5))
                        frames[_cord[_pn + _m]].append((_tid,) + _ib)
                        _n_coasted += 1
            _last_seen[_tid] = (_n, _b)
    if _n_coasted:
        print(f"render coasting: filled {_n_coasted} blinked box-frame(s) "
              f"(gap <= {_coast_max} analysed frames)")

    # displayed-count smoothing: rolling median over HUD_SMOOTH_S so a person
    # whose detection blinks for a few frames doesn't wobble the dashboard
    ordered = sorted(frames)
    half = max(1, int(round(HUD_SMOOTH_S * eff_fps / 2)))
    raw_counts, raw_zone = [], []
    for idx in ordered:
        zc = defaultdict(int)
        for tid, x1, y1, x2, y2 in frames[idx]:
            bc = ((x1 + x2) / 2.0, float(y2))
            ctr = ((x1 + x2) / 2.0, (y1 + y2) / 2.0)
            for name, poly in polygons.items():
                if _in_poly(poly, ctr if name in STAFF_ANCHOR_ZONES else bc):
                    zc[name] += 1
        raw_counts.append(len(frames[idx]))
        raw_zone.append(zc)

    def _med(vals):
        s = sorted(vals)
        return s[len(s) // 2]

    smooth_total, smooth_zone = {}, {}
    for k, idx in enumerate(ordered):
        lo, hi = max(0, k - half), min(len(ordered), k + half + 1)
        smooth_total[idx] = _med(raw_counts[lo:hi])
        smooth_zone[idx] = {name: _med([raw_zone[j].get(name, 0)
                                        for j in range(lo, hi)])
                            for name in polygons}
    max_idx = max(frames) if frames else -1
    _span_s = float(duration_s or ((max_idx + 1) / max(eff_fps, 1e-6)))
    if proxy_dir is not None:
        _idxs = sorted(int(p.stem) for p in Path(proxy_dir).glob("*.jpg"))
        print(f"   rendering {len(_idxs)} frames that actually had someone in them "
              f"(of {len(frames)} analysed)")
    else:
        _idxs = list(range(max_idx + 1))
    _prev_rendered_t = None
    _native_pos = 0   # cv2 fallback: next native frame index the cap will read
    try:
        for frame_idx in tqdm(_idxs, desc="render"):
            if proxy_dir is not None:
                frame = cv2.imread(str(Path(proxy_dir) / f"{frame_idx:07d}.jpg"))
                if frame is None:
                    continue
            else:
                # cv2 fallback: analysed index k lives at NATIVE frame k*step.
                # Sequential cap.read() drew native frames 0,1,2,... — i.e.
                # analysed frame k's boxes were drawn on the frame at t=k/30
                # instead of t=k*step/30 (4x temporal skew), at 4K while boxes
                # are in analysis coordinates. Skip-ahead and downscale.
                _tgt = frame_idx * step
                while _native_pos < _tgt:
                    if not cap.grab():
                        break
                    _native_pos += 1
                ok, frame = cap.read()
                _native_pos += 1
                if not ok:
                    break
                _amw = int(globals().get("ANALYSIS_MAX_W", 1280))
                if frame.shape[1] > _amw:
                    frame = cv2.resize(frame, (_amw,
                                       int(frame.shape[0] * _amw / frame.shape[1])))
            if frame_idx not in frames:
                continue
            t = times[frame_idx]
            if writer is None and _ff_pipe is None:
                h, w = frame.shape[:2]
                # v53: +62px letterbox band on top for the HUD, so the HUD never
                # covers people or labels inside the actual video area
                # v54: the output plays at PLAYBACK_FPS while the analysis ran
                # at eff_fps. Same frames, same numbers — only the playback clock
                # changes, so 10 h of footage becomes ~60-80 min to review.
                # Falls back to eff_fps when Cell 2e hasn't run (short clips).
                _play_fps = float(globals().get("PLAYBACK_FPS", eff_fps) or eff_fps)
                if globals().get("RENDER_DIRECT_H264") and _ffmpeg_ok():
                    # V64a: straight to h264 — the ~1.5GB raw mp4v intermediate
                    # never exists and the runner's re-encode pass is skipped.
                    import subprocess as _sp
                    if not Path(out_path).stem.endswith("_h264"):
                        out_path = Path(out_path).with_name(
                            Path(out_path).stem + "_h264.mp4")
                    _ff_pipe = _sp.Popen(
                        ["ffmpeg", "-y", "-loglevel", "error",
                         "-f", "rawvideo", "-pix_fmt", "bgr24",
                         "-s", f"{w}x{h + 88}", "-r", f"{_play_fps:.3f}",
                         "-i", "-",
                         "-c:v", "libx264", "-preset", "veryfast",
                         "-pix_fmt", "yuv420p",
                         "-crf", str(globals().get("RENDER_CRF", 28)),
                         "-movflags", "+faststart",   # browser can stream it
                         str(out_path)], stdin=_sp.PIPE,
                        stderr=_sp.PIPE)   # V73: never lose ffmpeg's reason
                    writer = None
                    print(f"   render -> h264 directly (no raw intermediate)")
                else:
                    writer = cv2.VideoWriter(str(out_path),
                                             cv2.VideoWriter_fourcc(*"mp4v"),
                                             _play_fps, (w, h + 88))
                    if not writer.isOpened():
                        # a missing codec makes writer.write() a silent no-op
                        # for 27k frames and leaves a 0-byte file behind
                        raise RuntimeError(
                            f"VideoWriter failed to open {out_path} "
                            f"(mp4v codec missing?)")
                if _play_fps > eff_fps:
                    print(f"   output plays {_play_fps/eff_fps:.1f}x faster than "
                          f"real time ({_play_fps:.0f} fps out, {eff_fps:.1f} analysed)")
            _lbl_placed = []   # v53: label rects drawn this frame (collision check)
            # PHASE11: zones are the MAP. Thin, dashed and dimmed, with a small name
            # tag at the polygon's top corner — never a solid rectangle, because a
            # solid rectangle now means exactly one thing: "this is a person".
            for name, poly in polygons.items():
                _zc = _dim(zone_bgr.get(name, (200,) * 3))
                _dashed_poly(frame, poly, _zc)
                _pt = min(poly.tolist(), key=lambda p: (p[1], p[0]))
                # clamp into frame: a polygon whose top corner sits at the right
                # edge had its name run off the side and vanish, which is exactly
                # the zone you most need named
                (_ztw, _zth), _ = cv2.getTextSize(str(name),
                                                  cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
                _zx = min(max(int(_pt[0]) + 3, 2), max(2, frame.shape[1] - _ztw - 2))
                _zy = min(max(int(_pt[1]) - 4, _zth + 2), frame.shape[0] - 2)
                cv2.putText(frame, str(name), (_zx, _zy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, _zc, 1, cv2.LINE_AA)
            # anything D3 removed, shown struck through: a filter you cannot see is
            # a filter you cannot trust or debug
            for _ph in (phantoms or []):
                _a, _b, _c, _d = [int(v) for v in _ph["box"]]
                cv2.rectangle(frame, (_a, _b), (_c, _d), (90, 90, 90), 1)
                cv2.line(frame, (_a, _b), (_c, _d), (90, 90, 90), 1)
                cv2.line(frame, (_a, _d), (_c, _b), (90, 90, 90), 1)
                cv2.putText(frame, "IGNORED (static phantom)", (_a + 3, _b + 14),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (90, 90, 90), 1, cv2.LINE_AA)
            n_in = bisect.bisect_right(in_times, t)
            n_out = bisect.bisect_right(out_times, t)
            if entry_line:
                (ex1, ey1), (ex2, ey2) = entry_line
                cv2.line(frame, (ex1, ey1), (ex2, ey2), (60, 60, 230), 3)
                mx, my = (ex1 + ex2) // 2, (ey1 + ey2) // 2
                tag = f"IN {n_in} | OUT {n_out}"
                (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                _ty = _claim_label_spot(_lbl_placed, mx - 6, my - th - 12,
                                        tw + 12, th + 12)
                cv2.rectangle(frame, (mx - 6, _ty), (mx + tw + 6, _ty + th + 12),
                              (25, 25, 25), -1)
                cv2.putText(frame, tag, (mx, _ty + th + 4), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (90, 90, 255), 2, cv2.LINE_AA)
            zone_counts = defaultdict(int)
            n_role = {"customer": 0, "staff": 0}
            boxes = frames[frame_idx]
            for tid, x1, y1, x2, y2 in boxes:
                cid = canon.get(tid, tid)
                role = roles.get(cid, "customer")
                n_role[role if role in n_role else "customer"] += 1
                bc = ((x1 + x2) / 2.0, float(y2))
                ctr = ((x1 + x2) / 2.0, (y1 + y2) / 2.0)
                trails[cid].append((int(bc[0]), int(bc[1])))
                for name, poly in polygons.items():
                    anchor = ctr if name in STAFF_ANCHOR_ZONES else bc
                    if _in_poly(poly, anchor):
                        zone_counts[name] += 1
                color = role_bgr.get(role, role_bgr["unknown"])
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                if isinstance(cid, str) and not str(cid).isdigit():
                    lbl = f"{cid} {role}"          # enrolled staff keep their name
                elif ENABLE_DISPLAY_RENUMBER and cid in _pnum:
                    # v53: "customer" is the default and just adds width in a
                    # crowded queue — show it only when the role is notable
                    lbl = (f"P{_pnum[cid]}" if role == "customer"
                           else f"P{_pnum[cid]} {role}")
                else:
                    lbl = f"#{cid} {role}"
                wc = wait_clock(cid, t)
                if wc is not None and role != "staff":
                    lbl += f" {mmss(wc)}"
                (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
                _ly0 = max(y1 - th - 10, 0)
                _ly = _claim_label_spot(_lbl_placed, x1, _ly0, tw + 8, th + 10)
                # PHASE11: collision avoidance can slide a label up to 176px away.
                # Without a leader line it then reads as belonging to whoever it
                # landed on — in a queue that is worse than an overlapping label.
                if _ly - _ly0 > 6:
                    cv2.line(frame, (x1 + 3, _ly + th + 10), (x1 + 3, y1),
                             color, 1, cv2.LINE_AA)
                cv2.rectangle(frame, (x1, _ly), (x1 + tw + 8, _ly + th + 10),
                              color, -1)
                cv2.putText(frame, lbl, (x1 + 4, _ly + th + 2),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2,
                            cv2.LINE_AA)
            present_now = {canon.get(tid, tid) for tid, *_ in boxes}
            for cid in list(trails):                 # prune departed people's
                if cid not in present_now:           # trails after ~1.5s
                    _trail_gap[cid] = _trail_gap.get(cid, 0) + 1
                    if _trail_gap[cid] > int(eff_fps * 1.5):
                        trails.pop(cid, None)
                        _trail_gap.pop(cid, None)
                else:
                    _trail_gap.pop(cid, None)
            if TRAIL_MODE != "off":
                for cid in present_now:
                    pts = trails.get(cid)
                    if not pts or len(pts) < 2:
                        continue
                    if TRAIL_MODE == "moving":
                        dx = pts[-1][0] - pts[0][0]
                        dy = pts[-1][1] - pts[0][1]
                        if dx * dx + dy * dy < 1600:  # <40px net = not walking
                            continue
                    cv2.polylines(frame, [np.array(pts, dtype=np.int32)], False,
                                  (200, 200, 60), 2)
            # v53: zone counts moved OFF the video onto the HUD band (3rd line) —
            # nothing is ever drawn over people any more; polygons stay outlined
            staffed = any(s <= t <= e2 for s, e2 in rec_staff_ivs)
            hud = (f"t={mmss(t)}  people in frame: {smooth_total[frame_idx]}"
                   f" ({n_role['staff']} staff)  "
                   f"entered={n_in} exited={n_out} (unique)")
            wait_now = sum(smooth_zone[frame_idx].get(z, 0) for z in WAIT_ZONES)
            hud2 = f"waiting now: {wait_now}"
            if _prev_rendered_t is not None and (t - _prev_rendered_t) > 30:
                hud2 += f"   [skipped {mmss(t - _prev_rendered_t)} — nobody present]"
            _prev_rendered_t = t
            if STAFF_ANCHOR_ZONES & set(polygons):
                _label = sorted(STAFF_ANCHOR_ZONES & set(polygons))[0]
                hud2 += f"   {_label}: {'STAFFED' if staffed else 'AWAY'}"
            # A7: the build that made this video, burned in — we could not tell
            # which build a reviewed video came from, so fixes looked like no-ops
            hud2 += f"   build {str(globals().get('_BUILD_ID', '?'))[:12]}"
            # v53: HUD lives on a band ADDED ABOVE the frame (letterbox), not
            # painted over the video — the full camera view stays visible
            frame = cv2.copyMakeBorder(frame, 88, 0, 0, 0,
                                       cv2.BORDER_CONSTANT, value=(25, 25, 25))
            cv2.putText(frame, hud, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.62,
                        (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(frame, hud2, (10, 51), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                        (170, 220, 170), 1, cv2.LINE_AA)
            # PHASE11: a mask zone exists to DELETE detections. Reporting "how many
            # people are in it" is meaningless and invites the reader to treat a
            # suppression region as a place guests stand.
            _mask_z = {z for z, rs in (zone_roles or {}).items() if "mask" in rs}
            hud3 = "   ".join(f"{n}: {smooth_zone[frame_idx].get(n, 0)}"
                              for n in polygons if n not in _mask_z)
            cv2.putText(frame, hud3, (10, 78), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                        (180, 200, 255), 1, cv2.LINE_AA)
            # v54: real wall clock + where we are in the true timeline. Without
            # these a fast-forwarded video is actively misleading — every gap
            # looks continuous and an abandoned desk reads as two seconds.
            _clock = wall(t) if globals().get("VIDEO_START_CLOCK") else mmss(t)
            _tz = globals().get("DRIVE_TZ", "")
            cv2.putText(frame, f"{_clock} {_tz}", (frame.shape[1] - 240, 24),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 235, 190), 2, cv2.LINE_AA)
            _span = max(_span_s, 1e-6)
            _fill = int((frame.shape[1] - 20) * min(1.0, t / _span))
            cv2.rectangle(frame, (10, 84), (frame.shape[1] - 10, 87), (70, 70, 70), -1)
            cv2.rectangle(frame, (10, 84), (10 + _fill, 87), (255, 190, 90), -1)
            cv2.putText(frame, f"{mmss(t)} / {mmss(_span)} real",
                        (frame.shape[1] - 240, 51), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (170, 170, 170), 1, cv2.LINE_AA)
            if _ff_pipe is not None:
                try:
                    _ff_pipe.stdin.write(frame.tobytes())
                except (BrokenPipeError, OSError) as _pe:
                    print(f"   !! h264 pipe died ({_pe}) — video stops here; "
                          f"analysis results are unaffected")
                    _ff_pipe = None
                    # stop the render entirely: looping on would keep growing
                    # render_index past the last written frame, and every
                    # moment clip cut from the .index.json would then land on
                    # the wrong seconds.
                    break
            elif writer is not None:
                writer.write(frame)
            render_index.append(round(float(t), 3))
            if t >= next_snap:
                small = cv2.resize(frame, (720, int(720 * frame.shape[0]
                                                    / frame.shape[1])))
                snapshots.append((t, cv2.cvtColor(small, cv2.COLOR_BGR2RGB)))
                next_snap += SNAPSHOT_EVERY_S
    finally:
        # finalize even on a crash: an unfinalized mp4 (no moov atom) is
        # unplayable. The pipe close was previously DEAD CODE nested under
        # `if writer:` — writer and _ff_pipe are mutually exclusive, so the
        # h264 file was only ever finalized by garbage collection.
        if cap is not None:
            cap.release()
        if _ff_pipe is not None:
            try:
                _ff_pipe.stdin.close()
                _ff_pipe.wait(timeout=600)
            except Exception as _fe:
                print(f"   !! ffmpeg finalize: {_fe}")
        if writer is not None:
            writer.release()
    try:
        # absolute night time, always: the consumer (moment clips) works on the
        # night's clock, and a chunk-local index silently mis-cuts every clip.
        _shift = float(globals().get("_RENDER_INDEX_SHIFT", 0.0))
        Path(str(out_path)).with_suffix(".index.json").write_text(
            json.dumps([round(v + _shift, 3) for v in render_index]))
    except Exception as _ix:
        print(f"(render index not written: {_ix})")
    return snapshots

def probe_environment(video_path, n_probes=10, start_s=0.0, span_s=None):
    """Is THIS chunk infrared? (Cell 2e answers it for chunk 1 only, and a
    night shift crosses dusk halfway through.) Colour-based identity signals
    are meaningless on a greyscale image, so the attire tier has to be
    decided per chunk, not per run."""
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    _fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    lo = int(start_s * _fps)
    hi = min(n, int((start_s + span_s) * _fps)) if span_s else n
    sats, chromas = [], []
    for k in range(n_probes):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(lo + (hi - lo) * k / n_probes))
        ok, fr = cap.read()
        if not ok:
            continue
        sats.append(float(cv2.cvtColor(fr, cv2.COLOR_BGR2HSV)[:, :, 1].mean()))
        # brightness-independent: how far apart the colour channels are. True
        # greyscale stays ~0-3 however dark the image is, where saturation
        # explodes on dark pixels because it divides by brightness.
        b, g, r = (fr[:, :, i].astype(np.int16) for i in range(3))
        chromas.append(float((np.abs(r - g) + np.abs(g - b)).mean()))
    cap.release()
    if not sats:
        return {"is_ir": False, "switches": False, "sat_min": None,
                "sat_max": None, "chroma_min": None, "chroma_max": None}
    thr = float(globals().get("IR_SAT_THRESHOLD", 12.0))
    cthr = float(globals().get("IR_CHROMA_THRESHOLD", 6.0))
    is_ir = (max(sats) < thr) or (max(chromas) < cthr)
    switches = (not is_ir) and (max(sats) > thr > min(sats)
                                or max(chromas) > cthr > min(chromas))
    return {"is_ir": is_ir, "switches": switches,
            "sat_min": min(sats), "sat_max": max(sats),
            "chroma_min": min(chromas), "chroma_max": max(chromas)}


def process_video(camera_id, video_path, zones_path, max_seconds=None, device=None,
                  chunk_tag="", start_seconds=0.0):
    # Open the video FIRST so we know its real frame size before loading
    # zones — zone JSON files are often drawn against a different reference
    # resolution (a screenshot, a different export of the same footage),
    # so load_zone_config auto-scales polygon coordinates to whatever this
    # video's actual frame_w x frame_h turns out to be.
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"cannot open {video_path}")
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    if not native_fps or native_fps != native_fps or native_fps < 1:
        native_fps = NATIVE_FPS_OVERRIDE or 25
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, round(native_fps / FPS_TARGET))
    frame_step_s = step / native_fps
    eff_fps = 1.0 / frame_step_s

    # v55 CRITICAL: frames are analysed at ANALYSIS width (<=1920), not at the
    # camera's native 4K. Zones must be scaled to the frame the detector
    # actually sees, or every polygon sits at twice the coordinates of every
    # box and no zone ever triggers correctly. Must match frame_source()'s
    # rounding exactly.
    ANALYSIS_MAX_W = int(globals().get("ANALYSIS_MAX_W", 1920))
    an_w = min(ANALYSIS_MAX_W, frame_w) // 2 * 2
    an_h = int(round(frame_h * an_w / max(frame_w, 1))) // 2 * 2
    if (an_w, an_h) != (frame_w, frame_h):
        print(f"\U0001f4d0 analysing at {an_w}x{an_h} (source {frame_w}x{frame_h}) — "
              f"zones scaled to the analysed frame, not the source")
    _src_w, _src_h = frame_w, frame_h   # V65: native dims, for face re-crop
    frame_w, frame_h = an_w, an_h
    polygons, entry_lines = load_zone_config(zones_path, frame_size=(frame_w, frame_h))
    zcolors = zone_color_map(polygons)
    # zone ROLES (wait / staff / seating / entry / service / other) are
    # derived from each zone's NAME via keyword matching (see classify_zones
    # in the ZONES cell) — nothing here is tied to "restaurant" vocabulary,
    # so the exact same process_video() runs on a cafe, a store, or anything
    # else whose zones_<stem>.json uses recognizable zone names.
    zone_roles = classify_zones(list(polygons.keys()))
    staff_zones_here = {z for z, rs in zone_roles.items() if "staff" in rs}
    _mask_names = {z for z, rs in zone_roles.items() if "mask" in rs}
    if _mask_names:
        print(f"DEAD AREAS active (reflection/poster suppression): "
              f"{sorted(_mask_names)}")

    def _drop_masked(dets):
        """v53: detections whose feet land inside a mask/ignore zone are
        phantoms (mirror, glass reflection, poster, TV) - dropped before
        they can mint ids or pollute counts."""
        if not _mask_names or len(dets) == 0:
            return dets
        _keep = []
        for (_mx1, _my1, _mx2, _my2) in dets.xyxy:
            _bc = ((_mx1 + _mx2) / 2.0, _my2)
            _keep.append(not any(_in_poly(polygons[_n], _bc)
                                 for _n in _mask_names))
        return dets[np.array(_keep, dtype=bool)]

    # ── ZONE-COMPLETENESS GUARD ─────────────────────────────────────────────
    # Silent failure mode discovered 2026-07-12: cafe_004 zones file only had
    # "customers"/"staff" polygons and no entry_line — Q1 (people entered)
    # came back as raw in=None/out=None (line_zone never built at all, not a
    # detection failure), and Q2/Q3/Q5/Q6/Q7 were silently ABSENT from
    # answers.json (each gated behind "if seating_zones:" / "if wait_zones:"
    # etc., which just skip quietly when nothing matches that role). Nothing
    # anywhere printed a warning — the run looked clean (self-audit green)
    # while 5 of 7 business questions were never computed. This guard makes
    # that loud instead of silent, at the one place both facts are known:
    # right after roles are classified and before any answer is computed.
    _present_roles = {r for rs in zone_roles.values() for r in rs}
    _missing_roles = [r for r in ("wait", "seating") if r not in _present_roles]
    if not entry_lines:
        _missing_roles = ["entry_line (no line-crossing at all)"] + _missing_roles
    if _missing_roles:
        print(f"🚨 ZONE GAP · {camera_id}: missing {_missing_roles} — the questions "
              f"depending on these will be silently ABSENT from answers.json, not "
              f"zero. Draw the missing zone(s)/entry_line in this venue's "
              f"zones_<stem>.json before treating any answer here as complete.")
    else:
        print(f"✅ {camera_id}: entry_line + entry/wait/seating zones all present")

    device_str = str(device or globals().get('DEVICE', 'cuda'))
    model = YOLO(DETECTOR_MODEL)
    _dk = ("fine-tuned dense-scene"
           if ("crowdhuman" in Path(DETECTOR_MODEL).name.lower()
               or Path(DETECTOR_MODEL).name.lower() == "best.pt") else "stock")
    print(f"ℹ️  {_dk} detector (person only) — staff vs customer comes "
          f"entirely from the staff-zone override (this video's own zones file)")

    det_conf = DETECT_CONF_FLOOR if ENABLE_CONF_HYSTERESIS else CONF_THRESHOLD
    if probe_environment(video_path, start_s=start_seconds,
                         span_s=max_seconds)["is_ir"]:
        # IR/greyscale: every body scores lower than it would in colour, so the
        # same floor silently drops people at night. CLAHE (below) restores
        # contrast; this restores recall.
        det_conf = min(det_conf, IR_DETECT_CONF_FLOOR)

    use_online_tracker = TRACKER_MODE in ("botsort-reid", "boosttrack",
                                          "occluboost")   # F6b: without
    # this the ablation variant built OccluBoost and then fell through to
    # BYTETRACK — the A/B would have measured nothing.
    online_tracker = None
    tracker = None

    def _fresh_tracker():
        """v55: a tracker counts FRAMES. After the motion gate skips a long
        empty stretch, its idea of 'recently lost' is minutes stale, and it
        will happily reattach an abandoned id to whoever walks in next. Start
        it clean instead — bridging a gap that long is the offline stitcher's
        job, and it has appearance evidence the live tracker does not."""
        if use_online_tracker:
            return build_online_tracker(eff_fps, device=device_str)
        return sv.ByteTrack(
            track_activation_threshold=max(0.1, CONF_THRESHOLD - 0.1),
            lost_track_buffer=int(30 * LOST_TRACK_BUFFER_S),
            minimum_matching_threshold=0.8,
            frame_rate=int(round(eff_fps)),
            minimum_consecutive_frames=2,
        )

    if use_online_tracker:
        online_tracker = build_online_tracker(eff_fps, device=device_str)
        if online_tracker is None:
            # fallback: previous path (Ultralytics-wrapped, detector-feature
            # proxy for with_reid — kept working, just weaker on overlap)
            tracker_yaml = _botsort_yaml(eff_fps * LOST_TRACK_BUFFER_S)
            tracker = None
            print(f"tracker: BotSORT + inline ReID ({tracker_yaml}, model=auto — "
                  f"detector-feature proxy; real CLIP-ReID/OSNet runs in the "
                  f"offline stitcher pass only)")
    else:
        tracker = _fresh_tracker()

    # ── v45 RUN DIAGNOSTIC: make the LIVE-association appearance source
    # (real ReID model vs weak auto-proxy) unmissable in every Kaggle log ─────
    if use_online_tracker and online_tracker is not None:
        _appear = f"REAL {(_REID_STATE.get('method') or '?').upper()} embeddings (online tracker)"
    elif use_online_tracker:
        _appear = "WEAK detector-feature proxy (with_reid=auto) — real ReID only runs OFFLINE!"
    else:
        _appear = "ByteTrack (motion only, no appearance)"
    _sc = ((frame_w**2 + frame_h**2) ** 0.5) / REF_DIAGONAL_PX
    print("+" + "-" * 68)
    print(f"| RUN DIAGNOSTIC · {camera_id}")
    print(f"|   detector        : {DETECTOR_MODEL} @ imgsz {YOLO_IMGSZ}")
    print(f"|   tracker          : {TRACKER_MODE}")
    print(f"|   live appearance  : {_appear}")
    print(f"|   sample fps       : {FPS_TARGET} (native {native_fps:.1f}, step {step})")
    print(f"|   conf hysteresis  : {'ON' if ENABLE_CONF_HYSTERESIS else 'off'} "
          f"(detect>={DETECT_CONF_FLOOR}, new>={NEW_TRACK_CONF}, keep>={KEEP_TRACK_CONF})")
    print(f"|   occlusion guard  : {'ON' if ENABLE_OCCLUSION_GUARD else 'off'}  "
          f"co-visibility: {'ON' if ENABLE_COVISIBILITY_BLOCK else 'off'}")
    print(f"|   GMC              : {GMC_METHOD if ENABLE_GMC else 'none'}")
    print(f"|   res-scaling      : {'ON' if ENABLE_RESOLUTION_SCALING else 'off'} "
          f"(frame {frame_w}x{frame_h}, scale {_sc:.2f}x)")
    print("+" + "-" * 68)

    # zone anchors: feet everywhere EXCEPT desk zones (desk-clipped people's
    # feet land outside the polygon — run-2 QA)
    zones = {}
    for name, poly in polygons.items():
        if name in _mask_names:
            continue
        anchor = (sv.Position.CENTER if name in staff_zones_here
                  else sv.Position.BOTTOM_CENTER)
        zones[name] = sv.PolygonZone(polygon=poly, triggering_anchors=(anchor,))
    # U3: one LineZone per door. Crossings carry which door, so a two-entrance
    # venue gets per-door counts and a correct total instead of nothing.
    line_zones, drawn_lines = {}, {}
    for _lname, _lpts in entry_lines.items():
        (x1, y1), (x2, y2) = _lpts
        if ENTRY_LINE_FLIP:
            (x1, y1), (x2, y2) = (x2, y2), (x1, y1)
        line_zones[_lname] = sv.LineZone(
            start=sv.Point(x1, y1), end=sv.Point(x2, y2),
            minimum_crossing_threshold=2,
            triggering_anchors=[sv.Position.BOTTOM_CENTER])
        drawn_lines[_lname] = [[x1, y1], [x2, y2]]
    if len(line_zones) > 1:
        print(f"\U0001f6aa {len(line_zones)} entry line(s): {sorted(line_zones)}")

    # U1/U2 CAMERA HEALTH — one extra decoded frame, before anything expensive.
    # Chunk 1 lays down the reference view; every later chunk is compared to it,
    # so a camera knocked at 21:00 is caught even though chunk 7 is internally
    # consistent with itself.
    _view = {"valid": True, "reasons": [], "checked": False}
    try:
        _ref_path = OUTPUT_DIR / f"viewref_{camera_id}"
        _probe = next((f for _i, _t, f in frame_source(
            video_path, 1.0, max_seconds=2.0, max_w=ANALYSIS_MAX_W,
            start_seconds=start_seconds)), None)
        if _probe is not None:
            _hc = CameraHealth.load(_ref_path)
            if _hc is None:
                _hc = CameraHealth.from_frame(
                    _probe, zone_tol_frac=VENUE_PROFILE["camera"]["zone_tol_frac"])
                _hc.save(_ref_path)
                print(f"\U0001f4f7 reference view saved for {camera_id} — later "
                      f"chunks are checked against THIS frame")
            _view = _hc.check(_probe, polygons)
            _view["checked"] = True
            print(f"\U0001f4f7 {verdict_line(_view)}")
    except Exception as _hex:
        print(f"(camera-health check skipped: {_hex})")

    rec = OccupancyRecorder(frame_step_s=frame_step_s,
                            gap_merge_s=GAP_MERGE_S, min_event_s=MIN_EVENT_S)
    crossings = []
    track_crops = defaultdict(list)     # tid -> best REID_CROPS_PER_TRACK (score, crop)
    track_pos = {}                      # tid -> [start_bc, last_bc]
    track_seated_frames = defaultdict(int)  # v42: tid -> count of frames where bbox looks seated
    track_total_frames = defaultdict(int)   # v42: tid -> total frames seen
    track_time = {}                     # tid -> [first_t, last_t]  (ALL tracks, not just zone-event ones)
    frame_log = []                      # (frame_idx, t, [(tid,x1,y1,x2,y2)])

    # ── LIVE IDENTITY MEMORY (v27) ──────────────────────────────────────────
    # Set up ONCE per video, before the frame loop. `_embed_one_live` reuses
    # the same lazy-cached embedder the offline stitcher uses — CLIP-ReID if
    # it loaded, else OSNet, else HSV (get_reid_embedder(device=device) is a no-op after
    # the first real call), so there's no separate model load — just an
    # extra call for each newly-BORN raw track_id, which is rare relative
    # to total frames.
    def _blur_score(crop_bgr, reference_size=128):
        if crop_bgr is None or crop_bgr.size == 0:
            return 0.0
        resized = cv2.resize(crop_bgr, (reference_size, reference_size), interpolation=cv2.INTER_AREA)
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
        return float(cv2.Laplacian(gray, cv2.CV_64F).var())

    def _embed_one_live(crop):
        fn = get_reid_embedder(device=device)
        if fn is None or crop is None:
            return None
        try:
            vecs = fn([crop])
            return vecs[0] if vecs else None
        except Exception:
            return None

    _dist_scale = (((frame_w**2 + frame_h**2) ** 0.5) / REF_DIAGONAL_PX) if ENABLE_RESOLUTION_SCALING else 1.0
    # F4: the live re-id gate is a budget for how far someone can move in ONE
    # frame, so it only means anything relative to eff_fps. At 4 fps this is
    # 560/4 = 140px, exactly the old hardcoded value; at 8 fps it correctly
    # halves instead of staying twice as loose as it should be.
    _live_max_dist = max(40.0, LIVE_REID_MAX_SPEED_PX_S / max(eff_fps, 1e-6))
    print(f"   live re-id gate: {_live_max_dist:.0f}px @ {eff_fps:.1f} fps "
          f"(derived from {LIVE_REID_MAX_SPEED_PX_S:.0f} px/s)")
    # G1: the plane is built lazily — the perspective fit needs samples, so
    # early frames run on pixel gates and everything after runs on metres. Same
    # pattern as carried-suppression, and for the same reason: a wrong metric
    # answer is worse than an honest pixel one.
    try:
        _zone_cfg_raw = json.loads(Path(zones_path).read_text())
    except Exception:
        _zone_cfg_raw = {}
    _ground = GroundPlane.none("perspective fit not ready yet")

    _ground_n = [0]

    def _refresh_ground():
        """B2: the plane used to be built once, the moment 200 samples existed,
        and never revisited — the first real run locked it to 200 of the 13,335
        detections eventually available and reported an implied camera height of
        1.12 m where the final fit implies 3.35 m. Now it is rebuilt whenever the
        evidence has doubled. An EXACT homography from ground_points never needs
        refitting and is left alone."""
        nonlocal _ground
        if not _persp.ready:
            return _ground
        _n = len(_persp.samples)
        if _ground.ok and (_ground.mode == "exact" or _n < _ground_n[0] * 2):
            return _ground
        _ground_n[0] = _n
        _ground = GroundPlane.from_zone_config(
            _zone_cfg_raw, (frame_w, frame_h), persp=_persp,
            person_h=VENUE_PROFILE["camera"]["person_height_m"],
            hfov_deg=VENUE_PROFILE["camera"]["hfov_deg"])
        if _ground.ok:
            print(f"\U0001f4d0 G1 {_ground.describe()}")
            for _w in _ground.sanity(frame_h):
                print(f"      !! {_w}")
            if _identity_memory is not None:
                _identity_memory.plane = _ground
        return _ground

    _identity_memory = (
        _IdentityMemory(embed_fn=_embed_one_live,
                        sim_threshold=LIVE_REID_SIM_THRESHOLD,
                        max_dist_px=_live_max_dist * _dist_scale,
                        memory_ttl_s=LIVE_REID_MEMORY_TTL_S,
                        dist_scale=_dist_scale,
                        max_speed_mps=MAX_WALK_SPEED_MPS)
        if ENABLE_LIVE_IDENTITY_MEMORY else None
    )
    _prev_occ_raw = set()   # v53: raw ids that were occluded LAST frame

    cap.release()   # v55: probing done; frame_source owns decoding from here
    duration_s = (n_frames / native_fps) if native_fps else 0.0
    if max_seconds:
        duration_s = min(duration_s, float(max_seconds))
    n_expected = int(duration_s * eff_fps) or None

    proxy_dir = None
    if PROXY_RENDER:
        proxy_dir = OUTPUT_DIR / f"_proxy_{camera_id}{chunk_tag}"
        if proxy_dir.exists():
            shutil.rmtree(proxy_dir, ignore_errors=True)
        proxy_dir.mkdir(parents=True, exist_ok=True)

    _env = probe_environment(video_path, start_s=start_seconds, span_s=max_seconds)
    _is_ir = bool(_env["is_ir"])
    if _is_ir:
        print(f"\U0001f319 INFRARED / NIGHT VISION (saturation "
              f"{_env['sat_min']:.0f}-{_env['sat_max']:.0f}, colour spread "
              f"{_env['chroma_min']:.1f}-{_env['chroma_max']:.1f}) — attire/HSV "
              f"merging disabled, CLAHE on, detection floor lowered to "
              f"{IR_DETECT_CONF_FLOOR}. Colour evidence does not exist here.")
    elif _env["switches"]:
        print(f"\u26a0\ufe0f  IR SWITCH INSIDE THIS CHUNK (saturation "
              f"{_env['sat_min']:.0f}-{_env['sat_max']:.0f}) — identities are NOT "
              f"merged on colour across it.")
    _clahe_on = bool(ENABLE_CLAHE or _is_ir)
    _attire_on = bool(ENABLE_ATTIRE_MERGE_TIER and not _is_ir)
    _staff_seen_names = set()

    _prev_small = None          # motion gate state
    # F1 per-frame IR state / F2 scene geometry / F3 head accounting
    _frame_ir = {}              # frame_idx -> bool
    _ir_state = [None]          # last seen IR state, for switch logging
    _ir_prev_main = [None]      # C4: last modality the MAIN loop acted on
    _ir_cut_last = [-1e9]       # C4: debounce clock for the hard cut
    _ir_switches = []           # (t, is_ir_now)
    _ir_pend = [0, None]        # V68b debounce: [streak, candidate]
    _track_ir_frames = defaultdict(int)
    _persp = _PerspectiveModel()
    _supp_stats = {}
    _has_head = _detector_has_head_class(model)
    _n_head_only = 0
    _last_det_t = -1e9
    _gate_since = None          # when the current gated silence began
    _n_skipped = _n_resets = 0
    _face_state = {}            # raw tid -> [tries, last_try_t, has_face]
    t = 0.0
    _eval_dir = None
    if globals().get("EVAL_EXPORT"):
        _eval_dir = OUTPUT_DIR / "eval_frames" / f"{camera_id}{chunk_tag}"
        _eval_dir.mkdir(parents=True, exist_ok=True)

    # v55 #9: detection runs on a BATCH of frames per call. Same model, same
    # frames, same results — Ultralytics batches internally and returns one
    # result per image — but 14,430 individual launches become ~3,600. The
    # motion gate lives in here too, so gated frames never enter a batch.
    _batchable = (use_online_tracker and online_tracker is not None) or (
        not use_online_tracker)

    def _analysis_stream():
        nonlocal _prev_small, _gate_since, _n_skipped
        buf, gaps = [], []

        def _flush():
            if not buf:
                return []
            # F1: if ANY frame in this batch is infrared, the whole batch runs
            # at the IR floor. Erring low costs a few extra weak detections the
            # tracker will drop; erring high loses people at night permanently.
            _bconf = (min(det_conf, IR_DETECT_CONF_FLOOR)
                      if any(_frame_ir.get(_i0) for _i0, _, _ in buf) else det_conf)
            if _batchable and DET_BATCH > 1:
                res = model([f for _, _, f in buf], conf=_bconf, iou=0.45,
                            imgsz=YOLO_IMGSZ, verbose=False, device=device_str,
                            half=globals().get("DETECTOR_HALF", False))
            elif _batchable:
                res = [model(f, conf=_bconf, iou=0.45, imgsz=YOLO_IMGSZ,
                             verbose=False, device=device_str,
                             half=globals().get("DETECTOR_HALF", False))[0]
                       for _, _, f in buf]
            else:
                res = [None] * len(buf)     # model.track() path: cannot batch
            out = list(zip(list(buf), res, list(gaps)))
            buf.clear(); gaps.clear()
            return out

        for _fi, _t, _fr in frame_source(video_path, eff_fps,
                                         max_seconds=max_seconds,
                                         max_w=ANALYSIS_MAX_W,
                                         start_seconds=start_seconds):
            # ── MOTION GATE ────────────────────────────────────────────────
            # A reception at 02:00 is empty most of the night and the detector
            # is the most expensive thing here. Skip it only when BOTH hold:
            # nothing detected for MOTION_IDLE_S, and the frame is static — so
            # a person standing perfectly still is never dropped.
            # F1: chroma on EVERY frame, measured BEFORE CLAHE (CLAHE moves L,
            # which would shift the channel spread we are testing). One 160x90
            # resize is shared with the motion gate, so this is ~free.
            _smallc = cv2.resize(_fr, (160, 90))
            _chroma71 = _frame_chroma(None, small=_smallc)
            # V71c: hysteresis — LEAVING infrared needs clearly-colour chroma
            # (1.35x), so a value hovering at the threshold cannot oscillate.
            _ir_raw = _chroma71 < (IR_CHROMA_THRESHOLD
                                   * (1.35 if _ir_state[0] else 1.0))
            # V68b: debounce — dusk chroma hovers at the threshold (43
            # flaps/hr measured). A flip must hold IR_DEBOUNCE_FRAMES
            # consecutive frames before the modality actually changes.
            if _ir_state[0] is None:
                _ir_state[0] = _ir_raw
                _ir_switches.append((_t, _ir_raw))
                _ir_pend[:] = [0, _ir_raw]
            elif _ir_raw != _ir_state[0]:
                _ir_pend[0] = _ir_pend[0] + 1 if _ir_pend[1] == _ir_raw else 1
                _ir_pend[1] = _ir_raw
                if _ir_pend[0] >= int(globals().get("IR_DEBOUNCE_FRAMES", 24)):
                    _ir_state[0] = _ir_raw
                    _ir_switches.append((_t, _ir_raw))
                    _ir_pend[0] = 0
            else:
                _ir_pend[0] = 0
            _ir_now = _ir_state[0]
            _frame_ir[_fi] = _ir_now
            if ENABLE_CLAHE or _ir_now:
                _fr = apply_clahe(_fr)

            if MOTION_GATE:
                _small = cv2.cvtColor(_smallc, cv2.COLOR_BGR2GRAY)
                if _prev_small is not None and (_t - _last_det_t) > MOTION_IDLE_S:
                    _chg = float((cv2.absdiff(_small, _prev_small) > 18).mean())
                    if _chg < MOTION_MIN_FRAC:
                        _prev_small = _small
                        if _gate_since is None:
                            _gate_since = _t
                        # full-length render: gated (still, empty) frames must
                        # still reach the proxy store or they vanish from the
                        # annotated video even with RENDER_ONLY_OCCUPIED=False
                        if proxy_dir is not None and not RENDER_ONLY_OCCUPIED:
                            cv2.imwrite(str(proxy_dir / f"{_fi:07d}.jpg"), _fr,
                                        [cv2.IMWRITE_JPEG_QUALITY,
                                         PROXY_JPEG_QUALITY])
                        frame_log.append((_fi, _t, []))
                        _n_skipped += 1
                        continue
                _prev_small = _small

            _gap = 0.0
            if _gate_since is not None:
                _gap = _t - _gate_since
                _gate_since = None
            if _eval_dir is not None and EVAL_WINDOW[0] <= _t <= EVAL_WINDOW[1]:
                cv2.imwrite(str(_eval_dir / f"{_fi:07d}.jpg"), _fr,
                            [cv2.IMWRITE_JPEG_QUALITY, 92])
            buf.append((_fi, _t, _fr)); gaps.append(_gap)
            if len(buf) >= max(1, int(DET_BATCH)):
                for item in _flush():
                    yield item
        for item in _flush():
            yield item

    for (frame_idx, t, frame), _pre_result, _gap_s in tqdm(
            _analysis_stream(), total=n_expected, desc=camera_id + " · analyze"):
        # C4: a colour<->IR flip is a modality change — every live appearance
        # anchor is garbage on the other side. Same rebuild as the gated-
        # silence path below, debounced against dusk flicker. OFF by default;
        # the ablation run decides whether it earns its place.
        _irf = _frame_ir.get(frame_idx)
        _ir_cut = (ENABLE_IR_HARD_CUT and _irf is not None
                   and _ir_prev_main[0] is not None
                   and _irf != _ir_prev_main[0]
                   and (t - _ir_cut_last[0]) >= IR_CUT_MIN_GAP_S)
        if _irf is not None:
            _ir_prev_main[0] = _irf
        if _ir_cut:
            _ir_cut_last[0] = t
            print(f"✂️  C4 IR hard-cut at t={t:.0f}s: tracker + live identity "
                  f"reset ({'-> IR' if _irf else '-> colour'})")
        # coming back from a long gated silence: the tracker has been frozen in
        # frame-time while real time moved on. Rebuild it (see _fresh_tracker).
        if _gap_s > LOST_TRACK_BUFFER_S or _ir_cut:
            if use_online_tracker and online_tracker is not None:
                online_tracker = _fresh_tracker()
            elif tracker is not None:
                tracker = _fresh_tracker()
            _n_resets += 1
            if _identity_memory is not None:
                _identity_memory.raw_to_canon.clear()

        if use_online_tracker and online_tracker is not None:
            # Real-OSNet path: detect only (no Ultralytics tracker wrapper),
            # then hand raw detections + the frame to boxmot's BotSort so its
            # live Hungarian match gets genuine appearance vectors.
            dets = sv.Detections.from_ultralytics(_pre_result)
            dets, _heads = _split_person_head(dets, _has_head)
            _n_head_only += len(_heads_without_person(_heads, dets))
            _feed_perspective(dets, _persp)
            _refresh_ground()
            dets = _suppress_carried(dets, _persp, _supp_stats)
            dets = _drop_implausible(dets, _persp, _supp_stats)
            dets = _drop_masked(dets)
            if len(dets):   # A1: two same-size boxes on one body -> one box
                dets = dets.with_nms(threshold=DEDUP_NMS_IOU, class_agnostic=True)
            tracked = online_tracker.update(_dets_to_boxmot(dets), frame)
            dets = _boxmot_to_dets(tracked)
        elif TRACKER_MODE == "botsort-reid":
            result = model.track(frame, persist=True, conf=det_conf,
                                 iou=0.45, imgsz=YOLO_IMGSZ,
                                 tracker=tracker_yaml, verbose=False,
                                 device=device_str,
                                 half=globals().get("DETECTOR_HALF", False))[0]
            dets = sv.Detections.from_ultralytics(result)
            dets, _heads = _split_person_head(dets, _has_head)
            _n_head_only += len(_heads_without_person(_heads, dets))
            _feed_perspective(dets, _persp)
            _refresh_ground()   # A3: metre gates were silently dead here
            dets = _suppress_carried(dets, _persp, _supp_stats)
            dets = _drop_implausible(dets, _persp, _supp_stats)
            dets = _drop_masked(dets)
            # F6: A1 deliberately NOT applied on this branch — ids were already
            # minted inside model.track(persist=True); post-hoc NMS makes the
            # SURVIVING id flip with per-frame confidence rank, which feeds the
            # live identity memory an oscillating id: worse than the duplicate.
            # The online-BoxMOT and ByteTrack branches NMS BEFORE the tracker.
            if dets.tracker_id is None:
                dets = dets[np.zeros(len(dets), dtype=bool)]
                dets.tracker_id = np.array([], dtype=object)
        else:
            dets = sv.Detections.from_ultralytics(_pre_result)
            dets, _heads = _split_person_head(dets, _has_head)
            _n_head_only += len(_heads_without_person(_heads, dets))
            _feed_perspective(dets, _persp)
            _refresh_ground()   # A3: metre gates were silently dead here
            dets = _suppress_carried(dets, _persp, _supp_stats)
            dets = _drop_implausible(dets, _persp, _supp_stats)
            dets = _drop_masked(dets)
            if len(dets):   # A1
                dets = dets.with_nms(threshold=DEDUP_NMS_IOU, class_agnostic=True)
            dets = tracker.update_with_detections(dets)

        # ── v44 OCCLUSION GUARD (computed once; used by identity memory
        # AND crop banking): detections whose boxes mutually overlap this frame
        # have crops contaminated by the neighbouring body. ──────────────────
        _occluded_idx = set()
        if ENABLE_OCCLUSION_GUARD and len(dets.xyxy) > 1:
            _bx = dets.xyxy
            for _a in range(len(_bx)):
                for _b in range(_a + 1, len(_bx)):
                    if _boxes_occluding(_bx[_a], _bx[_b]):
                        _occluded_idx.add(_a); _occluded_idx.add(_b)

        # ── LIVE IDENTITY MEMORY: resolve/remap BEFORE anything downstream
        # (positions, zones, crossings, crop banking, render log) ever sees
        # this frame's ids, so a corrected id is the ONLY id those consumers
        # ever observe -- no separate patch-up needed in each of them.
        if (_identity_memory is not None and len(dets.tracker_id)
                and dets.confidence is not None):
            remapped = []
            assigned_in_frame = set()
            _row_info = []   # v53: (det idx, raw id, clean crop, was_occluded)
            # v44 CO-VISIBILITY: canonical ids already on-screen this frame.
            _present_canons = set()
            if ENABLE_COVISIBILITY_BLOCK:
                for _t in dets.tracker_id:
                    _tc = _identity_memory.raw_to_canon.get(_safe_id(_t))
                    if _tc is not None:
                        _present_canons.add(_tc)
            # ── v55 PASS A: build the clean crops. No model calls here. ────
            _crops, _rawids, _confs, _boxes_i = [], [], [], []
            for _i, (bx1, by1, bx2, by2) in enumerate(dets.xyxy):
                raw_tid = _safe_id(dets.tracker_id[_i])
                bx1i, by1i, bx2i, by2i = int(bx1), int(by1), int(bx2), int(by2)
                conf_i = float(dets.confidence[_i])
                crop = None
                if conf_i >= 0.35 and (by2i - by1i) >= REID_MIN_CROP_H:
                    # v43 aspect ratio gate — v53: a body clipped by the
                    # frame border is legitimately short/wide, so skip the
                    # gate for edge boxes instead of discarding the person
                    aspect = float(by2i - by1i) / max(1, bx2i - bx1i)
                    _at_edge = (bx1i <= 2 or by1i <= 2
                                or bx2i >= frame.shape[1] - 3
                                or by2i >= frame.shape[0] - 3)
                    if _at_edge or MIN_BODY_ASPECT <= aspect <= MAX_BODY_ASPECT:
                        _c = frame[max(0, by1i):by2i, max(0, bx1i):bx2i]
                        if _c.size:
                            ch, cw = _c.shape[:2]
                            is_blurry = False
                            if min(ch, cw) >= MIN_CROP_PX_BLUR_GATE:
                                if _blur_score(_c) < MIN_BLUR_VARIANCE:
                                    is_blurry = True
                            if not is_blurry:
                                crop = cv2.resize(_c, (128, 256))
                _crops.append(crop); _rawids.append(raw_tid)
                _confs.append(conf_i); _boxes_i.append((bx1i, by1i, bx2i, by2i))

            # ── v55 PASS B: ONE batched Re-ID forward pass for the crops that
            # actually need a fresh vector. A track whose appearance we
            # embedded < REEMBED_EVERY_S ago does not need it again — it was
            # being recomputed for every person on every frame, at batch 1.
            _need = []
            for _i, _cr in enumerate(_crops):
                if _cr is None or _i in _occluded_idx:
                    continue
                _own_c = _identity_memory.raw_to_canon.get(_rawids[_i])
                if _own_c is None:
                    _need.append(_i)                       # a birth must have one
                else:
                    _rc = _identity_memory.bank.get(_own_c)
                    if _rc is None or (t - _rc.get("t_emb", -1e9)) >= REEMBED_EVERY_S:
                        _need.append(_i)
            _vecs = {}
            if _need:
                try:
                    _efn = get_reid_embedder(device=device)
                    if _efn is not None:
                        for _k, _v in zip(_need, _efn([_crops[k] for k in _need])):
                            _vecs[_k] = _v
                except Exception as _emb_exc:
                    if not globals().get("_EMB_WARNED"):
                        globals()["_EMB_WARNED"] = True
                        print(f"!! appearance embedding unavailable ({_emb_exc}) — "
                              f"tracking continues on motion only, identity will "
                              f"fragment more. Fix before trusting per-person numbers.")

            # ── v55 PASS C: faces, throttled. InsightFace on every crop on
            # every frame cost more than the detector did. A track only needs
            # ONE good face; after FACE_MAX_TRIES we stop paying for it.
            _faces = {}
            if ENABLE_FACE_CORROBORATION:
                for _i, _cr in enumerate(_crops):
                    if _cr is None:
                        continue
                    _fs = _face_state.setdefault(_rawids[_i], [0, -1e9, False])
                    if _fs[2] or _fs[0] >= FACE_MAX_TRIES or (t - _fs[1]) < FACE_RETRY_EVERY_S:
                        continue
                    _fs[0] += 1; _fs[1] = t
                    _emb, _dsc, _side = embed_face_scored(_cr)
                    if _emb is not None:
                        _fs[2] = True
                        _faces[_i] = _emb

            # ── v55 PASS D: resolve identities (no model calls left) ────────
            for _i in range(len(_crops)):
                raw_tid, conf_i, crop = _rawids[_i], _confs[_i], _crops[_i]
                bx1i, by1i, bx2i, by2i = _boxes_i[_i]
                face_emb = _faces.get(_i)

                # Check staff face match
                staff_match_name = None
                if face_emb is not None and _STAFF_FACE_GALLERY:
                    best_sim = -1
                    for sname, semb in _STAFF_FACE_GALLERY.items():
                        sim = _cosine(face_emb, semb)
                        if sim > best_sim:
                            best_sim = sim
                            if sim >= STAFF_MATCH_THRESHOLD:
                                staff_match_name = sname

                # If matched staff face and not yet assigned to another in this frame
                if staff_match_name and staff_match_name not in assigned_in_frame:
                    assigned_in_frame.add(staff_match_name)
                    if _identity_memory:
                        _identity_memory.raw_to_canon[raw_tid] = staff_match_name
                        _identity_memory._remember(staff_match_name, face_emb,
                                                   ((bx1i + bx2i) / 2, (by1i + by2i) / 2),
                                                   t, conf_i)
                    remapped.append(staff_match_name)
                    rec.vote_role(staff_match_name, "staff")
                    _staff_seen_names.add(staff_match_name)
                else:
                    _is_occ = _i in _occluded_idx
                    _own = _identity_memory.raw_to_canon.get(raw_tid)
                    _blocked = (_present_canons - {_own}) if _own is not None else _present_canons
                    resolved_id = _identity_memory.resolve(
                        raw_tid, crop, conf_i, (bx1i, by1i, bx2i, by2i), t,
                        frozen=_is_occ, blocked_canons=_blocked,
                        vec=_vecs.get(_i))
                    remapped.append(resolved_id)
                    _row_info.append((_i, raw_tid, crop, _is_occ))

            # v53 SWAP RE-VALIDATION: tracks that were overlapping last frame
            # and are clean now get their identities cross-checked - a trade
            # made mid-occlusion is undone here, on the first clean evidence.
            if ENABLE_SWAP_REVALIDATION:
                _exited = [(ix, r, c) for (ix, r, c, o) in _row_info
                           if (not o) and r in _prev_occ_raw and c is not None]
                for _ea in range(len(_exited)):
                    for _eb in range(_ea + 1, len(_exited)):
                        _ia, _ra, _ca = _exited[_ea]
                        _ib, _rb, _cb = _exited[_eb]
                        if _identity_memory.try_swap(_ra, _ca, _rb, _cb,
                                                     SWAP_MARGIN):
                            remapped[_ia] = _identity_memory.raw_to_canon[_ra]
                            remapped[_ib] = _identity_memory.raw_to_canon[_rb]
                _prev_occ_raw = {r for (_ix, r, _c, o) in _row_info if o}
            dets.tracker_id = np.array(remapped, dtype=object)

        boxes_this = []
        if len(dets.xyxy):
            _last_det_t = t
            if proxy_dir is not None:
                cv2.imwrite(str(proxy_dir / f"{frame_idx:07d}.jpg"), frame,
                            [cv2.IMWRITE_JPEG_QUALITY, PROXY_JPEG_QUALITY])
        elif proxy_dir is not None and (
                not RENDER_ONLY_OCCUPIED
                # F4: a solo person blinking out leaves an EMPTY frame —
                # without a proxy JPEG the A6 coasted box was silently
                # dropped at render. Keep proxies within the coast window.
                or (t - _last_det_t) <= globals().get("RENDER_COAST_S", 0.5)):
            cv2.imwrite(str(proxy_dir / f"{frame_idx:07d}.jpg"), frame,
                        [cv2.IMWRITE_JPEG_QUALITY, PROXY_JPEG_QUALITY])
        for (bx1, by1, bx2, by2), tid, cid_ in zip(dets.xyxy, dets.tracker_id,
                                                   dets.class_id):
            tid = _safe_id(tid)
            bx1, by1, bx2, by2 = int(bx1), int(by1), int(bx2), int(by2)
            boxes_this.append((tid, bx1, by1, bx2, by2))
            # v42: seated/standing detection via bbox aspect ratio
            _bw, _bh = max(bx2 - bx1, 1), max(by2 - by1, 1)
            track_total_frames[tid] += 1
            if _frame_ir.get(frame_idx):
                _track_ir_frames[tid] += 1      # F1: colour evidence is invalid
            if _bw / _bh > 0.45:  # seated people have wider/shorter bboxes
                track_seated_frames[tid] += 1
            bc = ((bx1 + bx2) / 2.0, float(by2))
            if tid not in track_pos:
                track_pos[tid] = [bc, bc]
                track_time[tid] = [t, t]
            else:
                track_pos[tid][1] = bc
                track_time[tid][1] = t
            role_vote = "staff" if (isinstance(tid, str) and not tid.isdigit()) else "customer"
            rec.vote_role(tid, role_vote)
        frame_log.append((frame_idx, t, boxes_this))

        for name, zone in zones.items():
            for tid in dets.tracker_id[zone.trigger(dets)]:
                rec.add(t, name, _safe_id(tid))
        for _lname, line_zone in line_zones.items():
            crossed_in, crossed_out = line_zone.trigger(dets)
            # G3: record WHERE the crossing happened. Tier A de-duplicates
            # in space and time, so a crossing without a position is a crossing
            # that can only fall back to trusting the track id.
            _bc_of = {}
            for (_cx1, _cy1, _cx2, _cy2), _ctid in zip(dets.xyxy, dets.tracker_id):
                _bc_of[_safe_id(_ctid)] = ((float(_cx1) + float(_cx2)) / 2.0,
                                           float(_cy2))
            for tid in dets.tracker_id[crossed_in]:
                _sid = _safe_id(tid)
                crossings.append({"t": round(t, 2), "track_id": _sid,
                                  "direction": "in", "line": _lname,
                                  "pos": _bc_of.get(_sid)})
            for tid in dets.tracker_id[crossed_out]:
                _sid = _safe_id(tid)
                crossings.append({"t": round(t, 2), "track_id": _sid,
                                  "direction": "out", "line": _lname,
                                  "pos": _bc_of.get(_sid)})
        if ENABLE_REID_STITCH and dets.confidence is not None:
            for _bi, (box, conf, tid) in enumerate(zip(dets.xyxy, dets.confidence,
                                      dets.tracker_id)):
                if _bi in _occluded_idx:
                    continue  # v44: don't bank a crop contaminated by overlap
                bx1, by1, bx2, by2 = [int(v) for v in box]
                if conf >= 0.35 and (by2 - by1) >= REID_MIN_CROP_H:
                    # v43 aspect ratio gate
                    aspect = float(by2 - by1) / max(1, bx2 - bx1)
                    if MIN_BODY_ASPECT <= aspect <= MAX_BODY_ASPECT:
                        crop = frame[max(0, by1):by2, max(0, bx1):bx2]
                        if crop.size:
                            # v43 blur gate
                            ch, cw = crop.shape[:2]
                            is_blurry = False
                            if min(ch, cw) >= MIN_CROP_PX_BLUR_GATE:
                                if _blur_score(crop) < MIN_BLUR_VARIANCE:
                                    is_blurry = True
                            if not is_blurry:
                                lst = track_crops[_safe_id(tid)]
                                lst.append((float(conf) * (by2 - by1),
                                            cv2.resize(crop, (128, 256))))
                                lst.sort(key=lambda sc: -sc[0])
                                del lst[REID_CROPS_PER_TRACK:]
                                # V76: the SOURCE height, before the resize to
                                # 256. Banked crops are all exactly 256 tall,
                                # so measuring them can only ever report 256.
                                try:
                                    _src_crop_h.append(int(by2 - by1))
                                except NameError:
                                    pass
    if start_seconds:
        print(f"\u23e9 analysed {mmss(start_seconds)} -> "
              f"{mmss(start_seconds + (max_seconds or 0))} of this chunk only")

    if MOTION_GATE and _n_skipped:
        print(f"\u26a1 motion gate: skipped the detector on {_n_skipped} empty, "
              f"static frames ({100.0 * _n_skipped / max(1, len(frame_log)):.0f}% "
              f"of this chunk); tracker restarted {_n_resets} time(s) after a "
              f"silence longer than {LOST_TRACK_BUFFER_S}s")

    # F1/F2/F3 run diagnostics — these were the silent failures.
    _n_ir = sum(1 for v in _frame_ir.values() if v)
    print(f"\U0001f319 F1 infrared: {_n_ir}/{len(_frame_ir)} analysed frames "
          f"({100.0 * _n_ir / max(1, len(_frame_ir)):.0f}%), "
          f"{max(0, len(_ir_switches) - 1)} switch(es) mid-chunk"
          + (" — colour/IR handled per frame" if _ir_switches else ""))
    for _st, _sir in _ir_switches[1:6]:
        print(f"      switch at {mmss(_st)} -> {'INFRARED' if _sir else 'colour'}")
    print(f"\U0001f4d0 F2 {_persp.describe()}")
    print(f"      carried-suppression removed "
          f"{_supp_stats.get('carried_suppressed', 0)} detection(s) this chunk "
          f"(needs containment AND off-floor height AND a low head — an "
          f"occluded guest fails the head test and is kept)")
    if _has_head:
        print(f"\U0001f9e0 F3 head class ACTIVE: {_n_head_only} head detection(s) "
              f"had no person box around them — each is a body the detector lost "
              f"behind someone else"
              + ("" if ENABLE_HEAD_RECOVERY else
                 " (recovery OFF until Phase 2 scores it)"))
    else:
        print("\U0001f9e0 F3 detector is not 2-class person/head — head signal "
              "unavailable (this is correct for stock COCO weights)")
    if _identity_memory is not None and _identity_memory.reassignments:
        print(f"\U0001f9e0 Live identity memory: "
              f"{_identity_memory.reassignments} raw track-id birth(s) "
              f"redirected to an existing identity in real time "
              f"(id would otherwise have changed abruptly)")

    # entry zones keep sub-2s doorway transits: a walking-pace pass through a
    # shallow main_entrance polygon lasts ~1s, and MIN_EVENT_S=2.0 was deleting
    # every one of them — which is why the region arrival count read 0 even
    # with a perfectly drawn entry polygon.
    events = rec.events(min_event_by_zone={z: 0.5 for z, rs in zone_roles.items()
                                           if "entry" in rs})
    if staff_zones_here:
        events = apply_staff_zone_override(events, staff_zones_here,
                                           STAFF_OVERRIDE_MIN_S,
                                           min_share_dwell_s=STAFF_OVERRIDE_MIN_S)

    # ── Re-ID stitching (appearance + spatio-temporal hand-off) ────────────
    id_merges, mapping = {}, {}
    _sweep_hits = {}   # F3: defined for EVERY flag combination — the
    #    protection call sites below reference it even when stitch is off
    calibration_report, face_validation_report = None, None
    face_veto_report, face_veto_flagged = None, None
    merge_diagnostics, identity_dossiers = None, {}
    if ENABLE_REID_STITCH and (events or track_time):
        try:
            embed = get_reid_embedder(device=device)
            windows = {tid: (tt[0], tt[1]) for tid, tt in track_time.items()}
            for e in events:   # widen using zone dwell bounds too, doesn't hurt
                w0 = windows.get(e["track_id"])
                lo = min(w0[0], e["t_in"]) if w0 else e["t_in"]
                hi = max(w0[1], e["t_out"]) if w0 else e["t_out"]
                windows[e["track_id"]] = (lo, hi)
            embeddings = {}
            anchor_embeddings = {}
            if embed is not None:
                embeddings = {tid: embed([c for _, c in crops])
                              for tid, crops in track_crops.items()
                              if tid in windows and crops}
                # ANCHOR SNAPSHOT (v25): crops in track_crops[tid] are kept
                # re-sorted best-first every time one is banked (see the
                # `lst.sort(key=lambda sc: -sc[0])` above), so crops[0] IS the
                # single highest-quality snapshot for that track — the
                # canonical reference image an operator would point to and
                # say "that's this person." Embed it separately (not just as
                # the first item pulled from the full gallery) so
                # merge_fragmented_tracks can do a clean 1:1 anchor-to-anchor
                # comparison, which is the strongest single piece of identity
                # evidence available and immune to a lucky/unlucky pairing
                # among the OTHER banked crops skewing the best-of-gallery
                # score in _gallery_sim.
                anchor_embeddings = {tid: embed([crops[0][1]])[0]
                                     for tid, crops in track_crops.items()
                                     if tid in windows and crops}
            # (v37) same banked crop IMAGES (not embeddings) — feeds the new
            # HSV attire merge tier in merge_fragmented_tracks, reusing the
            # exact crops already banked for appearance, no extra work.
            # F1: a track that lived under infrared has NO colour information,
            # so its HSV attire similarity is noise. Feeding it to the attire
            # merge tier and the HSV vetoes does not just add nothing — it
            # invents matches. Those tracks are withheld from every colour-based
            # signal; their face/appearance/hand-off evidence is untouched.
            _ir_tracks = {tid for tid, n in _track_ir_frames.items()
                          if n >= IR_TRACK_FRAC * max(1, track_total_frames.get(tid, 1))}
            raw_crops = {tid: [c for _, c in crops]
                        for tid, crops in track_crops.items()
                        if tid in windows and crops and tid not in _ir_tracks}
            if _ir_tracks:
                print(f"\U0001f319 F1: {len(_ir_tracks)} of {len(track_crops)} tracks "
                      f"lived under infrared — withheld from attire/HSV evidence "
                      f"(colour does not exist in those frames)")
            positions = {tid: (track_pos[tid][0], track_pos[tid][1])
                         for tid in windows if tid in track_pos}
            # A4: fraction of each track's life spent under IR — feeds the
            # cross-modality guard inside merge_fragmented_tracks
            _ir_frac = {tid: _track_ir_frames.get(tid, 0)
                        / max(1, track_total_frames.get(tid, 1))
                        for tid in windows}
            # (v36) hard number on crop resolution -- a common, boring, and
            # easily-missed reason appearance embeddings don't separate:
            # crops just too small for ANY model (or a human) to make out
            # real detail. Cheap to compute, worth printing every run.
            if track_crops:
                # V76: source heights if we collected them, else fall back
                # to the old (always-256) measurement rather than crashing.
                _sizes = list(globals().get("_src_crop_h") or [])
                _from_source = bool(_sizes)
                if not _sizes:
                    _sizes = [c.shape[0] for crops in track_crops.values()
                              for _, c in crops]
                if _sizes:
                    _sizes.sort()
                    _med = _sizes[len(_sizes)//2]
                    _p10 = _sizes[int(len(_sizes)*0.1)]
                    print(f"📐 {'SOURCE' if _from_source else 'Banked'} crop height: median {_med}px, p10 {_p10}px "
                          f"across {len(_sizes)} crops — ReID models are "
                          f"typically trained on ~256px-tall crops; well "
                          f"below that (say <100px) means the embeddings may "
                          f"not have enough real detail to separate people "
                          f"regardless of backbone or threshold")
            method = (_REID_CACHE.get(_boxmot_dev(device), _REID_STATE)
                      .get("method")) or "none"
            thresh = (REID_SIM_THRESHOLD if method in ("osnet", "clip")
                      else max(REID_SIM_THRESHOLD, 0.93))

            # PHASE8FIX_ROLE_HINT_HOISTED: hoisted above the FACE_SCOPE block, which
            # reads role_hint. It used to be defined 55 lines LATER, so the
            # read raised UnboundLocalError, the surrounding try/except
            # swallowed it, and the entire Re-ID stitching stage silently did
            # nothing — 95 fragmented 'people' instead of 22.
            # Role-boundary guard: only trust a role if the fragment actually
            # earned it — real dwell time, not a single frame's classification.
            # (v38) moved BEFORE calibration so calibrate_appearance_threshold
            # can exclude staff-vs-customer pairs from its same-person ground
            # truth too — see that function's role_hint docstring for why a
            # busy counter needs this guard on the ground truth, not just on
            # real merges.
            _zone_time = defaultdict(lambda: {"staff": 0.0, "other": 0.0})
            for e in events:
                bucket = "staff" if e["zone"] in staff_zones_here else "other"
                _zone_time[e["track_id"]][bucket] += e["duration"]
            role_hint = {}
            _vid_dur = max(1.0, float(t))          # seconds of footage seen
            for tid, dwell in _zone_time.items():
                _long_enough = dwell["staff"] >= STAFF_OVERRIDE_MIN_S
                _big_share   = dwell["staff"] >= STAFF_MIN_VIDEO_SHARE * _vid_dur
                _dominates   = dwell["staff"] >= STAFF_DOMINANCE_RATIO * dwell["other"]
                if _long_enough and _big_share and _dominates:
                    role_hint[tid] = "staff"
                elif dwell["other"] >= MIN_SEATED_S:
                    role_hint[tid] = "customer"

            # ── FACE EMBEDDINGS (v30, optional, corroborating only) ────────
            # PHASE 8: FACE_SCOPE gates whose face is computed post-tracking.
            # "staff_only" = only tracks known to be staff (gallery match or zone
            # dwell). Customer faces are never embedded in this phase.
            # "all" = every track (requires explicit consent in venue profile).
            face_embeddings = {}
            _face_rejected_sizes = []
            _sweep_hits = {}   # C2: {track_id: enrolled staff name}
            _sweep_scores = {}  # V68c: face score per sweep hit
            # V65: per-track box index for source-resolution face re-crops
            _v65_scale = _src_w / max(frame_w, 1)
            _v65_budget = [int(globals().get("FACE_RECROP_MAX_TRACKS", 120))]
            _v65_rescued = [0, 0]     # attempts, successes
            _v65_boxes = {}
            if globals().get("FACE_SOURCE_RECROP") and _v65_scale > 1.2:
                for _fi, _t65, _bs in frame_log:
                    for _b in _bs:
                        _v65_boxes.setdefault(_b[0], []).append(
                            (_fi, (_b[1], _b[2], _b[3], _b[4])))

            def _v65_retry(tid):
                if (not _v65_boxes or _v65_budget[0] <= 0
                        or tid not in _v65_boxes):
                    return None
                _v65_budget[0] -= 1
                _v65_rescued[0] += 1
                _fe = _recrop_face_from_source(
                    video_path, _v65_boxes[tid], _v65_scale, step, native_fps,
                    start_seconds, k=int(globals().get("FACE_RECROP_FRAMES", 4)))
                if _fe is not None:
                    _v65_rescued[1] += 1
                return _fe
            _face_scope = FACE_SCOPE if 'FACE_SCOPE' in dir() else "staff_only"
            if ENABLE_FACE_CORROBORATION:
                # Determine which track IDs are eligible for face embedding
                if _face_scope == "all":
                    _face_eligible = set(track_crops.keys())
                else:
                    # staff_only: gallery-matched IDs + zone-dwell staff IDs
                    _face_eligible = set()
                    # Staff identified by gallery match (string IDs from _identity_memory)
                    if _identity_memory:
                        for _raw, _canon in _identity_memory.raw_to_canon.items():
                            if isinstance(_canon, str) and _canon in _STAFF_FACE_GALLERY:
                                _face_eligible.add(_canon)
                                _face_eligible.add(_raw)
                    # Staff identified by zone dwell (role_hint computed above)
                    for _tid, _role in role_hint.items():
                        if _role == "staff":
                            _face_eligible.add(_tid)
                    if _face_eligible:
                        print(f"🔒 FACE_SCOPE={_face_scope!r}: face embeddings "
                              f"restricted to {len(_face_eligible)} staff track(s)")
                    else:
                        print(f"🔒 FACE_SCOPE={_face_scope!r}: no staff tracks "
                              f"identified — no post-processing face embeddings computed")

                for tid, crops in track_crops.items():
                    if tid not in windows or not crops:
                        continue
                    if tid not in _face_eligible:
                        continue
                    fe = get_track_face_embedding(crops, size_log=_face_rejected_sizes)
                    if fe is None:
                        fe = _v65_retry(tid)   # V65: 4K source re-crop
                    if fe is not None:
                        face_embeddings[tid] = fe

                # C2: STAFF-GALLERY SWEEP — breaks the staff_only circularity.
                # The eligibility gate above only embeds tracks ALREADY
                # believed staff, so a staff member the live gallery match
                # missed (fragmented at the door, face turned away for the
                # first minutes) could never be face-merged at all. Here every
                # remaining track's best crops get ONE chance against the
                # ENROLLED gallery. A match is the strongest identity evidence
                # this pipeline has; a non-match is discarded on the spot —
                # customer faces are still never stored.
                if ENABLE_STAFF_GALLERY_SWEEP and _STAFF_FACE_GALLERY:
                    for tid, crops in track_crops.items():
                        if (tid in face_embeddings or tid not in windows
                                or not crops):
                            continue
                        fe = get_track_face_embedding(crops)
                        if fe is None:
                            fe = _v65_retry(tid)   # V65: 4K source re-crop
                        if fe is None:
                            continue
                        _best, _bname = -1.0, None
                        for _sn, _se in _STAFF_FACE_GALLERY.items():
                            _s = _cosine(fe, _se)
                            if _s > _best:
                                _best, _bname = _s, _sn
                        if _best >= STAFF_MATCH_THRESHOLD:
                            _sweep_hits[tid] = _bname
                            _sweep_scores[tid] = _best
                            face_embeddings[tid] = fe   # a staff face — kept
                            role_hint[tid] = "staff"
                        # else: fe goes out of scope — nothing stored
                    # V68c: one body, one name — two tracks that OVERLAP in
                    # time cannot both be this staff member. Keep the best
                    # face score; losers stay role=staff but unnamed.
                    _by_name = {}
                    for _v68t in sorted(_sweep_hits,
                                        key=lambda k: -_sweep_scores.get(k, 0)):
                        _v68n = _sweep_hits[_v68t]
                        _v68w = windows.get(_v68t)
                        _kept = _by_name.setdefault(_v68n, [])
                        if _v68w and any(
                                _v68w[0] < windows[_k][1]
                                and windows[_k][0] < _v68w[1]
                                for _k in _kept if _k in windows):
                            del _sweep_hits[_v68t]
                            print(f"   V68c: track {_v68t} face-matched "
                                  f"{_v68n} but overlaps a stronger "
                                  f"{_v68n} track — name not applied")
                        else:
                            _kept.append(_v68t)
                    if _sweep_hits:
                        print(f"🧲 C2 gallery sweep: {len(_sweep_hits)} "
                              f"track(s) matched enrolled staff by face: "
                              + ", ".join(f"{t}->{n}" for t, n in
                                          list(_sweep_hits.items())[:6]))
                if _v65_rescued[0]:
                    print(f"👓 V65 source re-crop: {_v65_rescued[1]} face(s) "
                          f"rescued from {_v65_rescued[0]} 4K retr"
                          f"{'y' if _v65_rescued[0] == 1 else 'ies'} "
                          f"(scale {_v65_scale:.1f}x, budget left "
                          f"{_v65_budget[0]})")
                if track_crops:
                    _scope_label = ("STAFF ONLY — customer faces not processed"
                                    if _face_scope != "all"
                                    else "ALL TRACKS — customer consent required")
                    print(f"🔒 Face scope: {_scope_label}")
                    cov_pct = 100.0 * len(face_embeddings) / max(1, len(track_crops))
                    print(f"👤 Face corroboration: {len(face_embeddings)}/"
                          f"{len(track_crops)} tracks ({cov_pct:.0f}%) had a "
                          f"usable face crop (>= {FACE_MIN_FACE_PX}px, "
                          f"det_score >= {FACE_MIN_DET_SCORE})")
                    # (v32) most CCTV face misses are SIZE misses, not
                    # confidence misses -- surface that split so a low
                    # coverage number doesn't get misread as "faces rarely
                    # visible" when it's actually "faces visible but too
                    # small to trust," which is a different, more fixable
                    # problem (camera angle/zoom, not camera absence).
                    if _face_rejected_sizes:
                        _face_rejected_sizes.sort()
                        _med = _face_rejected_sizes[len(_face_rejected_sizes)//2]
                        print(f"   ℹ️  {len(_face_rejected_sizes)} additional "
                              f"crop(s) found A face but it was < "
                              f"{FACE_MIN_FACE_PX}px (median rejected size "
                              f"{_med:.0f}px) — too small to trust as identity "
                              f"evidence, excluded rather than risking a "
                              f"noisy embedding")

            # ── APPEARANCE-THRESHOLD CALIBRATION (v30, auto-applied v37) ────
            # Measures THIS run's actual same/different-person cosine
            # distribution for whichever backbone loaded, instead of trusting
            # OSNet's carried-over numbers. When CALIBRATION_AUTO_APPLY is on,
            # suggest_appearance_thresholds() turns that measurement into the
            # threshold actually used a few lines down — no more copying a
            # printed suggestion into the config cell by hand between runs.
            run_anchor_thresh = ANCHOR_SIM_THRESHOLD
            if ENABLE_REID_CALIBRATION and anchor_embeddings:
                cal = calibrate_appearance_threshold(
                    windows, positions, anchor_embeddings,
                    handoff_gap_s=REID_HANDOFF_GAP_S,
                    handoff_px=REID_HANDOFF_PX,
                    stationary_px=REID_STATIONARY_PX,
                    role_hint=role_hint)
                if cal["same_n"] and cal["diff_n"]:
                    sep_ok = (cal["same_p10"] is not None and cal["diff_p90"] is not None
                             and cal["same_p10"] > cal["diff_p90"])
                    print(f"📏 Calibration ({method}): same-person "
                          f"p10={cal['same_p10']:.3f} p50={cal['same_p50']:.3f} "
                          f"(n={cal['same_n']}) | different-person "
                          f"p50={cal['diff_p50']:.3f} p90={cal['diff_p90']:.3f} "
                          f"(n={cal['diff_n']})")
                    if cal.get("role_conflicts_excluded"):
                        print(f"   ℹ️  excluded {cal['role_conflicts_excluded']} "
                              f"spatially-close pair(s) from the same-person "
                              f"ground truth — they'd earned opposite staff/"
                              f"customer roles, so proximity alone doesn't "
                              f"make them one person (busy-counter guard)")
                    if cal.get("duplicate_excluded"):
                        print(f"   ℹ️  excluded {cal['duplicate_excluded']} "
                              f"co-visible pair(s) from the different-person "
                              f"ground truth — they stayed glued to the same "
                              f"spot the whole time, so they're one body "
                              f"wearing two track IDs, not two people "
                              f"(duplicate-track guard)")
                    if CALIBRATION_AUTO_APPLY:
                        thresh, run_anchor_thresh = suggest_appearance_thresholds(
                            cal, thresh, ANCHOR_SIM_THRESHOLD)
                        if sep_ok:
                            print(f"   ✅ clean separation — AUTO-APPLIED this "
                                  f"run: appearance/anchor thresholds = "
                                  f"{thresh}/{run_anchor_thresh} (config "
                                  f"defaults {REID_SIM_THRESHOLD}/"
                                  f"{ANCHOR_SIM_THRESHOLD} left untouched)")
                        else:
                            print(f"   ⚠️  distributions overlap — {method}'s "
                                  f"appearance signal alone can't cleanly "
                                  f"separate same/different person on this "
                                  f"footage; AUTO-APPLIED a conservative "
                                  f"appearance/anchor bar of {thresh}/"
                                  f"{run_anchor_thresh} for this run and "
                                  f"leaning on the hand-off/stationary, "
                                  f"attire (HSV), and face merge tiers to "
                                  f"cover what appearance alone is missing")
                    else:
                        suggestion = (round((cal["same_p10"] + cal["diff_p90"]) / 2, 3)
                                     if sep_ok else None)
                        if sep_ok:
                            print(f"   ✅ clean separation — suggested "
                                  f"REID_SIM_THRESHOLD/ANCHOR_SIM_THRESHOLD "
                                  f"≈ {suggestion} (currently "
                                  f"{REID_SIM_THRESHOLD}/{ANCHOR_SIM_THRESHOLD}, "
                                  f"CALIBRATION_AUTO_APPLY is off)")
                        else:
                            print(f"   ⚠️  distributions overlap — {method}'s "
                                  f"appearance signal is not cleanly separating "
                                  f"same/different person on this footage at any "
                                  f"single threshold; current thresholds "
                                  f"(REID_SIM_THRESHOLD={REID_SIM_THRESHOLD}, "
                                  f"ANCHOR_SIM_THRESHOLD={ANCHOR_SIM_THRESHOLD}) "
                                  f"are a best-effort compromise, not a clean "
                                  f"cut (CALIBRATION_AUTO_APPLY is off)")
                else:
                    print(f"📏 Calibration: not enough hand-off/stationary "
                          f"({cal['same_n']}) or co-visible ({cal['diff_n']}) "
                          f"pairs this run to measure — skipped")
                calibration_report = cal
                # F5: persist the suggestion so it can be reviewed and PINNED in
                # Cell 2, instead of silently changing between runs.
                try:
                    _sg, _sa = suggest_appearance_thresholds(
                        cal, REID_SIM_THRESHOLD, ANCHOR_SIM_THRESHOLD)
                    _cp = OUTPUT_DIR / f"calibration_{camera_id}{chunk_tag}.json"
                    _cp.write_text(json.dumps({
                        "backbone": method, "applied_this_run": bool(CALIBRATION_AUTO_APPLY),
                        "in_use": {"REID_SIM_THRESHOLD": thresh,
                                   "ANCHOR_SIM_THRESHOLD": run_anchor_thresh},
                        "suggested": {"REID_SIM_THRESHOLD": _sg,
                                      "ANCHOR_SIM_THRESHOLD": _sa},
                        "same_n": cal["same_n"], "diff_n": cal["diff_n"],
                        "same_p10": cal["same_p10"], "diff_p90": cal["diff_p90"],
                    }, indent=2, default=str))
                    if not CALIBRATION_AUTO_APPLY:
                        print(f"   \U0001f4cc F5: thresholds FROZEN at "
                              f"{thresh}/{run_anchor_thresh}. This run suggests "
                              f"{_sg}/{_sa} -> {_cp.name}. Pin it in Cell 2 only "
                              f"after a scored A/B, never automatically.")
                except Exception as _cex:
                    print(f"   (calibration not persisted: {_cex})")
                # (v36) show the ACTUAL crops behind the worst-separating
                # pairs -- a percentile number can't tell you WHY same/
                # different-person similarity overlaps; the images usually
                # can (too small/blurry to see detail, near-identical
                # uniforms, bad angle, heavy occlusion...).
                if cal.get("same_pairs_worst"):
                    plot_reid_pair_audit(
                        cal["same_pairs_worst"], track_crops,
                        f"{camera_id}: spatially-certain SAME person, "
                        f"LOWEST appearance similarity (should be high)")
                if cal.get("diff_pairs_worst"):
                    plot_reid_pair_audit(
                        cal["diff_pairs_worst"], track_crops,
                        f"{camera_id}: certainly DIFFERENT people, "
                        f"HIGHEST appearance similarity (should be low)")

            
            mapping, merge_edges, merge_diagnostics = merge_fragmented_tracks(
                windows, embeddings, thresh, REID_MAX_GAP_S,
                positions=positions,
                handoff_gap_s=REID_HANDOFF_GAP_S,
                handoff_px=REID_HANDOFF_PX,
                role_hint=role_hint,
                stationary_px=REID_STATIONARY_PX,
                anchor_embeddings=anchor_embeddings,
                anchor_sim_threshold=run_anchor_thresh,
                raw_crops=(raw_crops if _attire_on else None),
                plane=_ground, handoff_m=REID_HANDOFF_M,
                stationary_m=REID_STATIONARY_M,
                hsv_sim_threshold=HSV_MERGE_SIM_THRESHOLD,
                face_embeddings=(face_embeddings if ENABLE_FACE_MERGE_TIER else None),
                face_sim_threshold=FACE_MERGE_SIM_THRESHOLD,
                ir_hint=_ir_frac)
            _n_ident = len(set(mapping.values()))
            if ENABLE_GLOBAL_TRACKLET and _n_ident > GLOBAL_TRACKLET_MAX_IDS:
                print(f"\U0001F310 global tracklet pass SKIPPED: {_n_ident} identities "
                      f"> GLOBAL_TRACKLET_MAX_IDS={GLOBAL_TRACKLET_MAX_IDS} (the linker "
                      f"builds a dense {2*_n_ident}x{2*_n_ident} cost matrix — it would "
                      f"eat the RAM and never finish). Greedy tier merges still ran.")
            elif ENABLE_GLOBAL_TRACKLET:
                _gbefore = len(set(mapping.values()))
                mapping = _apply_global_tracklet_pass(
                    mapping, windows, positions, anchor_embeddings, thresh)
                _gafter = len(set(mapping.values()))
                print(f"\U0001F310 global tracklet pass: {_gbefore} -> {_gafter} "
                      f"identities ({_gbefore - _gafter} extra long-range merge(s))")
            if ENABLE_ATTIRE_MERGE_TIER or ENABLE_FACE_MERGE_TIER:
                _tiers_on = ", ".join(
                    t for t, on in (("attire/HSV", ENABLE_ATTIRE_MERGE_TIER),
                                    ("face", ENABLE_FACE_MERGE_TIER)) if on)
                print(f"🧷 merge tiers active this run: appearance/anchor "
                      f"(thresh={thresh}/{run_anchor_thresh}), "
                      f"hand-off/stationary, {_tiers_on} "
                      f"(hsv>={HSV_MERGE_SIM_THRESHOLD}, "
                      f"face>={FACE_MERGE_SIM_THRESHOLD})")

            # (v38) per-tier accepted-merge breakdown + rejection diagnostics
            # — answers "which lever is actually pulling weight" directly,
            # instead of inferring it from before/after fragment counts.
            if merge_diagnostics:
                _tc = merge_diagnostics["tier_counts"]
                if _tc:
                    _tc_str = ", ".join(f"{t}={n}" for t, n in
                                        sorted(_tc.items(), key=lambda kv: -kv[1]))
                    print(f"   📊 merges by tier: {_tc_str}")
                _rc = merge_diagnostics["role_conflicts_blocked"]
                _oc = merge_diagnostics["overlap_blocked_count"]
                if _rc or _oc:
                    print(f"   🧾 blocked candidates: {_rc} by role conflict "
                          f"(staff vs customer), {_oc} by time-window overlap "
                          f"(evidence said match, but groups were already "
                          f"co-visible under a different id)")
                if merge_diagnostics["overlap_blocked_samples"]:
                    print(f"   ℹ️  strongest overlap-blocked candidates "
                          f"(evidence was good, order/window said no):")
                    for sim, a, b, tier in merge_diagnostics["overlap_blocked_samples"][:5]:
                        print(f"      ID {a} <-> ID {b}: tier={tier} score={sim:.3f}")

            # ── FACE VETO (v32) ─────────────────────────────────────────────
            # Unlike the corroboration/cross-validation below (print-only),
            # this ACTIVELY reverses a merge when face evidence confidently
            # contradicts it — but only on the SPECIFIC direct edge that
            # caused the merge, and only when that edge's own evidence was
            # appearance-only (not hand-off/stationary/anchor tier, which is
            # independent of appearance and shouldn't be overridden by a
            # single face signal — see apply_face_veto docstring for why
            # v31's group-wide check caused visible ID flicker on evidence
            # that was actually correct). Runs before n_merged/remap_events
            # so every downstream count (id_merges, self-audit, exported
            # events) reflects the post-veto mapping.
            face_veto_flagged = None
            if ENABLE_FACE_VETO and face_embeddings:
                try:
                    mapping, face_veto_report, face_veto_flagged = apply_face_veto(
                        mapping, merge_edges, face_embeddings, windows,
                        sim_threshold=FACE_SIM_THRESHOLD,
                        veto_margin=FACE_VETO_MARGIN,
                        max_edge_score_for_veto=FACE_VETO_MAX_EDGE_SCORE)
                    if face_veto_report:
                        print(f"🚫 Face veto: reversed {len(face_veto_report)} "
                              f"merge(s) on a confident face mismatch "
                              f"(appearance-only evidence, no hand-off/"
                              f"stationary/anchor backing):")
                        for a, b, sim, escore in face_veto_report:
                            print(f"      ID {a} <-> ID {b}: face_sim={sim:.3f} "
                                  f"(edge evidence score {escore:.3f})")
                    if face_veto_flagged:
                        print(f"   ℹ️  {len(face_veto_flagged)} merge(s) had a "
                              f"confident face mismatch but were kept — the "
                              f"merge evidence was hand-off/stationary/anchor "
                              f"tier (independent of appearance), which "
                              f"outranks a single face signal; still worth a "
                              f"manual look:")
                        for a, b, sim, escore in face_veto_flagged:
                            print(f"      ID {a} <-> ID {b}: face_sim={sim:.3f} "
                                  f"(edge evidence score {escore:.3f})")
                except Exception as veto_exc:
                    print(f"(Face veto skipped: {veto_exc})")

            # C2: gallery-sweep pins — a fragment whose face matched an
            # enrolled staff member IS that staff member, overriding whatever
            # the appearance tiers concluded; its whole merge-group follows.
            for _tid, _sn in _sweep_hits.items():
                for _k, _v in list(mapping.items()):
                    if _v == _tid:
                        mapping[_k] = _sn
                mapping[_tid] = _sn
            n_merged = sum(1 for k, v in mapping.items() if k != v)

            # (v38) persistent identity memory — one dossier per confirmed
            # person, pooling every evidence snapshot (face / hand-off /
            # stationary / anchor / attire / plain-gallery) collected across
            # their merged fragments. This is what keeps an id FIXED: it's
            # the memory match_track_to_dossiers cross-verifies future
            # sightings against, instead of trusting a single signal.
            if mapping:
                try:
                    identity_dossiers = build_identity_dossiers(
                        mapping, merge_edges, windows,
                        embeddings=embeddings,
                        anchor_embeddings=anchor_embeddings,
                        face_embeddings=face_embeddings,
                        raw_crops=raw_crops, positions=positions)
                    _eq = Counter(t for d in identity_dossiers.values()
                                 for t in d["evidence_tiers_used"])
                    _eq_str = (", ".join(f"{t}={n}" for t, n in
                                         sorted(_eq.items(), key=lambda kv: -kv[1]))
                              if _eq else "none (all single-fragment people)")
                    print(f"🗂️  identity memory: {len(identity_dossiers)} "
                          f"person dossier(s) built (confirming evidence — "
                          f"{_eq_str})")
                except Exception as dossier_exc:
                    import traceback
                    print(f"(Identity dossier build skipped: {dossier_exc})")
                    traceback.print_exc()
                    identity_dossiers = {}

            # DEBUG — self-selecting: find close-but-unmerged fragment pairs
            _unmerged_near = []
            _dbg_ids = sorted(windows, key=lambda k: windows[k][0])[:400]  # v55: was every pair
            for a in _dbg_ids:
                for b in _dbg_ids:
                    if str(a) >= str(b) or mapping.get(a) == mapping.get(b, b):
                        continue
                    pa, pb = positions.get(a), positions.get(b)
                    if not pa or not pb:
                        continue
                    d = math.hypot(pa[1][0] - pb[0][0], pa[1][1] - pb[0][1])
                    g = windows[b][0] - windows[a][1]
                    if 0 <= g <= REID_MAX_GAP_S and d <= 150:
                        _unmerged_near.append((d, g, a, b))
            for d, g, a, b in sorted(_unmerged_near, key=lambda x: (x[0], x[1], str(x[2]), str(x[3])))[:5]:
                print(f"DEBUG unmerged near-pair: {a}->{b}  dist={d:.1f}px  gap={g:.1f}s")

            if n_merged:
                events, crossings, _ = remap_events(events, crossings, mapping)
                # F2: remap_events recomputes role from fragment majority-dwell
                # — which voted "customer" during the live loop. A face match
                # against the ENROLLED gallery outranks dwell: force it, or the
                # staff member C2 just rescued is counted as a guest all night.
                _pinned = set(_sweep_hits.values())
                if _pinned:
                    events = [dict(e, role="staff")
                              if e["track_id"] in _pinned else e
                              for e in events]
                # roles may change after merge (fragments pool their dwell)
                if staff_zones_here:
                    events = apply_staff_zone_override(
                        events, staff_zones_here, STAFF_OVERRIDE_MIN_S,
                        min_share_dwell_s=STAFF_OVERRIDE_MIN_S)
                id_merges = {k: v for k, v in mapping.items() if k != v}
                print(f"🔗 Re-ID stitching ({method} + hand-off, "
                      f"sim>={thresh}): {len(windows)} track fragments -> "
                      f"{len(set(mapping.values()))} people "
                      f"({n_merged} merges)")

                # ── AUTOMATIC CROSS-VALIDATION (v26) ────────────────────────
                # Independently re-check every OSNet merge with a second,
                # unrelated feature space (HSV histogram) on the SAME banked
                # crops from this run — track IDs line up exactly, no manual
                # cross-referencing against a separately re-tracked video.
                # Diagnostic only: never touches `mapping` or downstream
                # events, just flags what's worth a manual look.
                if ENABLE_CROSS_VALIDATION:
                    try:
                        xval = cross_validate_reid(
                            mapping, track_crops, windows,
                            gallery_sim_threshold=CROSS_VAL_GALLERY_SIM,
                            anchor_sim_threshold=CROSS_VAL_ANCHOR_SIM,
                            max_gap_s=REID_MAX_GAP_S)
                        n_agree = len(xval["agree"])
                        n_disagree = len(xval["disagree"])
                        n_hsv_only = len(xval["hsv_only"])
                        n_checked = n_agree + n_disagree
                        if n_checked:
                            print(f"🔍 Cross-validation (independent HSV check): "
                                  f"{n_agree}/{n_checked} {method.upper()} merges corroborated")
                        if n_disagree:
                            print(f"   ⚠️  {n_disagree} {method.upper()} merge(s) NOT "
                                  f"corroborated by HSV — review these:")
                            for a, b, gs, asim in sorted(
                                    xval["disagree"],
                                    key=lambda r: -max(r[2], r[3]))[:5]:
                                print(f"      ID {a} <-> ID {b}: "
                                      f"gallery_sim={gs:.3f} anchor_sim={asim:.3f}")
                        if n_hsv_only:
                            print(f"   ℹ️  {n_hsv_only} additional candidate(s) "
                                  f"HSV flagged that {method.upper()} did NOT merge:")
                            for a, b, gs, asim in sorted(
                                    xval["hsv_only"],
                                    key=lambda r: -max(r[2], r[3]))[:5]:
                                print(f"      ID {a} <-> ID {b}: "
                                      f"gallery_sim={gs:.3f} anchor_sim={asim:.3f}")
                    except Exception as xval_exc:
                        import traceback
                        print(f"(Cross-validation skipped: {xval_exc})")
                        traceback.print_exc()

                # ── FACE CROSS-VALIDATION (v30) ─────────────────────────────
                # Independent of both the appearance backbone AND HSV — where
                # a face resolved for both sides of a merge, this is the
                # strongest available check. Sparse by nature; print-only,
                # never touches `mapping`.
                if ENABLE_FACE_CORROBORATION and face_embeddings:
                    try:
                        fval = cross_validate_faces(
                            mapping, face_embeddings, windows,
                            sim_threshold=FACE_SIM_THRESHOLD,
                            max_gap_s=REID_MAX_GAP_S)
                        f_agree, f_disagree = len(fval["agree"]), len(fval["disagree"])
                        f_checked = f_agree + f_disagree
                        if f_checked:
                            print(f"👤 Face cross-validation: {f_agree}/"
                                  f"{f_checked} merges had a face on both "
                                  f"sides and matched")
                        if f_disagree:
                            print(f"   ⚠️  {f_disagree} merge(s) had a face on "
                                  f"both sides that did NOT match — strongest "
                                  f"available contradiction signal, review "
                                  f"these first:")
                            for a, b, sim in sorted(fval["disagree"],
                                                    key=lambda r: r[2])[:5]:
                                print(f"      ID {a} <-> ID {b}: face_sim={sim:.3f}")
                        if fval["face_only"]:
                            print(f"   ℹ️  {len(fval['face_only'])} unmerged "
                                  f"pair(s) had matching faces but were never "
                                  f"merged by {method} — possible missed "
                                  f"matches (e.g. person changed clothes):")
                            for a, b, sim in sorted(fval["face_only"],
                                                    key=lambda r: -r[2])[:5]:
                                print(f"      ID {a} <-> ID {b}: face_sim={sim:.3f}")
                        face_validation_report = fval
                    except Exception as fval_exc:
                        import traceback
                        print(f"(Face cross-validation skipped: {fval_exc})")
                        traceback.print_exc()
            else:
                print("Re-ID stitching: no fragments needed merging")
        except Exception as exc:
            import traceback
            print("=" * 78)
            print(f"\U0001f6a8 Re-ID STITCHING FAILED: {exc}")
            print("   Identity fragments were NOT merged. Every unique-people count "
                  "below is inflated — a previous run of this footage produced 22 "
                  "people with stitching working and 95 without it.")
            print("   Treat this run's Tier B numbers as INVALID until this is fixed.")
            print("=" * 78)
            traceback.print_exc()

    # D2: furniture does not fidget. The potted plant held a track for the whole
    # chunk at identical pixels and was labelled "staff" purely by zone dwell.
    _static = {}
    if ENABLE_STATIC_FILTER:
        try:
            _prot = protected_ids(crossings=crossings,
                                  face_ids=(list(_staff_seen_names)
                                            + list(_sweep_hits.values())),  # F3 protect
                                  canon=(mapping or {}))
            _static = static_track_ids(
                frame_log, canon=(mapping or {}), protected=_prot,
                min_life_s=STATIC_MIN_LIFE_S,
                max_centre_jitter=STATIC_CENTRE_JITTER,
                max_size_jitter=STATIC_SIZE_JITTER)
            if _static:
                events, crossings, frame_log = drop_tracks(
                    events, crossings, frame_log, _static,
                    canon=(mapping or {}))
                print(f"\U0001fab4 D2 dropped {len(_static)} static track(s) — "
                      f"furniture, not people:")
                for _cid, _d in sorted(_static.items(),
                                       key=lambda kv: -kv[1]["seconds"])[:5]:
                    print(f"      id {_cid} at {_d['at']} for {_d['seconds']:.0f}s, "
                          f"centre jitter {_d['centre_jitter']:.4f} of body height")
        except Exception as _sf:
            print(f"(static filter skipped: {_sf})")
    if _supp_stats.get("size_dropped"):
        _d1n, _d1s = _supp_stats["size_dropped"], _supp_stats.get("size_seen", 0)
        _d1pct = (100.0 * _d1n / _d1s) if _d1s else float("nan")
        print(f"\U0001f4cf D1 dropped {_d1n} of {_d1s} detection(s) "
              f"({_d1pct:.1f}%) too large to be a person at their own footline "
              f"(tol {SIZE_FILTER_TOL}x the predicted area)")
        if _d1s and _d1pct > 25:
            print(f"   \u26a0\ufe0f  D1 is dropping more than a quarter of every "
                  f"detection. Either the scene-geometry fit is wrong or this "
                  f"filter is eating real people — check the fit above before "
                  f"trusting any count from this run.")

    # ── D3: static phantoms whose IDS CHURN ────────────────────────────────
    # PHASE10_PHANTOM_AND_REGION_ARRIVALS. D2 above is per-track-id and is
    # structurally blind to this: the plant and the mirror re-mint a fresh id
    # every few seconds, so no single id ever lives long enough to be tested.
    # This asks the question of the LOCATION instead, and reuses drop_tracks so
    # events, crossings and frame_log stay consistent with each other.
    _phantoms = []
    if globals().get("ENABLE_PHANTOM_FILTER", True):
        try:
            _pprot = protected_ids(crossings=crossings,
                                   face_ids=(list(_staff_seen_names)
                                            + list(_sweep_hits.values())),  # F3 protect
                                   canon=(mapping or {}))
            _phantoms = phantom_regions(
                frame_log, frame_wh=(frame_w, frame_h), protected=_pprot,
                min_span_s=PHANTOM_MIN_SPAN_S,
                max_centre_jitter=PHANTOM_CENTRE_JITTER,
                max_size_cv=PHANTOM_SIZE_CV)
            if _phantoms:
                _pbox = {}
                for _fi, _t, _bs in frame_log:
                    for _tid, _a, _b, _c2, _d2 in _bs:
                        _pbox.setdefault(mapping.get(_tid, _tid) if mapping
                                         else _tid, []).append((_a, _b, _c2, _d2))
                _pdrop = {}
                for _cid, _bxs in _pbox.items():
                    if _cid in _pprot:
                        continue
                    _n = len(_bxs)
                    _med = tuple(sorted(v[k] for v in _bxs)[_n // 2]
                                 for k in range(4))
                    if in_phantom(_med, _phantoms):
                        _pdrop[_cid] = {"at": (round((_med[0] + _med[2]) / 2),
                                               round((_med[1] + _med[3]) / 2)),
                                        "sightings": _n}
                if _pdrop:
                    events, crossings, frame_log = drop_tracks(
                        events, crossings, frame_log, _pdrop,
                        canon=(mapping or {}))
                print(f"\U0001faa9 D3 dropped {len(_pdrop)} id(s) sitting on "
                      f"{len(_phantoms)} static phantom region(s) — the plant/"
                      f"mirror class of false positive that D2 cannot see "
                      f"because their ids churn:")
                for _r in _phantoms[:4]:
                    print(f"      at {_r['centre']}: {_r['why']}")
        except Exception as _pf:
            print(f"(phantom filter skipped: {_pf})")

    event_ids = {e["track_id"] for e in events}
    # a genuinely-tracked person whose only zone presence was a sub-threshold
    # doorway transit has NO zone event — discarding their crossing here would
    # delete a real arrival, so tracked ids are exempt from the ghost filter.
    ghosts = sum(1 for c in crossings if c["track_id"] not in event_ids
                 and c["track_id"] not in track_time)
    if ghosts:
        crossings = [c for c in crossings if c["track_id"] in event_ids
                     or c["track_id"] in track_time]
        print(f"filtered {ghosts} ghost line-crossings (flicker tracks with "
              f"no real zone presence)")

    # v55 #6: if the entry line's endpoints are the wrong way round, IN and
    # OUT swap and nothing complains — the headline number is simply backwards.
    # Over a whole chunk of a venue that is filling or steady, inward crossings
    # should not be dwarfed by outward ones.
    _n_in = sum(1 for c in crossings if c["direction"] == "in")
    _n_out = sum(1 for c in crossings if c["direction"] == "out")

    # A line that NEVER fires is indistinguishable from an empty venue in every
    # number downstream — "0 people entered" reads like a fact. On the first full
    # hour of CAM.112 the line triggered zero times while 46 people passed through
    # the waiting zone and reception was visited 74 times. Nothing said a word.
    _movers = len({e["track_id"] for e in events})
    if (_n_in + _n_out) == 0 and _movers >= 5:
        print("=" * 78)
        print(f"\U0001f6a8 ENTRY LINE NEVER TRIGGERED \u00b7 {camera_id}")
        print(f"   {_movers} people moved through the zones and NOT ONE crossed the "
              f"entry line.")
        print("   Every arrival/exit number is therefore 0 — that is a BROKEN LINE, "
              "not a quiet venue.")
        print("   Most likely, in order:")
        print("     1. the line is too SHORT and people walk around its ends —")
        print("        it must span the full width of the doorway, end to end")
        print("     2. it is drawn in the wrong place (e.g. across a corridor "
              "people never actually cross)")
        print("     3. it is positioned where feet are occluded — the trigger "
              "anchor is the BOTTOM of the box")
        print("   Fix: redraw entry_line in the zones JSON across the real "
              "threshold, on the floor, wall to wall.")
        print("=" * 78)
    elif (_n_in + _n_out) > 0 and _movers >= 20 and (_n_in + _n_out) < _movers * 0.2:
        print(f"\u26a0\ufe0f  entry line fired only {_n_in + _n_out} time(s) for "
              f"{_movers} people in the zones — it may be clipping most arrivals.")
    if (_n_in + _n_out) >= 20 and _n_out > _n_in * 1.6:
        print("=" * 78)
        print(f"🚨 ENTRY LINE LOOKS BACKWARDS · {camera_id}: {_n_out} people "
              f"crossed OUT vs {_n_in} IN over {(t or 1) / 60:.0f} min.")
        print(f"   Unless this camera really did watch a room empty out, set "
              f"ENTRY_LINE_FLIP = {not ENTRY_LINE_FLIP} in Cell 2 and re-run. "
              f"Every 'people entered' number below is the wrong direction "
              f"until you do.")
        print("=" * 78)

    roles = {**rec.final_roles(),
             **{e["track_id"]: e["role"] for e in events}}

    # A5: TIER-A crossing dedupe — one person crossing once, wearing several
    # ids while doing it, collapses to ONE event BEFORE anything counts it.
    # tier_a_crossings existed for exactly this and was wired into nothing the
    # HUD or the report reads; both count unique ids, which is what churn
    # inflates ("entered=5" for one re-minted person). Staff crossings pass
    # through untouched (tier A is a guest counter by design).
    _crossings_raw = list(crossings)   # F1: Tier B + the 1c alarm need the
    #    UN-deduped events, or the two estimators stop being independent and
    #    the disagreement alarm goes quiet exactly when identity fragments.
    if crossings:
        _staff_c = [c for c in crossings
                    if roles.get(c["track_id"]) == "staff"]
        _, _in_k = tier_a_crossings(crossings, plane=_ground,
                                    direction="in", roles=roles)
        _, _out_k = tier_a_crossings(crossings, plane=_ground,
                                     direction="out", roles=roles)
        _deduped = sorted(_staff_c + _in_k + _out_k, key=lambda c: c["t"])
        if len(_deduped) < len(crossings):
            print(f"tier-A crossing dedupe: {len(crossings)} raw crossing "
                  f"event(s) -> {len(_deduped)} (id churn at the line "
                  f"collapsed before counting)")
        crossings = _deduped
    print(f"{camera_id}: {len(events)} zone events, "
          f"{len(event_ids)} tracked people")

    # ── PASS 2: render the video from final identities ─────────────────────
    out_path = OUTPUT_DIR / f"{camera_id}{chunk_tag}_annotated.mp4"
    canon = dict(mapping) if mapping else {}
    snapshots = render_annotated(
        video_path, out_path, frame_log, canon, roles, events, crossings,
        polygons, zcolors,
        (list(drawn_lines.values())[0] if drawn_lines else None),
        eff_fps, step, native_fps, zone_roles=zone_roles,
        proxy_dir=proxy_dir, duration_s=duration_s, phantoms=_phantoms)
    if proxy_dir is not None:
        shutil.rmtree(proxy_dir, ignore_errors=True)   # ~1 GB per chunk
    # RENDER_DIRECT_H264 renames the file to *_annotated_h264.mp4 INSIDE
    # render_annotated; without picking that up here the run reported
    # "annotated_video": None and the export cell shipped no video at all.
    _h264_alt = Path(out_path).with_name(Path(out_path).stem + "_h264.mp4")
    if _h264_alt.exists():
        out_path = _h264_alt
    if not Path(out_path).exists():
        # nobody was in shot for this whole chunk, so no frame was written and
        # no file exists. Say so instead of handing downstream a dead path.
        print(f"(no annotated video for {camera_id}{chunk_tag}: nobody appeared "
              f"in any analysed frame of this chunk)")
        out_path = None
    else:
        print(f"annotated video (persistent ids) -> {out_path}")

    return {
        "camera_id": camera_id, "events": events, "roles": roles,
        "is_ir": _is_ir, "duration_s": duration_s,
        "staff_matched_names": sorted(_staff_seen_names),
        "frames_skipped_idle": _n_skipped, "tracker_resets": _n_resets,
        # B1: the labelling export re-decodes at THIS rate. Falling back to
        # FPS_TARGET (8) while analysis ran at eff_fps (7.5) drifted 6.7% and
        # tripped the 5% alignment guard, so no package was ever built.
        "eval_fps": eff_fps,
        "size_dropped": _supp_stats.get("size_dropped", 0),
        "size_seen": _supp_stats.get("size_seen", 0),
        "phantom_regions": [{k: v for k, v in _r.items() if k != "box"}
                            for _r in _phantoms],
        "zone_roles": dict(zone_roles),
        "static_dropped": len(_static),
        "ir_frames": sum(1 for v in _frame_ir.values() if v),
        "ir_frames_total": len(_frame_ir),
        "ir_switches": [(round(s, 1), bool(v)) for s, v in _ir_switches],
        "perspective_fit": _persp.describe(),
        "ground_plane": _ground.describe(),
        "ground_mode": _ground.mode,
        "perspective_coeffs": (list(_persp._refit()) if _persp.ready else None),
        "frame_size_analysed": [frame_w, frame_h],
        "carried_suppressed": _supp_stats.get("carried_suppressed", 0),
        "heads_without_person": _n_head_only,
        "live_max_dist_px": round(_live_max_dist, 1),
        "crossings": crossings,
        "crossings_raw": _crossings_raw,   # F1
        "t_end": t + frame_step_s,
        "custom_model": False,   # kept for downstream compat; always False now
        "line_in": (sum(lz.in_count for lz in line_zones.values())
                    if line_zones else None),
        "line_out": (sum(lz.out_count for lz in line_zones.values())
                     if line_zones else None),
        "per_line": {n: {"in": lz.in_count, "out": lz.out_count}
                     for n, lz in line_zones.items()},
        # U1: the verdict travels WITH the run, so the report cannot forget it
        "view_valid": bool(_view.get("valid", True)),
        "view_reasons": list(_view.get("reasons", [])),
        "view_checked": bool(_view.get("checked", False)),
        "snapshots": snapshots, "zcolors": zcolors,
        "id_merges": id_merges,
            "seated_ratios": {tid: round(track_seated_frames.get(tid, 0) / max(track_total_frames.get(tid, 1), 1), 2) for tid in set(list(track_seated_frames.keys()) + list(track_total_frames.keys()))},
        "online_reid": (_REID_STATE.get("method") if online_tracker is not None
                        else "auto-proxy" if use_online_tracker else None),
        "annotated_video": (str(out_path) if out_path else None),
        # v53: raw per-frame boxes + final identity mapping, so an offline
        # audit can reconstruct exactly who held which id in every frame
        "frame_log": frame_log,
        "canon_map": dict(mapping) if mapping else {},
        # (v31) diagnostic/veto reports, kept per-run so the self-audit and
        # any manual review can look back at exactly what fired without
        # re-running the pipeline. None when the corresponding feature was
        # off, skipped, or had nothing to report.
        "calibration_report": calibration_report,
        "face_validation_report": face_validation_report,
        "face_veto_report": face_veto_report,
        "face_veto_flagged": face_veto_flagged,
        # (v38) persistent per-person identity memory + the diagnostics that
        # explain which evidence tier proved/blocked each merge this run.
        "merge_diagnostics": merge_diagnostics,
        "identity_dossiers": identity_dossiers,
        # (v34) raw per-frame box centers -- already computed for rendering,
        # kept here too so a heatmap can be built without re-processing the
        # video. NOTE: this is a dwell-density VISUALIZATION built from the
        # same track positions the pipeline already has -- it does not, and
        # cannot, improve identity tracking itself (see plot_occupancy_heatmap).
        "frame_log": frame_log,
    }









In [ ]:
# Cell 8 — STEP CHECK: does the detector see people, for EVERY queued video
for v in VIDEO_QUEUE:
    if v["video"].exists():
        detection_sanity(v["video"], v["camera_id"])
    else:
        print(f"⚠️  {v['video']} not found — skipping")


In [ ]:
# Cell 9 — PROCESS EVERY QUEUED VIDEO, GENERICALLY (answers 1-7, whichever
# apply to that venue). Each video answers whichever questions its OWN zone
# roles support, discovered from its own zones_<stem>.json.
# v55: the answer-building is a function so the multi-chunk runner (Cell 9b)
# produces IDENTICAL answers from the merged night, and this cell no longer
# reprocesses chunk 1 when 9b is about to process every chunk anyway.
import pandas as pd

runs = []
VENUE_META = {}          # camera_id -> {wait_zones, staff_zones, seating_zones, zone_roles}
ANSWERS_BY_VENUE = {}     # camera_id -> {question: answer}
ANSWERS_DF_BY_VENUE = {}  # camera_id -> full answers DataFrame (question/answer/evidence)
PER_PARTY_BY_VENUE = {}   # camera_id -> per-party service rows


def answer_for_run(run, zones_path, show=True):
    """Every question this venue's zones can support, + the evidence line."""
    camera_id = run["camera_id"]
    zone_names = set(load_zone_config(zones_path)[0])
    zone_roles = classify_zones(zone_names)
    wait_zones    = zones_with_role(zone_names, "wait")
    staff_zones   = zones_with_role(zone_names, "staff")
    seating_zones = zones_with_role(zone_names, "seating")
    VENUE_META[camera_id] = {"wait_zones": wait_zones, "staff_zones": staff_zones,
                             "seating_zones": seating_zones, "zone_roles": zone_roles}
    events = run["events"]
    rows = []

    # TIER A — geometry over a ~3 s horizon. Does not depend on identity
    # holding together for the night, so this is the defensible number.
    _plane_a = _rebuild_plane(run)
    q1a, _kept = tier_a_crossings(run["crossings"], plane=_plane_a,
                                  roles=run["roles"])
    # TIER B — unique identities after the whole-night stitch. Same question,
    # but it inherits every Re-ID error over ten hours.
    # F1: Tier B must count from the RAW crossings — the deduped list has
    # already collapsed churned ids, so counting unique ids from it would
    # be Tier A wearing Tier B's label and the 1c alarm could never fire.
    q1b = entered_count(run.get("crossings_raw") or run["crossings"],
                        run["roles"])
    rows.append(("1. People entered (TIER A · geometry)", q1a,
                 f"de-duplicated crossing events within {6.0}s and "
                 f"{'1.2 m' if _plane_a.ok else '140 px'} — independent of long-term identity"))
    rows.append(("1b. People entered (TIER B · identity)", q1b,
                 f"unique stitched identities that crossed inward "
                 f"(raw in={run['line_in']}, out={run['line_out']})"))
    if q1a and abs(q1b - q1a) / max(q1a, 1) > 0.15:
        rows.append(("1c. !! Tier A vs Tier B disagree",
                     f"{q1a} vs {q1b}",
                     "identity is fragmenting or over-merging at the door — "
                     "trust Tier A and treat the guest count as a RANGE"))
    _groups = detect_groups(_kept, plane=_plane_a)
    if _groups:
        rows.append(("1d. Arriving parties (groups)", len(_groups),
                     "arrivals within 25 s and 3 m of each other, chained; "
                     f"sizes {sorted((g['size'] for g in _groups), reverse=True)[:8]}"))
    # U5: ENTRY_LINE_FLIP is a hand-set boolean that inverts this whole table
    # when wrong. Measure it from the footage every run and say so.
    _interior = set(zone_names) - {z for z, rs in zone_roles.items() if "entry" in rs}
    _flip, _conf, _ev = infer_entry_direction(run["crossings"], events, _interior)
    if _flip is True:
        rows.append(("1e. \U0001f6a8 ENTRY LINE LOOKS BACKWARDS", "IN/OUT SWAPPED",
                     f"{_ev['tracks_with_evidence']} people crossed and then dwelled "
                     f"on the WRONG side of the crossing "
                     f"(net {_ev['net_dwell_after_minus_before_s']}s). Set "
                     f"camera.entry_line_flip = "
                     f"{not globals().get('ENTRY_LINE_FLIP', True)} in the profile "
                     f"and re-run — every arrival number above is inverted."))
    elif _flip is False:
        rows.append(("1e. entry line direction", "confirmed by the footage",
                     f"{_ev['tracks_with_evidence']} people dwelled inside on the "
                     f"expected side of their crossing"))
    _per_line = run.get("per_line") or {}
    if len(_per_line) > 1:
        rows.append(("1f. Per door", len(_per_line),
                     "; ".join(f"{n}: in={d['in']} out={d['out']}"
                               for n, d in sorted(_per_line.items()))))
    q1 = q1a

    if seating_zones:
        q2 = seated_count(events, seating_zones, MIN_SEATED_S)
        rows.append(("2. Got seated", q2, f"customer dwell >= {MIN_SEATED_S}s in {sorted(seating_zones)}"))

    for wz in sorted(wait_zones):
        q3, wait_dist = waited_over(events, wz, WAIT_THRESHOLD_S)
        rows.append((f"3. Waited > {mmss(WAIT_THRESHOLD_S)} ({wz})", q3,
                     f"{len(wait_dist)} people used {wz}"))

    for sz in sorted(staff_zones):
        q4 = reception_absence(events, sz, 0, run["t_end"])
        n_visits = sum(1 for e in events if e["zone"] == sz and e["role"] == "staff")
        rows.append((f"4. {sz} unattended (s)", q4["total_away_s"],
                     f"longest gap {q4['longest_away_s']}s; staff present at {sz} "
                     f"{n_visits} separate times"))

    per_party, avgs = [], {}
    if seating_zones:
        per_party, avgs = table_service_metrics(events, seating_zones, PARTY_GAP_S,
                                                MIN_PARTY_S, VISIT_MIN_S)
        rows.append(("5. Avg seating -> order (s)", avgs.get("avg_seating_to_order_s"),
                     f"PROXY: first staff visit >= {VISIT_MIN_S}s after party sits"))
        rows.append(("6. Avg order -> food (s)", avgs.get("avg_order_to_food_s"),
                     "PROXY: second distinct staff visit"))
        rows.append(("7. Avg server visits / party", avgs.get("avg_visits_per_party"),
                     f"across {avgs.get('n_parties', 0)} parties"))
    PER_PARTY_BY_VENUE[camera_id] = per_party

    answers_df = pd.DataFrame(rows, columns=["question", "answer", "evidence"])
    if show:
        display(answers_df.style.hide(axis="index"))
    ANSWERS_BY_VENUE[camera_id] = {r.question: r.answer for r in answers_df.itertuples()}
    ANSWERS_DF_BY_VENUE[camera_id] = answers_df
    return answers_df


_MULTI = len(globals().get("QUEUE_REMOTE", [])) > 1
if globals().get("PEAK_ONLY"):
    # v55: with a SINGLE chunk queued this cell used to process the whole hour
    # and Cell 9c then analysed the peak window on top of it — the full cost
    # plus the peak. In peak mode nothing is processed here.
    print(f"PEAK_ONLY is on — Cell 9c analyses only the busiest "
          f"{globals().get('PEAK_WINDOW_MIN', 20)} minutes. Nothing processed here.")
elif globals().get("RUN_ABLATION"):
    print("RUN_ABLATION=True — skipping the full single-video pass; "
          "Cell 9e re-runs only the labelled windows.")
    runs = []
elif _MULTI:
    print(f"multi-chunk night ({len(QUEUE_REMOTE)} chunks) — Cell 9b does the "
          f"processing. Skipping the single-video pass so chunk 1 is not "
          f"processed twice.")
else:
    for v in VIDEO_QUEUE:
        camera_id, video_path, zones_path = v["camera_id"], v["video"], v["zones_path"]
        if not video_path.exists():
            print(f"⚠️  {camera_id}: video missing, skipping")
            continue
        print("=" * 70); print(camera_id); print("=" * 70)
        _w0 = float(globals().get("WINDOW_START_MIN") or 0) * 60.0
        _wl = globals().get("WINDOW_MIN")
        _max = (float(_wl) * 60.0) if _wl else globals().get("PROVE_SECONDS")
        if _wl:
            print(f"\u23f1  fixed window: {mmss(_w0)} -> {mmss(_w0 + _max)} "
                  f"of this video ({_wl} min)")
        globals()["_RENDER_INDEX_SHIFT"] = 0.0
        run = process_video(camera_id, video_path, zones_path,
                            max_seconds=_max, start_seconds=_w0)
        if _wl:
            run["chunk_bounds"] = [(0, _w0, _w0 + _max)]
            run["annotated_videos"] = ([run["annotated_video"]]
                                       if run.get("annotated_video") else [])
        show_gallery(run["snapshots"], f"{camera_id.upper()} — annotated snapshots", ncols=3)
        runs.append(run)
        answer_for_run(run, zones_path)
    print(f"\n✅ processed {len(runs)}/{len(VIDEO_QUEUE)} video(s)")


In [ ]:
# Cell 9b — MULTI-CHUNK RUNNER · real 2-GPU · resume · seam + whole-night identity
# One chunk = one hour = one immutable artifact on disk. Re-running a chunk is
# free; a crash costs one chunk, not the night.
import threading, queue, json, os, math, gzip, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

def _json_safe(o):
    """PATCH_V63_JSONSAFE: default=list killed run 2 — list(np.uint8) raises
    "not iterable". Scalars first, arrays second, iterables third, str last."""
    if getattr(o, "ndim", None) == 0 and hasattr(o, "item"):
        return o.item()
    if hasattr(o, "tolist"):
        return o.tolist()
    try:
        return list(o)
    except TypeError:
        return str(o)


SEAM_WINDOW_S   = 90.0
SEAM_FACE_SIM   = globals().get("FACE_MERGE_SIM_THRESHOLD", 0.45)
SEAM_ANCHOR_SIM = globals().get("ANCHOR_SIM_THRESHOLD", 0.75)
SEAM_GALLERY_SIM = globals().get("REID_SIM_THRESHOLD", 0.60)

# ── whole-night guest identity ──────────────────────────────────────────────
# The seam window only catches someone who was on screen at the chunk boundary.
# A guest who steps out at 18:50 and comes back at 19:40 is TWO people to the
# seam rule — and "count each guest once even if they leave and return" is the
# requirement. So after the seam pass, every guest who crossed the door is
# matched against every other guest across the whole night, on face first and
# best-crop appearance second, with deliberately strict bars: a wrong merge
# here under-counts guests, which is worse than a missed one.
GLOBAL_GUEST_LINK = True
GUEST_GLOBAL_GAP_S = 5400.0     # 90 min — a meal, a smoke, a walk round the block
GUEST_FACE_SIM     = 0.50       # stricter than the seam's 0.45
GUEST_ANCHOR_SIM   = 0.82       # stricter than the merge tier's 0.75

# V64b: a crashed/re-run session leaves the previous attempt's raw videos
# and proxy dirs behind — run 2 carried a 1.5GB corpse from run 1 into a
# 19.5GB disk. Finished *_h264.mp4 files are kept; everything re-creatable
# is swept.
_swept = 0
for _stale in list(OUTPUT_DIR.glob("*_annotated.mp4")):
    _swept += _stale.stat().st_size
    _stale.unlink(missing_ok=True)
# proxy dirs are named "_proxy_<cam><tag>" (see process_video) — the old
# "proxy_*" glob matched nothing, leaving ~2-3GB of JPEGs per crashed chunk
for _spd in list(OUTPUT_DIR.glob("_proxy_*")):
    if _spd.is_dir():
        shutil.rmtree(_spd, ignore_errors=True)
if _swept:
    print(f"🧹 swept {_swept // 1_000_000} MB of stale raw video from a "
          f"previous attempt")

EVENTS_DIR = OUTPUT_DIR / "chunk_events"
EVENTS_DIR.mkdir(parents=True, exist_ok=True)


def _offset_for(name, first_clock):
    c = _clock_from_name(name)
    if not c or not first_clock:
        return None
    return (c - first_clock).total_seconds()


def _ns(tid, k):
    if isinstance(tid, str) and not tid.isdigit():
        return tid          # gallery-named staff keep ONE identity all night
    return f"c{k}_{tid}"


def _slim(d):
    """Only what identity matching needs — an embedding gallery per person for
    10 chunks would not fit in RAM."""
    return {"person_id": d.get("person_id"), "t_first": d["t_first"], "t_last": d["t_last"],
            "anchor_embedding": d.get("anchor_embedding"),
            "face_embedding": d.get("face_embedding")}


def _seam_dossiers(dossiers, off, k, t_chunk_end, side):
    out = {}
    for cid, d in (dossiers or {}).items():
        if isinstance(cid, str) and not str(cid).isdigit():
            continue
        t_first, t_last = d["t_first"] + off, d["t_last"] + off
        near = (t_last >= off + t_chunk_end - SEAM_WINDOW_S) if side == "late" \
            else (t_first <= off + SEAM_WINDOW_S)
        if not near:
            continue
        out[_ns(cid, k)] = {**d, "person_id": _ns(cid, k),
                            "t_first": t_first, "t_last": t_last}
    return out


def _all_dossiers(dossiers, off, k):
    out = {}
    for cid, d in (dossiers or {}).items():
        if isinstance(cid, str) and not str(cid).isdigit():
            continue
        out[_ns(cid, k)] = _slim({**d, "person_id": _ns(cid, k),
                                  "t_first": d["t_first"] + off,
                                  "t_last": d["t_last"] + off})
    return out


def stitch_seam(late, early, roles):
    cands = []
    for nid, nd in early.items():
        for oid, od in late.items():
            if nd["t_first"] < od["t_last"] - 1.0:
                continue
            if roles.get(nid) and roles.get(oid) and roles[nid] != roles[oid]:
                continue
            if (nd.get("face_embedding") is not None
                    and od.get("face_embedding") is not None):
                s = _cosine(nd["face_embedding"], od["face_embedding"])
                if s >= SEAM_FACE_SIM:
                    cands.append((0.92 + 0.07 * s, "face", nid, oid, s))
                    continue
            if (nd.get("anchor_embedding") is not None
                    and od.get("anchor_embedding") is not None):
                s = _cosine(nd["anchor_embedding"], od["anchor_embedding"])
                if s >= SEAM_ANCHOR_SIM:
                    cands.append((0.80 + 0.10 * s, "anchor", nid, oid, s))
                    continue
            if nd.get("appearance_gallery") and od.get("appearance_gallery"):
                s = _gallery_sim(nd["appearance_gallery"], od["appearance_gallery"])
                if s >= SEAM_GALLERY_SIM:
                    cands.append((0.60 + 0.15 * s, "gallery", nid, oid, s))
    alias, used_new, used_old = {}, set(), set()
    for _score, tier, nid, oid, s in sorted(cands, key=lambda c: -c[0]):
        if nid in used_new or oid in used_old:
            continue
        alias[nid] = oid
        used_new.add(nid); used_old.add(oid)
        print(f"    stitched {nid} -> {oid}  ({tier} {s:.2f})")
    return alias


def link_returning_guests(dossiers, guest_ids, roles):
    """Same guest, hours apart -> one person. Greedy best-evidence-first, one
    partner each, no time-overlap. Returns {later_id: earlier_id}."""
    ids = [i for i in guest_ids if i in dossiers]
    ids.sort(key=lambda i: dossiers[i]["t_first"])
    cands = []
    for a_i, a in enumerate(ids):
        da = dossiers[a]
        for b in ids[a_i + 1:]:
            db = dossiers[b]
            gap = db["t_first"] - da["t_last"]
            if gap < 0:
                continue            # co-visible: cannot be one body
            if gap > GUEST_GLOBAL_GAP_S:
                break               # sorted by start: every later b is further
            if roles.get(a) == "staff" or roles.get(b) == "staff":
                continue
            fa, fb = da.get("face_embedding"), db.get("face_embedding")
            if fa is not None and fb is not None:
                s = _cosine(fa, fb)
                if s >= GUEST_FACE_SIM:
                    cands.append((2.0 + s, "face", b, a, s)); continue
            aa, ab = da.get("anchor_embedding"), db.get("anchor_embedding")
            if aa is not None and ab is not None:
                s = _cosine(aa, ab)
                if s >= GUEST_ANCHOR_SIM:
                    cands.append((1.0 + s, "anchor", b, a, s))
    alias, used_a, used_b = {}, set(), set()
    for _sc, tier, later, earlier, s in sorted(cands, key=lambda c: -c[0]):
        if later in used_a or earlier in used_b:
            continue
        alias[later] = earlier
        used_a.add(later); used_b.add(earlier)
        print(f"    same guest returned: {later} -> {earlier}  ({tier} {s:.2f})")
    return alias


def _resolve_chain(alias, t):
    seen = set()
    while t in alias and t not in seen:
        seen.add(t); t = alias[t]
    return t


def _apply_alias(events, crossings, roles, frame_log, alias):
    if not alias:
        return events, crossings, roles, frame_log
    R = lambda x: _resolve_chain(alias, x)
    events = [{**e, "track_id": R(e["track_id"])} for e in events]
    crossings = [{**c, "track_id": R(c["track_id"])} for c in crossings]
    roles = {R(t): v for t, v in roles.items()}
    frame_log = [(fi, t, [(R(b[0]),) + tuple(b[1:]) for b in bx])
                 for fi, t, bx in frame_log]
    return events, crossings, roles, frame_log


def run_multichunk(camera_id, zones_path, remote, first_clock):
    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    max_workers = min(n_gpus, 2) if n_gpus >= 2 else 1
    print(f"⚡ {n_gpus} GPU(s) -> {max_workers} parallel chunk worker(s)")

    chunk_tasks = [(k, f, _offset_for(f["name"], first_clock))
                   for k, f in enumerate(remote)]
    _t0 = time.time()

    def _process_one_chunk(task_item):
        # v55: one bad chunk must not kill the night. Log it, skip it, carry on —
        # the report then covers 4 of 5 hours instead of nothing at all.
        try:
            return _process_one_chunk_inner(task_item)
        except Exception as _chunk_exc:
            import traceback
            k = task_item[0]
            print("=" * 70)
            print(f"❌ CHUNK {k} FAILED: {_chunk_exc!r}")
            traceback.print_exc()
            print(f"   skipping it; the other chunks continue. Fix and re-run "
                  f"with RESUME=True to fill this hour in.")
            print("=" * 70)
            return k, None, None, []

    def _process_one_chunk_inner(task_item):
        k, f, off = task_item
        if off is None:
            off = k * 3600.0
        ck = EVENTS_DIR / f"chunk_{k:02d}.json"
        fl = EVENTS_DIR / f"chunk_{k:02d}_frames.json.gz"
        if globals().get("RESUME") and ck.exists():
            print(f"  .. resume chunk {k} from {ck.name}")
            d = json.loads(ck.read_text())
            # v55: the frame log used to be dropped here, so greet latency and
            # the ID audit silently ran on NO frames and reported zeros.
            if fl.exists():
                with gzip.open(fl, "rt") as fh:
                    d["frame_log"] = [(a, b, [tuple(x) for x in c])
                                      for a, b, c in json.load(fh)]
            else:
                d["frame_log"] = []
                print(f"     !! no frame log for chunk {k} — greet/contact metrics "
                      f"will be blank for this hour. Delete {ck.name} to redo it.")
            return k, off, d, []

        path = _pull(f)
        gpu_id = k % max_workers if n_gpus >= 2 else 0
        worker_device = f"cuda:{gpu_id}" if n_gpus else "cpu"
        print(f"\n{'=' * 70}\n  CHUNK {k + 1}/{len(remote)} on {worker_device} "
              f"| offset +{off / 3600:.2f} h\n{'=' * 70}")
        globals()["_RENDER_INDEX_SHIFT"] = off      # index in night time
        try:
            r = process_video(camera_id, Path(path), zones_path,
                              max_seconds=globals().get("PROVE_SECONDS"),
                              device=worker_device, chunk_tag=f"_c{k:02d}")
        finally:
            globals()["_RENDER_INDEX_SHIFT"] = 0.0
        d = {
            "events": [{**e, "track_id": _ns(e["track_id"], k),
                        "t_in": e["t_in"] + off, "t_out": e["t_out"] + off}
                       for e in r["events"]],
            "crossings": [{**c, "track_id": _ns(c["track_id"], k),
                           "t": c["t"] + off} for c in r["crossings"]],
            "crossings_raw": [{**c, "track_id": _ns(c["track_id"], k),
                               "t": c["t"] + off}
                              for c in r.get("crossings_raw",
                                             r["crossings"])],
            "ir_switches": [(s + off, v)
                            for s, v in r.get("ir_switches", [])],
            "roles": {_ns(t, k): v for t, v in r["roles"].items()},
            "frame_log": [(fi, t + off,
                           [(_ns(b[0], k),) + tuple(b[1:]) for b in bx])
                          for fi, t, bx in r.get("frame_log", [])],
            "t_end": r["t_end"] + off,
            "line_in": r.get("line_in", 0) or 0,
            "line_out": r.get("line_out", 0) or 0,
            "is_ir": r.get("is_ir", False),
            "build": str(globals().get("_BUILD_ID", "?")),   # A7
            "phasea": True,   # A7: artifact provenance
            "video": None,
            "t_start": off,
            "early": _seam_dossiers(r.get("identity_dossiers"), off, k, r["t_end"], "early"),
            "late":  _seam_dossiers(r.get("identity_dossiers"), off, k, r["t_end"], "late"),
            "dossiers": _all_dossiers(r.get("identity_dossiers"), off, k),
        }
        try:
            ck.write_text(json.dumps({kk: vv for kk, vv in d.items()
                                      if kk != "frame_log"},
                                     default=_json_safe))
            with gzip.open(fl, "wt") as fh:
                json.dump(d["frame_log"], fh, default=_json_safe)
        except Exception as _save_exc:
            # V63b: a cache write must never void the work it caches. Run 2
            # processed BOTH chunks perfectly and then discarded them because
            # this very dump crashed. Results stay in memory; only resume
            # loses this chunk.
            print(f"  !! chunk {k} artifact save FAILED ({_save_exc!r}) — "
                  f"results kept in memory, run continues; RESUME will not "
                  f"have this chunk cached.")
        # v55: re-encode THIS chunk now. You can watch hour 1 while hour 4
        # is still running, and a session that dies at hour 6 still leaves 6
        # finished, playable hours behind.
        _v = r.get("annotated_video")
        if _v and str(_v).endswith("_h264.mp4"):
            # V64a: render already produced h264 directly — nothing to do
            print(f"  ✅ chunk {k} video ready NOW: {Path(_v).name} "
                  f"({Path(_v).stat().st_size // 1_000_000} MB, direct h264)")
        elif _v and Path(_v).exists():
            _h = Path(_v).with_name(Path(_v).stem + "_h264.mp4")
            try:
                import subprocess as _sp
                _rc = _sp.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(_v),
                               "-vf", f"scale='min({globals().get('RENDER_MAX_W', 1280)},iw)':-2",
                               "-c:v", "libx264", "-pix_fmt", "yuv420p",
                               "-crf", str(globals().get("RENDER_CRF", 28)), str(_h)],
                              timeout=3600).returncode
                if _rc == 0:
                    Path(_v).unlink(missing_ok=True)
                    _ix = Path(_v).with_suffix(".index.json")
                    if _ix.exists():
                        _ix.replace(_h.with_suffix(".index.json"))
                    _v = str(_h)
                    print(f"  \u2705 chunk {k} video ready NOW: {_h.name} "
                          f"({_h.stat().st_size // 1_000_000} MB) — download it "
                          f"from the Output panel without waiting for the rest")
            except Exception as _enc:
                print(f"  (chunk {k} re-encode skipped: {_enc} — raw mp4v kept)")
        d["video"] = _v
        print(f"  chunk {k}: {len(d['events'])} events, "
              f"{len({c['track_id'] for c in d['crossings'] if c['direction'] == 'in'})} "
              f"inward crossings, artifacts in {EVENTS_DIR.name}/")

        snaps = r.get("snapshots", [])[:2]
        if Path(path).exists():
            try:
                os.remove(path)
                print(f"  deleted {Path(path).name} (events + annotated video kept)")
            except Exception:
                pass
        print(f"  chunk {k} done in {(time.time() - _t0) / 60:.1f} min elapsed")
        return k, off, d, snaps

    chunk_results, snapshots_all = {}, []
    if max_workers > 1:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for fut in as_completed([ex.submit(_process_one_chunk, t) for t in chunk_tasks]):
                k, off, d, snaps = fut.result()
                if d is None:
                    continue
                chunk_results[k] = (off, d); snapshots_all.extend(snaps)
    else:
        for task in chunk_tasks:
            k, off, d, snaps = _process_one_chunk(task)
            if d is None:
                continue
            chunk_results[k] = (off, d); snapshots_all.extend(snaps)

    _failed = [k for k, _f, _o in chunk_tasks if k not in chunk_results]
    if _failed:
        print(f"\n⚠️  {len(_failed)} chunk(s) failed and are NOT in the report: "
              f"{_failed}. Every total below covers only the chunks that "
              f"succeeded.")

    events, crossings, frame_log, roles = [], [], [], {}
    crossings_raw, ir_switches_all = [], []   # F1 / F5c
    dossiers_all, videos = {}, []
    t_end, line_in, line_out = 0.0, 0, 0
    late_prev, alias_all, n_stitched = {}, {}, 0
    _prev_ir = False   # A4b: modality of the previous chunk

    for k in sorted(chunk_results):
        off, d = chunk_results[k]
        if k and late_prev and d.get("early"):
            _early = d["early"]
            if bool(d.get("is_ir")) != _prev_ir:
                # A4b: colour<->IR seam — body appearance across the modality
                # change is noise (VI-ReID SOTA ~70% R1; NVRs don't try).
                # Stripping one side is enough: every seam tier needs BOTH.
                _early = {i: {**v, "anchor_embedding": None,
                              "appearance_gallery": None}
                          for i, v in _early.items()}
                print("    colour<->IR seam: stitching on FACE only")
            alias = stitch_seam(late_prev, _early, {**roles, **d["roles"]})
            if alias:
                n_stitched += len(alias)
                alias_all.update(alias)
                d["events"], d["crossings"], d["roles"], d["frame_log"] = _apply_alias(
                    d["events"], d["crossings"], d["roles"], d.get("frame_log", []), alias_all)
                d["dossiers"] = {_resolve_chain(alias_all, i): v
                                 for i, v in d.get("dossiers", {}).items()}
            else:
                print("    no identities spanned this seam")
        late_prev = d.get("late", {})
        _prev_ir = bool(d.get("is_ir"))   # A4b
        events += d["events"]; crossings += d["crossings"]
        crossings_raw += d.get("crossings_raw", d["crossings"])
        ir_switches_all += d.get("ir_switches", [])
        frame_log += d.get("frame_log", [])
        roles.update(d["roles"]); dossiers_all.update(d.get("dossiers", {}))
        t_end = max(t_end, d["t_end"])
        line_in += d["line_in"]; line_out += d["line_out"]
        if d.get("video"):
            videos.append(d["video"])

    n_returned = 0
    if GLOBAL_GUEST_LINK:
        _guests = {c["track_id"] for c in crossings
                   if c["direction"] == "in" and roles.get(c["track_id"]) != "staff"}
        print(f"\n  whole-night guest matching over {len(_guests)} door-crossing "
              f"identities (face >= {GUEST_FACE_SIM}, appearance >= {GUEST_ANCHOR_SIM}, "
              f"gap <= {GUEST_GLOBAL_GAP_S / 60:.0f} min)...")
        g_alias = link_returning_guests(dossiers_all, _guests, roles)
        n_returned = len(g_alias)
        events, crossings, roles, frame_log = _apply_alias(events, crossings, roles,
                                                           frame_log, g_alias)
        alias_all.update(g_alias)

    crossings.sort(key=lambda c: c["t"])
    frame_log.sort(key=lambda r: r[1])
    return {"camera_id": camera_id, "events": events, "crossings": crossings,
            "crossings_raw": crossings_raw,          # F1
            "ir_switches": sorted(ir_switches_all),  # F5c: without this the
            # merged run had NO ir data and the night_ir_busy / ir_switch
            # eval windows could never be picked on a multi-chunk night

            "roles": roles, "frame_log": frame_log, "t_end": t_end,
            "line_in": line_in, "line_out": line_out, "snapshots": snapshots_all,
            "zcolors": zone_color_map(load_zone_config(zones_path)[0]),
            "id_merges": alias_all, "seated_ratios": {},
            "online_reid": True, "annotated_video": (videos[0] if videos else None),
            "annotated_videos": videos, "custom_model": True,
            "canon_map": {}, "duration_s": t_end,
            "chunk_bounds": [(k, chunk_results[k][0], chunk_results[k][1]["t_end"])
                             for k in sorted(chunk_results)],
            "seam_stitches": n_stitched, "returning_guests": n_returned}


if (len(globals().get("QUEUE_REMOTE", [])) > 1
        and not globals().get("PEAK_ONLY")
        and not globals().get("RUN_ABLATION")):   # ABL: Cell 9e instead
    _cam = CAMERA_ID
    _zp = VIDEO_QUEUE[0]["zones_path"]
    runs = [run_multichunk(_cam, _zp, QUEUE_REMOTE, first_clock)]
    _r = runs[0]
    answer_for_run(_r, _zp)
    print(f"\n{'=' * 78}")
    print(f"  MERGED {len(QUEUE_REMOTE)} chunks -> {_r['t_end'] / 3600:.2f} h, "
          f"{len(_r['events'])} events, {len(_r['crossings'])} crossings, "
          f"{len(set(_r['roles']))} identities")
    print(f"  seam stitches      {_r['seam_stitches']} (spanned a chunk boundary)")
    print(f"  returning guests   {_r['returning_guests']} (left and came back — "
          f"counted ONCE)")
    _named = [t for t in _r["roles"] if isinstance(t, str) and not str(t).startswith("c")]
    print(f"  gallery staff      {_named or 'none — per-person staff numbers will over-count'}")
    print("=" * 78)
elif globals().get("PEAK_ONLY"):
    print(f"PEAK_ONLY is on — skipping the full-night run. Cell 9c below scans "
          f"all {len(globals().get('QUEUE_REMOTE', []))} chunk(s) and analyses "
          f"only the busiest {globals().get('PEAK_WINDOW_MIN', 20)} minutes.")
else:
    print(f"single chunk ({len(globals().get('QUEUE_REMOTE', []))}) — Cell 9 already ran it")


In [ ]:
# Cell 9c — PEAK WINDOW: the busiest N minutes of the night, and only those
# Full analytics on 10 hours is expensive. Most of the value is in the busy
# stretch. This scans every chunk cheaply (one frame every few seconds, ~1 min
# of GPU per chunk — the detector only, no tracking, no Re-ID), finds the
# densest window, and then runs the COMPLETE pipeline on just that window.
#
# Set PEAK_ONLY = True and run this INSTEAD of Cell 9b.
# All five knobs live in Cell 2 so Cell 9b (which runs BEFORE this cell) can
# see PEAK_ONLY and skip itself. Defaults here only for a stale kernel.
PEAK_ONLY         = globals().get("PEAK_ONLY", False)
PEAK_WINDOW_MIN   = globals().get("PEAK_WINDOW_MIN", 20)
PEAK_SCAN_EVERY_S = globals().get("PEAK_SCAN_EVERY_S", 6.0)
PEAK_SCAN_IMGSZ   = globals().get("PEAK_SCAN_IMGSZ", 640)
PEAK_CHUNKS       = globals().get("PEAK_CHUNKS", None)
PEAK_FORCE_START_MIN = globals().get("PEAK_FORCE_START_MIN", None)

if PEAK_ONLY and len(globals().get("QUEUE_REMOTE", [])) >= 1:
    import numpy as np, time, os
    from pathlib import Path

    def scan_density(video_path, device="cuda:0"):
        """people-per-sampled-frame over a whole chunk. Detector only."""
        m = YOLO(DETECTOR_MODEL)
        ts, ns = [], []
        for _i, _t, _fr in frame_source(video_path, 1.0 / PEAK_SCAN_EVERY_S,
                                        max_w=1280, keyframes_only=True):
            r = m(_fr, conf=CONF_THRESHOLD, iou=0.45, imgsz=PEAK_SCAN_IMGSZ,
                  verbose=False, device=device)[0]
            d = sv.Detections.from_ultralytics(r)
            ts.append(_t); ns.append(int((d.class_id == 0).sum()) if len(d) else 0)
        return np.array(ts), np.array(ns, dtype=float)

    def best_window(ts, ns, win_s):
        """Sliding window with the most person-seconds."""
        if len(ts) < 2:
            return 0.0, 0.0
        k = max(1, int(round(win_s / PEAK_SCAN_EVERY_S)))
        if len(ns) <= k:
            return float(ts[0]), float(ns.mean())
        c = np.convolve(ns, np.ones(k), mode="valid")
        i = int(c.argmax())
        return float(ts[i]), float(c[i] / k)

    _win_s = PEAK_WINDOW_MIN * 60.0
    _cands = []
    _t0 = time.time()
    _idxs = range(len(QUEUE_REMOTE)) if PEAK_CHUNKS is None else PEAK_CHUNKS
    # 9c claims to run "INSTEAD of Cell 9b" but _offset_for lives in 9b —
    # skipping 9b would NameError here, so degrade gracefully
    _offset_for = globals().get("_offset_for") or (lambda name, fc: None)
    for k in _idxs:
        f = QUEUE_REMOTE[k]
        off = _offset_for(f["name"], first_clock)
        off = k * 3600.0 if off is None else off
        try:
            path = _pull(f)
        except Exception as _dl:
            print(f"  \u26a0\ufe0f  chunk {k} could not be downloaded, skipping it:\n{_dl}")
            continue
        print(f"\n\u23f1  scanning chunk {k} ({f['name'][:40]}...) for the busy stretch")
        ts, ns = scan_density(Path(path), device="cuda:0" if torch.cuda.is_available() else "cpu")
        s, dens = best_window(ts, ns, _win_s)
        if PEAK_FORCE_START_MIN is not None:
            s = float(PEAK_FORCE_START_MIN) * 60.0
            _m = (ts >= s) & (ts < s + _win_s)
            dens = float(ns[_m].mean()) if _m.any() else 0.0
            print(f"   window PINNED to {mmss(s)} by PEAK_FORCE_START_MIN "
                  f"(auto-pick disabled)")
        peak_clock = wall(off + s) or mmss(off + s)
        print(f"   busiest {PEAK_WINDOW_MIN} min starts {peak_clock}  "
              f"avg {dens:.1f} people in frame  (chunk avg {ns.mean():.1f}, "
              f"max {ns.max():.0f})")
        _cands.append({"chunk": k, "file": f, "offset": off, "start": s,
                       "density": dens, "path": path, "ts": ts, "ns": ns})
        # keep the best-so-far ON DISK and delete the loser. Downloading a file
        # twice is what trips Drive's per-file quota, and 5 x 3.3 GB does not
        # fit in 20 GB either — this does both in one rule.
        _lead = max(_cands, key=lambda c: c["density"])
        for _c in _cands:
            if _c is not _lead and _c.get("path") and Path(_c["path"]).exists():
                try:
                    os.remove(_c["path"]); _c["path"] = None
                    print(f"      freed chunk {_c['chunk']} "
                          f"({_c['density']:.1f} < {_lead['density']:.1f} people)")
                except Exception:
                    pass

    if not _cands:
        raise RuntimeError("no chunk could be scanned — see the download errors above")
    _best = max(_cands, key=lambda c: c["density"])
    print("\n" + "=" * 78)
    print(f"  PEAK: chunk {_best['chunk']}, "
          f"{wall(_best['offset'] + _best['start']) or mmss(_best['start'])} -> "
          f"{wall(_best['offset'] + _best['start'] + _win_s) or ''}  "
          f"({_best['density']:.1f} people in frame on average)")
    print(f"  scan cost {(time.time() - _t0) / 60:.1f} min for "
          f"{len(_cands)} chunk(s)")
    print("=" * 78)

    # the density curve of the winning chunk, so the choice is visible
    fig, ax = plt.subplots(figsize=(14, 2.6))
    ax.fill_between(_best["ts"] / 60.0, _best["ns"], color=ROLE_HEXES["customer"],
                    alpha=.85, step="mid")
    ax.axvspan(_best["start"] / 60.0, (_best["start"] + _win_s) / 60.0,
               color=STATUS_CRIT, alpha=.25, label=f"analysed {PEAK_WINDOW_MIN} min")
    ax.set_xlabel(f"minutes into chunk {_best['chunk']}")
    ax.set_ylabel("people")
    ax.set_title(f"where the people are — chunk {_best['chunk']}", loc="left",
                 weight="bold")
    ax.legend(loc="upper right")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / "peak_scan.png", dpi=130,
                                    facecolor=SURFACE); plt.show()

    # ── full pipeline, on that window only ─────────────────────────────────
    _zp = VIDEO_QUEUE[0]["zones_path"]
    # the winner was never deleted, so this is a cache hit, not a second download
    _path = _best["path"] if (_best.get("path") and Path(_best["path"]).exists()) \
        else _pull(_best["file"])
    globals()["_RENDER_INDEX_SHIFT"] = _best["offset"]   # index in night time
    try:
        run = process_video(CAMERA_ID, Path(_path), _zp,
                            max_seconds=_win_s, start_seconds=_best["start"],
                            chunk_tag=f"_peak{_best['chunk']:02d}")
    finally:
        globals()["_RENDER_INDEX_SHIFT"] = 0.0

    # shift onto the night's clock so every wall time is real
    _sh = _best["offset"]
    run["events"] = [{**e, "t_in": e["t_in"] + _sh, "t_out": e["t_out"] + _sh}
                     for e in run["events"]]
    run["crossings"] = [{**c, "t": c["t"] + _sh} for c in run["crossings"]]
    run["frame_log"] = [(fi, t + _sh, b) for fi, t, b in run["frame_log"]]
    _w0 = _sh + _best["start"]
    run["t_end"] = _w0 + _win_s
    run["chunk_bounds"] = [(0, _w0, _w0 + _win_s)]   # position, not chunk no.
    # H.264 straight away: the raw mp4v is 300-600 MB and no browser plays it.
    # ~20-40 MB and playable, ready to download the second the window is done.
    if run.get("annotated_video") and Path(run["annotated_video"]).exists():
        import subprocess as _sp
        _v = Path(run["annotated_video"])
        _h = _v.with_name(_v.stem + "_h264.mp4")
        try:
            if _sp.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(_v),
                        "-vf", f"scale='min({globals().get('RENDER_MAX_W', 1280)},iw)':-2",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags",
                        "+faststart",     # plays while still downloading
                        "-crf", str(globals().get("RENDER_CRF", 28)), str(_h)],
                       timeout=3600).returncode == 0:
                _ix = _v.with_suffix(".index.json")
                if _ix.exists():
                    _ix.replace(_h.with_suffix(".index.json"))
                _v.unlink(missing_ok=True)
                run["annotated_video"] = str(_h)
                print(f"\U0001f3ac {_h.name} ready to download NOW "
                      f"({_h.stat().st_size / 1e6:.0f} MB, browser-playable)")
        except Exception as _enc:
            print(f"(h264 re-encode skipped: {_enc} — raw mp4v kept)")
    run["annotated_videos"] = [run["annotated_video"]] if run["annotated_video"] else []
    runs = [run]
    show_gallery(run["snapshots"], f"{CAMERA_ID} — PEAK {PEAK_WINDOW_MIN} min", ncols=3)
    answer_for_run(run, _zp)
    print(f"\n\u2705 peak window analysed. Every cell below now reports on "
          f"{wall(_w0) or mmss(_w0)} -> {wall(run['t_end']) or ''} only.")
else:
    print("PEAK_ONLY off (or nothing queued) — Cell 9b handles the full night")


In [ ]:
# Cell 18 — RECEPTION METRICS  (the numbers a GM reads)
# Pure functions over events + crossings. No model, no video, no new deps.
# Every metric here is either exact geometry or a labelled PROXY — nothing
# in between, because a number whose status is unclear is worse than none.

from collections import defaultdict

# ── thresholds, all named and all arguable — that is the point ──────────────
GREET_PROXIMITY_PX   = 220    # fallback only, when box height is unavailable
GREET_PROXIMITY_BODIES = 1.2  # contact = within 1.2x the two people's mean box
                              # HEIGHT. Scale-invariant: works the same near
                              # the camera and in the far corner, on a wide
                              # distorted lens, with no calibration.
GREET_MIN_CONTACT_S  = 3.0    # brief pass-by is not a greeting
TURNAWAY_MAX_S       = 90.0   # in and out this fast, unserved = walked out
LONG_WAIT_S          = 180.0  # "waited too long" line in the report
MICRO_ABSENCE_S      = 90.0   # desk gaps under this = doing the job, not gone
BREAK_ABSENCE_S      = 600.0  # over this = a break, reported separately
BUCKET_S             = 900.0  # 15-minute reporting buckets


def _canon_intervals(events, zone_pred, role=None):
    """Merged presence intervals per person for zones matching zone_pred."""
    per = defaultdict(list)
    for e in events:
        if role and e["role"] != role:
            continue
        if zone_pred(e["zone"]):
            per[e["track_id"]].append([e["t_in"], e["t_out"]])
    return {k: merge_intervals(v) for k, v in per.items()}


def _boxes_by_time(frame_log):
    """[(t, {tid: (cx, cy)})] — sampled centres, for proximity tests."""
    out = []
    for _idx, t, boxes in frame_log:
        out.append((t, {b[0]: ((b[1] + b[3]) / 2.0, (b[2] + b[4]) / 2.0,
                               abs(b[4] - b[2]))          # cx, cy, box height
                        for b in boxes}))
    return out


def staff_contacts(frame_log, roles, canon=None, plane=None):
    """PROXY: guest and staff boxes within GREET_PROXIMITY_PX for
    >= GREET_MIN_CONTACT_S. Proximity is not conversation — this is a
    service-touch signal, and every report must say so."""
    tl = _boxes_by_time(frame_log)
    running, done = {}, defaultdict(list)
    for i, (t, cent) in enumerate(tl):
        prev_t = tl[i - 1][0] if i else t
        staff = [k for k in cent if roles.get(k) == "staff"]
        guests = [k for k in cent if roles.get(k) == "customer"]
        near = set()
        for g in guests:
            gx, gy, gh = cent[g]
            for s in staff:
                sx, sy, sh = cent[s]
                # G2: a real distance on the floor. The body-height proxy
                # below was a clever workaround for not having one — it is kept
                # as the fallback, because it is still much better than a flat
                # pixel threshold.
                _d = None
                if plane is not None and plane.ok:
                    _d = plane.dist_m((gx, gy + gh / 2.0), (sx, sy + sh / 2.0))
                if _d is not None:
                    if _d <= GREET_PROXIMITY_M:
                        near.add(g)
                        break
                    continue
                scale = (gh + sh) / 2.0
                limit = (GREET_PROXIMITY_BODIES * scale) if scale > 1 \
                    else GREET_PROXIMITY_PX
                if math.hypot(gx - sx, gy - sy) <= limit:
                    near.add(g)
                    break
        for g in near:
            running.setdefault(g, t)
        for g in list(running):
            if g not in near:
                if prev_t - running[g] >= GREET_MIN_CONTACT_S:
                    done[g].append((running[g], prev_t))
                del running[g]
    for g, t0 in running.items():
        if tl and tl[-1][0] - t0 >= GREET_MIN_CONTACT_S:
            done[g].append((t0, tl[-1][0]))
    return dict(done)


def _arrival_verdict(run):
    """PHASE10: the arrival cross-check, as plain text for the brief."""
    events, roles = run["events"], run["roles"]
    n, _arr, why = arrivals_from_regions(events, run.get("zone_roles") or {},
                                         roles=roles)
    line_n = len({c["track_id"] for c in run["crossings"]
                  if c["direction"] == "in" and roles.get(c["track_id"]) != "staff"})
    xc = cross_check(line_n, n, movers=len({e["track_id"] for e in events}))
    return describe_arrivals(line_n, n, xc) + (f"\n     {why}" if why else "")


def reception_report(run, staff_zones, wait_zones, t_end):
    """One dict holding every reception number. Keys are plain English on
    purpose — they go straight into the brief and the spreadsheet."""
    events, crossings, roles = run["events"], run["crossings"], run["roles"]
    _plane = _rebuild_plane(run)
    # PHASE10: every guest number below is derived from `crossings`. On the
    # first real hour the entry line never fired, so all of them came back 0 —
    # while the waiters table in the same report listed 46 people. A second,
    # independent count (entry zone -> interior zone) needs no line at all, so
    # it survives exactly the drawing mistake a line dies from.
    # C1r fields join the report dict where the other counts are built
    _region_n, _region_arr, _region_why = arrivals_from_regions(
        events, run.get("zone_roles") or {}, roles=roles)
    _line_n = len({c["track_id"] for c in crossings
                   if c["direction"] == "in" and roles.get(c["track_id"]) != "staff"})
    _movers = len({e["track_id"] for e in events})
    # V74: a region count is only a cross-check if the entry zone is somewhere
    # people actually walk. When almost nobody is ever seen inside it, the
    # region number is a second silent failure, not a fallback.
    _cov74 = entry_zone_coverage(events, run.get("zone_roles") or {}, roles=roles)
    _xc = cross_check(_line_n, _region_n, movers=_movers, coverage=_cov74)
    if _cov74 and _xc.get("trust") == "neither":
        print("=" * 78)
        print(f"🚨 ENTRY ZONE MISPLACED · {camera_id}")
        print(f"   only {_cov74['with_entry']} of {_cov74['non_staff']} non-staff "
              f"people were ever seen inside an entry zone "
              f"({_cov74['share_with_entry']*100:.0f}%).")
        print("   The line AND the entry polygon both need redrawing. Every")
        print("   arrival number below is an ESTIMATE from identity births.")
        print("=" * 78)
    contacts = staff_contacts(run.get("frame_log", []), roles, plane=_plane)

    # ── arrivals, exits, re-entries ────────────────────────────────────────
    ins = [c for c in crossings if c["direction"] == "in"
           and roles.get(c["track_id"]) != "staff"]
    outs = [c for c in crossings if c["direction"] == "out"
            and roles.get(c["track_id"]) != "staff"]
    per_person_ins = defaultdict(list)
    for c in ins:
        per_person_ins[c["track_id"]].append(c["t"])

    # C1r: OCCUPANCY CONSISTENCY — running IN-OUT can never truly be negative;
    # every time it dips below zero, a crossing was missed or double-counted.
    # The dip count and depth are a FREE, gt-less measure of counting damage.
    # Reported, never silently corrected.
    _occ, _occ_min, _occ_dips = 0, 0, 0
    for _t2, _d2 in sorted([(c["t"], 1) for c in ins]
                           + [(c["t"], -1) for c in outs]):
        _occ += _d2
        if _occ < 0:
            if _occ < _occ_min:
                _occ_min = _occ
            if _occ == -1 and _d2 == -1:
                _occ_dips += 1
    reentries = {k: len(v) - 1 for k, v in per_person_ins.items() if len(v) > 1}

    # If the line is broken, fall back to the region count for the headline
    # arrival number rather than publishing a 0 that reads like a fact.
    # ORDER MATTERS: these fallbacks MUST run BEFORE the greet loop below —
    # in the shipped run they ran after it, so greeted/unserved/turnaways were
    # computed over an EMPTY dict and published as confident zeros while 16
    # guests and a staffed desk were in the same report.
    _arrivals_estimated = False
    if _xc["trust"] == "region" and _region_n:
        _arrivals_estimated = True   # zone-entry times, not door-crossing times
        for _a in _region_arr:
            per_person_ins.setdefault(_a["track_id"], []).append(_a["t"])
    # V69: third tier. Line AND region both blind while people demonstrably
    # exist means the DOORWAY is untrackable, not the venue empty. On a
    # single-public-door camera, a non-staff identity born mid-footage had
    # to have walked in. >=30s guard: present-at-start bodies didn't arrive.
    if not per_person_ins and _movers >= 5:
        _first3 = {}
        for _e3 in events:
            if roles.get(_e3["track_id"]) == "staff":
                continue
            if (_e3["track_id"] not in _first3
                    or _e3["t_in"] < _first3[_e3["track_id"]]):
                _first3[_e3["track_id"]] = _e3["t_in"]
        _n3 = 0
        for _tid3, _t3 in sorted(_first3.items(), key=lambda kv: kv[1]):
            if _t3 >= 30.0:
                per_person_ins[_tid3].append(_t3)
                _n3 += 1
        if _n3:
            _arrivals_estimated = True
            print(f"   V69 birth-count fallback: line={len(ins)} AND "
                  f"region={_region_n} while {_movers} moved — {_n3} non-staff "
                  f"identities first seen mid-footage counted as arrivals "
                  f"(ESTIMATE: the door itself was never trackable this chunk)")

    # ── greet latency: door -> first staff contact ─────────────────────────
    # Staff contacts are line-independent, so greet COUNTS are real even when
    # arrival times are estimates; only exit-based metrics (walkouts) stay
    # line-derived and are reported as unknown, never zero, when the line died.
    greets, unserved, turnaways = [], [], []
    for tid, times in per_person_ins.items():
        t_in = min(times)
        cs = [c[0] for c in contacts.get(tid, []) if c[0] >= t_in]
        # A walkout is someone who left AND DID NOT COME BACK. Using the last
        # exit alone counts a smoker's first trip outside as a lost guest —
        # so an exit followed by a later entry means they never really left.
        p_outs = sorted(c["t"] for c in outs if c["track_id"] == tid)
        last_out = p_outs[-1] if p_outs else None
        still_inside = last_out is not None and any(t > last_out for t in times)
        final_exit = None if (last_out is None or still_inside) else last_out
        if cs:
            greets.append({"person": tid, "arrived": t_in,
                           "greeted": min(cs), "latency_s": min(cs) - t_in})
        else:
            unserved.append({"person": tid, "arrived": t_in, "left": final_exit})
            if final_exit is not None and (final_exit - t_in) <= TURNAWAY_MAX_S:
                turnaways.append({"person": tid, "arrived": t_in,
                                  "left": final_exit,
                                  "seconds_inside": final_exit - t_in})
    lat = sorted(g["latency_s"] for g in greets)
    def _pct(p):
        return lat[min(len(lat) - 1, int(len(lat) * p))] if lat else None

    # ── desk coverage: per PERSON and per STATION ──────────────────────────
    # Station coverage is the fair, defensible number. Per-person is kept
    # for shift-change detection, NOT for judging an individual.
    per_staff = _canon_intervals(events, lambda z: z in staff_zones, role="staff")
    station = merge_intervals([iv for ivs in per_staff.values() for iv in ivs])
    # v55: only count gaps that fall inside footage we have.
    covered = covered_windows(run)
    covered_s = total_duration(covered)
    gaps = clip_to(complement_intervals(station, 0.0, t_end), covered)
    micro = [g for g in gaps if (g[1] - g[0]) < MICRO_ABSENCE_S]
    breaks = [g for g in gaps if (g[1] - g[0]) >= BREAK_ABSENCE_S]
    real = [g for g in gaps if MICRO_ABSENCE_S <= (g[1] - g[0]) < BREAK_ABSENCE_S]

    # guests who arrived while nobody was at the desk — from ALL arrival
    # evidence (line, region, or birth fallback), not just line crossings,
    # otherwise a dead line silently reports 0 here too
    _arr_recs = [{"track_id": tid, "t": t}
                 for tid, ts in per_person_ins.items() for t in ts]
    def _uncovered(t):
        return not any(a <= t <= b for a, b in station)
    unattended = [c for c in _arr_recs if _uncovered(c["t"])]

    # ── shift change: distinct staff identities and their windows ──────────
    shifts = sorted(
        ({"staff": k,
          "on_station_s": total_duration(v),
          "first": min(i[0] for i in v),
          "last":  max(i[1] for i in v)}
         for k, v in per_staff.items() if total_duration(v) > 300),
        key=lambda d: d["first"])

    # ── demand curve vs coverage, in 15-min buckets ────────────────────────
    nb = int(t_end // BUCKET_S) + 1
    buckets = []
    for b in range(nb):
        lo, hi = b * BUCKET_S, min((b + 1) * BUCKET_S, t_end)
        cov = sum(max(0.0, min(hi, e) - max(lo, s)) for s, e in station)
        foot = sum(max(0.0, min(hi, e) - max(lo, s)) for s, e in covered)
        if foot <= 1.0:
            buckets.append({"from": wall(lo) or mmss(lo), "arrivals": None,
                            "exits": None, "desk_covered_pct": None,
                            "unattended_arrivals": None, "footage": False})
            continue
        buckets.append({
            "footage": True,
            "from": wall(lo) or mmss(lo),
            # count ALL arrival evidence, so the demand curve is not flat-zero
            # (and "busiest 15 min" not garbage) whenever the line is dead
            "arrivals": sum(1 for c in _arr_recs if lo <= c["t"] < hi),
            "exits":    sum(1 for c in outs if lo <= c["t"] < hi),
            "desk_covered_pct": round(100 * cov / foot, 1),
            "unattended_arrivals": sum(1 for c in unattended if lo <= c["t"] < hi),
        })

    # ── waits ──────────────────────────────────────────────────────────────
    waits = []
    for wz in wait_zones:
        _, dist = waited_over(events, wz, 0.0)
        waits += list(dist.values()) if isinstance(dist, dict) else list(dist)

    _bk = [b for b in buckets if b.get("footage")]
    return {
        "occupancy_went_negative": _occ_dips,
        "occupancy_worst_deficit": -_occ_min,
        "footage_hours":             round(covered_s / 3600.0, 2),
        "night_span_hours":          round(t_end / 3600.0, 2),
        "missing_hours":             round((t_end - covered_s) / 3600.0, 2),
        "guests_tonight":            len(per_person_ins),
        "door_crossings_in":         len(ins),
        "people_who_came_back":      len(reentries),
        # all-zero buckets have NO busiest window: max() over zeros returns the
        # FIRST bucket, which the shipped run published as a fact ("busiest
        # 19:30, 0 arriving") — report unknown instead
        "busiest_15min":             (max(_bk, key=lambda b: b["arrivals"])["from"]
                                      if _bk and max(b["arrivals"] for b in _bk) > 0
                                      else None),
        "peak_arrivals_in_15min":    max((b["arrivals"] for b in _bk), default=0),
        "arrival_times_estimated":   _arrivals_estimated,

        "greeted":                   len(greets),
        "avg_greet_seconds":         round(sum(lat) / len(lat), 1) if lat else None,
        "median_greet_seconds":      round(_pct(0.50), 1) if lat else None,
        "slowest_greet_seconds":     round(lat[-1], 1) if lat else None,
        "waited_over_3min":          sum(1 for g in greets if g["latency_s"] > LONG_WAIT_S),

        "left_without_being_served": len(unserved),
        # a walkout needs an EXIT time, which only the line provides — when the
        # line never fired this is unknown, not a reassuring zero
        "walked_out_under_90s":      (len(turnaways) if ins or outs else None),
        "arrived_to_an_empty_desk":  len(unattended),

        "desk_covered_pct":          round(100 * total_duration(station) / max(covered_s, 1e-6), 1),
        "longest_desk_gap_min":      round(max((g[1] - g[0] for g in gaps), default=0) / 60, 1),
        "quick_steps_away":          len(micro),
        "real_absences":             len(real),
        "long_breaks":               len(breaks),

        # a fragmented staff identity never accumulates the 300s one-id shift
        # bar, so "0 people worked the desk" coexisted with 340s of measured
        # staff coverage — fall back to distinct staff ids seen at the station
        "people_working_the_desk":   (len(shifts) if shifts
                                      else (len(per_staff) if station else 0)),
        "_shifts":                   shifts,
        "_buckets":                  buckets,
        "_greets":                   sorted(greets, key=lambda g: -g["latency_s"]),
        "_turnaways":                turnaways,
        "_unattended":               unattended,
        "_gaps":                     sorted(gaps, key=lambda g: -(g[1] - g[0])),
        "_proxy_note": ("greet = staff box within "
                        f"{GREET_PROXIMITY_PX}px for {GREET_MIN_CONTACT_S}s. "
                        "Proximity, not conversation. Treat as a service-touch "
                        "signal, never as proof someone was spoken to."),
    }


RECEPTION = {}
for run in runs:
    meta = VENUE_META[run["camera_id"]]
    RECEPTION[run["camera_id"]] = reception_report(
        run, meta["staff_zones"], meta["wait_zones"], run["t_end"])

    r = RECEPTION[run["camera_id"]]
    print("=" * 78)
    print(f"  {run['camera_id']} — RECEPTION")
    print("=" * 78)
    if r["missing_hours"] > 0.02:
        print(f"  ⚠️  FOOTAGE GAP: the night spans {r['night_span_hours']}h but only "
              f"{r['footage_hours']}h of video exists ({r['missing_hours']}h missing). "
              f"Every percentage below is over the footage we HAVE, not the wall clock.")
    for k, v in r.items():
        if not k.startswith("_"):
            print(f"  {k:30s} {v}")
    print(f"\n  PROXY: {r['_proxy_note']}")


In [ ]:
# Cell 19 — THE BRIEF, THE STRIP, THE MOMENTS
# Nobody watches 10 hours. These three outputs are what actually gets read:
#   BRIEF    plain English, no jargon, every sentence traceable to a row
#   STRIP    the whole night in one image, readable in 3 seconds
#   MOMENTS  8-12 clips of 10 s — the exceptions, not the footage
# The brief is deterministic (template over the event log), NOT generated
# prose. That is deliberate: it cannot hallucinate, it works with no API
# key, and an LLM can polish the wording later without touching the facts.

import subprocess
from pathlib import Path

MOMENT_PAD_S   = 5.0     # seconds either side of the interesting instant
MOMENT_MAX     = 12      # cap — past this nobody watches either
MOMENTS_DIR    = OUTPUT_DIR / "moments"
MOMENTS_DIR.mkdir(parents=True, exist_ok=True)


def _hm(t):
    """Video seconds -> real wall clock, using the burned-in start time."""
    return wall(t) or mmss(t)


def write_brief(camera_id, r, t_end, t_start=0.0):
    """Plain English. No metric names, no thresholds, no jargon."""
    hrs = r.get("footage_hours", (t_end - t_start) / 3600.0)
    _span = (f"{hrs:.1f} hours of footage" if hrs >= 1
             else f"{hrs * 60:.0f} minutes of footage")   # a 20 min window is
    L = []                                                # not "0.3 hours"
    L.append(f"{camera_id} — {_hm(t_start)} to {_hm(t_end)}  ({_span})")
    L.append("")
    L.append(f"{r['guests_tonight']} people came through the door.")
    if r["busiest_15min"]:
        L.append(f"Busiest stretch was {r['busiest_15min']}, "
                 f"with {r['peak_arrivals_in_15min']} arriving in 15 minutes.")
    if r["avg_greet_seconds"] is not None:
        L.append(f"On average someone was with them in "
                 f"{r['avg_greet_seconds']:.0f} seconds "
                 f"(typical {r['median_greet_seconds']:.0f}s, "
                 f"slowest {r['slowest_greet_seconds']:.0f}s).")
    if r["waited_over_3min"]:
        L.append(f"{r['waited_over_3min']} guests waited more than 3 minutes "
                 f"before anyone reached them.")
    if r["arrived_to_an_empty_desk"]:
        L.append(f"{r['arrived_to_an_empty_desk']} guests walked in while "
                 f"nobody was at the desk.")
    if r["walked_out_under_90s"]:
        L.append(f"{r['walked_out_under_90s']} people came in, were not "
                 f"spoken to, and left within 90 seconds.")
    L.append("")
    if r["missing_hours"] > 0.02:
        L.append(f"NOTE: {r['missing_hours']:.1f} hours of this window have no "
                 f"footage at all. Everything here describes the "
                 f"{r['footage_hours']:.1f} hours we can see.")
    L.append(f"The desk was covered {r['desk_covered_pct']:.0f}% of the time we "
             f"have footage for. Longest gap: {r['longest_desk_gap_min']:.0f} minutes.")
    if r["people_working_the_desk"] > 1:
        L.append(f"{r['people_working_the_desk']} different people worked the "
                 f"desk — these numbers cover the station, not any one person.")
    if r["people_who_came_back"]:
        L.append(f"{r['people_who_came_back']} people stepped out and came "
                 f"back; they are counted once, not twice.")
    L.append("")
    L.append("Read this as service-touch timing, not conversation: we can see")
    L.append("that someone was with a guest, not what was said.")
    return "\n".join(L)


def plot_strip(camera_id, r, t_end):
    """The whole night in one image: demand on top, coverage underneath."""
    b = r["_buckets"]
    if not b:
        return None
    xs = list(range(len(b)))
    fig, ax = plt.subplots(2, 1, figsize=(15, 4.4), height_ratios=[3, 1],
                           sharex=True)

    # filter BEFORE the first bar: no-footage buckets carry arrivals=None
    # (TypeError in bar()), and filtering between the two bar calls drew the
    # red "unattended" bars against a different x-axis than the blue ones
    b = [x for x in b if x.get("footage", True)]
    xs = list(range(len(b)))
    if not b:
        plt.close(fig)
        return None
    ax[0].bar(xs, [x["arrivals"] for x in b], color=ROLE_HEXES["customer"],
              width=0.86, label="arrivals")
    bad = [i for i, x in enumerate(b) if x["unattended_arrivals"]]
    if bad:
        ax[0].bar(bad, [b[i]["unattended_arrivals"] for i in bad],
                  color=STATUS_CRIT, width=0.86, label="arrived to an empty desk")
    ax[0].set_ylabel("guests / 15 min")
    ax[0].legend(loc="upper left", ncol=2)
    ax[0].set_title(f"{camera_id} — the night in one picture", loc="left",
                    fontsize=12, weight="bold")

    cov = [x["desk_covered_pct"] for x in b]
    ax[1].imshow([cov], aspect="auto", cmap="RdYlGn", vmin=0, vmax=100,
                 extent=(-0.5, len(b) - 0.5, 0, 1))
    ax[1].set_yticks([])
    ax[1].set_ylabel("desk\ncovered", rotation=0, ha="right", va="center")
    step = max(1, len(b) // 12)
    ax[1].set_xticks(xs[::step])
    ax[1].set_xticklabels([b[i]["from"] for i in xs[::step]], rotation=0)
    ax[1].grid(False)
    plt.tight_layout()
    p = OUTPUT_DIR / f"{camera_id}_strip.png"
    plt.savefig(p, dpi=130, facecolor=SURFACE)
    plt.show()
    return p


def pick_moments(r, t_end):
    """The exceptions worth 10 seconds of a manager's attention."""
    m = []
    for g in r["_greets"][:3]:
        if g["latency_s"] > LONG_WAIT_S:
            m.append((f"waited {g['latency_s']:.0f}s to be greeted",
                      g["arrived"], g["greeted"]))
    for w in r["_turnaways"][:3]:
        m.append((f"walked out after {w['seconds_inside']:.0f}s",
                  w["arrived"], w["left"]))
    for u in r["_unattended"][:3]:
        m.append(("arrived, desk empty", u["t"], u["t"] + 10))
    for gap in r["_gaps"][:2]:
        if gap[1] - gap[0] >= BREAK_ABSENCE_S:
            m.append((f"desk empty {(gap[1]-gap[0])/60:.0f} min", gap[0], gap[0] + 10))
    peak = max((x for x in r["_buckets"] if x.get("footage")),
               key=lambda x: x["arrivals"], default=None)
    if peak and peak["arrivals"]:
        i = r["_buckets"].index(peak)
        m.append((f"busiest 15 min ({peak['arrivals']} arrivals)",
                  i * BUCKET_S, i * BUCKET_S + 10))
    return sorted(m, key=lambda x: x[1])[:MOMENT_MAX]


def _load_render_index(video_path):
    v = Path(str(video_path))
    for p in (v.with_suffix(".index.json"),
              Path(str(video_path).replace("_h264.mp4", ".mp4")).with_suffix(".index.json")):
        try:
            return json.loads(p.read_text())
        except Exception:
            continue
    return None


def _vt(t, index, t_offset=0.0):
    """night-seconds -> seconds inside the fast-forwarded annotated video.
    The index is written in ABSOLUTE night time, so no offset is applied to it;
    t_offset only helps the index-less fallback."""
    fps_out = float(globals().get("PLAYBACK_FPS", 30))
    if index:
        import bisect as _bs
        i = min(len(index) - 1, _bs.bisect_left(index, t))
        return i / fps_out
    speed = fps_out / float(globals().get("ANALYSIS_FPS", 4))
    return max(0.0, (t - t_offset) / max(speed, 1e-6))


def cut_moments(video_path, moments, camera_id, index=None, t_offset=0.0):
    """ffmpeg -ss/-to per moment, on the annotated video's own clock."""
    out = []
    for k, (label, t0, t1) in enumerate(moments):
        s = max(0.0, _vt(t0, index, t_offset) - MOMENT_PAD_S)
        e = _vt(t1, index, t_offset) + MOMENT_PAD_S
        safe = "".join(c if c.isalnum() else "_" for c in label)[:40]
        p = MOMENTS_DIR / f"{camera_id}_{k:02d}_{safe}.mp4"
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{s:.2f}",
               "-i", str(video_path), "-t", f"{e - s:.2f}",
               "-vf", f"scale='min({RENDER_MAX_W},iw)':-2",
               "-c:v", "libx264", "-crf", str(RENDER_CRF), "-an", str(p)]
        try:
            subprocess.run(cmd, check=True, timeout=180)
            out.append((label, _hm(t0), p))
        except Exception as ex:
            print(f"  WARN could not cut '{label}': {ex}")
    return out


_video_by_cam = {v["camera_id"]: v["video"] for v in VIDEO_QUEUE}
BRIEFS = {}
for run in runs:
    cam = run["camera_id"]
    r = RECEPTION.get(cam)
    if not r:
        continue
    if not run.get("events"):
        # V64c: run 2 crashed HERE (int + None) after every chunk failed —
        # the report layer must say "nothing to report", not die.
        print(f"(brief skipped for {cam}: run has no events — every chunk "
              f"failed or nothing was processed)")
        continue

    _cov = covered_windows(run)
    brief = write_brief(cam, r, run["t_end"], t_start=(_cov[0][0] if _cov else 0.0))
    BRIEFS[cam] = brief
    (OUTPUT_DIR / f"{cam}_brief.txt").write_text(brief)
    print("=" * 78); print(brief); print("=" * 78)

    plot_strip(cam, r, run["t_end"])

    mom = pick_moments(r, run["t_end"])
    print(f"\n  {len(mom)} MOMENTS — {len(mom) * (2 * MOMENT_PAD_S + 10) / 60:.1f} min "
          f"instead of {run['t_end'] / 3600:.1f} hours")
    for label, t0, _t1 in mom:
        print(f"    {_hm(t0):>10s}  {label}")

    # v55: cut from the ANNOTATED per-chunk videos (the source chunks are
    # deleted as we go), mapping night-time -> (chunk file, time inside it).
    _bounds = run.get("chunk_bounds")
    _vids = run.get("annotated_videos") or ([run["annotated_video"]]
                                            if run.get("annotated_video") else [])
    if _bounds and _vids:
        _done = 0
        for label, t0, t1 in mom:
            # position in the video list, NOT the chunk number — with PEAK mode
            # (or any skipped chunk) those are different, and indexing the list
            # by chunk number is an IndexError waiting to happen.
            _hit = next(((pos, s) for pos, (_k, s, e) in enumerate(_bounds)
                         if s <= t0 <= e), None)
            if _hit is None or _hit[0] >= len(_vids):
                continue
            _v = _vids[_hit[0]]
            _done += len(cut_moments(_v, [(label, t0, t1)], f"{cam}_{_hit[0]:02d}",
                                     index=_load_render_index(_v), t_offset=_hit[1]))
        print(f"  -> {_done} clips in {MOMENTS_DIR} (cut from the annotated video)")
    else:
        _v = run.get("annotated_video")
        src = _v if (_v and Path(_v).exists()) else _video_by_cam.get(cam)
        if src and Path(src).exists():
            clips = cut_moments(src, mom, cam, index=_load_render_index(src))
            print(f"  -> {len(clips)} clips in {MOMENTS_DIR}")
        else:
            print("  (source video already deleted — clips skipped)")


In [ ]:
# Cell 10 — VISUALS for every processed venue
_video_path_by_camera = {v["camera_id"]: v["video"] for v in VIDEO_QUEUE}
for run in runs:
    meta = VENUE_META[run["camera_id"]]
    plot_journey_gantt(run["events"], f"Per-person journey — {run['camera_id']}", run["zcolors"])
    plot_occupancy(run["events"], run["t_end"], f"People present over time — {run['camera_id']}")
    if run.get("frame_log"):
        vp = _video_path_by_camera.get(run["camera_id"])
        if vp and vp.exists():
            plot_occupancy_heatmap(run["frame_log"], vp,
                                   f"Dwell heatmap — {run['camera_id']}")
    for wz in sorted(meta["wait_zones"]):
        _, wait_dist = waited_over(run["events"], wz, WAIT_THRESHOLD_S)
        if wait_dist:
            plot_wait_distribution(wait_dist, WAIT_THRESHOLD_S,
                                   f"{wz} dwell per person — {run['camera_id']}")
    for sz in sorted(meta["staff_zones"]):
        plot_presence_timeline(run["events"], sz, run["t_end"],
                               f"{sz} coverage — {run['camera_id']}")
    if meta["seating_zones"]:
        plot_table_service(run["events"], meta["seating_zones"],
                           f"Table/service timeline — {run['camera_id']}")


In [ ]:
# Cell 13 — minute-by-minute table (the mentor's output contract), across
# every venue processed above (one or many, whatever was in VIDEO_QUEUE).
minutes = []
for run in runs:
    minutes += minute_summaries(run["events"], run["crossings"],
                                run["camera_id"], run["t_end"],
                                window_s=WINDOW_S, roles=run["roles"])
for row in minutes:                      # add wall-clock for the demo sheet
    row["clock"] = wall(int(row["time_window"].split("-")[0].split(":")[0]) * 60
                        + int(row["time_window"].split("-")[0].split(":")[1]))
(OUTPUT_DIR / "minute_summaries.json").write_text(json.dumps(minutes, indent=2))
print(f"{len(minutes)} window-rows ({WINDOW_S}s each) -> "
      f"{OUTPUT_DIR/'minute_summaries.json'}")
pd.DataFrame(minutes)


In [ ]:
def track_sort_key(tid):
    if isinstance(tid, str) and not tid.isdigit():
        return (0, tid)
    try:
        return (1, int(tid))
    except ValueError:
        return (2, str(tid))

# Cell 14 — EXCEL ANALYTICS WORKBOOK (styled — the demo hand-out)
# `runs` was already built by Cell 9 above (one entry per processed venue)
ROLE_FILL = {"customer": "background-color: #dbe9f9",
             "staff": "background-color: #fde3d5",
             "unknown": "background-color: #eeeeee"}
try:
    import jinja2  # noqa: F401 — pandas Styler needs it
    HAS_STYLER = True
except ImportError:
    HAS_STYLER = False
    print("jinja2 missing -> workbook written without cell colors")

def style_by_role(df):
    styler = df.style.apply(
        lambda row: [ROLE_FILL.get(str(row.get("role", "")), "")] * len(row),
        axis=1)
    flag_cols = [c for c in df.columns if "waited" in c]
    if flag_cols:
        styler = styler.map(
            lambda v: "background-color: #f5c6c6; font-weight: bold"
            if v == "YES" else "", subset=flag_cols)
    return styler

def _compute_id_confidence(tid, run, evs):
    """Compute a confidence score (0-100) for how reliable this person's ID is."""
    score = 0
    # Duration bonus: 0-25 points (max at 5+ minutes)
    total_dur = sum(e["duration"] for e in evs)
    score += min(25, total_dur / 12)
    # Evidence tier bonus: 0-30 points
    dossiers = run.get("identity_dossiers", {})
    if tid in dossiers:
        tiers = dossiers[tid].get("evidence_tiers_used", [])
        score += min(30, len(tiers) * 10)
        if "face" in tiers:
            score += 15
    # Staff gallery match: +20 points
    if isinstance(tid, str) and not str(tid).isdigit():
        score += 20
    # Door crossing: +10 points
    crossings = run.get("crossings", [])
    if any(c["track_id"] == tid and c["direction"] == "in" for c in crossings):
        score += 10
    # Single continuous track: +10 points
    merges = run.get("id_merges", {})
    n_fragments = sum(1 for v in merges.values() if v == tid)
    if n_fragments == 0:
        score += 10
    return min(100, round(score))

def build_person_log(run):
    events, crossings = run["events"], run["crossings"]
    roles = run["roles"]
    per = defaultdict(list)
    for e in events:
        per[e["track_id"]].append(e)
    entered_ids = {c["track_id"] for c in crossings if c["direction"] == "in"}
    merged_into = defaultdict(list)
    for frag, canon in run.get("id_merges", {}).items():
        merged_into[canon].append(f"#{frag}")
    rows = []
    for tid, evs in sorted(per.items(), key=lambda kv: track_sort_key(kv[0])):
        evs.sort(key=lambda e: e["t_in"])
        first, last = evs[0]["t_in"], max(e["t_out"] for e in evs)
        meta = VENUE_META[run["camera_id"]]
        wait_s = sum(e["duration"] for e in evs if e["zone"] in meta["wait_zones"])
        seated = any(e["zone"] in meta["seating_zones"]
                     and e["duration"] >= MIN_SEATED_S for e in evs)
        rows.append({
            "person_id": f"P{tid}" if (isinstance(tid, int) or str(tid).isdigit()) else str(tid),
            "role": roles.get(tid, evs[0]["role"]),
            "camera": run["camera_id"],
            "first_seen": mmss(first), "first_seen_clock": wall(first),
            "last_seen": mmss(last), "last_seen_clock": wall(last),
            "visible_min": round((last - first) / 60, 1),
            "zone_path": " > ".join(dict.fromkeys(e["zone"] for e in evs)),
            "n_zones": len({e["zone"] for e in evs}),
            "entered_via_door": "yes" if tid in entered_ids else "",
            "waiting_visits": sum(1 for e in evs if e["zone"] in meta["wait_zones"]),
            "wait_s": round(wait_s, 1),
            "waited_over": "YES" if wait_s >= WAIT_THRESHOLD_S else "",
            "seated": "yes" if seated else "",
            "pose_seated_ratio": run.get("seated_ratios", {}).get(tid, ""),
            "id_confidence": _compute_id_confidence(tid, run, evs),
            "also_labeled_in_video_as": ", ".join(sorted(merged_into.get(tid, []))),
        })
    return pd.DataFrame(rows)

def build_id_merges(run):
    """Cross-verification key: the annotated VIDEO shows live tracker labels;
    Re-ID stitching later folds returning people together. This sheet maps
    every merged video label to its final Excel person."""
    return pd.DataFrame([{"video_label": f"#{frag}",
                          "final_person_in_excel": f"P{canon}" if (isinstance(canon, int) or str(canon).isdigit()) else str(canon)}
                         for frag, canon in
                         sorted(run.get("id_merges", {}).items(), key=lambda kv: (str(type(kv[0]).__name__), str(kv[0])))])

def build_zone_visits(run):
    """Person x zone pivot: how many separate visits, how long in total.
    A person leaving a zone and coming back = visits 2 (same ID remembered)."""
    agg = defaultdict(lambda: {"visits": 0, "total_s": 0.0})
    for e in run["events"]:
        key = (e["track_id"], e["role"], e["zone"])
        agg[key]["visits"] += 1
        agg[key]["total_s"] += e["duration"]
    rows = [{"person_id": f"P{t}" if (isinstance(t, int) or str(t).isdigit()) else str(t), "role": r, "zone": z,
             "visits": v["visits"], "total_s": round(v["total_s"], 1)}
            for (t, r, z), v in sorted(agg.items(), key=lambda kv: track_sort_key(kv[0][0]))]
    return pd.DataFrame(rows)

def build_staff_zone_log(run):
    """Minute-accurate STAFFED/AWAY ledger, unioned across every zone this
    venue's own zones file classified as a "staff" zone (reception, till,
    checkout, counter, ... whatever it's called for this venue)."""
    staff_zones = VENUE_META[run["camera_id"]]["staff_zones"]
    if not staff_zones:
        return pd.DataFrame()
    staffed = merge_intervals([(e["t_in"], e["t_out"]) for e in run["events"]
                               if e["zone"] in staff_zones
                               and e["role"] == "staff"])
    gaps = complement_intervals(staffed, 0, run["t_end"])
    raw = ([("STAFFED ✓", s, e) for s, e in staffed]
           + [("AWAY ✗", s, e) for s, e in gaps if e - s >= 1.0])
    raw.sort(key=lambda r: r[1])
    return pd.DataFrame([{
        "status": st, "start": mmss(s), "start_clock": wall(s),
        "end": mmss(e), "end_clock": wall(e), "duration_s": round(e - s, 1),
    } for st, s, e in raw])

def build_zone_events(run):
    rows = [{
        "person_id": f"P{e['track_id']}" if (isinstance(e['track_id'], int) or str(e['track_id']).isdigit()) else str(e['track_id']), "role": e["role"],
        "zone": e["zone"], "entry": mmss(e["t_in"]),
        "entry_clock": wall(e["t_in"]), "exit": mmss(e["t_out"]),
        "exit_clock": wall(e["t_out"]), "duration_s": e["duration"],
    } for e in sorted(run["events"], key=lambda e: e["t_in"])]
    return pd.DataFrame(rows)

def build_zone_summary(run):
    zones = defaultdict(list)
    for e in run["events"]:
        zones[e["zone"]].append(e)
    rows = []
    for zone, evs in sorted(zones.items()):
        people = {e["track_id"] for e in evs}
        rows.append({
            "zone": zone,
            "unique_people": len(people),
            "customers": len({e["track_id"] for e in evs
                              if e["role"] == "customer"}),
            "staff": len({e["track_id"] for e in evs if e["role"] == "staff"}),
            "events": len(evs),
            "avg_dwell_s": round(sum(e["duration"] for e in evs) / len(evs), 1),
            "max_dwell_s": round(max(e["duration"] for e in evs), 1),
            "person_minutes": round(sum(e["duration"] for e in evs) / 60, 1),
        })
    return pd.DataFrame(rows)

xlsx_path = OUTPUT_DIR / "venue_analytics.xlsx"
answer_frames = []
for camera_id, adf in ANSWERS_DF_BY_VENUE.items():
    tagged = adf.copy()
    tagged.insert(0, "camera_id", camera_id)
    answer_frames.append(tagged)
sheets = []
if answer_frames:
    sheets.append(("Answers", pd.concat(answer_frames, ignore_index=True), False))
for run in runs:
    tag = "" if len(runs) == 1 else " " + run["camera_id"][:14]
    sheets.append((f"Person Log{tag}", build_person_log(run), True))
    sheets.append((f"Zone Visits{tag}", build_zone_visits(run), True))
    sheets.append((f"Zone Events{tag}", build_zone_events(run), True))
    sheets.append((f"Zone Summary{tag}", build_zone_summary(run), False))
    staff_log = build_staff_zone_log(run)
    if len(staff_log):
        sheets.append((f"Staff Zone Log{tag}", staff_log, False))
    merges_df = build_id_merges(run)
    if len(merges_df):
        sheets.append((f"ID Merges{tag}", merges_df, False))
for camera_id, per_party in PER_PARTY_BY_VENUE.items():
    if per_party:
        pp_df = pd.DataFrame(per_party)
        pp_df.insert(0, "camera_id", camera_id)
        pp_df["party_start_clock"] = pp_df["party_start"].map(wall)
        tag = "" if len(runs) == 1 else " " + camera_id[:14]
        sheets.append((f"Table Service{tag}", pp_df, False))
if minutes:
    win_df = pd.DataFrame(minutes)
    win_df["person_ids"] = win_df["person_ids"].map(
        lambda ids: ", ".join(ids[:20]) + (f" +{len(ids)-20} more"
                                           if len(ids) > 20 else ""))
    sheets.append((f"{WINDOW_S}s Windows", win_df, False))

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    used = set()
    for name, df, role_style in sheets:
        sheet = name[:28]
        while sheet in used:            # openpyxl needs unique sheet names
            sheet = sheet[:26] + "_2"
        used.add(sheet)
        if len(df) and role_style and HAS_STYLER:
            style_by_role(df).to_excel(xw, sheet_name=sheet, index=False)
        else:
            df.to_excel(xw, sheet_name=sheet, index=False)

# post-pass: bold headers, freeze top row, sensible column widths
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
wb = load_workbook(xlsx_path)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E79")
    for col in ws.columns:
        width = max((len(str(c.value)) for c in col[:80] if c.value), default=8)
        ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 9), 46)
wb.save(xlsx_path)
print(f"📊 Excel workbook -> {xlsx_path}")
print("   sheets:", [ws.title for ws in wb.worksheets])


In [ ]:
# Cell 15 — EXPORT: events CSV + answers JSON + browser-playable videos + zip
import csv, shutil
from IPython.display import Video, display as idisplay

# `runs` was already built by Cell 9 above (one entry per processed venue)
for run in runs:
    p = OUTPUT_DIR / f"{run['camera_id']}_events.csv"
    with open(p, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["track_id", "role", "zone",
                                          "t_in", "t_out", "duration"])
        w.writeheader(); w.writerows(run["events"])
    print(f"{p}  ({len(run['events'])} events)")

answers = {"thresholds": {
    "MIN_SEATED_S": MIN_SEATED_S, "WAIT_THRESHOLD_S": WAIT_THRESHOLD_S,
    "VISIT_MIN_S": VISIT_MIN_S, "PARTY_GAP_S": PARTY_GAP_S,
    "DEMO_SCALE": DEMO_SCALE}}
for camera_id, ans in ANSWERS_BY_VENUE.items():
    answers[camera_id] = ans
    if PER_PARTY_BY_VENUE.get(camera_id):
        answers[f"{camera_id}_per_party_detail"] = PER_PARTY_BY_VENUE[camera_id]
(OUTPUT_DIR / "answers.json").write_text(json.dumps(answers, indent=2, default=str))
print("answers ->", OUTPUT_DIR / "answers.json")

# LIGHT results zip FIRST (Excel/CSV/JSON only, a few MB — downloads
# instantly) so it exists even if the slow video re-encode below is skipped
import zipfile
zip_path = BASE / "poc_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUTPUT_DIR.iterdir()):
        if f.suffix.lower() != ".mp4":
            zf.write(f, f.name)
print(f"📦 results zip (no videos, small): {zip_path} "
      f"({zip_path.stat().st_size // 1024} KB)")

# Re-encode annotated videos to H.264 (browser-playable); the bulky raw
# mp4v original is deleted after a successful re-encode. Videos are meant
# to be downloaded individually from the Output panel, not zipped.
import subprocess
for run in runs:
    _vids = run.get("annotated_videos") or ([run["annotated_video"]]
                                            if run.get("annotated_video") else [])
    if not _vids:
        print(f"⚠️  {run['camera_id']}: no annotated video was produced this "
              f"run — check the render step above, do NOT ship this export "
              f"as if it were complete")
    _h264 = []
    for _v in _vids:                       # v55: one per chunk, not one per run
        srcp = Path(_v)
        if not srcp.exists():
            continue
        if srcp.stem.endswith("_h264"):        # already done per chunk
            _h264.append(str(srcp))
            continue
        dst = srcp.with_name(srcp.stem + "_h264.mp4")
        try:
            r = subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", str(srcp),
                 "-vf", f"scale='min({globals().get('RENDER_MAX_W', 1280)},iw)':-2",
                 "-c:v", "libx264", "-pix_fmt", "yuv420p",
                 "-crf", str(globals().get("RENDER_CRF", 26)),
                 "-movflags", "+faststart",   # browser can stream it
                 str(dst)], timeout=3600)
        except FileNotFoundError:
            print(f"⚠️  ffmpeg not installed — keeping the raw {srcp.name} "
                  f"(may not play in a browser)")
            _h264.append(str(srcp))
            continue
        except subprocess.TimeoutExpired:
            print(f"⚠️  ffmpeg timed out on {srcp.name} — keeping the raw file")
            _h264.append(str(srcp))
            continue
        if r.returncode == 0:
            srcp.unlink(missing_ok=True)
            _h264.append(str(dst))
            print(f"browser-playable -> {dst.name} "
                  f"({dst.stat().st_size // 1_000_000} MB)")
    run["annotated_videos_h264"] = _h264
    run["annotated_video_h264"] = _h264[0] if _h264 else None

import os
for run in runs:                      # inline playback of the final videos
    for v in (run.get("annotated_videos_h264") or [])[:3]:
        print(f"\n▶ {run['camera_id']} annotated video:")
        try:  # relative path — absolute paths 404 in the notebook's web server
            idisplay(Video(os.path.relpath(v, Path.cwd()), embed=False, width=760))
        except Exception as exc:
            print(f"(inline playback unavailable: {exc} — "
                  f"download poc_output.zip instead)")


In [ ]:
# Cell 17c — CLIP STAFF CHALLENGER  (measures, never overrides)
# Staff/customer is the biggest single error source in this pipeline: get it
# wrong and BOTH halves break at once — staff counted as guests inflates the
# guest count, guests counted as staff inflates desk coverage.
#
# Today that decision is a heuristic (Cell 2's STAFF_MIN_VIDEO_SHARE /
# STAFF_DOMINANCE_RATIO): "spent >=35% of the video in the staff zone and
# >=3x more time there than anywhere else". Clever, and it will fail on a
# host who roams, a manager passing through, and a guest who lingers.
#
# CLIP can judge from the PICTURE instead of from the geometry. But CLIP was
# trained on clean web photos; these crops are small, angled, often a
# person's back, and half the night they are infrared. That is out of
# distribution and the accuracy is genuinely unknown.
#
# So this runs as a CHALLENGER: it scores every person, reports where it
# disagrees with the heuristic, and CHANGES NOTHING. Look at the
# disagreements, decide who was right, and only then consider promoting it.
# Same champion/challenger discipline as the forecasting work.

CLIP_ENABLED   = True
CLIP_MODEL     = "openai/clip-vit-base-patch32"   # small, fast, good enough to judge
CLIP_MARGIN    = 0.02      # |staff - customer| below this = "unsure", not a vote
CLIP_MAX_CROPS = 4         # per person; more crops = steadier vote

# Several prompts per class, averaged. A single prompt is brittle — this is
# standard practice and costs nothing.
STAFF_PROMPTS = [
    "a photo of a restaurant host working behind a reception desk",
    "a photo of a staff member in a work uniform",
    "a photo of a waiter or hostess at work in a restaurant",
    "an employee working at a front desk",
]
GUEST_PROMPTS = [
    "a photo of a restaurant guest arriving",
    "a photo of a customer in a coat waiting in a lobby",
    "a photo of a visitor standing in a reception area",
    "a member of the public entering a venue",
]

CLIP_REPORT = {}
if not CLIP_ENABLED:
    print("CLIP challenger disabled (CLIP_ENABLED=False)")
else:
    try:
        import torch, numpy as np
        from transformers import CLIPModel, CLIPProcessor
        from PIL import Image

        _dev = DEVICE if torch.cuda.is_available() else "cpu"
        print(f"loading {CLIP_MODEL} on {_dev} ...")
        _clip = CLIPModel.from_pretrained(CLIP_MODEL).to(_dev).eval()
        _proc = CLIPProcessor.from_pretrained(CLIP_MODEL)

        def _as_embeds(x, proj):
            # PATCH_V58P2: newer transformers return a ModelOutput here, not a
            # tensor — `.norm` then crashed and the except below silently ate
            # the whole challenger every run. Accept tensor / *_embeds /
            # pooler_output (projected so text+image share one space).
            import torch as _t
            if _t.is_tensor(x):
                return x
            for _k in ("text_embeds", "image_embeds"):
                _v = getattr(x, _k, None)
                if _v is not None:
                    return _v
            _v = getattr(x, "pooler_output", None)
            if _v is not None:
                return proj(_v)
            return x[0]

        with torch.no_grad():
            _ti = _proc(text=STAFF_PROMPTS + GUEST_PROMPTS,
                        return_tensors="pt", padding=True).to(_dev)
            _tv = _as_embeds(_clip.get_text_features(**_ti),
                             _clip.text_projection)
            _tv = _tv / _tv.norm(dim=-1, keepdim=True)
            _staff_vec = _tv[:len(STAFF_PROMPTS)].mean(0)
            _guest_vec = _tv[len(STAFF_PROMPTS):].mean(0)
            _staff_vec = _staff_vec / _staff_vec.norm()
            _guest_vec = _guest_vec / _guest_vec.norm()

        def clip_judge(crops):
            """-> (verdict, staff_score, guest_score). 'unsure' when the two
            scores are within CLIP_MARGIN — abstaining beats guessing."""
            imgs = []
            for c in crops[:CLIP_MAX_CROPS]:
                arr = c[1] if isinstance(c, tuple) else c
                if arr is None or getattr(arr, "size", 0) == 0:
                    continue
                if arr.ndim == 3 and arr.shape[2] == 3:      # BGR -> RGB
                    arr = arr[:, :, ::-1]
                imgs.append(Image.fromarray(np.ascontiguousarray(arr)))
            if not imgs:
                return "no_crop", None, None
            with torch.no_grad():
                ii = _proc(images=imgs, return_tensors="pt").to(_dev)
                iv = _as_embeds(_clip.get_image_features(**ii),
                                _clip.visual_projection)
                iv = iv / iv.norm(dim=-1, keepdim=True)
                s = float((iv @ _staff_vec).mean())
                g = float((iv @ _guest_vec).mean())
            if abs(s - g) < CLIP_MARGIN:
                return "unsure", s, g
            return ("staff" if s > g else "customer"), s, g

        for run in runs:
            dossiers = run.get("identity_dossiers") or {}
            roles = run.get("roles", {})
            if not dossiers:
                print(f"  {run['camera_id']}: no dossiers returned — skipping")
                continue
            rows, agree, disagree, unsure, nocrop = [], 0, 0, 0, 0
            for cid, d in dossiers.items():
                verdict, s, g = clip_judge(d.get("attire_crops") or [])
                heur = roles.get(cid, "unknown")
                if verdict == "no_crop":
                    nocrop += 1
                elif verdict == "unsure":
                    unsure += 1
                elif heur == "unknown":
                    pass
                elif verdict == heur:
                    agree += 1
                else:
                    disagree += 1
                rows.append({"person": cid, "heuristic": heur, "clip": verdict,
                             "staff_score": None if s is None else round(s, 4),
                             "guest_score": None if g is None else round(g, 4),
                             "t_first": round(d["t_first"], 1),
                             "seconds_seen": round(d["t_last"] - d["t_first"], 1)})
            CLIP_REPORT[run["camera_id"]] = rows
            judged = agree + disagree
            print("=" * 78)
            print(f"  {run['camera_id']} — CLIP STAFF CHALLENGER  (advisory only)")
            print("=" * 78)
            print(f"  people scored     {len(rows)}")
            print(f"  agree             {agree}"
                  + (f"  ({100*agree/judged:.0f}%)" if judged else ""))
            print(f"  DISAGREE          {disagree}   <- look at these")
            print(f"  clip unsure       {unsure}   (scores within {CLIP_MARGIN})")
            print(f"  no usable crop    {nocrop}")
            if judged and agree / judged < 0.7:
                print("\n  !!  under 70% agreement. One of the two is badly wrong.")
                print("      Do NOT promote CLIP on this evidence — check crops by hand.")
            bad = [r for r in rows if r["clip"] in ("staff", "customer")
                   and r["heuristic"] in ("staff", "customer")
                   and r["clip"] != r["heuristic"]]
            for r in sorted(bad, key=lambda r: -abs(r["staff_score"] - r["guest_score"]))[:12]:
                print(f"    {str(r['person']):>10s}  heuristic={r['heuristic']:<9s}"
                      f" clip={r['clip']:<9s} (s={r['staff_score']} g={r['guest_score']})"
                      f"  seen {r['seconds_seen']:.0f}s")
            pd.DataFrame(rows).to_csv(
                OUTPUT_DIR / f"{run['camera_id']}_clip_challenger.csv", index=False)
            print(f"\n  -> {run['camera_id']}_clip_challenger.csv")
            print("  NOTHING WAS CHANGED. This is evidence, not a decision.")

    except ImportError as _e:
        print("CLIP challenger skipped (optional): the installed transformers "
              "needs a newer torch than this box has "
              f"({_e}). The install cell now pins transformers<4.47 to fix "
              "this on the next fresh session. Heuristic roles are unaffected.")
    except Exception as _e:
        print(f"CLIP challenger unavailable ({type(_e).__name__}: {_e})")
        print("  -> heuristic roles are unaffected; this cell is advisory only")


In [ ]:
# Cell 17b — ID TIMELINE AUDIT  (complete per-frame identity report)
# Exports a small text/CSV report covering EVERY frame, so tracking quality can
# be reviewed without exporting hundreds of images. Auto-flags the failure modes
# that matter: an identity that teleports, one that lives in disjoint blocks
# (classic sign of a wrong merge), and ones that blink in and out.
import csv

# Calibrated from a real run: a person walking in a 1270px-wide frame moves
# ~300-600 px/s, and box jitter adds ~300 px/s of noise, so 400 flagged normal
# walking (612 false alarms). Only genuinely impossible motion should flag.
TELEPORT_PX_PER_S = 1500.0  # ~12 m/s - sprinting; impossible inside a shop
TELEPORT_MIN_PX   = 150.0   # and the jump must be large in absolute terms too,
                            # so per-frame box jitter can never trigger it
BLINK_GAP_S       = 1.5     # gap longer than this splits a segment
REAPPEAR_GAP_S    = 5.0     # vanish this long then return = wrong-merge risk
ENCOUNTER_IOU     = 0.35    # boxes overlapping this much = swap-risk moment
SHADOW_PX         = 60.0    # two ids this close for a long time = double box
SHADOW_MIN_S      = 4.0     # ...sustained at least this long
EDGE_MARGIN_PX    = 60      # a track ending this close to a border = left frame     # vanish this long then return = classic
                            # wrong-merge signature; always report it

for _run in runs:
    _cam = _run["camera_id"]
    _flog = _run.get("frame_log") or []
    _canon = _run.get("canon_map") or {}
    _roles = _run.get("roles") or {}
    if not _flog:
        print(f"(no frame_log for {_cam} — re-run process_video to enable the audit)")
        continue

    # rebuild: identity -> [(t, cx, cy)] using the SAME canonical ids the video shows
    _seen = {}
    _order = []
    for _fi, _t, _boxes in _flog:
        for _tid, _x1, _y1, _x2, _y2 in _boxes:
            _cid = _canon.get(_tid, _tid)
            if _cid not in _seen:
                _seen[_cid] = []
                _order.append(_cid)
            _seen[_cid].append((_t, (_x1 + _x2) / 2.0, float(_y2)))
    _pnum = {c: i + 1 for i, c in enumerate(_order)}
    _seen_key = {}   # matches video P-numbers

    _rows, _flags = [], []
    for _cid, _pts in _seen.items():
        _pts.sort()
        _label = (f"{_cid}" if isinstance(_cid, str) and not str(_cid).isdigit()
                  else f"P{_pnum[_cid]}")
        # split into segments on long gaps
        _segs, _cur = [], [_pts[0]]
        for _prev, _now in zip(_pts, _pts[1:]):
            if _now[0] - _prev[0] > BLINK_GAP_S:
                _segs.append(_cur); _cur = [_now]
            else:
                _cur.append(_now)
            # teleport check between consecutive sightings
            _dt = max(1e-3, _now[0] - _prev[0])
            _d = ((_now[1] - _prev[1]) ** 2 + (_now[2] - _prev[2]) ** 2) ** 0.5
            if _d / _dt > TELEPORT_PX_PER_S and _d >= TELEPORT_MIN_PX:
                _flags.append(f"  TELEPORT {_label}: {_d:.0f}px in {_dt:.1f}s "
                              f"at t={_prev[0]:.1f}->{_now[0]:.1f} "
                              f"(possible wrong merge / id steal)")
        _segs.append(_cur)
        for _si, _s in enumerate(_segs):
            _rows.append({"identity": _label, "role": _roles.get(_cid, "customer"),
                          "segment": _si + 1, "t_start": round(_s[0][0], 1),
                          "t_end": round(_s[-1][0], 1),
                          "duration_s": round(_s[-1][0] - _s[0][0], 1),
                          "sightings": len(_s),
                          "x_start": int(_s[0][1]), "y_start": int(_s[0][2]),
                          "x_end": int(_s[-1][1]), "y_end": int(_s[-1][2])})
        for _k in range(len(_segs) - 1):
            _gap = _segs[_k + 1][0][0] - _segs[_k][-1][0]
            _jump = (((_segs[_k + 1][0][1] - _segs[_k][-1][1]) ** 2
                      + (_segs[_k + 1][0][2] - _segs[_k][-1][2]) ** 2) ** 0.5)
            if _gap >= REAPPEAR_GAP_S:
                _flags.append(
                    f"  REAPPEARED {_label}: gone {_gap:.0f}s "
                    f"(t={_segs[_k][-1][0]:.0f}->{_segs[_k+1][0][0]:.0f}), returned "
                    f"{_jump:.0f}px away - VERIFY same person (long gap + moved "
                    f"position is the classic wrong-merge signature)")
        if len(_segs) > 2:
            _flags.append(f"  SPLIT LIFE {_label}: {len(_segs)} separate blocks - "
                          f"one identity used at disjoint times often means two "
                          f"people were merged")


    # ---- extra checks the segment logic alone cannot see -------------------
    # (a) SWAP RISK: heavy box overlap between two identities in the same frame
    _enc = {}
    for _fi, _t2, _boxes in _flog:
        for _m in range(len(_boxes)):
            for _n in range(_m + 1, len(_boxes)):
                _ta, _ax1, _ay1, _ax2, _ay2 = _boxes[_m]
                _tb, _bx1, _by1, _bx2, _by2 = _boxes[_n]
                _ca = _canon.get(_ta, _ta); _cb = _canon.get(_tb, _tb)
                if _ca == _cb:
                    continue
                _ix = max(0, min(_ax2, _bx2) - max(_ax1, _bx1))
                _iy = max(0, min(_ay2, _by2) - max(_ay1, _by1))
                _inter = _ix * _iy
                if _inter <= 0:
                    continue
                _ua = (_ax2-_ax1)*(_ay2-_ay1) + (_bx2-_bx1)*(_by2-_by1) - _inter
                if _ua > 0 and _inter / _ua >= ENCOUNTER_IOU:
                    _k = tuple(sorted((str(_ca), str(_cb))))
                    _e = _enc.setdefault(_k, [_t2, _t2, 0])
                    _e[1] = _t2; _e[2] += 1
    for (_ca, _cb), (_t0, _t1, _cnt) in sorted(_enc.items(), key=lambda kv: -kv[1][2]):
        _la = next((r["identity"] for r in _rows if str(_seen_key.get(r["identity"], "")) == _ca), _ca)
        _flags.append(f"  SWAP RISK {_ca} <-> {_cb}: boxes overlapped for "
                      f"{_cnt} frames (t={_t0:.0f}-{_t1:.0f}s) - a clean id swap here "
                      f"leaves both tracks smooth and is INVISIBLE to gap checks; "
                      f"eyeball this timestamp")

    # (b) VANISHED mid-frame = likely missed detection / lost track
    _W = max((_b[3] for _f, _tt, _bx in _flog for _b in _bx), default=1280)
    _H = max((_b[4] for _f, _tt, _bx in _flog for _b in _bx), default=720)
    for _cid, _pts in _seen.items():
        _lab = (f"{_cid}" if isinstance(_cid, str) and not str(_cid).isdigit()
                else f"P{_pnum[_cid]}")
        _lt, _lx, _ly = _pts[-1]
        if _lt < _flog[-1][1] - 3.0:      # stopped well before the video ended
            _near_edge = (_lx <= EDGE_MARGIN_PX or _lx >= _W - EDGE_MARGIN_PX
                          or _ly <= EDGE_MARGIN_PX or _ly >= _H - EDGE_MARGIN_PX)
            if not _near_edge:
                _flags.append(f"  VANISHED {_lab}: track ends at t={_lt:.0f}s in the "
                              f"MIDDLE of frame ({int(_lx)},{int(_ly)}) - person did not "
                              f"exit, so detector likely lost them (missed detection)")

    # (c) SHADOW: two ids glued together = one person with two boxes
    _pos_at = {}
    for _cid, _pts in _seen.items():
        for _tt, _x, _y in _pts:
            _pos_at.setdefault(round(_tt, 1), {})[_cid] = (_x, _y)
    _pair_close = {}
    for _tt, _d in _pos_at.items():
        _ids = list(_d)
        for _m in range(len(_ids)):
            for _n in range(_m + 1, len(_ids)):
                _a, _b = _ids[_m], _ids[_n]
                _dx = _d[_a][0] - _d[_b][0]; _dy = _d[_a][1] - _d[_b][1]
                if (_dx*_dx + _dy*_dy) ** 0.5 <= SHADOW_PX:
                    # one co-located sample = one analysed frame-step of time.
                    # the old +0.1 assumed 10 samples/s while analysis runs at
                    # ~7.5 fps, understating shadow durations by ~25%
                    _pair_close[tuple(sorted((str(_a), str(_b))))] = \
                        _pair_close.get(tuple(sorted((str(_a), str(_b)))), 0) \
                        + 1.0 / float(globals().get("ANALYSIS_FPS", 8) or 8)
    for (_a, _b), _dur in _pair_close.items():
        if _dur >= SHADOW_MIN_S:
            _flags.append(f"  SHADOW {_a} <-> {_b}: stayed within {SHADOW_PX:.0f}px for "
                          f"~{_dur:.0f}s - likely ONE person carrying two ids "
                          f"(double box), which inflates the person count")

    _csv = OUTPUT_DIR / f"{_cam}_id_timeline.csv"
    with open(_csv, "w", newline="", encoding="utf-8") as _f:
        _w = csv.DictWriter(_f, fieldnames=list(_rows[0].keys()))
        _w.writeheader(); _w.writerows(_rows)

    _lines = []
    _lines.append("=" * 72)
    _lines.append(f"ID TIMELINE AUDIT - {_cam}   (covers EVERY frame, not a sample)")
    _lines.append("=" * 72)
    _lines.append(f"video length: {_flog[-1][1]:.1f}s   distinct identities: {len(_seen)}")
    _lines.append("")
    _lines.append(f"{'identity':<10}{'role':<10}{'first':>7}{'last':>8}{'secs':>7}"
                  f"{'segs':>6}   path")
    _byid = {}
    for _r in _rows:
        _byid.setdefault(_r["identity"], []).append(_r)
    for _lab in sorted(_byid, key=lambda L: _byid[L][0]["t_start"]):
        _rs = _byid[_lab]
        _lines.append(f"{_lab:<10}{_rs[0]['role']:<10}{_rs[0]['t_start']:>7.1f}"
                      f"{_rs[-1]['t_end']:>8.1f}"
                      f"{sum(r['duration_s'] for r in _rs):>7.1f}{len(_rs):>6}   "
                      f"({_rs[0]['x_start']},{_rs[0]['y_start']})->"
                      f"({_rs[-1]['x_end']},{_rs[-1]['y_end']})")
    _lines.append("")
    if _flags:
        _lines.append(f"SUSPICIOUS EVENTS ({len(_flags)}):")
        _lines.extend(_flags[:40])
        if len(_flags) > 40:
            _lines.append(f"  ... and {len(_flags) - 40} more")
    else:
        _lines.append("SUSPICIOUS EVENTS: none - no teleports or split lives detected")
    _txt = "\n".join(_lines)
    (OUTPUT_DIR / f"{_cam}_id_audit.txt").write_text(_txt, encoding="utf-8")
    # hand the count to the self-audit (Cell 16): 262 suspicious events and a
    # "demo-grade" verdict must never be printable in the same run again
    _run["id_audit_flags"] = len(_flags)
    print(_txt)
    print(f"\nsaved -> {_csv.name} and {_cam}_id_audit.txt "
          f"(paste the .txt into chat for a full review)")


In [ ]:
# Cell 16 — PIPELINE SELF-AUDIT: auto red-flags for the failure modes that
# corrupted earlier runs. Green ticks = numbers are trustworthy; warnings tell
# you exactly what to check before presenting anything.
for run in runs:
    ev = run["events"]
    life = {}
    for e in ev:
        lo, hi = life.get(e["track_id"], (e["t_in"], e["t_out"]))
        life[e["track_id"]] = (min(lo, e["t_in"]), max(hi, e["t_out"]))
    # VISIBLE time, not span: span (max-min) rewards exactly the failure this
    # check exists to catch — an over-merged identity visible 28s across a
    # 44-minute span scored 2664s and the audit called it healthy.
    from collections import defaultdict as _dd27
    _vis = _dd27(float)
    for e in ev:
        _vis[e["track_id"]] += e["duration"]
    lives = sorted(_vis.values())
    med_life = lives[len(lives) // 2] if lives else 0.0
    n_ids = len(life)
    entered = {c["track_id"] for c in run["crossings"] if c["direction"] == "in"}
    _meta = VENUE_META.get(run["camera_id"], {})
    _seat_z = _meta.get("seating_zones") or set()
    _staff_z = _meta.get("staff_zones") or set()
    seated = {e["track_id"] for e in ev if e["duration"] >= MIN_SEATED_S
              and e["zone"] in _seat_z}
    chain = len(entered & seated)
    rec_staff = sum(e["duration"] for e in ev
                    if e["zone"] in _staff_z and e["role"] == "staff")
    rec_cust = sum(e["duration"] for e in ev
                   if e["zone"] in _staff_z and e["role"] == "customer")
    _hours = max(total_duration(covered_windows(run)) / 3600.0, 1e-6)  # footage, not span
    _max_plausible = int(60 * _hours) + 60       # v55: scales with the night
    checks = [
        (med_life >= 45,
         f"median identity VISIBLE time {med_life:.0f}s",
         f"median identity VISIBLE time {med_life:.0f}s (<45s = fragmentation "
         f"or low recall; per-person dwell/greet numbers undercount)"),
        (run.get("id_audit_flags", 0) <= 20,
         f"id-timeline audit: {run.get('id_audit_flags', 0)} suspicious "
         f"event(s)" + ("" if "id_audit_flags" in run else
                        "  (audit cell not run yet — re-run Cell 17b first)"),
         f"id-timeline audit flagged {run.get('id_audit_flags')} suspicious "
         f"events (teleports/reappears/shadows) — per-person numbers are NOT "
         f"demo-grade until these are reviewed"),
        (n_ids <= _max_plausible,
         f"{n_ids} unique identities over {_hours:.1f} h — plausible "
         f"(<= {_max_plausible})",
         f"{n_ids} unique identities over {_hours:.1f} h — above the "
         f"{_max_plausible} plausibility bar, suspect ID churn"),
        (chain > 0 or not entered or not seated,
         f"door->seat identity chain intact ({chain} entrants seated)",
         "NO entrant was ever seated — identities break between door and "
         "table (the run-2 pathology)"),
        (rec_staff >= rec_cust or (rec_staff + rec_cust) < 30,
         f"reception owned by staff ({rec_staff:.0f}s vs {rec_cust:.0f}s)",
         f"reception owned by CUSTOMERS ({rec_cust:.0f}s vs {rec_staff:.0f}s)"
         " — staff rule / zone anchor suspect"),
        (bool(run.get("id_merges")) or n_ids <= 60,
         f"Re-ID stitching merged {len(run.get('id_merges', {}))} fragments",
         "Re-ID stitching merged NOTHING despite many ids — verify CLIP-ReID/OSNet"),
        (run.get("online_reid") in ("clip", "osnet"),
         f"live tracker used REAL {str(run.get('online_reid')).upper()} "
         f"embeddings (overlap-swap fix active)",
         f"live tracker fell back to the with_reid=auto proxy "
         f"(online_reid={run.get('online_reid')!r}) — ID swaps during "
         f"overlap are more likely on this run; check the CLIP-ReID/OSNet "
         f"load warning printed above"),
        (len(run.get('face_veto_report') or []) == 0,
         "face veto found no confident face-mismatched merges",
         f"face veto reversed {len(run.get('face_veto_report') or [])} merge(s) on a confident face "
         f"mismatch — the body-appearance backbone is producing wrong "
         f"merges on this footage; consider tightening REID_SIM_THRESHOLD/"
         f"ANCHOR_SIM_THRESHOLD (see 📏 Calibration report above)"),
        (len((run.get('face_validation_report') or {}).get('disagree', [])) == 0,
         "no borderline face disagreements left unresolved",
         f"{len((run.get('face_validation_report') or {}).get('disagree', []))} merge(s) still have a face-on-both-sides "
         f"disagreement below veto confidence — not strong enough to "
         f"auto-reverse, but worth a manual crop look before presenting"),
    ]
    print(f"— SELF-AUDIT · {run['camera_id']} —")
    ok_all = True
    for ok, good, bad in checks:
        print("  " + ("✅ " + good if ok else "⚠️  " + bad))
        ok_all &= ok
    print("  VERDICT:", "numbers are demo-grade" if ok_all
          else "fix warnings before presenting per-person numbers")


In [ ]:
# Cell 20 — FULL-NIGHT COVERAGE LEDGER
# The annotated video only shows the minutes somebody was in shot. The REPORT
# has to cover every minute of the night, including the empty ones — "nothing
# happened between 01:12 and 02:40" is itself a finding (and the only way to
# tell a quiet night apart from a pipeline that stopped seeing people).
import numpy as np, pandas as pd, json, math
from collections import defaultdict

LEDGER_BIN_S = 60.0          # one row per minute of real time

def coverage_ledger(run, meta, bin_s=LEDGER_BIN_S):
    cov = covered_windows(run)
    # v55: start at the first minute that HAS footage. A 20-minute peak window
    # at 18:41 otherwise produced 11 hours of NO FOOTAGE rows before it.
    t_origin = (int(cov[0][0] // bin_s) * bin_s) if cov else 0.0
    zones = sorted({e["zone"] for e in run["events"]})
    t_end = float(run["t_end"])
    nb = int(math.ceil((t_end - t_origin) / bin_s)) or 1
    per = {z: [set() for _ in range(nb)] for z in zones}
    staff_bins = [set() for _ in range(nb)]
    staff_secs = [0.0] * nb   # actual staffed SECONDS per bin, not touched-bins
    for e in run["events"]:
        b0 = max(0, int((e["t_in"] - t_origin) // bin_s))
        b1 = min(nb - 1, int((e["t_out"] - t_origin) // bin_s))
        for b in range(b0, b1 + 1):
            per[e["zone"]][b].add(e["track_id"])
            if e["role"] == "staff" and e["zone"] in meta["staff_zones"]:
                staff_bins[b].add(e["track_id"])
                staff_secs[b] += max(0.0, min(e["t_out"], t_origin + (b + 1) * bin_s)
                                     - max(e["t_in"], t_origin + b * bin_s))
    ins = defaultdict(int)
    for c in run["crossings"]:
        if c["direction"] == "in" and run["roles"].get(c["track_id"]) != "staff":
            ins[min(nb - 1, max(0, int((c["t"] - t_origin) // bin_s)))] += 1
    rows = []
    for b in range(nb):
        t0 = t_origin + b * bin_s
        present = set().union(*[per[z][b] for z in zones]) if zones else set()
        row = {"minute": b, "clock": wall(t0) or mmss(t0), "t_start_s": t0,
               "people_present": len(present),
               "staff_at_station": len(staff_bins[b]),
               "station_covered": bool(staff_bins[b]),
               "station_covered_s": round(min(staff_secs[b], bin_s), 1),
               "arrivals": ins.get(b, 0)}
        for z in zones:
            row[f"n_{z}"] = len(per[z][b])
        has_footage = any(s < t0 + bin_s and e > t0 for s, e in cov)
        row["has_footage"] = has_footage
        row["status"] = ("NO FOOTAGE" if not has_footage else
                         "EMPTY" if not present else
                         "guests, no staff at station" if not staff_bins[b]
                         else "normal")
        rows.append(row)
    return pd.DataFrame(rows), zones


def quiet_stretches(df, min_min=5):
    """Runs of consecutive EMPTY minutes — the 'nothing happened here' list."""
    out, start = [], None
    for i, r in df.iterrows():
        if not r.get("has_footage", True):
            start = None      # a missing hour is not a quiet hour
            continue
        if r["people_present"] == 0:
            start = r if start is None else start
        elif start is not None:
            if i - start["minute"] >= min_min:
                out.append((start["clock"], df.iloc[i - 1]["clock"], i - start["minute"]))
            start = None
    if start is not None and len(df) - start["minute"] >= min_min:
        out.append((start["clock"], df.iloc[-1]["clock"], len(df) - start["minute"]))
    return out


LEDGERS, QUIET = {}, {}
for run in runs:
    cam = run["camera_id"]
    meta = VENUE_META[cam]
    df, zones = coverage_ledger(run, meta)
    LEDGERS[cam] = df
    QUIET[cam] = quiet_stretches(df)
    p = OUTPUT_DIR / f"{cam}_coverage_by_minute.csv"
    df.to_csv(p, index=False)

    _f = df[df["has_footage"]]
    # SECOND-weighted, so this number agrees with Cell 18's desk_covered_pct.
    # The old boolean-minute mean gave 60s of credit for 1s of presence and
    # printed 27.9% while Cell 18 said 9.1% for the same night.
    covered = (100.0 * _f["station_covered_s"].sum() / max(60.0 * len(_f), 1e-6)
               if len(_f) else 0.0)
    covered_touch = 100.0 * _f["station_covered"].mean() if len(_f) else 0.0
    empty = 100.0 * (_f["people_present"] == 0).mean() if len(_f) else 0.0
    _missing = int((~df["has_footage"]).sum())
    print("=" * 78)
    print(f"  {cam} — FULL-NIGHT LEDGER · {len(df)} minutes · every minute accounted for")
    print("=" * 78)
    if _missing:
        print(f"  ⚠️  NO FOOTAGE       {_missing} of {len(df)} minutes "
              f"({100.0 * _missing / len(df):.0f}%) — missing video, NOT a quiet "
              f"room. Percentages below are over the {len(_f)} minutes we have.")
    print(f"  station attended     {covered:5.1f}% of footage time "
          f"(sec-weighted; touched {covered_touch:.0f}% of minutes)")
    print(f"  nobody in shot       {empty:5.1f}% of minutes WITH footage")
    print(f"  guests present, no staff at station: "
          f"{int((df['status'] == 'guests, no staff at station').sum())} minutes")
    print(f"  -> {p.name}")
    if QUIET[cam]:
        print(f"\n  quiet stretches (>=5 min with nobody in shot):")
        for a, b, n in QUIET[cam][:12]:
            print(f"     {a} -> {b}   ({n} min)")
        if len(QUIET[cam]) > 12:
            print(f"     ... and {len(QUIET[cam]) - 12} more (full list in the CSV)")

    # one picture of the whole night: zones down the side, time across
    if zones:
        M = np.array([[df.iloc[b][f"n_{z}"] for b in range(len(df))] for z in zones],
                     dtype=float)
        fig, ax = plt.subplots(figsize=(15, 0.55 * len(zones) + 1.8))
        ax.imshow(M, aspect="auto", cmap="magma", interpolation="nearest")
        ax.set_yticks(range(len(zones))); ax.set_yticklabels(zones)
        step = max(1, len(df) // 14)
        ax.set_xticks(range(0, len(df), step))
        ax.set_xticklabels([df.iloc[i]["clock"] for i in range(0, len(df), step)],
                           rotation=0, fontsize=8)
        ax.set_title(f"{cam} — people per zone, every minute of the night "
                     f"(black = nobody)", loc="left", weight="bold")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"{cam}_night_ledger.png", dpi=130, facecolor=SURFACE)
        plt.show()

# the exact night-time of every frame in the annotated video, so a moment
# timestamp can be turned into a video timestamp (Cell 19 uses this).
RENDER_INDEX = None


In [ ]:
# Cell 21 — THE DELIVERABLE: REPORT.md + end-card + free-text ask()
# Three things a person actually uses: a markdown report that renders anywhere,
# a summary card burned onto the end of the video so the video is self-
# contained, and a question box.
import json, os, subprocess, textwrap
from pathlib import Path

# ── EXACT vs PROXY. Every number carries its own status, so nobody has to
# remember which is which in the meeting. ────────────────────────────────────
STATUS = {
    "guests_tonight":            ("EXACT*", "unique inward door crossings; * identity is stitched across chunks, see limits"),
    "door_crossings_in":         ("EXACT",  "raw inward crossings incl. re-entries"),
    "people_who_came_back":      ("EXACT*", "guests with >1 inward crossing"),
    "busiest_15min":             ("EXACT",  "bucket with most arrivals"),
    "peak_arrivals_in_15min":    ("EXACT",  ""),
    "greeted":                   ("PROXY",  "staff box within 1.2 body-heights for >=3 s"),
    "avg_greet_seconds":         ("PROXY",  "proximity, not conversation"),
    "median_greet_seconds":      ("PROXY",  ""),
    "slowest_greet_seconds":     ("PROXY",  ""),
    "waited_over_3min":          ("PROXY",  "based on the greet proxy"),
    "left_without_being_served": ("PROXY",  "no proximity event found — a greeting from behind the desk can be missed"),
    "walked_out_under_90s":      ("PROXY",  ""),
    "arrived_to_an_empty_desk":  ("EXACT",  "arrival while no staff-role person was inside the station polygon"),
    "desk_covered_pct":          ("EXACT",  "station occupancy over the whole night"),
    "longest_desk_gap_min":      ("EXACT",  ""),
    "quick_steps_away":          ("EXACT",  ""),
    "real_absences":             ("EXACT",  ""),
    "long_breaks":               ("EXACT",  ""),
    "people_working_the_desk":   ("WEAK",   "distinct identities with >5 min at the station; without a face in the gallery this over-counts"),
}


def guest_confidence(run):
    """#5: the guest count depends on identity holding together for hours, so
    report it as a range. id_confidence is already computed per person (dwell,
    evidence tiers, face, door crossing, fragment count) — this just splits the
    guests on it instead of pretending one integer is certain."""
    per = defaultdict(list)
    for e in run["events"]:
        per[e["track_id"]].append(e)
    guests = {c["track_id"] for c in run["crossings"]
              if c["direction"] == "in" and run["roles"].get(c["track_id"]) != "staff"}
    scores = {g: _compute_id_confidence(g, run, per.get(g, [])) for g in guests}
    high = sum(1 for v in scores.values() if v >= 60)
    return {"total": len(guests), "high": high, "low": len(guests) - high,
            "scores": scores}


def target_check(r):
    """#8: measurement is only useful next to what good looks like."""
    rows = []
    for k, target in (globals().get("TARGETS") or {}).items():
        if target is None or k not in r or r[k] is None:
            continue
        v = r[k]
        lower_is_better = k in ("median_greet_seconds", "avg_greet_seconds",
                                "arrived_to_an_empty_desk", "walked_out_under_90s",
                                "waited_over_3min", "left_without_being_served")
        passed = (v <= target) if lower_is_better else (v >= target)
        rows.append((k, v, target, "PASS" if passed else "MISS",
                     "<=" if lower_is_better else ">="))
    return rows


def build_report(cam, r, run, ledger, quiet):
    L = []
    A = L.append
    hrs = run["t_end"] / 3600.0
    A(f"# {cam} — {_hm(0)} to {_hm(run['t_end'])}  ({hrs:.1f} h)")
    A("")
    # U1: if the view no longer matches the one the zones were drawn on, every
    # spatial number below is measuring the wrong part of the room. Refusing to
    # print them is the whole point — a missing number is recoverable, a
    # confident wrong one gets quoted in a meeting.
    if not run.get("view_valid", True):
        A("> # \U0001f6a8 THIS REPORT IS NOT VALID")
        A("> ")
        A("> The camera view no longer matches the one the zones were drawn on, "
          "so every zone, line and dwell number would be measuring the wrong "
          "part of the room. They are withheld deliberately.")
        A("> ")
        for _rsn in run.get("view_reasons", []):
            A(f"> - {_rsn}")
        A("> ")
        A("> **What still holds:** how many people were DETECTED, and the "
          "annotated video. Nothing that depends on where they were.")
        A("> ")
        A("> **To fix:** restore the camera to its original aim, or re-draw the "
          "zones on a current frame and delete `poc_output/viewref_*` so a new "
          "reference is taken.")
        A("")
        return "\n".join(L)
    if not run.get("view_checked", True):
        A("> \u26a0\ufe0f The camera view could not be verified this run, so nothing "
          "confirms the zones still line up with the room.")
        A("")
    A(BRIEFS.get(cam, ""))
    A("")
    _tc = target_check(r)
    if _tc:
        A("## Against target")
        A("")
        A("| what | measured | target | |")
        A("|---|---|---|---|")
        for k, v, target, verdict, op in _tc:
            A(f"| {k.replace('_', ' ')} | {v} | {op} {target} | "
              f"{'✅ PASS' if verdict == 'PASS' else '❌ MISS'} |")
        A("")
    _gc = guest_confidence(run)
    A("## How many guests, honestly")
    A("")
    A(f"**{_gc['total']} guests** — of which **{_gc['high']} are high-confidence "
      f"identities** and {_gc['low']} are uncertain (short appearance, no face, "
      f"or stitched from fragments).")
    A("")
    A(f"Read it as *{_gc['high']}-{_gc['total']} people*. The uncertainty is "
      f"identity, not detection: those people were definitely seen, we are less "
      f"sure whether two sightings were the same person. Per-person confidence "
      f"is in the Person Log sheet.")
    A("")
    A("## The numbers")
    A("")
    A("| metric | value | status | how it is computed |")
    A("|---|---|---|---|")
    for k, v in r.items():
        if k.startswith("_"):
            continue
        st, note = STATUS.get(k, ("", ""))
        A(f"| {k.replace('_', ' ')} | {v} | {st} | {note} |")
    A("")
    A("`EXACT` = geometry (a line was crossed, a polygon was occupied). "
      "`PROXY` = an inference from proximity or dwell — directionally right, "
      "not evidence of what was said. `WEAK` = do not quote without checking.")
    A("")
    A("## Every minute of the night")
    A("")
    _miss = int((~ledger["has_footage"]).sum()) if "has_footage" in ledger else 0
    if _miss:
        A(f"> ⚠️ **{_miss} minutes of this window have no video at all** "
          f"({r['missing_hours']:.1f} h missing of a {r['night_span_hours']:.1f} h "
          f"span). They are marked `NO FOOTAGE`, never `EMPTY`, and every "
          f"percentage here is over the {r['footage_hours']:.1f} h we do have.")
        A("")
    A(f"- minutes covered by the ledger: **{len(ledger)}** (`{cam}_coverage_by_minute.csv`)")
    A(f"- station covered: **{100.0 * ledger['station_covered'].mean():.1f}%** of minutes")
    A(f"- nobody in shot: **{100.0 * (ledger['people_present'] == 0).mean():.1f}%** of minutes")
    A(f"- guests present with no staff at the station: "
      f"**{int((ledger['status'] == 'guests, no staff at station').sum())}** minutes")
    if quiet:
        A("")
        A("Quiet stretches (nobody in shot for 5+ minutes):")
        A("")
        A("| from | to | minutes |")
        A("|---|---|---|")
        for a, b, n in quiet[:20]:
            A(f"| {a} | {b} | {n} |")
    A("")
    A("## Busiest / quietest")
    A("")
    A("| 15 min from | arrivals | exits | desk covered | arrived to empty desk |")
    A("|---|---|---|---|---|")
    for b in r["_buckets"]:
        A(f"| {b['from']} | {b['arrivals']} | {b['exits']} | "
          f"{b['desk_covered_pct']}% | {b['unattended_arrivals']} |")
    A("")
    if r["_shifts"]:
        A("## Who worked the station")
        A("")
        A("| identity | on station | first | last |")
        A("|---|---|---|---|")
        for s in r["_shifts"]:
            A(f"| {s['staff']} | {s['on_station_s'] / 60:.0f} min | "
              f"{_hm(s['first'])} | {_hm(s['last'])} |")
        _named = run.get("staff_matched_names") or []
        A("")
        A(f"Face-gallery matches this run: **{_named or 'none'}**. Identities that "
          f"are not a gallery match are numbered, not named — if a different "
          f"person took over the desk they appear as a separate row above. Add "
          f"their photo to `staff_gallery/` to name them.")
    A("")
    A("## How it was produced")
    A("")
    A(f"- detector `{Path(DETECTOR_MODEL).name}` @ imgsz {YOLO_IMGSZ}, "
      f"analysed at {FPS_TARGET} fps, played back at "
      f"{globals().get('PLAYBACK_FPS', FPS_TARGET)} fps")
    A(f"- tracker `{TRACKER_MODE}`, appearance `{run.get('online_reid')}`")
    A(f"- chunks: {len(run.get('annotated_videos') or [1])}, "
      f"seam stitches {run.get('seam_stitches', 0)}, "
      f"returning guests folded back into one person "
      f"{run.get('returning_guests', 0)}")
    A(f"- frames skipped by the motion gate (nobody there, nothing moving): "
      f"{run.get('frames_skipped_idle', 0)}")
    A("")
    A("## Limits — read before quoting a number")
    A("")
    A("- Per-person guest numbers over a whole night depend on re-identification "
      "across hours. Aggregate numbers (arrivals per 15 min, station coverage, "
      "zone occupancy) do not, and are the defensible ones.")
    A("- Greeting is proximity, never conversation.")
    A("- No ground-truth labels have been scored against this footage yet, so "
      "accuracy is estimated, not measured. Label 2 minutes and run TrackEval "
      "to turn that into a number.")
    return "\n".join(L)


def make_endcard(cam, r, run, out_png, size=(1280, 720)):
    """A summary board burned onto the end of the video, so the video is the
    report for anyone who never opens the spreadsheet."""
    W, H = size
    card = np.full((H, W, 3), 22, np.uint8)
    def txt(s, x, y, sc=0.8, col=(240, 240, 240), th=2):
        cv2.putText(card, s, (x, y), cv2.FONT_HERSHEY_SIMPLEX, sc, col, th, cv2.LINE_AA)
    txt(f"{cam}   {_hm(0)} - {_hm(run['t_end'])}", 40, 70, 1.0, (255, 220, 150), 2)
    txt(f"{run['t_end'] / 3600:.1f} hours of footage", 40, 108, 0.6, (170, 170, 170), 1)
    lines = [
        (f"{r['guests_tonight']}", "people came through the door"),
        (f"{r['greeted']}", "were reached by someone (proximity proxy)"),
        (f"{r['avg_greet_seconds'] if r['avg_greet_seconds'] is not None else '-'}s",
         "average time before someone was with them"),
        (f"{r['waited_over_3min']}", "waited more than 3 minutes"),
        (f"{r['arrived_to_an_empty_desk']}", "arrived while the desk was empty"),
        (f"{r['desk_covered_pct']:.0f}%", "of available footage the desk was covered"),
        (f"{r['longest_desk_gap_min']:.0f} min", "longest single gap at the desk"),
        (f"{r['people_who_came_back']}", "stepped out and came back (counted once)"),
    ]
    _gc = guest_confidence(run)
    lines[0] = (f"{_gc['high']}-{_gc['total']}", "people came through the door "
                                                 "(high-confidence - total)")
    y = 190
    for big, small in lines:
        txt(big, 60, y, 1.1, (120, 220, 255), 3)
        txt(small, 260, y, 0.62, (225, 225, 225), 1)
        y += 58
    txt("EXACT: door crossings, zone occupancy, desk coverage.   "
        "PROXY: greeting = proximity, not conversation.",
        40, H - 40, 0.5, (150, 150, 150), 1)
    cv2.imwrite(str(out_png), card)
    return out_png


REPORTS = {}
for run in runs:
    cam = run["camera_id"]
    r = RECEPTION.get(cam)
    if not r:
        continue
    md = build_report(cam, r, run, LEDGERS[cam], QUIET.get(cam, []))
    p = OUTPUT_DIR / f"{cam}_REPORT.md"
    p.write_text(md)
    REPORTS[cam] = md
    print(f"📝 {p}  ({len(md.splitlines())} lines)")

    png = make_endcard(cam, r, run, OUTPUT_DIR / f"{cam}_endcard.png")
    # burn it onto the end of the LAST annotated chunk (6 s), best effort
    _vids = run.get("annotated_videos_h264") or ([run.get("annotated_video_h264")]
                                                 if run.get("annotated_video_h264") else [])
    if _vids and Path(_vids[-1]).exists():
        last = Path(_vids[-1])
        card_mp4 = OUTPUT_DIR / f"{cam}_endcard.mp4"
        joined = last.with_name(last.stem + "_with_summary.mp4")
        try:
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1",
                            "-t", "6", "-i", str(png), "-vf",
                            f"scale={globals().get('RENDER_MAX_W', 1280)}:-2,"
                            f"fps={globals().get('PLAYBACK_FPS', 30)}",
                            "-c:v", "libx264", "-pix_fmt", "yuv420p", str(card_mp4)],
                           check=True, timeout=180)
            lst = OUTPUT_DIR / "_concat.txt"
            lst.write_text(f"file '{last.resolve()}'\nfile '{card_mp4.resolve()}'\n")
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat",
                            "-safe", "0", "-i", str(lst), "-c", "copy", str(joined)],
                           check=True, timeout=600)
            print(f"🎬 end-card burned on -> {joined.name}")
        except Exception as ex:
            print(f"(end-card not concatenated: {ex} — {png.name} and "
                  f"{card_mp4.name} are still there to show separately)")

from IPython.display import Markdown, display as _disp
for cam, md in REPORTS.items():
    _disp(Markdown(md))


# ── FREE TEXT: ask a question about the night ───────────────────────────────
# Grounded: the model only ever sees the numbers this pipeline produced (the
# ledger, the reception report, the event summary) — it cannot see the video,
# so it cannot invent something that is not in the data.
def _gemini_key():
    k = os.environ.get("GEMINI_API_KEY")
    if k:
        return k
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        return None


def _facts(cam):
    run = next(r for r in runs if r["camera_id"] == cam)
    r = RECEPTION[cam]
    led = LEDGERS[cam]
    return {
        "camera": cam,
        "from": _hm(0), "to": _hm(run["t_end"]),
        "hours": round(run["t_end"] / 3600.0, 2),
        "headline": {k: v for k, v in r.items() if not k.startswith("_")},
        "status_of_each_metric": {k: v[0] for k, v in STATUS.items()},
        "fifteen_min_buckets": r["_buckets"],
        "who_worked_the_station": r["_shifts"],
        "longest_desk_gaps_s": [round(b - a, 1) for a, b in r["_gaps"][:10]],
        "slowest_greets": r["_greets"][:10],
        "walkouts": r["_turnaways"][:10],
        "quiet_stretches": QUIET.get(cam, [])[:20],
        "zone_summary": build_zone_summary(run).to_dict("records"),
        "minutes_with_nobody": int((led["people_present"] == 0).sum()),
        "minutes_total": int(len(led)),
    }


def ask(question, cam=None, model="gemini-2.0-flash"):
    """Free-text question about the night, answered ONLY from the numbers above."""
    cam = cam or runs[0]["camera_id"]
    facts = _facts(cam)
    key = _gemini_key()
    if not key:
        print("No GEMINI_API_KEY (env var or Kaggle secret). Here are the facts "
              "the question would have been answered from:")
        print(json.dumps(facts, indent=2, default=str)[:4000])
        return None
    import urllib.request
    prompt = (
        "You are a restaurant operations analyst. Answer the question using ONLY "
        "the JSON facts below, which come from a computer-vision pipeline on one "
        "camera. Rules: quote the numbers you used; if a metric is marked PROXY "
        "say so in one short clause; if the facts do not contain the answer, say "
        "exactly what is missing instead of guessing. Be brief and concrete.\n\n"
        f"QUESTION: {question}\n\nFACTS:\n{json.dumps(facts, default=str)}")
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode()
    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{model}:generateContent?key={key}")
    try:
        req = urllib.request.Request(url, data=body,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=90) as resp:
            out = json.loads(resp.read())
        text = out["candidates"][0]["content"]["parts"][0]["text"]
        print(text)
        return text
    except Exception as ex:
        print(f"(ask failed: {ex}) — falling back to the raw facts")
        print(json.dumps(facts, indent=2, default=str)[:4000])
        return None


print("\n💬 ask() is ready. Examples:")
print('   ask("how many people waited more than 3 minutes and when?")')
print('   ask("when was the desk left empty the longest, and did anyone walk in then?")')
print('   ask("was the second half of the night busier than the first?")')


# ── V67 [PATCH_V67_NIGHTJSON]: the single-JSON night summary ────────────────
# One machine-readable file per camera: narrative + counts + every anomaly
# with timestamps. This is the product's data interface; Excel stays for
# humans. Appended last so every helper above is already defined.
for _run67 in runs:
    _cam67 = _run67["camera_id"]
    if _cam67 not in RECEPTION:
        continue
    _r67 = RECEPTION[_cam67]
    _f67 = _facts(_cam67)
    _anoms = []

    def _a67(t, typ, detail, sev):
        _anoms.append({"t_s": round(float(t or 0), 1), "clock": _hm(t or 0),
                       "type": typ, "detail": detail, "severity": sev})

    for _w in (_r67.get("_turnaways") or [])[:50]:
        _a67(_w.get("arrived"), "walkout_unserved",
             f"guest left after {round(_w.get('seconds_inside') or 0)}s "
             f"with no staff contact", "high")
    for _g in (_r67.get("_greets") or [])[:100]:
        if (_g.get("latency_s") or 0) > 180:
            _a67(_g.get("arrived"), "slow_greet",
                 f"waited {round(_g['latency_s'])}s before first staff "
                 f"contact (PROXY: proximity, not conversation)", "medium")
    for _ga, _gb in (_r67.get("_gaps") or [])[:20]:
        if (_gb - _ga) >= 300:
            _a67(_ga, "desk_uncovered",
                 f"reception empty for {round((_gb - _ga) / 60, 1)} min",
                 "high")
    for _t67, _v67 in _run67.get("ir_switches", []):
        _a67(_t67, "camera_mode_switch",
             "switched to infrared" if _v67 else "switched to colour", "info")
    if _r67.get("occupancy_went_negative"):
        _a67(0, "count_inconsistency",
             f"running IN-OUT went negative "
             f"{_r67['occupancy_went_negative']}x (worst deficit "
             f"{_r67.get('occupancy_worst_deficit', 0)}) — direct measure of "
             f"counting damage", "medium")
    _movers67 = len({_e["track_id"] for _e in _run67.get("events", [])})
    if not _run67.get("crossings") and _movers67 >= 5:
        _a67(0, "entry_line_dead",
             f"{_movers67} people moved through zones, the entry line fired "
             f"0x — arrival numbers use the region fallback", "high")
    _seen67 = _run67.get("size_seen", 0)
    _drop67 = _run67.get("size_dropped", 0)
    if _seen67 and _drop67 / _seen67 > 0.12:
        _a67(0, "d1_high_drop",
             f"size filter dropped {_drop67}/{_seen67} detections "
             f"({100 * _drop67 / _seen67:.0f}%) — scene-geometry fit suspect; "
             f"supply ground_points", "medium")
    _anoms.sort(key=lambda _x: _x["t_s"])

    _brief_p = OUTPUT_DIR / f"{_cam67}_brief.txt"
    _hl = _f67.get("headline", {})
    _sum67 = {
        "schema": "keva-vision/night-summary/v1",
        "build": str(globals().get("_BUILD_ID", "?")),
        "camera": _cam67,
        "window": {"from": _f67.get("from"), "to": _f67.get("to"),
                   "hours": _f67.get("hours")},
        "narrative": (_brief_p.read_text() if _brief_p.exists()
                      else "(brief not generated)"),
        "counts": {
            "guests_tonight": _hl.get("guests_tonight"),
            "door_crossings_in": _hl.get("door_crossings_in"),
            "people_who_came_back": _hl.get("people_who_came_back"),
            "busiest_15min": _hl.get("busiest_15min"),
            "peak_arrivals_in_15min": _hl.get("peak_arrivals_in_15min"),
        },
        "service": {
            "greeted": _hl.get("greeted"),
            "median_greet_seconds": _hl.get("median_greet_seconds"),
            "waited_over_3min": _hl.get("waited_over_3min"),
            "left_without_being_served": _hl.get("left_without_being_served"),
            "desk_covered_pct": _hl.get("desk_covered_pct"),
            "longest_desk_gap_min": _hl.get("longest_desk_gap_min"),
        },
        "metric_tiers": _f67.get("status_of_each_metric", {}),
        "anomalies": _anoms,
        "facts": _f67,
    }
    _sp67 = OUTPUT_DIR / f"{_cam67}_night_summary.json"
    _sp67.write_text(json.dumps(_sum67, indent=2, default=str))
    print(f"🧾 night summary (single JSON, {len(_anoms)} anomalies) -> "
          f"{_sp67.name}")


In [ ]:
# Cell 22 — GROUND TRUTH: pick slices, export for labelling, score  [EVAL_PHASE2]
# The gate for every future change. Until a gt.txt exists, every accuracy
# statement in the report is an ESTIMATE and is labelled as one.
import json, zipfile, shutil
from pathlib import Path

EVAL_DIR = OUTPUT_DIR / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

LABEL_WINDOW_S   = 120.0   # 2 min per condition. Long enough to contain real
                           # entries/exits and occlusions, short enough that a
                           # human will actually finish labelling it.
SWITCH_WINDOW_S  = 60.0    # a switch slice only needs to straddle the flip
MIN_PEOPLE_FOR_WINDOW = 1  # never ship an empty slice to be labelled


def _ir_at(run, t):
    """IR state at time t, reconstructed from the switch log Phase 1 records."""
    state = False
    for st, val in run.get("ir_switches", []):
        if st <= t:
            state = bool(val)
        else:
            break
    return state


def pick_eval_windows(run, want_s=LABEL_WINDOW_S):
    """Choose WHICH slices to label — one per condition, densest first.

    A single window is not enough. Daylight and infrared are different problems
    (colour evidence exists in one and not the other), and the moment the camera
    flips is a third. Scoring them together would average a broken night into a
    working day and show neither.
    """
    flog = run.get("frame_log") or []
    if not flog:
        return []
    series = [(t, len(b)) for _fi, t, b in flog]
    series.sort()
    ts = [t for t, _ in series]
    ns = [n for _, n in series]
    step = (ts[-1] - ts[0]) / max(1, len(ts) - 1) if len(ts) > 1 else 0.25
    k = max(1, int(round(want_s / max(step, 1e-6))))

    def best(mask, width_s):
        kk = max(1, int(round(width_s / max(step, 1e-6))))
        best_i, best_v = None, -1.0
        for i in range(0, max(1, len(ts) - kk)):
            # B3: a window is only INFRARED if most of it is. Requiring merely
            # "any" IR frame made night_ir_busy pick the exact same window as
            # day_busy, so the night condition was never actually sampled.
            if sum(1 for v in mask[i:i + kk] if v) < 0.6 * kk:
                continue
            v = sum(ns[i:i + kk])
            if v > best_v:
                best_v, best_i = v, i
        if best_i is None or best_v <= 0:
            return None
        return (ts[best_i], min(ts[best_i] + width_s, ts[-1]), best_v / kk)

    ir = [_ir_at(run, t) for t in ts]
    out = []
    day_mask = [not v for v in ir]
    w = best(day_mask, want_s)
    if w and w[2] >= MIN_PEOPLE_FOR_WINDOW:
        out.append(("day_busy", w[0], w[1], w[2]))
    w = best(ir, want_s)
    if w and w[2] >= MIN_PEOPLE_FOR_WINDOW:
        out.append(("night_ir_busy", w[0], w[1], w[2]))
    sw = [s for s, _v in run.get("ir_switches", [])[1:]]
    if sw:
        # the flip nearest to people being present, not just the first one
        cand = sorted(sw, key=lambda s: -sum(
            n for t, n in series if abs(t - s) <= SWITCH_WINDOW_S / 2))
        s0 = max(ts[0], cand[0] - SWITCH_WINDOW_S / 2)
        # B3: a MEAN, not a sum. The first run reported "avg 2190.0 people in
        # frame" for the switch window, which is a total wearing an average's
        # label and makes the three conditions incomparable.
        _in = [n for t, n in series if s0 <= t <= s0 + SWITCH_WINDOW_S]
        dens = (sum(_in) / len(_in)) if _in else 0.0
        if dens > 0:
            out.append(("ir_switch", s0, s0 + SWITCH_WINDOW_S, dens))
    return out


def export_label_package(camera_id, video_path, run, name, t0, t1, out_dir=EVAL_DIR):
    """Frames + our predictions + manifest + CVAT instructions, zipped.

    Frames are matched to analysed frames BY TIME. CVAT will renumber the images
    1..N in filename order, so the predictions are written with that SAME
    numbering. Index-based alignment would be one off-by-one away from a score
    that is confidently wrong.
    """
    pkg = out_dir / f"{camera_id}_{name}"
    if pkg.exists():
        shutil.rmtree(pkg)
    (pkg / "images").mkdir(parents=True, exist_ok=True)

    flog = sorted(((t, b) for _fi, t, b in (run.get("frame_log") or [])),
                  key=lambda r: r[0])
    if not flog:
        print(f"  !! {name}: no frame_log — cannot export")
        return None
    log_ts = [t for t, _ in flog]
    eff = run.get("eval_fps") or FPS_TARGET
    tol = 0.5 / max(eff, 1e-6)

    import bisect
    rows, n_written, unmatched = [], 0, 0
    for _i, t, frame in frame_source(video_path, eff, max_seconds=(t1 - t0),
                                     max_w=ANALYSIS_MAX_W, start_seconds=t0):
        n_written += 1
        cv2.imwrite(str(pkg / "images" / f"{n_written:06d}.jpg"), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        j = bisect.bisect_left(log_ts, t)
        cand = [c for c in (j - 1, j) if 0 <= c < len(log_ts)]
        hit = min(cand, key=lambda c: abs(log_ts[c] - t)) if cand else None
        if hit is None or abs(log_ts[hit] - t) > tol:
            unmatched += 1
            continue
        for tid, x1, y1, x2, y2 in flog[hit][1]:
            canon = (run.get("canon_map") or {}).get(tid, tid)
            rows.append((n_written, canon, x1, y1, x2 - x1, y2 - y1))

    if n_written == 0:
        # F5a: 0 > 0*0.05 is False — an empty seek used to ship an EMPTY
        # package that scored as if the pipeline saw nobody. Refuse instead.
        print(f"  !! {name}: seek produced ZERO frames — the window "
              f"({t0:.0f}-{t1:.0f}s) is not inside {Path(video_path).name}. "
              f"Package refused.")
        shutil.rmtree(pkg, ignore_errors=True)
        return None
    if unmatched > n_written * 0.05:
        print(f"  !! {name}: {unmatched}/{n_written} exported frames had no "
              f"matching analysed frame. REFUSING to build this package — the "
              f"labels would be compared against the wrong frames.")
        shutil.rmtree(pkg, ignore_errors=True)
        return None

    # ids must be small positive ints for MOT; staff carry string names
    idmap, num = {}, 0
    mot = []
    for fr, tid, x, y, w, h in rows:
        if tid not in idmap:
            num += 1
            idmap[tid] = num
        mot.append((fr, idmap[tid], x, y, w, h))
    write_mot(pkg / "predictions.txt", mot)

    manifest = {
        "camera_id": camera_id, "condition": name,
        "window_video_s": [round(t0, 2), round(t1, 2)],
        "window_clock": [wall(t0) or mmss(t0), wall(t1) or mmss(t1)],
        "frames": n_written, "analysis_fps": eff,
        "frame_size_analysed": [ANALYSIS_MAX_W, None],
        "is_infrared": _ir_at(run, (t0 + t1) / 2),
        "n_pred_boxes": len(mot), "n_pred_ids": len(idmap),
        "source_video": Path(video_path).name,   # ABL: to re-run this window
        "build": str(globals().get("_BUILD_ID", "?")),
        "id_map_pipeline_to_mot": {str(k): v for k, v in idmap.items()},
        "config": {k: globals().get(k) for k in (
            "DETECTOR_MODEL", "TRACKER_MODE", "ANALYSIS_FPS", "YOLO_IMGSZ",
            "ANALYSIS_MAX_W", "CONF_THRESHOLD", "DETECT_CONF_FLOOR",
            "NEW_TRACK_CONF", "REID_SIM_THRESHOLD", "ANCHOR_SIM_THRESHOLD",
            "LIVE_REID_MAX_SPEED_PX_S", "CALIBRATION_AUTO_APPLY",
            "ENABLE_CARRIED_SUPPRESS", "CARRIED_HEIGHT_TOL",
            "CARRIED_MIN_HEAD_DROP", "MOTION_GATE", "GAP_MERGE_S")},
    }
    manifest["config"]["DETECTOR_MODEL"] = str(manifest["config"]["DETECTOR_MODEL"])
    (pkg / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    (pkg / "HOW_TO_LABEL.txt").write_text(f"""HOW TO LABEL THIS PACKAGE   ({camera_id} / {name})
{'=' * 70}
{n_written} frames, {t1 - t0:.0f} seconds, {manifest['window_clock'][0]} -> {manifest['window_clock'][1]}
This slice is {'INFRARED / night vision' if manifest['is_infrared'] else 'daylight / colour'}.

WHY: nothing in this project has ever been measured. This is the file that
turns "we think it improved" into a number, and it only has to be made once.

STEPS
 1. Go to cvat.ai (free account) -> Create new task.
 2. Upload the images/ folder from this package (all {n_written} files).
 3. Add ONE label called:  person
 4. Open the task and draw a box around EVERY person, in EVERY frame.

RULES THAT DECIDE WHETHER THE SCORE IS MEANINGFUL
 * ONE track id per real human being. If someone walks out and comes back,
   reuse their original id. That is precisely what we are testing.
 * Box the WHOLE person, including the parts hidden behind the desk or
   another person - estimate where the hidden edge is. Our pipeline is
   judged on whether it finds the PERSON, not the visible pixels.
 * Include staff. Include children. Include someone being carried.
 * Include anyone even partly in frame, however small, as long as you can
   tell it is a person.
 * If you genuinely cannot tell whether something is a person, skip it -
   a wrong label is worse than a missing one.
 * Do not label reflections in glass, posters, or people on a screen.
 * Use CVAT's interpolation. You do not need to draw all {n_written} frames
   by hand - draw keyframes and let it fill in, then fix the drift.

WHEN DONE
 5. Menu -> Export task dataset -> format "MOT 1.1" -> download.
 6. Rename the gt.txt inside it to:   {camera_id}_{name}_gt.txt
 7. Put it in the Kaggle dataset (or in poc_output/eval/) and re-run Cell 22.
    It will print HOTA / DetA / AssA for this condition.

WHAT IS ALREADY IN HERE
   images/            the frames to label
   predictions.txt    what OUR pipeline currently says, in MOT 1.1 format.
                      Do NOT look at this before labelling.
                      Reading it first biases you toward agreeing with the very
                      thing we are trying to measure, and the score becomes
                      meaningless. Label blind, compare afterwards.
   manifest.json      exact config that produced predictions.txt, so a future
                      A/B knows what changed.
""")
    zp = out_dir / f"{camera_id}_{name}_to_label.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(pkg.rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(pkg))
    print(f"  package: {zp.name}  ({n_written} frames, {len(mot)} pred boxes, "
          f"{len(idmap)} identities, {zp.stat().st_size // 1024} KB)")
    return pkg


# ---------------------------------------------------------------------------
for run in runs:
    cam = run["camera_id"]
    vpath = next((v["video"] for v in VIDEO_QUEUE if v["camera_id"] == cam), None)
    print("=" * 78)
    print(f"  GROUND TRUTH · {cam}")
    print("=" * 78)

    windows = pick_eval_windows(run)
    if not windows:
        print("  no frame_log — nothing to export")
    else:
        print(f"  slices worth labelling ({len(windows)}):")
        for name, t0, t1, dens in windows:
            print(f"    {name:16s} {mmss(t0)} -> {mmss(t1)}  "
                  f"({wall(t0) or 'video time'})  avg {dens:.1f} people in frame")
        have = {w[0] for w in windows}
        for missing, why in (("night_ir_busy", "no infrared frames in this footage"),
                             ("ir_switch", "no colour<->IR switch in this footage")):
            if missing not in have:
                print(f"    {missing:16s} NOT AVAILABLE — {why}. That condition "
                      f"stays UNMEASURED; do not claim accuracy for it.")

    # score whatever ground truth already exists
    pairs = {}
    for name, *_ in windows:
        pkg = EVAL_DIR / f"{cam}_{name}"
        gt = next((p for p in [EVAL_DIR / f"{cam}_{name}_gt.txt",
                               EVAL_DIR / f"{name}_gt.txt"] if p.exists()), None)
        if gt is None and INPUT_ROOT.exists():
            gt = next(INPUT_ROOT.rglob(f"{cam}_{name}_gt.txt"), None)
        if gt is not None and (pkg / "predictions.txt").exists():
            pairs[name] = (gt, pkg / "predictions.txt")

    if pairs:
        print(f"\n  scoring {len(pairs)} labelled condition(s)")
        EVAL_RESULTS = score_conditions(pairs, out_dir=EVAL_DIR)
    else:
        EVAL_RESULTS = {}
        print("\n  no ground truth found yet — building the labelling packages")
        if vpath and Path(vpath).exists():
            for name, t0, t1, _d in windows:
                # F5b: on a multi-chunk night the window times are NIGHT-time
                # but vpath is chunk 1 — resolve to the chunk that actually
                # contains the window, re-pulling it if it was deleted.
                _vp, _t0, _t1 = Path(vpath), t0, t1
                for _cb in (run.get("chunk_bounds") or []):
                    _k, _off, _end = int(_cb[0]), float(_cb[1]), float(_cb[2])
                    if _off <= t0 < _end:
                        _rem = globals().get("QUEUE_REMOTE") or []
                        if _k < len(_rem):
                            _cand = DRIVE_DIR / _rem[_k]["name"]
                            if not _cand.exists():
                                try:
                                    _cand = Path(_pull(_rem[_k]))
                                except Exception as _pe:
                                    print(f"  !! {name}: chunk re-pull failed "
                                          f"({_pe}) — using merged-video path")
                                    _cand = None
                            if _cand is not None and Path(_cand).exists():
                                _vp, _t0, _t1 = Path(_cand), t0 - _off, t1 - _off
                        break
                export_label_package(cam, _vp, run, name, _t0, _t1)
            print(f"\n  -> download the *_to_label.zip files from the Output panel,")
            print(f"     follow HOW_TO_LABEL.txt, put the gt.txt back, re-run this cell.")
        else:
            print("  !! source video is gone (chunks are deleted as they finish) —")
            print("     re-run with RESUME=True and MAX_CHUNKS=1 to rebuild a package.")
        print("\n  UNTIL THEN: every accuracy statement in the report is an "
              "ESTIMATE, not a measurement.")


In [ ]:
# Cell 9e — ABLATION RUNNER  [PATCH_V58_PHASEC]
# Session-2 workflow: ONE Kaggle session answers every Phase C question.
#   1. Set RUN_ABLATION = True in Cell 2, Run All. The night runner is
#      skipped; each labelled eval window is re-processed under each variant.
#   2. Download the exported packages, score locally (seconds, free):
#        python gt_kit.py score <pkg_variant> <window_gt.txt>
#        python gt_kit.py compare baseline_score.json <variant_score.json>
# Variants only need to BEAT the baseline on the frozen gt — flags that lose
# stay off. ~4 min per (window, variant) on a T4: 3 windows x 4 variants ≈ 50m.
ABLATION_VARIANTS = globals().get("ABLATION_VARIANTS") or {
    "baseline":   {},
    "occluboost": {"TRACKER_MODE": "occluboost"},
    "ir_hardcut": {"ENABLE_IR_HARD_CUT": True},
    "no_sweep":   {"ENABLE_STAFF_GALLERY_SWEEP": False},
    # PATCH_V58P2: ~3.8k head-only detections/hour sit unused; whether
    # recovering them helps recall or floods the tracker is a measurement.
    "head_recov": {"ENABLE_HEAD_RECOVERY": True},
    # PATCH_V68: best.pt is a 10-epoch CrowdHuman tune with 70% recall on its
    # OWN val set — whether it beats stock yolo11x on THIS footage has never
    # been measured. The gt scores decide, not faith in a fine-tune.
    "stock_det":  {"DETECTOR_MODEL": "yolo11x.pt"},
}

if globals().get("RUN_ABLATION"):
    import json as _json

    _mans = sorted({p for p in list(EVAL_DIR.glob("*/manifest.json"))
                    + list(INPUT_ROOT.rglob("*_to_label*/manifest.json"))
                    + list(INPUT_ROOT.rglob("eval/*/manifest.json"))})
    if not _mans:
        print("RUN_ABLATION: no eval manifests found — run the night once "
              "first (Cell 22 exports the label packages this re-runs).")
    _done = []
    for _mp in _mans:
        _m = _json.loads(_mp.read_text())
        _cond = _m.get("condition")
        _srcv = _m.get("source_video")
        _t0, _t1 = _m.get("window_video_s", [None, None])
        _cam = _m.get("camera_id", globals().get("CAMERA_ID", "CAM"))
        if not (_srcv and _t0 is not None):
            print(f"  !! {_mp.parent.name}: manifest predates Phase C "
                  f"(no source_video) — re-export it once, then ablate.")
            continue
        if "__" in str(_cond):
            continue                     # a variant package, not an original
        _vid = next((p for p in [DRIVE_DIR / _srcv]
                     + list(INPUT_ROOT.rglob(_srcv)) if p.exists()), None)
        if _vid is None and globals().get("QUEUE_REMOTE"):
            _rem = next((r for r in QUEUE_REMOTE
                         if r.get("name") == _srcv), None)
            if _rem is not None:
                print(f"  pulling {_srcv} from Drive for ablation...")
                _vid = _pull(_rem)
        if _vid is None:
            print(f"  !! {_cond}: source video {_srcv} not found — skipped")
            continue
        _zp = (VIDEO_QUEUE[0]["zones_path"] if globals().get("VIDEO_QUEUE")
               else None)
        for _vn, _over in ABLATION_VARIANTS.items():
            _saved = {k: globals().get(k) for k in _over}
            globals().update(_over)
            print("=" * 70)
            print(f"  ABLATION · {_cond} · variant {_vn} "
                  f"({_over if _over else 'as-configured'})")
            print("=" * 70)
            try:
                _r = process_video(_cam, Path(_vid), _zp,
                                   max_seconds=(_t1 - _t0),
                                   start_seconds=_t0,
                                   chunk_tag=f"_abl_{_cond}_{_vn}")
                _pkg = export_label_package(_cam, Path(_vid), _r,
                                            f"{_cond}__{_vn}", _t0, _t1)
                if _pkg:
                    _done.append(f"{_cond}__{_vn}")
            except Exception as _ae:
                print(f"  !! variant {_vn} FAILED: {_ae!r} — other variants "
                      f"continue")
            finally:
                globals().update(_saved)
    print("=" * 70)
    print(f"  ABLATION DONE: {len(_done)} package(s): {_done}")
    print("  download poc_output/eval/*__*/predictions.txt and score locally:")
    print("    python gt_kit.py score <pkg> <the SAME window gt.txt>")
else:
    print("RUN_ABLATION = False — ablation runner skipped (Session-2 tool).")


## Notes — what this run needs, what it produces, what it cannot do

### What must be in the Drive folder before Run All
| item | name | what breaks without it |
|---|---|---|
| the chunks | `CAM.112 ... 7-28-2026, 4.30.00pm CDT - ....mp4` | nothing runs. The **date/time in the filename is the clock** — every wall-clock number comes from it |
| the zone map | any `*.json` with `zone` in the name | Cell 2d stops and writes `frame_0.jpg` for you to draw on |
| a polygon named `reception` (or `staff`/`host`/`counter`/`desk`) | inside that JSON | every staff metric comes back empty |
| an `entry_line` of 2 points | inside that JSON | "people entered" cannot be counted at all |
| staff photos | `sarah.jpg`, `mike.jpg`, ... | staff are numbered, not named, and per-person staff numbers over-count. Filename = the name shown on screen |

Zone names decide what is measured (`entry / wait / staff / seating / service / mask`).
A polygon named `mask`/`ignore`/`mirror`/`reflection` becomes a **dead area** —
detections there are dropped, which is how you kill reflections and posters.

### Scale settings that matter (Cell 2e)
`ANALYSIS_FPS` is accuracy, `PLAYBACK_FPS` is watchability, and they are
independent. `PROVE_SECONDS = 600` gives a 10-minute proof of one chunk before
committing a night. `RESUME = True` skips chunks that already have an artifact
in `poc_output/chunk_events/` — a crash costs one chunk, not the run.

### What is EXACT
Door crossings, zone dwell, station coverage, per-minute occupancy — geometry
over tracked positions, all traceable to `*_events.csv` and the annotated video.

### What is a PROXY (say it out loud)
"Greeted" = a staff box within 1.2 body-heights of a guest for ≥3 s. Proximity
is not conversation. "Order taken"/"food arrived" (venues with tables) = first
and second distinct staff visit. Present these as *service-touch timings*.

### Known limits
- **Identity across hours** is the weak point. Within a chunk it is strong
  (BoTSORT + real CLIP-ReID + a 6-tier offline stitcher). Across chunks it
  relies on the seam pass (±90 s) plus the whole-night guest matcher, so
  absolute "unique people over 10 h" is the least certain number on the page.
  Aggregate numbers do not depend on it.
- **Nothing has been scored against ground truth yet.** Accuracy here is
  estimated, not measured. Label two minutes in CVAT and run TrackEval to
  replace the estimate with HOTA/IDF1.
- The annotated video **skips stretches with nobody in them** and is
  fast-forwarded; the HUD shows the real wall clock and says when time was
  skipped. The per-minute ledger (Cell 20) is what covers every second.
- Infrared: colour-based attire matching is disabled automatically per chunk
  when the camera flips to IR, because colour no longer exists in the image.
- Zones are per-camera pixel coordinates — redraw for a new camera angle.


In [ ]:
# ============================================================================
# STAFF GALLERY BUILDER (run ONCE, then never again)
# ============================================================================
# Extracts the clearest face crop per person from your video.
# After running, look at the saved images, rename staff ones
# (e.g. "person_3.jpg" -> "jane.jpg"), and move them into staff_gallery/.
# Then re-run the main pipeline — the gallery will auto-load.
# ============================================================================

import cv2, os, shutil
import numpy as np
from pathlib import Path
from collections import defaultdict

# --- CONFIG (edit these) ---
VIDEO_PATH = None  # Set to your video path, e.g. "/kaggle/input/your-dataset/jane_camera.mp4"
                   # If None, auto-discovers from VIDEO_QUEUE
FACES_DIR = "extracted_faces"
SAMPLE_EVERY_N_FRAMES = 30  # Sample 1 frame every N (lower = more faces, slower)
MIN_FACE_SIZE = 50          # Minimum face bbox width in pixels
MIN_DET_SCORE = 0.60        # Minimum face detection confidence
MAX_FACES_PER_CLUSTER = 1   # Save top-N best crops per person
# ---------------------------

def build_staff_gallery():
    # Auto-discover video if not set
    video_path = VIDEO_PATH
    if video_path is None:
        if 'VIDEO_QUEUE' in dir() or 'VIDEO_QUEUE' in globals():
            for v in VIDEO_QUEUE:
                if v["video"].exists():
                    video_path = str(v["video"])
                    break
    if video_path is None:
        print("ERROR: Set VIDEO_PATH to your video file, or run the discovery cell first.")
        return

    print(f"Video: {video_path}")

    # Ensure InsightFace is available
    try:
        from insightface.app import FaceAnalysis
    except ImportError:
        print("ERROR: insightface not installed. Run the install cell first.")
        return

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                 if device == "cuda" else ["CPUExecutionProvider"])

    model_name = FACE_MODEL_NAME if 'FACE_MODEL_NAME' in globals() else "buffalo_sc"
    try:
        app = FaceAnalysis(name=model_name, providers=providers)
    except Exception:
        app = FaceAnalysis(name="buffalo_sc", providers=providers)
        model_name = "buffalo_sc"
    app.prepare(ctx_id=0 if device == "cuda" else -1, det_size=(320, 320))
    print(f"Face model: {model_name}")

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    print(f"Frames: {total_frames}, FPS: {fps:.1f}, Sampling every {SAMPLE_EVERY_N_FRAMES} frames")

    # Collect all face detections with embeddings
    all_faces = []  # list of (embedding, crop_image, det_score, frame_idx)
    frame_idx = 0

    while True:
        # decode-skip: cap.read() on EVERY frame does the full YUV->BGR
        # convert + ~24MB copy per 4K frame just to throw 29 of 30 away —
        # that's why the shipped run was still at 40% when it was cut off.
        # grab() advances without decoding.
        if frame_idx % SAMPLE_EVERY_N_FRAMES:
            if not cap.grab():
                break
            frame_idx += 1
            continue
        ret, frame = cap.read()
        if not ret:
            break
        if True:
            # BGR straight in — InsightFace expects cv2's BGR (the main
            # pipeline feeds BGR crops); the RGB convert was wrong AND slow
            faces = app.get(frame)
            for face in faces:
                bbox = face.bbox.astype(int)
                w = bbox[2] - bbox[0]
                if w < MIN_FACE_SIZE:
                    continue
                if face.det_score < MIN_DET_SCORE:
                    continue
                # Pad the crop slightly for better recognition
                h_frame, w_frame = frame.shape[:2]
                pad = int(w * 0.3)
                x1 = max(0, bbox[0] - pad)
                y1 = max(0, bbox[1] - pad)
                x2 = min(w_frame, bbox[2] + pad)
                y2 = min(h_frame, bbox[3] + pad)
                crop = frame[y1:y2, x1:x2].copy()
                all_faces.append({
                    "embedding": face.embedding,
                    "crop": crop,
                    "score": float(face.det_score),
                    "frame": frame_idx,
                    "time_s": frame_idx / fps,
                })
            if frame_idx % (SAMPLE_EVERY_N_FRAMES * 50) == 0:
                pct = frame_idx / max(total_frames, 1) * 100
                print(f"  Scanned {frame_idx}/{total_frames} frames ({pct:.0f}%) — {len(all_faces)} faces so far")
        frame_idx += 1

    cap.release()
    print(f"\nTotal face detections: {len(all_faces)}")

    if not all_faces:
        print("No faces found! Try lowering MIN_FACE_SIZE or MIN_DET_SCORE.")
        return

    # --- Cluster faces by identity (greedy nearest-neighbor) ---
    clusters = []  # list of lists of face dicts
    CLUSTER_THRESHOLD = 0.45  # cosine similarity to consider same person

    for face in sorted(all_faces, key=lambda f: -f["score"]):
        emb = face["embedding"]
        norm = np.linalg.norm(emb)
        if norm < 1e-6:
            continue
        emb_normed = emb / norm

        best_cluster = None
        best_sim = -1
        for ci, cluster in enumerate(clusters):
            # Compare against cluster centroid (average of all embeddings)
            centroid = np.mean([f["embedding"] / np.linalg.norm(f["embedding"])
                               for f in cluster], axis=0)
            centroid /= np.linalg.norm(centroid)
            sim = float(np.dot(emb_normed, centroid))
            if sim > best_sim:
                best_sim = sim
                best_cluster = ci

        if best_sim >= CLUSTER_THRESHOLD and best_cluster is not None:
            clusters[best_cluster].append(face)
        else:
            clusters.append([face])

    # Sort clusters by size (most frequently seen person first)
    clusters.sort(key=lambda c: -len(c))
    print(f"Found {len(clusters)} distinct people")

    # --- Save best crop per person ---
    out_dir = Path(FACES_DIR)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for pi, cluster in enumerate(clusters):
        # Sort by detection score, save top N
        cluster.sort(key=lambda f: -f["score"])
        for ci in range(min(MAX_FACES_PER_CLUSTER, len(cluster))):
            face = cluster[ci]
            fname = f"person_{pi+1}.jpg" if ci == 0 else f"person_{pi+1}_alt{ci}.jpg"
            path = out_dir / fname
            cv2.imwrite(str(path), face["crop"])
            t = face["time_s"]
            mins, secs = int(t // 60), t % 60
            print(f"  Saved {fname} (score={face['score']:.2f}, "
                  f"time={mins}m{secs:.0f}s, cluster_size={len(cluster)})")

    print(f"\n{'='*60}")
    print(f"  {len(clusters)} face crops saved to: {out_dir}/")
    print(f"  ")
    print(f"  NEXT STEPS:")
    print(f"  1. Look at the images — identify which ones are staff")
    print(f"  2. Rename them: person_3.jpg -> jane.jpg")
    print(f"  3. Copy/move them to staff_gallery/")
    print(f"  4. Re-run the main pipeline (it auto-loads the gallery)")
    print(f"{'='*60}")

build_staff_gallery()
